# ARC-AGI-2: conceptual lattice + RDR over ARCGraph objects (symbolic, no neural model)
Frozen prior vocabulary `fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl`. Per task, RDR decision lists over lattice concepts are induced
from the training pairs only and executed on the test inputs. Unresolved slots get a 1x1 placeholder,
so any non-zero score comes from the induced rules.



**Probe:** V21 (V20 + fill-largest-patch v2 + near-miss fallback for tasks with no verified program; deterministic)

In [ ]:
import hashlib, json, subprocess, sys, zipfile, os
from pathlib import Path
BUNDLE_SHA = '9eedcc404d6015a2c84db58c63ec97c3e55f282faaed5fb9eb479afe0a6ab4fb'
bundles = [p for p in Path('/kaggle/input').rglob('candidate.bundle')]
assert len(bundles) == 1, bundles
payload = bundles[0].parent
assert hashlib.sha256(bundles[0].read_bytes()).hexdigest() == BUNDLE_SHA
cand = Path('/kaggle/working/cand')
if not cand.exists():
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(cand)
deps = Path('/kaggle/working/deps')
try:
    import rdflib  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', str(deps),
                    str(payload / 'rdflib-7.6.0-py3-none-any.whl')], check=True)
challenges = [p for p in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')]
assert len(challenges) == 1, challenges
Path('/kaggle/working/probe').mkdir(exist_ok=True)
CODEX_SHA = '62d4c71378e546093b874b9a1a1ffc61758b85cd531c06b5d4ecab33df18693b'
codex = [p for p in Path('/kaggle/input').rglob('codex_atoms.tgz')]
if codex:
    import tarfile
    assert hashlib.sha256(codex[0].read_bytes()).hexdigest() == CODEX_SHA, 'codex atom bundle digest mismatch'
    cx = Path('/kaggle/working/codex')
    if not cx.exists():
        with tarfile.open(codex[0]) as t: t.extractall(cx)
    os.environ['CODEX_ROOT'] = str(cx); os.environ['S0_REPS'] = str(cx / 's0_reps.json')
print(payload, challenges[0], 'codex' if codex else 'no-codex')


In [ ]:
%%writefile /kaggle/working/probe/occupancy2.py
"""M1 occupancy probe: prior-concept layer + per-task conceptual lattice + RDR decision list.

Segmentation comes from registered ARCGraph abstractions. Attributes come only from a
fixed, task-independent prior layer (topology, geometry, number/extremes, property
chains). Rules are lattice concepts (extent = closure of a minimal generator of <= 2
attributes) arranged as an RDR-style decision list. No task identifiers; outputs are
used only on training pairs.
"""
from __future__ import annotations

import copy, json, os, signal, sys, time
from collections import Counter
from itertools import combinations
from types import SimpleNamespace

sys.path[:0] = ["/home/claude/work/stubs", "/home/claude/work/cand2"]
from image import Image  # noqa: E402

ABSTRACTIONS = ("nbccg", "ccgbr", "nbvcg", "nbhcg", "mcccg")
DIRS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
MAX_RULES = 5


class Timeout(BaseException):
    pass


# ---------------------------------------------------------------- segmentation (ARCGraph)
def segment(grid, abstraction):
    image = Image(SimpleNamespace(stop_search=False, check_time_limit=lambda: False,
                                  max_abstract_nodes=64), grid=copy.deepcopy(grid), name="m1")
    graph = getattr(image, Image.abstraction_ops[abstraction])()
    nodes = []
    for _, d in graph.graph.nodes(data=True):
        pix = [tuple(p) for p in d["nodes"]]
        colors = {p: grid[p[0]][p[1]] for p in pix}
        vals = set(colors.values())
        nodes.append({"pix": frozenset(pix), "colors": colors,
                      "color": next(iter(vals)) if len(vals) == 1 else None})
    if not nodes or len(nodes) > 64:
        raise ValueError("node_count")
    return nodes


def background(grid):
    return Counter(c for row in grid for c in row).most_common(1)[0][0]


# ---------------------------------------------------------------- prior layer
def norm_shape(pix):
    r0 = min(r for r, _ in pix); c0 = min(c for _, c in pix)
    return frozenset((r - r0, c - c0) for r, c in pix)


def bbox(pix):
    rs = [r for r, _ in pix]; cs = [c for _, c in pix]
    return min(rs), min(cs), max(rs), max(cs)


def has_hole(pix):
    r0, c0, r1, c1 = bbox(pix)
    if r1 - r0 < 2 or c1 - c0 < 2:
        return False
    free = {(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)} - set(pix)
    stack = [(r0 - 1, c0 - 1)]; seen = set(stack)
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            q = (r + dr, c + dc)
            if q in free and q not in seen:
                seen.add(q); stack.append(q)
    return len(seen) < len(free)


def interior(pix):
    """Cells enclosed by pix (not reachable from outside its bbox)."""
    r0, c0, r1, c1 = bbox(pix)
    free = {(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)} - set(pix)
    stack = [(r0 - 1, c0 - 1)]; seen = set(stack)
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            q = (r + dr, c + dc)
            if q in free and q not in seen:
                seen.add(q); stack.append(q)
    return free - seen


def attributes(nodes, grid):
    """Return per-node attribute sets plus chain-valued color sources."""
    H, W = len(grid), len(grid[0])
    sizes = [len(n["pix"]) for n in nodes]
    shapes = [norm_shape(n["pix"]) for n in nodes]
    colors = [n["color"] for n in nodes]
    ccount = Counter(c for c in colors if c is not None)
    scount = Counter(shapes)
    boxes = [bbox(n["pix"]) for n in nodes]
    inner = [interior(n["pix"]) for n in nodes]
    owner = {}
    for i, n in enumerate(nodes):
        for p in n["pix"]:
            owner[p] = i
    out, sources = [], []
    for i, n in enumerate(nodes):
        a = set(); src = {}
        pix = n["pix"]; r0, c0, r1, c1 = boxes[i]; h, w = r1 - r0 + 1, c1 - c0 + 1
        c = colors[i]
        a.add(f"color={c}" if c is not None else "multicolor")
        a.add(f"size={sizes[i]}" if sizes[i] <= 12 else "size>12")
        if sizes[i] == max(sizes): a.add("largest")
        if sizes[i] == min(sizes): a.add("smallest")
        if sizes.count(sizes[i]) == 1: a.add("size_unique")
        if sizes[i] == 1: a.add("single_pixel")
        if h == 1 and w > 1: a.add("hline")
        if w == 1 and h > 1: a.add("vline")
        if len(pix) == h * w and h * w > 1: a.add("filled_rect")
        if h == w: a.add("square_bbox")
        if len(pix) == 2 * (h + w) - 4 and h > 2 and w > 2 and has_hole(pix): a.add("hollow_rect")
        if inner[i]: a.add("has_hole")
        s = shapes[i]
        if s == frozenset((r, w - 1 - cc) for r, cc in s): a.add("sym_lr")
        if s == frozenset((h - 1 - r, cc) for r, cc in s): a.add("sym_ud")
        a.add("shape_unique" if scount[s] == 1 else "shape_shared")
        if c is not None:
            if ccount[c] == 1: a.add("color_unique")
            if ccount[c] == max(ccount.values()): a.add("color_most_common")
            if ccount[c] == min(ccount.values()): a.add("color_least_common")
        if r0 == 0 or c0 == 0 or r1 == H - 1 or c1 == W - 1: a.add("touches_border")
        if r0 == min(b[0] for b in boxes): a.add("topmost")
        if r1 == max(b[2] for b in boxes): a.add("bottommost")
        if c0 == min(b[1] for b in boxes): a.add("leftmost")
        if c1 == max(b[3] for b in boxes): a.add("rightmost")
        # topology: touching / containment (RCC-style) and property chains
        touch = set()
        for (r, cc) in pix:
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)):
                j = owner.get((r + dr, cc + dc))
                if j is not None and j != i:
                    touch.add(j)
        a.add("touches_other" if touch else "isolated")
        tcols = {colors[j] for j in touch if colors[j] is not None}
        for tc in tcols:
            a.add(f"adjColor={tc}")
        if len(tcols) == 1: src["adj"] = next(iter(tcols))
        containers = [j for j in range(len(nodes)) if j != i and pix <= inner[j]]
        if containers:
            a.add("inside_other")
            j = min(containers, key=lambda j: sizes[j])
            if colors[j] is not None:
                a.add(f"contextColor={colors[j]}"); src["ctx"] = colors[j]
        if any(nodes[j]["pix"] <= inner[i] for j in range(len(nodes)) if j != i): a.add("contains_other")
        aligned = {colors[j] for j in range(len(nodes)) if j != i and colors[j] is not None and
                   (not (boxes[j][2] < r0 or boxes[j][0] > r1) or not (boxes[j][3] < c0 or boxes[j][1] > c1))}
        for ac in aligned:
            a.add(f"alignedColor={ac}")
        if len(aligned) == 1: src["aligned"] = next(iter(aligned))
        big = max(range(len(nodes)), key=lambda j: sizes[j])
        small = min(range(len(nodes)), key=lambda j: sizes[j])
        if sizes.count(sizes[big]) == 1 and colors[big] is not None: src["largest"] = colors[big]
        if sizes.count(sizes[small]) == 1 and colors[small] is not None: src["smallest"] = colors[small]
        if ccount:
            least = [k for k, v in ccount.items() if v == min(ccount.values())]
            if len(least) == 1: src["least_common"] = least[0]
        same_shape_other = [j for j in range(len(nodes)) if j != i and shapes[j] == s and colors[j] is not None]
        if same_shape_other and len({colors[j] for j in same_shape_other}) == 1:
            src["same_shape"] = colors[same_shape_other[0]]
        out.append(a); sources.append(src)
    return out, sources


# ---------------------------------------------------------------- labels (training only)
def slide_distance(node, grid, bg, direction, others):
    dr, dc = DIRS[direction]
    H, W = len(grid), len(grid[0]); k = 0
    while True:
        nxt = [(r + dr * (k + 1), c + dc * (k + 1)) for r, c in node["pix"]]
        if any(not (0 <= r < H and 0 <= c < W) for r, c in nxt): return k
        if any(p in others for p in nxt): return k
        k += 1
        if k > 30: return k


def candidate_labels(node, src, grid, out, bg, others):
    H, W = len(out), len(out[0])
    cols = node["colors"]
    now = {p: out[p[0]][p[1]] for p in node["pix"]}
    labs = set()
    if all(now[p] == cols[p] for p in node["pix"]):
        labs.add("keep")
    vals = set(now.values())
    if node["color"] is not None and len(vals) == 1:
        v = next(iter(vals))
        if v != node["color"]:
            if v == bg: labs.add("remove")
            labs.add(f"recolor:const:{v}")
            for name, sc in src.items():
                if sc == v: labs.add(f"recolor:{name}")
    # translations
    p0 = next(iter(node["pix"]))
    for r in range(H):
        for c in range(W):
            if out[r][c] != cols[p0]: continue
            d = (r - p0[0], c - p0[1])
            if d == (0, 0): continue
            if all(0 <= p[0] + d[0] < H and 0 <= p[1] + d[1] < W and
                   out[p[0] + d[0]][p[1] + d[1]] == cols[p] for p in node["pix"]):
                labs.add(f"move:{d[0]},{d[1]}")
                for name, (dr, dc) in DIRS.items():
                    k = slide_distance(node, grid, bg, name, others)
                    if k and (dr * k, dc * k) == d: labs.add(f"slide:{name}")
    return labs


def apply_labels(grid, nodes, srcs, labels, bg, others_list):
    canvas = [row[:] for row in grid]
    H, W = len(grid), len(grid[0])
    paints = []
    for n, s, lab, others in zip(nodes, srcs, labels, others_list):
        if lab in ("keep", "refine"): continue
        if lab == "remove":
            for p in n["pix"]: canvas[p[0]][p[1]] = bg
            continue
        if lab.startswith("recolor:"):
            _, kind, *rest = lab.split(":")
            v = int(rest[0]) if kind == "const" else s.get(kind)
            if v is None: return None
            paints += [(p, v) for p in n["pix"]]
            continue
        if lab.startswith("move:"):
            d = tuple(int(x) for x in lab[5:].split(","))
        elif lab.startswith("slide:"):
            k = slide_distance(n, grid, bg, lab[6:], others)
            d = (DIRS[lab[6:]][0] * k, DIRS[lab[6:]][1] * k)
        else:
            return None
        for p in n["pix"]: canvas[p[0]][p[1]] = bg
        for p in n["pix"]:
            q = (p[0] + d[0], p[1] + d[1])
            if not (0 <= q[0] < H and 0 <= q[1] < W): return None
            paints.append((q, n["colors"][p]))
    for (r, c), v in paints:
        canvas[r][c] = v
    return canvas



import os
VOCAB = set(filter(None, os.environ.get("M1B_VOCAB", "").split(",")))
EFFECTS = [e for e in ("fill_interior", "ray", "connect", "border", "bbox_fill", "outline8") if e in VOCAB]


def effect_cells(name, node, grid, bg, owner_color):
    H, W = len(grid), len(grid[0]); pix = node["pix"]
    if name == "fill_interior":
        return [{q for q in interior(pix) if 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg}]
    if name == "bbox_fill":
        r0, c0, r1, c1 = bbox(pix)
        return [{(r, c) for r in range(r0, r1 + 1) for c in range(c0, c1 + 1) if grid[r][c] == bg}]
    if name == "border":
        r0, c0, r1, c1 = bbox(pix)
        return [{(r, c) for r in range(r0 - 1, r1 + 2) for c in range(c0 - 1, c1 + 2)
                 if 0 <= r < H and 0 <= c < W and not (r0 <= r <= r1 and c0 <= c <= c1) and grid[r][c] == bg}]
    if name == "outline8":
        hole = interior(pix)
        return [{(r + dr, c + dc) for r, c in pix for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                 if 0 <= r + dr < H and 0 <= c + dc < W and (r + dr, c + dc) not in pix and (r + dr, c + dc) not in hole
                 and grid[r + dr][c + dc] == bg}]
    if name == "ray":
        outs = []
        for dr, dc in list(DIRS.values()) + [(1, 1), (1, -1), (-1, 1), (-1, -1)]:
            cells = set()
            for r, c in pix:
                q = (r + dr, c + dc)
                if q in pix: continue
                while 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg:
                    cells.add(q); q = (q[0] + dr, q[1] + dc)
            outs.append(cells)
        return outs
    if name == "connect":
        cells = set()
        for r, c in pix:
            for dr, dc in DIRS.values():
                q = (r + dr, c + dc); walk = []
                if q in pix: continue
                while 0 <= q[0] < H and 0 <= q[1] < W and grid[q[0]][q[1]] == bg:
                    walk.append(q); q = (q[0] + dr, q[1] + dc)
                if walk and 0 <= q[0] < H and 0 <= q[1] < W and q not in pix and owner_color.get(q) == node["color"]:
                    cells.update(walk)
        return [cells]
    return []


def effect_options(node, src, grid, bg, owner_color, out=None, families=None):
    """All (effect, index, colorspec) options; with out given, keep those consistent with out."""
    opts = []
    for name in (families or EFFECTS):
        for idx, cells in enumerate(effect_cells(name, node, grid, bg, owner_color)):
            if not cells: continue
            if out is None:
                opts.append((f"{name}#{idx}", cells)); continue
            vals = {out[r][c] for r, c in cells}
            if len(vals) != 1: continue
            v = next(iter(vals))
            specs = [f"const:{v}"] + (["self"] if v == node["color"] else []) + [k for k, sv in src.items() if sv == v]
            opts.append((f"{name}#{idx}", cells, specs, v))
    return opts


def effect_labels(node, src, grid, out, bg, owner_color, changed, families=None):
    labs = set()
    for name, cells, specs, v in effect_options(node, src, grid, bg, owner_color, out, families):
        if cells & changed:
            labs |= {f"{name}@{s}" for s in specs}
    return labs or {"none"}


def apply_effects(canvas, grid, nodes, srcs, elabels, bg, owner_color):
    for n, s, lab in zip(nodes, srcs, elabels):
        if lab == "none": continue
        name, spec = lab.split("@")
        ename, idx = name.split("#")
        cells_list = effect_cells(ename, n, grid, bg, owner_color)
        idx = int(idx)
        if idx >= len(cells_list): return None
        if spec.startswith("const:"): v = int(spec[6:])
        elif spec == "self": v = n["color"]
        else: v = s.get(spec)
        if v is None: return None
        for r, c in cells_list[idx]: canvas[r][c] = v
    return canvas


# ---------------------------------------------------------------- DSL primitive: pixel refinement (cycle 3)
def px_attrs(node, grid, bg, nattrs):
    """Pixel-level concepts inside an object: position within the object's frame, boundary/interior,
    parity (checkerboard), neighbourhood colour, row/column alignment with outside colours."""
    H, W = len(grid), len(grid[0]); pix = node["pix"]
    r0, c0, r1, c1 = bbox(pix); h, w = r1 - r0 + 1, c1 - c0 + 1
    rowcols = {}; colcols = {}
    for r in range(H):
        for c in range(W):
            if (r, c) not in pix and grid[r][c] != bg:
                rowcols.setdefault(r, set()).add(grid[r][c]); colcols.setdefault(c, set()).add(grid[r][c])
    out = {}
    for (r, c) in pix:
        a = {"obj:" + x for x in nattrs}
        a.add(f"px:color={node['colors'][(r, c)]}")
        nb4 = [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]
        a.add("px:boundary" if any(q not in pix for q in nb4) else "px:interior")
        if (r in (r0, r1)) and (c in (c0, c1)): a.add("px:corner")
        if r == r0: a.add("px:top_edge")
        if r == r1: a.add("px:bottom_edge")
        if c == c0: a.add("px:left_edge")
        if c == c1: a.add("px:right_edge")
        a.add(f"px:row_parity={(r - r0) % 2}"); a.add(f"px:col_parity={(c - c0) % 2}")
        a.add(f"px:checker={(r - r0 + c - c0) % 2}")
        if h % 2 and r - r0 == h // 2: a.add("px:mid_row")
        if w % 2 and c - c0 == w // 2: a.add("px:mid_col")
        if r - r0 < 3: a.add(f"px:row_from_top={r - r0}")
        if r1 - r < 3: a.add(f"px:row_from_bottom={r1 - r}")
        if c - c0 < 3: a.add(f"px:col_from_left={c - c0}")
        if c1 - c < 3: a.add(f"px:col_from_right={c1 - c}")
        for q in nb4:
            if 0 <= q[0] < H and 0 <= q[1] < W and q not in pix: a.add(f"px:adj={grid[q[0]][q[1]]}")
        for v in rowcols.get(r, ()): a.add(f"px:row_has={v}")
        for v in colcols.get(c, ()): a.add(f"px:col_has={v}")
        out[(r, c)] = a
    return out


def px_labels(node, out_grid):
    res = {}
    for (r, c) in node["pix"]:
        v = out_grid[r][c]
        res[(r, c)] = {"keep"} if v == node["colors"][(r, c)] else {f"pc:{v}"}
    return res


def apply_pixel_rules(canvas, grid, nodes, at, labels, rp, bg):
    for n, a, lab in zip(nodes, at, labels):
        if lab != "refine": continue
        pa = px_attrs(n, grid, bg, a)
        pl = predict_labels_default(rp, [pa[p] for p in sorted(pa)], "keep")
        for p, l in zip(sorted(pa), pl):
            if l.startswith("pc:"): canvas[p[0]][p[1]] = int(l[3:])
    return canvas

# ---------------------------------------------------------------- lattice / RDR decision list

def clarified(cases):
    """FCA attribute clarification: drop attributes with identical extents or full extent; cap for pair search."""
    n = len(cases); full = (1 << n) - 1
    ext = {}
    for i, (at, _) in enumerate(cases):
        for a in at: ext[a] = ext.get(a, 0) | (1 << i)
    seen, attrs = set(), []
    for a in sorted(ext):
        e = ext[a]
        if e == full or e in seen: continue
        seen.add(e); attrs.append(a)
    pair_attrs = attrs if len(attrs) <= 150 else sorted(attrs, key=lambda a: -bin(ext[a]).count("1"))[:150]
    return attrs, ext, pair_attrs

def learn(cases, specific=False, mode="greedy"):
    if mode == "inertia":
        return learn_inertia(cases, specific)
    return learn_greedy(cases, specific)


def learn_inertia(cases, specific=False, default="keep"):
    """RDR with an inertia root: default conclusion = no change; exceptions are lattice concepts
    whose extent contains no case that must stay unchanged-incompatible with the exception label."""
    n = len(cases)
    if not any(default in ls for _, ls in cases): return None
    attrs, ext, pair_attrs = clarified(cases)
    labels = sorted({l for _, ls in cases for l in ls} - {default})
    valid = {l: sum(1 << i for i, (_, ls) in enumerate(cases) if l in ls) for l in labels}
    need = sum(1 << i for i, (_, ls) in enumerate(cases) if default not in ls)
    gens = [(a,) for a in sorted(ext)] + list(combinations(pair_attrs, 2))
    uncovered = (1 << n) - 1; rules = []
    while uncovered & need:
        if len(rules) >= MAX_RULES - 1: return None
        best = None
        for g in gens:
            e = uncovered
            for a in g: e &= ext[a]
            if not (e & need): continue
            full = (1 << n) - 1
            for a in g: full &= ext[a]
            gen = bin(full).count("1")
            for l in labels:
                if e & ~valid[l]: continue
                key = (bin(e & need).count("1"), (len(g) if specific else -len(g)), gen, tuple(g), l)
                if best is None or key > best[0]: best = (key, g, l, e)
        if best is None: return None
        _, g, l, e = best
        rules.append((g, l)); uncovered &= ~e
    return rules + [((), default)]


def learn_greedy(cases, specific=False):
    """cases: list of (attrset, validlabelset). Greedy decision list of lattice concepts."""
    n = len(cases)
    attrs, ext, pair_attrs = clarified(cases)
    labels = sorted({l for _, ls in cases for l in ls})
    valid = {l: sum(1 << i for i, (_, ls) in enumerate(cases) if l in ls) for l in labels}
    uncovered = (1 << n) - 1
    rules = []
    gens = [()] + [(a,) for a in sorted(ext)] + list(combinations(pair_attrs, 2))
    while uncovered:
        if len(rules) >= MAX_RULES: return None
        best = None
        for g in gens:
            e = uncovered
            for a in g: e &= ext[a]
            if not e: continue
            cov = bin(e).count("1")
            full = (1 << n) - 1
            for a in g: full &= ext[a]
            gen = bin(full).count("1")  # maximal concept subsumption: prefer the more general concept
            for l in labels:
                if e & ~valid[l]: continue
                key = (cov, (len(g) if specific else -len(g)), gen, l != "keep", tuple(g), l)
                if best is None or key > best[0]:
                    best = (key, g, l, e)
        if best is None: return None
        _, g, l, e = best
        rules.append((g, l)); uncovered &= ~e
    return rules


def predict_labels(rules, attrsets):
    labs = []
    for at in attrsets:
        for g, l in rules:
            if all(a in at for a in g):
                labs.append(l); break
        else:
            labs.append("keep")
    return labs


# ---------------------------------------------------------------- task driver

def n_holes(pix):
    cells = interior(pix); comps = 0; seen = set()
    for q in cells:
        if q in seen: continue
        comps += 1; stack = [q]; seen.add(q)
        while stack:
            r, c = stack.pop()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                x = (r + dr, c + dc)
                if x in cells and x not in seen:
                    seen.add(x); stack.append(x)
    return comps

def extra_attrs(nodes, grid):
    H, W = len(grid), len(grid[0])
    sizes = [len(n["pix"]) for n in nodes]; colors = [n["color"] for n in nodes]
    shapes = [norm_shape(n["pix"]) for n in nodes]; boxes = [bbox(n["pix"]) for n in nodes]
    inner = [interior(n["pix"]) for n in nodes]
    ranks = sorted(set(sizes), reverse=True)
    owner = {p: i for i, n in enumerate(nodes) for p in n["pix"]}
    res = []
    for i, n in enumerate(nodes):
        a = set(); r0, c0, r1, c1 = boxes[i]
        a.add(f"D1:size_rank={ranks.index(sizes[i])}" if ranks.index(sizes[i]) < 4 else "D1:size_rank>3")
        a.add(f"D1:size_rank_asc={sorted(set(sizes)).index(sizes[i])}")
        same = [j for j in range(len(nodes)) if colors[j] == colors[i]]
        a.add(f"D2:n_same_color={min(len(same),5)}")
        for j in range(len(nodes)):
            if j != i and shapes[j] == shapes[i] and colors[j] is not None: a.add(f"D3:shape_as_color={colors[j]}")
        touch = {owner[(r+dr, c+dc)] for r, c in n["pix"] for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)) if owner.get((r+dr, c+dc), i) != i}
        a.add(f"D4:n_touch={min(len(touch),4)}")
        a.add("D5:top_half" if (r0+r1)/2 < H/2 else "D5:bottom_half"); a.add("D5:left_half" if (c0+c1)/2 < W/2 else "D5:right_half")
        a.add(f"D8:height={r1-r0+1}"); a.add(f"D8:width={c1-c0+1}")
        for j in range(len(nodes)):
            if j != i and nodes[j]["pix"] <= inner[i] and colors[j] is not None: a.add(f"D7:contains_color={colors[j]}")
            if j != i and colors[j] is not None and not (boxes[j][2] < r0 or boxes[j][0] > r1): a.add(f"D9:row_with_color={colors[j]}")
            if j != i and colors[j] is not None and not (boxes[j][3] < c0 or boxes[j][1] > c1): a.add(f"D9:col_with_color={colors[j]}")
        if sizes[i] == max(sizes[j] for j in same): a.add("D10:largest_of_its_color")
        if sizes[i] == min(sizes[j] for j in same): a.add("D10:smallest_of_its_color")
        a.add(f"D11:n_cells_of_color_in_bbox={sum(1 for r in range(r0,r1+1) for c in range(c0,c1+1) if grid[r][c]==colors[i])==sizes[i]}")
        a.add(f"D12:n_holes={min(n_holes(n['pix']), 5)}")
        big = max(range(len(nodes)), key=lambda j: (sizes[j], -j))
        if big != i:
            br0, bc0, br1, bc1 = boxes[big]
            cy, cx = (r0 + r1) / 2, (c0 + c1) / 2; by, bx = (br0 + br1) / 2, (bc0 + bc1) / 2
            a.add("D13:above_largest" if cy < by else ("D13:below_largest" if cy > by else "D13:level_with_largest"))
            a.add("D13:left_of_largest" if cx < bx else ("D13:right_of_largest" if cx > bx else "D13:centered_on_largest"))
            if not (r1 < br0 or r0 > br1): a.add("D13:row_overlaps_largest")
            if not (c1 < bc0 or c0 > bc1): a.add("D13:col_overlaps_largest")
        dens = sizes[i] / ((r1 - r0 + 1) * (c1 - c0 + 1))
        a.add("D14:dense_full" if dens == 1 else ("D14:dense_high" if dens >= 0.5 else "D14:dense_low"))
        res.append(a)
    return res



# ---------------------------------------------------------------- Codex selectors as concepts (cycle 5)
_CODEX = None
_CODEX_CACHE = {}
ROLE_KEYS = ("role", "marker", "source", "separator", "foreground", "base", "color", "background")


def _codex():
    global _CODEX
    if _CODEX is None:
        import importlib, json as _j
        root = os.environ.get("CODEX_ROOT", "/mnt/user-data/uploads/arc_extended_arga")
        if root not in sys.path: sys.path.insert(0, root)
        cr = importlib.import_module("wake.composition_runtime")
        spec = _j.load(open(os.environ.get("CODEX_SELECTORS", "/home/claude/work/au/seltime.json")))
        _CODEX = [(name, getattr(cr, name), v["spec"]) for name, v in sorted(spec.items())]
    return _CODEX


def _cells(v, H, W):
    if isinstance(v, (list, tuple, set)) and v:
        items = list(v)[:2000]
        if all(isinstance(x, (list, tuple)) and len(x) == 2 and all(isinstance(y, int) for y in x) for x in items):
            cs = {(x[0], x[1]) for x in items}
            if all(0 <= r < H and 0 <= c < W for r, c in cs): return cs
    return None


def codex_features(grid):
    """Run each Codex selector on the grid; return (role attributes by colour, cell-set attributes, chain sources)."""
    key = tuple(map(tuple, grid))
    if key in _CODEX_CACHE: return _CODEX_CACHE[key]
    H, W = len(grid), len(grid[0])
    roles, cellsets, sources = {}, {}, {}
    for name, f, spec in _codex():
        try:
            out = f(grid, spec)
        except Exception:
            continue
        if not isinstance(out, dict): continue
        short = name[len("_select_"):] if name.startswith("_select_") else name
        for k, v in out.items():
            tag = f"cx:{short}.{k}"
            if isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 and any(r in k for r in ROLE_KEYS):
                roles.setdefault(v, set()).add(tag); sources[tag] = v
            else:
                cs = _cells(v, H, W)
                if cs: cellsets[tag] = cs
    _CODEX_CACHE[key] = (roles, cellsets, sources)
    if len(_CODEX_CACHE) > 5000: _CODEX_CACHE.clear()
    return roles, cellsets, sources



# ---------------------------------------------------------------- S0 atoms (Codex detectors) as a fallback stratum (cycle 6)
_ATOMS = None
_ATOM_CACHE = {}
ATOM_ACTIVE = False


def _atoms():
    global _ATOMS
    if _ATOMS is None:
        import importlib, json as _j
        root = os.environ.get("CODEX_ROOT", "/home/claude/work/codex")
        if root not in sys.path: sys.path.insert(0, root)
        reps = _j.load(open(os.environ.get("S0_REPS", "/home/claude/work/s0/s0_reps.json")))
        if "S_atoms_used" in VOCAB or "S_atoms_primary" in VOCAB:
            used = set(_j.load(open(os.environ.get("S0_REPS_USED", "/home/claude/work/s0/s0_reps_used.json"))))
            reps = [r for r in reps if r in used]
        fns = []
        for name in sorted(reps):
            mod, fn = name.rsplit(".", 1)
            try:
                fns.append((name, getattr(importlib.import_module("detectors." + mod), fn)))
            except Exception:
                pass
        _ATOMS = fns
    return _ATOMS


def _collect_cells(ev, H, W, out):
    if isinstance(ev, dict):
        if isinstance(ev.get("row"), int) and isinstance(ev.get("col"), int):
            if 0 <= ev["row"] < H and 0 <= ev["col"] < W: out.add((ev["row"], ev["col"]))
        for v in ev.values(): _collect_cells(v, H, W, out)
    elif isinstance(ev, (list, tuple, set, frozenset)):
        if len(ev) == 2 and all(isinstance(x, int) and not isinstance(x, bool) for x in ev):
            r, c = ev
            if 0 <= r < H and 0 <= c < W: out.add((r, c))
            return
        for v in ev: _collect_cells(v, H, W, out)


def atom_cells(grid):
    key = tuple(map(tuple, grid))
    if key in _ATOM_CACHE: return _ATOM_CACHE[key]
    H, W = len(grid), len(grid[0]); res = {}
    for name, fn in _atoms():
        try:
            ev = fn([row[:] for row in grid])
        except Exception:
            continue
        if not ev: continue
        cs = set(); _collect_cells(ev, H, W, cs)
        if cs: res["at:" + name.split(".", 1)[1][:-len("_evidence")] if name.endswith("_evidence") else "at:" + name] = cs
    _ATOM_CACHE[key] = res
    if len(_ATOM_CACHE) > 4000: _ATOM_CACHE.clear()
    return res


def prepare(grid, abstraction):
    nodes = segment(grid, abstraction)
    bg = background(grid)
    at, src = attributes(nodes, grid)
    if "S_holes" in VOCAB or "S_size" in VOCAB:
        holes = [n_holes(n["pix"]) for n in nodes]
        Hg, Wg = len(grid), len(grid[0])
        spans = [bbox(n["pix"])[0] == 0 or bbox(n["pix"])[1] == 0 or bbox(n["pix"])[2] == Hg - 1 or bbox(n["pix"])[3] == Wg - 1
                 for n in nodes]  # frames/separators anchored to the border are not reference tokens
        for i, s in enumerate(src):
            if nodes[i]["color"] is None: continue
            for key, vals in (("same_holes", holes), ("same_size", [len(n["pix"]) for n in nodes])):
                if ("S_holes" if key == "same_holes" else "S_size") not in VOCAB: continue
                cand = [j for j in range(len(nodes)) if j != i and vals[j] == vals[i] and not spans[j]
                        and nodes[j]["color"] is not None and nodes[j]["color"] != nodes[i]["color"]]
                if not cand: continue
                size_i = len(nodes[i]["pix"])
                gap = {j: abs(len(nodes[j]["pix"]) - size_i) for j in cand}
                best = [j for j in cand if gap[j] == min(gap.values())]
                cs = {nodes[j]["color"] for j in best}
                if len(cs) == 1:
                    s[key] = next(iter(cs)); at[i].add(f"exists_{key}")  # OWL existential restriction
    if "S_codex" in VOCAB:
        roles, cellsets, csrc = codex_features(grid)
        for n, a, s in zip(nodes, at, src):
            if n["color"] is not None:
                a |= roles.get(n["color"], set())
            for tag, cs in cellsets.items():
                if n["pix"] & cs: a.add(tag)
            for tag, v in csrc.items():
                s[tag] = v
    if ATOM_ACTIVE or "S_atoms_primary" in VOCAB:
        for tag, cs in atom_cells(grid).items():
            for n, a in zip(nodes, at):
                if n["pix"] & cs: a.add(tag)
    dv = {v for v in VOCAB if v.startswith("D")}
    if dv:
        for a, e in zip(at, extra_attrs(nodes, grid)):
            a |= {x for x in e if x.split(":")[0] in dv}
    allpix = [set().union(*[m["pix"] for j, m in enumerate(nodes) if j != i]) if len(nodes) > 1 else set()
              for i in range(len(nodes))]
    owner_color = {p: n["colors"][p] for n in nodes for p in n["pix"]}
    return nodes, bg, at, src, allpix, owner_color


def predict_grid(grid, rules, abstraction):
    rb, re_ = rules[:2]
    rp = rules[2] if len(rules) > 2 else None
    nodes, bg, at, src, others, oc = prepare(grid, abstraction)
    blabels = predict_labels(rb, at)
    canvas = apply_labels(grid, nodes, src, blabels, bg, others)
    if canvas is None: return None
    if rp:
        canvas = apply_pixel_rules(canvas, grid, nodes, at, blabels, rp, bg)
    if re_:
        groups, cur = [], None
        for item in re_:
            if isinstance(item, tuple) and len(item) == 1 and str(item[0]).startswith("#"):
                cur = []; groups.append(cur)
            else:
                cur.append(item)
        for fr in groups:
            el = predict_labels_default(fr, at, "none")
            canvas = apply_effects(canvas, grid, nodes, src, el, bg, oc)
            if canvas is None: return None
    return canvas


def predict_labels_default(rules, attrsets, default):
    labs = []
    for at in attrsets:
        for g, l in rules:
            if all(a in at for a in g):
                labs.append(l); break
        else:
            labs.append(default)
    return labs


def fit(pairs, abstraction, specific=False, mode="greedy"):
    cases, preps = [], []
    for p in pairs:
        gi, go = p["input"], p["output"]
        if (len(gi), len(gi[0])) != (len(go), len(go[0])): return None, "extent_change"
        pr = prepare(gi, abstraction)
        nodes, bg, at, src, others, oc = pr
        labs = [candidate_labels(n, s, gi, go, bg, o) for n, s, o in zip(nodes, src, others)]
        if PIXEL_ACTIVE:
            labs = [l if l else {"refine"} for l in labs]
        if EFFECTS:
            labs = [l if l else {"keep"} for l in labs]  # own pixels may be repainted by an effect
        if any(not l for l in labs): return None, "unlabelled_node"
        cases += list(zip(at, labs)); preps.append((gi, go, pr))
    rb = learn(cases, specific, mode)
    if rb is None: return None, "not_occupied"
    rp = None
    if PIXEL_ACTIVE and any(l == "refine" for _, l in rb):
        pcases = []
        for gi, go, (nodes, bg, at, src, others, oc) in preps:
            for n, a, lab in zip(nodes, at, predict_labels(rb, at)):
                if lab != "refine": continue
                pa, pl = px_attrs(n, gi, bg, a), px_labels(n, go)
                pcases += [(pa[p], pl[p]) for p in sorted(pa)]
        if pcases:
            rp = learn(pcases, specific, mode)
            if rp is None: return None, "not_occupied"
    re_ = None
    if EFFECTS:
        changes, residual_any = [], False
        for gi, go, (nodes, bg, at, src, others, oc) in preps:
            canvas = predict_grid(gi, (rb, None, rp), abstraction)
            if canvas is None: return None, "not_representable"
            changed = {(r, c) for r in range(len(gi)) for c in range(len(gi[0])) if canvas[r][c] != go[r][c]}
            residual_any |= bool(changed); changes.append(changed)
        if residual_any:
            re_ = []
            remaining = list(EFFECTS)
            while remaining and any(changes):
                best = None
                for fam in remaining:
                    ecases = []
                    for (gi, go, (nodes, bg, at, src, others, oc)), changed in zip(preps, changes):
                        ecases += list(zip(at, [effect_labels(n, s, gi, go, bg, oc, changed, [fam]) for n, s in zip(nodes, src)]))
                    if all(ls == {"none"} for _, ls in ecases): continue
                    fr = learn(ecases, specific, mode)
                    if fr is None or all(l == "none" for _, l in fr): continue
                    # MDL: residual cells this family explains, per rule
                    newchanges, explained = [], 0
                    for (gi, go, (nodes, bg, at, src, others, oc)), changed in zip(preps, changes):
                        canvas = [row[:] for row in gi]
                        for r, c in changed: canvas[r][c] = None
                        el = predict_labels_default(fr, at, "none")
                        canvas = apply_effects(canvas, gi, nodes, src, el, bg, oc)
                        if canvas is None: explained = -1; break
                        still = {(r, c) for r, c in changed if canvas[r][c] != go[r][c]}
                        explained += len(changed) - len(still); newchanges.append(still)
                    if explained <= 0: continue
                    key = (explained / len(fr), explained, fam)
                    if best is None or key > best[0]: best = (key, fam, fr, newchanges)
                if best is None: break
                _, fam, fr, changes = best
                remaining.remove(fam)
                re_ += [("#" + fam,)] + fr
            re_ = re_ or None
    rules = (rb, re_, rp)
    for gi, go, _ in preps:
        if predict_grid(gi, rules, abstraction) != go:
            return None, "not_representable"
    return rules, "occupied"


def run_rules(grid, rules, abstraction):
    return predict_grid(grid, rules, abstraction)


def nrules(rules):
    rb, re_ = rules[:2]
    rp = rules[2] if len(rules) > 2 else None
    return len(rb) + (sum(1 for x in re_ if not (len(x) == 1)) if re_ else 0) + (len(rp) if rp else 0)



# ---------------------------------------------------------------- DSL primitive: frame (cycle 3)
def frame_candidates(task, limit=3):
    """Select, by a lattice concept, the unique object whose bounding box becomes the output frame."""
    outs = []
    for ab in ("nbccg", "mcccg", "ccgbr"):
        try:
            per_pair = []
            for p in task["train"]:
                nodes, bg, at, src, others, oc = prepare(p["input"], ab)
                h, w = len(p["output"]), len(p["output"][0])
                dims = [(bbox(n["pix"])[2] - bbox(n["pix"])[0] + 1, bbox(n["pix"])[3] - bbox(n["pix"])[1] + 1) for n in nodes]
                per_pair.append((nodes, at, [d == (h, w) for d in dims]))
            tests = [prepare(t["input"], ab) for t in task["test"]]
        except Timeout:
            raise
        except Exception:
            continue
        if not all(any(c) for _, _, c in per_pair): continue
        attrs = sorted({x for _, at, _ in per_pair for a in at for x in a})
        gens = [(a,) for a in attrs] + list(combinations(attrs, 2))
        for g in gens:
            ok = True
            for nodes, at, cand in per_pair:
                hits = [i for i, a in enumerate(at) if all(x in a for x in g)]
                if len(hits) != 1 or not cand[hits[0]]: ok = False; break
            if not ok: continue
            thits = [[i for i, a in enumerate(tp[2]) if all(x in a for x in g)] for tp in tests]
            if any(len(h) != 1 for h in thits): continue
            sel = [nodes[[i for i, a in enumerate(at) if all(x in a for x in g)][0]] for nodes, at, _ in per_pair]
            tsel = [tp[0][h[0]] for tp, h in zip(tests, thits)]
            outs.append((ab, g, sel, tsel))
            if len(outs) >= limit: return outs
    return outs


def crop(grid, node):
    r0, c0, r1, c1 = bbox(node["pix"])
    return [row[c0:c1 + 1] for row in grid[r0:r1 + 1]]


def solve_frame(task):
    global FRAME_ACTIVE
    found = []
    FRAME_ACTIVE = True
    try:
        for ab, g, sel, tsel in frame_candidates(task):
            inner = {"train": [{"input": crop(p["input"], n), "output": p["output"]} for p, n in zip(task["train"], sel)],
                     "test": [{"input": crop(t["input"], n)} for t, n in zip(task["test"], tsel)]}
            att, _, fnd = solve(inner)
            for f in att:
                found.append(dict(f, frame=(ab, g), rules=f["rules"]))
            if found: break
    finally:
        FRAME_ACTIVE = False
    return found

FRAME_ACTIVE = False

PIXEL_ACTIVE = False


def solve(task, abstractions=ABSTRACTIONS):
    """Simpler vocabulary first (MDL): pixel refinement engages only if no object-level program exists."""
    global PIXEL_ACTIVE
    PIXEL_ACTIVE = False
    res = solve_once(task, abstractions)
    if not res[2] and "P_pixel" in VOCAB:
        PIXEL_ACTIVE = True
        try:
            res = solve_once(task, abstractions)
        finally:
            PIXEL_ACTIVE = False
    if not res[2] and ("S_atoms" in VOCAB or "S_atoms_used" in VOCAB):
        global ATOM_ACTIVE
        ATOM_ACTIVE = True
        try:
            res = solve_once(task, abstractions)
            if not res[2] and "P_pixel" in VOCAB:
                PIXEL_ACTIVE = True
                try: res = solve_once(task, abstractions)
                finally: PIXEL_ACTIVE = False
        finally:
            ATOM_ACTIVE = False
    if not res[2] and "P_frame" in VOCAB and not FRAME_ACTIVE and any(
            (len(p["input"]), len(p["input"][0])) != (len(p["output"]), len(p["output"][0])) for p in task["train"]):
        fr = solve_frame(task)
        if fr:
            res = (fr[:2], dict(res[1], frame="occupied"), fr)
    if res[2] and len(res[0]) < 2 and "G_dsl" in VOCAB and not FRAME_ACTIVE:
        # attempt 2 (cycle 10): if the lattice has a single distinct prediction, the cheapest verified
        # whole-grid program with different predictions fills the unused second attempt
        import gdsl
        for p in gdsl.search(task):
            if p["preds"] != res[0][0]["preds"]:
                g2 = {"abstraction": "gdsl", "specific": False, "mode": "gdsl", "rules": ([], []), "program": p["program"], "preds": p["preds"]}
                res = (res[0] + [g2], res[1], res[2])
                break
    if not res[2] and "G_dsl" in VOCAB and not FRAME_ACTIVE:
        # whole-grid stratum (cycle 8): verified programs over grid primitives, engaged only when
        # the object lattice, pixel refinement, atoms and frame strata found no program
        import gdsl
        progs = gdsl.search(task)
        if progs:
            found = [{"abstraction": "gdsl", "specific": False, "mode": "gdsl", "rules": ([], []),
                      "program": p["program"], "preds": p["preds"]} for p in progs]
            res = (found[:2], dict(res[1], gdsl="occupied"), found)
    return res


def solve_once(task, abstractions=ABSTRACTIONS):
    found, statuses = [], {}
    for ab in abstractions:
        variants = ([(False, "inertia")] if "L_inertia" in VOCAB else []) + [(False, "greedy"), (True, "greedy")]
        for specific, mode in variants:
            try:
                rules, st = fit(task["train"], ab, specific, mode)
            except Timeout:
                raise
            except Exception as e:
                rules, st = None, f"error:{type(e).__name__}"
            statuses.setdefault(ab, st)
            if rules is None: continue
            try:
                preds = [run_rules(t["input"], rules, ab) for t in task["test"]]
            except Timeout:
                raise
            except Exception:
                continue
            if any(p is None for p in preds): continue
            found.append({"abstraction": ab, "specific": specific, "mode": mode, "rules": rules, "preds": preds})
    distinct = {json.dumps(f["preds"]) for f in found}
    if len(distinct) > 2:  # regeneration stability: concepts must survive leave-one-out
        for f in found:
            f["stable"] = bool(loo(task, [f]))
    attempts, seen = [], set()
    for f in sorted(found, key=lambda f: (not f.get("stable", True), nrules(f["rules"]))):
        key = json.dumps(f["preds"])
        if key not in seen:
            seen.add(key); attempts.append(f)
    return attempts[:2], statuses, found


def loo(task, found):
    if not found or len(task["train"]) < 2: return None
    f = found[0]
    for i in range(len(task["train"])):
        rest = task["train"][:i] + task["train"][i + 1:]
        try:
            rules, _ = fit(rest, f["abstraction"], f["specific"], f.get("mode", "greedy"))
            if rules is None or run_rules(task["train"][i]["input"], rules, f["abstraction"]) != task["train"][i]["output"]:
                return False
        except Timeout:
            raise
        except Exception:
            return False
    return True


def main():
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--challenges"); ap.add_argument("--solutions"); ap.add_argument("--keys", default="")
    ap.add_argument("--out"); ap.add_argument("--seconds", type=int, default=60)
    args = ap.parse_args()
    ch = json.load(open(args.challenges)); so = json.load(open(args.solutions))
    keys = sorted(ch) if not args.keys else [k.strip() for k in open(args.keys) if k.strip()]
    done = set()
    try:
        done = {json.loads(l)["task"] for l in open(args.out)}
    except FileNotFoundError:
        pass
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(Timeout()))
    for k in keys:
        if k in done: continue
        row = {"task": k}; t0 = time.time()
        try:
            signal.alarm(args.seconds)
            attempts, statuses, found = solve(ch[k])
            row["statuses"] = statuses
            row["occupied"] = bool(found)
            row["n_programs"] = len(found)
            sol = so[k]
            row["test_exact"] = bool(attempts) and all(
                any(a["preds"][i] == sol[i] for a in attempts) for i in range(len(sol)))
            row["loo"] = loo(ch[k], found)
            if attempts:
                rb, re_ = attempts[0]["rules"][:2]
                rp = attempts[0]["rules"][2] if len(attempts[0]["rules"]) > 2 else None
                row["rules"] = [[list(g), l] for g, l in rb] + [[list(x[0]), "effect:" + x[1]] if len(x) == 2 else ["family", x[0]] for x in (re_ or [])] + [[list(g), "pixel:" + l] for g, l in (rp or [])]
                row["n_rules"] = nrules(attempts[0]["rules"])
                row["abstraction"] = attempts[0]["abstraction"]
                if "program" in attempts[0]: row["program"] = attempts[0]["program"]
        except Timeout:
            row["status"] = "timeout"
        finally:
            signal.alarm(0)
        row["seconds"] = round(time.time() - t0, 2)
        with open(args.out, "a") as f:
            f.write(json.dumps(row) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/probe/codex_ops.py
"""Bind Codex's grid-level DSL (extended_transformations/*_grid_based) into the verified G-DSL search.

Parameter domains are read from each function's own source (string enums) and from the task's
training pairs (colours present, size ratios). Nothing task-specific: every value is either an
enum the function declares, a colour seen in the task's training grids, or a small integer.
"""
from __future__ import annotations
import glob, importlib, inspect, os, re, sys

_OPS = None
SKIP = {"fill_grid_based"}          # needs an object argument, not a grid-level op
ENUM_HINT = ("type", "axis", "corner", "mirror_grid", "combine_pattern", "concat_axis", "fill_direction",
             "shifting_direction", "direction", "connect_mode", "component_mode", "selection", "output_mode",
             "mirror_axis", "center_mode", "duplicate_mode", "split_axis", "fold", "overlay")
INTS = {"factor": (2, 3, 4), "grid_size": (2, 3, 4, 5), "output_width": (1, 2, 3), "degrees": (90, 180, 270),
        "duplicate": (2, 3), "duplicate_arbitrary": (2, 3), "connectivity": (4, 8)}


def _root():
    for r in (os.environ.get("CODEX_DSL_ROOT", ""), "/home/claude/work/cand2", "/kaggle/working/cand"):
        if r and os.path.isdir(os.path.join(r, "extended_transformations")): return r
    return None


def load():
    global _OPS
    if _OPS is not None: return _OPS
    _OPS = []
    root = _root()
    if not root: return _OPS
    for p in (root, os.path.join(root, "extended_transformations")):
        if p not in sys.path: sys.path.append(p)
    for f in sorted(glob.glob(os.path.join(root, "extended_transformations", "*_grid.py"))):
        name = os.path.basename(f)[:-3]
        try:
            mod = importlib.import_module("extended_transformations." + name)
            src = inspect.getsource(mod)
        except Exception:
            continue
        for fn in sorted(n for n in dir(mod) if n.endswith("_grid_based")):
            if fn in SKIP: continue
            f_ = getattr(mod, fn)
            try: sig = inspect.signature(f_)
            except Exception: continue
            params = []
            for pn, p in list(sig.parameters.items())[1:]:
                d = p.default if p.default is not inspect.Parameter.empty else None
                if pn == "classifier_params": continue
                kind, vals = "fixed", [d]
                if any(pn.endswith(h) or pn == h for h in ENUM_HINT) and (isinstance(d, str) or d is None):
                    vs = set(re.findall(rf'\b{pn}\s*==\s*["\']([^"\']+)["\']', src))
                    for grp in re.findall(rf'\b{pn}\s+in\s*[\(\[\{{]([^\)\]\}}]+)[\)\]\}}]', src):
                        vs |= set(re.findall(r'["\']([^"\']+)["\']', grp))
                    if vs: kind, vals = "enum", ([d] if d is not None else []) + sorted(vs - {d})
                elif "color" in pn:
                    kind = "colour"
                elif isinstance(d, bool):
                    kind, vals = "bool", [d, not d]
                elif pn in INTS:
                    kind, vals = "int", [d] + [v for v in INTS[pn] if v != d]
                params.append((pn, kind, vals, d))
            _OPS.append((fn, f_, params))
    return _OPS


_WL = None
def whitelist():
    global _WL
    if _WL is None:
        import json
        p = os.path.join(os.path.dirname(os.path.abspath(__file__)), "codex_types_general.json")
        _WL = {tuple(x) for x in json.load(open(p))["types"]} if os.path.exists(p) and os.environ.get("CODEX_ALL_TYPES") != "1" else None
    return _WL

def programs(train, per_op=70):
    """Yield (name, cost, fn) for Codex ops: enum type values x (default colours, then one colour varied)."""
    ops = load()
    if not ops: return
    cols = sorted({v for p in train[:1] for g in (p["input"], p["output"]) for r in g for v in r})
    for fn, f_, params in ops:
        n = 0
        enum_ps = [p for p in params if p[1] == "enum"]
        main = enum_ps[0] if enum_ps else None
        others = [p for p in params if p is not main]
        base = {pn: d for pn, _, _, d in params}
        wl = whitelist()
        for tv in (main[2] if main else [None]):
            if n > per_op: break
            if wl is not None and (fn.replace("_grid_based", ""), str(tv)) not in wl: continue
            kw = dict(base)
            if main: kw[main[0]] = tv
            variants = [dict(kw)]
            for pn, kind, vals, d in others:
                opts = cols if kind == "colour" else vals[1:] if kind in ("enum", "bool", "int") else []
                for v in opts:
                    if v == d: continue
                    k2 = dict(kw); k2[pn] = v; variants.append(k2)
            # two colour params varied together over induced colours (e.g. colour1 -> colour2)
            cps = [p for p in others if p[1] == "colour"]
            if len(cps) >= 2:
                for a in cols:
                    for b in cols:
                        if a != b:
                            k2 = dict(kw); k2[cps[0][0]] = a; k2[cps[1][0]] = b; variants.append(k2)
            for k2 in variants:
                n += 1
                if n > per_op: break
                nd = sum(1 for pn, _, _, d in params if k2.get(pn) != d and (not main or pn != main[0]))
                label = f"codex:{fn.replace('_grid_based', '')}[{tv}]" + ("" if not nd else "(" + ",".join(f"{pn}={k2[pn]}" for pn, _, _, d in params if k2.get(pn) != d and (not main or pn != main[0])) + ")")
                yield (label, 3 + nd, (lambda g, f_=f_, k2=k2: f_([list(r) for r in g], **k2)))


In [ ]:
%%writefile /kaggle/working/probe/codex_types_general.json
{
 "rule": "Codex grid-op types admitted only if they solve >=2 distinct training tasks (cycle 8)",
 "types": [
  [
   "beam",
   "box_based"
  ],
  [
   "connect",
   "connect_rectangles"
  ],
  [
   "downscale",
   "horizontal"
  ],
  [
   "extract",
   "periodic_hole_patch"
  ],
  [
   "mirror",
   "diagonal"
  ],
  [
   "placement",
   "align_all_colors_to_guide_top"
  ],
  [
   "tile",
   "macro_mask_self_substitution"
  ]
 ]
}

In [ ]:
%%writefile /kaggle/working/probe/compose_ctx.py
"""compose_ctx: inter-part CONTEXT composition.   part_out = OP(part_in, ctx)   [optionally + a library program]

A program here is   partition -> for every part: ctx = ROLE(part) -> OP(part, ctx) on the parts a PREDICATE
selects -> reassemble, every step induced from the training pairs; the whole program is verified on all
training pairs, then must pass a held-out-pair (leave-one-out) re-induction check.

Search space
  partitions   the partitions of compose_lift (imported, not copied): sep panels, bgsep bands (rc / r / c),
               frame interiors, equal blocks (nh x nw), object bounding boxes (8-conn multi-colour, 4- and 8-conn
               single-colour; INPLACE only, boxes must be disjoint); plus 'sepx': separator lines of any colour,
               the grid's majority colour included.
  ROLE (ctx)   'sel'  one global context part chosen by a lift selector (max / min attribute, odd-one-out,
                      has / lacks / most of a colour, fixed index): template, key, marker part ...
               'next' / 'prev' in reading order (cyclic or not); 'right' / 'left' / 'down' / 'up' neighbour in
                      the part lattice (cyclic or not); 'mh' / 'mv' / 'mp' mirror part (row / column / point
                      mirror of the lattice position).
  OP           'cell' cellwise table   out[y][x] = E(key)   with the ctx optionally transformed by one of D8
                      (dims must agree).  key = nothing ('u'), the ctx cell or the part cell or both, as a colour
                      ROLE (rank in its part by count; ties: scene background, first cell) or as a literal colour.
                      E = the part cell, the ctx cell, a role colour of the part or of the ctx, or a literal
                      colour; per key the consistent expressions are intersected (bit masks) and the first in
                      prior order is used.  The simplest key mode that is consistent wins.  Covers copy ctx,
                      overlay either way, stamp, swap, recolour ctx with the part's colours, boolean and/or/xor.
               'recol' per-colour recolour of the part (no cell alignment with ctx): part colour (role or
                      literal) -> the same colour, a role colour of the ctx or of the part, or a literal colour
                      (fill the part's background with ctx's colour, band i recoloured by band i+1 ...).
               'lib'  (after 'cell' / 'recol' when no table fits) one library program on the pooled
                      (OP result, output part) pairs, re-induced by compose_lift.lib_search.
  PREDICATE    which parts get OP (the others must be unchanged): all, not-the-ctx, has / lacks colour c,
               number of colours; the first predicate (prior order) that takes every changed part and leaves out
               every part OP would change wrongly.
  modes        INPLACE  same-size tasks, disjoint regions, cells outside regions unchanged.
               PROG     output = the NEXT panel of the input's panel sequence (reading order, forward or
                      backward, blank panels skipped as the canvas): per colour translation extrapolated by one
                      step, or growth continued (the cells added between the last two panels, shifted by the
                      unit step).  Output: the panel or the panel with its one-cell separator frame.
Generalisation check: re-induction on all training pairs but one predicts the held-out pair exactly (last pair
with a change; also the first one when >= 3 pairs change).  Test predictions proposed by several hypotheses
are ranked first (votes); at most 3 programs with distinct predictions are returned.
Budget: the DETERMINISTIC work counter of compose_lift (tick / Budget / work_cap / SEC); no clocks, no timers,
no signals.  Charges: one partition (lift's cost), one table fit / prediction (HYP_BASE + CELL_UNIT per cell
visited), library enumerations (lift's cost model).  <= 19 s of work per task.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen'); sys.path.insert(0, '/home/claude/work/latent')
import gdsl
import compose_lift as L
from gdsl import H, W, colours
from collections import Counter

TOTAL_BUDGET = 19.0
DEBUG = False
DIAG = Counter()

HYP_BASE = 2_000                 # work units: one hypothesis fit / prediction pass (base)
CELL_UNIT = 6                    # work units per cell visited by a table fit or prediction (calibrated: >= CPU)
tick = L.tick
_Timeout = L._Timeout
SEC = L.SEC


def _cells(g):
    return len(g) * len(g[0]) if g and g[0] else 0


def dims(g):
    return (len(g), len(g[0]) if g else 0)


def d8(g, t):
    if t == 0: return g
    if t == 1: return [r[::-1] for r in g]
    if t == 2: return g[::-1]
    if t == 3: return [r[::-1] for r in g[::-1]]
    T = [list(r) for r in zip(*g)]
    if t == 4: return T
    if t == 5: return [r[::-1] for r in T]
    if t == 6: return T[::-1]
    return [r[::-1] for r in T[::-1]]


D8N = ('id', 'flipH', 'flipV', 'rot180', 'transpose', 'rot90', 'rot270', 'antitranspose')


def roles(g, sbg, rs=0):
    """Colours of a part in role order.  rs 0: the scene background first (when present), then by count, first
    cell; rs 1: by count (ties: the scene background first, then first cell)."""
    cnt = Counter(); first = {}
    for y, row in enumerate(g):
        for x, v in enumerate(row):
            cnt[v] += 1
            if v not in first: first[v] = (y, x)
    if rs: return sorted(cnt, key=lambda c: (-cnt[c], c != sbg, first[c]))
    return sorted(cnt, key=lambda c: (c != sbg, -cnt[c], first[c]))


MAXR = 4


def _ridx(rl, v):
    try:
        k = rl.index(v)
    except ValueError:
        return MAXR
    return k if k < MAXR else MAXR


# expression bits: 0 P (part cell), 1 C (ctx cell), 2..5 part role k, 6..9 ctx role k, 10..19 literal colour
def _cand(o, p, c, pr, cr, aligned):
    m = 1 << (10 + o)
    if o == p: m |= 1
    if aligned and o == c: m |= 2
    for k in range(min(MAXR, len(pr))):
        if pr[k] == o: m |= 1 << (2 + k)
    if cr is not None:
        for k in range(min(MAXR, len(cr))):
            if cr[k] == o: m |= 1 << (6 + k)
    return m


def _eval(e, p, c, pr, cr):
    if e == 0: return p
    if e == 1: return c
    if e < 6:
        k = e - 2
        return pr[k] if k < len(pr) else None
    if e < 10:
        k = e - 6
        return cr[k] if cr is not None and k < len(cr) else None
    return e - 10


def _low(m):
    return (m & -m).bit_length() - 1


def _enames(e):
    if e == 0: return 'P'
    if e == 1: return 'C'
    if e < 6: return f'p{e - 2}'
    if e < 10: return f'c{e - 6}'
    return f'#{e - 10}'


KEYMODES_ALIGNED = ('u', 'cr', 'pr', 'pcr', 'cl', 'pl', 'pcl')
KEYMODES_RECOL = ('u', 'pr', 'pl')


def _key(mode, p, c, pr, cr):
    if mode == 'u': return 0
    if mode == 'cr': return _ridx(cr, c)
    if mode == 'pr': return _ridx(pr, p)
    if mode == 'pcr': return (_ridx(pr, p), _ridx(cr, c))
    if mode == 'cl': return c
    if mode == 'pl': return p
    return (p, c)


def table_fit(recs, aligned, t, rs=0):
    """recs: [(part grid, ctx grid, out grid, scene bg)].  Returns (keymode, {key: expr}) for the simplest
    consistent key mode, or None."""
    prep = []
    for pg, cg, og, sbg in recs:
        if dims(og) != dims(pg): return None
        pr = roles(pg, sbg, rs); cr = roles(cg, sbg, rs) if cg is not None else None
        cgt = d8(cg, t) if aligned else None
        if aligned and dims(cgt) != dims(pg): return None
        cells = []
        for y in range(len(pg)):
            prow, orow = pg[y], og[y]
            crow = cgt[y] if aligned else None
            for x in range(len(prow)):
                p = prow[x]; c = crow[x] if aligned else None
                cells.append((p, c, _cand(orow[x], p, c, pr, cr, aligned)))
        tick(HYP_BASE + CELL_UNIT * len(cells))
        prep.append((pr, cr, cells))
    for mode in (KEYMODES_ALIGNED if aligned else KEYMODES_RECOL):
        tab = {}; ok = True
        for pr, cr, cells in prep:
            for p, c, m in cells:
                k = _key(mode, p, c, pr, cr)
                v = tab.get(k, -1) & m
                if not v: ok = False; break
                tab[k] = v
            if not ok: break
        if ok:
            tick(CELL_UNIT * sum(len(x[2]) for x in prep))
            return mode, {k: _low(v) for k, v in tab.items()}
    tick(CELL_UNIT * len(KEYMODES_ALIGNED) * sum(len(x[2]) for x in prep))
    return None


def table_apply(model, pg, cg, sbg, aligned, t, rs=0):
    mode, tab = model
    tick(HYP_BASE + CELL_UNIT * _cells(pg))
    pr = roles(pg, sbg, rs); cr = roles(cg, sbg, rs) if cg is not None else None
    if aligned:
        cgt = d8(cg, t)
        if dims(cgt) != dims(pg): return None
    out = []
    for y in range(len(pg)):
        prow = pg[y]; crow = cgt[y] if aligned else None; orow = []
        for x in range(len(prow)):
            p = prow[x]; c = crow[x] if aligned else None
            e = tab.get(_key(mode, p, c, pr, cr))
            if e is None: return None
            v = _eval(e, p, c, pr, cr)
            if v is None: return None
            orow.append(v)
        out.append(orow)
    return out


def model_name(model):
    mode, tab = model
    if mode == 'u': return _enames(tab[0])
    items = sorted(tab.items(), key=lambda kv: str(kv[0]))
    return mode + '{' + ','.join(f"{k}:{_enames(e)}" for k, e in items[:8]) + (',..' if len(items) > 8 else '') + '}'


# ------------------------------------------------------------------ partitions
def sepx_layout(g):
    """Like compose_lift.sep_layout, but the separator colour may be the grid's majority colour too (panels
    between full-length single-colour lines; the colour giving the most panels, non-majority first on ties)."""
    h, w = H(g), W(g); bg = L._ORIG_BG(g); best = None
    for sc in sorted(colours(g)):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols: continue
        if len(rows) == h or len(cols) == w: continue
        ri, ci = L._intervals(rows, h), L._intervals(cols, w)
        n = len(ri) * len(ci)
        if n < 2: continue
        if best is None or (n, sc != bg) > best[0]: best = ((n, sc != bg), sc, ri, ci)
    return best[1:] if best else None


def partition(spec, g):
    if spec[0] != 'sepx': return L.partition(spec, g)
    tick(L.PART_BASE + L.PART_CELL * _cells(g))
    lay = sepx_layout(g)
    if lay is None: return None
    sc, ri, ci = lay
    ps = [L.Part(a, b, a1 - a + 1, b1 - b + 1, [row[b:b1 + 1] for row in g[a:a1 + 1]]) for a, a1 in ri for b, b1 in ci]
    if len(ps) > L.MAX_PARTS: return None
    for p in ps: p.a['sepc'] = sc
    L._attrs(ps, g)
    return ps


def spec_name(spec):
    return 'sepx' if spec[0] == 'sepx' else L.spec_name(spec)


# ------------------------------------------------------------------ lattice and context roles
def lattice(ps):
    rows = sorted({p.r0 for p in ps}); cols = sorted({p.c0 for p in ps})
    at = {}
    for k, p in enumerate(ps):
        key = (rows.index(p.r0), cols.index(p.c0))
        if key in at: return None
        at[key] = k
    return len(rows), len(cols), at


REL_RULES = [('next', True), ('prev', True), ('next', False), ('prev', False), ('right', True), ('left', True),
             ('down', True), ('up', True), ('right', False), ('left', False), ('down', False), ('up', False),
             ('mh',), ('mv',), ('mp',)]


def ctx_map(rule, ps):
    """Index of the context part of every part (None: no context), or None when the rule does not apply."""
    n = len(ps); kind = rule[0]
    if kind == 'sel':
        c = L.select(ps, rule[1])
        if c is None: return None
        i = next(k for k, p in enumerate(ps) if p is c)
        return [i] * n
    if kind in ('next', 'prev'):
        d = 1 if kind == 'next' else -1
        if rule[1]: return [(k + d) % n for k in range(n)]
        return [k + d if 0 <= k + d < n else None for k in range(n)]
    lat = lattice(ps)
    if lat is None: return None
    R, C, at = lat
    pos = {k: rc for rc, k in at.items()}
    out = []
    for k in range(n):
        r, c = pos[k]
        if kind in ('right', 'left', 'down', 'up'):
            dr, dc = {'right': (0, 1), 'left': (0, -1), 'down': (1, 0), 'up': (-1, 0)}[kind]
            if (kind in ('right', 'left') and C < 2) or (kind in ('down', 'up') and R < 2): return None
            rr, cc = r + dr, c + dc
            if rule[1]: rr %= R; cc %= C
        elif kind == 'mh': rr, cc = r, C - 1 - c
        elif kind == 'mv': rr, cc = R - 1 - r, c
        else: rr, cc = R - 1 - r, C - 1 - c
        out.append(at.get((rr, cc)))
    return out


def rule_name(rule):
    if rule[0] == 'sel': return f"sel-{rule[1][0]}-{rule[1][1]}"
    if len(rule) == 1: return rule[0]
    return rule[0] + ('-cyc' if rule[1] else '')


# ------------------------------------------------------------------ predicates (which parts get OP)
def _pred_list(palette, ncols):
    ps = [('all',), ('notctx',)]
    for c in sorted(palette):
        ps += [('has', c), ('lacks', c)]
    for n in sorted(ncols):
        ps.append(('ncol', n))
    ps.append(('ncol>=', 2))
    return ps


def pred_ok(pred, pt, is_ctx):
    k = pred[0]
    if k == 'all': return True
    if k == 'notctx': return not is_ctx
    cs = pt.a.get('allc')
    if cs is None:
        cs = frozenset(v for r in pt.g for v in r); pt.a['allc'] = cs
    if k == 'has': return pred[1] in cs
    if k == 'lacks': return pred[1] not in cs
    if k == 'ncol': return len(cs) == pred[1]
    if k == 'ncol>=': return len(cs) >= pred[1]
    return False


# ------------------------------------------------------------------ INPLACE-CTX
INPLACE_SPECS = [('sepx',), ('sep',), ('bgsep', 'rc'), ('bgsep', 'r'), ('bgsep', 'c'), ('frame',),
                 ('obj', True, False, 0, False), ('obj', False, True, 0, False), ('obj', True, True, 0, False)]


def setup_inplace(spec, grids_io):
    """[(parts, out crops or None, scene bg)] per grid; None when the spec does not give disjoint regions that
    hold every change."""
    res = []
    for gi, go in grids_io:
        ps = partition(spec, gi)
        if not ps or len(ps) < 2: return None
        occ = L._disjoint(ps, H(gi), W(gi))
        if occ is None: return None
        sbg = L._ORIG_BG(gi)
        outs = None
        if go is not None:
            if dims(go) != dims(gi): return None
            for y in range(H(gi)):
                ri, ro = gi[y], go[y]
                for x in range(W(gi)):
                    if ri[x] != ro[x] and (y, x) not in occ: return None
            outs = [L.region_crop(go, p.r0, p.c0, p.h, p.w, sbg) for p in ps]
        res.append((ps, outs, sbg))
    return res


class Hyp:
    """One context hypothesis: spec, role rule, op kind ('cell' with D8 t, or 'recol')."""
    def __init__(self, spec, rule, aligned, t, rs=0):
        self.spec, self.rule, self.aligned, self.t, self.rs = spec, rule, aligned, t, rs

    def name(self, model=None, pred=None):
        op = ('cell' + ('' if self.t == 0 else '@' + D8N[self.t])) if self.aligned else 'recol'
        op += '/major' if self.rs else ''
        s = f"ctx[{spec_name(self.spec)}]:{rule_name(self.rule)}:{op}"
        if model is not None: s += ':' + model_name(model)
        if pred is not None and pred != ('all',): s += ':if-' + '-'.join(map(str, pred))
        return s


def fit_hyp(h, setup, palette, ncols):
    """Induce (table model, predicate) of hypothesis h from the setup of some training pairs.  None if none."""
    recs = []; allrecs = []
    for ps, outs, sbg in setup:
        cm = ctx_map(h.rule, ps)
        if cm is None: return None
        for k, pt in enumerate(ps):
            ch = pt.g != outs[k]
            ci = cm[k]
            if ci is None:
                if ch: return None
                continue
            allrecs.append((pt, ps[ci], outs[k], sbg, ci == k, ch))
            if ch: recs.append((pt.g, ps[ci].g, outs[k], sbg))
    if not recs: return None
    model = table_fit(recs, h.aligned, h.t, h.rs)
    if model is None: return None
    good = []
    for pt, cp, o, sbg, isc, ch in allrecs:
        r = o if ch else table_apply(model, pt.g, cp.g, sbg, h.aligned, h.t, h.rs)
        good.append(r == o)
    for pred in _pred_list(palette, ncols):
        if all((pred_ok(pred, rc[0], rc[4]) if rc[5] else (g or not pred_ok(pred, rc[0], rc[4])))
               for rc, g in zip(allrecs, good)):
            return model, pred
    return None


def apply_hyp(h, model, pred, g):
    ps = partition(h.spec, g)
    if not ps or len(ps) < 2: return None
    if L._disjoint(ps, H(g), W(g)) is None: return None
    cm = ctx_map(h.rule, ps)
    if cm is None: return None
    sbg = L._ORIG_BG(g); canvas = L.copyg(g)
    for k, pt in enumerate(ps):
        ci = cm[k]
        if ci is None or not pred_ok(pred, pt, ci == k): continue
        r = table_apply(model, pt.g, ps[ci].g, sbg, h.aligned, h.t, h.rs)
        if r is None: return None
        L.paste(canvas, r, pt.r0, pt.c0)
    return canvas


def _palette(setup):
    pal = set(); nc = set()
    for ps, _, _ in setup:
        for pt in ps:
            cs = frozenset(v for r in pt.g for v in r); pt.a['allc'] = cs
            pal |= cs; nc.add(len(cs))
    return pal, nc


def try_inplace_ctx(task, budget, emit):
    train, test = task['train'], task['test']
    io = [(p['input'], p['output']) for p in train]
    tin = [p['input'] for p in train]
    specs = list(INPLACE_SPECS) + L._block_specs(tin + [t['input'] for t in test], limit=3)
    changed_pairs = [j for j, (a, b) in enumerate(io) if a != b]
    if not changed_pairs: return
    folds = [changed_pairs[-1]] + ([changed_pairs[0]] if len(changed_pairs) >= 3 else [])
    seen = set()
    for spec in specs:
        if budget.left() < 0.5: return
        st = setup_inplace(spec, io)
        if st is None: continue
        key = tuple((p.r0, p.c0, p.h, p.w) for ps, _, _ in st for p in ps)
        if key in seen: continue
        seen.add(key)
        tst = setup_inplace(spec, [(t['input'], None) for t in test])
        if tst is None: continue
        pal, nc = _palette(st)
        _palette(tst)
        ns = {len(ps) for ps, _, _ in st + tst}
        rules = [('sel', s) for s in L._selectors(tin, ns.pop() if len(ns) == 1 else 0)] + REL_RULES
        if DEBUG: print(' spec', spec_name(spec), 'parts', [len(x[0]) for x in st], 'rules', len(rules), flush=True)
        n_emit = 0
        for rule in rules:
            if budget.left() < 0.5: return
            sig = []
            ok = True
            for ps, _, _ in st + tst:
                cm = ctx_map(rule, ps)
                if cm is None: ok = False; break
                sig.append(tuple(cm))
            if not ok: continue
            sig = (tuple(sig))
            if (key, sig) in seen: continue          # same context assignment as an earlier rule
            seen.add((key, sig))
            for aligned, t, rs in OPS:
                h = Hyp(spec, rule, aligned, t, rs)
                fm = fit_hyp(h, st, pal, nc)
                if fm is None: continue
                model, pred = fm
                if not _loo(h, st, folds, pal, nc, io):
                    DIAG['loo_reject'] += 1
                    continue
                F = lambda g, h=h, model=model, pred=pred: apply_hyp(h, model, pred, g)
                cost = 20 + (0 if aligned else 1) + (2 if t else 0) + rs + _mode_cost(model) + (0 if pred == ('all',) else 1) \
                    + (0 if rule[0] != 'sel' else 1)
                if emit(h.name(model, pred), cost, F):
                    n_emit += 1
                break                                  # first op kind that fits for this rule
            if n_emit >= 4: break


OPS = [(True, 0, 0), (False, 0, 0), (True, 0, 1), (False, 0, 1)] + [(True, t, 0) for t in range(1, 8)]


def _mode_cost(model):
    return {'u': 0, 'cr': 1, 'pr': 1, 'pcr': 2, 'cl': 2, 'pl': 2, 'pcl': 3}[model[0]]


def _loo(h, st, folds, pal, nc, io):
    for j in folds:
        rest = [s for i, s in enumerate(st) if i != j]
        if not any(any(pt.g != o for pt, o in zip(ps, outs)) for ps, outs, _ in rest): return False
        fm = fit_hyp(h, rest, pal, nc)
        if fm is None: return False
        if apply_hyp(h, fm[0], fm[1], io[j][0]) != io[j][1]: return False
    return True




# ------------------------------------------------------------------ INPLACE-CTX + library (part level)
def _base_op(op, pg, cg, sbg):
    """A fixed binary operator whose result a library program then transforms: 'C' copy ctx, 'Co' ctx overlaid
    on the part (ctx non-background cells win), 'Po' the part overlaid on the ctx."""
    if dims(pg) != dims(cg): return None
    if op == 'C': return [r[:] for r in cg]
    pb, cb = roles(pg, sbg, 1)[0], roles(cg, sbg, 1)[0]
    if op == 'Co': return [[c if c != cb else p for p, c in zip(rp, rc)] for rp, rc in zip(pg, cg)]
    return [[p if p != pb else c for p, c in zip(rp, rc)] for rp, rc in zip(pg, cg)]


def try_inplace_lib(task, budget, emit, max_calls=4):
    train, test = task['train'], task['test']
    io = [(p['input'], p['output']) for p in train]
    tin = [p['input'] for p in train]
    tbg = L._task_bg(task)
    calls = 0; seen = set()
    for spec in INPLACE_SPECS[:6]:
        st = setup_inplace(spec, io)
        if st is None: continue
        tst = setup_inplace(spec, [(t['input'], None) for t in test])
        if tst is None: continue
        pal, nc = _palette(st); _palette(tst)
        ns = {len(ps) for ps, _, _ in st + tst}
        rules = [('sel', s) for s in L._selectors(tin, ns.pop() if len(ns) == 1 else 0)] + REL_RULES
        for rule in rules:
            maps = [ctx_map(rule, ps) for ps, _, _ in st]
            tmaps = [ctx_map(rule, ps) for ps, _, _ in tst]
            if any(m is None for m in maps + tmaps): continue
            ok = True; recs = []
            for (ps, outs, sbg), cm, j in zip(st, maps, range(len(st))):
                for k, pt in enumerate(ps):
                    if pt.g == outs[k]: continue
                    if cm[k] is None or cm[k] == k or dims(ps[cm[k]].g) != dims(pt.g): ok = False; break
                    recs.append((pt, ps[cm[k]], outs[k], sbg, j))
                if not ok: break
            if not ok or not recs or len({r[4] for r in recs}) < 2: continue
            for op in ('C', 'Co', 'Po'):
                if budget.left() < 1.0 or calls >= max_calls: return
                meta = [(r[0], _base_op(op, r[0].g, r[1].g, r[3]), r[2], r[4]) for r in recs]
                if any(m[1] is None for m in meta): continue
                if all(m[1] == m[2] for m in meta): continue       # the table search owns plain copies
                sig = str([(m[1], m[2]) for m in meta])
                if sig in seen: continue
                seen.add(sig)
                if not L._functional([(m[1], m[2]) for m in meta]): continue
                calls += 1
                res = L.lib_search([(m[1], m[2]) for m in meta], [], budget, cap=2.5, maxp=3, fbg=tbg)
                res = L.loo_filter(meta, res, budget, tbg, None, folds=2, occam=True)
                for name, cost, _, ap in res:
                    F = lambda g, spec=spec, rule=rule, op=op, ap=ap, pal=pal: _apply_lib(spec, rule, op, ap, g)
                    emit(f"ctx[{spec_name(spec)}]:{rule_name(rule)}:{op}>{name}", 32 + cost, F)


def _apply_lib(spec, rule, op, ap, g):
    """Every part whose ctx has its dims and differs from it gets LIB(OP(part, ctx)); other parts kept."""
    ps = partition(spec, g)
    if not ps or len(ps) < 2 or L._disjoint(ps, H(g), W(g)) is None: return None
    cm = ctx_map(rule, ps)
    if cm is None: return None
    sbg = L._ORIG_BG(g); canvas = L.copyg(g)
    for k, pt in enumerate(ps):
        ci = cm[k]
        if ci is None or ci == k or dims(ps[ci].g) != dims(pt.g): continue
        b = _base_op(op, pt.g, ps[ci].g, sbg)
        r = ap(b)
        if r is None or dims(r) != dims(pt.g): return None
        L.paste(canvas, r, pt.r0, pt.c0)
    return canvas


# ------------------------------------------------------------------ PROG (panel sequence extrapolation)
PROG_SPECS = [('sepx',), ('sep',), ('bgsep', 'rc'), ('bgsep', 'r'), ('bgsep', 'c')]
UNIT = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]


def _colour_sets(g, pbg):
    d = {}
    for y, row in enumerate(g):
        for x, v in enumerate(row):
            if v != pbg: d.setdefault(v, set()).add((y, x))
    return d


def _shift(cells, v):
    return {(y + v[0], x + v[1]) for y, x in cells}


def _translation(a, b):
    """The vector v with a shifted by v == b (None when there is none)."""
    if len(a) != len(b) or not a: return None
    (ya, xa), (yb, xb) = min(a), min(b)
    v = (yb - ya, xb - xa)
    return v if _shift(a, v) == b else None


def _pbg(panels):
    cnt = Counter(v for p in panels for r in p for v in r)
    return cnt.most_common(1)[0][0]


def extrapolate(seq, kind, pbg):
    """Next panel of a panel sequence.  kind 'shift': every changing colour moves by one constant vector per step
    (checked on every consecutive pair of the sequence); 'grow': every changing colour c continues its last step
    by one unit vector v:  out_c = (B_c + shift(added, v)) - shift(removed, v), where added = B_c - A_c,
    removed = A_c - B_c and added lies in A_c shifted by v."""
    A, B = seq[-2], seq[-1]
    h, w = dims(B)
    if any(dims(p) != (h, w) for p in seq): return None
    tick(HYP_BASE + CELL_UNIT * h * w * len(seq))
    sa, sb = _colour_sets(A, pbg), _colour_sets(B, pbg)
    out = [r[:] for r in B]
    if kind == 'shift':
        adds = []
        for c in sorted(set(sa) | set(sb)):
            a, b = sa.get(c, set()), sb.get(c, set())
            if a == b: continue
            v = _translation(a, b)
            if v is None or v == (0, 0): return None
            for k in range(len(seq) - 2):
                pa, pb = _colour_sets(seq[k], pbg).get(c, set()), _colour_sets(seq[k + 1], pbg).get(c, set())
                if _translation(pa, pb) != v: return None
            for y, x in b: out[y][x] = pbg
            adds.append((c, _shift(b, v)))
        if not adds: return None
        for c, cells in adds:
            for y, x in cells:
                if not (0 <= y < h and 0 <= x < w): return None
                out[y][x] = c
        return out
    rem, add = [], {}
    for c in sorted(set(sa) | set(sb)):
        a, b = sa.get(c, set()), sb.get(c, set())
        if a == b: continue
        D, R = b - a, a - b
        vs = [v for v in UNIT if D and D <= _shift(a, v)] if D else [v for v in UNIT if R <= _shift(b, (-v[0], -v[1]))]
        if len(vs) != 1: return None
        v = vs[0]
        rem.append((c, _shift(R, v)))
        for y, x in _shift(D, v) - b:
            if not (0 <= y < h and 0 <= x < w): continue
            if add.get((y, x), c) != c: return None
            add[(y, x)] = c
    if not rem: return None
    for c, cells in rem:
        for y, x in cells:
            if 0 <= y < h and 0 <= x < w and out[y][x] == c and (y, x) not in add: out[y][x] = pbg
    for (y, x), c in add.items(): out[y][x] = c
    return out


def prog_predict(spec, order, drop, kind, asm, g):
    ps = partition(spec, g)
    if not ps or len(ps) < 3 - drop: return None
    idx = list(range(len(ps)))
    if order: idx.reverse()
    seq = [ps[k] for k in idx]
    if drop:
        last, prev = seq[-1], seq[-2]
        if len(colours(last.g)) >= len(colours(prev.g)): return None
        seq = seq[:-1]
    if len(seq) < 2: return None
    pbg = _pbg([p.g for p in seq])
    r = extrapolate([p.g for p in seq], kind, pbg)
    if r is None: return None
    if asm == 'panel': return r
    base = seq[-1]
    ring = L.region_crop(g, base.r0 - 1, base.c0 - 1, base.h + 2, base.w + 2, L._ORIG_BG(g))
    L.paste(ring, r, 1, 1)
    return ring


def try_prog(task, budget, emit):
    train = task['train']
    for spec in PROG_SPECS:
        ps0 = partition(spec, train[0]['input'])
        if not ps0 or len({(p.h, p.w) for p in ps0}) != 1: continue
        h, w = ps0[0].h, ps0[0].w
        o = train[0]['output']
        asms = [a for a, d in (('panel', (h, w)), ('pad1', (h + 2, w + 2))) if dims(o) == d]
        for asm in asms:
            for kind in ('shift', 'grow'):
                for drop in (0, 1):
                    for order in (0, 1):
                        if budget.left() < 0.3: return
                        name = f"ctx-prog[{spec_name(spec)}]:{'rev' if order else 'fwd'}{'-dropblank' if drop else ''}:{kind}:{asm}"
                        F = lambda g, a=(spec, order, drop, kind, asm): prog_predict(*a, g)
                        emit(name, 30 + drop + order + (kind == 'grow'), F)



# ------------------------------------------------------------------ SELCTX (output = OP(selected part, its ctx))
SELCTX_SPECS = [('sepx',), ('sep',), ('bgsep', 'rc'), ('bgsep', 'r'), ('bgsep', 'c'), ('frame',)]


def selctx_pick(spec, sel, rule, g):
    ps = partition(spec, g)
    if not ps or len(ps) < 2: return None
    a = L.select(ps, sel)
    if a is None: return None
    k = next(i for i, p in enumerate(ps) if p is a)
    cm = ctx_map(rule, ps)
    if cm is None or cm[k] is None or cm[k] == k: return None
    return a.g, ps[cm[k]].g, L._ORIG_BG(g)


def try_selctx(task, budget, emit):
    train, test = task['train'], task['test']
    tin = [p['input'] for p in train]
    folds = [len(train) - 1] + ([0] if len(train) >= 3 else [])
    specs = SELCTX_SPECS + L._block_specs(tin + [t['input'] for t in test], limit=3)
    seen = set(); n = 0
    for spec in specs:
        pin = [partition(spec, g) for g in tin]
        if any(not ps or len(ps) < 2 for ps in pin): continue
        pte = [partition(spec, t['input']) for t in test]
        if any(not ps or len(ps) < 2 for ps in pte): continue
        ns = {len(ps) for ps in pin + pte}
        sels = L._selectors(tin, ns.pop() if len(ns) == 1 else 0)
        for sel in sels:
            if budget.left() < 0.3 or n >= 3: return
            tick(L.SEL_BASE + sum(_cells(g) for g in tin) // 2)
            ch = [L.select(ps, sel) for ps in pin]
            if any(c is None or dims(c.g) != dims(p['output']) for c, p in zip(ch, train)): continue
            for rule in [('sel', s2) for s2 in sels if s2 != sel] + REL_RULES:
                picks = [selctx_pick(spec, sel, rule, g) for g in tin]
                if any(x is None for x in picks): continue
                tp = [selctx_pick(spec, sel, rule, t['input']) for t in test]
                if any(x is None for x in tp): continue
                key = (str(picks), str(tp))
                if key in seen: continue
                seen.add(key)
                for aligned, t, rs in OPS:
                    recs = [(a, b, p['output'], sbg) for (a, b, sbg), p in zip(picks, train)]
                    model = table_fit(recs, aligned, t, rs)
                    if model is None: continue
                    if all(a == p['output'] for (a, b, sbg), p in zip(picks, train)): break
                    ok = True
                    for j in folds:
                        m2 = table_fit([r for i, r in enumerate(recs) if i != j], aligned, t, rs)
                        a, b, sbg = picks[j]
                        if m2 is None or table_apply(m2, a, b, sbg, aligned, t, rs) != train[j]['output']:
                            ok = False; break
                    if not ok:
                        DIAG['loo_reject'] += 1; continue
                    h = Hyp(spec, rule, aligned, t, rs)
                    F = lambda g, a=(spec, sel, rule), m=model, h=h: (lambda x: None if x is None else
                                                                       table_apply(m, x[0], x[1], x[2], h.aligned, h.t, h.rs))(selctx_pick(*a, g))
                    name = f"ctx-sel[{spec_name(spec)}]:{sel[0]}-{sel[1]}+{rule_name(rule)}:" + h.name(model).split(':', 3)[-1]
                    if emit(name, 25 + _mode_cost(model) + (2 if t else 0) + rs, F): n += 1
                    break


# ------------------------------------------------------------------ SEARCH
def SEARCH(task):
    L._WORK[0] = 0; del L._CAPS[:]
    budget = L.Budget(TOTAL_BUDGET)
    DIAG.clear()
    found = []; seen = {}
    train, test = task['train'], task['test']

    def emit(name, cost, F):
        try:
            for p in train:
                if F(p['input']) != p['output']: return False
            preds = [F(t['input']) for t in test]
        except _Timeout:
            raise
        except Exception as e:
            if DEBUG: print('  emit error', name, repr(e)[:100])
            return False
        if any(p is None for p in preds): return False
        k = str(preds)
        if k in seen:
            seen[k]['votes'] += 1
            return False
        seen[k] = {'program': name, 'cost': cost, 'preds': preds, 'votes': 1}
        found.append(seen[k])
        if DEBUG: print('  EMIT', name, cost, flush=True)
        return True

    patched = L._install_patch()
    try:
        with L.work_cap(TOTAL_BUDGET + 0.5):
            same = all(dims(p['input']) == dims(p['output']) for p in train)
            if same:
                try_inplace_ctx(task, budget.sub(12.0), emit)
                if not found:
                    try_inplace_lib(task, budget, emit)
            else:
                try_prog(task, budget.sub(3.0), emit)
                try_selctx(task, budget.sub(12.0), emit)
    except _Timeout:
        pass
    finally:
        L._remove_patch(patched)
        L._FORCED[0] = None
        del L._CAPS[:]
    found.sort(key=lambda r: (-r['votes'], r['cost']))
    SEARCH.stats = {'work': round(L._WORK[0] / SEC, 3), 'diag': dict(DIAG)}
    return [{'program': r['program'] + (f" (x{r['votes']})" if r['votes'] > 1 else ''), 'cost': r['cost'], 'preds': r['preds']}
            for r in found[:3]]


In [ ]:
%%writefile /kaggle/working/probe/compose_lift.py
"""compose_lift: lift the whole G-DSL library from grids to PARTS of grids.

A program here is   partition -> (pair | select | group) -> library program on parts -> reassemble,
every step induced from the training pairs; the whole program is verified on all training pairs.

Search space
  partitions   sep      panels between full-length lines of one non-background colour (any thickness)
               bgsep    rectangles between all-background rows / columns (bands in one or both axes)
               frame    interiors of closed single-colour rectangular frames
               lines    every row (or every column)
               blocks   equal tiles, nh x nw split (SELECT only)
               obj      connected objects: 4/8-connected, single-/multi-colour, pad 0/1, masked object or
                        whole bbox crop
               layers   one full-size part per colour
  modes        INPLACE  same-size task: parts keep their regions.  Regions (panels, bands, frames, lines):
                        disjoint, output part = output crop, the rest unchanged.  Masked objects: layered -
                        all objects erased, each transformed part's foreground painted back (boxes may overlap;
                        every changed cell must lie in a box).  Bbox crops: pasted, overlaps must agree.
                        The (in,out) part pairs of ALL training pairs are pooled and the library searched once.
               GROUP    INPLACE / OBJMAP / LAYERS when no single program fits: parts grouped by one attribute
                        (colour, size, shape, extremal size, border contact, holes, orientation, adjacent colours,
                        minority-colour count, contains colour c, line parity ...); one program per group
                        (>= 2 examples from >= 2 training pairs); never-changing groups get the identity; the
                        changing groups must share ONE mechanism (family) whose parameter varies by group.
               SELIN    same-size task where only ONE selected part changes (selectors as in SELECT).
               OBJMAP   same-size task whose objects change extent: each object becomes a part of dims
                        rule(h,w) (transpose, grow/shrink by one ring or along one axis, x2, x3) anchored at its
                        centre or a corner, painted on the background canvas (disjoint parts).
               LAYERS   same-size task: every colour layer transformed (one program, or one per colour).
               SELECT   output = library(one selected part): extreme attribute (size, area, colours, density,
                        position), odd-one-out (unique colour set / shape / content / size / symmetry / holes),
                        contains / lacks / most of a marker colour, or fixed index when the part count is fixed.
                        Hypotheses selecting the same training parts share one library search.
               PANELMAP input panel lattice (separators or background gaps) and output with the same R x C
                        layout (own separators / margins, or bare R x C or C x R blocks): library on the pooled
                        panel pairs, output re-assembled as induced.
  library      gdsl.candidates() on the pooled part pairs (every family induces its parameters from the part
               pairs; at most INDUCE pairs, changed first, used for induction; a restricted family list without
               the few families whose cost on parts is unbounded, see _EXCLUDE) + identity / erase + optional
               global colour map, verified exactly on every part pair.  Parts of the scene see the scene
               background (bg_of is wrapped while a part-level program runs).  Fallback: the same search on
               colour-ROLE-normalised parts (colours renamed by frequency rank, mapped back at test).
Generalisation check (against over-firing): a program is kept only if the same program, re-induced on the parts
of all training pairs but one, predicts the held-out pair's parts exactly: last changed pair, plus the first
one for programs of cost >= 3 (occam) and always for SELECT / SELIN / GROUP.  Test predictions proposed by
several hypotheses are ranked first (votes); at most 3 programs with distinct predictions are returned.
Budget: DETERMINISTIC work budget (no clocks, no timers, no signals), so the predictions are identical on any
machine under any load.  A counter is advanced at the engine's natural units of work, each charged a cost that
depends only on the data: one library enumeration (gdsl.candidates) on a pool of part pairs (charged before it
runs, from the pool's pairs and cells), one candidate program applied to one part (by its cells), one partition
of a grid (by its cells), one selector evaluated on the parts.  Costs are calibrated so that 1 "second" of budget
(SEC work units) is about one CPU second on the calibration machine.  <= 19 s of work per task, shared by the
modes (same-size: INPLACE 11 s of which plain pooled search 7 s, then LAYERS 5 s, OBJMAP 3 s, SELIN 3 s, each only
while nothing is found; other tasks: PANELMAP 4 s, SELECT the rest); every library call has a work cap (an
enumeration that would exceed it is not started); hypotheses in prior order, identical part sets deduplicated,
context-free fits required (_functional).
"""
import sys, re
sys.path.insert(0, '/home/claude/work/widen')
import gdsl
from gdsl import H, W, colours, bbox
from collections import Counter, defaultdict

TOTAL_BUDGET = 19.0              # budget in calibrated seconds of WORK (see SEC), never wall-clock time
INDUCE = 10
DIAG = Counter()                 # per-task diagnostics: library searches, fits, generalisation checks
DEBUG = False
MAX_PARTS = 40
_ORIG_BG = gdsl.bg_of


# ------------------------------------------------------------------ deterministic work budget
# Work is counted in integer work units (wu); SEC wu ~ one CPU second on the calibration machine, so every budget
# and cap in this file reads as (calibrated) seconds.  Each charge depends only on the data, never on a clock.
SEC = 10_000_000                 # 1 wu ~ 0.1 microsecond
# Cost model (wu), fitted by non-negative least squares to this engine's own CPU profile (1400 library
# enumerations from 190 training tasks: every 7th task of the eval order plus the slowest ones; per-task R^2 0.85,
# 0.84 leave-one-task-out).  One enumeration gdsl.candidates(pool), charged BEFORE it runs, from the pool alone:
# same-size pool: base + per pair + per input cell + per foreground (non-background) input cell, more per
# foreground cell when every pair changes;  other pools: base + per (input cells x largest-grid cells).
CAND_SAME, CAND_DIFF = 2_430_000, 1_000_000
CAND_PAIR, CAND_CELL, CAND_FG, CAND_FG_CHANGED = 30_600, 1_620, 26_550, 13_700
CAND_CELLMX_DIFF = 180           # per 10 (input cells x largest-grid cells)
RUN_BASE, RUN_CELL = 190, 13     # one program applied to one grid: base + per cell
PART_BASE, PART_CELL = 400, 18   # one partition of a grid (objects, attributes): base + per cell
SEL_BASE = 20                    # one selector evaluated on the parts of every training grid: base + cells / 2
# Library families left out of part-level enumerations (a restricted family list, the deterministic bound on
# one library call): heavy, content-dependent induction cost that no pool feature predicts, and no program in
# the timed engine's full eval.  Always: 3 families measured at up to 2.4 s, 3.4 s and 275 s for ONE call on
# parts (the second on three grids of at most 10x10 cells, since the library dropped its own wall budget; the
# third runs an unbudgeted search over panel orders, exponential when no full chain exists).  On same-size
# pools whose largest grid has >= BIG_GRID cells also 4 families that take up to 1-2.4 s there on parts and
# 4-10 s (one > 30 s) on whole grids (they stay in on smaller grids, where they are cheap and do contribute).
_EXCLUDE = (('fam_stamp_local_stencil', 'fam_local_stencil'), ('fam_stamp_complete_partial_matches', 'fam_complete_partial'),
            ('prior_combinatorics', 'fam_panel_chain'))
_EXCLUDE_BIG = (('fam_move_slide_until_contact', 'fam_slide_until_contact'), ('fam_ray_diagonal_from_object', 'fam_corner_rays'),
                ('prior_mechanics', 'fam_mech_gravity'), ('prior_mechanics', 'fam_mech_lockkey'))
BIG_GRID = 200


class _Timeout(BaseException):
    pass


_WORK = [0]              # work units spent by the current SEARCH
_CAPS = []               # stack of [work deadline, fired]; outermost first


def _wu(sec):
    return int(round(sec * SEC))


def tick(u):
    """Charge u work units BEFORE doing that work.  A cap whose deadline the charge passes raises _Timeout (once,
    outermost cap first) and the counter stops at that deadline: the work is not done, the cap is used up."""
    n = _WORK[0] + u
    for e in _CAPS:
        if not e[1] and n > e[0]:
            e[1] = True
            _WORK[0] = max(_WORK[0], e[0])
            raise _Timeout()
    _WORK[0] = n


class Budget:
    def __init__(self, total, parent=None):
        self.t0 = _WORK[0]; self.end = self.t0 + _wu(total)
        if parent is not None: self.end = min(self.end, parent.end)
    def left(self):
        """Remaining work, in calibrated seconds."""
        return (self.end - _WORK[0]) / SEC
    def sub(self, sec):
        """A share of the remaining work for one mode (unused work flows to the next mode)."""
        return Budget(sec, self)


class work_cap:
    """Raise _Timeout (once) at the first charge that brings the work spent inside the block over `sec`."""
    def __init__(self, sec):
        self.u = _wu(max(0.05, sec))
    def __enter__(self):
        _CAPS.append([_WORK[0] + self.u, False])
    def __exit__(self, *a):
        _CAPS.pop()
        return False


def _cells(g):
    return len(g) * len(g[0]) if g and g[0] else 0


def _run(fn, g):
    """gdsl.run charged by the grid's cells (the natural unit: one candidate program applied to one part)."""
    tick(RUN_BASE + RUN_CELL * _cells(g))
    return gdsl.run(fn, g)


def _pool_shape(train):
    """(same-size pool, largest grid cells) of the pairs a library enumeration induces on."""
    same = all(dims(p['input']) == dims(p['output']) for p in train)
    mx = max(max(_cells(p['input']), _cells(p['output'])) for p in train)
    return same, mx


def _cand_cost(train):
    """Deterministic cost of one library enumeration on these pairs (a function of the pairs only)."""
    same, mx = _pool_shape(train)
    ci = sum(_cells(p['input']) for p in train)
    if not same:
        return CAND_DIFF + CAND_CELLMX_DIFF * ci * mx // 10
    fg = 0
    for p in train:
        g = p['input']; bg = _ORIG_BG(g)
        fg += sum(v != bg for row in g for v in row)
    allch = all(p['input'] != p['output'] for p in train)
    return CAND_SAME + CAND_PAIR * len(train) + CAND_CELL * ci + (CAND_FG + (CAND_FG_CHANGED if allch else 0)) * fg


def _candidates(train):
    """gdsl.candidates(train) on the restricted family list (see _EXCLUDE)."""
    same, mx = _pool_shape(train)
    ex = _EXCLUDE + (_EXCLUDE_BIG if same and mx >= BIG_GRID else ())
    full = gdsl.FAMILIES
    gdsl.FAMILIES = tuple(f for f in full if (f.__module__, f.__name__) not in ex)
    try:
        return gdsl.candidates(train)
    finally:
        gdsl.FAMILIES = full


# ------------------------------------------------------------------ scene background forced on part-level runs
_FORCED = [None]


def _bg_patched(g):
    f = _FORCED[0]
    return f if f is not None else _ORIG_BG(g)


def _install_patch():
    """Every module holding gdsl.bg_of gets a wrapper that is transparent unless a part-level run forces the
    scene background (an object crop's own majority colour is not the scene's background)."""
    done = []
    for mod in list(sys.modules.values()):
        try:
            if mod is not None and getattr(mod, 'bg_of', None) is _ORIG_BG:
                setattr(mod, 'bg_of', _bg_patched); done.append(mod)
        except Exception:
            pass
    return done


def _remove_patch(done):
    for mod in done:
        try:
            setattr(mod, 'bg_of', _ORIG_BG)
        except Exception:
            pass


class forced_bg:
    def __init__(self, c):
        self.c = c
    def __enter__(self):
        self.old = _FORCED[0]; _FORCED[0] = self.c
    def __exit__(self, *a):
        _FORCED[0] = self.old
        return False


# ------------------------------------------------------------------ grid helpers
def region_crop(g, r0, c0, h, w, bg):
    hh, ww = H(g), W(g); out = []
    for y in range(r0, r0 + h):
        if 0 <= y < hh:
            row = g[y]
            out.append([row[x] if 0 <= x < ww else bg for x in range(c0, c0 + w)])
        else:
            out.append([bg] * w)
    return out


def paste(canvas, sub, r0, c0, only_fg=None):
    hh, ww = H(canvas), W(canvas)
    for dy, row in enumerate(sub):
        y = r0 + dy
        if not 0 <= y < hh: continue
        for dx, v in enumerate(row):
            x = c0 + dx
            if 0 <= x < ww and (only_fg is None or v != only_fg):
                canvas[y][x] = v


def copyg(g):
    return [r[:] for r in g]


def dims(g):
    return (H(g), W(g))


def _ident(g):
    return [r[:] for r in g]


def _erase(g):
    bg = _bg_patched(g)
    return [[bg] * W(g) for _ in range(H(g))]


# ------------------------------------------------------------------ parts
class Part:
    __slots__ = ('r0', 'c0', 'h', 'w', 'g', 'cells', 'a')
    def __init__(self, r0, c0, h, w, g, cells=None):
        self.r0, self.c0, self.h, self.w, self.g, self.cells = r0, c0, h, w, g, cells
        self.a = {}


def _intervals(seps, n):
    s = set(seps); out = []; y = 0
    while y < n:
        if y in s:
            y += 1; continue
        y1 = y
        while y1 + 1 < n and y1 + 1 not in s: y1 += 1
        out.append((y, y1)); y = y1 + 1
    return out


def sep_layout(g):
    """(separator colour, row intervals, col intervals) for full-length lines of one non-background colour."""
    h, w = H(g), W(g); bg = _ORIG_BG(g); best = None
    for sc in colours(g):
        if sc == bg: continue
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols: continue
        if len(rows) == h or len(cols) == w: continue
        ri, ci = _intervals(rows, h), _intervals(cols, w)
        n = len(ri) * len(ci)
        if n < 2: continue
        if best is None or n > best[0]: best = (n, sc, ri, ci)
    return best[1:] if best else None


def part_sep(g):
    lay = sep_layout(g)
    if lay is None: return None
    _, ri, ci = lay
    return [Part(a, b, a1 - a + 1, b1 - b + 1, [row[b:b1 + 1] for row in g[a:a1 + 1]]) for a, a1 in ri for b, b1 in ci]


def part_bgsep(g, axes):
    h, w = H(g), W(g); bg = _ORIG_BG(g)
    rows = [r for r in range(h) if all(v == bg for v in g[r])] if 'r' in axes else []
    cols = [c for c in range(w) if all(g[r][c] == bg for r in range(h))] if 'c' in axes else []
    ri, ci = _intervals(rows, h), _intervals(cols, w)
    out = []
    for a, a1 in ri:
        for b, b1 in ci:
            sub = [row[b:b1 + 1] for row in g[a:a1 + 1]]
            if any(v != bg for r in sub for v in r):
                out.append(Part(a, b, a1 - a + 1, b1 - b + 1, sub))
    return out if len(out) >= 2 else None


def part_blocks(g, nh, nw):
    h, w = H(g), W(g)
    if h % nh or w % nw: return None
    bh, bw = h // nh, w // nw
    return [Part(a, b, bh, bw, [row[b:b + bw] for row in g[a:a + bh]]) for a in range(0, h, bh) for b in range(0, w, bw)]


def part_lines(g, axis):
    if axis == 'r':
        return [Part(y, 0, 1, W(g), [g[y][:]]) for y in range(H(g))]
    return [Part(0, x, H(g), 1, [[g[y][x]] for y in range(H(g))]) for x in range(W(g))]


def part_frame(g):
    """Interiors of closed single-colour rectangular frames (4-connected objects owning their whole bbox border)."""
    bg = _ORIG_BG(g); out = []
    for ob in gdsl.objects(g, bg, False, True):
        r0, c0, r1, c1 = bbox(ob)
        if r1 - r0 < 2 or c1 - c0 < 2: continue
        col = g[ob[0][0]][ob[0][1]]
        if any(g[y][x] != col for y in (r0, r1) for x in range(c0, c1 + 1)): continue
        if any(g[y][x] != col for x in (c0, c1) for y in range(r0, r1 + 1)): continue
        p = Part(r0 + 1, c0 + 1, r1 - r0 - 1, c1 - c0 - 1, [row[c0 + 1:c1] for row in g[r0 + 1:r1]])
        p.a['fcol'] = col
        out.append(p)
    return out or None


def part_obj(g, diag, byc, pad, masked):
    bg = _ORIG_BG(g)
    obs = gdsl.objects(g, bg, diag, byc)
    if not obs or len(obs) > MAX_PARTS: return None
    out = []
    for ob in obs:
        r0, c0, r1, c1 = bbox(ob)
        r0 -= pad; c0 -= pad; r1 += pad; c1 += pad
        hh, ww = r1 - r0 + 1, c1 - c0 + 1
        if masked:
            sub = [[bg] * ww for _ in range(hh)]
            for y, x in ob: sub[y - r0][x - c0] = g[y][x]
        else:
            sub = region_crop(g, r0, c0, hh, ww, bg)
        out.append(Part(r0, c0, hh, ww, sub, ob))
    return out


def part_layers(g):
    """One full-size part per foreground colour (that colour's cells on the background)."""
    bg = _ORIG_BG(g)
    cols = sorted(colours(g) - {bg})
    return [Part(0, 0, H(g), W(g), [[v if v == c else bg for v in row] for row in g]) for c in cols] or None


def partition(spec, g):
    tick(PART_BASE + PART_CELL * _cells(g))
    try:
        k = spec[0]
        if k == 'lines': ps = part_lines(g, spec[1])
        elif k == 'layers': ps = part_layers(g)
        elif k == 'sep': ps = part_sep(g)
        elif k == 'bgsep': ps = part_bgsep(g, spec[1])
        elif k == 'frame': ps = part_frame(g)
        elif k == 'blocks': ps = part_blocks(g, spec[1], spec[2])
        elif k == 'obj': ps = part_obj(g, spec[1], spec[2], spec[3], spec[4])
        else: ps = None
    except _Timeout:
        raise
    except Exception:
        return None
    if not ps or len(ps) > (30 if k == 'lines' else MAX_PARTS): return None
    _attrs(ps, g)
    return ps


def spec_name(spec):
    k = spec[0]
    if k == 'obj':
        return f"obj[{'8' if spec[1] else '4'}{'' if spec[2] else ',multi'}{',pad' + str(spec[3]) if spec[3] else ''}{'' if spec[4] else ',bbox'}]"
    if k == 'blocks':
        return f"blocks[{spec[1]}x{spec[2]}-split]"
    if k == 'bgsep':
        return f"bgsep[{spec[1]}]"
    if k == 'lines':
        return 'rows' if spec[1] == 'r' else 'cols'
    return k


# ------------------------------------------------------------------ part attributes
def _shape_key(cells):
    r0, c0, _, _ = bbox(cells)
    return tuple(sorted((y - r0, x - c0) for y, x in cells))


def _has_hole(cells):
    """True when the cell set encloses background (4-connected region not reaching its padded bbox border)."""
    r0, c0, r1, c1 = bbox(cells)
    h, w = r1 - r0 + 3, c1 - c0 + 3
    occ = [[False] * w for _ in range(h)]
    for y, x in cells: occ[y - r0 + 1][x - c0 + 1] = True
    seen = [[False] * w for _ in range(h)]
    st = [(0, 0)]; seen[0][0] = True; n = 1
    while st:
        y, x = st.pop()
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and not occ[yy][xx]:
                seen[yy][xx] = True; st.append((yy, xx)); n += 1
    return n + len(cells) < h * w


def _attrs(ps, g):
    bg = _ORIG_BG(g); hh, ww = H(g), W(g)
    for p in ps:
        a = p.a
        if p.cells is not None:
            cells = p.cells
            cols = Counter(g[y][x] for y, x in cells)
            r0, c0, r1, c1 = bbox(cells)
        else:
            cols = Counter(v for r in p.g for v in r if v != bg)
            cells = [(y, x) for y, r in enumerate(p.g) for x, v in enumerate(r) if v != bg]
            r0, c0, r1, c1 = p.r0, p.c0, p.r0 + p.h - 1, p.c0 + p.w - 1
        a['size'] = sum(cols.values())
        a['shape'] = _shape_key(cells) if cells else ()
        a['bh'], a['bw'] = r1 - r0 + 1, c1 - c0 + 1
        a['border'] = r0 <= 0 or c0 <= 0 or r1 >= hh - 1 or c1 >= ww - 1
        a['top'], a['left'], a['bottom'], a['right'] = r0, c0, r1, c1
        a['colours'] = frozenset(cols)
        a['colour'] = next(iter(cols)) if len(cols) == 1 else ('multi', len(cols))
        a['ncol'] = len(cols)
        a['major'] = cols.most_common(1)[0][0] if cols else bg
        a['area'] = a['bh'] * a['bw']
        a['density'] = a['size'] / (p.h * p.w)
        a['content'] = str(p.g)
        a['sym'] = p.g == [r[::-1] for r in p.g] or p.g == p.g[::-1]
        a['solid'] = a['size'] == a['area']
        a['mcount'] = min(cols.most_common()[-1][1], 3) if len(cols) >= 2 else 0
        a['orient'] = 'h' if a['bw'] > a['bh'] else ('v' if a['bh'] > a['bw'] else 's')
        a['hole'] = _has_hole(cells) if cells and a['size'] < a['area'] and a['area'] <= 900 else False
        if p.cells is not None:                     # colours touching the object from outside (context)
            own = set(p.cells); adj = set()
            for y, x in p.cells:
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < hh and 0 <= xx < ww and (yy, xx) not in own and g[yy][xx] != bg:
                        adj.add(g[yy][xx])
            a['adj'] = frozenset(adj)
    if all(p.h == 1 and p.w == ww for p in ps) or all(p.w == 1 and p.h == hh for p in ps):
        for p in ps:
            p.a['par'] = (p.r0 + p.c0) % 2          # parity of a line's index (rows / cols only)
    sizes = [p.a['size'] for p in ps]
    mx, mn = max(sizes), min(sizes)
    ranks = sorted(set(sizes), reverse=True)
    shc = Counter(p.a['shape'] for p in ps)
    for p in ps:
        p.a['extreme'] = 'max' if p.a['size'] == mx else ('min' if p.a['size'] == mn else 'mid')
        p.a['srank'] = ranks.index(p.a['size'])
        p.a['shape_n'] = min(shc[p.a['shape']], 3)


# ------------------------------------------------------------------ library search on part pairs
def _applier(fn, m, fbg):
    def ap(g):
        with forced_bg(fbg):
            r = _run(fn, g)
            if r is not None and m is not None: r = gdsl.apply_cmap(r, m)
            return r
    return ap


def _base(name):
    return name[:-5] if name.endswith('+cmap') else name


def lib_search(pairs, tests, budget, cap=3.0, maxp=2, fbg=None, stats=None, want=None, names=None):
    """Library programs verified on pooled (in, out) part pairs: [(name, cost, test_preds, apply)].
    names: only these programs (re-induced on these pairs).  want: return every program whose test
    predictions equal it (generalisation check)."""
    seen = set(); P = []
    for i, o in pairs:
        k = (str(i), str(o))
        if k not in seen:
            seen.add(k); P.append((i, o))
    changed = [io for io in P if io[0] != io[1]]
    if not changed: return []
    changed.sort(key=lambda io: -(H(io[1]) * W(io[1])))
    P = changed + [io for io in P if io[0] == io[1]]
    w_start = _WORK[0]
    cap = min(cap, budget.left() - 0.3)
    if cap <= 0.1: return []
    # families induce their parameters from at most INDUCE pairs (changed ones first); every candidate is then
    # verified on all pairs, so the cap only trades recall for speed on large pools
    train = [{'input': i, 'output': o} for i, o in P[:INDUCE]]
    outs = [o for _, o in P]
    found = []
    with forced_bg(fbg):
        try:
            with work_cap(cap):
                tick(_cand_cost(train))       # charged up front: an enumeration that would exceed the cap is skipped
                cands = [('identity', 0, _ident), ('erase', 0, _erase)] + _candidates(train)
                i0, o0 = P[0]
                for name, cost, fn in cands:
                    if names is not None and name not in names: continue
                    r = _run(fn, i0)
                    if r is None or dims(r) != dims(o0): continue
                    if r == o0:
                        if any(_run(fn, i) != o for i, o in P[1:]): continue
                        prog = (name, cost, fn, None)
                    else:
                        m = gdsl.fit_cmap([r], [o0])
                        if m is None: continue
                        preds = [r]; ok = True
                        for i, o in P[1:]:
                            rr = _run(fn, i)
                            if rr is None or dims(rr) != dims(o): ok = False; break
                            preds.append(rr)
                        if not ok: continue
                        m = gdsl.fit_cmap(preds, outs)
                        if not m or any(gdsl.apply_cmap(a, m) != b for a, b in zip(preds, outs)): continue
                        prog = (name + '+cmap', cost + 1, fn, m)
                    ap = _applier(prog[2], prog[3], fbg)
                    tp = []
                    for t in tests:
                        r = ap(t)
                        if r is None: break
                        tp.append(r)
                    if len(tp) != len(tests): continue
                    if not tests:                     # no test parts given: keep distinct programs
                        found.append((prog[0], prog[1], tp, ap))
                        if len(found) >= maxp: break
                        continue
                    if want is not None:
                        if tp == want:
                            found.append((prog[0], prog[1], tp, ap))
                            if names is None: break
                        continue
                    if any(tp == f[2] for f in found): continue
                    found.append((prog[0], prog[1], tp, ap))
                    if len(found) >= maxp: break
        except _Timeout:
            pass
    if stats is not None:
        stats.append(round((_WORK[0] - w_start) / SEC, 2))
    if want is None:
        DIAG['search'] += 1
        DIAG['search_fit'] += bool(found)
    if DEBUG:
        print(f"   lib_search n={len(P)} changed={len(changed)} tests={len(tests)} want={want is not None} "
              f"-> {[f[0] for f in found]} {(_WORK[0] - w_start) / SEC:.2f}s(work)", flush=True)
    return found


def _held_out(meta, first=False):
    """Index of the training pair held out by the generalisation check (last, or first, pair with a changed
    part), or None when fewer than two training pairs show a change."""
    js = sorted({m[3] for m in meta if m[1] != m[2]})
    if len(js) < 2: return None
    return js[0] if first else js[-1]


def loo_filter(meta, res, budget, fbg, stats, folds=1, occam=False):
    """Generalisation check: keep the programs of `res` that, re-induced on the parts of all training pairs but
    one, predict the held-out pair's parts exactly (fold 1: last changed pair; fold 2: also the first).
    occam: the second fold only for programs of cost >= 3 (more capacity, more evidence required)."""
    if not res: return []
    ok = {_base(r[0]) for r in res}
    for first in (False, True)[:folds]:
        need = {_base(r[0]) for r in res if not (first and occam and r[1] < 3)} & ok
        if not need: continue
        jh = _held_out(meta, first)
        if jh is None: return []
        rest = [(i, o) for _, i, o, j in meta if j != jh]
        held = [(i, o) for _, i, o, j in meta if j == jh]
        got = lib_search(rest, [i for i, _ in held], budget, cap=3.0, fbg=fbg, stats=stats, want=[o for _, o in held],
                         names=need)
        ok -= need - {_base(r[0]) for r in got}
        if not ok:
            DIAG['loo_reject'] += 1
            return []
    DIAG['loo_pass'] += 1
    return [r for r in res if _base(r[0]) in ok]


# ------------------------------------------------------------------ colour-role normalisation of parts
class Norm:
    """Colours of a part renamed by role: its non-background colours ranked by (count desc, first occurrence)
    become canonical colours 1, 2, ... (background and the task's constant new output colours kept), so a
    library program induced on normalised parts binds colour ROLES instead of absolute colours."""
    def __init__(self, bg, keep):
        self.bg, self.keep = bg, keep
        self.canon = [c for c in (1, 2, 3, 4, 5, 6, 7, 8, 9, 0) if c != bg and c not in keep]

    def fwd(self, g):
        cnt = Counter(); first = {}
        for y, row in enumerate(g):
            for x, v in enumerate(row):
                if v != self.bg:
                    cnt[v] += 1
                    first.setdefault(v, (y, x))
        order = sorted(cnt, key=lambda c: (-cnt[c], first[c]))
        if len(order) > len(self.canon) or any(c in self.keep for c in order): return None
        m = {self.bg: self.bg}
        for k, c in enumerate(order): m[c] = self.canon[k]
        return m

    def apply(self, ap, g):
        m = self.fwd(g)
        if m is None: return None
        r = ap([[m[v] for v in row] for row in g])
        if r is None: return None
        inv = {v: k for k, v in m.items()}
        for c in self.keep: inv[c] = c
        if any(v not in inv for row in r for v in row): return None
        return [[inv[v] for v in row] for row in r]


def norm_meta(meta, bg):
    """Normalised copy of part pairs (None when normalisation cannot apply or changes nothing)."""
    keep = set()
    for m in meta: keep |= colours(m[2]) - colours(m[1]) - {bg}
    if len({frozenset(colours(m[1]) - {bg}) for m in meta}) < 2: return None, None
    N = Norm(bg, keep)
    out = []
    for pt, i, o, j in meta:
        mp = N.fwd(i)
        if mp is None: return None, None
        if any(v not in mp and v not in keep for row in o for v in row): return None, None
        out.append((pt, [[mp[v] for v in row] for row in i], [[mp.get(v, v) for v in row] for row in o], j))
    return N, out


def norm_search(meta, budget, fbg, nbg, stats, folds, maxp=4, occam=False):
    """Library search on colour-normalised parts (background nbg kept); returns [(name, cost, None, apply)]
    whose apply maps the part to roles, runs the program and maps the roles back."""
    N, nmeta = norm_meta(meta, nbg if nbg is not None else 0)
    if N is None: return []
    res = lib_search([(m[1], m[2]) for m in nmeta], [], budget, cap=2.5, maxp=maxp, fbg=fbg, stats=stats)
    res = loo_filter(nmeta, res, budget, fbg, stats, folds=folds, occam=occam)
    return [(name + '@norm', cost + 2, None, (lambda g, ap=ap: N.apply(ap, g))) for name, cost, _, ap in res]


# ------------------------------------------------------------------ helpers for hypotheses
def _same_bg(grids):
    bgs = {_ORIG_BG(g) for g in grids}
    return bgs.pop() if len(bgs) == 1 else None


def _task_bg(task):
    return _same_bg([p['input'] for p in task['train']] + [t['input'] for t in task['test']])


def _disjoint(parts, h, w):
    occ = set()
    for p in parts:
        for y in range(max(0, p.r0), min(h, p.r0 + p.h)):
            for x in range(max(0, p.c0), min(w, p.c0 + p.w)):
                if (y, x) in occ: return None
                occ.add((y, x))
    return occ


def _functional(pairs):
    """Identical input parts must have identical output parts (else no context-free part program exists)."""
    m = {}
    for i, o in pairs:
        if m.setdefault(str(i), str(o)) != str(o): return False
    return True


def _regions_key(pl):
    return tuple((p.r0, p.c0, p.h, p.w, p.a.get('content')) for p in pl)


# ------------------------------------------------------------------ GROUP (attribute-dependent programs)
GROUP_ATTRS = ('colour', 'extreme', 'size', 'shape', 'border', 'ncol', 'shape_n', 'bh', 'bw', 'major', 'sym', 'srank',
               'fcol', 'solid', 'hole', 'orient', 'adj', 'mcount', 'par')


def try_group(meta, tflat, builder, prefix, budget, fbg, emit, stats, gcalls, max_calls=6):
    """One library program per attribute group; groups whose parts never change get the identity.  Every
    changing group needs >= 2 examples from >= 2 training pairs, and the grouped hypothesis must pass the
    held-out-pair check group by group."""
    jh = _held_out(meta)
    if jh is None: return
    pal = set()
    for m in meta: pal |= m[0].a.get('colours', frozenset())
    extra = []
    if len(pal) <= 6:
        for c in sorted(pal):
            extra.append(('has', c))
            for pt in [m[0] for m in meta] + list(tflat):
                pt.a[('has', c)] = c in pt.a.get('colours', ())
    cands = []; tried = set()
    for an in GROUP_ATTRS + tuple(extra):
        groups = defaultdict(list)
        for m in meta:
            if an not in m[0].a: break
            groups[m[0].a[an]].append(m)
        else:
            if len(groups) < 2 or len(groups) > 4: continue
            if any(pt.a.get(an) not in groups for pt in tflat): continue
            sig = frozenset(frozenset(id(m) for m in v) for v in groups.values())
            if sig in tried: continue
            tried.add(sig)
            bad = False
            for v in groups.values():
                ch = [m for m in v if m[1] != m[2]]
                if ch and (len(v) < 2 or len({m[3] for m in v}) < 2): bad = True
                if not _functional([(m[1], m[2]) for m in v]): bad = True
            if bad: continue
            n_ident = sum(all(m[1] == m[2] for m in v) for v in groups.values())
            if n_ident == len(groups): continue
            # groups that are purely changed or purely unchanged separate the rule's domain cleanly
            mixed = sum(any(m[1] == m[2] for m in v) and any(m[1] != m[2] for m in v) for v in groups.values())
            cands.append((mixed, len(groups) - n_ident, -n_ident, len(cands), an, groups))
    cands.sort(key=lambda c: c[:4])
    folds = [jx for jx in (jh, _held_out(meta, first=True)) if jx is not None]   # last and first changed pair
    for _, n_search, _, _, an, groups in cands:
        if budget.left() < 1.0 or gcalls[0] + n_search > max_calls: return
        progs = {}; ok = True
        for val, v in sorted(groups.items(), key=lambda kv: str(kv[0])):
            if all(m[1] == m[2] for m in v):
                progs[val] = ('id', _ident); continue
            if budget.left() < 0.8: return
            gcalls[0] += 1
            tg = [pt.g for pt in tflat if pt.a.get(an) == val]
            res = lib_search([(m[1], m[2]) for m in v], tg, budget, cap=2.5, maxp=3, fbg=fbg, stats=stats)
            res = _group_loo(v, res, folds, budget, fbg, stats)
            if not res: ok = False; break
            progs[val] = (res[0][0], res[0][3])
        if not ok: continue
        fams = {_family(p[0]) for p in progs.values()} - {None}
        if len(fams) > 1: continue          # one mechanism whose parameter depends on the attribute
        F = builder(lambda pt, progs=progs, an=an: progs[pt.a.get(an)][1] if pt.a.get(an) in progs else None)
        name = f"{prefix}:group-by-{an}(" + '; '.join(f"{k}->{v[0]}" for k, v in sorted(progs.items(), key=lambda kv: str(kv[0]))) + ')'
        if emit(name, 30, F):
            gcalls[0] = max_calls + 1          # one grouped explanation is enough
            return


def _family(name):
    """Mechanism of a program name (None for the neutral identity / erase / pure colour map)."""
    b = _base(name)
    if b in ('id', 'identity', 'erase'): return None
    return re.split(r'[\[:]', b, 1)[0]


def _group_loo(v, res, folds, budget, fbg, stats):
    """Held-out-pair check for one group: each program of `res`, re-induced on the group's parts outside the
    held-out training pair, must predict the group's parts inside it (for every fold)."""
    ok = {_base(r[0]) for r in res}
    for jx in folds:
        held = [m for m in v if m[3] == jx]
        rest = [m for m in v if m[3] != jx]
        if not held: continue
        if not any(m[1] != m[2] for m in rest): return []
        got = lib_search([(m[1], m[2]) for m in rest], [m[1] for m in held], budget, cap=2.5, fbg=fbg, stats=stats,
                         want=[m[2] for m in held], names=ok)
        ok &= {_base(r[0]) for r in got}
        if not ok: return []
    return [r for r in res if _base(r[0]) in ok]


# ------------------------------------------------------------------ INPLACE
INPLACE_SPECS = [('sep',), ('obj', False, True, 0, True), ('obj', True, False, 0, True), ('frame',),
                 ('bgsep', 'rc'), ('bgsep', 'r'), ('bgsep', 'c'), ('obj', True, True, 0, True), ('obj', False, False, 0, True),
                 ('obj', False, True, 1, True), ('obj', True, False, 1, True), ('obj', True, True, 1, True),
                 ('obj', False, False, 1, True), ('obj', True, True, 0, False), ('obj', False, True, 0, False),
                 ('lines', 'r'), ('lines', 'c')]


def _block_specs(grids, limit=3):
    """Equal tiles as an nh x nw split (<= 4 per axis, >= 4 cells per tile) valid for every given grid;
    fewest tiles first."""
    specs = []
    for nh in range(1, 5):
        for nw in range(1, 5):
            if nh * nw < 2: continue
            if all(H(g) % nh == 0 and W(g) % nw == 0 and (H(g) // nh) * (W(g) // nw) >= 4 for g in grids):
                specs.append((nh * nw, ('blocks', nh, nw)))
    specs.sort(key=lambda s: s[0])
    return [s for _, s in specs[:limit]]


def _obj_targets(ps, gi, go, bg):
    """Output part of every object (layered semantics, bboxes may overlap): inside its (padded) box, its own
    cells and the changed cells owned by no other object take the output colour; the rest is background."""
    owner = {}
    for k, pt in enumerate(ps):
        for c in pt.cells: owner[c] = k
    covered = set(); outs = []
    for k, pt in enumerate(ps):
        t = [[bg] * pt.w for _ in range(pt.h)]
        for y in range(max(0, pt.r0), min(H(gi), pt.r0 + pt.h)):
            for x in range(max(0, pt.c0), min(W(gi), pt.c0 + pt.w)):
                o = owner.get((y, x))
                if o == k or (o is None and gi[y][x] != go[y][x]):
                    t[y - pt.r0][x - pt.c0] = go[y][x]
                    covered.add((y, x))
        outs.append(t)
    return outs, covered


def inplace_setup(task, spec):
    """Region semantics (panels, bands, frames, lines): disjoint regions, output part = output crop, cells
    outside every region unchanged.  Object semantics: every changed cell inside some (padded) object box,
    output part = _obj_targets; the lifted program erases the objects and paints the transformed parts."""
    meta = []            # (part, in, out, train index)
    is_obj = spec[0] == 'obj' and spec[4]
    overlap_ok = spec[0] == 'obj' and not spec[4]          # whole bbox crops: overlapping boxes allowed
    whole = True
    for j, p in enumerate(task['train']):
        gi, go = p['input'], p['output']
        ps = partition(spec, gi)
        if not ps: return None
        whole = whole and len(ps) == 1 and ps[0].r0 <= 0 and ps[0].c0 <= 0 and \
            ps[0].r0 + ps[0].h >= H(gi) and ps[0].c0 + ps[0].w >= W(gi)
        bg = _ORIG_BG(gi)
        if is_obj:
            outs, covered = _obj_targets(ps, gi, go, bg)
            for y in range(H(gi)):
                for x in range(W(gi)):
                    if gi[y][x] != go[y][x] and (y, x) not in covered: return None
            for pt, t in zip(ps, outs):
                meta.append((pt, pt.g, t, j))
            continue
        if overlap_ok:
            occ = set()
            for pt in ps:
                occ |= {(y, x) for y in range(max(0, pt.r0), min(H(gi), pt.r0 + pt.h))
                        for x in range(max(0, pt.c0), min(W(gi), pt.c0 + pt.w))}
        else:
            occ = _disjoint(ps, H(gi), W(gi))
            if occ is None: return None
        for y in range(H(gi)):
            ry, oy = gi[y], go[y]
            for x in range(W(gi)):
                if ry[x] != oy[x] and (y, x) not in occ: return None
        for pt in ps:
            meta.append((pt, pt.g, region_crop(go, pt.r0, pt.c0, pt.h, pt.w, bg), j))
    if whole: return None           # one part covering every training grid: that is the whole-grid library's job
    if all(m[1] == m[2] for m in meta): return None
    tparts = []
    for t in task['test']:
        ps = partition(spec, t['input'])
        if not ps or (not is_obj and not overlap_ok and _disjoint(ps, H(t['input']), W(t['input'])) is None): return None
        tparts.append(ps)
    return meta, tparts


def lifted_inplace(spec, chooser):
    """Whole-grid program: partition, transform each part by chooser(part) (an apply fn), put it back
    (regions: pasted; objects: all objects erased, then every transformed part's foreground painted)."""
    is_obj = spec[0] == 'obj' and spec[4]
    overlap_ok = spec[0] == 'obj' and not spec[4]
    def F(g):
        ps = partition(spec, g)
        if not ps: return None
        if not is_obj and not overlap_ok and _disjoint(ps, H(g), W(g)) is None: return None
        canvas = copyg(g); bg = _ORIG_BG(g)
        if is_obj:
            for pt in ps:
                for y, x in pt.cells: canvas[y][x] = bg
        for pt in ps:
            ap = chooser(pt)
            if ap is None: return None
            r = ap(pt.g)
            if r is None or dims(r) != (pt.h, pt.w): return None
            paste(canvas, r, pt.r0, pt.c0, only_fg=bg if is_obj else None)
        return canvas
    return F


def _pad_stable(task, spec, F, ap):
    """The margin of padded object parts must not be an arbitrary choice: if the same program with one more cell
    of margin also reproduces every training pair, both margins are consistent with the evidence and they must
    agree on the test inputs (else the transformed objects may need more room than the margin allows)."""
    wider = spec[:3] + (spec[3] + 1,) + spec[4:]
    G = lifted_inplace(wider, lambda pt: ap)
    try:
        if any(G(p['input']) != p['output'] for p in task['train']): return True     # the margin is determined
        return all(F(t['input']) == G(t['input']) for t in task['test'])
    except _Timeout:
        raise
    except Exception:
        return True


def try_inplace(task, budget_all, emit, stats, specs):
    seen = set(); setups = []; ok_pad0 = set(); calls = 0
    tbg = _task_bg(task)
    n_ok = 0; n_norm = 0
    budget = budget_all.sub(7.0)                 # plain pooled search; the rest is left for GROUP
    for spec in specs:
        if budget.left() < 1.0 or calls >= 6 or n_ok >= 3: break
        if spec[0] == 'obj' and spec[3] and (spec[1], spec[2]) in ok_pad0: continue
        st = inplace_setup(task, spec)
        if st is None: continue
        if spec[0] == 'obj' and not spec[3] and spec[4]: ok_pad0.add((spec[1], spec[2]))
        meta, tparts = st
        key = (_regions_key([m[0] for m in meta]), tuple(_regions_key(ps) for ps in tparts))
        if key in seen: continue
        seen.add(key)
        fbg = tbg if spec[0] in ('obj', 'frame', 'lines', 'bgsep') else None
        tflat = [pt for ps in tparts for pt in ps]
        pairs = [(m[1], m[2]) for m in meta]
        if DEBUG: print(' INPLACE', spec_name(spec), 'parts', len(meta), 'functional', _functional(pairs), flush=True)
        res = []
        if _functional(pairs) and _held_out(meta) is not None:
            calls += 1
            res = lib_search(pairs, [pt.g for pt in tflat], budget, cap=3.0, maxp=4, fbg=fbg, stats=stats)
            res = loo_filter(meta, res, budget, fbg, stats, folds=2, occam=True)
            if not res and n_norm < 2 and budget.left() > 2.0:
                n_norm += 1
                res = norm_search(meta, budget, fbg, tbg, stats, folds=2, occam=True)
            for name, cost, tp, ap in res:
                F = lifted_inplace(spec, lambda pt, ap=ap: ap)
                if spec[0] == 'obj' and spec[3] and not _pad_stable(task, spec, F, ap): continue
                n_ok += 2 * emit(f"lift[{spec_name(spec)}]:{name}", 10 + cost, F)
        if not res and sum(m[1] != m[2] for m in meta) >= 2 and spec[0] != 'blocks':
            setups.append((spec, meta, tparts, fbg))
    gcalls = [0]
    for spec, meta, tparts, fbg in (setups if not n_ok else []):
        if budget_all.left() < 1.5 or gcalls[0] >= 6: break
        try_group(meta, [pt for ps in tparts for pt in ps], lambda ch, spec=spec: lifted_inplace(spec, ch),
                  f"lift[{spec_name(spec)}]", budget_all, fbg, emit, stats, gcalls)


# ------------------------------------------------------------------ SELIN (only the selected part changes)
SELIN_SPECS = [('obj', False, True, 0, True), ('obj', True, False, 0, True), ('sep',), ('frame',), ('bgsep', 'rc'),
               ('obj', True, True, 0, True), ('obj', False, False, 0, True), ('obj', False, True, 1, True),
               ('obj', True, False, 1, True)]


def lifted_selin(spec, sel, ap, isolate=False):
    """The selected part transformed in place; the rest kept (or, isolate, cleared to the background)."""
    def F(g):
        ps = partition(spec, g)
        if not ps: return None
        c = select(ps, sel)
        if c is None: return None
        r = ap(c.g)
        if r is None or dims(r) != (c.h, c.w): return None
        canvas = [[_ORIG_BG(g)] * W(g) for _ in range(H(g))] if isolate else copyg(g)
        paste(canvas, r, c.r0, c.c0)
        return canvas
    return F


def try_selin(task, budget, emit, stats, max_calls=6):
    train, test = task['train'], task['test']
    tin = [p['input'] for p in train]
    tbg = _task_bg(task)
    diffs = [{(y, x) for y in range(H(p['input'])) for x in range(W(p['input'])) if p['input'][y][x] != p['output'][y][x]}
             for p in train]
    sel_cost = SEL_BASE + sum(_cells(g) for g in tin) // 2          # one selector checked on every training grid
    cands = []; seen = set()
    for spec in SELIN_SPECS:
        if budget.left() < 1.0: break
        pin = [partition(spec, g) for g in tin]
        if any(not ps or len(ps) < 2 for ps in pin): continue
        pte = [partition(spec, t['input']) for t in test]
        if any(not ps for ps in pte): continue
        ns = {len(ps) for ps in pin + pte}
        sels = _selectors(tin, ns.pop() if len(ns) == 1 and spec[0] in ('sep', 'bgsep', 'frame') else 0)
        for sel in sels:
            tick(sel_cost)
            ch = [select(ps, sel) for ps in pin]
            if any(c is None for c in ch): continue
            inside = lambda c, y, x: c.r0 <= y < c.r0 + c.h and c.c0 <= x < c.c0 + c.w
            keep = all(inside(c, y, x) for c, d in zip(ch, diffs) for y, x in d)
            isolate = not keep and all(p['output'][y][x] == _ORIG_BG(p['input'])
                                       for c, p in zip(ch, train) for y in range(H(p['output'])) for x in range(W(p['output']))
                                       if not inside(c, y, x))
            if not keep and not isolate: continue
            pairs = [(c.g, region_crop(p['output'], c.r0, c.c0, c.h, c.w, _ORIG_BG(p['input']))) for c, p in zip(ch, train)]
            if keep and sum(i != o for i, o in pairs) < 2: continue
            ct = [select(ps, sel) for ps in pte]
            if any(c is None for c in ct): continue
            key = (str(pairs), tuple(str(c.g) for c in ct), isolate)
            if key in seen: continue
            seen.add(key)
            cands.append((spec, sel, pairs, [c.g for c in ct], isolate))
    n = 0
    for spec, sel, pairs, tg, isolate in cands:
        if budget.left() < 1.0 or n >= max_calls: break
        tag = 'isolate' if isolate else 'keep'
        if all(i == o for i, o in pairs):          # isolate the selected part unchanged
            if emit(f"lift-selin[{spec_name(spec)},{tag}]:select-{sel[0]}-{sel[1]}", 8, lifted_selin(spec, sel, _ident, isolate)):
                return
            continue
        n += 1
        fbg = tbg if spec[0] in ('obj', 'frame', 'bgsep') else None
        meta = [(None, i, o, j) for j, (i, o) in enumerate(pairs)]
        res = lib_search(pairs, tg, budget, cap=2.5, maxp=4, fbg=fbg, stats=stats)
        res = loo_filter(meta, res, budget, fbg, stats, folds=2)
        if not res and budget.left() > 2.0:
            res = norm_search(meta, budget, fbg, tbg, stats, folds=2)
        for name, cost, tp, ap in res:
            if emit(f"lift-selin[{spec_name(spec)},{tag}]:select-{sel[0]}-{sel[1]}>{name}", 14 + cost,
                    lifted_selin(spec, sel, ap, isolate)):
                return


# ------------------------------------------------------------------ LAYERS (one part per colour)
def layers_setup(task):
    meta = []
    for j, p in enumerate(task['train']):
        gi, go = p['input'], p['output']
        bg = _ORIG_BG(gi)
        ps = partition(('layers',), gi)
        if not ps: return None
        cin = colours(gi) - {bg}
        if not colours(go) - {bg} <= cin: return None
        for pt in ps:
            c = pt.a['colour']
            meta.append((pt, pt.g, [[v if v == c else bg for v in row] for row in go], j))
    if all(m[1] == m[2] for m in meta) or max(Counter(m[3] for m in meta).values()) < 2: return None
    tparts = []
    for t in task['test']:
        ps = partition(('layers',), t['input'])
        if not ps: return None
        tparts.append(ps)
    return meta, tparts


def lifted_layers(chooser):
    def F(g):
        bg = _ORIG_BG(g)
        ps = partition(('layers',), g)
        if not ps: return None
        canvas = [[bg] * W(g) for _ in range(H(g))]
        for pt in ps:
            ap = chooser(pt)
            if ap is None: return None
            r = ap(pt.g)
            if r is None or dims(r) != dims(g): return None
            for y in range(H(g)):
                for x in range(W(g)):
                    v = r[y][x]
                    if v != bg:
                        if canvas[y][x] != bg and canvas[y][x] != v: return None
                        canvas[y][x] = v
        return canvas
    return F


def try_layers(task, budget, emit, stats):
    st = layers_setup(task)
    if st is None: return
    meta, tparts = st
    fbg = _task_bg(task)
    tflat = [pt for ps in tparts for pt in ps]
    pairs = [(m[1], m[2]) for m in meta]
    if _functional(pairs) and _held_out(meta) is not None:
        plain = budget.sub(2.5)                   # the rest is left for the per-colour grouping
        res = lib_search(pairs, [pt.g for pt in tflat], plain, cap=2.0, maxp=4, fbg=fbg, stats=stats)
        res = loo_filter(meta, res, budget, fbg, stats, folds=2, occam=True)
        ok = False
        for name, cost, tp, ap in res:
            ok = emit(f"lift[layers]:{name}", 12 + cost, lifted_layers(lambda pt, ap=ap: ap)) or ok
        if ok: return
    if sum(m[1] != m[2] for m in meta) >= 2:
        try_group(meta, tflat, lifted_layers, "lift[layers]", budget, fbg, emit, stats, [0], max_calls=4)


# ------------------------------------------------------------------ OBJMAP (objects change extent in place)
ANCHORS = ('centre', 'tl', 'tr', 'bl', 'br')
DIM_RULES = {
    'T': lambda h, w: (w, h),
    'grow1': lambda h, w: (h + 2, w + 2),
    'shrink1': lambda h, w: (h - 2, w - 2) if h > 2 and w > 2 else None,
    'x2': lambda h, w: (2 * h, 2 * w),
    'x3': lambda h, w: (3 * h, 3 * w),
    'w+1': lambda h, w: (h, w + 1), 'w-1': lambda h, w: (h, w - 1) if w > 1 else None,
    'h+1': lambda h, w: (h + 1, w), 'h-1': lambda h, w: (h - 1, w) if h > 1 else None,
    'w+2': lambda h, w: (h, w + 2), 'w-2': lambda h, w: (h, w - 2) if w > 2 else None,
    'h+2': lambda h, w: (h + 2, w), 'h-2': lambda h, w: (h - 2, w) if h > 2 else None,
}


def _anchor_pt(r0, c0, r1, c1, a):
    if a == 'centre': return (r0 + r1, c0 + c1)
    if a == 'tl': return (r0, c0)
    if a == 'tr': return (r0, c1)
    if a == 'bl': return (r1, c0)
    return (r1, c1)


def _place(pt, h, w, a):
    if a == 'centre':
        sy, sx = pt
        if (sy - h + 1) % 2 or (sx - w + 1) % 2: return None
        return (sy - h + 1) // 2, (sx - w + 1) // 2
    if a == 'tl': return pt
    if a == 'tr': return pt[0], pt[1] - w + 1
    if a == 'bl': return pt[0] - h + 1, pt[1]
    return pt[0] - h + 1, pt[1] - w + 1


def _out_region(pt, anchor, rule):
    d = DIM_RULES[rule](pt.a['bh'], pt.a['bw'])
    if d is None: return None
    pos = _place(_anchor_pt(*bbox(pt.cells), anchor), d[0], d[1], anchor)
    if pos is None: return None
    return pos[0], pos[1], d[0], d[1]


def objmap_setup(task, diag, byc, anchor, rule):
    """Each object is replaced by a part of dims rule(h, w) anchored at the same point; the output is the
    background canvas with every transformed part's foreground painted on it."""
    spec = ('obj', diag, byc, 0, True)
    meta = []; resized = False
    for j, p in enumerate(task['train']):
        gi, go = p['input'], p['output']
        bg = _ORIG_BG(gi)
        ps = partition(spec, gi)
        if not ps or len(ps) < 2: return None
        canvas = copyg(gi)
        for pt in ps:
            for y, x in pt.cells: canvas[y][x] = bg
        regs = []
        for pt in ps:
            reg = _out_region(pt, anchor, rule)
            if reg is None: return None
            regs.append(Part(reg[0], reg[1], reg[2], reg[3], None))
            if (reg[2], reg[3]) != (pt.h, pt.w): resized = True
            if reg[2] * reg[3] * 2 > H(go) * W(go): return None          # a part as large as the scene: not per-object
            op = region_crop(go, reg[0], reg[1], reg[2], reg[3], bg)
            paste(canvas, op, reg[0], reg[1], only_fg=bg)
            meta.append((pt, pt.g, op, j))
        if canvas != go: return None
        if _disjoint(regs, H(go), W(go)) is None: return None      # every output cell explained by one part
    if not resized: return None                                     # same extents: INPLACE covers it
    tparts = []
    for t in task['test']:
        ps = partition(spec, t['input'])
        if not ps: return None
        tparts.append(ps)
    return meta, tparts


def lifted_objmap(diag, byc, anchor, rule, chooser):
    spec = ('obj', diag, byc, 0, True)
    def F(g):
        bg = _ORIG_BG(g)
        ps = partition(spec, g)
        if not ps: return None
        canvas = copyg(g)
        for pt in ps:
            for y, x in pt.cells: canvas[y][x] = bg
        for pt in ps:
            ap = chooser(pt)
            if ap is None: return None
            r = ap(pt.g)
            if r is None: return None
            reg = _out_region(pt, anchor, rule)
            if reg is None or (reg[2], reg[3]) != dims(r): return None
            paste(canvas, r, reg[0], reg[1], only_fg=bg)
        return canvas
    return F


def try_objmap(task, budget, emit, stats):
    fbg = _task_bg(task)
    seen = set(); setups = []; calls = 0
    for diag, byc in ((False, True), (True, False), (True, True), (False, False)):
        for rule in DIM_RULES:
            for anchor in ANCHORS:
                if budget.left() < 1.0: return
                st = objmap_setup(task, diag, byc, anchor, rule)
                if st is None: continue
                meta, tparts = st
                key = str([(m[1], m[2]) for m in meta]) + str([[pt.g for pt in ps] for ps in tparts])
                if key in seen: continue
                seen.add(key)
                tflat = [pt for ps in tparts for pt in ps]
                pairs = [(m[1], m[2]) for m in meta]
                pre = f"lift-objmap[{spec_name(('obj', diag, byc, 0, True))},{anchor},{rule}]"
                res = []
                if _functional(pairs) and _held_out(meta) is not None and calls < 4:
                    calls += 1
                    res = lib_search(pairs, [pt.g for pt in tflat], budget, cap=3.0, maxp=4, fbg=fbg, stats=stats)
                    res = loo_filter(meta, res, budget, fbg, stats, folds=2, occam=True)
                for name, cost, tp, ap in res:
                    emit(f"{pre}:{name}", 12 + cost, lifted_objmap(diag, byc, anchor, rule, lambda pt, ap=ap: ap))
                if not res and sum(m[1] != m[2] for m in meta) >= 2:
                    setups.append((diag, byc, anchor, rule, meta, tflat, pre))
    gcalls = [0]
    for diag, byc, anchor, rule, meta, tflat, pre in setups:
        if budget.left() < 1.5 or gcalls[0] >= 4: break
        try_group(meta, tflat, lambda ch, d=diag, b=byc, a=anchor, r=rule: lifted_objmap(d, b, a, r, ch), pre, budget, fbg,
                  emit, stats, gcalls, max_calls=4)


# ------------------------------------------------------------------ SELECT
SELECT_SPECS = [('sep',), ('obj', False, True, 0, True), ('obj', True, True, 0, True), ('frame',),
                ('obj', True, False, 0, True), ('obj', False, False, 0, True), ('obj', True, False, 0, False),
                ('obj', False, True, 0, False), ('bgsep', 'rc'), ('bgsep', 'r'), ('bgsep', 'c')]


def _selectors(train_inputs, fixed_n):
    common = None
    for g in train_inputs:
        cs = colours(g) - {_ORIG_BG(g)}
        common = cs if common is None else common & cs
    sels = []
    for an in ('size', 'area', 'ncol', 'bh', 'bw', 'density'):
        sels.append(('max', an)); sels.append(('min', an))
    sels += [('min', 'top'), ('min', 'left'), ('max', 'bottom'), ('max', 'right')]
    for an in ('colours', 'shape', 'content', 'size', 'ncol', 'sym', 'colour', 'area', 'fcol', 'solid', 'hole', 'orient'):
        sels.append(('unique', an))
    for c in sorted(common or ()):
        sels += [('has', c), ('lacks', c), ('maxcount', c), ('touch', c)]
    if fixed_n:
        sels += [('index', k) for k in range(fixed_n)]
    return sels


def select(ps, sel):
    kind, an = sel
    if kind == 'index':
        return ps[an] if len(ps) > an else None
    if kind in ('max', 'min'):
        vals = [p.a.get(an) for p in ps]
        if any(v is None for v in vals): return None
        best = max(vals) if kind == 'max' else min(vals)
        hits = [p for p, v in zip(ps, vals) if v == best]
    elif kind == 'unique':
        if len(ps) < 3: return None
        vals = [p.a.get(an) for p in ps]
        if any(v is None for v in vals): return None
        cnt = Counter(vals)
        hits = [p for p, v in zip(ps, vals) if cnt[v] == 1]
    elif kind in ('has', 'lacks'):
        hits = [p for p in ps if (any(v == an for r in p.g for v in r)) == (kind == 'has')]
    elif kind == 'touch':                         # the one object touched by a cell of colour c
        if any('adj' not in p.a for p in ps): return None
        hits = [p for p in ps if an in p.a['adj']]
    elif kind == 'maxcount':
        vals = [sum(v == an for r in p.g for v in r) for p in ps]
        best = max(vals)
        if best == 0: return None
        hits = [p for p, v in zip(ps, vals) if v == best]
    else:
        return None
    return hits[0] if len(hits) == 1 else None


def _dim_rel(pairs):
    """Strength of the size relation part -> output (lower is stronger); None when no relation holds."""
    rels = None
    for i, o in pairs:
        (h, w), (oh, ow) = dims(i), dims(o)
        r = set()
        if (h, w) == (oh, ow): r.add('eq')
        if (h, w) == (ow, oh): r.add('T')
        if oh % h == 0 and ow % w == 0: r.add(('up', oh // h, ow // w))
        if h % oh == 0 and w % ow == 0: r.add(('down', h // oh, w // ow))
        r.add(('const', oh, ow))
        if oh <= h and ow <= w: r.add('le')
        rels = r if rels is None else rels & r
    if not rels: return None
    if 'eq' in rels or 'T' in rels: return 0
    if any(isinstance(x, tuple) and x[0] in ('up', 'down') for x in rels): return 1
    if 'le' in rels: return 2
    return 3


def lifted_select(spec, sel, ap):
    def F(g):
        ps = partition(spec, g)
        if not ps: return None
        c = select(ps, sel)
        return ap(c.g) if c is not None else None
    return F


def try_select(task, budget, emit, stats):
    train, test = task['train'], task['test']
    tin = [p['input'] for p in train]
    tbg = _task_bg(task)
    sel_cost = SEL_BASE + sum(_cells(g) for g in tin) // 2          # one selector checked on every training grid
    cands = []; seen = set()
    for spec in SELECT_SPECS + _block_specs(tin + [t['input'] for t in test], limit=2):
        if budget.left() < 1.0: break
        pin = [partition(spec, g) for g in tin]
        if any(not ps for ps in pin): continue
        pte = [partition(spec, t['input']) for t in test]
        if any(not ps for ps in pte): continue
        ns = {len(ps) for ps in pin + pte}
        sels = _selectors(tin, ns.pop() if len(ns) == 1 and spec[0] in ('sep', 'bgsep', 'frame', 'blocks') else 0)
        for sel in sels:
            tick(sel_cost)
            ch = [select(ps, sel) for ps in pin]
            if any(c is None for c in ch): continue
            if all((c.h, c.w) == dims(g) for c, g in zip(ch, tin)): continue     # not a selection: whole grid
            ct = [select(ps, sel) for ps in pte]
            if any(c is None for c in ct): continue
            key = (tuple(str(c.g) for c in ch), tuple(str(c.g) for c in ct))
            if key in seen: continue
            seen.add(key)
            pairs = [(c.g, p['output']) for c, p in zip(ch, train)]
            rel = _dim_rel(pairs)
            if rel is None: continue
            cands.append((rel, len(cands), spec, sel, pairs, [c.g for c in ct]))
    cands.sort(key=lambda x: (x[0], x[1]))
    n_id = 0
    for rel, _, spec, sel, pairs, tg in cands:          # the selected part is the output itself
        if all(i == o for i, o in pairs):
            n_id += emit(f"lift[{spec_name(spec)}]:select-{sel[0]}-{sel[1]}", 5, lifted_select(spec, sel, _ident))
    if n_id: return
    # hypotheses selecting the same training parts share one library search (they differ only at test)
    groups = {}
    for rel, idx, spec, sel, pairs, tg in cands:
        if all(i == o for i, o in pairs): continue
        fbg = tbg if spec[0] in ('obj', 'frame') else None
        k = (str(pairs), fbg)
        if k not in groups: groups[k] = (rel, idx, pairs, fbg, [])
        groups[k][4].append((spec, sel))
    n = 0
    for rel, _, pairs, fbg, members in sorted(groups.values(), key=lambda g: g[:2]):
        if budget.left() < 1.0 or n >= 8: break
        n += 1
        meta = [(None, i, o, j) for j, (i, o) in enumerate(pairs)]
        res = lib_search(pairs, [], budget, cap=2.5, maxp=6, fbg=fbg, stats=stats)
        res = loo_filter(meta, res, budget, fbg, stats, folds=2)
        semantic = any(s[0] in ('obj', 'frame', 'sep') for s, _ in members)
        if not res and semantic and budget.left() > 2.0:
            res = norm_search(meta, budget, fbg, tbg, stats, folds=2)
        for name, cost, tp, ap in res:
            for spec, sel in members:
                if name.endswith('@norm') and spec[0] not in ('obj', 'frame', 'sep'): continue
                emit(f"lift[{spec_name(spec)}]:select-{sel[0]}-{sel[1]}>{name}", 12 + cost, lifted_select(spec, sel, ap))


# ------------------------------------------------------------------ PANELMAP
def in_layout(kind, g):
    """Input panel lattice: (panels in reading order, R, C, separator colour) from separator lines ('sep') or
    from all-background rows / columns when every lattice cell holds something ('bgsep')."""
    if kind == 'sep':
        lay = sep_layout(g)
        if lay is None: return None
        sc, ri, ci = lay
    else:
        h, w = H(g), W(g); sc = _ORIG_BG(g)
        ri = _intervals([r for r in range(h) if all(v == sc for v in g[r])], h)
        ci = _intervals([c for c in range(w) if all(g[r][c] == sc for r in range(h))], w)
        if len(ri) * len(ci) < 2: return None
    ps = [[row[b:b1 + 1] for row in g[a:a1 + 1]] for a, a1 in ri for b, b1 in ci]
    if kind == 'bgsep' and any(all(v == sc for r in p for v in r) for p in ps): return None
    return ps, len(ri), len(ci), sc


def _out_layout(train, kind):
    """Output layout rule, constant over pairs: the input's R x C panel grid with the output's own separators
    ('sep', colour 'same'|c, row sep, col sep, top, bottom, left, right margins); or R x C equal blocks
    ('blocks',); or C x R equal blocks in reading order ('blocksT',)."""
    struct = set(); same = True; outc = set(); blocks = blocksT = True
    for p in train:
        li, lo = in_layout(kind, p['input']), sep_layout(p['output'])
        if li is None: return None
        _, R, C, sci = li
        go = p['output']
        if H(go) % R or W(go) % C: blocks = False
        if H(go) % C or W(go) % R or R == C: blocksT = False
        if lo is None:
            struct.add(None); continue
        sco, rio, cio = lo
        if (R, C) != (len(rio), len(cio)):
            struct.add(None); continue
        gaps_r = {b[0] - a[1] - 1 for a, b in zip(rio, rio[1:])} or {0}
        gaps_c = {b[0] - a[1] - 1 for a, b in zip(cio, cio[1:])} or {0}
        if len(gaps_r) != 1 or len(gaps_c) != 1:
            struct.add(None); continue
        struct.add((gaps_r.pop(), gaps_c.pop(), rio[0][0], H(go) - 1 - rio[-1][1], cio[0][0], W(go) - 1 - cio[-1][1]))
        same = same and sco == sci
        outc.add(sco)
    rules = []
    if len(struct) == 1 and None not in struct:
        if same: rules.append(('sep', 'same') + next(iter(struct)))
        elif len(outc) == 1: rules.append(('sep', next(iter(outc))) + next(iter(struct)))
    if all(sep_layout(p['output']) is None for p in train):
        if blocks: rules.append(('blocks',))
        if blocksT: rules.append(('blocksT',))
    return rules


def _out_panels(rule, R, C, g_out):
    if rule[0] == 'sep':
        ps = part_sep(g_out)
        return [p.g for p in ps] if ps else None
    nr, nc = (R, C) if rule[0] == 'blocks' else (C, R)
    ps = part_blocks(g_out, nr, nc)
    return [p.g for p in ps] if ps else None


def _build_panels(panels, nr, nc, sc, geom):
    tr_, tc, mt, mb, ml, mr = geom
    rows = [panels[a * nc:(a + 1) * nc] for a in range(nr)]
    hs = [{H(p) for p in row} for row in rows]
    ws = [{W(rows[a][b]) for a in range(nr)} for b in range(nc)]
    if any(len(s) != 1 for s in hs + ws): return None
    hs = [s.pop() for s in hs]; ws = [s.pop() for s in ws]
    Ht = mt + mb + sum(hs) + tr_ * (nr - 1); Wt = ml + mr + sum(ws) + tc * (nc - 1)
    if Ht > 30 or Wt > 30: return None
    out = [[sc] * Wt for _ in range(Ht)]
    y = mt
    for a in range(nr):
        x = ml
        for b in range(nc):
            paste(out, rows[a][b], y, x); x += ws[b] + tc
        y += hs[a] + tr_
    return out


def lifted_panelmap(kind, rule, ap):
    def F(g):
        lay = in_layout(kind, g)
        if lay is None: return None
        ps, R, C, sci = lay
        outs = [ap(p) for p in ps]
        if any(o is None for o in outs): return None
        if rule[0] == 'blocks':
            return _build_panels(outs, R, C, 0, (0, 0, 0, 0, 0, 0))
        if rule[0] == 'blocksT':
            return _build_panels(outs, C, R, 0, (0, 0, 0, 0, 0, 0))
        return _build_panels(outs, R, C, sci if rule[1] == 'same' else rule[1], rule[2:])
    return F


def try_panelmap(task, budget, emit, stats):
    train = task['train']
    for kind in ('sep', 'bgsep'):
        for rule in _out_layout(train, kind) or ():
            if budget.left() < 1.5: return
            meta = []; ok = True
            for j, p in enumerate(train):
                ps, R, C, _ = in_layout(kind, p['input'])
                po = _out_panels(rule, R, C, p['output'])
                if not po or len(po) != len(ps): ok = False; break
                for a, b in zip(ps, po):
                    meta.append((None, a, b, j))
            if not ok: continue
            tl = [in_layout(kind, t['input']) for t in task['test']]
            if any(x is None for x in tl): continue
            pairs = [(m[1], m[2]) for m in meta]
            if not _functional(pairs) or _held_out(meta) is None: continue
            res = lib_search(pairs, [p for x in tl for p in x[0]], budget, cap=3.0, maxp=4, stats=stats)
            res = loo_filter(meta, res, budget, None, stats, folds=2, occam=True)
            for name, cost, tp, ap in res:
                emit(f"lift-panelmap[{kind}->{rule[0]}]:{name}", 12 + cost, lifted_panelmap(kind, rule, ap))


# ------------------------------------------------------------------ SEARCH
def SEARCH(task):
    _WORK[0] = 0; del _CAPS[:]                   # every task starts from zero work: independent of earlier tasks
    budget = Budget(TOTAL_BUDGET)
    DIAG.clear()
    found = []; seen = {}; stats = []
    train, test = task['train'], task['test']

    def emit(name, cost, F):
        """Verify the whole lifted program on every training pair, then predict the test inputs."""
        try:
            for p in train:
                if F(p['input']) != p['output']: return False
            preds = [F(t['input']) for t in test]
        except _Timeout:
            raise
        except Exception:
            return False
        if any(p is None for p in preds): return False
        k = str(preds)
        if k in seen:
            seen[k]['votes'] += 1            # another hypothesis agrees on the test prediction
            return 0.5
        seen[k] = {'program': name, 'cost': cost, 'preds': preds, 'votes': 1, 't': round((_WORK[0] - budget.t0) / SEC, 2)}
        found.append(seen[k])
        return True

    patched = _install_patch()
    try:
        with work_cap(TOTAL_BUDGET + 0.5):
            same = all(dims(p['input']) == dims(p['output']) for p in train)
            if same:
                try_inplace(task, budget.sub(11.0), emit, stats, INPLACE_SPECS)
                if not found:
                    try_layers(task, budget.sub(5.0), emit, stats)
                if not found:
                    try_objmap(task, budget.sub(3.0), emit, stats)
                if not found:
                    try_selin(task, budget.sub(3.0), emit, stats)
            else:
                try_panelmap(task, budget.sub(4.0), emit, stats)
                try_select(task, budget, emit, stats)
    except _Timeout:
        pass
    finally:
        _remove_patch(patched)
        _FORCED[0] = None
        del _CAPS[:]
    found.sort(key=lambda r: (-r['votes'], r['cost']))
    SEARCH.stats = stats + [('first', min((r['t'] for r in found), default=None)), ('diag', dict(DIAG)),
                            ('work', round(_WORK[0] / SEC, 3))]
    return [{'program': r['program'] + (f" (x{r['votes']})" if r['votes'] > 1 else ''), 'cost': r['cost'], 'preds': r['preds']}
            for r in found[:3]]


In [ ]:
%%writefile /kaggle/working/probe/compose_objmap.py
"""compose_objmap: object correspondence + per-object action learner (composition engine, Dream lane 3).

SEARCH(task) -> [{"program", "preds"}] (best first, <= 3); every program reproduces all training pairs.

Pipeline (per task; per background mode (grid / task-level) x segmentation c4 / c8 / m8 / m4 / colour-groups):
 1. Segment every training input into objects (Obj: cells, colours, bbox, D8 images of its patch).
 2. Correspondence (least action).  For each input object find how it appears in the output: in-place keep /
    uniform recolour / colour map / delete, template matches of every D8 image of its coloured patch (exact or
    uniformly recoloured), or cells it creates.  A placement is only usable through an ACTION DESCRIPTOR that
    predicts it from the input alone:
      keep | delete | (vector-expr, D8 transform, colour-expr, keep-original) | add(creation, colour-expr)
      vector-expr : const(dy,dx) | border d | own-size d x m | toward anchor until contact | onto anchor
                    | snap: least displacement touching an anchor / colour | align row/col with an anchor
                    (start/centre/end) | mirror across an anchor | into the matching hole (slot) of another object
                    | slide d / toward an anchor / toward the nearest edge / where the object's marker points,
                      until at rest (any contact / must hit / must hit colour c); sequential per direction group,
                      local test = the placement is at rest IN THE OUTPUT
                    anchors: nearest, nearest of colour c, largest, nearest / largest of own colour, same shape
      colour-expr : same | const c | colour of the touching / containing / contained / nearest / same-shape /
                    largest / row-marker / col-marker / only-other-coloured object | minority / majority own
                    colour | colour map / swap | lookup table over a feature (shape, size, holes, ranks, the key
                    object's shape ...) | colour of the (interior) reference object with the same holes / size /
                    shape / h / w (legend lookup)
      creation    : rays (8 dirs / groups, stop or through) | ray toward an anchor or from the marker cell |
                    outline 4/8 | fill holes | fill bbox
    Each object gets the SET of consistent descriptors with their gains (output cells made correct).
 3. The action is learned as a function of object attributes: greedy decision lists over bitsets (predicates:
    colour, colour-frequency rank, has colour, size / size rank, h, w, shape class, holes, rect / line / single,
    border, unique colour / shape, same-shape count, touches colour c, inside colour c, contains colour c,
    row/col aligned with colour c, same-shape twin, has a legend reference, extreme / rank position, minority-
    colour count, and negations); rules chosen by coverage, then gain, then description length, with
    branching on the first rules; predicate and descriptor alternatives with the same training support are
    tried as variants (they may differ on the test input; the top-2 are returned).
 4. Render (erase movers, in-place recolours, static moves, sliding groups, creations; paint over / under) and
    verify on ALL training pairs.  MDL acceptance: every rule fires in >= 2 pairs or on >= 2 objects (lists of
    4+ rules: >= 2 pairs each); lookup tables need support per entry.  Novelty guard: a test object of an unseen
    colour may not fall into a colour-decided rule learned from a single colour.  Segmentations whose test
    objects fall outside the training size range are penalised.
 5. Composition with the library (only when no single object map fits):
      FIRST: output-independent programs of the original G-DSL families (ranked by agreement with the outputs)
             -> object map (or -> whole-grid D8 for non-reshaper steps);
      LAST : near-miss object map (renders every pair, only makes correct changes, >= 10% of them) -> a second
             object map (<= 3 rules in all) or the library search (lib_search: gdsl.search over the families
             with a deterministic work bound) on the residual pairs (rendered, output).
Conservation laws (per-colour cell counts) pre-classify the pairs (rearrangement / recolour / delete / create)
and prune the descriptor space.  Whole-grid D8 / global colour-map tasks are left to the library.
No task ids or literal task constants: every colour, vector, predicate value and table entry is induced from
the training pairs.

DETERMINISM (Kaggle parity): the engine has no clock and no signal.  Every search is bounded by a WORK budget
(class Budget): the natural units of work -- segmentation contexts, placement scans, (object x transform x
vocabulary expression) tests and placement hits, slide steps, ray / creation cells, snap and anchor scans,
feature / predicate tables, decision-list nodes, rule-list variants and their renders, library programs
generated and applied to a grid -- are charged to an integer counter, with weights calibrated (NNLS on the
1050-task harness) so that WU_S = 1e8 work units ~ 1 s of CPU on the reference machine; the old wall-clock
limits are kept as WU_S x seconds (15 s per task, 2 s / 3 s per first / last-step learner, gates 2-7 s).
Set iteration never decides an order (hash-seed independent).  Predictions are therefore identical on any
machine, under any load.  (The wall-clock original is compose_objmap_timed.py.)
"""
import sys
from collections import Counter
sys.path.insert(0, '/home/claude/work/widen')
import gdsl

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))
DIR8 = {'N': (-1, 0), 'S': (1, 0), 'W': (0, -1), 'E': (0, 1), 'NW': (-1, -1), 'NE': (-1, 1), 'SW': (1, -1), 'SE': (1, 1)}
DIR4 = ('N', 'S', 'W', 'E')
TXS = ('id', 'fh', 'fv', 'r180', 'T', 'r90', 'r270', 'aT')
SEGS = ('c4', 'c8', 'm8', 'm4', 'col')
KEEP = ('keep',)
DEL = ('del',)
ZERO = ('const', 0, 0)
DIAG = None


# ============================================================ deterministic work budget

class Budget:
    """Deterministic work budget: integer work units charged by the engine (no clock).  A sub-budget also
    charges its parent and never has more left than its parent."""
    __slots__ = ('cap', 'used', 'parent')

    def __init__(self, units, parent=None):
        self.cap = int(units); self.used = 0; self.parent = parent

    def spend(self, n):
        b = self
        while b is not None:
            b.used += n; b = b.parent

    def left(self):
        v = self.cap - self.used
        return v if self.parent is None else min(v, self.parent.left())

    def over(self): return self.left() <= 0


NOBUDGET = Budget(1 << 62)

# Work-unit weights (integers): units charged per natural unit of work.  Calibrated by non-negative least squares
# on the 1050-task harness (CPU time per learner call / library phase on the reference machine) so that WU_S units
# ~ 1 s of CPU (1 unit ~ 10 ns).  Integer arithmetic only: the budget is exact on every platform.
WU_S = 100_000_000
W_CALL, W_CALL_A = 0, 81                               # a learner call: base, per training input cell
W_CTX, W_CTX_AREA, W_CTX_OBJ = 0, 102, 1549            # segmentation contexts (per grid: cells, objects)
W_HOLES_A, W_TOWARD, W_CR, W_CR_STEP = 223, 27, 576, 16 # holes (bbox cells), toward (steps x cells), creations
W_SNAP, W_SNAP_RING, W_SNAP_PAIR = 1800, 18, 42        # snap: base, anchor cells x nbrs, ring cells x patch cells
W_ANCH, W_CX, W_CX_N = 459, 279, 19                    # anchors (per object), relative colour (base, per object)
W_SEG, W_SEG_OBJ = 0, 1021                             # a new segmentation: signature, output index, descriptors
W_NM, W_NM_TX = 1021, 1189                             # move / transform pre-scan: per object, per D8 image
W_OD, W_OD_CELL, W_ADD, W_ADD_S = 0, 1923, 577, 59     # object_descs: per object, per cell, per creation kind, per cell
W_REL, W_HIT, W_HIT_CELL, W_REST = 1343, 628, 0, 35    # relative-colour scan; placement hit (+ cells); rest test cells
W_VFILT, W_MONO_OCC = 2, 75                            # vocabulary filtered for unchanged objects; recoloured-copy scan
W_TX, W_PL, W_PL_OCC = 5755, 0, 153                    # per D8 image, per placement scan, per scanned output cell
W_VX, W_VX_CELL, W_SLIDE_K = 66, 4, 44                 # per vocabulary expression, x patch cells, per slide step
W_FEAT, W_FEAT2, W_PM = 2534, 418, 70                  # object features (n, n^2), predicate table (preds x objects)
W_LL, W_LL_D, W_LL_P = 0, 751, 0                       # decision-list setup (descriptors, predicates)
W_NODE, W_NODE_PRED, W_NODE_MASK, W_NODE_CAND, W_NODE_CM = 0, 0, 140, 162, 8 # per decision-list search node
W_STRUCT, W_STRUCT_P = 0, 0                            # per rule list: variant construction
W_VAR, W_VAR_F, W_PAINT = 0, 17, 1568                  # per variant: MDL test (objects x rules); per paint mode
W_REND, W_REND_A, W_REND_O = 546, 9, 410               # per render: base, cells, objects
W_NEAR_A = 3                                           # near-miss test, per output cell
W_FS_GEN, W_FS_PROG = 789962, 0                        # first steps: generation, per program generated
W_RUN, W_RUN_A = 3112, 123                             # a library program applied to one grid: base, per cell
W_LIB_GEN, W_LIB_GEN_A, W_LIB_PROG = 60738876, 115488, 0 # lib_candidates on residual pairs: base, per cell, per program
LIB_MAX_PROGS = 3000                                   # at most this many library programs tried on a residual
# residual last step: library families whose generation has no deterministic internal bound are not searched
# (fam_complete_partial lost its 2 s wall-clock cap for Kaggle parity and can take > 10 s on one residual)
LIB_SKIP = ('fam_complete_partial',)


def tx_cell(t, y, x, h, w):
    if t == 'id': return y, x
    if t == 'fh': return y, w - 1 - x
    if t == 'fv': return h - 1 - y, x
    if t == 'r180': return h - 1 - y, w - 1 - x
    if t == 'T': return x, y
    if t == 'r90': return x, h - 1 - y
    if t == 'r270': return w - 1 - x, y
    return w - 1 - x, h - 1 - y          # aT


def segment(g, bg, seg):
    if seg == 'col':
        d = {}
        for y, row in enumerate(g):
            for x, v in enumerate(row):
                if v != bg: d.setdefault(v, []).append((y, x))
        return [d[c] for c in sorted(d)]
    return gdsl.objects(g, bg, seg[1] == '8', seg[0] == 'c')


# ============================================================ objects and context

class Obj:
    __slots__ = ('i', 'cells', 'cset', 'cols', 'r0', 'c0', 'r1', 'c1', 'h', 'w', 'size', 'colour', 'colours',
                 'rel', 'shape', '_tx', 'ccount')

    def __init__(self, i, cells, g):
        cells = sorted(cells)
        self.i = i; self.cells = cells; self.cset = frozenset(cells)
        self.cols = [g[y][x] for y, x in cells]
        ys = [y for y, _ in cells]; xs = [x for _, x in cells]
        self.r0, self.c0, self.r1, self.c1 = min(ys), min(xs), max(ys), max(xs)
        self.h = self.r1 - self.r0 + 1; self.w = self.c1 - self.c0 + 1
        self.size = len(cells)
        cnt = Counter(self.cols); self.ccount = cnt
        self.colour = min(cnt, key=lambda c: (-cnt[c], c))
        self.colours = frozenset(cnt)
        self.rel = tuple((y - self.r0, x - self.c0, c) for (y, x), c in zip(cells, self.cols))
        self.shape = frozenset((y, x) for y, x, _ in self.rel)
        self._tx = {}

    def patch(self, t):
        """D8 image of the coloured patch: (cells sorted (dy,dx,c), nh, nw, in-place top-left ay, ax)."""
        r = self._tx.get(t)
        if r is None:
            h, w = self.h, self.w
            cells = tuple(sorted(tx_cell(t, y, x, h, w) + (c,) for y, x, c in self.rel))
            nh, nw = (h, w) if t in ('id', 'fh', 'fv', 'r180') else (w, h)
            r = (cells, nh, nw, self.r0 + (h - nh) // 2, self.c0 + (w - nw) // 2)
            self._tx[t] = r
        return r


def canon_shape(shape):
    """D8-canonical key of a cell shape."""
    ys = [y for y, _ in shape]; xs = [x for _, x in shape]
    h, w = max(ys) + 1, max(xs) + 1
    best = None
    for t in TXS:
        k = tuple(sorted(tx_cell(t, y, x, h, w) for y, x in shape))
        if best is None or k < best: best = k
    return best


class Ctx:
    def __init__(self, g, bg, seg, budget=NOBUDGET):
        self.g = g; self.h = len(g); self.w = len(g[0]); self.bg = bg; self.seg = seg
        self.objs = [Obj(i, c, g) for i, c in enumerate(segment(g, bg, seg))]
        self.owner = [[-1] * self.w for _ in range(self.h)]
        for o in self.objs:
            for y, x in o.cells: self.owner[y][x] = o.i
        self._c = {}
        self.budget = budget        # the relation / creation computations below charge their work here
        budget.spend(W_CTX_AREA * self.h * self.w + W_CTX_OBJ * len(self.objs))

    # ---------------------------------------------------------------- relations (lazy)
    def holes(self, o):
        """(hole cells, number of hole components): 4-conn regions of non-o cells inside o's bbox that are
        not connected to the outside."""
        k = ('holes', o.i)
        if k in self._c: return self._c[k]
        self.budget.spend(W_HOLES_A * (o.h + 2) * (o.w + 2))
        r0, c0, r1, c1 = o.r0 - 1, o.c0 - 1, o.r1 + 1, o.c1 + 1
        seen = {(r0, c0)}; st = [(r0, c0)]
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                q = (y + dy, x + dx)
                if r0 <= q[0] <= r1 and c0 <= q[1] <= c1 and q not in seen and q not in o.cset:
                    seen.add(q); st.append(q)
        hole = set()
        for y in range(o.r0, o.r1 + 1):
            for x in range(o.c0, o.c1 + 1):
                if (y, x) not in seen and (y, x) not in o.cset: hole.add((y, x))
        n = 0; rem = set(hole)
        while rem:
            n += 1; st = [rem.pop()]
            while st:
                y, x = st.pop()
                for dy, dx in N4:
                    q = (y + dy, x + dx)
                    if q in rem: rem.discard(q); st.append(q)
        r = (hole, n)
        self._c[k] = r
        return r

    def touching(self, o, diag=True):
        k = ('touch', o.i, diag)
        if k in self._c: return self._c[k]
        res = set()
        for y, x in o.cells:
            for dy, dx in (N8 if diag else N4):
                yy, xx = y + dy, x + dx
                if 0 <= yy < self.h and 0 <= xx < self.w:
                    j = self.owner[yy][xx]
                    if j >= 0 and j != o.i: res.add(j)
        self._c[k] = res
        return res

    def containers(self):
        """dict obj index -> index of the smallest object whose holes contain it."""
        if 'cont' in self._c: return self._c['cont']
        res = {}
        for a in sorted(self.objs, key=lambda a: -a.size):
            if a.h < 3 or a.w < 3: continue
            hole, n = self.holes(a)
            if not n: continue
            for b in self.objs:
                if b.i != a.i and b.cells[0] in hole: res[b.i] = a.i
        self._c['cont'] = res
        return res

    def inner_colours(self, o):
        hole, n = self.holes(o)
        return frozenset(self.g[y][x] for y, x in hole if self.g[y][x] != self.bg)

    def dist(self, a, b):
        gy = max(0, b.r0 - a.r1, a.r0 - b.r1); gx = max(0, b.c0 - a.c1, a.c0 - b.c1)
        return (max(gy, gx), gy + gx)

    def anchors(self, o):
        """dict anchor-name -> Obj."""
        k = ('anch', o.i)
        if k in self._c: return self._c[k]
        self.budget.spend(W_ANCH * len(self.objs))
        res = {}
        others = [b for b in self.objs if b.i != o.i]
        if others:
            ds = [(self.dist(o, b), b) for b in others]
            m = min(d for d, _ in ds); hits = [b for d, b in ds if d == m]
            if len(hits) == 1: res['near'] = hits[0]
            bycol = {}
            for d, b in ds: bycol.setdefault(b.colour, []).append((d, b))
            for c, lst in bycol.items():
                m = min(d for d, _ in lst); hits = [b for d, b in lst if d == m]
                if len(hits) == 1:
                    res[('nc', c)] = hits[0]
                    if c == o.colour: res['samec'] = hits[0]
                if c == o.colour:
                    mx = max(b.size for _, b in lst); hits = [b for _, b in lst if b.size == mx]
                    if len(hits) == 1 and hits[0].size > o.size: res['bigsame'] = hits[0]
            mx = max(b.size for b in others); hits = [b for b in others if b.size == mx]
            if len(hits) == 1 and hits[0].size > o.size: res['large'] = hits[0]
            ck = self.canon(o); hits = [b for b in others if self.canon(b) == ck]
            if len(hits) == 1: res['twin'] = hits[0]
        self._c[k] = res
        return res

    def hole_slots(self):
        """Background hole components of every object: list of (shape (normalised cell set), top-left, owner)."""
        if 'slots' in self._c: return self._c['slots']
        res = []
        for a in self.objs:
            if a.h < 3 or a.w < 3: continue
            hole, n = self.holes(a)
            rem = {c for c in hole if self.g[c[0]][c[1]] == self.bg}
            while rem:
                st = [rem.pop()]; comp = [st[0]]
                while st:
                    y, x = st.pop()
                    for dy, dx in N4:
                        q = (y + dy, x + dx)
                        if q in rem: rem.discard(q); st.append(q); comp.append(q)
                r0 = min(y for y, _ in comp); c0 = min(x for _, x in comp)
                res.append((frozenset((y - r0, x - c0) for y, x in comp), (r0, c0), a.i))
        self._c['slots'] = res
        return res

    def feat(self, o):
        k = ('feat', o.i)
        if k not in self._c: self._c[k] = obj_features(self, o)
        return self._c[k]

    def bfeat(self, o):
        """Intrinsic features (no relations), used to match objects with reference objects."""
        k = ('bfeat', o.i)
        if k not in self._c:
            self._c[k] = {'holes': self.holes(o)[1], 'size': o.size, 'shapex': o.shape, 'h': o.h, 'w': o.w,
                          'shape': self.canon(o),
                          'border': o.r0 == 0 or o.c0 == 0 or o.r1 == self.h - 1 or o.c1 == self.w - 1}
        return self._c[k]

    def canon(self, o):
        k = ('canon', o.i)
        if k not in self._c: self._c[k] = canon_shape(o.shape)
        return self._c[k]

    # ---------------------------------------------------------------- relative colour expressions
    def colour_expr(self, o, name):
        k = ('cx', o.i, name)
        if k in self._c: return self._c[k]
        self.budget.spend(W_CX + W_CX_N * len(self.objs))
        v = None
        if name in ('touch', 'touch4'):
            cs = {self.objs[j].colour for j in self.touching(o, name == 'touch')} - {o.colour}
            if len(cs) == 1: v = cs.pop()
        elif name == 'container':
            j = self.containers().get(o.i)
            if j is not None: v = self.objs[j].colour
        elif name == 'inner':
            cs = self.inner_colours(o) - {o.colour}
            if len(cs) == 1: v = next(iter(cs))
        elif name == 'near':
            a = self.anchors(o).get('near')
            if a is not None: v = a.colour
        elif name == 'twin':
            ck = self.canon(o)
            cs = {b.colour for b in self.objs if b.i != o.i and b.colour != o.colour and self.canon(b) == ck}
            if len(cs) == 1: v = cs.pop()
        elif name == 'minor':
            if len(o.colours) == 2: v = min(o.colours, key=lambda c: (o.ccount[c], c))
        elif name == 'major':
            if len(o.colours) >= 2: v = o.colour
        elif name == 'large':
            a = self.anchors(o).get('large')
            if a is not None: v = a.colour
        elif name[:3] in ('eq_', 'eqi'):
            # colour of the reference object(s) sharing feature F (eqi: references not touching the border)
            F = name[4:] if name[:4] == 'eqi_' else name[3:]; fv = self.bfeat(o)[F]
            cs = {b.colour for b in self.objs if b.i != o.i and b.colour != o.colour and self.bfeat(b)[F] == fv
                  and (name[:4] != 'eqi_' or not self.bfeat(b)['border'])}
            if len(cs) == 1: v = cs.pop()
        elif name == 'ocol':
            cs = {b.colour for b in self.objs if b.colour != o.colour}
            if len(cs) == 1: v = cs.pop()
        elif name in ('rowmark', 'colmark'):
            cs = set()
            for b in self.objs:
                if b.i == o.i or b.colour == o.colour: continue
                if name == 'rowmark' and b.r0 <= o.r1 and o.r0 <= b.r1: cs.add(b.colour)
                if name == 'colmark' and b.c0 <= o.c1 and o.c0 <= b.c1: cs.add(b.colour)
            if len(cs) == 1: v = cs.pop()
        self._c[k] = v
        return v

    # ---------------------------------------------------------------- vector expressions
    def dir_to(self, o, a):
        """Orthogonal direction from o toward anchor a along the axis where their projections overlap."""
        colov = o.c0 <= a.c1 and a.c0 <= o.c1; rowov = o.r0 <= a.r1 and a.r0 <= o.r1
        if colov == rowov: return None
        if colov: return 'S' if a.r0 > o.r1 else 'N'
        return 'E' if a.c0 > o.c1 else 'W'

    def mark_dir(self, o):
        """Direction pointed at by the object's marker: its single minority-colour cell (multicolour object) or
        the single-cell object touching it, seen from the object's centroid."""
        k = ('mark', o.i)
        if k in self._c: return self._c[k]
        pts = None
        if len(o.colours) == 2:
            mc = min(o.colours, key=lambda c: (o.ccount[c], c))
            if o.ccount[mc] == 1 and o.size > 2: pts = [cell for cell, c in zip(o.cells, o.cols) if c == mc]
        else:
            ts = [self.objs[j] for j in self.touching(o, True) if self.objs[j].size == 1]
            if len(ts) == 1: pts = ts[0].cells
        d = None
        if pts:
            cy = sum(y for y, _ in o.cells) / o.size; cx = sum(x for _, x in o.cells) / o.size
            vy, vx = pts[0][0] - cy, pts[0][1] - cx
            sy = (vy > 0.25) - (vy < -0.25); sx = (vx > 0.25) - (vx < -0.25)
            for name, v in DIR8.items():
                if v == (sy, sx): d = name
        self._c[k] = d
        self._c[('markpts', o.i)] = pts if d else None
        return d

    def slide_dir(self, o, d):
        if isinstance(d, str): return d
        if d[0] == 'mark': return self.mark_dir(o)
        if d[0] == 'edge':      # toward the nearest grid edge (unique)
            ds = {'N': o.r0, 'S': self.h - 1 - o.r1, 'W': o.c0, 'E': self.w - 1 - o.c1}
            m = min(ds.values()); hits = [k for k, v in ds.items() if v == m]
            return hits[0] if len(hits) == 1 else None
        a = self.anchors(o).get(d[1])
        return self.dir_to(o, a) if a is not None else None

    def border_steps(self, o, t, d):
        cells, nh, nw, ay, ax = o.patch(t); sy, sx = DIR8[d]
        ky = (self.h - nh - ay if sy > 0 else ay) if sy else 99
        kx = (self.w - nw - ax if sx > 0 else ax) if sx else 99
        return min(ky, kx)

    def toward(self, o, t, a):
        """Move straight toward anchor a (along the axis where the projections overlap) until contact."""
        d = self.dir_to(o, a)
        if d is None: return None
        sy, sx = DIR8[d]
        cells, nh, nw, ay, ax = o.patch(t)
        abs_cells = [(ay + dy, ax + dx) for dy, dx, _ in cells]
        acs = a.cset
        for step in range(1, 31):
            for y, x in abs_cells:
                if (y + step * sy, x + step * sx) in acs:
                    self.budget.spend(W_TOWARD * step * len(abs_cells))
                    return ((step - 1) * sy, (step - 1) * sx)
        self.budget.spend(W_TOWARD * 30 * len(abs_cells))
        return None

    def snap(self, o, t, a, conn):
        """Least displacement that makes the (transformed) object touch anchor a -- an object, or a set of cells --
        (4/8-adjacency) without overlapping any other cell; None unless unique."""
        cells, nh, nw, ay, ax = o.patch(t)
        rel = [(ay + dy, ax + dx) for dy, dx, _ in cells]
        nb = N8 if conn == 8 else N4
        acs = a.cset if isinstance(a, Obj) else a
        ring = {(y + dy, x + dx) for y, x in acs for dy, dx in nb} - acs
        self.budget.spend(W_SNAP + W_SNAP_RING * len(acs) * len(nb) + W_SNAP_PAIR * len(ring) * len(rel))
        best = None; ties = 0
        for ry, rx in ring:
            for cy, cx in rel:
                off = (ry - cy, rx - cx)
                if off == (0, 0):
                    return (0, 0) if all(not (0 <= y < self.h and 0 <= x < self.w) or True for y, x in rel) else None
                ok = True
                for y, x in rel:
                    yy, xx = y + off[0], x + off[1]
                    if not (0 <= yy < self.h and 0 <= xx < self.w): ok = False; break
                    j = self.owner[yy][xx]
                    if j >= 0 and j != o.i: ok = False; break
                if not ok: continue
                d = (abs(off[0]) + abs(off[1]), max(abs(off[0]), abs(off[1])))
                if best is None or d < best[0]: best = (d, off); ties = 0
                elif d == best[0] and off != best[1]: ties += 1
        if best is None or ties: return None
        return best[1]

    def vx_offset(self, o, t, vx):
        """Offset (dy, dx) of the in-place transformed patch predicted by a STATIC vector expression."""
        kind = vx[0]
        if kind == 'const': return (vx[1], vx[2])
        cells, nh, nw, ay, ax = o.patch(t)
        if kind == 'border':
            k = self.border_steps(o, t, vx[1]); sy, sx = DIR8[vx[1]]
            return (k * sy, k * sx)
        if kind == 'own':
            d, m = vx[1], vx[2]; sy, sx = DIR8[d]
            return (sy * nh * m, sx * nw * m)
        if kind in ('snap', 'snapc') and (o.size > 6 or t != 'id'): return None     # small pieces only (cost)
        if kind == 'snapc':        # touch the nearest cells of colour c (any object of that colour)
            acs = frozenset(cell for b in self.objs if b.i != o.i and b.colour == vx[1] for cell in b.cells)
            return self.snap(o, t, acs, vx[2]) if acs else None
        if kind == 'slot':
            # the unique background hole of another object whose shape is exactly this (transformed) patch
            shp = frozenset((dy, dx) for dy, dx, _ in cells)
            hits = [tl for s, tl, owner in self.hole_slots() if owner != o.i and s == shp]
            if len(hits) != 1: return None
            return (hits[0][0] - ay, hits[0][1] - ax)
        a = self.anchors(o).get(vx[1])
        if a is None: return None
        if kind == 'toward': return self.toward(o, t, a)
        if kind == 'snap': return self.snap(o, t, a, vx[2])
        if kind == 'onto':
            ny, nx = a.r0 + a.r1 - (2 * ay + nh - 1), a.c0 + a.c1 - (2 * ax + nw - 1)
            if ny % 2 or nx % 2: return None
            return (ny // 2, nx // 2)
        if kind == 'alrow':
            m = vx[2]
            if m == 's': return (a.r0 - ay, 0)
            if m == 'e': return (a.r1 - (ay + nh - 1), 0)
            ny = a.r0 + a.r1 - (2 * ay + nh - 1)
            return None if ny % 2 else (ny // 2, 0)
        if kind == 'alcol':
            m = vx[2]
            if m == 's': return (0, a.c0 - ax)
            if m == 'e': return (0, a.c1 - (ax + nw - 1))
            nx = a.c0 + a.c1 - (2 * ax + nw - 1)
            return None if nx % 2 else (0, nx // 2)
        if kind == 'mirh':
            return (0, a.c0 + a.c1 - (ax + nw - 1) - ax)
        if kind == 'mirv':
            return (a.r0 + a.r1 - (ay + nh - 1) - ay, 0)
        return None


EQ_FEATS = ('holes', 'size', 'shapex', 'shape', 'h', 'w')
REL_CX = ('touch', 'touch4', 'container', 'inner', 'near', 'twin', 'minor', 'major', 'large', 'rowmark', 'colmark',
          'ocol') + \
    tuple('eq_' + F for F in EQ_FEATS) + tuple('eqi_' + F for F in EQ_FEATS)
ANCHORS_FIXED = ('near', 'large', 'samec', 'bigsame', 'twin')


def static_vocab(colours):
    v = [('slot',)] + [('snapc', c, k) for c in colours for k in (4, 8)]
    for d in DIR8:
        v.append(('border', d))
        for m in (1, 2): v.append(('own', d, m))
    anchors = list(ANCHORS_FIXED) + [('nc', c) for c in colours]
    for a in anchors:
        v.append(('toward', a)); v.append(('onto', a)); v.append(('snap', a, 4)); v.append(('snap', a, 8))
        for m in ('s', 'c', 'e'):
            v.append(('alrow', a, m)); v.append(('alcol', a, m))
        v.append(('mirh', a)); v.append(('mirv', a))
    return v


def slide_vocab(colours):
    """Sliding expressions: (direction or ('to', anchor), stop) with stop None / 'hit' / colour."""
    v = []
    dirs = list(DIR8) + [('edge',), ('mark',)] + [('to', a) for a in ANCHORS_FIXED] + [('to', ('nc', c)) for c in colours]
    for d in dirs:
        v.append(('slide', d, None)); v.append(('slide', d, 'hit'))
        if isinstance(d, str) and len(d) == 1:
            for c in colours: v.append(('slide', d, c))
    return v


RAY_DIRS = {'N': ('N',), 'S': ('S',), 'E': ('E',), 'W': ('W',), 'NE': ('NE',), 'NW': ('NW',), 'SE': ('SE',),
            'SW': ('SW',), 'orth': ('N', 'S', 'E', 'W'), 'diag': ('NE', 'NW', 'SE', 'SW'), 'vert': ('N', 'S'),
            'horiz': ('E', 'W'), 'all': tuple(DIR8)}


def add_kinds(colours):
    ks = []
    for dn in RAY_DIRS:
        for stop in ('stop', 'through'):
            ks.append(('ray', (dn, stop)))
    for a in ('near', 'samec', 'large', 'mark') + tuple(('nc', c) for c in colours):
        ks.append(('rayto', a))
    ks += [('outline', False), ('outline', True), ('holes', None), ('bbox', None)]
    return ks


def created(ctx, o, kind, param):
    """Cells created by a creation action (on the input geometry): {cell: emitting colour}."""
    key = ('cr', o.i, kind, param)
    if key in ctx._c: return ctx._c[key]
    g = ctx.g; bg = ctx.bg; h, w = ctx.h, ctx.w; S = {}
    steps = 0           # cells visited (charged as work)
    if kind in ('ray', 'rayto'):
        if kind == 'ray':
            dirs = RAY_DIRS[param[0]]; stop = param[1]
        elif param == 'mark':      # a ray from the marker cell in the direction it points
            d = ctx.mark_dir(o); dirs = (d,) if d else (); stop = 'stop'
            srcs = ctx._c.get(('markpts', o.i)) or []
        else:
            a = ctx.anchors(o).get(param)
            d = ctx.dir_to(o, a) if a is not None else None
            dirs = (d,) if d else (); stop = 'stop'
        emit = list(zip(o.cells, o.cols))
        if kind == 'rayto' and param == 'mark':
            emit = [((y, x), g[y][x]) for y, x in srcs]
        for (y, x), col in emit:
            for d in dirs:
                sy, sx = DIR8[d]; yy, xx = y + sy, x + sx
                while 0 <= yy < h and 0 <= xx < w:
                    if (yy, xx) not in o.cset:
                        if g[yy][xx] != bg:
                            if stop == 'stop': break
                        elif (yy, xx) not in S:
                            S[(yy, xx)] = col
                    yy += sy; xx += sx
                steps += max(abs(yy - y), abs(xx - x))
    elif kind == 'outline':
        steps = o.size * (8 if param else 4)
        for y, x in o.cells:
            for dy, dx in (N8 if param else N4):
                yy, xx = y + dy, x + dx
                if 0 <= yy < h and 0 <= xx < w and g[yy][xx] == bg and (yy, xx) not in o.cset: S[(yy, xx)] = o.colour
    elif kind == 'holes':
        hole, n = ctx.holes(o)
        steps = len(hole)
        for y, x in hole:
            if g[y][x] == bg: S[(y, x)] = o.colour
    elif kind == 'bbox':
        steps = o.h * o.w
        for y in range(o.r0, o.r1 + 1):
            for x in range(o.c0, o.c1 + 1):
                if g[y][x] == bg: S[(y, x)] = o.colour
    elif kind == 'rep':
        # repeated copies along d with period (own extent + gap), until they leave the grid
        d, gap = param; sy, sx = DIR8[d]
        py, px = sy * (o.h + gap), sx * (o.w + gap)
        for k in range(1, 31):
            inside = False; steps += o.size
            for (y, x), col in zip(o.cells, o.cols):
                yy, xx = y + k * py, x + k * px
                if 0 <= yy < h and 0 <= xx < w:
                    inside = True
                    if g[yy][xx] == bg and (yy, xx) not in S: S[(yy, xx)] = col
            if not inside: break
    ctx.budget.spend(W_CR + W_CR_STEP * steps)
    ctx._c[key] = S
    return S


def add_cost(kind, param):
    if kind == 'ray': return 2.2 + (0 if len(param[0]) <= 2 else 0.3) + (0.2 if param[1] == 'through' else 0)
    if kind == 'rep': return 2.4 + 0.2 * param[1] + (0.3 if len(param[0]) == 2 else 0)
    if kind == 'rayto': return 2.6
    return {'outline': 2.0, 'holes': 1.8, 'bbox': 2.2}[kind]


def vx_cost(vx):
    k = vx[0]
    if k == 'const': return 0 if vx[1:] == (0, 0) else 3.0 + 0.1 * (abs(vx[1]) + abs(vx[2]))
    if k == 'slide':
        d = vx[1]
        c = 2.0 if isinstance(d, str) else 2.3 if d[0] in ('edge', 'mark') else 2.4 + (0.3 if d[1] not in ANCHORS_FIXED else 0)
        if isinstance(d, str) and len(d) == 2: c += 0.3
        if vx[2] == 'hit': c += 0.2
        elif vx[2] is not None: c += 0.5
        return c
    if k == 'border': return 2.1 + (0.3 if len(vx[1]) == 2 else 0)
    if k == 'own': return 2.8 + 0.3 * vx[2]
    if k == 'slot': return 2.5
    if k == 'snapc': return 2.9
    base = {'toward': 2.5, 'onto': 2.6, 'alrow': 2.8, 'alcol': 2.8, 'mirh': 2.9, 'mirv': 2.9, 'snap': 2.7}[k]
    return base + (0.3 if vx[1] not in ANCHORS_FIXED else 0)


def cx_cost(cx):
    if cx == 'same': return 0.0
    k = cx[0]
    if k == 'const': return 1.2
    if k == 'rel': return 1.6
    if k == 'tab': return 1.5 + 0.4 * len(cx[2])
    if k == 'swap': return 1.8
    return 2.5


def desc_cost(d):
    if d == KEEP: return 0.0
    if d == DEL: return 1.0
    if d[0] == 'add': return add_cost(d[1], d[2]) + cx_cost(d[3])
    vx, t, cx, copy = d
    return vx_cost(vx) + (0 if t == 'id' else 1.0) + cx_cost(cx) + (0.5 if copy else 0)


# ============================================================ correspondence (placements) and descriptor sets

def out_index(O):
    idx = {}
    for y, row in enumerate(O):
        for x, v in enumerate(row):
            idx.setdefault(v, []).append((y, x))
    return idx


def placements(o, t, O, Oidx):
    """Offsets where the D8 image t of o's coloured patch appears (exact colours) in O."""
    cells, nh, nw, ay, ax = o.patch(t)
    y0, x0, c0 = cells[0]
    Hh, Ww = len(O), len(O[0]); res = set()
    for (y, x) in Oidx.get(c0, ()):
        ty, tx = y - y0, x - x0
        if ty < 0 or tx < 0 or ty + nh > Hh or tx + nw > Ww: continue
        for dy, dx, c in cells:
            if O[ty + dy][tx + dx] != c: break
        else:
            res.add((ty - ay, tx - ax))
    return res


def mono_placements(o, t, O, Oidx, bg, cap=40):
    """Offsets where the shape of t(o) appears uniformly coloured c (c != bg) in O: {offset: c}."""
    cells, nh, nw, ay, ax = o.patch(t)
    y0, x0, _ = cells[0]
    Hh, Ww = len(O), len(O[0]); res = {}
    for c, lst in Oidx.items():
        if c == bg or len(lst) < len(cells): continue
        for (y, x) in lst:
            ty, tx = y - y0, x - x0
            if ty < 0 or tx < 0 or ty + nh > Hh or tx + nw > Ww: continue
            for dy, dx, _ in cells:
                if O[ty + dy][tx + dx] != c: break
            else:
                res[(ty - ay, tx - ax)] = c
                if len(res) > cap: return res
    return res


def conservation(pairs):
    """'rearrange' (per-colour counts equal) / 'recolour' (fg total equal) / 'delete' / 'create' / 'mixed'."""
    kinds = set()
    for i, o in pairs:
        if len(i) != len(o) or len(i[0]) != len(o[0]): return 'diffsize'
        bg = gdsl.bg_of(i)
        ci = Counter(v for r in i for v in r if v != bg); co = Counter(v for r in o for v in r if v != bg)
        if ci == co: kinds.add('rearrange')
        elif sum(ci.values()) == sum(co.values()): kinds.add('recolour')
        elif sum(co.values()) < sum(ci.values()): kinds.add('delete')
        else: kinds.add('create')
    return kinds.pop() if len(kinds) == 1 else 'mixed'


def at_rest(O, bg, cellsabs, sy, sx):
    """(border?, contact colours) one step further along (sy,sx) from the placed cells, read in O."""
    own = set(cellsabs); cols = set(); border = False
    h, w = len(O), len(O[0])
    for y, x in cellsabs:
        yy, xx = y + sy, x + sx
        if not (0 <= yy < h and 0 <= xx < w): border = True; continue
        if (yy, xx) in own: continue
        v = O[yy][xx]
        if v != bg: cols.add(v)
    return border, cols


def rest_ok(stop, border, cols):
    if stop is None: return border or bool(cols)
    if stop == 'hit': return bool(cols)
    return stop in cols


class DSet(dict):
    """descriptor -> gain (number of output cells it makes correct that the input had wrong)."""
    def add(self, d, gain=0):
        if gain > self.get(d, -1): self[d] = gain

    def discard(self, d):
        self.pop(d, None)


def object_descs(ctx, O, Oidx, o, svocab, lvocab, consts, allow, used=None, budget=NOBUDGET):
    """Descriptors consistent with output O for object o (local test), with their gains.
    used: (vector-expr, transform) pairs used by the changed objects; an unchanged object without other
    placements only needs those (a rule whose descriptor is a no-op on every object it covers is useless).
    The work done is charged to budget."""
    g = ctx.g; bg = ctx.bg; res = DSet()
    wk = W_OD + W_OD_CELL * o.size
    ocols = [O[y][x] for y, x in o.cells]
    keep = all(a == b for a, b in zip(ocols, o.cols))
    gone = all(a != b for a, b in zip(ocols, o.cols)) and bg in ocols
    if keep: res.add(KEEP)
    if gone and allow['del']: res.add(DEL, o.size)
    if not keep and allow['recolour']:
        s = set(ocols)
        nchg = sum(1 for a, b in zip(o.cols, ocols) if a != b)
        if len(s) == 1:
            c = ocols[0]
            if c != bg:
                res.add((ZERO, 'id', ('const', c), False), nchg)
                wk += W_REL
                for n in REL_CX:
                    if ctx.colour_expr(o, n) == c: res.add((ZERO, 'id', ('rel', n), False), nchg)
        elif len(o.colours) >= 2 and bg not in s:
            m = {}
            for a, b in zip(o.cols, ocols):
                if m.setdefault(a, b) != b: m = None; break
            if m:
                res.add((ZERO, 'id', ('map', tuple(sorted(m.items()))), False), nchg)
                if len(m) == 2:
                    (a1, b1), (a2, b2) = sorted(m.items())
                    if a1 == b2 and a2 == b1: res.add((ZERO, 'id', ('swap',), False), nchg)
    if keep and allow.get('add'):
        wk += W_ADD * len(allow['add'])
        for kind, param in allow['add']:
            S = created(ctx, o, kind, param)
            if not S: continue
            wk += W_ADD_S * len(S)
            ocs = {O[y][x] for y, x in S}
            if len(ocs) == 1:
                c = next(iter(ocs))
                if c != bg:
                    res.add(('add', kind, param, ('const', c)), len(S))
                    wk += W_REL
                    for n in REL_CX:
                        if ctx.colour_expr(o, n) == c: res.add(('add', kind, param, ('rel', n)), len(S))
            if all(O[y][x] == c for (y, x), c in S.items()): res.add(('add', kind, param, 'same'), len(S))
    if not allow['move']:
        budget.spend(wk)
        return res
    pcache = {}
    hit = [0, 0, 0]     # placement hits tested by add(): count, cells touched, relative-colour scans
    for t in (TXS if allow['tx'] else ('id',)):
        cells, nh, nw, ay, ax = o.patch(t)
        key = (cells, nh, nw, ay, ax)
        wk += W_TX
        if key not in pcache:
            wk += W_PL + W_PL_OCC * len(Oidx.get(cells[0][2], ()))
            pl = placements(o, t, O, Oidx)
            mono = {}
            if allow['recolour'] and t == 'id' and not (pl - {(0, 0)}):
                wk += W_MONO_OCC * sum(len(v) for c, v in Oidx.items() if c != bg)
                mono = mono_placements(o, t, O, Oidx, bg)
                if len(o.colours) == 1: mono = {k: v for k, v in mono.items() if v != o.colour}
            pcache[key] = (pl, mono)
        pl, mono = pcache[key]
        if not pl and not mono: continue

        def add(vx, off, tabs):
            if off not in pl and off not in mono: return
            hit[0] += 1; hit[1] += len(cells) + o.size
            tgt = {(ay + off[0] + dy, ax + off[1] + dx) for dy, dx, _ in cells}
            vac = [(y, x) for y, x in o.cells if (y, x) not in tgt]
            gp = sum(1 for y, x in tgt if g[y][x] != O[y][x])
            gv = sum(1 for y, x in vac if g[y][x] != O[y][x])
            if off in pl:
                still = bool(vac) and all(O[y][x] == g[y][x] for y, x in vac)
                if not still: res.add((vx, t, 'same', False), gp + gv)
                if keep and allow['copy'] and off != (0, 0): res.add((vx, t, 'same', True), gp)
            if off in mono:
                c = mono[off]; hit[2] += 1
                for cx in [('const', c)] + [('rel', n) for n in REL_CX if ctx.colour_expr(o, n) == c]:
                    res.add((vx, t, cx, False), gp + gv)
                    if keep and allow['copy']: res.add((vx, t, cx, True), gp)
        sv, lv, cs = svocab, lvocab, consts
        if used is not None and keep and not mono and pl <= {(0, 0)}:
            wk += W_VFILT * (len(svocab) + len(lvocab) + len(consts))
            sv = [vx for vx in svocab if (vx, t) in used]
            lv = [vx for vx in lvocab if (vx, t) in used]
            cs = [c for c in consts if (('const',) + c, t) in used]
        wk += W_VX * (len(sv) + len(cs) + len(lv)) + W_VX_CELL * (len(sv) + len(lv)) * len(cells)
        # static expressions
        for vx in sv:
            off = ctx.vx_offset(o, t, vx)
            if off is None: continue
            if off == (0, 0) and t == 'id':
                if keep: res.add((vx, 'id', 'same', False))
                continue
            add(vx, off, None)
        for c in cs:
            if c == (0, 0) and t == 'id': continue
            add(('const',) + c, c, None)
        if t != 'id' and (used is None or not keep or (ZERO, t) in used):
            add(ZERO, (0, 0), None)          # transform in place
        # sliding expressions: placement along the direction that is at rest in the output
        for vx in lv:
            d = ctx.slide_dir(o, vx[1])
            if d is None: continue
            sy, sx = DIR8[d]; stop = vx[2]
            kmax = ctx.border_steps(o, t, d)
            wk += W_SLIDE_K * (kmax + 1)
            for k in range(0, kmax + 1):
                off = (k * sy, k * sx)
                if off not in pl and off not in mono: continue
                cabs = [(ay + off[0] + dy, ax + off[1] + dx) for dy, dx, _ in cells]
                wk += W_REST * len(cells)
                border, cols = at_rest(O, bg, cabs, sy, sx)
                if not rest_ok(stop, border, cols): continue
                if off == (0, 0) and t == 'id':
                    if keep: res.add((vx, 'id', 'same', False))
                else:
                    add(vx, off, None)
                break
    wk += W_HIT * hit[0] + W_HIT_CELL * hit[1] + W_REL * hit[2]
    # least action: an object whose disappearance is explained by a move did not vanish
    if DEL in res and any(len(d) == 4 and d[0] != 'add' and not d[3] and (d[0] != ZERO or d[1] != 'id') and d[2] == 'same'
                          and d[0][0] not in ('slot', 'onto') for d in res):
        res.discard(DEL)    # least action: the disappearance is explained by a move
    budget.spend(wk)
    return res


# ============================================================ predicates

def obj_features(ctx, o):
    """dict feature -> value (hashable) for predicate construction."""
    f = {}
    objs = ctx.objs
    f['col'] = o.colour
    f['size'] = o.size
    sizes = sorted({b.size for b in objs}, reverse=True)
    f['srank'] = sizes.index(o.size)
    f['srank_r'] = len(sizes) - 1 - sizes.index(o.size)
    f['h'] = o.h; f['w'] = o.w
    f['rect'] = o.size == o.h * o.w
    f['line'] = (o.h == 1 or o.w == 1) and o.size == o.h * o.w and o.size > 1
    f['single'] = o.size == 1
    f['square'] = o.h == o.w
    hole, nh = ctx.holes(o)
    f['holes'] = min(nh, 6)
    f['border'] = o.r0 == 0 or o.c0 == 0 or o.r1 == ctx.h - 1 or o.c1 == ctx.w - 1
    ck = ctx.canon(o)
    f['shape'] = ck
    f['shapex'] = tuple(sorted(o.shape))
    f['ushape'] = sum(1 for b in objs if ctx.canon(b) == ck) == 1
    f['ucol'] = sum(1 for b in objs if b.colour == o.colour) == 1
    f['ncol'] = min(len(o.colours), 3)
    f['has'] = o.colours
    f['touch'] = frozenset(objs[j].colour for j in ctx.touching(o, True))
    f['ntouch'] = min(len(ctx.touching(o, True)), 2)
    cont = ctx.containers()
    f['in'] = frozenset([objs[cont[o.i]].colour]) if o.i in cont else frozenset()
    f['contains'] = ctx.inner_colours(o)
    rowc = set(); colc = set()
    for b in objs:
        if b.i == o.i: continue
        if b.r0 <= o.r1 and o.r0 <= b.r1: rowc.add(b.colour)
        if b.c0 <= o.c1 and o.c0 <= b.c1: colc.add(b.colour)
    f['rowal'] = frozenset(rowc); f['colal'] = frozenset(colc)
    f['twin'] = frozenset(b.colour for b in objs if b.i != o.i and ctx.canon(b) == ck)
    ext = set()
    if o.r0 == min(b.r0 for b in objs) and sum(1 for b in objs if b.r0 == o.r0) == 1: ext.add('top')
    if o.r1 == max(b.r1 for b in objs) and sum(1 for b in objs if b.r1 == o.r1) == 1: ext.add('bottom')
    if o.c0 == min(b.c0 for b in objs) and sum(1 for b in objs if b.c0 == o.c0) == 1: ext.add('left')
    if o.c1 == max(b.c1 for b in objs) and sum(1 for b in objs if b.c1 == o.c1) == 1: ext.add('right')
    f['ext'] = frozenset(ext)
    f['minorcnt'] = min(o.ccount.values()) if len(o.colours) >= 2 else 0
    f['hasref'] = frozenset(F for F in EQ_FEATS if ctx.colour_expr(o, 'eqi_' + F) is not None)
    others = [b for b in objs if b.colour != o.colour]
    f['oshape'] = ctx.canon(others[0]) if len(others) == 1 else None     # shape of the single key object
    f['nsame'] = min(sum(1 for b in objs if ctx.canon(b) == ck), 4)
    ccnt = Counter(b.colour for b in objs); vals = sorted(set(ccnt.values()), reverse=True)
    f['colrank'] = vals.index(ccnt[o.colour]); f['colrank_r'] = len(vals) - 1 - f['colrank']
    f['ncolsame'] = min(sum(1 for b in objs if b.colour == o.colour), 4)
    ys = sorted({b.r0 for b in objs}); xs = sorted({b.c0 for b in objs})
    f['yrank'] = ys.index(o.r0); f['xrank'] = xs.index(o.c0)
    f['yrank_r'] = len(ys) - 1 - f['yrank']; f['xrank_r'] = len(xs) - 1 - f['xrank']
    return f


SET_FEATS = ('has', 'touch', 'in', 'contains', 'rowal', 'colal', 'twin', 'ext', 'hasref')
NOPRED = ('shapex', 'oshape')
TAB_FEATS = ('shape', 'shapex', 'size', 'holes', 'h', 'w', 'minorcnt', 'ntouch', 'srank', 'srank_r', 'yrank', 'xrank',
             'yrank_r', 'xrank_r', 'nsame', 'ncolsame', 'oshape')
PRED_COST = {'col': 1.0, 'size': 1.6, 'size>=': 2.0, 'size<=': 2.0, 'srank': 1.2, 'srank_r': 1.2, 'h': 2.0, 'w': 2.0,
             'rect': 1.2, 'line': 1.2, 'single': 1.0, 'square': 1.5, 'holes': 1.3, 'border': 1.2, 'shape': 2.5,
             'ushape': 1.5, 'ucol': 1.5, 'ncol': 1.5, 'has': 1.3, 'touch': 1.4, 'ntouch': 1.6, 'in': 1.4,
             'contains': 1.4, 'rowal': 1.8, 'colal': 1.8, 'twin': 1.8, 'ext': 2.0, 'minorcnt': 2.0,
             'touch*': 1.3, 'in*': 1.3, 'contains*': 1.3, 'twin*': 1.6, 'hasref': 1.8,
             'yrank': 2.0, 'xrank': 2.0, 'yrank_r': 2.0, 'xrank_r': 2.0, 'nsame': 1.8, 'ncolsame': 1.8,
             'colrank': 1.3, 'colrank_r': 1.3}


def pred_eval(p, f):
    """Evaluate predicate p = (feat, value[, neg]) on feature dict f."""
    k = p[0]
    if k == 'true': return True
    if k == 'size>=': r = f['size'] >= p[1]
    elif k == 'size<=': r = f['size'] <= p[1]
    elif k in ('touch*', 'in*', 'contains*', 'twin*'): r = bool(f[k[:-1]])
    elif k in SET_FEATS: r = p[1] in f[k]
    else: r = f[k] == p[1]
    return (not r) if len(p) > 2 else r


def build_preds(feats):
    """All predicates with values observed on training objects."""
    vals = {}
    for f in feats:
        for k, v in f.items():
            if k in NOPRED: continue
            if k in SET_FEATS:
                for c in v: vals.setdefault(k, set()).add(c)
            else:
                vals.setdefault(k, set()).add(v)
    preds = [('true', None)]
    for k, vs in vals.items():
        if k == 'shape':
            cnt = Counter(f['shape'] for f in feats)
            vs = {v for v in vs if cnt[v] >= 2}
        if k in ('srank', 'srank_r', 'yrank', 'xrank', 'yrank_r', 'xrank_r', 'colrank', 'colrank_r'):
            vs = {v for v in vs if v <= (1 if k[0] == 's' else 0)}
        if len(vs) > 12 and k not in SET_FEATS: continue
        for v in sorted(vs, key=repr):
            preds.append((k, v)); preds.append((k, v, True))
        if k == 'size':
            for v in sorted(vs):
                preds.append(('size>=', v)); preds.append(('size<=', v))
    for k in ('touch*', 'in*', 'contains*', 'twin*'):
        preds.append((k, None)); preds.append((k, None, True))
    return preds


def pred_cost(p):
    if p[0] == 'true': return 0.0
    return PRED_COST.get(p[0], 2.0) + (0.4 if len(p) > 2 else 0.0)


# ============================================================ decision-list learning

def popcount(x): return bin(x).count('1')


def learn_lists(n, dmask, pmask, dgain=None, max_rules=5, max_lists=20, budget=NOBUDGET):
    """Greedy decision lists with diversified branching on the first rules.
    dmask: {desc: bitmask of objects it is consistent with}; pmask: {pred: bitmask of objects it holds on}.
    Returns (lists of (pred, mask) whose last pred is 'true', {mask: descriptors sorted by cost}).
    The work done (per search node) is charged to budget."""
    full = (1 << n) - 1
    bymask = {}
    for d, m in dmask.items():
        bymask.setdefault(m, []).append(d)
    dg = dgain or {}
    for m in bymask: bymask[m].sort(key=lambda d: (-dg.get(d, 0), desc_cost(d), repr(d)))
    mcost = {m: desc_cost(ds[0]) for m, ds in bymask.items()}
    mgain = {m: dg.get(ds[0], 0) for m, ds in bymask.items()}
    # a mask whose best descriptor changes nothing on any object it covers is dominated by 'keep'
    masks = sorted((m for m in bymask if mgain[m] > 0 or KEEP in bymask[m]), key=lambda m: mcost[m])
    preds = [(p, pm) for p, pm in sorted(pmask.items(), key=lambda kv: pred_cost(kv[0])) if p[0] != 'true']
    results = []; seen = set()

    def rec(R, rules, depth):
        if len(results) >= max_lists: return
        closing = sorted(((mcost[m], m) for m in masks if R & ~m == 0), key=lambda c: c[0])
        ncl = 2 if depth == 0 else 1
        used_m = set()
        for cst, m in closing:
            if len(used_m) >= ncl: break
            used_m.add(m)
            key = tuple(rules) + ((('true', None), m, R),)
            if key not in seen:
                seen.add(key); results.append(list(key))
        if closing and depth >= 1: return
        if depth + 1 >= max_rules: return
        budget.spend(W_NODE + W_NODE_PRED * len(preds) + W_NODE_MASK * len(masks))
        covmap = {}
        for p, pm in preds:                      # cheapest predicate per distinct cover
            cov = pm & R
            if cov and cov != R and cov not in covmap: covmap[cov] = p
        live = [m for m in masks if m & R]
        cands = []
        for cov, p in covmap.items():
            k = popcount(cov); pc = pred_cost(p)
            for m in live:
                if not (cov & ~m):
                    cands.append((k, pc + mcost[m], p, m, cov))
        budget.spend(W_NODE_CAND * len(cands) + W_NODE_CM * len(covmap) * len(live))
        if not cands: return
        picks = []
        by_cov = sorted(cands, key=lambda c: (-c[0], -mgain[c[3]], c[1]))
        width = 4 if depth == 0 else (2 if depth == 1 else 1)
        seen_cov = set(); seen_m = set()
        for c in by_cov:
            if c[4] in seen_cov or c[3] in seen_m: continue
            seen_cov.add(c[4]); seen_m.add(c[3]); picks.append(c)
            if len(picks) >= width: break
        if depth == 0:   # cheapest-per-object alternatives (different descriptor)
            by_eff = sorted(cands, key=lambda c: (c[1] / c[0], -c[0]))
            extra = 0
            for c in by_eff:
                if c[4] in seen_cov or c[3] in seen_m or c[0] < 2: continue
                seen_cov.add(c[4]); seen_m.add(c[3]); picks.append(c); extra += 1
                if extra >= 2: break
        for c in picks:
            rules.append((c[2], c[3], R))
            rec(R & ~c[4], rules, depth + 1)
            rules.pop()
            if len(results) >= max_lists: break

    rec(full, [], 0)
    return results, bymask


# ============================================================ rendering

def cx_colour(ctx, o, cx, c):
    if cx == 'same': return c
    k = cx[0]
    if k == 'const': return cx[1]
    if k == 'rel': return ctx.colour_expr(o, cx[1])
    if k == 'map': return dict(cx[1]).get(c)
    if k == 'swap':
        if len(o.colours) != 2: return None
        a, b = sorted(o.colours)
        return b if c == a else a
    if k == 'tab':
        return dict(cx[2]).get(ctx.feat(o)[cx[1]])
    return None


def canvas_slide(out, bg, abs_cells, sy, sx, stop):
    """Free steps along (sy,sx) on the current canvas (stop at any non-bg cell or the border);
    None when the stop condition ('hit' / colour) is not met at rest."""
    h, w = len(out), len(out[0]); own = set(abs_cells); step = 1
    while True:
        cols = set(); border = False
        for y, x in abs_cells:
            yy, xx = y + step * sy, x + step * sx
            if not (0 <= yy < h and 0 <= xx < w): border = True; continue
            if (yy, xx) in own: continue
            v = out[yy][xx]
            if v != bg: cols.add(v)
        if border or cols:
            return step - 1 if rest_ok(stop, border, cols) else None
        step += 1


def render(ctx, descs, paint='over'):
    """Apply one descriptor per object; returns the output grid or None."""
    g = ctx.g; bg = ctx.bg; out = [r[:] for r in g]
    static, sliders = [], []
    for o, d in zip(ctx.objs, descs):
        if d == KEEP or d[0] == 'add': continue
        if d == DEL:
            for y, x in o.cells: out[y][x] = bg
            continue
        vx, t, cx, copy = d
        if vx == ZERO and t == 'id':
            if cx == 'same': continue
            for (y, x), c in zip(o.cells, o.cols):
                nc = cx_colour(ctx, o, cx, c)
                if nc is None: return None
                out[y][x] = nc
            continue
        if vx[0] == 'slide':
            dd = ctx.slide_dir(o, vx[1])
            if dd is None: return None
            sliders.append((o, t, cx, copy, dd, vx[2]))
        else:
            off = ctx.vx_offset(o, t, vx)
            if off is None: return None
            static.append((o, t, cx, copy, off))
    for item in static + sliders:
        if not item[3]:
            for y, x in item[0].cells: out[y][x] = bg
    adds = [(o, d) for o, d in zip(ctx.objs, descs) if d[0] == 'add']

    def put(o, t, cx, off):
        cells, nh, nw, ay, ax = o.patch(t)
        for dy, dx, c in cells:
            y, x = ay + off[0] + dy, ax + off[1] + dx
            if not (0 <= y < ctx.h and 0 <= x < ctx.w): continue
            nc = cx_colour(ctx, o, cx, c)
            if nc is None: return False
            if paint == 'over' or out[y][x] == bg: out[y][x] = nc
        return True
    for o, t, cx, copy, off in static:
        if not put(o, t, cx, off): return None

    # sliding objects: each direction group moves on its own copy of the canvas (objects moving in different
    # directions pass each other); within a group the leading object moves first and the others stack on it
    groups = {}
    for item in sliders: groups.setdefault(item[4], []).append(item)
    base = [r[:] for r in out]; paints = []
    for dd in DIR8:
        if dd not in groups: continue
        sy, sx = DIR8[dd]
        cv = [r[:] for r in base]
        for o, t, cx, copy, _, stop in sorted(groups[dd], key=lambda it: -max(y * sy + x * sx for y, x in it[0].cells)):
            cells, nh, nw, ay, ax = o.patch(t)
            abs_cells = [(ay + dy, ax + dx) for dy, dx, _ in cells]
            k = canvas_slide(cv, bg, abs_cells, sy, sx, stop)
            if k is None: return None
            for dy, dx, c in cells:
                y, x = ay + k * sy + dy, ax + k * sx + dx
                if not (0 <= y < ctx.h and 0 <= x < ctx.w): continue
                nc = cx_colour(ctx, o, cx, c)
                if nc is None: return None
                cv[y][x] = nc; paints.append((y, x, nc))
    for y, x, nc in paints:
        if paint == 'over' or out[y][x] == bg: out[y][x] = nc
    for o, (_, kind, param, cx) in adds:
        for (y, x), c in created(ctx, o, kind, param).items():
            nc = cx_colour(ctx, o, cx, c)
            if nc is None: return None
            if paint == 'over' or out[y][x] == bg: out[y][x] = nc
    return out


def assign(feats, rules):
    res = []
    for f in feats:
        for p, d in rules:
            if pred_eval(p, f): res.append(d); break
        else:
            return None
    return res


def fmt_pred(p):
    if p[0] == 'true': return 'else'
    s = f"{p[0]}={p[1]!r}" if p[1] is not None else p[0]
    return ('not ' + s) if len(p) > 2 else s


def fmt_desc(d):
    if d == KEEP: return 'keep'
    if d == DEL: return 'delete'
    if d[0] == 'add':
        cx = d[3]
        return f"add:{d[1]}:{d[2]}" + ('' if cx == 'same' else '|colour=' + ','.join(str(v) for v in cx))
    vx, t, cx, copy = d
    s = ('copy' if copy else 'move') + ':' + ','.join(str(v) for v in vx)
    if vx == ZERO: s = 'inplace'
    if t != 'id': s += '|' + t
    if cx != 'same':
        if cx[0] == 'tab': s += f'|colour=table[{cx[1]}:{len(cx[2])}]'
        else: s += '|colour=' + ','.join(str(v) for v in cx)
    return s


def fmt_rules(seg, rules, paint):
    s = seg[0] + ('' if seg[1] is None else f',bg{seg[1]}')
    return f"objmap[{s},{paint}]: " + ' ; '.join(f"{fmt_pred(p)} -> {fmt_desc(d)}" for p, d in rules)


# ============================================================ learning driver

def grid_bg(g, bgc):
    """Background of a grid: the task-level colour bgc when given and present, else the most common colour."""
    if bgc is not None and any(bgc in r for r in g): return bgc
    return gdsl.bg_of(g)


def same_shape_task(train):
    return all(len(p['input']) == len(p['output']) and len(p['input'][0]) == len(p['output'][0]) for p in train)


def seg_signature(ctxs):
    return tuple(tuple(sorted(tuple(o.cells) for o in c.objs)) for c in ctxs)


def add_tables(ctxs, outs, objs_all, descs_all):
    """Lookup-table recolours: output colour of the (in-place, uniformly coloured) object as a function of one
    shape feature, induced from all objects; added to the descriptor sets of the objects it explains."""
    outc = []
    for pi, o in objs_all:
        O = outs[pi]; cs = {O[y][x] for y, x in o.cells}
        c = cs.pop() if len(cs) == 1 else None
        outc.append(c if c is not None and c != ctxs[pi].bg else None)
    if sum(1 for c in outc if c is not None) < 3: return
    for F in TAB_FEATS:
        groups = {}
        for j, (pi, o) in enumerate(objs_all):
            if outc[j] is None: continue
            groups.setdefault(ctxs[pi].feat(o)[F], Counter())[outc[j]] += 1
        table = {v: next(iter(cnt)) for v, cnt in groups.items() if len(cnt) == 1}
        if len(table) < 2 or len(set(table.values())) < 2: continue
        expl = [j for j, (pi, o) in enumerate(objs_all) if outc[j] is not None and table.get(ctxs[pi].feat(o)[F]) == outc[j]]
        if len(expl) < (len(table) + 2 if len(table) <= 3 else 2 * len(table)): continue
        cx = ('tab', F, tuple(sorted(table.items(), key=repr)))
        d = (ZERO, 'id', cx, False)
        for j in expl:
            pi, o = objs_all[j]
            descs_all[j].add(d, sum(1 for c in o.cols if c != outc[j]))


def learn_objmap(ins, outs, tests, budget, segs=SEGS, max_objs=40, allow_over=None, tag='', near=None):
    """Core learner on (input, output) grids of equal size.  Returns list of (cost, program, preds).
    near (list or None): collects near misses -- object maps that render every pair and only change cells to
    their output value (progress), for a residual last step: (cost, rules, seg, paint, renders).
    budget: work budget (checked between segmentations, objects, rule lists and variants; all work charged)."""
    found = []
    npairs = len(ins)
    budget.spend(W_CALL + W_CALL_A * sum(len(g) * len(g[0]) for g in ins))
    # a whole-grid dihedral transform or a global colour map is not an object map (the library owns those)
    for f in gdsl.D8.values():
        if all(f(i) == o for i, o in zip(ins, outs)): return found
    m = gdsl.fit_cmap(ins, outs)
    if m and all(gdsl.apply_cmap(i, m) == o for i, o in zip(ins, outs)): return found
    cons = conservation(list(zip(ins, outs)))
    if cons == 'rearrange':
        allow = {'del': False, 'recolour': False, 'copy': False, 'move': True, 'tx': True}
    elif cons == 'recolour':
        allow = {'del': False, 'recolour': True, 'copy': False, 'move': True, 'tx': True}
    elif cons == 'delete':
        allow = {'del': True, 'recolour': True, 'copy': False, 'move': True, 'tx': True}
    else:
        allow = {'del': True, 'recolour': True, 'copy': True, 'move': True, 'tx': True}
    if allow_over: allow.update(allow_over)
    seen_sig = set()
    colours = sorted({v for g in ins for r in g for v in r} - {gdsl.bg_of(g) for g in ins})
    tot = Counter(v for g in ins for r in g for v in r)
    tbg = max((c for c in tot if all(any(c in r for r in g) for g in ins)), key=lambda c: tot[c], default=None)
    bgmodes = [None] + ([tbg] if tbg is not None and any(gdsl.bg_of(g) != tbg for g in ins) else [])
    allow['add'] = add_kinds(colours) if cons in ('create', 'mixed') and allow.get('add', True) else None
    train_cols = set()
    for bgc, seg in [(b, s) for b in bgmodes for s in segs]:
        if budget.over(): break
        budget.spend(W_CTX)
        ctxs = [Ctx(g, grid_bg(g, bgc), seg, budget) for g in ins]
        if any(len(c.objs) == 0 or len(c.objs) > max_objs for c in ctxs): continue
        sig = (seg_signature(ctxs), seg_signature([Ctx(g, grid_bg(g, bgc), seg, budget) for g in tests]))
        if sig in seen_sig: continue
        seen_sig.add(sig)
        budget.spend(W_SEG + W_SEG_OBJ * sum(len(c.objs) for c in ctxs))
        Oidxs = [out_index(O) for O in outs]
        # does any object move / need a D8 transform?
        need_move = False; need_tx = False; cc = Counter(); ntx = 0
        for c, O, Oi in zip(ctxs, outs, Oidxs):
            for o in c.objs:
                pl = placements(o, 'id', O, Oi)
                if pl - {(0, 0)}:
                    need_move = True
                    if len(pl) <= 3:
                        for off in pl - {(0, 0)}: cc[off] += 1
                elif allow['recolour'] and (0, 0) not in pl and len({O[y][x] for y, x in o.cells}) > 1:
                    need_move = True
                if allow['tx'] and (0, 0) not in pl and o.size > 1:
                    base = o.patch('id')[0]
                    for t in TXS[1:]:
                        ntx += 1
                        if o.patch(t)[0] != base and placements(o, t, O, Oi):
                            need_tx = True; need_move = True; break
        budget.spend(W_NM * sum(len(c.objs) for c in ctxs) + W_NM_TX * ntx)
        consts = [off for off, n in cc.most_common(8)]
        al = dict(allow); al['tx'] = need_tx; al['move'] = allow['move'] and need_move
        svocab = static_vocab(colours) if al['move'] else []
        lvocab = slide_vocab(colours) if al['move'] else []
        objs_all = []; descs_all = []; nbad = 0
        # changed objects first: the vector expressions they use bound the work on unchanged objects
        order = []
        for pi, (c, O) in enumerate(zip(ctxs, outs)):
            for o in c.objs:
                kp = all(O[y][x] == v for (y, x), v in zip(o.cells, o.cols))
                order.append((kp, pi, o))
        used = set(); dsmap = {}
        for kp, pi, o in sorted(order, key=lambda r: r[0]):
            c, O, Oi = ctxs[pi], outs[pi], Oidxs[pi]
            ds = object_descs(c, O, Oi, o, svocab, lvocab, consts, al, used if kp else None, budget)
            if not kp:
                for d in ds:
                    if len(d) == 4 and d[0] != 'add': used.add((d[0], d[1]))
            if not ds:
                nbad += 1
                if DIAG is None: break
            dsmap[(pi, o.i)] = ds
            if budget.over(): break
        for pi, c in enumerate(ctxs):
            for o in c.objs:
                if (pi, o.i) in dsmap:
                    objs_all.append((pi, o)); descs_all.append(dsmap[(pi, o.i)])
        if not nbad and not budget.over() and allow['recolour']:
            add_tables(ctxs, outs, objs_all, descs_all)
        if not nbad and allow['add']:
            # creation actions that create nothing on an unchanged object are vacuously consistent
            adds = sorted({d for ds in descs_all for d in ds if d[0] == 'add'}, key=repr)   # hash-seed independent
            for (pi, o), ds in zip(objs_all, descs_all):
                if KEEP not in ds: continue
                for d in adds:
                    if d not in ds and not created(ctxs[pi], o, d[1], d[2]): ds.add(d)
        if DIAG is not None:
            DIAG.setdefault('segs', {})[seg] = {'unexpl': nbad, 'n': len(descs_all)}
        if nbad or budget.over(): continue
        budget.spend(sum(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 for c in ctxs))
        feats_by_pair = [[c.feat(o) for o in c.objs] for c in ctxs]
        feats_all = [f for fl in feats_by_pair for f in fl]
        pair_of = [pi for pi, _ in objs_all]
        n = len(objs_all)
        dmask = {}; dgain = Counter()
        for j, ds in enumerate(descs_all):
            for d, gn in ds.items():
                dmask[d] = dmask.get(d, 0) | (1 << j); dgain[d] += gn
        pmask = {}
        preds_all = build_preds(feats_all)
        budget.spend(W_PM * len(preds_all) * len(feats_all))
        for p in preds_all:
            m = 0
            for j, f in enumerate(feats_all):
                if pred_eval(p, f): m |= 1 << j
            if m: pmask[p] = m
        budget.spend(W_LL + W_LL_D * len(dmask) + W_LL_P * len(pmask))
        lists, bymask = learn_lists(n, dmask, pmask, dgain, budget=budget)
        if DIAG is not None: DIAG['segs'][seg]['lists'] = len(lists)
        test_ctxs = None; tried = set()
        for struct in lists:
            if budget.over(): break
            budget.spend(W_STRUCT + W_STRUCT_P * len(pmask) * len(struct))
            # variants: cheapest predicate / descriptor per rule first, then single substitutions by
            # alternatives with the same training support (they may differ on the test inputs)
            base = [(p, bymask[m][0]) for p, m, R in struct]
            variants = [base]
            for r, (p, m, R) in enumerate(struct):
                for alt in bymask[m][1:3]:
                    v = list(base); v[r] = (p, alt); variants.append(v)
                if p[0] != 'true':
                    cov = pmask[p] & R
                    alts = sorted((q for q, qm in pmask.items() if q != p and q[0] != 'true' and qm & R == cov),
                                  key=pred_cost)[:2]
                    for q in alts:
                        v = list(base); v[r] = (q, base[r][1]); variants.append(v)
            nok = 0
            for rules in variants:
                key = tuple(rules)
                if key in tried: continue
                tried.add(key)
                if budget.over(): break
                budget.spend(W_VAR + W_VAR_F * len(feats_all) * len(rules))
                # MDL acceptance: every rule fires in >= 2 pairs or on >= 2 objects
                fires = [set() for _ in rules]; nf = [0] * len(rules); colsets = [set() for _ in rules]
                for j, f in enumerate(feats_all):
                    for r, (p, d) in enumerate(rules):
                        if pred_eval(p, f):
                            fires[r].add(pair_of[j]); nf[r] += 1; colsets[r].add(f['col']); break
                if any(len(fs) < 2 and k < 2 for fs, k in zip(fires, nf)) and npairs >= 2: break
                # longer lists: every rule must be supported by at least two training pairs
                if len(rules) >= 4 and npairs >= 2 and any(len(fs) < 2 for fs in fires): continue
                # a lookup table must be supported by at least two objects per entry among those its rule covers
                if any(len(d) == 4 and isinstance(d[2], tuple) and d[2][0] == 'tab' and
                       k < (len(d[2][2]) + 2 if len(d[2][2]) <= 3 else 2 * len(d[2][2]))
                       for (p, d), k in zip(rules, nf)): continue
                for paint in ('over', 'under'):
                    good = True; rends = []
                    budget.spend(W_PAINT)
                    for c, O, fl in zip(ctxs, outs, feats_by_pair):
                        ds = assign(fl, rules)
                        budget.spend(W_REND + W_REND_A * c.h * c.w + W_REND_O * len(c.objs))
                        r = render(c, ds, paint) if ds is not None else None
                        rends.append(r)
                        if r != O:
                            good = False
                            if near is None or r is None: break
                    if not good:
                        if near is not None and len(rends) == npairs and all(r is not None for r in rends):
                            budget.spend(W_NEAR_A * sum(len(O) * len(O[0]) for O in outs))
                            prog = all(r[y][x] == O[y][x] for r, g, O in zip(rends, ins, outs)
                                       for y in range(len(O)) for x in range(len(O[0])) if r[y][x] != g[y][x])
                            # the object map must do a real share of the work (>= 10% of the changed cells)
                            prog = prog and all(10 * sum(1 for ra, ga in zip(r, g) for a, b in zip(ra, ga) if a != b) >=
                                                sum(1 for ga, oa in zip(g, O) for a, b in zip(ga, oa) if a != b)
                                                for r, g, O in zip(rends, ins, outs))
                            if prog:
                                cost = sum(pred_cost(p) + desc_cost(d) for p, d in rules) + 1.0 * len(rules)
                                near.append((cost, rules, (seg, bgc), paint, rends))
                        continue
                    if test_ctxs is None:
                        test_ctxs = [Ctx(g, grid_bg(g, bgc), seg, budget) for g in tests]
                        budget.spend(sum(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 for c in test_ctxs))
                        test_feats = [[c.feat(o) for o in c.objs] for c in test_ctxs]
                        train_cols = {f['col'] for f in feats_all}
                        # segmentation plausibility on the test inputs: objects outside the training size range
                        lo = min(f['size'] for f in feats_all); hi = max(f['size'] for f in feats_all)
                        odd = sum(1 for fl in test_feats for f in fl if f['size'] < lo or f['size'] > hi)
                        seg_pen = 0.6 * odd
                    uses_col = any(p[0] == 'col' for p, d in rules)
                    preds_t = []
                    for c, fl in zip(test_ctxs, test_feats):
                        ds = assign(fl, rules)
                        if ds is None: break
                        if uses_col:
                            bad = False
                            for f in fl:
                                if f['col'] in train_cols: continue
                                for r, (p, d) in enumerate(rules):
                                    if pred_eval(p, f):
                                        if len(colsets[r]) < 2: bad = True
                                        break
                            if bad: break
                        budget.spend(W_REND + W_REND_A * c.h * c.w + W_REND_O * len(c.objs))
                        r = render(c, ds, paint)
                        if r is None: break
                        preds_t.append(r)
                    if len(preds_t) != len(tests): continue
                    cost = sum(pred_cost(p) + desc_cost(d) for p, d in rules) + 1.0 * len(rules) + SEGS.index(seg) * 0.05 + seg_pen
                    found.append((cost + (0.3 if bgc is not None else 0), tag + fmt_rules((seg, bgc), rules, paint), preds_t))
                    nok += 1
                    break
    return found


OLD_FAMS_SKIP = ('fam_codex', 'fam_recolour_by_property')


def first_steps(train):
    """Output-independent programs of the original G-DSL families (usable as a FIRST step)."""
    out = []
    for fam in gdsl.FAMILIES:
        if getattr(fam, '__module__', '') != 'gdsl' or fam.__name__ in OLD_FAMS_SKIP: continue
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(gdsl.compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out


def agree(a, b):
    return sum(1 for ra, rb in zip(a, b) for x, y in zip(ra, rb) if x == y)


def compose_first(task, budget, max_steps=16):
    """library first step (output-independent) -> object map learned on (step(input), output).
    Candidate steps are ranked by their agreement with the outputs (same-size tasks) and cost."""
    train = task['train']; tests = [t['input'] for t in task['test']]
    outs = [p['output'] for p in train]; ins = [p['input'] for p in train]
    same = same_shape_task(train)
    base = sum(agree(i, o) for i, o in zip(ins, outs)) if same else 0
    cands = []; seen = set()
    steps = first_steps(train)
    budget.spend(W_FS_GEN + W_FS_PROG * len(steps))
    for name, cost, fn in steps:
        if budget.left() < 3 * WU_S: break
        if name.startswith('dihedral:'): continue      # a whole-grid rotation is not a first step for objects
        mids = []
        for p, O in zip(ins, outs):
            budget.spend(W_RUN + W_RUN_A * len(p) * len(p[0]))
            r = gdsl.run(fn, p)
            if r is None or len(r) != len(O) or len(r[0]) != len(O[0]): mids = None; break
            mids.append(r)
        if mids is None or mids == outs: continue
        if same and all(m == i for m, i in zip(mids, ins)): continue
        k = repr(mids)
        if k in seen: continue
        seen.add(k)
        score = sum(agree(m, o) for m, o in zip(mids, outs)) - base
        cands.append((-score, cost, name, fn, mids))
    cands.sort(key=lambda c: (c[0], c[1]))
    found = []; n = 0
    for _, cost, name, fn, mids in cands:
        if budget.left() < 2 * WU_S or n >= max_steps: break
        budget.spend(sum(W_RUN + W_RUN_A * len(t) * len(t[0]) for t in tests))
        tm = [gdsl.run(fn, t) for t in tests]
        if any(t is None for t in tm): continue
        # first step followed by a whole-grid dihedral transform (two library steps).  The library already
        # composes its reshapers with every D8 element, so that case would only duplicate a library program.
        if not name.startswith(gdsl.RESHAPERS):
            for dk, f in gdsl.D8.items():
                if dk != 'id' and all(f(m) == o for m, o in zip(mids, outs)):
                    found.append((cost + 2.0, f"{name} ; dihedral:{dk}", [f(t) for t in tm]))
            if found: break
        n += 1
        sub = Budget(min(budget.left() - WU_S, 2 * WU_S), budget)
        for c, prog, preds in learn_objmap(mids, outs, tm, sub, segs=('c4', 'c8', 'm8'), tag=name + ' ; '):
            found.append((c + cost + 1.0, prog, preds))
        if found: break
    return found


def lib_candidates(train):
    """gdsl.candidates(train) restricted to the families with a deterministic internal work bound (LIB_SKIP
    excluded; same program order and costs otherwise)."""
    out = []
    for fam in gdsl.FAMILIES:
        if fam.__name__ in LIB_SKIP: continue
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(gdsl.compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out


def lib_search(task, budget, max_programs=6):
    """gdsl.search(task, max_programs, allow2=False) bounded by WORK instead of time: the library's candidate
    programs (lib_candidates) are consumed in their own (cost) order, at most LIB_MAX_PROGS of them and only while
    the budget lasts; generating the candidates and applying a program to a pair are charged.  Identical to
    gdsl.search over those families whenever neither bound binds."""
    train = task['train']; outs = [p['output'] for p in train]
    found = []
    ident = [p['input'] for p in train]
    m = gdsl.fit_cmap(ident, outs)
    if m and any(k != v for k, v in m.items()) and all(gdsl.apply_cmap(a, m) == b for a, b in zip(ident, outs)):
        found.append(("colour-map", 1, lambda g, m=m: gdsl.apply_cmap(g, m)))
    area = sum(len(p['input']) * len(p['input'][0]) for p in train)
    cands = lib_candidates(train)
    budget.spend(W_LIB_GEN + W_LIB_GEN_A * area + W_LIB_PROG * len(cands))
    for name, cost, fn in cands[:LIB_MAX_PROGS]:
        if budget.over(): break
        preds = []
        for p, o in zip(train, outs):
            budget.spend(W_RUN + W_RUN_A * len(p['input']) * len(p['input'][0]))
            r = gdsl.run(fn, p["input"])
            if r is None or len(r) != len(o) or len(r[0]) != len(o[0]): preds = None; break
            if not preds and gdsl.fit_cmap([r], [o]) is None: preds = None; break
            preds.append(r)
        if not preds: continue
        if preds == outs:
            found.append((name, cost, fn))
        else:
            m = gdsl.fit_cmap(preds, outs)
            if m and all(gdsl.apply_cmap(a, m) == b for a, b in zip(preds, outs)):
                found.append((name + "+colour-map", cost + 1, lambda g, fn=fn, m=m: gdsl.apply_cmap(gdsl.run(fn, g), m)))
        if len(found) >= max_programs: break
    res, seen = [], set()
    for name, cost, fn in sorted(found, key=lambda x: x[1]):
        preds = [gdsl.run(fn, t["input"]) for t in task["test"]]
        if any(p is None for p in preds): continue
        k = str(preds)
        if k in seen: continue
        seen.add(k); res.append({"program": name, "cost": cost, "preds": preds})
    return res


def compose_last(task, near, budget, k=2, lib=True):
    """object map (near miss: progress on every pair) -> a LAST step learned on the residual pairs
    (rendered, output): first a second object map (cheap), then the library search."""
    train = task['train']; tests = [t['input'] for t in task['test']]
    outs = [p['output'] for p in train]
    found = []; done = []
    for cost, rules, seg, paint, rends in sorted(near, key=lambda r: r[0]):
        if budget.left() < 3 * WU_S or len(done) >= k: break
        if len(rules) > 2: continue
        key = repr(rends)
        if key in done: continue
        done.append(key)
        tr = []
        for g in tests:
            c = Ctx(g, grid_bg(g, seg[1]), seg[0], budget)
            budget.spend(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 + W_REND + W_REND_A * c.h * c.w +
                         W_REND_O * len(c.objs))
            ds = assign([c.feat(o) for o in c.objs], rules)
            r = render(c, ds, paint) if ds is not None else None
            if r is None: tr = None; break
            tr.append(r)
        if tr is None: continue
        head = fmt_rules(seg, rules, paint)
        sub_b = Budget(min(3 * WU_S, budget.left() - WU_S), budget)
        for c2, prog2, preds2 in learn_objmap(rends, outs, tr, sub_b):
            if prog2.count(' -> ') + len(rules) > 3: continue     # two object maps: at most 3 rules in all
            found.append((cost + c2 + 1.0, head + ' ; ' + prog2, preds2))
        if found or not lib or budget.left() < 7 * WU_S: continue
        sub = {'train': [{'input': r, 'output': O} for r, O in zip(rends, outs)], 'test': [{'input': r} for r in tr]}
        try:
            res = lib_search(sub, budget, max_programs=2)
        except Exception:
            res = []
        for r in res:
            if r['program'].startswith('codex'): continue        # library programs of our own families only
            found.append((cost + r.get('cost', 3) + 2.0, head + ' ; ' + r['program'], r['preds']))
    return found


def SEARCH(task, budget_s=15.0, compose=True):
    """budget_s: work budget in reference-CPU seconds (WU_S work units each); deterministic, no clock."""
    budget = Budget(budget_s * WU_S)
    train = task['train']; tests = [t['input'] for t in task['test']]
    res = []; near = []
    same = same_shape_task(train)
    # defer to the library when a whole-grid dihedral transform already explains every pair
    for k, f in gdsl.D8.items():
        if all(f(p['input']) == p['output'] for p in train): return []
    m = gdsl.fit_cmap([p['input'] for p in train], [p['output'] for p in train])
    if m and all(gdsl.apply_cmap(p['input'], m) == p['output'] for p in train): return []
    if same:
        ins = [p['input'] for p in train]; outs = [p['output'] for p in train]
        res = learn_objmap(ins, outs, tests, budget, near=near)
    if compose and not res and budget.left() > 4 * WU_S:
        res = compose_first(task, budget)
    if compose and not res and near and budget.left() > 3 * WU_S:
        res = compose_last(task, near, budget)
    res.sort(key=lambda r: r[0])
    out, seen = [], set()
    for cost, prog, preds in res:
        k = repr(preds)
        if k in seen: continue
        seen.add(k); out.append({'program': prog, 'preds': preds})
        if len(out) >= 3: break
    return out


In [ ]:
%%writefile /kaggle/working/probe/compose_objmap2.py
"""compose_objmap: object correspondence + per-object action learner (composition engine, Dream lane 3).

SEARCH(task) -> [{"program", "preds"}] (best first, <= 3); every program reproduces all training pairs.

Pipeline (per task; per background mode (grid / task-level) x segmentation c4 / c8 / m8 / m4 / colour-groups):
 1. Segment every training input into objects (Obj: cells, colours, bbox, D8 images of its patch).
 2. Correspondence (least action).  For each input object find how it appears in the output: in-place keep /
    uniform recolour / colour map / delete, template matches of every D8 image of its coloured patch (exact or
    uniformly recoloured), or cells it creates.  A placement is only usable through an ACTION DESCRIPTOR that
    predicts it from the input alone:
      keep | delete | (vector-expr, D8 transform, colour-expr, keep-original) | add(creation, colour-expr)
      vector-expr : const(dy,dx) | border d | own-size d x m | toward anchor until contact | onto anchor
                    | snap: least displacement touching an anchor / colour | align row/col with an anchor
                    (start/centre/end) | mirror across an anchor | into the matching hole (slot) of another object
                    | slide d / toward an anchor / toward the nearest edge / where the object's marker points,
                      until at rest (any contact / must hit / must hit colour c); sequential per direction group,
                      local test = the placement is at rest IN THE OUTPUT
                    anchors: nearest, nearest of colour c, largest, nearest / largest of own colour, same shape
      colour-expr : same | const c | colour of the touching / containing / contained / nearest / same-shape /
                    largest / row-marker / col-marker / only-other-coloured object | minority / majority own
                    colour | colour map / swap | lookup table over a feature (shape, size, holes, ranks, the key
                    object's shape ...) | colour of the (interior) reference object with the same holes / size /
                    shape / h / w (legend lookup)
      creation    : rays (8 dirs / groups, stop or through) | ray toward an anchor or from the marker cell |
                    outline 4/8 | fill holes | fill bbox
    Each object gets the SET of consistent descriptors with their gains (output cells made correct).
 3. The action is learned as a function of object attributes: greedy decision lists over bitsets (predicates:
    colour, colour-frequency rank, has colour, size / size rank, h, w, shape class, holes, rect / line / single,
    border, unique colour / shape, same-shape count, touches colour c, inside colour c, contains colour c,
    row/col aligned with colour c, same-shape twin, has a legend reference, extreme / rank position, minority-
    colour count, and negations); rules chosen by coverage, then gain, then description length, with
    branching on the first rules; predicate and descriptor alternatives with the same training support are
    tried as variants (they may differ on the test input; the top-2 are returned).
 4. Render (erase movers, in-place recolours, static moves, sliding groups, creations; paint over / under) and
    verify on ALL training pairs.  MDL acceptance: every rule fires in >= 2 pairs or on >= 2 objects (lists of
    4+ rules: >= 2 pairs each); lookup tables need support per entry.  Novelty guard: a test object of an unseen
    colour may not fall into a colour-decided rule learned from a single colour.  Segmentations whose test
    objects fall outside the training size range are penalised.
 5. Composition with the library (only when no single object map fits):
      FIRST: output-independent programs of the original G-DSL families (ranked by agreement with the outputs)
             -> object map (or -> whole-grid D8 for non-reshaper steps);
      LAST : near-miss object map (renders every pair, only makes correct changes, >= 10% of them) -> a second
             object map (<= 3 rules in all) or the library search (lib_search: gdsl.search over the families
             with a deterministic work bound) on the residual pairs (rendered, output).
Conservation laws (per-colour cell counts) pre-classify the pairs (rearrangement / recolour / delete / create)
and prune the descriptor space.  Whole-grid D8 / global colour-map tasks are left to the library.
No task ids or literal task constants: every colour, vector, predicate value and table entry is induced from
the training pairs.

DETERMINISM (Kaggle parity): the engine has no clock and no signal.  Every search is bounded by a WORK budget
(class Budget): the natural units of work -- segmentation contexts, placement scans, (object x transform x
vocabulary expression) tests and placement hits, slide steps, ray / creation cells, snap and anchor scans,
feature / predicate tables, decision-list nodes, rule-list variants and their renders, library programs
generated and applied to a grid -- are charged to an integer counter, with weights calibrated (NNLS on the
1050-task harness) so that WU_S = 1e8 work units ~ 1 s of CPU on the reference machine; the old wall-clock
limits are kept as WU_S x seconds (15 s per task, 2 s / 3 s per first / last-step learner, gates 2-7 s).
Set iteration never decides an order (hash-seed independent).  Predictions are therefore identical on any
machine, under any load.  (The wall-clock original is compose_objmap_timed.py.)
"""
import sys
from collections import Counter
sys.path.insert(0, '/home/claude/work/widen')
import gdsl

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))
DIR8 = {'N': (-1, 0), 'S': (1, 0), 'W': (0, -1), 'E': (0, 1), 'NW': (-1, -1), 'NE': (-1, 1), 'SW': (1, -1), 'SE': (1, 1)}
DIR4 = ('N', 'S', 'W', 'E')
TXS = ('id', 'fh', 'fv', 'r180', 'T', 'r90', 'r270', 'aT')
SEGS = ('c4', 'c8', 'm8', 'm4', 'col')
KEEP = ('keep',)
DEL = ('del',)
ZERO = ('const', 0, 0)
DIAG = None


# ============================================================ deterministic work budget

class Budget:
    """Deterministic work budget: integer work units charged by the engine (no clock).  A sub-budget also
    charges its parent and never has more left than its parent."""
    __slots__ = ('cap', 'used', 'parent')

    def __init__(self, units, parent=None):
        self.cap = int(units); self.used = 0; self.parent = parent

    def spend(self, n):
        b = self
        while b is not None:
            b.used += n; b = b.parent

    def left(self):
        v = self.cap - self.used
        return v if self.parent is None else min(v, self.parent.left())

    def over(self): return self.left() <= 0


NOBUDGET = Budget(1 << 62)

# Work-unit weights (integers): units charged per natural unit of work.  Calibrated by non-negative least squares
# on the 1050-task harness (CPU time per learner call / library phase on the reference machine) so that WU_S units
# ~ 1 s of CPU (1 unit ~ 10 ns).  Integer arithmetic only: the budget is exact on every platform.
WU_S = 100_000_000
W_CALL, W_CALL_A = 0, 81                               # a learner call: base, per training input cell
W_CTX, W_CTX_AREA, W_CTX_OBJ = 0, 102, 1549            # segmentation contexts (per grid: cells, objects)
W_HOLES_A, W_TOWARD, W_CR, W_CR_STEP = 223, 27, 576, 16 # holes (bbox cells), toward (steps x cells), creations
W_SNAP, W_SNAP_RING, W_SNAP_PAIR = 1800, 18, 42        # snap: base, anchor cells x nbrs, ring cells x patch cells
W_ANCH, W_CX, W_CX_N = 459, 279, 19                    # anchors (per object), relative colour (base, per object)
W_SEG, W_SEG_OBJ = 0, 1021                             # a new segmentation: signature, output index, descriptors
W_NM, W_NM_TX = 1021, 1189                             # move / transform pre-scan: per object, per D8 image
W_OD, W_OD_CELL, W_ADD, W_ADD_S = 0, 1923, 577, 59     # object_descs: per object, per cell, per creation kind, per cell
W_REL, W_HIT, W_HIT_CELL, W_REST = 1343, 628, 0, 35    # relative-colour scan; placement hit (+ cells); rest test cells
W_VFILT, W_MONO_OCC = 2, 75                            # vocabulary filtered for unchanged objects; recoloured-copy scan
W_TX, W_PL, W_PL_OCC = 5755, 0, 153                    # per D8 image, per placement scan, per scanned output cell
W_VX, W_VX_CELL, W_SLIDE_K = 66, 4, 44                 # per vocabulary expression, x patch cells, per slide step
W_FEAT, W_FEAT2, W_PM = 2534, 418, 70                  # object features (n, n^2), predicate table (preds x objects)
W_LL, W_LL_D, W_LL_P = 0, 751, 0                       # decision-list setup (descriptors, predicates)
W_NODE, W_NODE_PRED, W_NODE_MASK, W_NODE_CAND, W_NODE_CM = 0, 0, 140, 162, 8 # per decision-list search node
W_STRUCT, W_STRUCT_P = 0, 0                            # per rule list: variant construction
W_VAR, W_VAR_F, W_PAINT = 0, 17, 1568                  # per variant: MDL test (objects x rules); per paint mode
W_REND, W_REND_A, W_REND_O = 546, 9, 410               # per render: base, cells, objects
W_NEAR_A = 3                                           # near-miss test, per output cell
W_FS_GEN, W_FS_PROG = 789962, 0                        # first steps: generation, per program generated
W_RUN, W_RUN_A = 3112, 123                             # a library program applied to one grid: base, per cell
W_LIB_GEN, W_LIB_GEN_A, W_LIB_PROG = 60738876, 115488, 0 # lib_candidates on residual pairs: base, per cell, per program
LIB_MAX_PROGS = 3000                                   # at most this many library programs tried on a residual
# residual last step: library families whose generation has no deterministic internal bound are not searched
# (fam_complete_partial lost its 2 s wall-clock cap for Kaggle parity and can take > 10 s on one residual)
LIB_SKIP = ('fam_complete_partial',)


def tx_cell(t, y, x, h, w):
    if t == 'id': return y, x
    if t == 'fh': return y, w - 1 - x
    if t == 'fv': return h - 1 - y, x
    if t == 'r180': return h - 1 - y, w - 1 - x
    if t == 'T': return x, y
    if t == 'r90': return x, h - 1 - y
    if t == 'r270': return w - 1 - x, y
    return w - 1 - x, h - 1 - y          # aT


def segment(g, bg, seg):
    if seg == 'col':
        d = {}
        for y, row in enumerate(g):
            for x, v in enumerate(row):
                if v != bg: d.setdefault(v, []).append((y, x))
        return [d[c] for c in sorted(d)]
    return gdsl.objects(g, bg, seg[1] == '8', seg[0] == 'c')


# ============================================================ objects and context

class Obj:
    __slots__ = ('i', 'cells', 'cset', 'cols', 'r0', 'c0', 'r1', 'c1', 'h', 'w', 'size', 'colour', 'colours',
                 'rel', 'shape', '_tx', 'ccount')

    def __init__(self, i, cells, g):
        cells = sorted(cells)
        self.i = i; self.cells = cells; self.cset = frozenset(cells)
        self.cols = [g[y][x] for y, x in cells]
        ys = [y for y, _ in cells]; xs = [x for _, x in cells]
        self.r0, self.c0, self.r1, self.c1 = min(ys), min(xs), max(ys), max(xs)
        self.h = self.r1 - self.r0 + 1; self.w = self.c1 - self.c0 + 1
        self.size = len(cells)
        cnt = Counter(self.cols); self.ccount = cnt
        self.colour = min(cnt, key=lambda c: (-cnt[c], c))
        self.colours = frozenset(cnt)
        self.rel = tuple((y - self.r0, x - self.c0, c) for (y, x), c in zip(cells, self.cols))
        self.shape = frozenset((y, x) for y, x, _ in self.rel)
        self._tx = {}

    def patch(self, t):
        """D8 image of the coloured patch: (cells sorted (dy,dx,c), nh, nw, in-place top-left ay, ax)."""
        r = self._tx.get(t)
        if r is None:
            h, w = self.h, self.w
            cells = tuple(sorted(tx_cell(t, y, x, h, w) + (c,) for y, x, c in self.rel))
            nh, nw = (h, w) if t in ('id', 'fh', 'fv', 'r180') else (w, h)
            r = (cells, nh, nw, self.r0 + (h - nh) // 2, self.c0 + (w - nw) // 2)
            self._tx[t] = r
        return r


def canon_shape(shape):
    """D8-canonical key of a cell shape."""
    ys = [y for y, _ in shape]; xs = [x for _, x in shape]
    h, w = max(ys) + 1, max(xs) + 1
    best = None
    for t in TXS:
        k = tuple(sorted(tx_cell(t, y, x, h, w) for y, x in shape))
        if best is None or k < best: best = k
    return best


class Ctx:
    def __init__(self, g, bg, seg, budget=NOBUDGET):
        self.g = g; self.h = len(g); self.w = len(g[0]); self.bg = bg; self.seg = seg
        self.objs = [Obj(i, c, g) for i, c in enumerate(segment(g, bg, seg))]
        self.owner = [[-1] * self.w for _ in range(self.h)]
        for o in self.objs:
            for y, x in o.cells: self.owner[y][x] = o.i
        self._c = {}
        self.budget = budget        # the relation / creation computations below charge their work here
        budget.spend(W_CTX_AREA * self.h * self.w + W_CTX_OBJ * len(self.objs))

    # ---------------------------------------------------------------- relations (lazy)
    def holes(self, o):
        """(hole cells, number of hole components): 4-conn regions of non-o cells inside o's bbox that are
        not connected to the outside."""
        k = ('holes', o.i)
        if k in self._c: return self._c[k]
        self.budget.spend(W_HOLES_A * (o.h + 2) * (o.w + 2))
        r0, c0, r1, c1 = o.r0 - 1, o.c0 - 1, o.r1 + 1, o.c1 + 1
        seen = {(r0, c0)}; st = [(r0, c0)]
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                q = (y + dy, x + dx)
                if r0 <= q[0] <= r1 and c0 <= q[1] <= c1 and q not in seen and q not in o.cset:
                    seen.add(q); st.append(q)
        hole = set()
        for y in range(o.r0, o.r1 + 1):
            for x in range(o.c0, o.c1 + 1):
                if (y, x) not in seen and (y, x) not in o.cset: hole.add((y, x))
        n = 0; rem = set(hole)
        while rem:
            n += 1; st = [rem.pop()]
            while st:
                y, x = st.pop()
                for dy, dx in N4:
                    q = (y + dy, x + dx)
                    if q in rem: rem.discard(q); st.append(q)
        r = (hole, n)
        self._c[k] = r
        return r

    def touching(self, o, diag=True):
        k = ('touch', o.i, diag)
        if k in self._c: return self._c[k]
        res = set()
        for y, x in o.cells:
            for dy, dx in (N8 if diag else N4):
                yy, xx = y + dy, x + dx
                if 0 <= yy < self.h and 0 <= xx < self.w:
                    j = self.owner[yy][xx]
                    if j >= 0 and j != o.i: res.add(j)
        self._c[k] = res
        return res

    def containers(self):
        """dict obj index -> index of the smallest object whose holes contain it."""
        if 'cont' in self._c: return self._c['cont']
        res = {}
        for a in sorted(self.objs, key=lambda a: -a.size):
            if a.h < 3 or a.w < 3: continue
            hole, n = self.holes(a)
            if not n: continue
            for b in self.objs:
                if b.i != a.i and b.cells[0] in hole: res[b.i] = a.i
        self._c['cont'] = res
        return res

    def inner_colours(self, o):
        hole, n = self.holes(o)
        return frozenset(self.g[y][x] for y, x in hole if self.g[y][x] != self.bg)

    def dist(self, a, b):
        gy = max(0, b.r0 - a.r1, a.r0 - b.r1); gx = max(0, b.c0 - a.c1, a.c0 - b.c1)
        return (max(gy, gx), gy + gx)

    def anchors(self, o):
        """dict anchor-name -> Obj."""
        k = ('anch', o.i)
        if k in self._c: return self._c[k]
        self.budget.spend(W_ANCH * len(self.objs))
        res = {}
        others = [b for b in self.objs if b.i != o.i]
        if others:
            ds = [(self.dist(o, b), b) for b in others]
            m = min(d for d, _ in ds); hits = [b for d, b in ds if d == m]
            if len(hits) == 1: res['near'] = hits[0]
            bycol = {}
            for d, b in ds: bycol.setdefault(b.colour, []).append((d, b))
            for c, lst in bycol.items():
                m = min(d for d, _ in lst); hits = [b for d, b in lst if d == m]
                if len(hits) == 1:
                    res[('nc', c)] = hits[0]
                    if c == o.colour: res['samec'] = hits[0]
                if c == o.colour:
                    mx = max(b.size for _, b in lst); hits = [b for _, b in lst if b.size == mx]
                    if len(hits) == 1 and hits[0].size > o.size: res['bigsame'] = hits[0]
            mx = max(b.size for b in others); hits = [b for b in others if b.size == mx]
            if len(hits) == 1 and hits[0].size > o.size: res['large'] = hits[0]
            ck = self.canon(o); hits = [b for b in others if self.canon(b) == ck]
            if len(hits) == 1: res['twin'] = hits[0]
        self._c[k] = res
        return res

    def hole_slots(self):
        """Background hole components of every object: list of (shape (normalised cell set), top-left, owner)."""
        if 'slots' in self._c: return self._c['slots']
        res = []
        for a in self.objs:
            if a.h < 3 or a.w < 3: continue
            hole, n = self.holes(a)
            rem = {c for c in hole if self.g[c[0]][c[1]] == self.bg}
            while rem:
                st = [rem.pop()]; comp = [st[0]]
                while st:
                    y, x = st.pop()
                    for dy, dx in N4:
                        q = (y + dy, x + dx)
                        if q in rem: rem.discard(q); st.append(q); comp.append(q)
                r0 = min(y for y, _ in comp); c0 = min(x for _, x in comp)
                res.append((frozenset((y - r0, x - c0) for y, x in comp), (r0, c0), a.i))
        self._c['slots'] = res
        return res

    def feat(self, o):
        k = ('feat', o.i)
        if k not in self._c: self._c[k] = obj_features(self, o)
        return self._c[k]

    def bfeat(self, o):
        """Intrinsic features (no relations), used to match objects with reference objects."""
        k = ('bfeat', o.i)
        if k not in self._c:
            self._c[k] = {'holes': self.holes(o)[1], 'size': o.size, 'shapex': o.shape, 'h': o.h, 'w': o.w,
                          'shape': self.canon(o),
                          'border': o.r0 == 0 or o.c0 == 0 or o.r1 == self.h - 1 or o.c1 == self.w - 1}
        return self._c[k]

    def canon(self, o):
        k = ('canon', o.i)
        if k not in self._c: self._c[k] = canon_shape(o.shape)
        return self._c[k]

    # ---------------------------------------------------------------- relative colour expressions
    def colour_expr(self, o, name):
        k = ('cx', o.i, name)
        if k in self._c: return self._c[k]
        self.budget.spend(W_CX + W_CX_N * len(self.objs))
        v = None
        if name in ('touch', 'touch4'):
            cs = {self.objs[j].colour for j in self.touching(o, name == 'touch')} - {o.colour}
            if len(cs) == 1: v = cs.pop()
        elif name == 'container':
            j = self.containers().get(o.i)
            if j is not None: v = self.objs[j].colour
        elif name == 'inner':
            cs = self.inner_colours(o) - {o.colour}
            if len(cs) == 1: v = next(iter(cs))
        elif name == 'near':
            a = self.anchors(o).get('near')
            if a is not None: v = a.colour
        elif name == 'twin':
            ck = self.canon(o)
            cs = {b.colour for b in self.objs if b.i != o.i and b.colour != o.colour and self.canon(b) == ck}
            if len(cs) == 1: v = cs.pop()
        elif name == 'minor':
            if len(o.colours) == 2: v = min(o.colours, key=lambda c: (o.ccount[c], c))
        elif name == 'major':
            if len(o.colours) >= 2: v = o.colour
        elif name == 'large':
            a = self.anchors(o).get('large')
            if a is not None: v = a.colour
        elif name[:3] in ('eq_', 'eqi'):
            # colour of the reference object(s) sharing feature F (eqi: references not touching the border)
            F = name[4:] if name[:4] == 'eqi_' else name[3:]; fv = self.bfeat(o)[F]
            cs = {b.colour for b in self.objs if b.i != o.i and b.colour != o.colour and self.bfeat(b)[F] == fv
                  and (name[:4] != 'eqi_' or not self.bfeat(b)['border'])}
            if len(cs) == 1: v = cs.pop()
        elif name == 'ocol':
            cs = {b.colour for b in self.objs if b.colour != o.colour}
            if len(cs) == 1: v = cs.pop()
        elif name in ('rowmark', 'colmark'):
            cs = set()
            for b in self.objs:
                if b.i == o.i or b.colour == o.colour: continue
                if name == 'rowmark' and b.r0 <= o.r1 and o.r0 <= b.r1: cs.add(b.colour)
                if name == 'colmark' and b.c0 <= o.c1 and o.c0 <= b.c1: cs.add(b.colour)
            if len(cs) == 1: v = cs.pop()
        self._c[k] = v
        return v

    # ---------------------------------------------------------------- vector expressions
    def dir_to(self, o, a):
        """Orthogonal direction from o toward anchor a along the axis where their projections overlap."""
        colov = o.c0 <= a.c1 and a.c0 <= o.c1; rowov = o.r0 <= a.r1 and a.r0 <= o.r1
        if colov == rowov: return None
        if colov: return 'S' if a.r0 > o.r1 else 'N'
        return 'E' if a.c0 > o.c1 else 'W'

    def mark_dir(self, o):
        """Direction pointed at by the object's marker: its single minority-colour cell (multicolour object) or
        the single-cell object touching it, seen from the object's centroid."""
        k = ('mark', o.i)
        if k in self._c: return self._c[k]
        pts = None
        if len(o.colours) == 2:
            mc = min(o.colours, key=lambda c: (o.ccount[c], c))
            if o.ccount[mc] == 1 and o.size > 2: pts = [cell for cell, c in zip(o.cells, o.cols) if c == mc]
        else:
            ts = [self.objs[j] for j in self.touching(o, True) if self.objs[j].size == 1]
            if len(ts) == 1: pts = ts[0].cells
        d = None
        if pts:
            cy = sum(y for y, _ in o.cells) / o.size; cx = sum(x for _, x in o.cells) / o.size
            vy, vx = pts[0][0] - cy, pts[0][1] - cx
            sy = (vy > 0.25) - (vy < -0.25); sx = (vx > 0.25) - (vx < -0.25)
            for name, v in DIR8.items():
                if v == (sy, sx): d = name
        self._c[k] = d
        self._c[('markpts', o.i)] = pts if d else None
        return d

    def slide_dir(self, o, d):
        if isinstance(d, str): return d
        if d[0] == 'mark': return self.mark_dir(o)
        if d[0] == 'away':      # straight away from an anchor (opposite of the direction toward it)
            a = self.anchors(o).get(d[1])
            t = self.dir_to(o, a) if a is not None else None
            return {'N': 'S', 'S': 'N', 'W': 'E', 'E': 'W'}.get(t)
        if d[0] == 'edge':      # toward the nearest grid edge (unique)
            ds = {'N': o.r0, 'S': self.h - 1 - o.r1, 'W': o.c0, 'E': self.w - 1 - o.c1}
            m = min(ds.values()); hits = [k for k, v in ds.items() if v == m]
            return hits[0] if len(hits) == 1 else None
        a = self.anchors(o).get(d[1])
        return self.dir_to(o, a) if a is not None else None

    def border_steps(self, o, t, d):
        cells, nh, nw, ay, ax = o.patch(t); sy, sx = DIR8[d]
        ky = (self.h - nh - ay if sy > 0 else ay) if sy else 99
        kx = (self.w - nw - ax if sx > 0 else ax) if sx else 99
        return min(ky, kx)

    def toward(self, o, t, a):
        """Move straight toward anchor a (along the axis where the projections overlap) until contact."""
        d = self.dir_to(o, a)
        if d is None: return None
        sy, sx = DIR8[d]
        cells, nh, nw, ay, ax = o.patch(t)
        abs_cells = [(ay + dy, ax + dx) for dy, dx, _ in cells]
        acs = a.cset
        for step in range(1, 31):
            for y, x in abs_cells:
                if (y + step * sy, x + step * sx) in acs:
                    self.budget.spend(W_TOWARD * step * len(abs_cells))
                    return ((step - 1) * sy, (step - 1) * sx)
        self.budget.spend(W_TOWARD * 30 * len(abs_cells))
        return None

    def snap(self, o, t, a, conn):
        """Least displacement that makes the (transformed) object touch anchor a -- an object, or a set of cells --
        (4/8-adjacency) without overlapping any other cell; None unless unique."""
        cells, nh, nw, ay, ax = o.patch(t)
        rel = [(ay + dy, ax + dx) for dy, dx, _ in cells]
        nb = N8 if conn == 8 else N4
        acs = a.cset if isinstance(a, Obj) else a
        ring = {(y + dy, x + dx) for y, x in acs for dy, dx in nb} - acs
        self.budget.spend(W_SNAP + W_SNAP_RING * len(acs) * len(nb) + W_SNAP_PAIR * len(ring) * len(rel))
        best = None; ties = 0
        for ry, rx in ring:
            for cy, cx in rel:
                off = (ry - cy, rx - cx)
                if off == (0, 0):
                    return (0, 0) if all(not (0 <= y < self.h and 0 <= x < self.w) or True for y, x in rel) else None
                ok = True
                for y, x in rel:
                    yy, xx = y + off[0], x + off[1]
                    if not (0 <= yy < self.h and 0 <= xx < self.w): ok = False; break
                    j = self.owner[yy][xx]
                    if j >= 0 and j != o.i: ok = False; break
                if not ok: continue
                d = (abs(off[0]) + abs(off[1]), max(abs(off[0]), abs(off[1])))
                if best is None or d < best[0]: best = (d, off); ties = 0
                elif d == best[0] and off != best[1]: ties += 1
        if best is None or ties: return None
        return best[1]

    def vx_offset(self, o, t, vx):
        """Offset (dy, dx) of the in-place transformed patch predicted by a STATIC vector expression."""
        kind = vx[0]
        if kind == 'const': return (vx[1], vx[2])
        cells, nh, nw, ay, ax = o.patch(t)
        if kind == 'border':
            k = self.border_steps(o, t, vx[1]); sy, sx = DIR8[vx[1]]
            return (k * sy, k * sx)
        if kind == 'own':
            d, m = vx[1], vx[2]; sy, sx = DIR8[d]
            return (sy * nh * m, sx * nw * m)
        if kind in ('snap', 'snapc') and (o.size > 6 or t != 'id'): return None     # small pieces only (cost)
        if kind == 'snapc':        # touch the nearest cells of colour c (any object of that colour)
            acs = frozenset(cell for b in self.objs if b.i != o.i and b.colour == vx[1] for cell in b.cells)
            return self.snap(o, t, acs, vx[2]) if acs else None
        if kind == 'slot':
            # the unique background hole of another object whose shape is exactly this (transformed) patch
            shp = frozenset((dy, dx) for dy, dx, _ in cells)
            hits = [tl for s, tl, owner in self.hole_slots() if owner != o.i and s == shp]
            if len(hits) != 1: return None
            return (hits[0][0] - ay, hits[0][1] - ax)
        a = self.anchors(o).get(vx[1])
        if a is None: return None
        if kind == 'toward': return self.toward(o, t, a)
        if kind == 'snap': return self.snap(o, t, a, vx[2])
        if kind == 'onto':
            ny, nx = a.r0 + a.r1 - (2 * ay + nh - 1), a.c0 + a.c1 - (2 * ax + nw - 1)
            if ny % 2 or nx % 2: return None
            return (ny // 2, nx // 2)
        if kind == 'alrow':
            m = vx[2]
            if m == 's': return (a.r0 - ay, 0)
            if m == 'e': return (a.r1 - (ay + nh - 1), 0)
            ny = a.r0 + a.r1 - (2 * ay + nh - 1)
            return None if ny % 2 else (ny // 2, 0)
        if kind == 'alcol':
            m = vx[2]
            if m == 's': return (0, a.c0 - ax)
            if m == 'e': return (0, a.c1 - (ax + nw - 1))
            nx = a.c0 + a.c1 - (2 * ax + nw - 1)
            return None if nx % 2 else (0, nx // 2)
        if kind == 'mirh':
            return (0, a.c0 + a.c1 - (ax + nw - 1) - ax)
        if kind == 'mirv':
            return (a.r0 + a.r1 - (ay + nh - 1) - ay, 0)
        return None


EQ_FEATS = ('holes', 'size', 'shapex', 'shape', 'h', 'w')
REL_CX = ('touch', 'touch4', 'container', 'inner', 'near', 'twin', 'minor', 'major', 'large', 'rowmark', 'colmark',
          'ocol') + \
    tuple('eq_' + F for F in EQ_FEATS) + tuple('eqi_' + F for F in EQ_FEATS)
ANCHORS_FIXED = ('near', 'large', 'samec', 'bigsame', 'twin')


def static_vocab(colours):
    v = [('slot',)] + [('snapc', c, k) for c in colours for k in (4, 8)]
    for d in DIR8:
        v.append(('border', d))
        for m in (1, 2): v.append(('own', d, m))
    anchors = list(ANCHORS_FIXED) + [('nc', c) for c in colours]
    for a in anchors:
        v.append(('toward', a)); v.append(('onto', a)); v.append(('snap', a, 4)); v.append(('snap', a, 8))
        for m in ('s', 'c', 'e'):
            v.append(('alrow', a, m)); v.append(('alcol', a, m))
        v.append(('mirh', a)); v.append(('mirv', a))
    return v


def slide_vocab(colours):
    """Sliding expressions: (direction or ('to', anchor), stop) with stop None / 'hit' / colour."""
    v = []
    dirs = list(DIR8) + [('edge',), ('mark',)] + [('to', a) for a in ANCHORS_FIXED] + [('to', ('nc', c)) for c in colours] + \
        [('away', a) for a in ANCHORS_FIXED]
    for d in dirs:
        v.append(('slide', d, None)); v.append(('slide', d, 'hit'))
        if isinstance(d, str) and len(d) == 1:
            for c in colours: v.append(('slide', d, c))
    return v


RAY_DIRS = {'N': ('N',), 'S': ('S',), 'E': ('E',), 'W': ('W',), 'NE': ('NE',), 'NW': ('NW',), 'SE': ('SE',),
            'SW': ('SW',), 'orth': ('N', 'S', 'E', 'W'), 'diag': ('NE', 'NW', 'SE', 'SW'), 'vert': ('N', 'S'),
            'horiz': ('E', 'W'), 'all': tuple(DIR8)}


def add_kinds(colours):
    ks = []
    for dn in RAY_DIRS:
        for stop in ('stop', 'through'):
            ks.append(('ray', (dn, stop)))
    for a in ('near', 'samec', 'large', 'mark') + tuple(('nc', c) for c in colours):
        ks.append(('rayto', a))
    ks += [('outline', False), ('outline', True), ('holes', None), ('bbox', None)]
    ks += [('complete', 'id'), ('complete', 'd8')]
    return ks


def created(ctx, o, kind, param):
    """Cells created by a creation action (on the input geometry): {cell: emitting colour}."""
    key = ('cr', o.i, kind, param)
    if key in ctx._c: return ctx._c[key]
    g = ctx.g; bg = ctx.bg; h, w = ctx.h, ctx.w; S = {}
    steps = 0           # cells visited (charged as work)
    if kind in ('ray', 'rayto'):
        if kind == 'ray':
            dirs = RAY_DIRS[param[0]]; stop = param[1]
        elif param == 'mark':      # a ray from the marker cell in the direction it points
            d = ctx.mark_dir(o); dirs = (d,) if d else (); stop = 'stop'
            srcs = ctx._c.get(('markpts', o.i)) or []
        else:
            a = ctx.anchors(o).get(param)
            d = ctx.dir_to(o, a) if a is not None else None
            dirs = (d,) if d else (); stop = 'stop'
        emit = list(zip(o.cells, o.cols))
        if kind == 'rayto' and param == 'mark':
            emit = [((y, x), g[y][x]) for y, x in srcs]
        for (y, x), col in emit:
            for d in dirs:
                sy, sx = DIR8[d]; yy, xx = y + sy, x + sx
                while 0 <= yy < h and 0 <= xx < w:
                    if (yy, xx) not in o.cset:
                        if g[yy][xx] != bg:
                            if stop == 'stop': break
                        elif (yy, xx) not in S:
                            S[(yy, xx)] = col
                    yy += sy; xx += sx
                steps += max(abs(yy - y), abs(xx - x))
    elif kind == 'outline':
        steps = o.size * (8 if param else 4)
        for y, x in o.cells:
            for dy, dx in (N8 if param else N4):
                yy, xx = y + dy, x + dx
                if 0 <= yy < h and 0 <= xx < w and g[yy][xx] == bg and (yy, xx) not in o.cset: S[(yy, xx)] = o.colour
    elif kind == 'holes':
        hole, n = ctx.holes(o)
        steps = len(hole)
        for y, x in hole:
            if g[y][x] == bg: S[(y, x)] = o.colour
    elif kind == 'bbox':
        steps = o.h * o.w
        for y in range(o.r0, o.r1 + 1):
            for x in range(o.c0, o.c1 + 1):
                if g[y][x] == bg: S[(y, x)] = o.colour
    elif kind == 'front':
        # re-layering by an induced priority: the (monochrome) object's bounding box is completed over the cells
        # of the objects of lower priority that occlude it (param 'small': the smaller bounding box is on top,
        # 'large': the larger one); background cells of the box stay background
        steps = o.h * o.w
        if len(o.colours) == 1:
            a0 = o.h * o.w
            for y in range(o.r0, o.r1 + 1):
                for x in range(o.c0, o.c1 + 1):
                    j = ctx.owner[y][x]
                    if j < 0 or j == o.i or g[y][x] == bg: continue
                    b = ctx.objs[j]; a1 = b.h * b.w
                    if (a1 > a0) if param == 'small' else (a1 < a0): S[(y, x)] = o.colour
    elif kind == 'complete':
        # template completion: the unique larger object (template, in its own orientation or any D8 image) that
        # contains this object's coloured pattern at exactly one position is stamped there (background cells only)
        rel = o.rel; oy0, ox0, oc0 = rel[0]
        results = set()
        for T in ctx.objs:
            if T.i == o.i or T.size <= o.size or not (o.colours <= T.colours): continue
            for t in (TXS if param == 'd8' else ('id',)):
                tc = T.patch(t)[0]
                tmap = {(dy, dx): c for dy, dx, c in tc}
                steps += len(tc)
                for ty, tx, c in tc:
                    if c != oc0: continue
                    sy_, sx_ = ty - oy0, tx - ox0
                    steps += o.size
                    if all(tmap.get((dy + sy_, dx + sx_)) == cc for dy, dx, cc in rel):
                        results.add(frozenset((o.r0 + dy - sy_, o.c0 + dx - sx_, cc) for dy, dx, cc in tc))
                if len(results) > 1: break
            if len(results) > 1: break
        if len(results) == 1:
            for y, x, c in next(iter(results)):
                if 0 <= y < h and 0 <= x < w and (y, x) not in o.cset and g[y][x] == bg: S[(y, x)] = c
    elif kind == 'rep':
        # repeated copies along d with period (own extent + gap), until they leave the grid
        d, gap = param; sy, sx = DIR8[d]
        py, px = sy * (o.h + gap), sx * (o.w + gap)
        for k in range(1, 31):
            inside = False; steps += o.size
            for (y, x), col in zip(o.cells, o.cols):
                yy, xx = y + k * py, x + k * px
                if 0 <= yy < h and 0 <= xx < w:
                    inside = True
                    if g[yy][xx] == bg and (yy, xx) not in S: S[(yy, xx)] = col
            if not inside: break
    ctx.budget.spend(W_CR + W_CR_STEP * steps)
    ctx._c[key] = S
    return S


# ------------------------------------------------------------ per-object shape edits in the object frame (cycle 18)
# An edit keeps a subset of the object's cells (the rest become background) and may recolour the kept cells.
RELAYER = False     # re-layering ('front' creations): off -- fits 52df9849 but predicts its test wrong, and its
                    # vacuous masks perturbed the rule search on create / mixed tasks (regression)
EDIT_KINDS = tuple(('trim', (d, k)) for d in DIR4 for k in (1, 2, 3)) + \
    (('erode', 4), ('erode', 8), ('hollow', 4), ('hollow', 8), ('minor', None), ('major', None),
     ('core', None), ('thin', None))


def edit_kept(ctx, o, kind, param):
    """frozenset of the object's cells kept by a shape edit (object frame), or None when undefined."""
    key = ('ed', o.i, kind, param)
    if key in ctx._c: return ctx._c[key]
    cs = o.cset; r = None
    if kind == 'trim':
        d, k = param
        if d == 'N': r = frozenset(c for c in o.cells if c[0] - o.r0 >= k)
        elif d == 'S': r = frozenset(c for c in o.cells if o.r1 - c[0] >= k)
        elif d == 'W': r = frozenset(c for c in o.cells if c[1] - o.c0 >= k)
        else: r = frozenset(c for c in o.cells if o.c1 - c[1] >= k)
    elif kind in ('erode', 'hollow'):
        nb = N4 if param == 4 else N8
        inner = frozenset((y, x) for y, x in o.cells if all((y + dy, x + dx) in cs for dy, dx in nb))
        r = inner if kind == 'erode' else cs - inner
    elif kind in ('minor', 'major'):
        if len(o.colours) == 2:
            mc = min(o.colours, key=lambda c: (o.ccount[c], c))
            if o.ccount[mc] * 2 != o.size:
                r = frozenset(c for c, v in zip(o.cells, o.cols) if (v != mc) == (kind == 'minor'))
    elif kind == 'core':
        # the cells of the rows and columns where the object is widest / tallest (removes protrusions)
        rows = Counter(y for y, _ in o.cells); cols = Counter(x for _, x in o.cells)
        mr = max(rows.values()); mc = max(cols.values())
        r = frozenset((y, x) for y, x in o.cells if rows[y] == mr and cols[x] == mc)
    elif kind == 'thin':
        # the cells of the object's rows / columns that are only one cell thick are removed (protruding lines)
        rows = Counter(y for y, _ in o.cells); cols = Counter(x for _, x in o.cells)
        r = frozenset((y, x) for y, x in o.cells if rows[y] > 1 and cols[x] > 1)
    ctx.budget.spend(W_OD_CELL * o.size)
    ctx._c[key] = r
    return r


# ------------------------------------------------------------ template stamping (cycle 18)
# A receiver object is overwritten by a template object (a role: the unique object with the most colours, or the
# largest object), in one of its D8 images, aligned on the receiver's bounding box (same size) or centre.
STAMP_ROLES = ('mostcol', 'large', 'near')
STAMP_ALIGN = ('bbox', 'centre', 'scale', 'inner')    # scale: k x k blocks filling the bbox; inner: the hole bbox


def stamp_template(ctx, o, role):
    k = ('stT', role)
    if k not in ctx._c:
        T = None
        if role == 'mostcol':
            m = max(len(b.colours) for b in ctx.objs)
            hits = [b for b in ctx.objs if len(b.colours) == m]
            if m >= 2 and len(hits) == 1: T = hits[0]
        elif role == 'large':
            m = max(b.size for b in ctx.objs)
            hits = [b for b in ctx.objs if b.size == m]
            if len(hits) == 1: T = hits[0]
        elif role == 'near':
            T = ctx.anchors(o).get('near')
            return T
        ctx._c[k] = T
    T = ctx._c[k]
    return None if T is None or T.i == o.i else T


def stamp_cells(ctx, o, role, align, t):
    """{cell: colour} painted by stamping the template on o (None when undefined)."""
    key = ('st', o.i, role, align, t)
    if key in ctx._c: return ctx._c[key]
    T = stamp_template(ctx, o, role); r = None
    if T is not None:
        cells, nh, nw, _, _ = T.patch(t)
        k = 1; ty = tx = None
        if align == 'bbox':
            if (nh, nw) == (o.h, o.w): ty, tx = o.r0, o.c0
        elif align == 'centre':
            dy2, dx2 = o.h - nh, o.w - nw
            if dy2 % 2 == 0 and dx2 % 2 == 0: ty, tx = o.r0 + dy2 // 2, o.c0 + dx2 // 2
        elif align == 'scale':
            if o.h % nh == 0 and o.w % nw == 0 and o.h // nh == o.w // nw >= 2: k = o.h // nh; ty, tx = o.r0, o.c0
        else:       # inner: scaled (k >= 1) into the bounding box of the object's holes
            hole = ctx.holes(o)[0] if o.h >= 3 and o.w >= 3 else ()
            if hole:
                hy0 = min(y for y, _ in hole); hx0 = min(x for _, x in hole)
                hh = max(y for y, _ in hole) - hy0 + 1; hw = max(x for _, x in hole) - hx0 + 1
                if hh % nh == 0 and hw % nw == 0 and hh // nh == hw // nw >= 1:
                    k = hh // nh; ty, tx = hy0, hx0
        if ty is not None:
            r = {}
            for dy, dx, c in cells:
                for a in range(k):
                    for b in range(k):
                        y, x = ty + dy * k + a, tx + dx * k + b
                        if not (0 <= y < ctx.h and 0 <= x < ctx.w): r = None; break
                        r[(y, x)] = c
                    if r is None: break
                if r is None: break
        ctx.budget.spend(W_CR + W_CR_STEP * T.size)
    ctx._c[key] = r
    return r


def add_cost(kind, param):
    if kind == 'ray': return 2.2 + (0 if len(param[0]) <= 2 else 0.3) + (0.2 if param[1] == 'through' else 0)
    if kind == 'rep': return 2.4 + 0.2 * param[1] + (0.3 if len(param[0]) == 2 else 0)
    if kind == 'rayto': return 2.6
    if kind == 'complete': return 2.6 + (0.4 if param == 'd8' else 0)
    if kind == 'front': return 2.4
    return {'outline': 2.0, 'holes': 1.8, 'bbox': 2.2}[kind]


def vx_cost(vx):
    k = vx[0]
    if k == 'const': return 0 if vx[1:] == (0, 0) else 3.0 + 0.1 * (abs(vx[1]) + abs(vx[2]))
    if k == 'slide':
        d = vx[1]
        c = 2.0 if isinstance(d, str) else 2.3 if d[0] in ('edge', 'mark') else 2.4 + (0.3 if d[1] not in ANCHORS_FIXED else 0) + \
            (0.2 if d[0] == 'away' else 0)
        if isinstance(d, str) and len(d) == 2: c += 0.3
        if vx[2] == 'hit': c += 0.2
        elif vx[2] is not None: c += 0.5
        return c
    if k == 'border': return 2.1 + (0.3 if len(vx[1]) == 2 else 0)
    if k == 'own': return 2.8 + 0.3 * vx[2]
    if k == 'slot': return 2.5
    if k == 'snapc': return 2.9
    base = {'toward': 2.5, 'onto': 2.6, 'alrow': 2.8, 'alcol': 2.8, 'mirh': 2.9, 'mirv': 2.9, 'snap': 2.7}[k]
    return base + (0.3 if vx[1] not in ANCHORS_FIXED else 0)


def cx_cost(cx):
    if cx == 'same': return 0.0
    k = cx[0]
    if k == 'const': return 1.2
    if k == 'rel': return 1.6
    if k == 'tab': return 1.5 + 0.4 * len(cx[2])
    if k == 'swap': return 1.8
    return 2.5


def desc_cost(d):
    if d == KEEP: return 0.0
    if d == DEL: return 1.0
    if d[0] == 'add': return add_cost(d[1], d[2]) + cx_cost(d[3])
    if d[0] == 'edit': return 2.0 + (0.2 * d[2][1] if d[1] == 'trim' else 0.3) + cx_cost(d[3])
    if d[0] == 'stamp': return 2.6 + (0 if d[2][1] == 'id' else 1.0) + (0 if d[2][0] == 'bbox' else 0.3) + \
            (0.3 if d[1] == 'near' else 0)
    vx, t, cx, copy = d
    return vx_cost(vx) + (0 if t == 'id' else 1.0) + cx_cost(cx) + (0.5 if copy else 0)


# ============================================================ correspondence (placements) and descriptor sets

def out_index(O):
    idx = {}
    for y, row in enumerate(O):
        for x, v in enumerate(row):
            idx.setdefault(v, []).append((y, x))
    return idx


def placements(o, t, O, Oidx):
    """Offsets where the D8 image t of o's coloured patch appears (exact colours) in O."""
    cells, nh, nw, ay, ax = o.patch(t)
    y0, x0, c0 = cells[0]
    Hh, Ww = len(O), len(O[0]); res = set()
    for (y, x) in Oidx.get(c0, ()):
        ty, tx = y - y0, x - x0
        if ty < 0 or tx < 0 or ty + nh > Hh or tx + nw > Ww: continue
        for dy, dx, c in cells:
            if O[ty + dy][tx + dx] != c: break
        else:
            res.add((ty - ay, tx - ax))
    return res


def mono_placements(o, t, O, Oidx, bg, cap=40):
    """Offsets where the shape of t(o) appears uniformly coloured c (c != bg) in O: {offset: c}."""
    cells, nh, nw, ay, ax = o.patch(t)
    y0, x0, _ = cells[0]
    Hh, Ww = len(O), len(O[0]); res = {}
    for c, lst in Oidx.items():
        if c == bg or len(lst) < len(cells): continue
        for (y, x) in lst:
            ty, tx = y - y0, x - x0
            if ty < 0 or tx < 0 or ty + nh > Hh or tx + nw > Ww: continue
            for dy, dx, _ in cells:
                if O[ty + dy][tx + dx] != c: break
            else:
                res[(ty - ay, tx - ax)] = c
                if len(res) > cap: return res
    return res


def conservation(pairs):
    """'rearrange' (per-colour counts equal) / 'recolour' (fg total equal) / 'delete' / 'create' / 'mixed'."""
    kinds = set()
    for i, o in pairs:
        if len(i) != len(o) or len(i[0]) != len(o[0]): return 'diffsize'
        bg = gdsl.bg_of(i)
        ci = Counter(v for r in i for v in r if v != bg); co = Counter(v for r in o for v in r if v != bg)
        if ci == co: kinds.add('rearrange')
        elif sum(ci.values()) == sum(co.values()): kinds.add('recolour')
        elif sum(co.values()) < sum(ci.values()): kinds.add('delete')
        else: kinds.add('create')
    return kinds.pop() if len(kinds) == 1 else 'mixed'


def at_rest(O, bg, cellsabs, sy, sx):
    """(border?, contact colours) one step further along (sy,sx) from the placed cells, read in O."""
    own = set(cellsabs); cols = set(); border = False
    h, w = len(O), len(O[0])
    for y, x in cellsabs:
        yy, xx = y + sy, x + sx
        if not (0 <= yy < h and 0 <= xx < w): border = True; continue
        if (yy, xx) in own: continue
        v = O[yy][xx]
        if v != bg: cols.add(v)
    return border, cols


def rest_ok(stop, border, cols):
    if stop is None: return border or bool(cols)
    if stop == 'hit': return bool(cols)
    return stop in cols


class DSet(dict):
    """descriptor -> gain (number of output cells it makes correct that the input had wrong)."""
    def add(self, d, gain=0):
        if gain > self.get(d, -1): self[d] = gain

    def discard(self, d):
        self.pop(d, None)


def object_descs(ctx, O, Oidx, o, svocab, lvocab, consts, allow, used=None, budget=NOBUDGET):
    """Descriptors consistent with output O for object o (local test), with their gains.
    used: (vector-expr, transform) pairs used by the changed objects; an unchanged object without other
    placements only needs those (a rule whose descriptor is a no-op on every object it covers is useless).
    The work done is charged to budget."""
    g = ctx.g; bg = ctx.bg; res = DSet()
    wk = W_OD + W_OD_CELL * o.size
    ocols = [O[y][x] for y, x in o.cells]
    keep = all(a == b for a, b in zip(ocols, o.cols))
    gone = all(a != b for a, b in zip(ocols, o.cols)) and bg in ocols
    if keep: res.add(KEEP)
    elif allow.get('occl') and not gone:
        # occluded: the changed cells all took the colour of an object touching this one (it paints over them)
        tc = {ctx.objs[j].colour for j in ctx.touching(o, True)}
        if all(a == b or a in tc for a, b in zip(ocols, o.cols)): res.add(KEEP)
    if gone and allow['del']: res.add(DEL, o.size)
    if not keep and allow['recolour']:
        s = set(ocols)
        nchg = sum(1 for a, b in zip(o.cols, ocols) if a != b)
        if len(s) == 1:
            c = ocols[0]
            if c != bg:
                res.add((ZERO, 'id', ('const', c), False), nchg)
                wk += W_REL
                for n in REL_CX:
                    if ctx.colour_expr(o, n) == c: res.add((ZERO, 'id', ('rel', n), False), nchg)
        elif len(o.colours) >= 2 and bg not in s:
            m = {}
            for a, b in zip(o.cols, ocols):
                if m.setdefault(a, b) != b: m = None; break
            if m:
                res.add((ZERO, 'id', ('map', tuple(sorted(m.items()))), False), nchg)
                if len(m) == 2:
                    (a1, b1), (a2, b2) = sorted(m.items())
                    if a1 == b2 and a2 == b1: res.add((ZERO, 'id', ('swap',), False), nchg)
    if allow.get('edit') and allow['del'] and len(o.cells) > 1:
        # shape edits: the output keeps a subset of the object's cells (possibly recoloured uniformly), the rest
        # became background; unchanged objects take the edits that are a no-op on them (vacuously consistent)
        kept = frozenset(c for c, v in zip(o.cells, ocols) if v != bg)
        # a changed object keeps a non-empty strict subset; vanished and unchanged objects take the edits that
        # the edited objects use in a post-pass (learn_objmap: vacuous / total edits, like the creations)
        if not keep and kept and kept != o.cset:
            kcols = {O[y][x] for y, x in kept}
            same_k = all(O[y][x] == v for (y, x), v in zip(o.cells, o.cols) if (y, x) in kept)
            cxs = []
            if same_k: cxs.append('same')
            elif len(kcols) == 1:
                c = next(iter(kcols)); cxs.append(('const', c))
                wk += W_REL
                cxs += [('rel', n) for n in REL_CX if ctx.colour_expr(o, n) == c]
            nchg = sum(1 for a, b in zip(o.cols, ocols) if a != b)
            wk += W_ADD * len(EDIT_KINDS)
            for kind, param in EDIT_KINDS:
                if edit_kept(ctx, o, kind, param) != kept: continue
                for cx in cxs: res.add(('edit', kind, param, cx), nchg)
    if allow.get('stamp'):
        # template stamping: every painted cell takes the template colour, o's uncovered cells are unchanged
        for role in STAMP_ROLES:
            if stamp_template(ctx, o, role) is None: continue
            for t in (TXS if allow['tx'] else ('id',)):
                for align in STAMP_ALIGN:
                    S = stamp_cells(ctx, o, role, align, t)
                    if S is None: continue
                    wk += W_ADD_S * (len(S) + o.size)
                    if any(O[y][x] != c for (y, x), c in S.items()): continue
                    if any(O[y][x] != v for (y, x), v in zip(o.cells, o.cols) if (y, x) not in S): continue
                    gain = sum(1 for (y, x), c in S.items() if g[y][x] != c)
                    res.add(('stamp', role, (align, t), 'same'), gain)
    if keep and allow.get('add'):
        wk += W_ADD * len(allow['add'])
        for kind, param in allow['add']:
            S = created(ctx, o, kind, param)
            if not S: continue
            wk += W_ADD_S * len(S)
            ocs = {O[y][x] for y, x in S}
            if len(ocs) == 1:
                c = next(iter(ocs))
                if c != bg:
                    res.add(('add', kind, param, ('const', c)), len(S))
                    wk += W_REL
                    for n in REL_CX:
                        if ctx.colour_expr(o, n) == c: res.add(('add', kind, param, ('rel', n)), len(S))
            if all(O[y][x] == c for (y, x), c in S.items()): res.add(('add', kind, param, 'same'), len(S))
    if not allow['move']:
        budget.spend(wk)
        return res
    pcache = {}
    hit = [0, 0, 0]     # placement hits tested by add(): count, cells touched, relative-colour scans
    for t in (TXS if allow['tx'] else ('id',)):
        cells, nh, nw, ay, ax = o.patch(t)
        key = (cells, nh, nw, ay, ax)
        wk += W_TX
        if key not in pcache:
            wk += W_PL + W_PL_OCC * len(Oidx.get(cells[0][2], ()))
            pl = placements(o, t, O, Oidx)
            mono = {}
            if allow['recolour'] and t == 'id' and not (pl - {(0, 0)}):
                wk += W_MONO_OCC * sum(len(v) for c, v in Oidx.items() if c != bg)
                mono = mono_placements(o, t, O, Oidx, bg)
                if len(o.colours) == 1: mono = {k: v for k, v in mono.items() if v != o.colour}
            pcache[key] = (pl, mono)
        pl, mono = pcache[key]
        if not pl and not mono: continue

        def add(vx, off, tabs):
            if off not in pl and off not in mono: return
            hit[0] += 1; hit[1] += len(cells) + o.size
            tgt = {(ay + off[0] + dy, ax + off[1] + dx) for dy, dx, _ in cells}
            vac = [(y, x) for y, x in o.cells if (y, x) not in tgt]
            gp = sum(1 for y, x in tgt if g[y][x] != O[y][x])
            gv = sum(1 for y, x in vac if g[y][x] != O[y][x])
            if off in pl:
                still = bool(vac) and all(O[y][x] == g[y][x] for y, x in vac)
                if not still: res.add((vx, t, 'same', False), gp + gv)
                if keep and allow['copy'] and off != (0, 0): res.add((vx, t, 'same', True), gp)
            if off in mono:
                c = mono[off]; hit[2] += 1
                for cx in [('const', c)] + [('rel', n) for n in REL_CX if ctx.colour_expr(o, n) == c]:
                    res.add((vx, t, cx, False), gp + gv)
                    if keep and allow['copy']: res.add((vx, t, cx, True), gp)
        sv, lv, cs = svocab, lvocab, consts
        if used is not None and keep and not mono and pl <= {(0, 0)}:
            wk += W_VFILT * (len(svocab) + len(lvocab) + len(consts))
            sv = [vx for vx in svocab if (vx, t) in used]
            lv = [vx for vx in lvocab if (vx, t) in used]
            cs = [c for c in consts if (('const',) + c, t) in used]
        wk += W_VX * (len(sv) + len(cs) + len(lv)) + W_VX_CELL * (len(sv) + len(lv)) * len(cells)
        # static expressions
        for vx in sv:
            off = ctx.vx_offset(o, t, vx)
            if off is None: continue
            if off == (0, 0) and t == 'id':
                if keep: res.add((vx, 'id', 'same', False))
                continue
            add(vx, off, None)
        for c in cs:
            if c == (0, 0) and t == 'id': continue
            add(('const',) + c, c, None)
        if t != 'id' and (used is None or not keep or (ZERO, t) in used):
            add(ZERO, (0, 0), None)          # transform in place
        # sliding expressions: placement along the direction that is at rest in the output
        for vx in lv:
            d = ctx.slide_dir(o, vx[1])
            if d is None: continue
            sy, sx = DIR8[d]; stop = vx[2]
            kmax = ctx.border_steps(o, t, d)
            wk += W_SLIDE_K * (kmax + 1)
            for k in range(0, kmax + 1):
                off = (k * sy, k * sx)
                if off not in pl and off not in mono: continue
                cabs = [(ay + off[0] + dy, ax + off[1] + dx) for dy, dx, _ in cells]
                wk += W_REST * len(cells)
                border, cols = at_rest(O, bg, cabs, sy, sx)
                if not rest_ok(stop, border, cols): continue
                if off == (0, 0) and t == 'id':
                    if keep: res.add((vx, 'id', 'same', False))
                else:
                    add(vx, off, None)
                break
    wk += W_HIT * hit[0] + W_HIT_CELL * hit[1] + W_REL * hit[2]
    # least action: an object whose disappearance is explained by a move did not vanish
    if DEL in res and any(len(d) == 4 and d[0] != 'add' and not d[3] and (d[0] != ZERO or d[1] != 'id') and d[2] == 'same'
                          and d[0][0] not in ('slot', 'onto') for d in res):
        res.discard(DEL)    # least action: the disappearance is explained by a move
        if allow.get('edit') and o.h > 1 and o.w > 1:
            # ... unless the object (a 2-D piece) is erased by a shape edit that removes all of it
            for kind, param in EDIT_KINDS:
                if edit_kept(ctx, o, kind, param) == frozenset(): res.add(('edit', kind, param, 'same'), o.size)
    budget.spend(wk)
    return res


# ============================================================ predicates

def obj_features(ctx, o):
    """dict feature -> value (hashable) for predicate construction."""
    f = {}
    objs = ctx.objs
    f['col'] = o.colour
    f['size'] = o.size
    sizes = sorted({b.size for b in objs}, reverse=True)
    f['srank'] = sizes.index(o.size)
    f['srank_r'] = len(sizes) - 1 - sizes.index(o.size)
    f['h'] = o.h; f['w'] = o.w
    f['rect'] = o.size == o.h * o.w
    f['line'] = (o.h == 1 or o.w == 1) and o.size == o.h * o.w and o.size > 1
    f['single'] = o.size == 1
    f['square'] = o.h == o.w
    hole, nh = ctx.holes(o)
    f['holes'] = min(nh, 6)
    f['border'] = o.r0 == 0 or o.c0 == 0 or o.r1 == ctx.h - 1 or o.c1 == ctx.w - 1
    ck = ctx.canon(o)
    f['shape'] = ck
    f['shapex'] = tuple(sorted(o.shape))
    f['ushape'] = sum(1 for b in objs if ctx.canon(b) == ck) == 1
    f['ucol'] = sum(1 for b in objs if b.colour == o.colour) == 1
    f['ncol'] = min(len(o.colours), 3)
    f['has'] = o.colours
    f['touch'] = frozenset(objs[j].colour for j in ctx.touching(o, True))
    f['ntouch'] = min(len(ctx.touching(o, True)), 2)
    cont = ctx.containers()
    f['in'] = frozenset([objs[cont[o.i]].colour]) if o.i in cont else frozenset()
    f['contains'] = ctx.inner_colours(o)
    # colours of the larger objects whose bounding box contains this object's bounding box (open enclosure)
    f['inbb'] = frozenset(b.colour for b in objs if b.i != o.i and b.size > o.size and b.r0 <= o.r0 and o.r1 <= b.r1
                          and b.c0 <= o.c0 and o.c1 <= b.c1)
    rowc = set(); colc = set()
    for b in objs:
        if b.i == o.i: continue
        if b.r0 <= o.r1 and o.r0 <= b.r1: rowc.add(b.colour)
        if b.c0 <= o.c1 and o.c0 <= b.c1: colc.add(b.colour)
    f['rowal'] = frozenset(rowc); f['colal'] = frozenset(colc)
    # relational: an object of the same colour shares a row / column band with this one (key matching)
    f['rowsame'] = o.colour in rowc; f['colsame'] = o.colour in colc
    f['twin'] = frozenset(b.colour for b in objs if b.i != o.i and ctx.canon(b) == ck)
    ext = set()
    if o.r0 == min(b.r0 for b in objs) and sum(1 for b in objs if b.r0 == o.r0) == 1: ext.add('top')
    if o.r1 == max(b.r1 for b in objs) and sum(1 for b in objs if b.r1 == o.r1) == 1: ext.add('bottom')
    if o.c0 == min(b.c0 for b in objs) and sum(1 for b in objs if b.c0 == o.c0) == 1: ext.add('left')
    if o.c1 == max(b.c1 for b in objs) and sum(1 for b in objs if b.c1 == o.c1) == 1: ext.add('right')
    f['ext'] = frozenset(ext)
    f['minorcnt'] = min(o.ccount.values()) if len(o.colours) >= 2 else 0
    f['hasref'] = frozenset(F for F in EQ_FEATS if ctx.colour_expr(o, 'eqi_' + F) is not None)
    others = [b for b in objs if b.colour != o.colour]
    f['oshape'] = ctx.canon(others[0]) if len(others) == 1 else None     # shape of the single key object
    f['nsame'] = min(sum(1 for b in objs if ctx.canon(b) == ck), 4)
    ccnt = Counter(b.colour for b in objs); vals = sorted(set(ccnt.values()), reverse=True)
    f['colrank'] = vals.index(ccnt[o.colour]); f['colrank_r'] = len(vals) - 1 - f['colrank']
    f['ncolsame'] = min(sum(1 for b in objs if b.colour == o.colour), 4)
    ys = sorted({b.r0 for b in objs}); xs = sorted({b.c0 for b in objs})
    f['yrank'] = ys.index(o.r0); f['xrank'] = xs.index(o.c0)
    f['yrank_r'] = len(ys) - 1 - f['yrank']; f['xrank_r'] = len(xs) - 1 - f['xrank']
    return f


SET_FEATS = ('has', 'touch', 'in', 'contains', 'rowal', 'colal', 'twin', 'ext', 'hasref', 'inbb')
NOPRED = ('shapex', 'oshape')
TAB_FEATS = ('shape', 'shapex', 'size', 'holes', 'h', 'w', 'minorcnt', 'ntouch', 'srank', 'srank_r', 'yrank', 'xrank',
             'yrank_r', 'xrank_r', 'nsame', 'ncolsame', 'oshape')
PRED_COST = {'col': 1.0, 'size': 1.6, 'size>=': 2.0, 'size<=': 2.0, 'srank': 1.2, 'srank_r': 1.2, 'h': 2.0, 'w': 2.0,
             'rect': 1.2, 'line': 1.2, 'single': 1.0, 'square': 1.5, 'holes': 1.3, 'border': 1.2, 'shape': 2.5,
             'ushape': 1.5, 'ucol': 1.5, 'ncol': 1.5, 'has': 1.3, 'touch': 1.4, 'ntouch': 1.6, 'in': 1.4,
             'contains': 1.4, 'rowal': 1.8, 'colal': 1.8, 'twin': 1.8, 'ext': 2.0, 'minorcnt': 2.0,
             'touch*': 1.3, 'in*': 1.3, 'contains*': 1.3, 'twin*': 1.6, 'hasref': 1.8, 'inbb': 1.6, 'inbb*': 1.5,
             'yrank': 2.0, 'xrank': 2.0, 'yrank_r': 2.0, 'xrank_r': 2.0, 'nsame': 1.8, 'ncolsame': 1.8,
             'colrank': 1.3, 'colrank_r': 1.3, 'rowsame': 1.8, 'colsame': 1.8}


def pred_eval(p, f):
    """Evaluate predicate p = (feat, value[, neg]) on feature dict f."""
    k = p[0]
    if k == 'true': return True
    if k == 'size>=': r = f['size'] >= p[1]
    elif k == 'size<=': r = f['size'] <= p[1]
    elif k in ('touch*', 'in*', 'contains*', 'twin*', 'inbb*'): r = bool(f[k[:-1]])
    elif k in SET_FEATS: r = p[1] in f[k]
    else: r = f[k] == p[1]
    return (not r) if len(p) > 2 else r


def build_preds(feats):
    """All predicates with values observed on training objects."""
    vals = {}
    for f in feats:
        for k, v in f.items():
            if k in NOPRED: continue
            if k in SET_FEATS:
                for c in v: vals.setdefault(k, set()).add(c)
            else:
                vals.setdefault(k, set()).add(v)
    preds = [('true', None)]
    for k, vs in vals.items():
        if k == 'shape':
            cnt = Counter(f['shape'] for f in feats)
            vs = {v for v in vs if cnt[v] >= 2}
        if k in ('srank', 'srank_r', 'yrank', 'xrank', 'yrank_r', 'xrank_r', 'colrank', 'colrank_r'):
            vs = {v for v in vs if v <= (1 if k[0] == 's' else 0)}
        if len(vs) > 12 and k not in SET_FEATS: continue
        for v in sorted(vs, key=repr):
            preds.append((k, v)); preds.append((k, v, True))
        if k == 'size':
            for v in sorted(vs):
                preds.append(('size>=', v)); preds.append(('size<=', v))
    for k in ('touch*', 'in*', 'contains*', 'twin*', 'inbb*'):
        preds.append((k, None)); preds.append((k, None, True))
    return preds


def pred_cost(p):
    if p[0] == 'true': return 0.0
    return PRED_COST.get(p[0], 2.0) + (0.4 if len(p) > 2 else 0.0)


# ============================================================ decision-list learning

def popcount(x): return bin(x).count('1')


def learn_lists(n, dmask, pmask, dgain=None, max_rules=5, max_lists=20, budget=NOBUDGET):
    """Greedy decision lists with diversified branching on the first rules.
    dmask: {desc: bitmask of objects it is consistent with}; pmask: {pred: bitmask of objects it holds on}.
    Returns (lists of (pred, mask) whose last pred is 'true', {mask: descriptors sorted by cost}).
    The work done (per search node) is charged to budget."""
    full = (1 << n) - 1
    bymask = {}
    for d, m in dmask.items():
        bymask.setdefault(m, []).append(d)
    dg = dgain or {}
    for m in bymask: bymask[m].sort(key=lambda d: (-dg.get(d, 0), desc_cost(d), repr(d)))
    mcost = {m: desc_cost(ds[0]) for m, ds in bymask.items()}
    mgain = {m: dg.get(ds[0], 0) for m, ds in bymask.items()}
    # a mask whose best descriptor changes nothing on any object it covers is dominated by 'keep'
    masks = sorted((m for m in bymask if mgain[m] > 0 or KEEP in bymask[m]), key=lambda m: mcost[m])
    preds = [(p, pm) for p, pm in sorted(pmask.items(), key=lambda kv: pred_cost(kv[0])) if p[0] != 'true']
    results = []; seen = set()

    def rec(R, rules, depth):
        if len(results) >= max_lists: return
        closing = sorted(((mcost[m], m) for m in masks if R & ~m == 0), key=lambda c: c[0])
        ncl = 2 if depth == 0 else 1
        used_m = set()
        for cst, m in closing:
            if len(used_m) >= ncl: break
            used_m.add(m)
            key = tuple(rules) + ((('true', None), m, R),)
            if key not in seen:
                seen.add(key); results.append(list(key))
        if closing and depth >= 1: return
        if depth + 1 >= max_rules: return
        budget.spend(W_NODE + W_NODE_PRED * len(preds) + W_NODE_MASK * len(masks))
        covmap = {}
        for p, pm in preds:                      # cheapest predicate per distinct cover
            cov = pm & R
            if cov and cov != R and cov not in covmap: covmap[cov] = p
        live = [m for m in masks if m & R]
        cands = []
        for cov, p in covmap.items():
            k = popcount(cov); pc = pred_cost(p)
            for m in live:
                if not (cov & ~m):
                    cands.append((k, pc + mcost[m], p, m, cov))
        budget.spend(W_NODE_CAND * len(cands) + W_NODE_CM * len(covmap) * len(live))
        if not cands: return
        picks = []
        by_cov = sorted(cands, key=lambda c: (-c[0], -mgain[c[3]], c[1]))
        width = 4 if depth == 0 else (2 if depth == 1 else 1)
        seen_cov = set(); seen_m = set()
        for c in by_cov:
            if c[4] in seen_cov or c[3] in seen_m: continue
            seen_cov.add(c[4]); seen_m.add(c[3]); picks.append(c)
            if len(picks) >= width: break
        if depth == 0:   # cheapest-per-object alternatives (different descriptor)
            by_eff = sorted(cands, key=lambda c: (c[1] / c[0], -c[0]))
            extra = 0
            for c in by_eff:
                if c[4] in seen_cov or c[3] in seen_m or c[0] < 2: continue
                seen_cov.add(c[4]); seen_m.add(c[3]); picks.append(c); extra += 1
                if extra >= 2: break
        for c in picks:
            rules.append((c[2], c[3], R))
            rec(R & ~c[4], rules, depth + 1)
            rules.pop()
            if len(results) >= max_lists: break

    rec(full, [], 0)
    return results, bymask


# ============================================================ rendering

def cx_colour(ctx, o, cx, c):
    if cx == 'same': return c
    k = cx[0]
    if k == 'const': return cx[1]
    if k == 'rel': return ctx.colour_expr(o, cx[1])
    if k == 'map': return dict(cx[1]).get(c)
    if k == 'swap':
        if len(o.colours) != 2: return None
        a, b = sorted(o.colours)
        return b if c == a else a
    if k == 'tab':
        return dict(cx[2]).get(ctx.feat(o)[cx[1]])
    return None


def canvas_slide(out, bg, abs_cells, sy, sx, stop):
    """Free steps along (sy,sx) on the current canvas (stop at any non-bg cell or the border);
    None when the stop condition ('hit' / colour) is not met at rest."""
    h, w = len(out), len(out[0]); own = set(abs_cells); step = 1
    while True:
        cols = set(); border = False
        for y, x in abs_cells:
            yy, xx = y + step * sy, x + step * sx
            if not (0 <= yy < h and 0 <= xx < w): border = True; continue
            if (yy, xx) in own: continue
            v = out[yy][xx]
            if v != bg: cols.add(v)
        if border or cols:
            return step - 1 if rest_ok(stop, border, cols) else None
        step += 1


def render(ctx, descs, paint='over'):
    """Apply one descriptor per object; returns the output grid or None."""
    g = ctx.g; bg = ctx.bg; out = [r[:] for r in g]
    static, sliders, stamps = [], [], []
    for o, d in zip(ctx.objs, descs):
        if d == KEEP or d[0] == 'add': continue
        if d[0] == 'stamp':
            S = stamp_cells(ctx, o, d[1], d[2][0], d[2][1])
            if S is None: return None
            stamps.append(S)
            continue
        if d[0] == 'edit':
            kept = edit_kept(ctx, o, d[1], d[2])
            if kept is None: return None
            for (y, x), c in zip(o.cells, o.cols):
                if (y, x) not in kept: out[y][x] = bg
                elif d[3] != 'same':
                    nc = cx_colour(ctx, o, d[3], c)
                    if nc is None: return None
                    out[y][x] = nc
            continue
        if d == DEL:
            for y, x in o.cells: out[y][x] = bg
            continue
        vx, t, cx, copy = d
        if vx == ZERO and t == 'id':
            if cx == 'same': continue
            for (y, x), c in zip(o.cells, o.cols):
                nc = cx_colour(ctx, o, cx, c)
                if nc is None: return None
                out[y][x] = nc
            continue
        if vx[0] == 'slide':
            dd = ctx.slide_dir(o, vx[1])
            if dd is None: return None
            sliders.append((o, t, cx, copy, dd, vx[2]))
        else:
            off = ctx.vx_offset(o, t, vx)
            if off is None: return None
            static.append((o, t, cx, copy, off))
    for item in static + sliders:
        if not item[3]:
            for y, x in item[0].cells: out[y][x] = bg
    adds = [(o, d) for o, d in zip(ctx.objs, descs) if d[0] == 'add']
    # re-layering: objects of higher priority are painted last (stable order otherwise)
    adds.sort(key=lambda od: (0, 0) if od[1][1] != 'front' else (1, (-1 if od[1][2] == 'small' else 1) * od[0].h * od[0].w))

    def put(o, t, cx, off):
        cells, nh, nw, ay, ax = o.patch(t)
        for dy, dx, c in cells:
            y, x = ay + off[0] + dy, ax + off[1] + dx
            if not (0 <= y < ctx.h and 0 <= x < ctx.w): continue
            nc = cx_colour(ctx, o, cx, c)
            if nc is None: return False
            if paint == 'over' or out[y][x] == bg: out[y][x] = nc
        return True
    for o, t, cx, copy, off in static:
        if not put(o, t, cx, off): return None
    for S in stamps:
        for (y, x), c in S.items():
            if paint == 'over' or out[y][x] == bg: out[y][x] = c

    # sliding objects: each direction group moves on its own copy of the canvas (objects moving in different
    # directions pass each other); within a group the leading object moves first and the others stack on it
    groups = {}
    for item in sliders: groups.setdefault(item[4], []).append(item)
    base = [r[:] for r in out]; paints = []
    for dd in DIR8:
        if dd not in groups: continue
        sy, sx = DIR8[dd]
        cv = [r[:] for r in base]
        for o, t, cx, copy, _, stop in sorted(groups[dd], key=lambda it: -max(y * sy + x * sx for y, x in it[0].cells)):
            cells, nh, nw, ay, ax = o.patch(t)
            abs_cells = [(ay + dy, ax + dx) for dy, dx, _ in cells]
            k = canvas_slide(cv, bg, abs_cells, sy, sx, stop)
            if k is None: return None
            for dy, dx, c in cells:
                y, x = ay + k * sy + dy, ax + k * sx + dx
                if not (0 <= y < ctx.h and 0 <= x < ctx.w): continue
                nc = cx_colour(ctx, o, cx, c)
                if nc is None: return None
                cv[y][x] = nc; paints.append((y, x, nc))
    for y, x, nc in paints:
        if paint == 'over' or out[y][x] == bg: out[y][x] = nc
    for o, (_, kind, param, cx) in adds:
        for (y, x), c in created(ctx, o, kind, param).items():
            nc = cx_colour(ctx, o, cx, c)
            if nc is None: return None
            if paint == 'over' or out[y][x] == bg: out[y][x] = nc
    return out


def assign(feats, rules):
    res = []
    for f in feats:
        for p, d in rules:
            if pred_eval(p, f): res.append(d); break
        else:
            return None
    return res


def fmt_pred(p):
    if p[0] == 'true': return 'else'
    s = f"{p[0]}={p[1]!r}" if p[1] is not None else p[0]
    return ('not ' + s) if len(p) > 2 else s


def fmt_desc(d):
    if d == KEEP: return 'keep'
    if d == DEL: return 'delete'
    if d[0] == 'stamp':
        return f"stamp:{d[1]}:{d[2][0]}" + ('' if d[2][1] == 'id' else '|' + d[2][1])
    if d[0] in ('add', 'edit'):
        cx = d[3]
        return f"{d[0]}:{d[1]}:{d[2]}" + ('' if cx == 'same' else '|colour=' + ','.join(str(v) for v in cx))
    vx, t, cx, copy = d
    s = ('copy' if copy else 'move') + ':' + ','.join(str(v) for v in vx)
    if vx == ZERO: s = 'inplace'
    if t != 'id': s += '|' + t
    if cx != 'same':
        if cx[0] == 'tab': s += f'|colour=table[{cx[1]}:{len(cx[2])}]'
        else: s += '|colour=' + ','.join(str(v) for v in cx)
    return s


def fmt_rules(seg, rules, paint):
    s = seg[0] + ('' if seg[1] is None else f',bg{seg[1]}')
    return f"objmap[{s},{paint}]: " + ' ; '.join(f"{fmt_pred(p)} -> {fmt_desc(d)}" for p, d in rules)


# ============================================================ learning driver

def diag_class(o, O, bg):
    """DIAG only: how an unexplained object appears in the output at its own cells."""
    oc = [O[y][x] for y, x in o.cells]
    nk = sum(a == b for a, b in zip(oc, o.cols)); nbg = sum(c == bg for c in oc)
    if nk == o.size: return 'kept'
    if nk and nbg and nk + nbg == o.size: return 'part-erase'
    if nk: return 'part-change'
    if nbg == o.size: return 'vanish'
    if not nbg: return 'recol-multi'
    return 'mixed'


def grid_bg(g, bgc):
    """Background of a grid: the task-level colour bgc when given and present, else the most common colour."""
    if bgc is not None and any(bgc in r for r in g): return bgc
    return gdsl.bg_of(g)


def same_shape_task(train):
    return all(len(p['input']) == len(p['output']) and len(p['input'][0]) == len(p['output'][0]) for p in train)


def seg_signature(ctxs):
    return tuple(tuple(sorted(tuple(o.cells) for o in c.objs)) for c in ctxs)


def add_tables(ctxs, outs, objs_all, descs_all):
    """Lookup-table recolours: output colour of the (in-place, uniformly coloured) object as a function of one
    shape feature, induced from all objects; added to the descriptor sets of the objects it explains."""
    outc = []
    for pi, o in objs_all:
        O = outs[pi]; cs = {O[y][x] for y, x in o.cells}
        c = cs.pop() if len(cs) == 1 else None
        outc.append(c if c is not None and c != ctxs[pi].bg else None)
    if sum(1 for c in outc if c is not None) < 3: return
    for F in TAB_FEATS:
        groups = {}
        for j, (pi, o) in enumerate(objs_all):
            if outc[j] is None: continue
            groups.setdefault(ctxs[pi].feat(o)[F], Counter())[outc[j]] += 1
        table = {v: next(iter(cnt)) for v, cnt in groups.items() if len(cnt) == 1}
        if len(table) < 2 or len(set(table.values())) < 2: continue
        expl = [j for j, (pi, o) in enumerate(objs_all) if outc[j] is not None and table.get(ctxs[pi].feat(o)[F]) == outc[j]]
        if len(expl) < (len(table) + 2 if len(table) <= 3 else 2 * len(table)): continue
        cx = ('tab', F, tuple(sorted(table.items(), key=repr)))
        d = (ZERO, 'id', cx, False)
        for j in expl:
            pi, o = objs_all[j]
            descs_all[j].add(d, sum(1 for c in o.cols if c != outc[j]))


def learn_objmap(ins, outs, tests, budget, segs=SEGS, max_objs=40, allow_over=None, tag='', near=None):
    """Core learner on (input, output) grids of equal size.  Returns list of (cost, program, preds).
    near (list or None): collects near misses -- object maps that render every pair and only change cells to
    their output value (progress), for a residual last step: (cost, rules, seg, paint, renders).
    budget: work budget (checked between segmentations, objects, rule lists and variants; all work charged)."""
    found = []
    npairs = len(ins)
    budget.spend(W_CALL + W_CALL_A * sum(len(g) * len(g[0]) for g in ins))
    # a whole-grid dihedral transform or a global colour map is not an object map (the library owns those)
    for f in gdsl.D8.values():
        if all(f(i) == o for i, o in zip(ins, outs)): return found
    m = gdsl.fit_cmap(ins, outs)
    if m and all(gdsl.apply_cmap(i, m) == o for i, o in zip(ins, outs)): return found
    cons = conservation(list(zip(ins, outs)))
    if cons == 'rearrange':
        allow = {'del': False, 'recolour': False, 'copy': False, 'move': True, 'tx': True}
    elif cons == 'recolour':
        allow = {'del': False, 'recolour': True, 'copy': False, 'move': True, 'tx': True}
    elif cons == 'delete':
        allow = {'del': True, 'recolour': True, 'copy': False, 'move': True, 'tx': True}
    else:
        allow = {'del': True, 'recolour': True, 'copy': True, 'move': True, 'tx': True}
    allow['edit'] = True
    if allow_over: allow.update(allow_over)
    seen_sig = set()
    colours = sorted({v for g in ins for r in g for v in r} - {gdsl.bg_of(g) for g in ins})
    tot = Counter(v for g in ins for r in g for v in r)
    tbg = max((c for c in tot if all(any(c in r for r in g) for g in ins)), key=lambda c: tot[c], default=None)
    bgmodes = [None] + ([tbg] if tbg is not None and any(gdsl.bg_of(g) != tbg for g in ins) else [])
    # separator mode: a colour drawing full rows / columns in every input (panel grids) as the background
    for c in sorted({v for r in ins[0] for v in r}):
        if c in bgmodes or c == gdsl.bg_of(ins[0]): continue
        if all(any(all(v == c for v in r) for r in g) or any(all(r[x] == c for r in g) for x in range(len(g[0])))
               for g in ins + tests):
            bgmodes.append(c)
            break
    add_ok = allow.get('add', True)
    allow['stamp'] = cons in ('create', 'mixed') and add_ok
    allow['add'] = add_kinds(colours) if cons in ('create', 'mixed') and add_ok else None
    if cons == 'recolour' and add_ok and RELAYER:
        allow['add'] = [('front', 'small'), ('front', 'large')]   # re-layering conserves the fg cell count
        allow['occl'] = True
    train_cols = set()
    for bgc, seg in [(b, s) for b in bgmodes for s in segs]:
        if budget.over(): break
        budget.spend(W_CTX)
        ctxs = [Ctx(g, grid_bg(g, bgc), seg, budget) for g in ins]
        if any(len(c.objs) == 0 or len(c.objs) > max_objs for c in ctxs): continue
        sig = (seg_signature(ctxs), seg_signature([Ctx(g, grid_bg(g, bgc), seg, budget) for g in tests]))
        if sig in seen_sig: continue
        seen_sig.add(sig)
        budget.spend(W_SEG + W_SEG_OBJ * sum(len(c.objs) for c in ctxs))
        Oidxs = [out_index(O) for O in outs]
        # does any object move / need a D8 transform?
        need_move = False; need_tx = False; cc = Counter(); ntx = 0
        for c, O, Oi in zip(ctxs, outs, Oidxs):
            for o in c.objs:
                pl = placements(o, 'id', O, Oi)
                if pl - {(0, 0)}:
                    need_move = True
                    if len(pl) <= 3:
                        for off in pl - {(0, 0)}: cc[off] += 1
                elif allow['recolour'] and (0, 0) not in pl and len({O[y][x] for y, x in o.cells}) > 1:
                    need_move = True
                if allow['tx'] and (0, 0) not in pl and o.size > 1:
                    base = o.patch('id')[0]
                    for t in TXS[1:]:
                        ntx += 1
                        if o.patch(t)[0] != base and placements(o, t, O, Oi):
                            need_tx = True; need_move = True; break
        budget.spend(W_NM * sum(len(c.objs) for c in ctxs) + W_NM_TX * ntx)
        consts = [off for off, n in cc.most_common(8)]
        al = dict(allow); al['tx'] = need_tx; al['move'] = allow['move'] and need_move
        svocab = static_vocab(colours) if al['move'] else []
        lvocab = slide_vocab(colours) if al['move'] else []
        objs_all = []; descs_all = []; nbad = 0
        # changed objects first: the vector expressions they use bound the work on unchanged objects
        order = []
        for pi, (c, O) in enumerate(zip(ctxs, outs)):
            for o in c.objs:
                kp = all(O[y][x] == v for (y, x), v in zip(o.cells, o.cols))
                order.append((kp, pi, o))
        used = set(); dsmap = {}
        for kp, pi, o in sorted(order, key=lambda r: r[0]):
            c, O, Oi = ctxs[pi], outs[pi], Oidxs[pi]
            ds = object_descs(c, O, Oi, o, svocab, lvocab, consts, al, used if kp else None, budget)
            if not kp:
                for d in ds:
                    if len(d) == 4 and d[0] not in ('add', 'edit', 'stamp'): used.add((d[0], d[1]))
            if not ds:
                nbad += 1
                if DIAG is None: break
                DIAG.setdefault('cls', {}).setdefault(seg, Counter())[diag_class(o, O, c.bg)] += 1
            dsmap[(pi, o.i)] = ds
            if budget.over(): break
        for pi, c in enumerate(ctxs):
            for o in c.objs:
                if (pi, o.i) in dsmap:
                    objs_all.append((pi, o)); descs_all.append(dsmap[(pi, o.i)])
        if not nbad and not budget.over() and allow['recolour']:
            add_tables(ctxs, outs, objs_all, descs_all)
        if not nbad and allow.get('edit') and allow['del']:
            # shape edits used by the edited objects: a no-op on an unchanged object, or erasing all of a vanished
            # object, is consistent with it
            eds = sorted({d for ds in descs_all for d in ds if d[0] == 'edit' and d[3] == 'same'}, key=repr)
            for (pi, o), ds in zip(objs_all, descs_all):
                if not eds or o.size < 2: continue
                if KEEP in ds and ds.get(KEEP, 0) == 0 and all(outs[pi][y][x] == v for (y, x), v in zip(o.cells, o.cols)):
                    tgt = o.cset
                elif DEL in ds:
                    tgt = frozenset()
                else:
                    continue
                for d in eds:
                    if d not in ds and edit_kept(ctxs[pi], o, d[1], d[2]) == tgt: ds.add(d, 0 if tgt else o.size)
        if not nbad and allow['add']:
            # creation actions that create nothing on an unchanged object are vacuously consistent
            adds = sorted({d for ds in descs_all for d in ds if d[0] == 'add'}, key=repr)   # hash-seed independent
            for (pi, o), ds in zip(objs_all, descs_all):
                if KEEP not in ds: continue
                for d in adds:
                    if d not in ds and not created(ctxs[pi], o, d[1], d[2]): ds.add(d)
        if DIAG is not None:
            DIAG.setdefault('segs', {})[(seg, bgc)] = {'unexpl': nbad, 'n': len(descs_all),
                                                        'cls': DIAG.get('cls', {}).pop(seg, None)}
        if nbad or budget.over(): continue
        budget.spend(sum(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 for c in ctxs))
        feats_by_pair = [[c.feat(o) for o in c.objs] for c in ctxs]
        feats_all = [f for fl in feats_by_pair for f in fl]
        pair_of = [pi for pi, _ in objs_all]
        n = len(objs_all)
        dmask = {}; dgain = Counter()
        for j, ds in enumerate(descs_all):
            for d, gn in ds.items():
                dmask[d] = dmask.get(d, 0) | (1 << j); dgain[d] += gn
        pmask = {}
        preds_all = build_preds(feats_all)
        budget.spend(W_PM * len(preds_all) * len(feats_all))
        for p in preds_all:
            m = 0
            for j, f in enumerate(feats_all):
                if pred_eval(p, f): m |= 1 << j
            if m: pmask[p] = m
        budget.spend(W_LL + W_LL_D * len(dmask) + W_LL_P * len(pmask))
        lists, bymask = learn_lists(n, dmask, pmask, dgain, budget=budget)
        if DIAG is not None: DIAG['segs'][(seg, bgc)]['lists'] = len(lists)
        test_ctxs = None; tried = set()
        for struct in lists:
            if budget.over(): break
            budget.spend(W_STRUCT + W_STRUCT_P * len(pmask) * len(struct))
            # variants: cheapest predicate / descriptor per rule first, then single substitutions by
            # alternatives with the same training support (they may differ on the test inputs)
            base = [(p, bymask[m][0]) for p, m, R in struct]
            variants = [base]
            for r, (p, m, R) in enumerate(struct):
                for alt in bymask[m][1:3]:
                    v = list(base); v[r] = (p, alt); variants.append(v)
                if p[0] != 'true':
                    cov = pmask[p] & R
                    alts = sorted((q for q, qm in pmask.items() if q != p and q[0] != 'true' and qm & R == cov),
                                  key=pred_cost)[:2]
                    for q in alts:
                        v = list(base); v[r] = (q, base[r][1]); variants.append(v)
            nok = 0
            for rules in variants:
                key = tuple(rules)
                if key in tried: continue
                tried.add(key)
                if budget.over(): break
                budget.spend(W_VAR + W_VAR_F * len(feats_all) * len(rules))
                # MDL acceptance: every rule fires in >= 2 pairs or on >= 2 objects
                fires = [set() for _ in rules]; nf = [0] * len(rules); colsets = [set() for _ in rules]
                for j, f in enumerate(feats_all):
                    for r, (p, d) in enumerate(rules):
                        if pred_eval(p, f):
                            fires[r].add(pair_of[j]); nf[r] += 1; colsets[r].add(f['col']); break
                if any(len(fs) < 2 and k < 2 for fs, k in zip(fires, nf)) and npairs >= 2:
                    if DIAG is not None:
                        DIAG['mdl'] = DIAG.get('mdl', 0) + 1
                        DIAG.setdefault('mdl_rules', []).append(fmt_rules((seg, bgc), rules, 'over'))
                    break
                # longer lists: every rule must be supported by at least two training pairs
                if len(rules) >= 4 and npairs >= 2 and any(len(fs) < 2 for fs in fires): continue
                # a lookup table must be supported by at least two objects per entry among those its rule covers
                if any(len(d) == 4 and isinstance(d[2], tuple) and d[2][0] == 'tab' and
                       k < (len(d[2][2]) + 2 if len(d[2][2]) <= 3 else 2 * len(d[2][2]))
                       for (p, d), k in zip(rules, nf)): continue
                for paint in ('over', 'under'):
                    good = True; rends = []
                    budget.spend(W_PAINT)
                    for c, O, fl in zip(ctxs, outs, feats_by_pair):
                        ds = assign(fl, rules)
                        budget.spend(W_REND + W_REND_A * c.h * c.w + W_REND_O * len(c.objs))
                        r = render(c, ds, paint) if ds is not None else None
                        rends.append(r)
                        if r != O:
                            good = False
                            if near is None or r is None: break
                    if not good:
                        if DIAG is not None:
                            DIAG['rendfail'] = DIAG.get('rendfail', 0) + 1
                            if all(r is not None for r in rends) and len(rends) == npairs:
                                wc = sum(1 for r, O in zip(rends, outs) for ra, oa in zip(r, O) for a, b in zip(ra, oa) if a != b)
                                if wc < DIAG.get('wrongcells', 1 << 30):
                                    DIAG['wrongcells'] = wc
                                    DIAG['best'] = (fmt_rules((seg, bgc), rules, paint), [
                                        (pi, y, x, r[y][x], O[y][x]) for pi, (r, O) in enumerate(zip(rends, outs))
                                        for y in range(len(O)) for x in range(len(O[0])) if r[y][x] != O[y][x]][:12])
                        if near is not None and len(rends) == npairs and all(r is not None for r in rends):
                            budget.spend(W_NEAR_A * sum(len(O) * len(O[0]) for O in outs))
                            prog = all(r[y][x] == O[y][x] for r, g, O in zip(rends, ins, outs)
                                       for y in range(len(O)) for x in range(len(O[0])) if r[y][x] != g[y][x])
                            # the object map must do a real share of the work (>= 10% of the changed cells)
                            prog = prog and all(10 * sum(1 for ra, ga in zip(r, g) for a, b in zip(ra, ga) if a != b) >=
                                                sum(1 for ga, oa in zip(g, O) for a, b in zip(ga, oa) if a != b)
                                                for r, g, O in zip(rends, ins, outs))
                            if prog:
                                cost = sum(pred_cost(p) + desc_cost(d) for p, d in rules) + 1.0 * len(rules)
                                near.append((cost, rules, (seg, bgc), paint, rends))
                        continue
                    if test_ctxs is None:
                        test_ctxs = [Ctx(g, grid_bg(g, bgc), seg, budget) for g in tests]
                        budget.spend(sum(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 for c in test_ctxs))
                        test_feats = [[c.feat(o) for o in c.objs] for c in test_ctxs]
                        train_cols = {f['col'] for f in feats_all}
                        # segmentation plausibility on the test inputs: objects outside the training size range
                        lo = min(f['size'] for f in feats_all); hi = max(f['size'] for f in feats_all)
                        odd = sum(1 for fl in test_feats for f in fl if f['size'] < lo or f['size'] > hi)
                        seg_pen = 0.6 * odd
                    uses_col = any(p[0] == 'col' for p, d in rules)
                    preds_t = []
                    for c, fl in zip(test_ctxs, test_feats):
                        ds = assign(fl, rules)
                        if ds is None: break
                        if uses_col:
                            bad = False
                            for f in fl:
                                if f['col'] in train_cols: continue
                                for r, (p, d) in enumerate(rules):
                                    if pred_eval(p, f):
                                        if len(colsets[r]) < 2: bad = True
                                        break
                            if bad: break
                        budget.spend(W_REND + W_REND_A * c.h * c.w + W_REND_O * len(c.objs))
                        r = render(c, ds, paint)
                        if r is None: break
                        preds_t.append(r)
                    if len(preds_t) != len(tests):
                        if DIAG is not None:
                            DIAG['testfail'] = DIAG.get('testfail', 0) + 1
                            DIAG.setdefault('tf_rules', []).append(fmt_rules((seg, bgc), rules, paint))
                        continue
                    cost = sum(pred_cost(p) + desc_cost(d) for p, d in rules) + 1.0 * len(rules) + SEGS.index(seg) * 0.05 + seg_pen
                    found.append((cost + (0.3 if bgc is not None else 0), tag + fmt_rules((seg, bgc), rules, paint), preds_t))
                    nok += 1
                    break
    return found


OLD_FAMS_SKIP = ('fam_codex', 'fam_recolour_by_property')


def first_steps(train):
    """Output-independent programs of the original G-DSL families (usable as a FIRST step)."""
    out = []
    for fam in gdsl.FAMILIES:
        if getattr(fam, '__module__', '') != 'gdsl' or fam.__name__ in OLD_FAMS_SKIP: continue
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(gdsl.compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out


def agree(a, b):
    return sum(1 for ra, rb in zip(a, b) for x, y in zip(ra, rb) if x == y)


def compose_first(task, budget, max_steps=16):
    """library first step (output-independent) -> object map learned on (step(input), output).
    Candidate steps are ranked by their agreement with the outputs (same-size tasks) and cost."""
    train = task['train']; tests = [t['input'] for t in task['test']]
    outs = [p['output'] for p in train]; ins = [p['input'] for p in train]
    same = same_shape_task(train)
    base = sum(agree(i, o) for i, o in zip(ins, outs)) if same else 0
    cands = []; seen = set()
    steps = first_steps(train)
    budget.spend(W_FS_GEN + W_FS_PROG * len(steps))
    for name, cost, fn in steps:
        if budget.left() < 3 * WU_S: break
        if name.startswith('dihedral:'): continue      # a whole-grid rotation is not a first step for objects
        mids = []
        for p, O in zip(ins, outs):
            budget.spend(W_RUN + W_RUN_A * len(p) * len(p[0]))
            r = gdsl.run(fn, p)
            if r is None or len(r) != len(O) or len(r[0]) != len(O[0]): mids = None; break
            mids.append(r)
        if mids is None or mids == outs: continue
        if same and all(m == i for m, i in zip(mids, ins)): continue
        k = repr(mids)
        if k in seen: continue
        seen.add(k)
        score = sum(agree(m, o) for m, o in zip(mids, outs)) - base
        cands.append((-score, cost, name, fn, mids))
    cands.sort(key=lambda c: (c[0], c[1]))
    found = []; n = 0
    for _, cost, name, fn, mids in cands:
        if budget.left() < 2 * WU_S or n >= max_steps: break
        budget.spend(sum(W_RUN + W_RUN_A * len(t) * len(t[0]) for t in tests))
        tm = [gdsl.run(fn, t) for t in tests]
        if any(t is None for t in tm): continue
        # first step followed by a whole-grid dihedral transform (two library steps).  The library already
        # composes its reshapers with every D8 element, so that case would only duplicate a library program.
        if not name.startswith(gdsl.RESHAPERS):
            for dk, f in gdsl.D8.items():
                if dk != 'id' and all(f(m) == o for m, o in zip(mids, outs)):
                    found.append((cost + 2.0, f"{name} ; dihedral:{dk}", [f(t) for t in tm]))
            if found: break
        n += 1
        sub = Budget(min(budget.left() - WU_S, 2 * WU_S), budget)
        for c, prog, preds in learn_objmap(mids, outs, tm, sub, segs=('c4', 'c8', 'm8'), tag=name + ' ; '):
            found.append((c + cost + 1.0, prog, preds))
        if found: break
    return found


def lib_candidates(train):
    """gdsl.candidates(train) restricted to the families with a deterministic internal work bound (LIB_SKIP
    excluded; same program order and costs otherwise)."""
    out = []
    for fam in gdsl.FAMILIES:
        if fam.__name__ in LIB_SKIP: continue
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(gdsl.compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out


def lib_search(task, budget, max_programs=6):
    """gdsl.search(task, max_programs, allow2=False) bounded by WORK instead of time: the library's candidate
    programs (lib_candidates) are consumed in their own (cost) order, at most LIB_MAX_PROGS of them and only while
    the budget lasts; generating the candidates and applying a program to a pair are charged.  Identical to
    gdsl.search over those families whenever neither bound binds."""
    train = task['train']; outs = [p['output'] for p in train]
    found = []
    ident = [p['input'] for p in train]
    m = gdsl.fit_cmap(ident, outs)
    if m and any(k != v for k, v in m.items()) and all(gdsl.apply_cmap(a, m) == b for a, b in zip(ident, outs)):
        found.append(("colour-map", 1, lambda g, m=m: gdsl.apply_cmap(g, m)))
    area = sum(len(p['input']) * len(p['input'][0]) for p in train)
    cands = lib_candidates(train)
    budget.spend(W_LIB_GEN + W_LIB_GEN_A * area + W_LIB_PROG * len(cands))
    for name, cost, fn in cands[:LIB_MAX_PROGS]:
        if budget.over(): break
        preds = []
        for p, o in zip(train, outs):
            budget.spend(W_RUN + W_RUN_A * len(p['input']) * len(p['input'][0]))
            r = gdsl.run(fn, p["input"])
            if r is None or len(r) != len(o) or len(r[0]) != len(o[0]): preds = None; break
            if not preds and gdsl.fit_cmap([r], [o]) is None: preds = None; break
            preds.append(r)
        if not preds: continue
        if preds == outs:
            found.append((name, cost, fn))
        else:
            m = gdsl.fit_cmap(preds, outs)
            if m and all(gdsl.apply_cmap(a, m) == b for a, b in zip(preds, outs)):
                found.append((name + "+colour-map", cost + 1, lambda g, fn=fn, m=m: gdsl.apply_cmap(gdsl.run(fn, g), m)))
        if len(found) >= max_programs: break
    res, seen = [], set()
    for name, cost, fn in sorted(found, key=lambda x: x[1]):
        preds = [gdsl.run(fn, t["input"]) for t in task["test"]]
        if any(p is None for p in preds): continue
        k = str(preds)
        if k in seen: continue
        seen.add(k); res.append({"program": name, "cost": cost, "preds": preds})
    return res


def compose_last(task, near, budget, k=2, lib=True):
    """object map (near miss: progress on every pair) -> a LAST step learned on the residual pairs
    (rendered, output): first a second object map (cheap), then the library search."""
    train = task['train']; tests = [t['input'] for t in task['test']]
    outs = [p['output'] for p in train]
    found = []; done = []
    for cost, rules, seg, paint, rends in sorted(near, key=lambda r: r[0]):
        if budget.left() < 3 * WU_S or len(done) >= k: break
        if len(rules) > 2: continue
        key = repr(rends)
        if key in done: continue
        done.append(key)
        tr = []
        for g in tests:
            c = Ctx(g, grid_bg(g, seg[1]), seg[0], budget)
            budget.spend(W_FEAT * len(c.objs) + W_FEAT2 * len(c.objs) ** 2 + W_REND + W_REND_A * c.h * c.w +
                         W_REND_O * len(c.objs))
            ds = assign([c.feat(o) for o in c.objs], rules)
            r = render(c, ds, paint) if ds is not None else None
            if r is None: tr = None; break
            tr.append(r)
        if tr is None: continue
        head = fmt_rules(seg, rules, paint)
        sub_b = Budget(min(3 * WU_S, budget.left() - WU_S), budget)
        for c2, prog2, preds2 in learn_objmap(rends, outs, tr, sub_b):
            if prog2.count(' -> ') + len(rules) > 3: continue     # two object maps: at most 3 rules in all
            found.append((cost + c2 + 1.0, head + ' ; ' + prog2, preds2))
        if found or not lib or budget.left() < 7 * WU_S: continue
        sub = {'train': [{'input': r, 'output': O} for r, O in zip(rends, outs)], 'test': [{'input': r} for r in tr]}
        try:
            res = lib_search(sub, budget, max_programs=2)
        except Exception:
            res = []
        for r in res:
            if r['program'].startswith('codex'): continue        # library programs of our own families only
            found.append((cost + r.get('cost', 3) + 2.0, head + ' ; ' + r['program'], r['preds']))
    return found


def SEARCH(task, budget_s=15.0, compose=True):
    """budget_s: work budget in reference-CPU seconds (WU_S work units each); deterministic, no clock."""
    budget = Budget(budget_s * WU_S)
    train = task['train']; tests = [t['input'] for t in task['test']]
    res = []; near = []
    same = same_shape_task(train)
    # defer to the library when a whole-grid dihedral transform already explains every pair
    for k, f in gdsl.D8.items():
        if all(f(p['input']) == p['output'] for p in train): return []
    m = gdsl.fit_cmap([p['input'] for p in train], [p['output'] for p in train])
    if m and all(gdsl.apply_cmap(p['input'], m) == p['output'] for p in train): return []
    if same:
        ins = [p['input'] for p in train]; outs = [p['output'] for p in train]
        res = learn_objmap(ins, outs, tests, budget, near=near)
    if compose and not res and budget.left() > 4 * WU_S:
        res = compose_first(task, budget)
    if compose and not res and near and budget.left() > 3 * WU_S:
        res = compose_last(task, near, budget)
    res.sort(key=lambda r: r[0])
    out, seen = [], set()
    for cost, prog, preds in res:
        k = repr(preds)
        if k in seen: continue
        seen.add(k); out.append({'program': prog, 'preds': preds})
        if len(out) >= 3: break
    return out


In [ ]:
%%writefile /kaggle/working/probe/fam_combine_boolean_panels.py
"""combine.boolean_panels (+ combine.region_select_panels): per-cell table over aligned source panels.

A grid yields k aligned "source" layers (separator panels, equal halves/thirds, or the grid and one of its
mirrors).  Every output cell is a function T(key) of the corresponding source cells, where the key is
  bits   : which sources are foreground at that cell (boolean combination, k<=3), or
  region : which side of a divider drawn in the first source the cell lies on (divider / marker side / other).
T maps each key to one value option induced from the training pairs:
  copy source i | source i overlaid on source j | a constant output colour.
Output placement: the panel itself, or the input with its single blank panel filled in.
Background: one task-level colour (most frequent over all training sources) or per grid.

Why gdsl.fam_panels misses these tasks: it takes bg_of(whole grid) per grid, which flips to the ink colour
whenever ink outnumbers background in some pair (fafffa47, 1b2d62fb, 94f9d214, ...); it only allows a constant
"on" colour (not copy/overlay per key, d47aa2ff) and never pairs a grid with its own mirror (ce039d91, a8610ef7).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
import gdsl
from gdsl import H, W, colours, split_panels

MIRRORS = {"fh": gdsl.fh, "fv": gdsl.fv, "r180": gdsl.r180, "T": gdsl.T, "aT": gdsl.at}


def _cut(g, axis, k):
    h, w = H(g), W(g)
    if axis == "LR":
        if w % k or w // k < 2: return None
        s = w // k; return [[r[i * s:(i + 1) * s] for r in g] for i in range(k)]
    if h % k or h // k < 2: return None
    s = h // k; return [g[i * s:(i + 1) * s] for i in range(k)]


def _uniform(p): return len(colours(p)) == 1


def sources(g, mode):
    """-> (list of source layers, target index or None).  target = index of the blank panel to fill."""
    kind = mode[0]
    if kind == "sep":
        sp = split_panels(g)
        if not sp: return None
        ps = sp[0]
        if mode[1] == "fill":
            blank = [i for i, p in enumerate(ps) if _uniform(p)]
            if len(blank) != 1 or len(ps) < 3: return None
            return [p for i, p in enumerate(ps) if i != blank[0]], blank[0]
        return ps, None
    if kind == "cut":
        ps = _cut(g, mode[1], mode[2]); return (ps, None) if ps else None
    if kind == "mirror":
        if mode[1] in ("T", "aT") and H(g) != W(g): return None
        return [g, MIRRORS[mode[1]](g)], None
    return None


def place_fill(g, panel):
    """Write panel into the blank separator panel of g."""
    sp = split_panels(g)
    h, w = H(g), W(g); sc = sp[1]
    rows = [-1] + [r for r in range(h) if all(v == sc for v in g[r])] + [h]
    cols = [-1] + [c for c in range(w) if all(g[r][c] == sc for r in range(h))] + [w]
    out = [r[:] for r in g]
    for a in range(len(rows) - 1):
        for b in range(len(cols) - 1):
            y0, y1, x0, x1 = rows[a] + 1, rows[a + 1], cols[b] + 1, cols[b + 1]
            if y1 - y0 < 2 or x1 - x0 < 2: continue
            if len({g[y][x] for y in range(y0, y1) for x in range(x0, x1)}) == 1:
                for y in range(y0, y1):
                    for x in range(x0, x1): out[y][x] = panel[y - y0][x - x0]
                return out
    return None


# ---------------------------------------------------------------- per-cell keys
def keys_bits(ps, bg):
    return [[tuple(int(p[y][x] != bg) for p in ps) for x in range(W(ps[0]))] for y in range(H(ps[0]))]


def keys_region(ps, bg):
    """Divider = largest 8-connected ink component of source 0; other ink = markers.
    Cells not on the divider split into 4-connected regions; regions holding a marker -> 'A', others -> 'B'."""
    m = ps[0]; h, w = H(m), W(m)
    comps = gdsl.objects(m, bg, diag=True, by_colour=False)
    if len(comps) < 2: return None
    comps.sort(key=len, reverse=True)
    if len(comps[0]) == len(comps[1]): return None
    div = set(comps[0]); marks = {c for cc in comps[1:] for c in cc}
    lab = [[None] * w for _ in range(h)]
    for y, x in div: lab[y][x] = "D"
    seen = set(div); nA = nB = 0
    for sy in range(h):
        for sx in range(w):
            if (sy, sx) in seen: continue
            st, comp = [(sy, sx)], []; seen.add((sy, sx))
            while st:
                y, x = st.pop(); comp.append((y, x))
                for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen:
                        seen.add((yy, xx)); st.append((yy, xx))
            t = "A" if any(c in marks for c in comp) else "B"
            nA += t == "A"; nB += t == "B"
            for y, x in comp: lab[y][x] = t
    if not nA or not nB: return None
    return lab


FEATURES = {"bits": keys_bits, "region": keys_region}


def value(opt, ps, bg, y, x):
    if opt[0] == "copy": return ps[opt[1]][y][x]
    if opt[0] == "over":
        v = ps[opt[1]][y][x]; return v if v != bg else ps[opt[2]][y][x]
    return opt[1]


def options(k, feat, out_cols):
    lo = 1 if feat == "region" else 0          # region: source 0 is the mask, values come from the rest
    ops = [("copy", i) for i in range(lo, k)]
    ops += [("over", i, j) for i in range(lo, k) for j in range(lo, k) if i != j]
    ops += [("const", c) for c in sorted(out_cols)]
    return ops


def grid_bg(ps):
    return Counter(v for p in ps for r in p for v in r).most_common(1)[0][0]


def induce(train, mode, feat, bgpol):
    data = []
    for p in train:
        s = sources(p["input"], mode)
        if not s: return None
        ps, tgt = s
        o = p["output"]
        if tgt is None:
            tgt_out = o
        else:
            if (H(o), W(o)) != (H(p["input"]), W(p["input"])): return None
            so = split_panels(o)
            if not so or len(so[0]) != len(ps) + 1: return None
            tgt_out = so[0][tgt]
        if any((H(q), W(q)) != (H(tgt_out), W(tgt_out)) for q in ps): return None
        if feat == "bits" and len(ps) > 3: return None
        if feat == "region" and len(ps) < 3: return None
        data.append((ps, tgt_out))
    if bgpol == "task":
        pooled = Counter(v for ps, _ in data for q in ps for r in q for v in r)
        bg = pooled.most_common(1)[0][0]
        if not all(any(bg in colours(q) for q in ps) for ps, _ in data): return None
    k = len(data[0][0])
    if any(len(ps) != k for ps, _ in data): return None
    out_cols = set.intersection(*[colours(o) for _, o in data]) | {v for _, o in data for r in o for v in r}
    cand = {}
    for ps, o in data:
        b = bg if bgpol == "task" else grid_bg(ps)
        ks = FEATURES[feat](ps, b)
        if ks is None: return None
        for y in range(H(o)):
            for x in range(W(o)):
                key = ks[y][x]
                if key not in cand: cand[key] = options(k, feat, out_cols)
                cand[key] = [op for op in cand[key] if value(op, ps, b, y, x) == o[y][x]]
                if not cand[key]: return None
    table = {key: ops[0] for key, ops in cand.items()}
    # reject trivial tables: every key copies the same source (output == a source)
    if len({v for v in table.values()}) == 1 and table[next(iter(table))][0] == "copy": return None
    return table, (bg if bgpol == "task" else None), k


def make(mode, feat, table, tbg, k):
    def fn(g):
        s = sources(g, mode)
        if not s: return None
        ps, tgt = s
        if len(ps) != k or any((H(q), W(q)) != (H(ps[0]), W(ps[0])) for q in ps): return None
        b = tbg if tbg is not None else grid_bg(ps)
        ks = FEATURES[feat](ps, b)
        if ks is None: return None
        out = []
        for y in range(H(ps[0])):
            row = []
            for x in range(W(ps[0])):
                op = table.get(ks[y][x])
                if op is None: return None
                row.append(value(op, ps, b, y, x))
            out.append(row)
        return out if tgt is None else place_fill(g, out)
    return fn


MODES = [("sep", "all"), ("sep", "fill")] + [("cut", a, k) for a in ("LR", "TB") for k in (2, 3)] + \
        [("mirror", m) for m in MIRRORS]


def fam_panel_table(train):
    i, o = train[0]["input"], train[0]["output"]
    same = (H(i), W(i)) == (H(o), W(o))
    for mode in MODES:
        if mode == ("sep", "fill") or mode[0] == "mirror":
            if not same: continue
        elif same: continue
        for feat in ("bits", "region"):
            seen = set()
            for bgpol in ("task", "grid"):
                r = induce(train, mode, feat, bgpol)
                if not r: continue
                table, tbg, k = r
                sig = str(sorted(table.items()))
                if sig in seen: continue
                seen.add(sig)
                cost = 3 + (feat == "region") + (bgpol == "grid") + (len(table) > 4)
                name = f"panel-table[{':'.join(map(str, mode))},{feat},bg={bgpol}]"
                yield (name, cost, make(mode, feat, table, tbg, k))


FAMILIES = (fam_panel_table,)


In [ ]:
%%writefile /kaggle/working/probe/fam_connect_aligned_pair.py
"""connect.aligned_pair (+ ray.full_line): straight-line drawing between / through items.

fam_connect_pairs
  Items = same-colour 8-connected objects.  Two items are a pair when their centres lie on a
  common row / column / diagonal (align set induced), they match on an induced key (same shape,
  same colour, or any), and no other item of the same class lies on the segment between them
  (nearest facing partner).  The segment is the run of line cells strictly between the last cell
  of item A and the first cell of item B.  Induced options:
     clear    - segment must be all background (otherwise the pair is not joined)
     paint    - own colour | constant k | halves (A-colour half, k at the midpoint, B-colour half)
     over     - paint background only | paint over everything
     filter   - all items | items on opposite grid borders (line spans the grid)
     base     - keep the input | blank canvas (only joined pairs + segments survive)
     emit     - foreign cells lying on a segment emit a full perpendicular line of their colour
fam_full_lines
  Seeds (objects filtered by an induced shape class) paint their full row and/or column through
  their centre.  Axis: both | row | col | orientation of the seed (horizontal/vertical segment) |
  per-colour table induced from training.  Line colour own | constant.  Crossings of a row line and
  a column line from different seeds: row wins | column wins | constant colour.  Layer: paint
  background only | everything but seeds | everything.  Clip: whole grid | the region (enclosing
  component) around the seed, line across its short side | crop to the largest one-colour
  component's bbox first.
Every parameter combination is verified internally on all training pairs; only exact fits are
yielded (bounded)."""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from itertools import product
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox


def _same_shape_io(train):
    return all((H(p['input']), W(p['input'])) == (H(p['output']), W(p['output'])) for p in train)


def _changed_colours(train, n=4):
    """Colours painted onto changed cells, most widely used first (pairs containing it, then count)."""
    cs = Counter(); npair = Counter()
    for p in train:
        i, o = p['input'], p['output']
        if (H(i), W(i)) != (H(o), W(o)): continue
        here = set()
        for y in range(H(i)):
            for x in range(W(i)):
                if i[y][x] != o[y][x]: cs[o[y][x]] += 1; here.add(o[y][x])
        for c in here: npair[c] += 1
    incol = {v for p in train for r in p['input'] for v in r}
    return sorted(cs, key=lambda c: (-npair[c], c in incol, -cs[c]))[:n]


# ------------------------------------------------------------------ connect pairs
class _Item:
    __slots__ = ('col', 'cells', 'r2', 'c2', 'shape', 'bb')

    def __init__(self, g, cells):
        self.cells = set(cells); self.col = g[cells[0][0]][cells[0][1]]
        self.bb = bbox(cells); r0, c0, r1, c1 = self.bb
        self.r2, self.c2 = r0 + r1, c0 + c1
        self.shape = tuple(sorted((y - r0, x - c0) for y, x in cells))


def _line_cells(a, b):
    """Integer cells on the line through the two centres, ordered from a to b, strictly between centres
    (endpoints included when integer).  None when not aligned."""
    dr, dc = b.r2 - a.r2, b.c2 - a.c2
    if dr == 0 and dc == 0: return None
    if dr == 0:
        if a.r2 % 2: return None
        y = a.r2 // 2; s = 1 if dc > 0 else -1
        xs = range(-(-a.c2 // 2) if s > 0 else a.c2 // 2, (b.c2 // 2) + s if s > 0 else -(-b.c2 // 2) + s, s)
        return 'orth', [(y, x) for x in xs]
    if dc == 0:
        if a.c2 % 2: return None
        x = a.c2 // 2; s = 1 if dr > 0 else -1
        ys = range(-(-a.r2 // 2) if s > 0 else a.r2 // 2, (b.r2 // 2) + s if s > 0 else -(-b.r2 // 2) + s, s)
        return 'orth', [(y, x) for y in ys]
    if abs(dr) != abs(dc): return None
    if (a.r2 - a.c2) % 2 and (a.r2 + a.c2) % 2: return None
    sr = 1 if dr > 0 else -1; sc = 1 if dc > 0 else -1
    out = []
    y = -(-a.r2 // 2) if sr > 0 else a.r2 // 2
    while (2 * y - b.r2) * sr <= 0:
        x2 = a.c2 + sc * sr * (2 * y - a.r2)
        if x2 % 2 == 0: out.append((y, x2 // 2))
        y += sr
    return 'diag', out


_IC = {}


def _items(g):
    kk = tuple(map(tuple, g))
    if kk not in _IC:
        if len(_IC) > 64: _IC.clear()
        bg = bg_of(g); _IC[kk] = (bg, [_Item(g, cs) for cs in objects(g, bg, True, True)])
    return _IC[kk]


def _pairs(g, key, align, clear, filt, ic=None):
    h, w = H(g), W(g)
    bg, items = _items(g)
    if ic is not None and not any(it.col == ic for it in items): return None
    if len(items) > 80: return None
    owner = {}
    for k, it in enumerate(items):
        for c in it.cells: owner[c] = k
    kf = {'shape': lambda it: (it.col, it.shape), 'colour': lambda it: it.col, 'any': lambda it: 0}[key]
    if filt == 'border':
        def onb(it):
            return {e for y, x in it.cells for e, t in (('t', y == 0), ('b', y == h - 1), ('l', x == 0), ('r', x == w - 1)) if t}
        eb = [onb(it) for it in items]
    res = []
    for ia in range(len(items)):
        for ib in range(ia + 1, len(items)):
            a, b = items[ia], items[ib]
            if ic is not None and (a.col != ic or b.col != ic): continue
            if kf(a) != kf(b): continue
            if filt == 'border':
                ea, ebb = eb[ia], eb[ib]
                if not (('t' in ea and 'b' in ebb) or ('b' in ea and 't' in ebb) or ('l' in ea and 'r' in ebb) or ('r' in ea and 'l' in ebb)):
                    continue
            lc = _line_cells(a, b)
            if lc is None: continue
            kind, cells = lc
            if align == 'row' and a.r2 != b.r2 or align == 'col' and a.c2 != b.c2: continue
            if align in ('orth', 'diag') and kind != align: continue
            ia_last = max([k for k, c in enumerate(cells) if c in a.cells], default=-1)
            ib_first = min([k for k, c in enumerate(cells) if c in b.cells], default=len(cells))
            seg = cells[ia_last + 1:ib_first]
            if not seg: continue
            if any(c in a.cells or c in b.cells for c in seg): continue
            blocked = False
            for c in seg:
                o = owner.get(c)
                if o is not None and o not in (ia, ib) and kf(items[o]) == kf(a): blocked = True; break
                if clear and g[c[0]][c[1]] != bg: blocked = True; break
            if blocked: continue
            res.append((a, b, kind, seg))
    return bg, res


def _render(g, pr, paint, k, over, base, emit):
    bg, prs = pr
    out = [r[:] for r in g] if base == 'keep' else [[bg] * W(g) for _ in range(H(g))]
    if base == 'blank':
        for a, b, _, _ in prs:
            for it in (a, b):
                for y, x in it.cells: out[y][x] = it.col
    emitters = []
    for a, b, kind, seg in prs:
        n = len(seg)
        for j, (y, x) in enumerate(seg):
            if paint == 'own': col = a.col
            elif paint == 'const': col = k
            else:
                if n % 2 == 0: return None
                col = a.col if j < n // 2 else (b.col if j > n // 2 else k)
            if g[y][x] != bg and g[y][x] not in (a.col,):
                emitters.append((y, x, kind, a, b))
            if over == 'bg' and g[y][x] != bg: continue
            out[y][x] = col
    if emit != 'none':
        for y, x, kind, a, b in emitters:
            if kind == 'orth':
                dirs = [(0, 1), (0, -1)] if a.r2 != b.r2 else [(1, 0), (-1, 0)]
            else:
                sr = 1 if b.r2 > a.r2 else -1; sc = 1 if b.c2 > a.c2 else -1
                dirs = [(sr, -sc), (-sr, sc)]
            col = g[y][x]
            for dy, dx in dirs:
                yy, xx = y + dy, x + dx
                while 0 <= yy < H(g) and 0 <= xx < W(g):
                    if emit == 'all' or out[yy][xx] == bg: out[yy][xx] = col
                    yy += dy; xx += dx
    return out


def fam_connect_pairs(train):
    if not _same_shape_io(train): return
    if any(p['input'] == p['output'] for p in train): return
    ks = _changed_colours(train)
    ins = [p['input'] for p in train]; outs = [p['output'] for p in train]
    found = 0
    common = set.intersection(*[{v for r in g for v in r} - {bg_of(g)} for g in ins])
    for ic, key, align, clear, filt in product(sorted(common) + [None], ('shape', 'colour', 'any'), ('orth', 'diag', 'both', 'row', 'col'), (True, False), ('all', 'border')):
        if ic is not None and key == 'any': continue
        prs = []
        for g in ins:
            pr = _pairs(g, key, align, clear, filt, ic)
            if pr is None or not pr[1]: prs = None; break
            prs.append(pr)
        if prs is None: continue
        fgseg = any(g[y][x] != pr[0] for g, pr in zip(ins, prs) for *_, seg in pr[1] for y, x in seg)
        paints = [('own', None)] + [('const', k) for k in ks] + [('halves', k) for k in ks]
        for (paint, k), over, base, emit in product(paints, ('bg', 'all'), ('keep', 'blank'), ('none', 'bg', 'all')):
            if base == 'blank' and filt != 'border': continue
            if not fgseg and (over == 'all' or emit != 'none'): continue
            if all(_render(g, pr, paint, k, over, base, emit) == o for g, pr, o in zip(ins, prs, outs)):
                def fn(g, key=key, align=align, clear=clear, filt=filt, paint=paint, k=k, over=over, base=base, emit=emit, ic=ic):
                    pr = _pairs(g, key, align, clear, filt, ic)
                    if pr is None or not pr[1]: return None
                    return _render(g, pr, paint, k, over, base, emit)
                yield (f"connect-pairs[items={ic},key={key},align={align},clear={int(clear)},filt={filt},paint={paint}{'' if k is None else k},"
                       f"over={over},base={base},emit={emit}]", 4, fn)
                found += 1
                if found >= 6: return


# ------------------------------------------------------------------ full lines
def _seed_ok(cls, cells):
    r0, c0, r1, c1 = bbox(cells); hh, ww = r1 - r0 + 1, c1 - c0 + 1; n = len(cells)
    if cls == 'all': return True
    if cls == 'single': return n == 1
    if cls == 'segment': return n >= 2 and (hh == 1 or ww == 1)
    if cls == 'block': return hh >= 2 and ww >= 2 and n == hh * ww
    if cls == 'nonblock': return not (hh >= 2 and ww >= 2 and n == hh * ww) and n < 0.2 * 900
    return False


def _crop_main(g):
    """Bbox of the largest one-colour component, trimmed of edge rows/cols that are mostly another colour."""
    obs = objects(g, -1, False, True)
    big = max(obs, key=len); c = g[big[0][0]][big[0][1]]
    r0, c0, r1, c1 = bbox(big)
    frac = lambda cells: sum(g[y][x] == c for y, x in cells) / len(cells)
    while r1 > r0 and c1 > c0:
        e = [(frac([(r0, x) for x in range(c0, c1 + 1)]), 't'), (frac([(r1, x) for x in range(c0, c1 + 1)]), 'b'),
             (frac([(y, c0) for y in range(r0, r1 + 1)]), 'l'), (frac([(y, c1) for y in range(r0, r1 + 1)]), 'r')]
        f, side = min(e)
        if f >= 0.5: break
        if side == 't': r0 += 1
        elif side == 'b': r1 -= 1
        elif side == 'l': c0 += 1
        else: c1 -= 1
    if r1 - r0 < 1 or c1 - c0 < 1: return None
    return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]


def _seeds(g, cls, clip):
    bg = bg_of(g)
    obs = objects(g, bg, False, True)
    if len(obs) > 60: return None
    seeds = []
    for cs in obs:
        if not _seed_ok(cls, cs): continue
        r0, c0, r1, c1 = bbox(cs)
        col = g[cs[0][0]][cs[0][1]]
        hh, ww = r1 - r0 + 1, c1 - c0 + 1
        orient = 'row' if hh == 1 and ww > 1 else 'col' if ww == 1 and hh > 1 else 'both'
        ry = (r0 + r1) / 2; cx = (c0 + c1) / 2
        region = (0, 0, H(g) - 1, W(g) - 1)
        if clip == 'region':
            if len(cs) != 1: continue
            y, x = cs[0]
            nb = Counter(g[yy][xx] for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1))
                         if 0 <= yy < H(g) and 0 <= xx < W(g) and g[yy][xx] != col)
            if not nb: return None
            rc = nb.most_common(1)[0][0]
            comp = [o for o in objects(g, -1, False, True) if g[o[0][0]][o[0][1]] == rc and any(abs(a - y) + abs(b - x) == 1 for a, b in o)]
            if not comp: return None
            region = bbox(max(comp, key=len))
        seeds.append({'cells': set(cs), 'col': col, 'orient': orient, 'r': ry, 'c': cx, 'region': region})
    return bg, seeds


def _axes(s, amode, table):
    if amode in ('row', 'col'): return {amode}
    if amode == 'both': return {'row', 'col'}
    if amode == 'orient': return {s['orient']} if s['orient'] != 'both' else {'row', 'col'}
    if amode == 'short':
        r0, c0, r1, c1 = s['region']
        return {'col'} if (r1 - r0) < (c1 - c0) else {'row'} if (r1 - r0) > (c1 - c0) else set()
    if amode == 'table':
        return table.get(s['col'])


def _draw_lines(g, sd, amode, table, lcol, cross, layer):
    bg, seeds = sd
    h, w = H(g), W(g)
    rowc, colc = {}, {}
    seedcells = {}
    for k, s in enumerate(seeds):
        for c in s['cells']: seedcells[c] = k
    for k, s in enumerate(seeds):
        ax = _axes(s, amode, table)
        if ax is None: return None
        col = s['col'] if lcol is None else lcol
        r0, c0, r1, c1 = s['region']
        if 'row' in ax:
            if s['r'] != int(s['r']): return None
            y = int(s['r'])
            for x in range(c0, c1 + 1): rowc.setdefault((y, x), (k, col))
        if 'col' in ax:
            if s['c'] != int(s['c']): return None
            x = int(s['c'])
            for y in range(r0, r1 + 1): colc.setdefault((y, x), (k, col))
    out = [r[:] for r in g]
    for cell in set(rowc) | set(colc):
        y, x = cell
        if layer == 'bg' and g[y][x] != bg: continue
        if layer == 'nonseed' and cell in seedcells: continue
        a, b = rowc.get(cell), colc.get(cell)
        if a and b and a[0] != b[0]:
            if cross == 'row': v = a[1]
            elif cross == 'col': v = b[1]
            elif cross == 'bg': v = bg
            else: v = cross
        else: v = (a or b)[1]
        out[y][x] = v
    return out


def _induce_table(ins, sds, outs, lcol):
    table = {}
    for g, sd, o in zip(ins, sds, outs):
        for s in sd[1]:
            col = s['col'] if lcol is None else lcol
            ax = set()
            r0, c0, r1, c1 = s['region']
            if s['r'] == int(s['r']):
                y = int(s['r']); n = c1 - c0 + 1
                if sum(o[y][x] == col for x in range(c0, c1 + 1)) * 2 > n: ax.add('row')
            if s['c'] == int(s['c']):
                x = int(s['c']); n = r1 - r0 + 1
                if sum(o[y][x] == col for y in range(r0, r1 + 1)) * 2 > n: ax.add('col')
            if table.setdefault(s['col'], ax) != ax: return None
    if len(set(map(frozenset, table.values()))) < 2: return None
    return table


def fam_full_lines(train):
    ins = [p['input'] for p in train]; outs = [p['output'] for p in train]
    if any(i == o for i, o in zip(ins, outs)): return
    crops = [False]
    if not _same_shape_io(train):
        cr = [_crop_main(g) for g in ins]
        if any(c is None or (H(c), W(c)) != (H(o), W(o)) for c, o in zip(cr, outs)): return
        crops = [True]; ins = cr
    ks = _changed_colours([{'input': i, 'output': o} for i, o in zip(ins, outs)])
    found = 0
    for crop in crops:
        for cls, clip in product(('all', 'single', 'segment', 'block', 'nonblock'), ('none', 'region')):
            sds = [_seeds(g, cls, clip) for g in ins]
            if any(sd is None or not sd[1] for sd in sds): continue
            amodes = ('both', 'row', 'col', 'orient', 'table') if clip == 'none' else ('short',)
            for amode, lcol in product(amodes, [None] + ks):
                table = None
                if amode == 'table':
                    table = _induce_table(ins, sds, outs, lcol)
                    if table is None: continue
                for cross, layer in product(['row', 'col', 'bg'] + ks, ('bg', 'nonseed', 'all')):
                    ok = True
                    for g, sd, o in zip(ins, sds, outs):
                        if _draw_lines(g, sd, amode, table, lcol, cross, layer) != o: ok = False; break
                    if not ok: continue
                    def fn(g, crop=crop, cls=cls, clip=clip, amode=amode, table=table, lcol=lcol, cross=cross, layer=layer):
                        if crop:
                            g = _crop_main(g)
                            if g is None: return None
                        sd = _seeds(g, cls, clip)
                        if sd is None or not sd[1]: return None
                        return _draw_lines(g, sd, amode, table, lcol, cross, layer)
                    yield (f"full-lines[crop={int(crop)},seed={cls},clip={clip},axis={amode},col={lcol},cross={cross},layer={layer}]", 4, fn)
                    found += 1
                    if found >= 6: return


FAMILIES = (fam_connect_pairs, fam_full_lines)


In [ ]:
%%writefile /kaggle/working/probe/fam_crop_by_score.py
"""crop.by_score: segment the input into units, pick the unit that is the arg-extremum of a score,
output a part of it (bbox / interior / bbox-minus-marker / enclosed content).

One parametrised primitive  crop-by-score[units, score, extremum, part]:
  units    : c4 | c8 (single-colour comps) | m4 | m8 (multi-colour comps) | k2 (clusters, gap<=1)
             | frames (rectangles with a complete one-colour border) | panels (separator lines) | halves
  score    : area | bbox-area | n-colours | minority | count[c] | n-comps[c] | n-shape[c,S]
             | near[c,k] (marker cells within k of the unit, unit itself not of colour c) | has[c]
  extremum : max | min (unique winner required) | all (union of every unit with has[c])
  part     : bbox | inner (bbox shrunk by 1) | nomark[c] (marker colour erased, bbox of rest) | content
All parameters (unit kind, colour c, shape S, distance k, extremum, part) are induced: a program is
only yielded when on every training pair the chosen unit's part equals the output.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, crop

# ------------------------------------------------------------------ units
def _rect_cells(r0, c0, r1, c1):
    return [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)]

def _clusters(g, bg, gap):
    h, w = H(g), W(g); seen = set(); out = []
    fg = {(y, x) for y in range(h) for x in range(w) if g[y][x] != bg}
    d = gap + 1
    for s in fg:
        if s in seen: continue
        seen.add(s); st = [s]; cells = []
        while st:
            y, x = st.pop(); cells.append((y, x))
            for dy in range(-d, d + 1):
                for dx in range(-d, d + 1):
                    q = (y + dy, x + dx)
                    if q in fg and q not in seen: seen.add(q); st.append(q)
        out.append(cells)
    return out

def _frames(g, bg):
    """Rectangles >=3x3 whose whole border is one non-bg colour (noise elsewhere allowed); maximal only."""
    h, w = H(g), W(g)
    R = [[0] * (w + 1) for _ in range(h + 1)]; D = [[0] * (w + 1) for _ in range(h + 1)]
    for y in range(h - 1, -1, -1):
        for x in range(w - 1, -1, -1):
            v = g[y][x]
            R[y][x] = 1 + (R[y][x + 1] if x + 1 < w and g[y][x + 1] == v else 0)
            D[y][x] = 1 + (D[y + 1][x] if y + 1 < h and g[y + 1][x] == v else 0)
    rects = []
    for y in range(h):
        for x in range(w):
            if g[y][x] == bg or R[y][x] < 3 or D[y][x] < 3: continue
            for ww in range(R[y][x], 2, -1):
                for hh in range(D[y][x], 2, -1):
                    x1, y1 = x + ww - 1, y + hh - 1
                    if D[y][x1] >= hh and R[y1][x] >= ww:
                        rects.append((y, x, y1, x1))
    rects = [r for r in rects if not any(o != r and o[0] <= r[0] and o[1] <= r[1] and o[2] >= r[2] and o[3] >= r[3] and g[o[0]][o[1]] == g[r[0]][r[1]] for o in rects)]
    if len(rects) > 40: return []
    return [_rect_cells(*r) for r in rects]

def _panels(g):
    h, w = H(g), W(g)
    for sc in sorted({v for r in g for v in r}):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols or len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]; out = []
        for a in range(len(rb) - 1):
            for b in range(len(cb) - 1):
                if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1:
                    out.append(_rect_cells(rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1))
        if len(out) >= 2: return out
    return []

def _halves(g):
    h, w = H(g), W(g); out = []
    if w % 2 == 0: out += [_rect_cells(0, 0, h - 1, w // 2 - 1), _rect_cells(0, w // 2, h - 1, w - 1)]
    if h % 2 == 0: out += [_rect_cells(0, 0, h // 2 - 1, w - 1), _rect_cells(h // 2, 0, h - 1, w - 1)]
    return out

UNITS = ('m4', 'm8', 'c4', 'c8', 'k1', 'frames', 'panels', 'halves')
RECT_UNITS = ('frames', 'panels', 'halves')

def border_bg(g):
    h, w = H(g), W(g)
    cells = [g[0][x] for x in range(w)] + [g[h - 1][x] for x in range(w)] + [g[y][0] for y in range(h)] + [g[y][w - 1] for y in range(h)]
    return Counter(cells).most_common(1)[0][0]

BGS = {'mode': bg_of, 'border': border_bg}

def units(g, kind, bg):
    if kind == 'c4': return objects(g, bg, False, True)
    if kind == 'c8': return objects(g, bg, True, True)
    if kind == 'm4': return objects(g, bg, False, False)
    if kind == 'm8': return objects(g, bg, True, False)
    if kind == 'k1': return _clusters(g, bg, 1)
    if kind == 'frames': return _frames(g, bg)
    if kind == 'panels': return _panels(g)
    if kind == 'halves': return _halves(g)
    return []

# ------------------------------------------------------------------ parts
def part(g, u, how, bg):
    if how == 'bbox': return crop(g, bbox(u))
    if how == 'inner':
        r0, c0, r1, c1 = bbox(u)
        if r1 - r0 < 2 or c1 - c0 < 2: return None
        return crop(g, (r0 + 1, c0 + 1, r1 - 1, c1 - 1))
    if how.startswith('nomark'):
        c = int(how[6:])
        if not any(g[y][x] == c for y, x in u): return None
        rest = [(y, x) for y, x in u if g[y][x] != c]
        if not rest: return None
        return [[bg if v == c else v for v in r] for r in crop(g, bbox(rest))]
    if how == 'content':
        r0, c0, r1, c1 = bbox(u); us = set(u)
        sub = [[g[y][x] if (y, x) in us else bg for x in range(W(g))] for y in range(H(g))]
        inner = [ob for ob in objects(sub, bg, False, True)
                 if all(r0 < y < r1 and c0 < x < c1 for y, x in ob)]
        cells = [c for ob in inner for c in ob]
        if not cells: return None
        return crop(g, bbox(cells))
    return None

# ------------------------------------------------------------------ scores
def _shape(cells):
    r0, c0, _, _ = bbox(cells); return tuple(sorted((y - r0, x - c0) for y, x in cells))

def score(g, u, s, bg):
    """Numeric score of unit u, or None when the unit is not a candidate for this score."""
    kind = s[0]
    if kind == 'area': return sum(g[y][x] != bg for y, x in u)
    if kind == 'barea':
        r0, c0, r1, c1 = bbox(u); return (r1 - r0 + 1) * (c1 - c0 + 1)
    vals = [g[y][x] for y, x in u if g[y][x] != bg]
    if kind == 'ncol': return len(set(vals))
    if kind == 'minority':
        if not vals: return 0
        return len(vals) - Counter(vals).most_common(1)[0][1]
    c = s[1]
    if kind == 'cnt': return sum(v == c for v in vals)
    if kind == 'has': return int(c in vals)
    if kind == 'ncomp':
        cells = [(y, x) for y, x in u if g[y][x] == c]
        if not cells: return 0
        sub = [[bg] * W(g) for _ in range(H(g))]
        for y, x in cells: sub[y][x] = c
        return len(objects(sub, bg, False, True))
    if kind == 'nshape':   # occurrences of template S (all cells colour c) inside the unit
        us = set(u); S = s[2]
        return sum(all((y + dy, x + dx) in us and g[y + dy][x + dx] == c for dy, dx in S) for y, x in u)
    if kind == 'near':
        if c in vals: return None
        k = s[2]; r0, c0, r1, c1 = bbox(u)
        return sum(g[y][x] == c for y in range(max(0, r0 - k), min(H(g), r1 + k + 1))
                   for x in range(max(0, c0 - k), min(W(g), c1 + k + 1)))
    return None

def choose(g, us, s, ext, bg, pt=None):
    sc = [score(g, u, s, bg) if (pt is None or ext == 'all' or part(g, u, pt, bg) is not None) else None for u in us]
    idx = [i for i, v in enumerate(sc) if v is not None]
    if len(idx) < 2: return None
    if ext == 'all':
        hits = [i for i in idx if sc[i] == 1]
        if not hits or len(hits) == len(idx): return None
        return [c for i in hits for c in us[i]]
    key = (lambda i: sc[i]) if ext == 'max' else (lambda i: -sc[i])
    best = max(key(i) for i in idx)
    hits = [i for i in idx if key(i) == best]
    return us[hits[0]] if len(hits) == 1 else None

def _sname(s):
    return s[0] if len(s) == 1 else s[0] + '[' + ','.join('S' if isinstance(a, tuple) else str(a) for a in s[1:]) + ']'

# ------------------------------------------------------------------ family
def _programs(train, bgm):
    bgf = BGS[bgm]
    o0 = train[0]['output']
    bgs = [bgf(p['input']) for p in train]
    cols = sorted(set.intersection(*[{v for r in p['input'] for v in r} - {b} for p, b in zip(train, bgs)]))
    parts = ['bbox', 'inner', 'content'] + ['nomark%d' % c for c in cols]
    # score library (colour / shape / distance parameters drawn from the training pairs)
    lib = [('area',), ('barea',), ('ncol',), ('minority',)]
    for c in cols:
        lib += [('cnt', c), ('ncomp', c), ('has', c)] + [('near', c, k) for k in (1, 2, 3)]
        # templates: multi-cell colour-c shapes present as components in every training output
        shp = None
        for p in train:
            o = p['output']; ob = bg_of(o)
            here = {_shape(q) for q in objects(o, ob, False, True) if o[q[0][0]][q[0][1]] == c and len(q) > 1}
            shp = here if shp is None else shp & here
        if shp and len(shp) <= 3: lib += [('nshape', c, S) for S in sorted(shp)]
    for uk in UNITS:
        per = []
        for p, bg in zip(train, bgs):
            g = p['input']; us = units(g, uk, bg)
            if len(us) < 2 or len(us) > 60: per = None; break
            per.append((g, us, bg, p['output']))
        if not per: continue
        for pt in parts:
            if pt == 'content' and uk != 'k1': continue
            match = [{i for i, u in enumerate(us) if part(g, u, pt, bg) == o} for g, us, bg, o in per]
            single_ok = all(match)
            for s in lib:
                exts = ('max', 'min') + (('all',) if s[0] == 'has' else ())
                for ext in exts:
                    if ext != 'all' and not single_ok: continue
                    ok = True
                    for (g, us, bg, o), m in zip(per, match):
                        ch = choose(g, us, s, ext, bg, pt)
                        if ch is None: ok = False; break
                        if ext == 'all':
                            if part(g, ch, pt, bg) != o: ok = False; break
                        elif not any(ch is us[i] for i in m): ok = False; break
                    if not ok: continue
                    def fn(g, uk=uk, pt=pt, s=s, ext=ext, bgf=bgf):
                        bg = bgf(g); us = units(g, uk, bg)
                        if len(us) < 2: return None
                        ch = choose(g, us, s, ext, bg, pt)
                        return part(g, ch, pt, bg) if ch else None
                    yield (f"crop-by-score[{uk},{_sname(s)},{ext},{pt}{',bg=mode' if bgm == 'mode' else ''}]", 4, fn)

def fam_crop_by_score(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if H(o0) > H(i0) or W(o0) > W(i0) or (H(o0), W(o0)) == (H(i0), W(i0)): return
    for p in train:
        if H(p['output']) > H(p['input']) or W(p['output']) > W(p['input']): return
    bgmodes = ['border'] + (['mode'] if any(border_bg(p['input']) != bg_of(p['input']) for p in train) else [])
    n = 0
    for bgm in bgmodes:
        for prog in _programs(train, bgm):
            yield prog; n += 1
            if n >= 12: return

FAMILIES = (fam_crop_by_score,)


In [ ]:
%%writefile /kaggle/working/probe/fam_crop_fit_by_anchor.py
"""crop.fit_by_anchor: pieces carrying anchor cells are translated so that their anchors land on matching
anchor cells of a target (canvas / assembly); the result is drawn on the chosen output canvas.

One parametrised primitive  fit(target, match, out):
  target : rect    - a single-colour region whose bbox holds only tiny (<=2-cell) marker blobs; its markers
                     are the free anchors; pieces are the 8-connected multicolour components outside that bbox
                     (on the most common colour outside it); anchor colours = the marker colours
           | root  - an assembly started from the piece that contains a root colour (induced: the colour whose
                     piece never moves in training); anchors = the connector colour (least frequent colour
                     shared by every piece)
           | largest - as root, but the assembly starts from the largest piece
  match  : all     - every anchor of the piece lands on a free anchor of the same colour
           | two   - at least two anchors of the piece coincide with free anchors (connectors), the rest become
                     new free anchors; bodies may not overlap
           | comp  - complementary glue: the piece's anchors land on assembly body cells, and its body covers
                     assembly anchors (which take the covering colour); bodies may not overlap
  out    : target (target bbox) | full (input-size canvas on bg) | union (bbox of the assembly)
Pieces are placed greedily (unique best translation first) until none fits.  All parameters are enumerated
from these small domains or induced from training pairs; programs are verified on all pairs before yield.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def comps(g, ok, nb=N8, region=None):
    h, w = H(g), W(g); seen = set(); out = []
    for r in range(h):
        for c in range(w):
            if (r, c) in seen or not ok(r, c): continue
            st = [(r, c)]; seen.add((r, c)); cells = {}
            while st:
                y, x = st.pop(); cells[(y, x)] = g[y][x]
                for dy, dx in nb:
                    p = (y + dy, x + dx)
                    if 0 <= p[0] < h and 0 <= p[1] < w and p not in seen and ok(*p):
                        seen.add(p); st.append(p)
            out.append(cells)
    return out


def bb(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    return min(ys), min(xs), max(ys), max(xs)


# ------------------------------------------------------------------ targets
def rect_target(g):
    """Largest single-colour 4-component whose bbox contains only tiny marker blobs (>=1 marker)."""
    best = None
    for col in {v for r in g for v in r}:
        for cp in comps(g, lambda y, x, col=col: g[y][x] == col, N4):
            r0, c0, r1, c1 = bb(cp); area = (r1 - r0 + 1) * (c1 - c0 + 1)
            if area < 9 or len(cp) * 2 < area or r1 == r0 or c1 == c0: continue
            if best and area <= best[0]: continue
            inb = lambda y, x: r0 <= y <= r1 and c0 <= x <= c1 and g[y][x] != col
            marks = comps(g, inb, N4)
            if not marks or any(len(m) > 2 for m in marks): continue
            best = (area, col, (r0, c0, r1, c1))
    return best


def shared_colour(pieces):
    common = None; cnt = Counter()
    for p in pieces:
        cs = set(p.values()); cnt.update(p.values())
        common = cs if common is None else common & cs
    if not common or len(set(cnt)) < 2: return None
    return min(common, key=lambda c: (cnt[c], c))


def setup(g, target, root_col, bgc=None):
    """-> dict(canvas, free, body, pieces, acols, base_bbox, bg) or None."""
    if target == 'rect':
        t = rect_target(g)
        if not t: return None
        _, fill, (r0, c0, r1, c1) = t
        inside = lambda y, x: r0 <= y <= r1 and c0 <= x <= c1
        outc = Counter(g[y][x] for y in range(H(g)) for x in range(W(g)) if not inside(y, x))
        if not outc: return None
        sbg = outc.most_common(1)[0][0]
        pieces = comps(g, lambda y, x: not inside(y, x) and g[y][x] != sbg)
        if not pieces or len(pieces) > 30: return None
        canvas = {(y, x): g[y][x] for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)}
        free = {p: v for p, v in canvas.items() if v != fill}
        return dict(canvas=canvas, free=free, body=set(), pieces=pieces, acols=set(free.values()),
                    bbox=(r0, c0, r1, c1), bg=fill, clip=True)
    bg = bgc if bgc is not None and any(bgc in r for r in g) else bg_of(g)
    pieces = comps(g, lambda y, x: g[y][x] != bg)
    if len(pieces) < 2 or len(pieces) > 30: return None
    k = shared_colour(pieces)
    if k is None: return None
    if target == 'root':
        roots = [p for p in pieces if root_col in p.values()]
        if len(roots) != 1 or root_col == k: return None
        root = roots[0]
    else:
        sz = sorted(len(p) for p in pieces)
        if sz[-1] == sz[-2]: root = max(pieces, key=lambda p: (len(p), -min(p)[0], -min(p)[1]))
        else: root = max(pieces, key=len)
    pieces = [p for p in pieces if p is not root]
    free = {p: v for p, v in root.items() if v == k}
    body = {p for p, v in root.items() if v != k}
    return dict(canvas=dict(root), free=free, body=body, pieces=pieces, acols={k},
                bbox=(0, 0, H(g) - 1, W(g) - 1), bg=bg, clip=False, fdir=adirs(root, {k}))


def adirs(piece, acols):
    """For each anchor cell: the 4-directions in which its own piece's body lies."""
    return {(y, x): {(a, b) for a, b in N4 if (y + a, x + b) in piece and piece[(y + a, x + b)] not in acols}
            for (y, x), v in piece.items() if v in acols}


# ------------------------------------------------------------------ placement
def conn_moves(st, piece, pd):
    """Connector matching: translations where >=1 piece connector coincides with a free connector, the two
    bodies meet it from different sides, and nothing else collides.  -> [(hits, bridges, t)]"""
    pa = {p: v for p, v in piece.items() if v in st['acols']}
    pb = [p for p, v in piece.items() if v not in st['acols']]
    free, body, fd = st['free'], st['body'], st['fdir']
    h, w = st['bbox'][2] + 1, st['bbox'][3] + 1
    ts = {(f[0] - q[0], f[1] - q[1]) for q, v in pa.items() for f, fv in free.items() if fv == v}
    res = []
    for dy, dx in ts:
        ok = True; hit = br = 0
        for y, x in pb:
            p = (y + dy, x + dx)
            if not (0 <= p[0] < h and 0 <= p[1] < w) or p in body or p in free: ok = False; break
        if not ok: continue
        for (y, x), v in pa.items():
            p = (y + dy, x + dx)
            if not (0 <= p[0] < h and 0 <= p[1] < w) or p in body: ok = False; break
            if p in free:
                a, b = pd[(y, x)], fd.get(p, set())
                if free[p] != v or a & b: ok = False; break
                hit += 1; br += any((-u, -z) in b for u, z in a)
        if ok and hit: res.append((hit, br, (dy, dx)))
    return res



def assemble(st, limit=4000):
    """Search placements of all pieces (DFS); best = (placed, hits, bridges); None unless unique best."""
    pieces = st['pieces']; pds = [adirs(p, st['acols']) for p in pieces]
    best = [None, set()]; seen = set(); nodes = [0]

    def rec(st, todo, placed, score):
        key = frozenset(placed)
        if key in seen or nodes[0] > limit: return
        seen.add(key); nodes[0] += 1
        sc = (len(placed),) + score
        if best[0] is None or sc > best[0]: best[0] = sc; best[1] = {key}
        elif sc == best[0]: best[1].add(key)
        moves = [(hit, br, i, t) for i in todo for hit, br, t in conn_moves(st, pieces[i], pds[i])]
        if not moves: return
        top = max(m[:2] for m in moves)                  # branch only over the strongest joins
        for hit, br, i, t in moves:
            if (hit, br) != top: continue
            s2 = dict(canvas=dict(st['canvas']), free=dict(st['free']), body=set(st['body']),
                      fdir=dict(st['fdir']), acols=st['acols'], bbox=st['bbox'])
            place(s2, pieces[i], t, 'two')
            rec(s2, todo - {i}, placed | {(i, t)}, (score[0] + hit, score[1] + br))

    rec(st, frozenset(range(len(pieces))), frozenset(), (0, 0))

    if nodes[0] > limit or len(best[1]) != 1: return None
    (sol,) = best[1]
    for i, t in sol: place(st, pieces[i], t, 'two')
    return len(sol)


def candidates(st, piece, match, minhit=2):
    pa = {p: v for p, v in piece.items() if v in st['acols']}
    pb = {p: v for p, v in piece.items() if v not in st['acols']}
    free, body = st['free'], st['body']
    if not pa: return []
    ts = set()
    if match == 'comp':
        for f in free:
            for q in pb: ts.add((f[0] - q[0], f[1] - q[1]))
    else:
        for q, v in pa.items():
            for f, fv in free.items():
                if fv == v: ts.add((f[0] - q[0], f[1] - q[1]))
    r0, c0, r1, c1 = st['bbox']
    res = []
    for dy, dx in ts:
        mb = {(y + dy, x + dx) for y, x in pb}
        ma = {(y + dy, x + dx): v for (y, x), v in pa.items()}
        if match == 'comp':
            if mb & body or not all(p in body for p in ma): continue
            cov = sum(1 for p in free if p in mb)
            if cov == 0: continue
            adj = sum(1 for (y, x) in ma if any((y + a, x + b) in free for a, b in N4))
            res.append(((cov, adj), (dy, dx)))
            continue
        hit = sum(1 for p, v in ma.items() if free.get(p) == v)
        if match == 'two':                         # coinciding connectors must bridge the two bodies
            pd = adirs(piece, st['acols']); fd = st['fdir']
            if any(free.get((y + dy, x + dx)) == v and not any((-a, -b) in fd.get((y + dy, x + dx), ())
                                                              for a, b in pd[(y, x)])
                   for (y, x), v in pa.items()): continue
        if match == 'all':
            if hit != len(ma) or mb & body: continue
            if not st['clip'] and any(not (r0 <= y <= r1 and c0 <= x <= c1) for y, x in mb): continue
        else:
            if hit < minhit or mb & body or mb & set(free): continue
            if any(p in body for p in ma if free.get(p) != ma[p]): continue
            if any(not (r0 <= y <= r1 and c0 <= x <= c1) for y, x in list(mb) + list(ma)): continue
        res.append((hit, (dy, dx)))
    return res


def place(st, piece, t, match):
    dy, dx = t
    acols = st['acols']
    if 'fdir' in st:
        for (y, x), d in adirs(piece, acols).items(): st['fdir'][(y + dy, x + dx)] = d
    for (y, x), v in piece.items():
        p = (y + dy, x + dx)
        if v in acols:
            if match == 'comp': continue               # glue lands on assembly body: keep body colour
            if st['free'].get(p) == v: del st['free'][p]
            elif match == 'two': st['free'][p] = v
            st['canvas'][p] = v
        else:
            if match == 'comp' and p in st['free']: del st['free'][p]
            st['canvas'][p] = v; st['body'].add(p)


def fit(g, target, match, out, root_col=None, need_all=False, bgc=None):
    st = setup(g, target, root_col, bgc)
    if st is None: return None
    todo = list(st['pieces']); placed = 0
    minhit = 2
    if match == 'two':
        placed = assemble(st) or 0
        todo = todo[placed:] if placed else todo
    while todo and match != 'two':
        prog = False
        for piece in sorted(todo, key=lambda p: -len(p)):
            c = candidates(st, piece, match, minhit)
            if not c: continue
            m = max(h for h, _ in c); best = [t for h, t in c if h == m]
            if len(best) != 1: continue
            place(st, piece, best[0], match); todo.remove(piece); placed += 1; prog = True
            break
        if prog: minhit = 2
        elif match == 'two' and minhit == 2: minhit = 1   # fall back to a single bridging connector
        else: break
    if placed == 0 or (need_all and todo): return None
    canvas = st['canvas']
    if out == 'target':
        r0, c0, r1, c1 = st['bbox']
    elif out == 'full':
        r0, c0, r1, c1 = 0, 0, H(g) - 1, W(g) - 1
    else:
        r0, c0, r1, c1 = bb(canvas)
    if r1 - r0 >= 30 or c1 - c0 >= 30: return None
    res = [[st['bg']] * (c1 - c0 + 1) for _ in range(r1 - r0 + 1)]
    for (y, x), v in canvas.items():
        if r0 <= y <= r1 and c0 <= x <= c1: res[y - r0][x - c0] = v
    return res


# ------------------------------------------------------------------ induction
def root_colours(train):
    """Colours whose (unique) piece sits unchanged at the same place in every training output."""
    ok = None
    for p in train:
        i, o = p['input'], p['output']
        if (H(i), W(i)) != (H(o), W(o)): return set()
        bg = bg_of(i)
        pcs = comps(i, lambda y, x: i[y][x] != bg)
        cs = set()
        for c in {v for r in i for v in r} - {bg}:
            ps = [q for q in pcs if c in q.values()]
            if len(ps) == 1 and len(ps[0]) >= 3 and all(o[y][x] == v for (y, x), v in ps[0].items()):
                cs.add(c)
        ok = cs if ok is None else ok & cs
    return ok or set()


CONFIGS = (('rect', 'all', 'target'), ('root', 'two', 'full'), ('largest', 'two', 'full'),
           ('largest', 'comp', 'union'), ('largest', 'all', 'union'), ('largest', 'comp', 'full'))


def fam_fit_by_anchor(train):
    i0, o0 = train[0]['input'], train[0]['output']
    same = (H(i0), W(i0)) == (H(o0), W(o0))
    for target, match, out in CONFIGS:
        if out == 'full' and not same: continue
        if out != 'full' and same: continue
        roots = sorted(root_colours(train)) if target == 'root' else [None]
        for rc in roots[:3]:
          for bgc in ((None,) if target == 'rect' else (None, glob_bg(train))):
            for need in (True, False):
                fn = lambda g, a=target, b=match, c=out, rc=rc, nd=need, bgc=bgc: fit(g, a, b, c, rc, nd, bgc)
                good = True
                for p in train:
                    try: r = fn(p['input'])
                    except Exception: r = None
                    if r != p['output']: good = False; break
                if good:
                    yield (f"fit-by-anchor[{target},{match},{out}{',bg=' + str(bgc) if bgc is not None else ''}{',root=' + str(rc) if rc is not None else ''}"
                           f"{',all' if need else ''}]", 5, fn)
                    break
            else: continue
            break


def glob_bg(train):
    """Background role induced over the whole task: most common colour over all training inputs."""
    c = Counter(v for p in train for r in p['input'] for v in r)
    return c.most_common(1)[0][0]


FAMILIES = (fam_fit_by_anchor,)


# ================================================================== crop.fit_by_shape
# Same idea with the anchor being the SHAPE of a hole: a host component with bg holes inside its bbox; every
# hole is filled by a loose piece whose key cells (all its cells | its host-coloured cells) have the hole's
# shape (identity | dihedral; pieces optionally drawn at scale s); output = host bbox (optionally x s).
def _d8(cells):
    """cells: {(y,x): v} -> list of 8 normalised dihedral images."""
    out = []
    for k in range(8):
        t = {}
        for (y, x), v in cells.items():
            a, b = (y, x) if k < 4 else (y, -x)
            for _ in range(k % 4): a, b = b, -a
            t[(a, b)] = v
        out.append(_norm(t))
    return out


def _norm(cells):
    y0 = min(y for y, _ in cells); x0 = min(x for _, x in cells)
    return {(y - y0, x - x0): v for (y, x), v in cells.items()}


def _down(cells, s):
    if s == 1: return cells
    c = _norm(cells); out = {}
    for (y, x), v in c.items():
        q = (y // s, x // s)
        if out.setdefault(q, v) != v: return None
    for (y, x), v in out.items():
        if any(c.get((y * s + a, x * s + b)) != v for a in range(s) for b in range(s)): return None
    if len(out) * s * s != len(c): return None
    return out


def fit_shape(g, key, hconn, dih, s, up, bgc=None):
    if s == "auto":                    # piece scale induced per grid: the largest s at which every hole fits
        for k in (4, 3, 2, 1):
            r = fit_shape(g, key, hconn, dih, k, k if up == 's' else up, bgc)
            if r is not None: return r
        return None
    bg = bgc if bgc is not None and any(bgc in r for r in g) else bg_of(g)
    best = None
    for col in {v for r in g for v in r} - {bg}:
        for h in comps(g, lambda y, x, col=col: g[y][x] == col, N4):
            if len(h) < 4: continue
            r0, c0, r1, c1 = bb(h)
            inb = lambda y, x: r0 <= y <= r1 and c0 <= x <= c1 and g[y][x] == bg
            holes = comps(g, inb, N8 if hconn == 8 else N4)
            if not holes or len(holes) > 12: continue
            hk = set(h)
            pieces = [p for p in comps(g, lambda y, x: g[y][x] != bg and (y, x) not in hk)
                      if not (r0 <= min(p)[0] <= r1 and c0 <= min(p, key=lambda q: q[1])[1] <= c1)]
            fills = {}; ok = True; used = set()
            for hole in sorted(holes, key=len):
                hn = _norm(hole); hy, hx = min(y for y, _ in hole), min(x for _, x in hole)
                opts = {}
                for pi, p in enumerate(pieces):
                    if pi in used: continue
                    pd = _down(p, s)
                    if pd is None: continue
                    for t in (_d8(pd) if dih else [_norm(pd)]):
                        kc = {q: v for q, v in t.items() if key == 'all' or v == col}
                        if not kc or len(kc) != len(hn): continue
                        kn = _norm(kc)
                        if set(kn) != set(hn): continue
                        ky, kx = min(y for y, _ in kc), min(x for _, x in kc)
                        dy, dx = hy - ky, hx - kx
                        paint = {(y + dy, x + dx): v for (y, x), v in t.items()}
                        if any(not (r0 <= y <= r1 and c0 <= x <= c1) or (g[y][x] != col and (y, x) not in hole)
                               for y, x in paint): continue
                        opts.setdefault(tuple(sorted(paint.items())), pi)
                        break                                 # first fitting transform: rotations before flips
                if len(opts) != 1: ok = False; break
                (pt, pi), = opts.items(); used.add(pi); fills.update(dict(pt))
            if not ok: continue
            area = (r1 - r0 + 1) * (c1 - c0 + 1)
            if best is None or area > best[0]: best = (area, (r0, c0, r1, c1), fills, col)
            elif area == best[0]: best = (area, None, None, None)
    if not best or best[1] is None: return None
    r0, c0, r1, c1 = best[1]
    out = [[g[y][x] for x in range(c0, c1 + 1)] for y in range(r0, r1 + 1)]
    for (y, x), v in best[2].items(): out[y - r0][x - c0] = v
    if up > 1: out = [[v for v in row for _ in range(up)] for row in out for _ in range(up)]
    if len(out) > 30 or len(out[0]) > 30: return None
    return out


def fam_fit_by_shape(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if H(o0) >= H(i0) and W(o0) >= W(i0) and (H(o0), W(o0)) == (H(i0), W(i0)): return
    for key in ('all', 'host'):
        for hconn in (4, 8):
            for dih in (False, True):
                for s, up in ((1, 1), ('auto', 's'), (2, 2), (2, 1), (1, 2)):
                    fn = lambda g, a=key, b=hconn, c=dih, d=s, e=up: fit_shape(g, a, b, c, d, e)
                    try: good = all(fn(p['input']) == p['output'] for p in train)
                    except Exception: good = False
                    if good:
                        yield (f"fit-by-shape[{key},{hconn},{'d8' if dih else 'id'},s={s},up={up}]", 5, fn)
                        return


FAMILIES = (fam_fit_by_anchor, fam_fit_by_shape)


In [ ]:
%%writefile /kaggle/working/probe/fam_extend_periodic_fill.py
"""extend.periodic_fill: a periodic pattern visible in a known region is continued over a domain.

One primitive, parameters induced from training pairs:
  canvas : output size rule (same | 2x | 2x width | 2x height | square)           -- matched on all pairs
  domain : whole grid | each connected non-background region | each row/column line
  known  : all cells | cells not of an occluder colour (fixed colour absent from outputs, or the
           colour filling an edge strip) | bounding box of non-background cells
  key    : 'min'  smallest translation lattice {(py,s),(0,px)} (either orientation, py/px may be
                  infinite) consistent with the known cells
           'tile' period = known bounding box (anchored on it)
           'ring' L-shaped rings max(y,x) mod p (smallest consistent p)
  phase  : output offset (dy,dx) in [-2,2]^2 induced from the first pair (0 preferred)
Unknown lattice classes keep the input value (inside the input) or become background.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
import numpy as np
from gdsl import H, W, bg_of, colours

INF = None

# ------------------------------------------------------------------ lattices
def _reduce(y, x, lat):
    py, s, px = lat
    if py:
        k = y // py; y -= k * py; x -= k * s
    if px:
        x %= px
    return (y, x)

def _consistent(known, lat):
    d = {}
    rep = False
    for (y, x), v in known:
        k = _reduce(y, x, lat)
        u = d.get(k)
        if u is None:
            d[k] = v
        elif u != v:
            return None
        else:
            rep = True
    return d if rep else None

def _ncls(py, s, px, hk, wk):
    """Number of lattice classes met by a full hk x wk rectangle (the description length)."""
    if py is INF: return hk * px
    if px: return py * px
    tot = 0
    for r in range(min(py, hk)):
        nk = (hk - 1 - r) // py + 1
        tot += wk + abs(s) * (nk - 1)
    return tot

from functools import lru_cache
@lru_cache(maxsize=256)
def _cands(hk, wk):
    cands = []
    for py in list(range(1, hk)) + [INF]:
        for px in list(range(1, wk)) + [INF]:
            if py is INF and px is INF: continue
            if py is INF:
                cands.append((_ncls(py, 0, px, hk, wk), 0, (py, 0, px)))
                continue
            ss = range(px) if px else range(-(wk - 1), wk)
            for s in ss:
                sa = abs(s if not px or s <= px // 2 else s - px)
                a = _ncls(py, s, px, hk, wk)
                if a < hk * wk: cands.append((a, sa, (py, s, px)))
    cands.sort(key=lambda c: (c[0], c[1]))
    return tuple(cands)

def _min_lattice(known, hk, wk, limit=20000):
    """Translation lattice with the fewest classes that is consistent with the known cells.
    known: list of ((y,x),v) relative to the known bbox (0..hk-1, 0..wk-1)."""
    cands = _cands(hk, wk)
    A = np.full((hk, wk), -1, dtype=np.int16)
    for (y, x), v in known: A[y, x] = v
    memo = {}
    def vec_ok(dy, dx):
        k = (dy, dx)
        if k not in memo:
            if dy >= hk or abs(dx) >= wk: memo[k] = True
            else:
                a = A[:hk - dy, max(0, -dx):wk - max(0, dx)]
                b = A[dy:, max(0, dx):max(0, dx) + a.shape[1]]
                m = (a >= 0) & (b >= 0)
                memo[k] = not bool(np.any(a[m] != b[m]))
        return memo[k]
    for n, (area, _, lat) in enumerate(cands):
        if n > limit: break
        py, s, px = lat
        if px and not vec_ok(0, px): continue
        if py and not vec_ok(py, s): continue
        d = _consistent(known, lat)
        if d is not None:
            return area, lat, d
    return None

def _T(g): return [list(r) for r in zip(*g)]

# ------------------------------------------------------------------ roles
def _strip_colour(g):
    """Colour filling an entire edge row/column (occluding strip); None unless unique."""
    h, w = H(g), W(g); cs = set()
    for line in (g[0], g[h - 1], [r[0] for r in g], [r[w - 1] for r in g]):
        if len(set(line)) == 1: cs.add(line[0])
    if len(cs) != 1: return None
    c = cs.pop()
    return c if c != bg_of(g) or True else None

def _known_cells(g, known, cells=None, bg=None):
    """Return list of ((y,x),v) known cells within `cells` (default whole grid)."""
    if cells is None:
        cells = [(y, x) for y in range(H(g)) for x in range(W(g))]
    if known == 'all':
        return [((y, x), g[y][x]) for y, x in cells]
    if known == 'bbox':
        nb = [(y, x) for y, x in cells if g[y][x] != bg]
        if not nb: return []
        y0 = min(y for y, _ in nb); y1 = max(y for y, _ in nb)
        x0 = min(x for _, x in nb); x1 = max(x for _, x in nb)
        return [((y, x), g[y][x]) for y, x in cells if y0 <= y <= y1 and x0 <= x <= x1]
    if known == 'strip':
        c = _strip_colour(g)
        if c is None: return []
    else:
        c = known  # integer occluder colour
    return [((y, x), g[y][x]) for y, x in cells if g[y][x] != c]

# ------------------------------------------------------------------ class functions
_CACHE = {}
def _class_fn(known, key):
    ck = (tuple(known), key)
    if ck in _CACHE: return _CACHE[ck]
    if len(_CACHE) > 4000: _CACHE.clear()
    r = _CACHE[ck] = _class_fn0(known, key)
    return r

def _class_fn0(known, key):
    """Return (lookup(y,x)->colour|None, lean) for known cells ((y,x),v) in grid coordinates.
    lean: -1 if the lattice extends leftward (for anchoring on enlarged canvases), 'T' flag."""
    if not known: return None
    ys = [p[0] for p, _ in known]; xs = [p[1] for p, _ in known]
    y0, x0 = min(ys), min(xs); hk = max(ys) - y0 + 1; wk = max(xs) - x0 + 1
    rel = [((y - y0, x - x0), v) for (y, x), v in known]
    if key == 'tile':
        if len(rel) != hk * wk: return None
        d = {p: v for p, v in rel}
        return (lambda y, x: d.get(((y - y0) % hk, (x - x0) % wk))), 0
    if key == 'ring':
        best = None
        ks = {max(y, x) for (y, x), _ in known}
        for p in range(1, max(ks) + 2):
            d = {}; ok = True
            for (y, x), v in known:
                if d.setdefault(max(y, x) % p, v) != v: ok = False; break
            if ok:
                best = (p, d); break
        if not best: return None
        p, d = best
        return (lambda y, x: d.get(max(y, x) % p) if y >= 0 and x >= 0 else None), 0
    # min lattice, both orientations
    a = _min_lattice(rel, hk, wk)
    b = _min_lattice([((x, y), v) for (y, x), v in rel], wk, hk)
    if a is None and b is None: return None
    if b is None or (a is not None and a[0] <= b[0]):
        _, lat, d = a
        return (lambda y, x: d.get(_reduce(y - y0, x - x0, lat))), (-1 if lat[0] and lat[1] < 0 and not lat[2] else 0)
    _, lat, d = b
    return (lambda y, x: d.get(_reduce(x - x0, y - y0, lat))), (-2 if lat[0] and lat[1] < 0 and not lat[2] else 0)

# ------------------------------------------------------------------ canvas rules
CANVAS = {
    'same': lambda h, w: (h, w),
    '2x': lambda h, w: (2 * h, 2 * w),
    '2w': lambda h, w: (h, 2 * w),
    '2h': lambda h, w: (2 * h, w),
    'square': lambda h, w: (max(h, w), max(h, w)),
}

def _regions(g, bg):
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; out = []
    for r in range(h):
        for c in range(w):
            if seen[r][c] or g[r][c] == bg: continue
            st = [(r, c)]; seen[r][c] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != bg:
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out

def _line_fill(seq, key):
    """1D: seq = list of (pos,v) contiguous span; return (lookup(pos), ok)."""
    n = len(seq); a = seq[0][0]; vals = [v for _, v in seq]
    p = n
    if key == 'min':
        for q in range(1, n):
            if all(vals[i] == vals[i % q] for i in range(n)): p = q; break
    return lambda t: vals[(t - a) % p]

def render(g, canvas, domain, known, key, off=(0, 0)):
    h, w = H(g), W(g); bg = bg_of(g)
    Ho, Wo = CANVAS[canvas](h, w)
    if Ho > 30 or Wo > 30: return None
    dy, dx = off
    if domain == 'whole':
        kc = _known_cells(g, known, bg=bg)
        cf = _class_fn(kc, key)
        if cf is None: return None
        f, lean = cf
        ay = ax = 0
        if lean == -1: ax = Wo - w
        elif lean == -2: ay = Ho - h
        out = [[bg] * Wo for _ in range(Ho)]
        for Y in range(Ho):
            for X in range(Wo):
                y, x = Y - ay + dy, X - ax + dx
                v = f(y, x)
                if v is None:
                    yy, xx = Y - ay, X - ax
                    v = g[yy][xx] if 0 <= yy < h and 0 <= xx < w else bg
                out[Y][X] = v
        return out
    if canvas != 'same' or off != (0, 0): return None
    out = [r[:] for r in g]
    if domain == 'regions':
        regs = _regions(g, bg)
        if not regs: return None
        for cells in regs:
            rbg = Counter(g[y][x] for y, x in cells).most_common(1)[0][0]
            kc = _known_cells(g, known, cells, rbg)
            if not kc or len(kc) == len(cells): continue
            cf = _class_fn(kc, key)
            if cf is None: continue
            f, _ = cf
            for y, x in cells:
                v = f(y, x)
                if v is not None: out[y][x] = v
        return out
    if domain == 'lines':
        any_ = False
        for axis in (0, 1):
            L = g if axis == 0 else _T(g)
            for r, row in enumerate(L):
                idx = [i for i, v in enumerate(row) if v != bg]
                if len(idx) < 2: continue
                a, b = idx[0], idx[-1]
                f = _line_fill([(i, row[i]) for i in range(a, b + 1)], key)
                any_ = True
                for t in range(len(row)):
                    if axis == 0: out[r][t] = f(t)
                    else: out[t][r] = f(t)
        return out if any_ else None
    return None

# ------------------------------------------------------------------ family
OFFS = sorted(((a, b) for a in range(-2, 3) for b in range(-2, 3)), key=lambda o: (abs(o[0]) + abs(o[1]), abs(o[0]), -o[1], -o[0]))

def fam_periodic_fill(train):
    i0, o0 = train[0]['input'], train[0]['output']
    canv = [c for c, f in CANVAS.items() if all(f(H(p['input']), W(p['input'])) == (H(p['output']), W(p['output'])) for p in train)]
    if not canv: return
    if 'square' in canv and len(canv) > 1 and all(H(p['input']) == W(p['input']) for p in train):
        canv.remove('square')
    ins = [p['input'] for p in train]; outs = [p['output'] for p in train]
    occ = sorted(c for c in set().union(*map(colours, ins)) if not any(c in colours(o) for o in outs))
    knowns = ['all', 'bbox', 'strip'] + occ
    configs = []
    for cv in canv:
        for kn in knowns:
            for key in ('min', 'tile', 'ring'):
                configs.append((cv, 'whole', kn, key))
        if cv == 'same':
            for key in ('tile', 'min'):
                configs.append((cv, 'regions', 'bbox', key))
                configs.append((cv, 'lines', 'bbox', key))
    n = 0
    for cv, dom, kn, key in configs:
        try:
            base = render(i0, cv, dom, kn, key)
        except Exception:
            continue
        if base is None: continue
        offs = OFFS if dom == 'whole' else [(0, 0)]
        hits = 0
        for off in offs:
            try:
                r = base if off == (0, 0) else render(i0, cv, dom, kn, key, off)
            except Exception:
                r = None
            if r == o0:
                if cv == 'same' and dom == 'whole' and kn == 'all' and off == (0, 0) and r == i0:
                    break
                ks = kn if isinstance(kn, str) else f'c{kn}'
                name = f"periodic-fill:{key}[{cv},{dom},{ks}" + (f",off{off[0]},{off[1]}" if off != (0, 0) else '') + ']'
                cost = 4 + (off != (0, 0)) + (dom != 'whole')
                yield (name, cost, lambda g, cv=cv, dom=dom, kn=kn, key=key, off=off: render(g, cv, dom, kn, key, off))
                n += 1; hits += 1
                if off == (0, 0) or hits >= 2: break
        if n >= 12: return

FAMILIES = (fam_periodic_fill,)


In [ ]:
%%writefile /kaggle/working/probe/fam_extend_repeat_stamp.py
"""extend.repeat_stamp: a template is copied repeatedly along a translation vector.

One primitive  REPEAT(template T, vector v, stop, colour)  -- paint T + k*v for k = 1, 2, ... until the
copy leaves the grid (or, with stop=hit, until the copy overlaps its target, inclusive).
Every parameter is induced per task; the *inducer* says where T, v, stop and colour come from:

  fragment  T = the unique largest object; every other object M is a fragment of a copy: v is the one
            direction d (8 dirs) whose step-shift of T contains M; step in {bbox+g, minfree+g}, g in {0,1}
            (minfree = smallest shift with no overlap).  Copies take M's colour, M is erased first.
  toward    T = template object, aligned target object (perpendicular spans overlap) lies in direction d;
            copies of T step size+g until one overlaps the target.  Target role: 'share' (single-colour
            object whose colour occurs in the multi-colour T) or 'vanish' (colour that disappears in all
            training outputs).  Targets optionally erased.
  self      every object repeats itself along (sy*(h+g), sx*(w+g)); the sign set is induced from the first
            training pair (directions whose first copy agrees with the output).
  stub      stub = component of non-anchor colours touching an anchor-colour object on one side;
            v = away from the anchor, |v| = stub extent (the stub's own period) -> continued to the edge.
  legend    identical shapes stacked with pitch = shape extent; a striped legend object gives the colour
            order; the stack is completed (both ways) to the full legend sequence, legend erased.
  motif     single-cell seeds; motif M (cells in the box between seed and seed+v, colours as roles
            seed/constant) and the set of vectors v are learned from the first pair's difference.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox

D8 = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
D4 = D8[:4]


def same_size(train):
    return all((H(p["input"]), W(p["input"])) == (H(p["output"]), W(p["output"])) for p in train)


def chain(out, cells, v, stop=None, kmax=40):
    """cells: list of (y,x,c). Paint cells+k*v, k=1.. while any cell is inside; stop(inside) ends it."""
    h, w = H(out), W(out); vy, vx = v
    if v == (0, 0): return
    for k in range(1, kmax):
        ins = [(y + k * vy, x + k * vx, c) for y, x, c in cells
               if 0 <= y + k * vy < h and 0 <= x + k * vx < w]
        if not ins: return
        for y, x, c in ins: out[y][x] = c
        if stop and stop(ins): return


def comps(g, keep, diag=False):
    """Multicolour components of cells where keep(y,x)."""
    h, w = H(g), W(g); seen = set(); res = []
    nb = D8 if diag else D4
    for y in range(h):
        for x in range(w):
            if (y, x) in seen or not keep(y, x): continue
            st = [(y, x)]; seen.add((y, x)); cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in nb:
                    q = (a + dy, b + dx)
                    if 0 <= q[0] < h and 0 <= q[1] < w and q not in seen and keep(*q):
                        seen.add(q); st.append(q)
            res.append(cs)
    return res


def step_vec(cells, d, mode):
    dy, dx = d
    y0, x0, y1, x1 = bbox(cells)
    kind, g = mode[:-1], int(mode[-1])
    if kind == "bbox":
        return (dy * (y1 - y0 + 1 + g), dx * (x1 - x0 + 1 + g))
    s = set(cells)
    for k in range(1, 31):
        if not any((y + k * dy, x + k * dx) in s for y, x in cells):
            return (dy * (k + g), dx * (k + g))
    return None


# ------------------------------------------------------------------ fragment
def fam_fragment(train):
    if not same_size(train): return
    for diag in (True, False):
        for mode in ("minfree0", "bbox1", "bbox0", "minfree1"):
            def fn(g, diag=diag, mode=mode):
                bg = bg_of(g); obs = objects(g, bg, diag, True)
                if len(obs) < 2: return None
                obs.sort(key=len, reverse=True)
                if len(obs[0]) == len(obs[1]) or len(obs[0]) < 3: return None
                T = obs[0]; out = [r[:] for r in g]
                for M in obs[1:]:
                    for y, x in M: out[y][x] = bg
                for M in obs[1:]:
                    ms = set(M); hits = []
                    for d in D8:
                        v = step_vec(T, d, mode)
                        if v and ms <= {(y + v[0], x + v[1]) for y, x in T}: hits.append(v)
                    if len(hits) != 1: return None
                    c = g[M[0][0]][M[0][1]]
                    chain(out, [(y, x, c) for y, x in T], hits[0])
                return out
            yield (f"repeat:fragment[{mode},{'8' if diag else '4'}]", 4, fn)


# ------------------------------------------------------------------ toward
def vanish_colours(train):
    v = None
    for p in train:
        ci = {c for r in p["input"] for c in r}; co = {c for r in p["output"] for c in r}
        v = (ci - co) if v is None else v & (ci - co)
    return v or set()


def fam_toward(train):
    if not same_size(train): return
    van = vanish_colours(train)
    roles = ["share"] + (["vanish"] if van else [])
    for role in roles:
        for erase in (False, True):
            for gap in (0, 1):
                def fn(g, role=role, erase=erase, gap=gap):
                    bg = bg_of(g); obs = comps(g, lambda y, x: g[y][x] != bg)
                    col = lambda o: {g[y][x] for y, x in o}
                    if role == "share":
                        tps = [o for o in obs if len(col(o)) >= 2]
                        tgs = [o for o in obs if len(col(o)) == 1]
                    else:
                        tps = [o for o in obs if not col(o) & van]
                        tgs = [o for o in obs if col(o) <= van]
                    if not tps or not tgs: return None
                    out = [r[:] for r in g]
                    if erase:
                        for o in tgs:
                            for y, x in o: out[y][x] = bg
                    n = 0
                    for T in tps:
                        y0, x0, y1, x1 = bbox(T); ct = col(T)
                        for dy, dx in D4:
                            best = None
                            for M in tgs:
                                a0, b0, a1, b1 = bbox(M)
                                if dy:
                                    if b1 < x0 or b0 > x1: continue
                                    dist = a0 - y1 if dy > 0 else y0 - a1
                                else:
                                    if a1 < y0 or a0 > y1: continue
                                    dist = b0 - x1 if dx > 0 else x0 - b1
                                if dist <= 0: continue
                                if best is None or dist < best[0]: best = (dist, M)
                            if best is None: continue
                            M = best[1]
                            if role == "share" and not col(M) <= ct: continue
                            ms = set(M)
                            v = (dy * (y1 - y0 + 1 + gap), dx * (x1 - x0 + 1 + gap))
                            chain(out, [(y, x, g[y][x]) for y, x in T], v,
                                  stop=lambda ins, ms=ms: any((y, x) in ms for y, x, _ in ins))
                            n += 1
                    return out if n else None
                yield (f"repeat:toward[{role},{'erase' if erase else 'keep'},g{gap}]", 5, fn)


# ------------------------------------------------------------------ self
def fam_self(train):
    if not same_size(train): return
    i0, o0 = train[0]["input"], train[0]["output"]
    signs = [d for d in D8]
    for diag in (False, True):
        for gap in (0, 1):
            bg = bg_of(i0); obs = objects(i0, bg, diag, False)
            if not obs or len(obs) > 12: continue
            ok = []
            for sy, sx in signs:
                new = False; good = True
                for T in obs:
                    y0, x0, y1, x1 = bbox(T)
                    v = (sy * (y1 - y0 + 1 + gap), sx * (x1 - x0 + 1 + gap))
                    for y, x in T:
                        yy, xx = y + v[0], x + v[1]
                        if 0 <= yy < H(i0) and 0 <= xx < W(i0):
                            if o0[yy][xx] != i0[y][x]: good = False; break
                            if i0[yy][xx] != i0[y][x]: new = True
                    if not good: break
                if good and new: ok.append((sy, sx))
            if not ok: continue
            def fn(g, diag=diag, gap=gap, ok=tuple(ok)):
                bg = bg_of(g); obs = objects(g, bg, diag, False)
                if not obs: return None
                out = [r[:] for r in g]
                for T in obs:
                    y0, x0, y1, x1 = bbox(T)
                    for sy, sx in ok:
                        chain(out, [(y, x, g[y][x]) for y, x in T],
                              (sy * (y1 - y0 + 1 + gap), sx * (x1 - x0 + 1 + gap)))
                return out
            yield (f"repeat:self[{','.join('%d%d' % d for d in ok)},g{gap},{'8' if diag else '4'}]", 4, fn)


# ------------------------------------------------------------------ stub
def fam_stub(train):
    if not same_size(train): return
    cs = None
    for p in train:
        c = {v for r in p["input"] for v in r}
        cs = c if cs is None else cs & c
    for a in sorted(cs or ()):
        def fn(g, a=a):
            bg = bg_of(g)
            if a == bg: return None
            h, w = H(g), W(g)
            stubs = comps(g, lambda y, x: g[y][x] not in (bg, a))
            out = [r[:] for r in g]; n = 0
            for C in stubs:
                dirs = set()
                for y, x in C:
                    for dy, dx in D4:
                        yy, xx = y - dy, x - dx
                        if 0 <= yy < h and 0 <= xx < w and g[yy][xx] == a: dirs.add((dy, dx))
                if not dirs: continue
                if len(dirs) != 1: return None
                dy, dx = dirs.pop(); y0, x0, y1, x1 = bbox(C)
                chain(out, [(y, x, g[y][x]) for y, x in C], (dy * (y1 - y0 + 1), dx * (x1 - x0 + 1)))
                n += 1
            return out if n else None
        yield (f"repeat:stub[anchor={a}]", 4, fn)


# ------------------------------------------------------------------ legend
def legend_of(g, bg):
    """Multicolour rectangle whose colour is constant along one axis: returns (cells, colour order, axis)."""
    found = []
    for C in comps(g, lambda y, x: g[y][x] != bg):
        cols = {g[y][x] for y, x in C}
        if len(cols) < 2: continue
        y0, x0, y1, x1 = bbox(C)
        if len(C) != (y1 - y0 + 1) * (x1 - x0 + 1): continue
        if all(g[y][x] == g[y0][x] for y, x in C):
            seq = [g[y0][x] for x in range(x0, x1 + 1)]
        elif all(g[y][x] == g[y][x0] for y, x in C):
            seq = [g[y][x0] for y in range(y0, y1 + 1)]
        else: continue
        if len(set(seq)) != len(seq): continue
        found.append((C, seq))
    return found[0] if len(found) == 1 else None


def fam_legend(train):
    if not same_size(train): return
    for axis in (0, 1):
        for rev in (False, True):
            def fn(g, axis=axis, rev=rev):
                bg = bg_of(g); L = legend_of(g, bg)
                if not L: return None
                LC, seq = L
                if rev: seq = seq[::-1]
                lset = set(LC)
                obs = [o for o in objects(g, bg, True, True) if not set(o) & lset]
                if not obs: return None
                norm = lambda o: sorted((y - bbox(o)[0], x - bbox(o)[1]) for y, x in o)
                shape = norm(obs[0])
                if any(norm(o) != shape for o in obs): return None
                cols = [g[o[0][0]][o[0][1]] for o in obs]
                if len(set(cols)) != len(cols) or not set(cols) <= set(seq): return None
                obs = sorted(obs, key=lambda o: bbox(o)[axis])
                y0, x0, y1, x1 = bbox(obs[0])
                pitch = (y1 - y0 + 1) if axis == 0 else (x1 - x0 + 1)
                for j, o in enumerate(obs):
                    b = bbox(o)
                    if b[axis] != bbox(obs[0])[axis] + j * pitch or b[1 - axis] != bbox(obs[0])[1 - axis]: return None
                idx = [seq.index(c) for c in cols]
                if idx != list(range(idx[0], idx[0] + len(idx))): return None
                out = [r[:] for r in g]
                for y, x in LC: out[y][x] = bg
                base = obs[0]; bi = idx[0]; h, w = H(g), W(g)
                for j, c in enumerate(seq):
                    k = j - bi
                    off = (k * pitch, 0) if axis == 0 else (0, k * pitch)
                    for y, x in base:
                        yy, xx = y + off[0], x + off[1]
                        if not (0 <= yy < h and 0 <= xx < w): return None
                        out[yy][xx] = c
                return out
            yield (f"repeat:legend[{'rows' if axis == 0 else 'cols'},{'rev' if rev else 'fwd'}]", 5, fn)


# ------------------------------------------------------------------ motif
def _seeds(g, bg):
    """Seeds = multicolour 4-connected objects; (cells, bbox, colour or None if multicolour)."""
    res = []
    for C in comps(g, lambda y, x: g[y][x] != bg):
        cs = {g[y][x] for y, x in C}
        res.append((set(C), bbox(C), cs.pop() if len(cs) == 1 else None))
    return res


def _ref(bb, v):
    y0, x0, y1, x1 = bb
    ry = y0 if v[0] < 0 else y1 if v[0] > 0 else (y0 + y1) / 2
    rx = x0 if v[1] < 0 else x1 if v[1] > 0 else (x0 + x1) / 2
    if ry != int(ry) or rx != int(rx): return None
    return (int(ry), int(rx))


def _motif_paint(g, seeds, motifs):
    out = [r[:] for r in g]
    for C, bb, sc in seeds:
        for v, M in motifs:
            ref = _ref(bb, v)
            if ref is None: return None
            if any(r == "S" for r in M.values()) and sc is None: return None
            cells = [(ref[0] + a - v[0], ref[1] + b - v[1], sc if r == "S" else r) for (a, b), r in M.items()]
            chain(out, cells, v)
    return out


def fam_motif(train):
    """Seeds emit a periodic motif along learned vectors to the border (motif learned from all pairs)."""
    if not same_size(train): return
    data = []
    for p in train:
        i, o = p["input"], p["output"]; bg = bg_of(i); h, w = H(i), W(i)
        sd = _seeds(i, bg)
        if not sd or len(sd) > 10 or any(len(C) > 9 for C, _, _ in sd): return
        D = {(y, x): o[y][x] for y in range(h) for x in range(w) if o[y][x] != i[y][x]}
        if not D or any(i[y][x] != bg for y, x in D): return
        data.append((i, o, sd, D))
    newc = {c for _, _, _, D in data for c in D.values()}

    def motif_for(v):
        vy, vx = v; M = {}
        for i, o, sd, D in data:
            for C, bb, sc in sd:
                ref = _ref(bb, v)
                if ref is None: return None
                for a in range(min(0, vy), max(0, vy) + 1):
                    for b in range(min(0, vx), max(0, vx) + 1):
                        q = (ref[0] + a, ref[1] + b)
                        if q in C or q not in D: continue
                        r = "S" if sc is not None and D[q] == sc else D[q]
                        if M.setdefault((a, b), r) != r: return None
        return M

    cand = sorted({(a, b) for a in range(-4, 5) for b in range(-4, 5) if (a, b) != (0, 0)},
                  key=lambda v: (max(abs(v[0]), abs(v[1])), abs(v[0]) + abs(v[1])))
    active = []; covered = [set() for _ in data]
    for v in cand:
        M = motif_for(v)
        if not M or any(r != "S" and r not in newc for r in M.values()): continue
        diffs = []; good = True
        for (i, o, sd, D) in data:
            p = _motif_paint(i, sd, [(v, M)])
            if p is None: good = False; break
            diff = {(y, x) for y in range(H(i)) for x in range(W(i)) if p[y][x] != i[y][x]}
            if any(p[y][x] != o[y][x] for y, x in diff): good = False; break
            diffs.append(diff)
        if not good or all(d <= c for d, c in zip(diffs, covered)): continue
        active.append((v, M))
        for c, d in zip(covered, diffs): c |= d
        if len(active) > 12: return
    if any(c != set(D) for c, (_, _, _, D) in zip(covered, data)): return

    def fn(g, active=tuple(active)):
        bgc = bg_of(g); sd = _seeds(g, bgc)
        if not sd or len(sd) > 10 or any(len(C) > 9 for C, _, _ in sd): return None
        return _motif_paint(g, sd, active)
    yield (f"repeat:motif[{';'.join('%d,%d' % v for v, _ in active)}]", 5, fn)


FAMILIES = (fam_fragment, fam_toward, fam_self, fam_stub, fam_legend, fam_motif)


In [ ]:
%%writefile /kaggle/working/probe/fam_fill_bg_windows.py
"""fill.bg_windows: paint square patches made only of a target colour r with a paint colour p.

Reviewer rule for mechanism group M050 (Len, 2026-09-29): "cover the largest square background patch with colour".
Topology / geometry prior: a maximal empty square is the largest disc of the L-infinity metric inside a region.

Everything is induced from the training pairs:
  r         the single input colour of the changed cells: a literal colour when constant across pairs,
            else the per-grid role 'bg' (most frequent colour)
  p         the single output colour of the changed cells (constant across pairs)
  selector  which squares are painted (m = minimum side, induced from the smallest painted square):
    win k       every k x k window made only of r (union of placements)
    maxwin>=m   the windows of the LARGEST side K for which an all-r K x K window exists in this grid, if K >= m
    compsq>=m   4-connected components of r that are solid squares with side >= m
    comprect>=m 4-connected components of r that are solid rectangles with both sides >= m
    greedy>=m   repeatedly the largest all-r square not overlapping earlier picks (ties: reading order of the
                top-left corner), while its side is >= m
    maxrect/s   the unique largest-AREA all-r axis rectangle (s = 1: its interior, border ring left unpainted)
    maxrect2/s  the same among rectangles with both sides >= 2 (a patch, not a line)
    maxsq/s     the unique largest all-r square (s = 1: interior)
    maxdiag     the unique longest diagonal run of r cells (either diagonal direction)
Mechanism group M067 "fill.largest empty" is the same concept (reviewer's parent concept "patch").
A program is kept only if it reproduces every training output exactly.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


def _sq(g, r, blocked=None):
    """S[y][x] = side of the largest all-r square whose bottom-right corner is (y, x)."""
    h, w = H(g), W(g)
    S = [[0] * w for _ in range(h)]
    for y in range(h):
        for x in range(w):
            if g[y][x] != r or (blocked and (y, x) in blocked):
                continue
            S[y][x] = 1 if y == 0 or x == 0 else 1 + min(S[y - 1][x], S[y][x - 1], S[y - 1][x - 1])
    return S


def _windows(g, r, k):
    S = _sq(g, r)
    cells = set()
    for y in range(H(g)):
        for x in range(W(g)):
            if S[y][x] >= k:
                for dy in range(k):
                    for dx in range(k):
                        cells.add((y - dy, x - dx))
    return cells


def _maxwin(g, r, m):
    S = _sq(g, r)
    K = max((v for row in S for v in row), default=0)
    return _windows(g, r, K) if K >= m else set()


def _components(g, r):
    h, w = H(g), W(g)
    seen = set()
    for y in range(h):
        for x in range(w):
            if g[y][x] != r or (y, x) in seen:
                continue
            st, comp = [(y, x)], {(y, x)}
            seen.add((y, x))
            while st:
                a, b = st.pop()
                for dy, dx in N4:
                    n = (a + dy, b + dx)
                    if 0 <= n[0] < h and 0 <= n[1] < w and n not in seen and g[n[0]][n[1]] == r:
                        seen.add(n); comp.add(n); st.append(n)
            yield comp


def _comp_rect(g, r, m, square):
    cells = set()
    for c in _components(g, r):
        ys = [a for a, _ in c]; xs = [b for _, b in c]
        hh, ww = max(ys) - min(ys) + 1, max(xs) - min(xs) + 1
        if len(c) != hh * ww or min(hh, ww) < m or (square and hh != ww):
            continue
        cells |= c
    return cells


def _greedy(g, r, m):
    taken = set()
    for _ in range(H(g) * W(g)):
        S = _sq(g, r, taken)
        K = max((v for row in S for v in row), default=0)
        if K < m:
            break
        y, x = next((y, x) for y in range(H(g)) for x in range(W(g)) if S[y][x] == K)
        for dy in range(K):
            for dx in range(K):
                taken.add((y - dy, x - dx))
    return taken


SELECT = {'win': _windows, 'maxwin': _maxwin, 'greedy': _greedy,
          'compsq': lambda g, r, m: _comp_rect(g, r, m, True),
          'comprect': lambda g, r, m: _comp_rect(g, r, m, False)}


def _maxrects(g, r, minside=1):
    """all maximal-AREA axis rectangles made only of r with both sides >= minside (histogram method); (y0, x0, h, w)."""
    h, w = H(g), W(g)
    hist = [0] * w; best = 0; rects = []
    for y in range(h):
        for x in range(w):
            hist[x] = hist[x] + 1 if g[y][x] == r else 0
        for x0 in range(w):
            mh = 10 ** 9
            for x1 in range(x0, w):
                mh = min(mh, hist[x1])
                if mh == 0: break
                if mh < minside or x1 - x0 + 1 < minside: continue
                a = mh * (x1 - x0 + 1)
                if a > best: best, rects = a, [(y - mh + 1, x0, mh, x1 - x0 + 1)]
                elif a == best: rects.append((y - mh + 1, x0, mh, x1 - x0 + 1))
    return list(dict.fromkeys(rects))


def _cells_rect(y0, x0, hh, ww, shrink):
    return {(y, x) for y in range(y0 + shrink, y0 + hh - shrink) for x in range(x0 + shrink, x0 + ww - shrink)}


def _maxrect2(g, r, shrink):
    rs = _maxrects(g, r, 2)
    if len(rs) != 1: return set()
    return _cells_rect(*rs[0], shrink)


def _maxrect(g, r, shrink):
    rs = _maxrects(g, r)
    if len(rs) != 1: return set()          # the largest patch must be unique
    return _cells_rect(*rs[0], shrink)


def _maxsq(g, r, shrink):
    S = _sq(g, r); K = max((v for row in S for v in row), default=0)
    corners = [(y, x) for y in range(H(g)) for x in range(W(g)) if S[y][x] == K]
    if K < 2 or len(corners) != 1: return set()
    y, x = corners[0]
    return _cells_rect(y - K + 1, x - K + 1, K, K, shrink)


def _maxdiag(g, r, _):
    h, w = H(g), W(g); best, runs = 0, []
    for dy, dx in ((1, 1), (1, -1)):
        for y in range(h):
            for x in range(w):
                if g[y][x] != r: continue
                py, px = y - dy, x - dx
                if 0 <= py < h and 0 <= px < w and g[py][px] == r: continue   # not a run start
                run = []; a, b = y, x
                while 0 <= a < h and 0 <= b < w and g[a][b] == r: run.append((a, b)); a += dy; b += dx
                if len(run) > best: best, runs = len(run), [run]
                elif len(run) == best: runs.append(run)
    if best < 2 or len(runs) != 1: return set()
    return set(runs[0])


SELECT.update({'maxrect2': _maxrect2, 'maxrect': _maxrect, 'maxsq': _maxsq, 'maxdiag': _maxdiag})


def apply(g, rr, p, sel, k):
    r = bg_of(g) if rr == 'bg' else rr
    cells = SELECT[sel](g, r, k)
    if not cells:
        return None
    out = [row[:] for row in g]
    for y, x in cells:
        out[y][x] = p
    return out


def fam_fill_bg_windows(train):
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train):
        return
    cin, cout, per_bg = set(), set(), set()
    for p in train:
        gi, go = p['input'], p['output']
        ch = [(y, x) for y in range(H(gi)) for x in range(W(gi)) if gi[y][x] != go[y][x]]
        if not ch:
            return
        cin |= {gi[y][x] for y, x in ch}; cout |= {go[y][x] for y, x in ch}
        per_bg.add(gi[ch[0][0]][ch[0][1]] == bg_of(gi))
    if len(cin) != 1 or len(cout) != 1:
        return
    r, p = next(iter(cin)), next(iter(cout))
    roles = [r] + (['bg'] if per_bg == {True} and len({bg_of(q['input']) for q in train}) > 1 else [])
    n = 0
    for rr in roles:
        for sel, ks in (('win', (2, 3, 4, 5, 6)), ('maxwin', (2, 3)), ('compsq', (2, 1)), ('comprect', (2,)),
                        ('greedy', (2, 3)), ('maxrect', (0, 1)), ('maxrect2', (0, 1)), ('maxsq', (0, 1)), ('maxdiag', (0,))):
            for k in ks:
                ok = True
                for q in train:
                    o = apply(q['input'], rr, p, sel, k)
                    if o != q['output']:
                        ok = False
                        break
                if ok:
                    n += 1
                    tag = f"{sel}{k}" if sel == 'win' else f"{sel}:shrink{k}" if sel.startswith('max') and sel != 'maxwin' else f"{sel}>={k}"
                    yield (f"fill-bg-windows:{tag}[r{rr},p{p}]", 3,
                           lambda g, a=(rr, p, sel, k): apply(g, *a))
                    if n >= 3:
                        return


FAMILIES = (fam_fill_bg_windows,)


In [ ]:
%%writefile /kaggle/working/probe/fam_fill_region_by_property.py
"""fill.region_by_property: segment the grid into regions separated by walls, then paint each whole
region by a rule on one region feature.  Everything is induced from the training pairs:
  region colour r  : the input colour of (most) changed cells
  membership mode  : 'col'  region cells = colour r
                     'wall' region cells = every colour except the dominant wall colour w (most frequent non-r colour)
                     'seed' region cells = r plus isolated single-cell seeds (8-isolated, size-1 objects)
  connectivity     : 4 | 8
  feature          : border (touches edge) | size-rank | is-rect | size parity | seed majority colour |
                     adjacent-marker majority colour | adjacent-has colour c ; optionally paired with border
  table            : feature value -> action, action in {keep, constant colour, W (dominant wall colour),
                     self (the feature value is itself a colour)}.  Unseen feature values refuse (None).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def dominant_wall(g, r):
    c = Counter(v for row in g for v in row if v != r)
    return c.most_common(1)[0][0] if c else None


def member_mask(g, r, mode):
    h, w = H(g), W(g)
    if mode == 'col':
        return [[g[y][x] == r for x in range(w)] for y in range(h)]
    if mode == 'wall':
        wc = dominant_wall(g, r)
        return [[g[y][x] != wc for x in range(w)] for y in range(h)]
    m = [[g[y][x] == r for x in range(w)] for y in range(h)]
    for y in range(h):
        for x in range(w):
            v = g[y][x]
            if v == r: continue
            if all(not (0 <= y + dy < h and 0 <= x + dx < w) or g[y + dy][x + dx] != v for dy, dx in N8):
                m[y][x] = True
    return m


def regions(g, mask, diag):
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; out = []
    nb = N8 if diag else N4
    for y0 in range(h):
        for x0 in range(w):
            if seen[y0][x0] or not mask[y0][x0]: continue
            st = [(y0, x0)]; seen[y0][x0] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and mask[yy][xx]:
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out


def neighbours(g, cells, mask):
    h, w = H(g), W(g); s = set(cells); nb = set()
    for y, x in cells:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in s and not mask[yy][xx]:
                nb.add((yy, xx))
    return nb


def majority(cnt):
    if not cnt: return None
    mc = cnt.most_common(2)
    if len(mc) > 1 and mc[0][1] == mc[1][1]: return 'tie'
    return mc[0][0]


def feature_values(g, regs, mask, r, feat):
    h, w = H(g), W(g)
    if feat == 'border':
        return [any(y in (0, h - 1) or x in (0, w - 1) for y, x in c) for c in regs]
    if feat == 'size_rank':
        sizes = sorted({len(c) for c in regs}, reverse=True)
        def rk(n):
            if len(sizes) == 1: return 'only'
            if n == sizes[0]: return 'max'
            if n == sizes[-1]: return 'min'
            return '2nd' if n == sizes[1] else 'mid'
        return [rk(len(c)) for c in regs]
    if feat == 'is_rect':
        out = []
        for c in regs:
            ys = [y for y, _ in c]; xs = [x for _, x in c]
            out.append(len(c) == (max(ys) - min(ys) + 1) * (max(xs) - min(xs) + 1))
        return out
    if feat == 'parity':
        return [len(c) % 2 for c in regs]
    if feat == 'seedmaj':
        return [majority(Counter(g[y][x] for y, x in c if g[y][x] != r)) for c in regs]
    wc = dominant_wall(g, r)
    if feat == 'adjmaj':
        return [majority(Counter(g[y][x] for y, x in neighbours(g, c, mask) if g[y][x] != wc)) for c in regs]
    if feat.startswith('adjhas'):
        col = int(feat[6:])
        return [any(g[y][x] == col for y, x in neighbours(g, c, mask)) or any(g[y][x] == col for y, x in c) for c in regs]
    if feat == 'seedrank':   # region holding the most / fewest foreign (non-region-colour) cells
        cnt = [sum(g[y][x] != r for y, x in c) for c in regs]
        mx, mn = max(cnt), min(cnt)
        if mx == mn: return ['only'] * len(regs)
        return ['max' if k == mx and cnt.count(mx) == 1 else 'min' if k == mn else 'mid' for k in cnt]
    raise ValueError(feat)


SELF_FEATS = ('seedmaj', 'adjmaj')


def region_features(g, r, mode, diag, feat, with_border, scope='all'):
    if r == 'bg': r = bg_of(g)
    mask = member_mask(g, r, mode)
    regs = regions(g, mask, diag)
    if not regs or len(regs) > 200: return None
    vals = feature_values(g, regs, mask, r, feat)
    if with_border:
        bs = feature_values(g, regs, mask, r, 'border')
        vals = [(b, v) for b, v in zip(bs, vals)]
    if scope == 'r':   # only the region-colour cells are repainted; seeds/noise inside stay
        regs = [[(y, x) for y, x in c if g[y][x] == r] for c in regs]
    return regs, vals


def base_val(v, with_border):
    return v[1] if with_border else v


def is_col(v):
    return isinstance(v, int) and not isinstance(v, bool)


def skey(v, with_border):
    """In self mode every colour-valued feature shares one table key (its action is 'self' or a constant)."""
    bv = base_val(v, with_border)
    if not is_col(bv): return v
    return (v[0], '*') if with_border else '*'


def candidate_actions(gi, go, cells, wc):
    acts = set()
    if all(gi[y][x] == go[y][x] for y, x in cells): acts.add('keep')
    cs = {go[y][x] for y, x in cells}
    if len(cs) == 1:
        c = cs.pop(); acts.add(c)
        if c == wc: acts.add('W')
    return acts


def induce(train, r, mode, diag, feat, with_border, selfmode, scope):
    table = {}; any_change = False
    for p in train:
        gi, go = p['input'], p['output']
        rf = region_features(gi, r, mode, diag, feat, with_border, scope)
        if rf is None: return None
        regs, vals = rf
        cover = set()
        wc = dominant_wall(gi, bg_of(gi) if r == 'bg' else r)
        for c, v in zip(regs, vals):
            if not c: continue
            cover.update(c)
            acts = candidate_actions(gi, go, c, wc)
            if not acts: return None
            if selfmode:
                bv = base_val(v, with_border)
                if is_col(bv) and bv in acts: acts = acts | {'self'}
                v = skey(v, with_border)
            prev = table.get(v)
            table[v] = acts if prev is None else prev & acts
            if not table[v]: return None
        # every changed cell must lie in some region
        for y in range(H(gi)):
            for x in range(W(gi)):
                if gi[y][x] != go[y][x] and (y, x) not in cover: return None
    fixed = {}
    for v, acts in table.items():
        # prefer keep; else self; else the single consistent constant colour; else the dominant-wall-colour role
        ints = [k for k in acts if isinstance(k, int)]
        a = 'keep' if 'keep' in acts else 'self' if 'self' in acts else ints[0] if len(ints) == 1 else 'W'
        fixed[v] = a
        if a != 'keep': any_change = True
    if not any_change: return None
    return fixed


def apply(g, r, mode, diag, feat, with_border, selfmode, scope, table):
    rf = region_features(g, r, mode, diag, feat, with_border, scope)
    if rf is None: return None
    regs, vals = rf
    wc = dominant_wall(g, bg_of(g) if r == 'bg' else r)
    out = [row[:] for row in g]
    for c, v in zip(regs, vals):
        bv = base_val(v, with_border)
        k = skey(v, with_border) if selfmode else v
        if k not in table: return None
        a = table[k]
        if a == 'keep': continue
        col = wc if a == 'W' else bv if a == 'self' else a
        if not is_col(col): return None
        for y, x in c: out[y][x] = col
    return out


def fam_region_by_property(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    ch = Counter()
    for p in train:
        gi, go = p['input'], p['output']
        for y in range(H(gi)):
            for x in range(W(gi)):
                if gi[y][x] != go[y][x]: ch[gi[y][x]] += 1
    if not ch: return
    rs = [ch.most_common(1)[0][0]]
    if bg_of(i0) not in rs: rs.append(bg_of(i0))
    if len({bg_of(p['input']) for p in train}) > 1: rs.append('bg')   # region colour as a per-grid role
    # colours relevant for adjhas: colours painted in outputs
    painted = set()
    for p in train:
        gi, go = p['input'], p['output']
        painted |= {go[y][x] for y in range(H(gi)) for x in range(W(gi)) if gi[y][x] != go[y][x]}
    feats = ['border', 'size_rank', 'is_rect', 'parity', 'seedrank', 'seedmaj', 'adjmaj'] + ['adjhas%d' % c for c in sorted(painted)][:3]
    from itertools import product
    n = 0
    for r, mode, diag, feat in product(rs, ('col', 'seed', 'wall'), (False, True), feats):
        if mode == 'col' and feat == 'seedmaj': continue
        sm = feat in SELF_FEATS
        for wb, scope in product((False,) if feat == 'border' else (False, True), ('all',) if mode == 'col' else ('all', 'r')):
            t = induce(train, r, mode, diag, feat, wb, sm, scope)
            if t is None: continue
            n += 1
            if n > 40: return
            name = f"region-fill:{feat}{'+border' if wb else ''}[r{r},{mode},{'8' if diag else '4'}{',r-only' if scope == 'r' else ''}]"
            yield (name, 4 + wb, lambda g, a=(r, mode, diag, feat, wb, sm, scope, t): apply(g, *a))

FAMILIES = (fam_region_by_property,)


In [ ]:
%%writefile /kaggle/working/probe/fam_lines.py
"""Own primitives (no Codex code): edge-marker projection with combination."""
import sys; import os;sys.path.insert(0,os.path.dirname(os.path.abspath(__file__)))
import gdsl
from gdsl import H, W, bg_of

EDGES=('top','bottom','left','right')
def edge_marks(g,bg,e):
    h,w=H(g),W(g)
    if e=='top': return {c for c in range(w) if g[0][c]!=bg}
    if e=='bottom': return {c for c in range(w) if g[h-1][c]!=bg}
    if e=='left': return {r for r in range(h) if g[r][0]!=bg}
    return {r for r in range(h) if g[r][w-1]!=bg}
def edge_colour(g,bg,e,i):
    h,w=H(g),W(g)
    return {'top':g[0][i] if e=='top' else None,'bottom':g[h-1][i] if e=='bottom' else None,
            'left':g[i][0] if e=='left' else None,'right':g[i][w-1] if e=='right' else None}[e]

def added_colour(train):
    """The single colour painted onto background cells in every pair (role: new paint)."""
    cs=None
    for p in train:
        i,o=p['input'],p['output']
        if (H(i),W(i))!=(H(o),W(o)): return None
        bg=bg_of(i)
        ch={o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x]!=o[y][x] and i[y][x]==bg}
        cs=ch if cs is None else cs|ch
    return next(iter(cs)) if cs and len(cs)==1 else None

def project(g,rows_from,cols_from,op,paint):
    bg=bg_of(g);h,w=H(g),W(g)
    R=set().union(*[edge_marks(g,bg,e) for e in rows_from]) if rows_from else set()
    C=set().union(*[edge_marks(g,bg,e) for e in cols_from]) if cols_from else set()
    if not R and not C: return None
    used_r={0} if 'top' in cols_from else set(); used_r|={h-1} if 'bottom' in cols_from else set()
    used_c={0} if 'left' in rows_from else set(); used_c|={w-1} if 'right' in rows_from else set()
    out=[r[:] for r in g]
    for y in range(h):
        if y in used_r: continue
        for x in range(w):
            if x in used_c or g[y][x]!=bg: continue
            a,b=y in R,x in C
            hit={'and':a and b,'or':a or b,'xor':a!=b}[op]
            if hit: out[y][x]=paint
    return out

def fam_edge_projection(train):
    i0,o0=train[0]['input'],train[0]['output']
    if (H(i0),W(i0))!=(H(o0),W(o0)): return
    paint=added_colour(train)
    if paint is None: return
    # which edges carry markers in every training input (role: marker edges)
    def edges_with(g): bg=bg_of(g); return {e for e in EDGES if edge_marks(g,bg,e) and len(edge_marks(g,bg,e))<(W(g) if e in('top','bottom') else H(g))}
    common=None
    for p in train:
        es=edges_with(p['input']); common=es if common is None else common&es
    if not common: return
    rowE=[e for e in ('left','right') if e in common]; colE=[e for e in ('top','bottom') if e in common]
    for op in ('and','or','xor'):
        if rowE and colE:
            yield (f"project:{op}[rows={'+'.join(rowE)},cols={'+'.join(colE)}]",4,lambda g,rowE=rowE,colE=colE,op=op:project(g,rowE,colE,op,paint))
    if rowE: yield (f"project:rows[{'+'.join(rowE)}]",4,lambda g,rowE=rowE:project(g,rowE,[], 'or',paint))
    if colE: yield (f"project:cols[{'+'.join(colE)}]",4,lambda g,colE=colE:project(g,[],colE,'or',paint))

FAMILIES=(fam_edge_projection,)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_by_vector.py
"""move.by_vector: every mover object is translated rigidly by a vector computed from its own properties.

One parametrised primitive  translate(seg, sel, rule, edge, collide):
  seg     : object segmentation   c4 | c8 (single colour) | m8 (multicolour 8-conn)
  sel     : which objects move    all | nonstatic-colour | interior (not touching grid edge) | shared-colour
  rule    : v(obj) = unit(obj) * mag(obj)
              unit : one of 8 constant directions | toward nearest anchor | toward same-colour anchor
                     | toward grid centre | toward attached marker
              mag  : 1 | 2 | 3 | cell count | width | height | number of marker cells
            or v(obj) = table[key(obj)], key in {const, colour, part-role in composite (top/mid/bottom)}
  edge    : clip (cells leaving the grid vanish) | wrap (toroidal)
  collide : over (movers paint over) | block (advance at most |v| unit steps while landing on background)
Markers = colours present in inputs but absent from outputs (induced); they are erased, or absorbed into the
object whose bounding box contains them (recoloured) when the rule is 'toward attached marker'.
All parameters are induced from training pairs; candidates are pre-filtered with per-object displacement sets
and fully verified on all training pairs before being yielded.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, static_colours

DIR8 = {"N": (-1, 0), "S": (1, 0), "W": (0, -1), "E": (0, 1), "NW": (-1, -1), "NE": (-1, 1), "SW": (1, -1), "SE": (1, 1)}
SEGS = ("c4", "c8", "m8")
SELS = ("all", "nonstatic", "interior", "shared")
UNITS = tuple(DIR8) + ("anchor", "anchor_same", "centre", "marker")
MAGS = (1, 2, 3, "size", "w", "h", "mk")
TABLES = ("const", "colour", "role")


def sgn(v): return (v > 0) - (v < 0)


def marker_colours(train):
    m = None
    for p in train:
        ci = {v for r in p["input"] for v in r}; co = {v for r in p["output"] for v in r}
        m = (ci - co) if m is None else (m & (ci - co)) | (m - ci)
    return m or set()


def segment(g, bg, seg, markers):
    gg = [[bg if v in markers else v for v in r] for r in g]
    obs = objects(gg, bg, diag=seg != "c4", by_colour=seg != "m8")
    return [[(y, x, g[y][x]) for y, x in ob] for ob in obs]


def select(obs, g, sel, st):
    h, w = H(g), W(g)
    if sel == "all": mv = [True] * len(obs)
    elif sel == "nonstatic": mv = [not all(c in st for _, _, c in ob) for ob in obs]
    elif sel == "interior": mv = [not any(y in (0, h - 1) or x in (0, w - 1) for y, x, _ in ob) for ob in obs]
    else:
        cnt = Counter(frozenset(c for _, _, c in ob) for ob in obs)
        mv = [cnt[frozenset(c for _, _, c in ob)] >= 2 for ob in obs]
    return [ob for ob, m in zip(obs, mv) if m], [ob for ob, m in zip(obs, mv) if not m]


def attach_markers(g, movers, markers):
    """Marker cells inside a mover's bounding box join it, recoloured to the mover's main colour."""
    h, w = H(g), W(g); out = []
    for ob in movers:
        cells = {(y, x) for y, x, _ in ob}
        main = Counter(c for _, _, c in ob).most_common(1)[0][0]
        r0, c0, r1, c1 = bbox(list(cells))
        mk = {(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if g[y][x] in markers}
        out.append((ob + [(y, x, main) for y, x in mk], mk))
    return out


def centre(cells):
    r0, c0, r1, c1 = bbox([(y, x) for y, x, *_ in cells])
    return (r0 + r1) / 2, (c0 + c1) / 2, (r0, c0, r1, c1)


def toward(ob, target_cells):
    """Unit step from object toward target; the component along an axis where projections overlap is 0."""
    _, _, (r0, c0, r1, c1) = centre(ob)
    _, _, (a0, b0, a1, b1) = centre(target_cells)
    dy = 0 if (a0 <= r1 and r0 <= a1) else sgn(a0 - r0)
    dx = 0 if (b0 <= c1 and c0 <= b1) else sgn(b0 - c0)
    return dy, dx


class Ctx:
    """Everything about one grid needed to evaluate a rule."""
    def __init__(self, g, seg, sel, markers, st, attach):
        self.g = g; self.bg = bg = bg_of(g); self.markers = markers
        obs = segment(g, bg, seg, markers)
        mov, rest = select(obs, g, sel, st)
        if attach:
            am = attach_markers(g, mov, markers)
            mov = [a for a, _ in am]; self.mk = [m for _, m in am]
        else:
            self.mk = [set() for _ in mov]
        # role items: split multicolour movers into colour parts
        self.parts = None
        if seg == "m8":
            parts = []
            for ob in mov:
                byc = {}
                for y, x, c in ob: byc.setdefault(c, []).append((y, x, c))
                ps = sorted(byc.values(), key=lambda p: centre(p)[0])
                for k, p in enumerate(ps):
                    rl = "solo" if len(ps) == 1 else "top" if k == 0 else "bottom" if k == len(ps) - 1 else "mid"
                    parts.append((p, rl))
            self.parts = parts
        self.movers, self.anchors = mov, rest
        self.nmk = sum(v in markers for r in g for v in r)

    def unit(self, i, u):
        ob = self.movers[i]
        if u in DIR8: return DIR8[u]
        if u == "centre":
            h, w = H(self.g), W(self.g)
            return toward(ob, [((h - 1) // 2, (w - 1) // 2), (h // 2, w // 2)])
        if u == "marker":
            if not self.mk[i]: return (0, 0)
            cy, cx, _ = centre(ob); my, mx, _ = centre(list(self.mk[i]))
            return sgn(my - cy), sgn(mx - cx)
        cands = self.anchors
        if u == "anchor_same":
            cs = {c for _, _, c in ob}
            cands = [a for a in cands if {c for _, _, c in a} & cs]
        if not cands: return (0, 0)
        cy, cx, _ = centre(ob)
        best = min(cands, key=lambda a: min(abs(y - cy) + abs(x - cx) for y, x, _ in a))
        return toward(ob, best)

    def mag(self, i, m):
        if isinstance(m, int): return m
        ob = self.movers[i]; r0, c0, r1, c1 = centre(ob)[2]
        return {"size": len(ob), "w": c1 - c0 + 1, "h": r1 - r0 + 1, "mk": self.nmk}[m]

    def items(self, table):
        """(cells, key) list for table rules."""
        if table == "role":
            return [(p, r) for p, r in self.parts] if self.parts is not None else None
        if table == "const": return [(ob, 0) for ob in self.movers]
        return [(ob, ob[0][2]) for ob in self.movers if len({c for _, _, c in ob}) == 1] \
            if all(len({c for _, _, c in ob}) == 1 for ob in self.movers) else None


def paint(g, bg, items, edge, collide):
    """items: list of (cells, (dy,dx), unit or None, steps)."""
    h, w = H(g), W(g); out = [r[:] for r in g]
    for cells, *_ in items:
        for y, x, _ in cells: out[y][x] = bg
    base = [r[:] for r in out]
    for cells, (dy, dx), unit, steps in items:
        if collide == "block":
            k = 0
            while k < steps:
                ny, nx = (k + 1) * unit[0], (k + 1) * unit[1]
                ok = True
                for y, x, _ in cells:
                    yy, xx = y + ny, x + nx
                    if not (0 <= yy < h and 0 <= xx < w) or base[yy][xx] != bg: ok = False; break
                if not ok: break
                k += 1
            dy, dx = k * unit[0], k * unit[1]
        for y, x, c in cells:
            yy, xx = y + dy, x + dx
            if edge == "wrap": yy %= h; xx %= w
            if 0 <= yy < h and 0 <= xx < w: out[yy][xx] = c
    return out


def disp_ok(o, cells, d):
    h, w = H(o), W(o); dy, dx = d
    for y, x, c in cells:
        yy, xx = y + dy, x + dx
        if 0 <= yy < h and 0 <= xx < w and o[yy][xx] != c: return False
    return True


def fam_move_by_vector(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    if all(p["input"] == p["output"] for p in train): return
    markers = marker_colours(train)
    st = static_colours(train)
    yielded = 0
    for seg in SEGS:
        for sel in SELS:
            if sel == "nonstatic" and not st: continue
            for attach in ((False, True) if markers else (False,)):
                ctxs = []
                for p in train:
                    c = Ctx(p["input"], seg, sel, markers, st, attach)
                    if not c.movers or len(c.movers) > 40: ctxs = None; break
                    ctxs.append(c)
                if not ctxs: continue
                progs = []
                # ---- unit * magnitude rules
                units = [u for u in UNITS if (u != "marker") == (not attach)]
                if attach: units = ["marker"]
                for u in units:
                    if u == "anchor_same" and seg == "m8": continue
                    ulist = [[c.unit(k, u) for k in range(len(c.movers))] for c in ctxs]
                    if all(v == (0, 0) for ul in ulist for v in ul): continue
                    for m in MAGS:
                        if m == "mk" and (not markers or attach): continue
                        mlist = [[c.mag(k, m) for k in range(len(c.movers))] for c in ctxs]
                        # prefilter: exact displacement must be consistent (ignoring blocking)
                        pre = all(disp_ok(p["output"], c.movers[k], (ul[k][0] * ml[k], ul[k][1] * ml[k]))
                                  for p, c, ul, ml in zip(train, ctxs, ulist, mlist) for k in range(len(c.movers)))
                        modes = [("over", "clip"), ("over", "wrap")] if pre else []
                        if not isinstance(m, int) or m <= 3: modes.append(("block", "clip"))
                        for collide, edge in modes:
                            progs.append((f"{u}*{m}", u, m, None, None, collide, edge))
                # ---- table rules: vector looked up by key
                for tb in TABLES:
                    if tb == "role" and seg != "m8": continue
                    if attach: continue
                    tabs = {}; good = True
                    for p, c in zip(train, ctxs):
                        its = c.items(tb)
                        if its is None: good = False; break
                        for cells, key in its:
                            h, w = H(p["output"]), W(p["output"])
                            y0, x0, col = cells[0]
                            D = {(yy - y0, xx - x0) for yy in range(h) for xx in range(w) if p["output"][yy][xx] == col}
                            D = {d for d in D if disp_ok(p["output"], cells, d)}
                            tabs[key] = tabs[key] & D if key in tabs else D
                            if not tabs[key]: good = False; break
                        if not good: break
                    if not good or not tabs: continue
                    tab = {k: min(v, key=lambda d: (abs(d[0]) + abs(d[1]), d)) for k, v in tabs.items()}
                    if all(v == (0, 0) for v in tab.values()): continue
                    if tb == "const" and sum(v != (0, 0) for v in tab.values()) == 0: continue
                    for edge in ("clip", "wrap"):
                        progs.append((f"{tb}", None, None, tb, tab, "over", edge))
                for name, u, m, tb, tab, collide, edge in progs:
                    fn = make(seg, sel, attach, markers, st, u, m, tb, tab, collide, edge)
                    if all(fn(p["input"]) == p["output"] for p in train):
                        yield (f"move-by-vector:{name}[{seg},{sel},{collide},{edge}{',attach' if attach else ''}]",
                               (4 if tb is None else 5) + (edge == 'wrap'), fn)
                        yielded += 1
                        if yielded >= 6: return
                        break


def make(seg, sel, attach, markers, st, u, m, tb, tab, collide, edge):
    def fn(g):
        c = Ctx(g, seg, sel, markers, st, attach)
        if not c.movers: return None
        g2 = [[c.bg if v in markers else v for v in r] for r in g]
        if tb is None:
            items = []
            for k, ob in enumerate(c.movers):
                un = c.unit(k, u); mg = c.mag(k, m)
                items.append((ob, (un[0] * mg, un[1] * mg), un, mg))
        else:
            its = c.items(tb)
            if its is None: return None
            items = []
            for cells, key in its:
                if key not in tab: return None
                items.append((cells, tab[key], None, 0))
        return paint(g2, c.bg, items, edge, collide)
    return fn


FAMILIES = (fam_move_by_vector,)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_slide_until_contact.py
"""Own primitive (no Codex code): slide-until-contact.

One parametrised primitive: every mover unit is translated along one of 4 directions until the next
step would hit a blocking cell (contact mode) or leave the grid, or (through mode) straight to the
grid edge ignoring obstacles, optionally painting the swept cells with an induced trail colour.
All parameters are induced from the training pairs:
  movers   : a colour set (a single colour whose cells change, or the whole set of changing colours)
  objects  : 4-conn single colour | 8-conn single colour | 4-conn multi colour
  unit     : whole object | 1-wide strips parallel to the motion axis
  floor    : global background | local background (the colour the unit sits on: region-aware)
  direction: const d | map(key->d) with key in {colour, hollow, open-concavity, size-threshold,
             region colour} | context {away from the anchor it touches, toward the full static line,
             away from the grid edge it touches}; 'stay' is an allowed map value
  recolour : optional map(key->colour) when moved units change colour by class
  stop     : contact | through(+trail colour)
The map / context is fitted from observed per-unit displacements in the training outputs; the search
harness then verifies the resulting program exactly on all pairs.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, fit_cmap, apply_cmap, static_colours

DIRS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
OPP = {'up': 'down', 'down': 'up', 'left': 'right', 'right': 'left'}
N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


# ------------------------------------------------------------------ objects and units
def components(g, M, diag, byc):
    h, w = H(g), W(g); seen = set(); out = []
    nb = N4 + (((-1, -1), (-1, 1), (1, -1), (1, 1)) if diag else ())
    for r in range(h):
        for c in range(w):
            if (r, c) in seen or g[r][c] not in M: continue
            col = g[r][c]; st = [(r, c)]; seen.add((r, c)); cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and g[yy][xx] in M \
                            and (not byc or g[yy][xx] == col):
                        seen.add((yy, xx)); st.append((yy, xx))
            out.append(sorted(cells))
    return out


def strips(cells, axis):
    """Split an object into maximal contiguous 1-wide runs along axis ('v' = columns, 'h' = rows)."""
    s = set(cells); out = []
    for y, x in sorted(cells, key=(lambda p: (p[1], p[0])) if axis == 'v' else (lambda p: p)):
        prev = (y - 1, x) if axis == 'v' else (y, x - 1)
        if prev in s: continue
        run = []; cy, cx = y, x
        while (cy, cx) in s:
            run.append((cy, cx)); cy, cx = (cy + 1, cx) if axis == 'v' else (cy, cx + 1)
        out.append(run)
    return out


# ------------------------------------------------------------------ unit features
def floor_of(g, cells, allmov, gbg, local):
    if not local: return gbg
    s = set(cells); cnt = Counter()
    for y, x in cells:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in s and (yy, xx) not in allmov:
                cnt[g[yy][xx]] += 1
    return cnt.most_common(1)[0][0] if cnt else gbg


def bbox_bg_split(cells):
    """(has enclosed hole, has open concavity) wrt the object's bounding box."""
    s = set(cells); ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    r0, r1, c0, c1 = min(ys), max(ys), min(xs), max(xs)
    empty = {(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s}
    if not empty: return False, False
    outside = {(y, x) for (y, x) in empty if y in (r0, r1) or x in (c0, c1)}
    st = list(outside); seen = set(outside)
    while st:
        y, x = st.pop()
        for dy, dx in N4:
            p = (y + dy, x + dx)
            if p in empty and p not in seen: seen.add(p); st.append(p)
    return len(seen) < len(empty), bool(seen)


def key_fns():
    return {
        'colour': lambda u: tuple(sorted({u['g'][y][x] for y, x in u['cells']})),
        'hollow': lambda u: u['shape'][0],
        'open': lambda u: u['shape'][1],
        'region': lambda u: u['floor'],
    }


# ------------------------------------------------------------------ contextual directions
def ctx_away_contact(u):
    g, s, fl = u['g'], set(u['cells']), u['floor']; sides = set()
    for y, x in u['cells']:
        for d, (dy, dx) in DIRS.items():
            yy, xx = y + dy, x + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in s and (yy, xx) not in u['allmov'] \
                    and g[yy][xx] != fl:
                sides.add(d)
    return OPP[next(iter(sides))] if len(sides) == 1 else 'stay'


def full_lines(g, fl):
    rows = [r for r in range(H(g)) if len(set(g[r])) == 1 and g[r][0] != fl]
    cols = [c for c in range(W(g)) if len({g[r][c] for r in range(H(g))}) == 1 and g[0][c] != fl]
    return rows, cols


def ctx_toward_line(u):
    rows, cols = u['lines']; ys = [y for y, _ in u['cells']]; xs = [x for _, x in u['cells']]
    best = None
    for r in rows:
        if r > max(ys): cand = (r - max(ys), 'down')
        elif r < min(ys): cand = (min(ys) - r, 'up')
        else: continue
        if best is None or cand[0] < best[0]: best = cand
    for c in cols:
        if c > max(xs): cand = (c - max(xs), 'right')
        elif c < min(xs): cand = (min(xs) - c, 'left')
        else: continue
        if best is None or cand[0] < best[0]: best = cand
    return best[1] if best else 'stay'


def ctx_away_edge(u):
    g = u['g']; ys = [y for y, _ in u['cells']]; xs = [x for _, x in u['cells']]
    t = [d for d, hit in (('down', min(ys) == 0), ('up', max(ys) == H(g) - 1),
                           ('right', min(xs) == 0), ('left', max(xs) == W(g) - 1)) if hit]
    return t[0] if len(t) == 1 else 'stay'


CTX = {'away-contact': ctx_away_contact, 'toward-line': ctx_toward_line, 'away-edge': ctx_away_edge}


# ------------------------------------------------------------------ units of a grid under a config
def make_units(g, M, omode, local, unit):
    gbg = bg_of(g)
    kind, arg = M
    M = {v for r in g for v in r} - {gbg}
    if kind == 'excl': M -= arg
    elif kind == 'only': M &= arg
    if gbg in M: return None
    diag, byc = omode
    objs = components(g, M, diag, byc)
    if not objs or len(objs) > 40: return None
    allmov = {p for ob in objs for p in ob}
    units = []
    for ob in objs:
        parts = [ob] if unit == 'obj' else strips(ob, unit)
        for cells in parts:
            fl = floor_of(g, cells, allmov, gbg, local)
            units.append({'g': g, 'cells': cells, 'floor': fl, 'allmov': allmov,
                          'size': len(ob), 'shape': bbox_bg_split(ob)})
    lines = full_lines(g, gbg)
    incol = {v for r in g for v in r}
    for u in units: u['lines'] = lines; u['incol'] = incol
    return units


# ------------------------------------------------------------------ simulation
def simulate(g, units, dirs, stop, trail, recol):
    h, w = H(g), W(g)
    out = [r[:] for r in g]
    pos = []
    for u, d in zip(units, dirs):
        pos.append([(y, x, g[y][x]) for y, x in u['cells']])
    if stop == 'through':
        paint = []
        for u, d, cur in zip(units, dirs, pos):
            for y, x, _ in cur: out[y][x] = u['floor']
        for i, (u, d) in enumerate(zip(units, dirs)):
            if d == 'stay': paint.append(pos[i]); continue
            dy, dx = DIRS[d]; cur = pos[i]; swept = []
            while all(0 <= y + dy < h and 0 <= x + dx < w for y, x, _ in cur):
                swept += [(y, x) for y, x, _ in cur]
                cur = [(y + dy, x + dx, c) for y, x, c in cur]
            if trail is not None:
                for y, x in swept: out[y][x] = trail
            paint.append(cur)
        for i, cur in enumerate(paint):
            for y, x, c in cur: out[y][x] = recol[i] if recol and recol[i] is not None else c
        return out
    # contact: step-wise simultaneous-ish settling, leading units first
    order = sorted(range(len(units)), key=lambda i: 0 if dirs[i] == 'stay' else
                   -max(y * DIRS[dirs[i]][0] + x * DIRS[dirs[i]][1] for y, x, _ in pos[i]))
    moving = True; guard = 0
    while moving and guard < 64:
        moving = False; guard += 1
        for i in order:
            d = dirs[i]
            if d == 'stay': continue
            dy, dx = DIRS[d]; cur = pos[i]; own = {(y, x) for y, x, _ in cur}; fl = units[i]['floor']
            k = 0
            while True:
                nxt = [(y + dy, x + dx) for y, x, _ in cur]
                if any(not (0 <= y < h and 0 <= x < w) or ((y, x) not in own and out[y][x] != fl)
                       for y, x in nxt):
                    break
                for y, x, _ in cur: out[y][x] = fl
                cur = [(y + dy, x + dx, c) for y, x, c in cur]
                for y, x, c in cur: out[y][x] = c
                own = {(y, x) for y, x, _ in cur}; k += 1
            if k: moving = True; pos[i] = cur
    if recol:
        for i, cur in enumerate(pos):
            if recol[i] is not None:
                for y, x, _ in cur: out[y][x] = recol[i]
    return out


# ------------------------------------------------------------------ induction from training
def observe(u, o):
    """Directions (and resulting uniform colour, or None if colours kept) consistent with the output."""
    g = u['g']; h, w = H(g), W(g); res = {}
    cells = u['cells']
    if all(o[y][x] == g[y][x] for y, x in cells): res['stay'] = None
    for d, (dy, dx) in DIRS.items():
        for k in range(1, max(h, w)):
            sh = [(y + dy * k, x + dx * k) for y, x in cells]
            if any(not (0 <= y < h and 0 <= x < w) for y, x in sh): break
            if all(o[y2][x2] == g[y][x] for (y, x), (y2, x2) in zip(cells, sh)):
                res.setdefault(d, None)
            else:
                cs = {o[y2][x2] for y2, x2 in sh}
                if len(cs) == 1 and len({g[y][x] for y, x in cells}) == 1:
                    c = next(iter(cs))
                    if c not in u['incol'] and d not in res: res[d] = c
    return res


def mover_sets(train):
    """Mover colour roles: all non-background | non-background minus the static colours | one colour
    whose cells change."""
    ch = set()
    for p in train:
        i, o = p['input'], p['output']
        for y in range(H(i)):
            for x in range(W(i)):
                if i[y][x] != o[y][x]: ch.add(i[y][x])
    out = [('nonbg', None)]
    st = frozenset(static_colours(train) - {bg_of(p['input']) for p in train})
    if st: out.append(('excl', st))
    out += [('only', frozenset([c])) for c in sorted(ch)]
    return out


def allowed(ob):
    """Directions consistent with an observation: a unit seen in place may have been blocked."""
    return set(ob) | (set(DIRS) if 'stay' in ob else set())


def colour_for(ob, d):
    return ob[d] if d in ob else ob.get('stay')


def fit(train_units, obs, keyname):
    """Yield (dir source, recolour map or None) consistent with the observations."""
    flat = [(u, ob) for us, os_ in zip(train_units, obs) for u, ob in zip(us, os_)]
    if keyname in CTX:
        f = CTX[keyname]; rec = {}
        for u, ob in flat:
            d = f(u)
            if d not in allowed(ob): return
            c = colour_for(ob, d)
            if rec.setdefault(d, c) != c: return
        yield 'ctx', (rec if any(v is not None for v in rec.values()) else None)
        return
    if keyname.startswith('size>'):
        t = int(keyname[5:]); kf = lambda u: u['size'] > t
    elif keyname == 'const':
        kf = lambda u: 0
    else:
        kf = key_fns()[keyname]
    cand = {}
    for u, ob in flat:
        k = kf(u); a = allowed(ob)
        cand[k] = cand[k] & a if k in cand else a
        if not cand[k]: return
    opts = {}
    for k, s in cand.items():
        mv = sorted(s - {'stay'})
        opts[k] = ['stay'] if 'stay' in s and len(mv) == 4 else mv
        if not opts[k]: return
    ks = sorted(opts, key=str); n = 0
    for combo in product(*[opts[k] for k in ks]):
        dmap = dict(zip(ks, combo))
        if all(v == 'stay' for v in dmap.values()): continue
        rec = {}; ok = True
        for u, ob in flat:
            k = kf(u); c = colour_for(ob, dmap[k])
            if rec.setdefault(k, c) != c: ok = False; break
        if not ok: continue
        yield (kf, dmap), (rec if any(v is not None for v in rec.values()) else None)
        n += 1
        if n >= 4: return


def fam_slide_until_contact(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    yielded = 0
    for M in mover_sets(train):
        for omode in ((False, True), (True, True), (False, False)):
            for local in (False, True):
                for unit in ('obj', 'v', 'h'):
                    TU = [make_units(p['input'], M, omode, local, unit) for p in train]
                    if any(t is None for t in TU): continue
                    OB = [[observe(u, p['output']) for u in us] for us, p in zip(TU, train)]
                    if any(not ob for os_ in OB for ob in os_): continue
                    sizes = sorted({u['size'] for us in TU for u in us})
                    keys = ['const', 'colour', 'hollow', 'open', 'region'] + \
                           ['size>%d' % s for s in sizes[:-1]] + list(CTX)
                    if unit != 'obj':
                        keys = ['const', 'colour', 'region'] + list(CTX)
                    for kn in keys:
                      for dsrc, rec in fit(TU, OB, kn):
                        for stop in ('contact', 'through'):
                            trail = None
                            prog = make_prog(M, omode, local, unit, kn, dsrc, rec, stop, None)
                            if stop == 'through':
                                trail = induce_trail(train, prog)
                                if trail == 'fail': continue
                                prog = make_prog(M, omode, local, unit, kn, dsrc, rec, stop, trail)
                            if not fits(train, prog): continue
                            name = 'slide:%s[movers=%s,%s%s,%s,unit=%s,dir=%s%s%s]' % (
                                stop, M[0] if M[0] == 'nonbg' else M[0] + '|'.join(map(str, sorted(M[1]))), '8' if omode[0] else '4',
                                '' if omode[1] else ',multi', 'local' if local else 'global', unit, kn,
                                '' if dsrc == 'ctx' else ':' + ','.join('%s>%s' % kv for kv in sorted(dsrc[1].items(), key=str)),
                                ',recolour' if rec else '') + ('' if trail is None else '+trail%d' % trail)
                            yield (name, 4 + (kn != 'const') + (rec is not None) + (trail is not None), prog)
                            yielded += 1
                            if yielded >= 8: return


def make_prog(M, omode, local, unit, kn, dsrc, rec, stop, trail):
    def fn(g):
        units = make_units(g, M, omode, local, unit)
        if not units: return None
        if dsrc == 'ctx':
            dirs = [CTX[kn](u) for u in units]
        else:
            kf, dmap = dsrc
            ks = [kf(u) for u in units]
            if any(k not in dmap for k in ks): return None
            dirs = [dmap[k] for k in ks]
        if all(d == 'stay' for d in dirs): return None
        recol = None
        if rec:
            if dsrc == 'ctx': recol = [rec.get(d) for d in dirs]
            else:
                if any(k not in rec for k in ks): return None
                recol = [rec[k] for k in ks]
        return simulate(g, units, dirs, stop, trail, recol)
    return fn


def fits(train, prog):
    preds = []
    for p in train:
        try: r = prog(p['input'])
        except Exception: return False
        if r is None: return False
        preds.append(r)
    outs = [p['output'] for p in train]
    if preds == outs: return True
    m = fit_cmap(preds, outs)
    return bool(m) and all(apply_cmap(a, m) == b for a, b in zip(preds, outs))


def induce_trail(train, prog):
    """Through mode: the one colour painted on swept cells, or None when no trail is needed."""
    cs = set()
    for p in train:
        r = prog(p['input'])
        if r is None: return 'fail'
        o = p['output']
        cs |= {o[y][x] for y in range(H(o)) for x in range(W(o)) if r[y][x] != o[y][x]}
    if not cs: return None
    return next(iter(cs)) if len(cs) == 1 else 'fail'


FAMILIES = (fam_slide_until_contact,)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_to_position.py
"""move.to_position: a mover is placed so that a reference point of it coincides with a target point defined by an
anchor that exists in the input; the original is erased.

One primitive  place(movers, ref, target, dihedral, rest)  with the target resolver chosen from a small enumerated set:
  key_marker   ref = key-colour cells (or the mover's corner cell diagonal to its single key cell);
               target = isolated monochrome marker(s) of the same colour; optional dihedral, ties broken by making
               line-coloured cells face the full-length line (border / divider) of their colour; one-to-one assignment.
  clamp        ref = singleton mover; target = nearest cell of the (unique) multi-cell anchor object's bbox.
  lattice      ref = mover bbox; target = centre of the lattice cell (cells between rows/cols of a static dot colour).
  corner       ref = singleton inside a frame; target = outer corner of the frame (opposite / same / h- / v-mirrored).
  codes        ref = mover centre; target = every interior cell whose row-code and column-code (frame border cells)
               equal the mover colour (copies); unmatched movers stay.
  dock         ref = piece's hub-facing edge; target = hub side of the same colour; dihedral chosen so the piece's
               first layer is exactly flush with that side.
  edge_line    ref = full-length line through a frame; target = a frame edge (per orientation).
Every parameter (colours, roles, which edge/corner, paint order, dihedral) is induced from the training pairs by
enumerating a small domain; the harness verifies exactness on all pairs.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, static_colours

# ------------------------------------------------------------------ shared helpers
DIH = {
    "id": lambda y, x: (y, x), "r90": lambda y, x: (x, -y), "r180": lambda y, x: (-y, -x), "r270": lambda y, x: (-x, y),
    "fh": lambda y, x: (y, -x), "fv": lambda y, x: (-y, x), "T": lambda y, x: (x, y), "aT": lambda y, x: (-x, -y)}


def same_shape(train):
    return all(H(p["input"]) == H(p["output"]) and W(p["input"]) == W(p["output"]) for p in train)


def tcells(cells, d):
    """cells: list of (y,x,c) -> transformed and normalised to min (0,0)."""
    f = DIH[d]; t = [(*f(y, x), c) for y, x, c in cells]
    my = min(y for y, _, _ in t); mx = min(x for _, x, _ in t)
    return [(y - my, x - mx, c) for y, x, c in t]


def coloured(g, ob): return [(y, x, g[y][x]) for y, x in ob]


def touches_edge(ob, g):
    return any(y in (0, H(g) - 1) or x in (0, W(g) - 1) for y, x in ob)


def spans(ob, g):
    y0, x0, y1, x1 = bbox(ob)
    return y1 - y0 + 1 == H(g) or x1 - x0 + 1 == W(g)


def major_lines(g, bg):
    """Rows/cols more than half covered by one non-bg colour: list of (axis, index, colour)."""
    out = []
    for y in range(H(g)):
        c, n = Counter(g[y]).most_common(1)[0]
        if c != bg and 2 * n > W(g): out.append(("r", y, c))
    for x in range(W(g)):
        c, n = Counter(g[y][x] for y in range(H(g))).most_common(1)[0]
        if c != bg and 2 * n > H(g): out.append(("c", x, c))
    return out


def full_lines(g, bg):
    """Full-length uniform non-bg rows/cols: list of (axis, index, colour)."""
    out = []
    for y in range(H(g)):
        if len(set(g[y])) == 1 and g[y][0] != bg: out.append(("r", y, g[y][0]))
    for x in range(W(g)):
        col = {g[y][x] for y in range(H(g))}
        if len(col) == 1 and g[0][x] != bg: out.append(("c", x, g[0][x]))
    return out


def inb(g, y, x): return 0 <= y < H(g) and 0 <= x < W(g)


# ------------------------------------------------------------------ key_marker
def _key_marker(g, conn, ref, dih):
    bg = bg_of(g)
    lines = major_lines(g, bg)
    band = {(y, x) for a, i, c in lines for (y, x) in ([(i, x) for x in range(W(g))] if a == "r" else
                                                        [(y, i) for y in range(H(g))]) if g[y][x] == c}
    gm = [[bg if (y, x) in band else g[y][x] for x in range(W(g))] for y in range(H(g))]
    obs = [coloured(g, ob) for ob in objects(gm, bg, diag=conn == 8, by_colour=False)]
    multi = [ob for ob in obs if len({c for _, _, c in ob}) > 1 and not spans([(y, x) for y, x, _ in ob], g)]
    mono = [ob for ob in obs if len({c for _, _, c in ob}) == 1 and not spans([(y, x) for y, x, _ in ob], g)]
    mcols = {c for ob in multi for _, _, c in ob}
    markers = [ob for ob in mono if ob[0][2] in mcols]
    if not markers or not multi: return None
    kcols = {ob[0][2] for ob in markers}
    mk_of = {}
    for i, ob in enumerate(markers):
        for y, x, c in ob: mk_of[(y, x)] = (i, c)
    lcols = {c for _, _, c in lines}

    def score(cells):
        s = 0
        for y, x, c in cells:
            if c in lcols:
                s += min(abs((y if a == "r" else x) - i) for a, i, cc in lines if cc == c)
        return s

    plans = []
    for mi, mv in enumerate(multi):
        keys = [(y, x, c) for y, x, c in mv if c in kcols]
        if not keys: continue
        if ref == "corner" and len(keys) != 1: continue
        cands = {}
        for d in (DIH if dih else ("id",)):
            tc = tcells(mv, d); cs = {(y, x): c for y, x, c in tc}
            tk = [(y, x, c) for y, x, c in tc if c in kcols]
            if ref == "key":
                refs = tk
            else:
                ky, kx, kc = tk[0]; rr = []
                for dy in (-1, 1):
                    for dx in (-1, 1):
                        if (ky + dy, kx + dx) in cs and (ky + dy, kx) in cs and (ky, kx + dx) in cs:
                            rr.append((ky + dy, kx + dx, kc))
                if len(rr) != 1: continue
                refs = rr
            y0, x0, c0 = refs[0]
            for (my, mx), (i, c) in mk_of.items():
                if c != c0: continue
                oy, ox = my - y0, mx - x0
                used = set(); ok = True
                for y, x, c in refs:
                    m = mk_of.get((y + oy, x + ox))
                    if m is None or m[1] != c: ok = False; break
                    used.add(m[0])
                if not ok: continue
                cov = {(y + oy, x + ox) for y, x, _ in refs}
                if any((y, x) not in cov for u in used for y, x, _ in markers[u]): continue
                placed = tuple(sorted((y + oy, x + ox, c) for y, x, c in tc))
                if not all(inb(g, y, x) for y, x, _ in placed): continue
                sc = (score(placed), d not in ("id", "r90", "r180", "r270"))
                if placed not in cands or sc < cands[placed][0]: cands[placed] = (sc, frozenset(used))
        if cands:
            plans.append((mi, len(keys), cands))
    if not plans: return None
    out = [r[:] for r in g]
    taken = set(); todo = []
    for mi, nk, cands in sorted(plans, key=lambda p: -p[1]):
        opts = [(sc, pl, u) for pl, (sc, u) in cands.items() if not (u & taken)]
        if not opts: continue
        opts.sort(key=lambda t: t[0])
        if len(opts) > 1 and opts[1][0] == opts[0][0]: return None
        _, pl, u = opts[0]; taken |= u; todo.append((mi, pl))
    if not todo: return None
    for mi, _ in todo:
        for y, x, _ in multi[mi]: out[y][x] = bg
    for u in taken:
        for y, x, _ in markers[u]: out[y][x] = bg
    for _, pl in todo:
        for y, x, c in pl: out[y][x] = c
    return out


def fam_key_marker(train):
    if not same_shape(train): return
    for conn in (4, 8):
        for ref in ("key", "corner"):
            for dih in (False, True):
                yield (f"to-position:key-marker[{conn},{ref},{'dih' if dih else 'id'}]", 4 + dih,
                       lambda g, conn=conn, ref=ref, dih=dih: _key_marker(g, conn, ref, dih))


# ------------------------------------------------------------------ clamp into anchor
def _clamp(g, rest):
    bg = bg_of(g)
    obs = objects(g, bg, diag=True, by_colour=True)
    big = [ob for ob in obs if len(ob) > 1]; small = [ob for ob in obs if len(ob) == 1]
    if len(big) != 1 or not small: return None
    r0, c0, r1, c1 = bbox(big[0])
    out = [r[:] for r in g]
    if rest == "erase":
        for y, x in big[0]: out[y][x] = bg
    tgt = {}
    for (y, x), in small:
        t = (min(max(y, r0), r1), min(max(x, c0), c1))
        if t in tgt or t == (y, x): return None
        tgt[t] = g[y][x]; out[y][x] = bg
    for (y, x), c in tgt.items(): out[y][x] = c
    return out


def fam_clamp(train):
    if not same_shape(train): return
    for rest in ("keep", "erase"):
        yield (f"to-position:clamp-into-anchor[{rest}]", 4, lambda g, rest=rest: _clamp(g, rest))


# ------------------------------------------------------------------ lattice cell centre
def _intervals(seps, n):
    b = [-1] + sorted(seps) + [n]
    return [(a + 1, c - 1) for a, c in zip(b, b[1:]) if c - 1 >= a + 1]


def _lattice(g, L, rnd):
    bg = bg_of(g)
    cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == L]
    if len(cells) < 4: return None
    R = {y for y, _ in cells}; C = {x for _, x in cells}
    if len(R) < 2 or len(C) < 2: return None
    # lattice: every L cell on a separator row AND column, or L covers whole rows/cols
    RI, CI = _intervals(R, H(g)), _intervals(C, W(g))
    gg = [[bg if v == L else v for v in r] for r in g]
    obs = objects(gg, bg, diag=True, by_colour=False)
    if not obs: return None
    out = [r[:] for r in g]
    paint = []
    for ob in obs:
        y0, x0, y1, x1 = bbox(ob); cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
        ri = [iv for iv in RI if iv[0] - .5 <= cy <= iv[1] + .5]; ci = [iv for iv in CI if iv[0] - .5 <= cx <= iv[1] + .5]
        if len(ri) != 1 or len(ci) != 1: return None
        (a, b), (c, d) = ri[0], ci[0]
        h, w = y1 - y0 + 1, x1 - x0 + 1
        if h > b - a + 1 or w > d - c + 1: return None
        ny = a + ((b - a + 1 - h) + rnd) // 2; nx = c + ((d - c + 1 - w) + rnd) // 2
        for y, x in ob:
            paint.append((y - y0 + ny, x - x0 + nx, g[y][x])); out[y][x] = bg
    for y, x, c in paint:
        if g[y][x] == L: return None
        out[y][x] = c
    return out


def fam_lattice(train):
    if not same_shape(train): return
    st = static_colours(train)
    cols = [c for c in st if c != bg_of(train[0]["input"]) and all(any(c in r for r in p["input"]) for p in train)]
    for L in cols[:3]:
        for rnd in (0, 1):
            yield (f"to-position:lattice-centre[{'floor' if not rnd else 'ceil'}]", 4,
                   lambda g, L=L, rnd=rnd: _lattice(g, L, rnd))


# ------------------------------------------------------------------ frame corner
def _corner(g, mode):
    bg = bg_of(g)
    obs = objects(g, bg, diag=True, by_colour=True)
    frames = [ob for ob in obs if len(ob) >= 8]
    if len(frames) != 1: return None
    r0, c0, r1, c1 = bbox(frames[0]); fs = set(frames[0])
    inner = [ob for ob in obs if ob is not frames[0]]
    if not inner or any(len(ob) != 1 for ob in inner): return None
    out = [r[:] for r in g]; paint = {}
    for (y, x), in inner:
        if not (r0 < y < r1 and c0 < x < c1): return None
        dy, dx = (y - r0) - (r1 - y), (x - c0) - (c1 - x)
        if dy == 0 or dx == 0: return None
        top, left = dy < 0, dx < 0
        if mode in ("opp", "v"): top = not top
        if mode in ("opp", "h"): left = not left
        t = (r0 - 1 if top else r1 + 1, c0 - 1 if left else c1 + 1)
        if not inb(g, *t) or t in paint: return None
        paint[t] = g[y][x]; out[y][x] = bg
    for (y, x), c in paint.items(): out[y][x] = c
    return out


def fam_corner(train):
    if not same_shape(train): return
    for mode in ("opp", "same", "h", "v"):
        yield (f"to-position:frame-outer-corner[{mode}]", 4, lambda g, mode=mode: _corner(g, mode))


# ------------------------------------------------------------------ border-code intersections
def _rect_bg(g, bg):
    best = None
    for ob in objects([[1 if v == bg else 0 for v in r] for r in g], 0, diag=False, by_colour=True):
        y0, x0, y1, x1 = bbox(ob)
        if len(ob) == (y1 - y0 + 1) * (x1 - x0 + 1) and y1 - y0 >= 2 and x1 - x0 >= 2:
            if best is None or len(ob) > best[0]: best = (len(ob), (y0, x0, y1, x1))
    return best and best[1]


def _codes(g, rowside, colside, order):
    bg = bg_of(g)
    rb = _rect_bg(g, bg)
    if not rb: return None
    r0, c0, r1, c1 = rb
    crs = [r0 - 1, r1 + 1] if rowside == "both" else [r0 - 1 if rowside == "top" else r1 + 1]
    ccs = [c0 - 1, c1 + 1] if colside == "both" else [c0 - 1 if colside == "left" else c1 + 1]
    if not all(inb(g, cr, c0) for cr in crs) or not all(inb(g, r0, cc) for cc in ccs): return None
    F = (r0 - 1, c0 - 1, r1 + 1, c1 + 1)
    movers = []
    for ob in objects(g, bg, diag=True, by_colour=True):
        y0, x0, y1, x1 = bbox(ob)
        if any(F[0] <= y <= F[2] and F[1] <= x <= F[3] for y, x in ob): continue
        if y1 - y0 + 1 >= H(g) or x1 - x0 + 1 >= W(g): continue
        if (y1 - y0) % 2 or (x1 - x0) % 2: continue
        movers.append(ob)
    if not movers: return None
    if order == "rev": movers = movers[::-1]
    out = [r[:] for r in g]; paint = []; hit = False
    for ob in movers:
        k = g[ob[0][0]][ob[0][1]]
        rows = [y for y in range(r0, r1 + 1) if all(g[y][cc] == k for cc in ccs)]
        cols = [x for x in range(c0, c1 + 1) if all(g[cr][x] == k for cr in crs)]
        if not rows or not cols: continue
        hit = True
        y0, x0, y1, x1 = bbox(ob); my, mx = (y0 + y1) // 2, (x0 + x1) // 2
        for y, x in ob: out[y][x] = bg
        for ty in rows:
            for tx in cols:
                paint.extend((y - my + ty, x - mx + tx, k) for y, x in ob)
    if not hit: return None
    for y, x, c in paint:
        if r0 <= y <= r1 and c0 <= x <= c1: out[y][x] = c
    return out


def fam_codes(train):
    if not same_shape(train): return
    for rs, cs in (("top", "left"), ("both", "both"), ("top", "right"), ("bottom", "left"), ("bottom", "right")):
            for order in ("fwd", "rev"):
                yield (f"to-position:code-intersections[{rs},{cs},{order}]", 5,
                       lambda g, rs=rs, cs=cs, order=order: _codes(g, rs, cs, order))


# ------------------------------------------------------------------ dock onto hub side
def _dock(g, rest):
    bg = bg_of(g)
    hubs = [ob for ob in objects(g, bg, diag=True, by_colour=False) if len({g[y][x] for y, x in ob}) >= 2]
    if len(hubs) != 1: return None
    hub = hubs[0]; hs = set(hub)
    hy0, hx0, hy1, hx1 = bbox(hub)
    pieces = [ob for ob in objects(g, bg, diag=True, by_colour=True) if not (set(ob) & hs)]
    if not pieces: return None
    out = [r[:] for r in g]
    for ob in pieces:
        for y, x in ob: out[y][x] = bg
    placedany = False
    for ob in pieces:
        k = g[ob[0][0]][ob[0][1]]
        S = [(y, x) for y, x in hub if g[y][x] == k]
        sides = []
        if S and all(x == hx0 for _, x in S): sides.append((0, -1))
        if S and all(x == hx1 for _, x in S): sides.append((0, 1))
        if S and all(y == hy0 for y, _ in S): sides.append((-1, 0))
        if S and all(y == hy1 for y, _ in S): sides.append((1, 0))
        if len(sides) != 1:
            if rest == "keep":
                for y, x in ob: out[y][x] = k
            continue
        dy, dx = sides[0]
        layer1 = {(y + dy, x + dx) for y, x in S}
        # projection coordinate along the outward direction
        proj = lambda y, x: y * dy + x * dx
        res = set()
        for d in DIH:
            tc = [(y, x) for y, x, _ in tcells([(y, x, 0) for y, x in ob], d)]
            lo = min(proj(y, x) for y, x in tc)
            first = [(y, x) for y, x in tc if proj(y, x) == lo]
            if len(first) != len(layer1): continue
            fy, fx = min(first); ty, tx = min(layer1)
            oy, ox = ty - fy, tx - fx
            if {(y + oy, x + ox) for y, x in first} != layer1: continue
            pl = frozenset((y + oy, x + ox) for y, x in tc)
            perp = (lambda y, x: x) if dy else (lambda y, x: y)
            if min(perp(*c) for c in pl) + max(perp(*c) for c in pl) != \
                    min(perp(*c) for c in S) + max(perp(*c) for c in S): continue
            if all(inb(g, y, x) for y, x in pl) and not (pl & hs): res.add(pl)
        if len(res) != 1:
            if res: return None
            if rest == "keep":
                for y, x in ob: out[y][x] = k
            continue
        placedany = True
        for y, x in next(iter(res)): out[y][x] = k
    return out if placedany else None


def fam_dock(train):
    if not same_shape(train): return
    for rest in ("erase", "keep"):
        yield (f"to-position:dock-on-hub-side[{rest}]", 5, lambda g, rest=rest: _dock(g, rest))


# ------------------------------------------------------------------ full-length lines snap to frame edge
def _edge_line(g, vside, hside):
    bg = bg_of(g)
    frames = []
    for ob in objects(g, bg, diag=False, by_colour=True):
        y0, x0, y1, x1 = bbox(ob)
        if y1 - y0 >= 2 and x1 - x0 >= 2 and set(ob) >= {(y, x) for y in (y0, y1) for x in range(x0, x1 + 1)} | \
                {(y, x) for x in (x0, x1) for y in range(y0, y1 + 1)}:
            frames.append((ob, (y0, x0, y1, x1)))
    if len(frames) != 1: return None
    ob, (r0, c0, r1, c1) = frames[0]; fc = g[ob[0][0]][ob[0][1]]
    vl = [x for x in range(W(g)) if all(g[y][x] != bg for y in range(H(g))) and
          len({g[y][x] for y in range(H(g))} - {fc}) == 1 and c0 < x < c1]
    hl = [y for y in range(H(g)) if all(v != bg for v in g[y]) and len(set(g[y]) - {fc}) == 1 and r0 < y < r1]
    if len(vl) > 1 or len(hl) > 1 or not (vl or hl): return None
    out = [r[:] for r in g]
    new = []
    for x in vl:
        L = ({g[y][x] for y in range(H(g))} - {fc}).pop()
        for y in range(H(g)):
            if out[y][x] == L: out[y][x] = bg
        tx = c0 if vside == "left" else c1
        new += [(y, tx, L) for y in range(H(g)) if not (r0 <= y <= r1)]
    for y in hl:
        L = (set(g[y]) - {fc}).pop()
        for x in range(W(g)):
            if out[y][x] == L: out[y][x] = bg
        ty = r0 if hside == "top" else r1
        new += [(ty, x, L) for x in range(W(g)) if not (c0 <= x <= c1)]
    for y, x, c in new: out[y][x] = c
    return out


def fam_edge_line(train):
    if not same_shape(train): return
    for vs in ("left", "right"):
        for hs in ("top", "bottom"):
            yield (f"to-position:line-to-frame-edge[{vs},{hs}]", 4, lambda g, vs=vs, hs=hs: _edge_line(g, vs, hs))


FAMILIES = (fam_key_marker, fam_clamp, fam_lattice, fam_corner, fam_codes, fam_dock, fam_edge_line)


In [ ]:
%%writefile /kaggle/working/probe/fam_move_toward_anchor.py
"""move.toward_anchor: every mover object travels straight toward the anchor it is paired with.

One parametrised primitive  attract(anchors, conn, pairing, mode, stop, unmatched, paint):
  anchors  : lines   (full rows/cols or frame sides of one non-bg colour)
             | static (components of colours that never change in training)
             | big    (per colour, the unique largest component of size >= 2)
             | all    (every component; used with the table pairing)
  conn     : mover segmentation, single-colour 4- or 8-connected components (non-anchor components)
  pairing  : same colour | any colour (nearest) | table (colour-compatibility pairs + allowed directions,
             both induced from the displacements observed in the training pairs)
  mode     : orth (only anchors whose row/col projection overlaps the mover; straight perpendicular motion)
             | king (step = sign vector toward the anchor's bbox, recomputed each step; 8 directions)
  stop     : touch8 / touch4 (stop when 8/4-adjacent to the anchor, obstacles ignored)
             | contact (stop when the next step hits any non-background cell; nearest movers settle first)
             | into (enter the anchor region and settle against its far side)
  unmatched: keep | erase   (movers with no reachable paired anchor)
  paint    : keep | rim     (moved mover drawn in the anchor's rim colour: the non-bg colour bordering it)
Every parameter is enumerated from a small generic domain or induced from the training pairs; programs are
verified on all training pairs before being yielded.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
sys.path.insert(0, '/home/claude/work/latent')
from collections import Counter
from gdsl import H, W, bg_of, objects, static_colours
from fam_move_slide_until_contact import fits

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def sgn(v): return (v > 0) - (v < 0)


def bb(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    return min(ys), min(xs), max(ys), max(xs)


# ------------------------------------------------------------------ anchors
def line_anchors(g, bg):
    h, w = H(g), W(g); out = []
    for r in range(h):
        inner = set(g[r][1:w - 1])
        if w >= 4 and len(inner) == 1 and g[r][1] != bg:
            c = g[r][1]; out.append((c, [(r, x) for x in range(w) if g[r][x] == c]))
    for x in range(w):
        inner = {g[r][x] for r in range(1, h - 1)}
        if h >= 4 and len(inner) == 1 and g[1][x] != bg:
            c = g[1][x]; out.append((c, [(r, x) for r in range(h) if g[r][x] == c]))
    return out


def split(g, sel, conn, st):
    """-> (anchors [(colour, cells)], movers [(colour, cells)]) or None."""
    bg = bg_of(g)
    comps = [(g[ob[0][0]][ob[0][1]], ob) for ob in objects(g, bg, diag=conn == 8, by_colour=True)]
    if len(comps) > 80: return None
    if sel == 'lines':
        an = line_anchors(g, bg)
        if not an: return None
        ac = {p for _, cs in an for p in cs}
        mv = [(c, [p for p in cs if p not in ac]) for c, cs in comps]
        return an, [(c, cs) for c, cs in mv if cs]
    if sel == 'static':
        an = [(c, cs) for c, cs in comps if c in st]
        mv = [(c, cs) for c, cs in comps if c not in st]
        return an, mv
    if sel == 'big':
        an = []; mv = []
        byc = {}
        for c, cs in comps: byc.setdefault(c, []).append(cs)
        for c, L in byc.items():
            L = sorted(L, key=len, reverse=True)
            if len(L[0]) >= 2 and (len(L) == 1 or len(L[0]) > len(L[1])):
                an.append((c, L[0])); mv += [(c, cs) for cs in L[1:]]
            else:
                mv += [(c, cs) for cs in L]
        return an, mv
    if sel == 'all':
        return comps, comps
    return None


def rim_colour(g, cells, bg):
    s = set(cells); col = g[cells[0][0]][cells[0][1]]; cnt = Counter()
    for y, x in cells:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in s and g[yy][xx] not in (bg, col):
                cnt[g[yy][xx]] += 1
    return cnt.most_common(1)[0][0] if cnt else None


# ------------------------------------------------------------------ geometry
def rel(mb, ab):
    """Sign vector from mover bbox toward anchor bbox (0 on overlapping axes) and the Chebyshev gap."""
    r0, c0, r1, c1 = mb; a0, b0, a1, b1 = ab
    dy = 1 if a0 > r1 else -1 if a1 < r0 else 0
    dx = 1 if b0 > c1 else -1 if b1 < c0 else 0
    gy = a0 - r1 if dy > 0 else r0 - a1 if dy < 0 else 0
    gx = b0 - c1 if dx > 0 else c0 - b1 if dx < 0 else 0
    return dy, dx, max(gy, gx)


def adjacent(cells, aset, nb):
    return any((y + dy, x + dx) in aset for y, x in cells for dy, dx in nb)


def choose(mc, mcells, anchors, pairing, mode, table, dirs):
    mb = bb(mcells); best = None; ms = set(mcells)
    for k, (ac, acells, ab) in enumerate(anchors):
        if acells is mcells or ms & set(acells): continue
        if pairing == 'same' and ac != mc: continue
        if pairing == 'table' and ac not in table.get(mc, ()): continue
        dy, dx, gap = rel(mb, ab)
        if dy == 0 and dx == 0: continue
        if mode == 'orth' and dy and dx: continue
        if dirs is not None and (sgn(dy), sgn(dx)) not in dirs: continue
        if best is None or gap < best[0]: best = (gap, k)
        elif gap == best[0]: best = (gap, None)
    return None if best is None or best[1] is None else best[1]


# ------------------------------------------------------------------ simulation
def attract(g, P, st, table=None, dirs=None):
    sel, conn, pairing, mode, stop, unmatched, paint = P
    sp = split(g, sel, conn, st)
    if sp is None: return None
    an, mv = sp
    if not an or not mv: return None
    h, w = H(g), W(g); bg = bg_of(g)
    anchors = [(c, cs, bb(cs)) for c, cs in an]
    out = [r[:] for r in g]
    plan = []
    for mc, cells in mv:
        k = choose(mc, cells, anchors, pairing, mode, table, dirs)
        plan.append((mc, cells, k))
    if all(k is None for _, _, k in plan) and unmatched == 'keep': return None
    moving = [(mc, cells, k) for mc, cells, k in plan if k is not None or unmatched == 'erase']
    for mc, cells, k in moving:
        for y, x in cells: out[y][x] = bg
    # nearest movers settle first
    order = sorted([p for p in moving if p[2] is not None], key=lambda p: rel(bb(p[1]), anchors[p[2]][2])[2])
    for mc, cells, k in order:
        ac, acells, ab = anchors[k]; aset = set(acells)
        cols = [g[y][x] for y, x in cells]; cur = list(cells)
        dy0, dx0, _ = rel(bb(cur), ab)
        for _ in range(h + w):
            dy, dx, _g = rel(bb(cur), ab)
            if mode == 'orth' or stop == 'into': dy, dx = dy0, dx0
            if stop in ('touch8', 'touch4'):
                if adjacent(cur, aset, N8 if stop == 'touch8' else N4) or (dy == 0 and dx == 0): break
            nxt = [(y + dy, x + dx) for y, x in cur]
            if any(not (0 <= y < h and 0 <= x < w) for y, x in nxt): break
            if stop == 'contact':
                if any(out[y][x] != bg for y, x in nxt): break
            elif stop == 'into':
                if all(p in aset for p in cur) and not all(p in aset for p in nxt): break
            else:
                if any(p in aset for p in nxt): break
            cur = nxt
        if stop == 'into' and not all(p in aset for p in cur): return None
        rc = rim_colour(g, acells, bg) if paint == 'rim' else None
        if paint == 'rim' and rc is None: return None
        for (y, x), c in zip(cur, cols): out[y][x] = rc if rc is not None else c
    return out


# ------------------------------------------------------------------ induction of the table pairing
def induce_table(train, conn):
    """Colour pairs (mover, partner) and step directions observed: a component that reappears translated
    straight along one of 8 directions with a different component 8-adjacent ahead of it."""
    table = {}; dirs = set()
    for p in train:
        i, o = p['input'], p['output']; bg = bg_of(i); h, w = H(i), W(i)
        for ob in objects(i, bg, diag=conn == 8, by_colour=True):
            c = i[ob[0][0]][ob[0][1]]
            if all(o[y][x] == c for y, x in ob): continue
            hit = None
            for dy, dx in N8:
                for k in range(1, max(h, w)):
                    sh = [(y + dy * k, x + dx * k) for y, x in ob]
                    if any(not (0 <= y < h and 0 <= x < w) for y, x in sh): break
                    if all(o[y][x] == c for y, x in sh):
                        ahead = {(y + dy, x + dx) for y, x in sh} - set(sh)
                        pc = {o[y][x] for y, x in ahead if 0 <= y < h and 0 <= x < w and o[y][x] not in (bg,)}
                        if len(pc) == 1:
                            hit = ((dy, dx), next(iter(pc))); break
                if hit: break
            if not hit: continue
            dirs.add(hit[0]); table.setdefault(c, set()).add(hit[1])
    if not table: return None, None
    # symmetric compatibility
    sym = {}
    for a, bs in table.items():
        for b in bs: sym.setdefault(a, set()).add(b); sym.setdefault(b, set()).add(a)
    return sym, dirs


# ------------------------------------------------------------------ family
CONFIGS = []
for sel in ('lines', 'static', 'big'):
    for pairing in ('same', 'any'):
        for mode in ('orth', 'king'):
            for stop in ('touch8', 'touch4', 'contact', 'into'):
                if stop == 'into' and (pairing != 'same' or mode != 'orth'): continue
                if mode == 'king' and stop == 'touch4' and sel == 'lines': continue
                for paint in (('keep', 'rim') if stop == 'into' else ('keep',)):
                    CONFIGS.append((sel, pairing, mode, stop, paint))
for mode in ('orth', 'king'):
    CONFIGS.append(('all', 'table', mode, 'touch8', 'keep'))


def fam_move_toward_anchor(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    st = static_colours(train) - {bg_of(p['input']) for p in train}
    n = 0; seen_ok = set()
    for conn in (4, 8):
        tabs = {}
        for sel, pairing, mode, stop, paint in CONFIGS:
            if sel == 'static' and not st: continue
            if not all(split(p['input'], sel, conn, st) for p in train[:1]): continue
            table = dirs = None
            if pairing == 'table':
                if conn not in tabs: tabs[conn] = induce_table(train, conn)
                table, dirs = tabs[conn]
                if table is None: continue
            for unmatched in ('keep', 'erase'):
                P = (sel, conn, pairing, mode, stop, unmatched, paint)
                prog = (lambda g, P=P, table=table, dirs=dirs: attract(g, P, st, table, dirs))
                try:
                    r0 = prog(i0)
                except Exception:
                    r0 = None
                if r0 is None or r0 == i0: continue
                if not fits(train, prog): continue
                key = tuple(tuple(map(tuple, prog(p['input']))) for p in train)
                if key in seen_ok: continue
                seen_ok.add(key)
                name = 'toward-anchor[%s,%d,%s,%s,%s,%s%s]' % (sel, conn, pairing, mode, stop, unmatched,
                                                                  ',rim' if paint == 'rim' else '')
                yield (name, 4 + (pairing == 'table') + (paint == 'rim') + (unmatched == 'erase'), prog)
                n += 1
                if n >= 6: return


FAMILIES = (fam_move_toward_anchor,)


In [ ]:
%%writefile /kaggle/working/probe/fam_ray_diagonal_from_object.py
"""ray.diagonal_from_object: rays emitted from selected corners of objects.

One parametrised primitive  corner_rays(pre, sel, src, key, paint, width, stop):
  pre   : none | scale k | canvas enlarge (input pasted at an induced corner of a larger blank canvas);
          k in {fixed ratio, #colours, #non-bg cells}, canvas base (H,W) or (M,M) with M=max(H,W)
  sel   : where rays start and which way they go
            outer   - convex outer corners of cells: a diagonal direction d is emitted when the diagonal
                      neighbour and both orthogonal neighbours on that side are background
            missing - an object (single colour, 8-conn) whose bounding box lacks exactly one corner cell
                      emits from that missing corner, outward
            radial  - every cell of a multicolour object emits in sign(cell - bbox centre) (8 directions),
                      when the next cell leaves the object
  src   : which cells may emit: all | big (colour with the largest bounding box) | vanish (colours present
          in every train input and absent from every output: flags)
  key   : directions on/off induced from training either globally or per source colour
  paint : own colour | the object's / grid's other colour | a constant colour seen in all outputs
  width : 1 (single-cell ray) | object (the whole object is swept along the direction, and recoloured)
  stop  : edge (paint background cells only, pass over the rest) | block (stop at the first non-background)
Everything is induced from the training pairs and every candidate is verified on all of them before it is
yielded.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox

DIAG = [(-1, -1), (-1, 1), (1, -1), (1, 1)]


def sgn(v): return (v > 0) - (v < 0)


def ncol(g, bg): return len({v for r in g for v in r} - {bg})
def ncell(g, bg): return sum(v != bg for r in g for v in r)


KRULES = {"ncol": ncol, "ncell": ncell}


def pre_candidates(train, bgc):
    """Yield (name, fn(g, bg) -> grid) consistent with all training shapes."""
    i0, o0 = train[0]["input"], train[0]["output"]
    shp = [((H(p["input"]), W(p["input"])), (H(p["output"]), W(p["output"]))) for p in train]
    if all(a == b for a, b in shp):
        yield "none", lambda g, bg: [r[:] for r in g]
        return
    def kfix(g, bg, k): return k
    rules = []
    if H(o0) % H(i0) == 0 and W(o0) % W(i0) == 0 and H(o0) // H(i0) == W(o0) // W(i0):
        rules.append(("k%d" % (H(o0) // H(i0)), lambda g, bg, k=H(o0) // H(i0): k))
    M0 = max(H(i0), W(i0))
    if H(o0) == W(o0) and H(o0) % M0 == 0 and ("k%d" % (H(o0) // M0)) not in [r[0] for r in rules]:
        rules.append(("k%d" % (H(o0) // M0), lambda g, bg, k=H(o0) // M0: k))
    for kn, kf in KRULES.items():
        rules.append((kn, kf))
    for kname, kf in rules:
        # scale
        def scale(g, bg, kf=kf):
            k = kf(g, bg)
            if k < 1 or k * max(H(g), W(g)) > 30: return None
            return [[v for v in r for _ in range(k)] for r in g for _ in range(k)]
        # canvas
        for base in ("hw", "mm"):
            for ay in (0, 1):
                for ax in (0, 1):
                    def canvas(g, bg, kf=kf, base=base, ay=ay, ax=ax):
                        k = kf(g, bg)
                        bh, bw = (H(g), W(g)) if base == "hw" else (max(H(g), W(g)),) * 2
                        oh, ow = bh * k, bw * k
                        if k < 1 or oh > 30 or ow > 30 or oh < H(g) or ow < W(g): return None
                        out = [[bg] * ow for _ in range(oh)]
                        y0 = 0 if ay == 0 else oh - H(g); x0 = 0 if ax == 0 else ow - W(g)
                        for y in range(H(g)):
                            for x in range(W(g)): out[y0 + y][x0 + x] = g[y][x]
                        return out
                    yield f"canvas[{base},{kname},{'tb'[ay]}{'lr'[ax]}]", canvas
        yield f"scale[{kname}]", scale


def grid_other(g, bg, own):
    cnt = Counter(v for r in g for v in r if v != bg and v != own)
    return cnt.most_common(1)[0][0] if cnt else None


def emissions(g, bg, sel, srcset):
    """List of (oy, ox, dy, dx, own, other, cells)."""
    h, w = H(g), W(g); em = []
    comps = objects(g, bg, diag=True, by_colour=False)
    comp_of = {}
    for ci, ob in enumerate(comps):
        for c in ob: comp_of[c] = ci
    def other_of(ci, own):
        cnt = Counter(g[y][x] for y, x in comps[ci] if g[y][x] != own)
        return cnt.most_common(1)[0][0] if cnt else grid_other(g, bg, own)
    def nb(y, x): return not (0 <= y < h and 0 <= x < w) or g[y][x] == bg
    if sel == "outer":
        for y in range(h):
            for x in range(w):
                v = g[y][x]
                if v == bg or (srcset is not None and v not in srcset): continue
                for dy, dx in DIAG:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and g[yy][xx] == bg and nb(yy, x) and nb(y, xx):
                        ci = comp_of[(y, x)]
                        em.append((y, x, dy, dx, v, other_of(ci, v), comps[ci]))
    elif sel == "cell":
        for y in range(h):
            for x in range(w):
                v = g[y][x]
                if v == bg or (srcset is not None and v not in srcset): continue
                ci = comp_of[(y, x)]
                for dy, dx in DIAG:
                    em.append((y, x, dy, dx, v, other_of(ci, v), comps[ci]))
    elif sel == "missing":
        for ob in objects(g, bg, diag=True, by_colour=True):
            v = g[ob[0][0]][ob[0][1]]
            if len(ob) < 2 or (srcset is not None and v not in srcset): continue
            r0, c0, r1, c1 = bbox(ob)
            if r0 == r1 or c0 == c1: continue
            s = set(ob)
            miss = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s]
            if len(miss) != 1: continue
            y, x = miss[0]
            if y not in (r0, r1) or x not in (c0, c1) or g[y][x] != bg: continue
            dy = -1 if y == r0 else 1; dx = -1 if x == c0 else 1
            ci = comp_of[ob[0]]
            em.append((y, x, dy, dx, v, other_of(ci, v), ob))
    elif sel == "radial":
        for ci, ob in enumerate(comps):
            if len(ob) < 2: continue
            r0, c0, r1, c1 = bbox(ob); s = set(ob)
            for y, x in ob:
                v = g[y][x]
                if srcset is not None and v not in srcset: continue
                dy, dx = sgn(2 * y - r0 - r1), sgn(2 * x - c0 - c1)
                if (dy, dx) == (0, 0) or (y + dy, x + dx) in s: continue
                em.append((y, x, dy, dx, v, other_of(ci, v), ob))
    return em


def paint_of(e, paint):
    if paint == "own": return e[4]
    if paint == "other": return e[5]
    return paint


def cast(g, bg, ems, on, keymode, paint, width, stop):
    h, w = H(g), W(g); out = [r[:] for r in g]
    for e in ems:
        oy, ox, dy, dx, own = e[:5]
        key = (dy, dx) if keymode == "global" else (own, dy, dx)
        if keymode != "global" and own not in {k[0] for k in on}: return None  # unseen source colour
        if not on.get(key, True): continue
        col = paint_of(e, paint)
        if col is None: return None
        if width == 1:
            y, x = oy + dy, ox + dx
            while 0 <= y < h and 0 <= x < w:
                if g[y][x] != bg:
                    if stop == "block": break
                else: out[y][x] = col
                y += dy; x += dx
        else:
            cells = e[6]; s = set(cells)
            for y, x in cells: out[y][x] = col
            k = 1
            while True:
                pts = [(y + k * dy, x + k * dx) for y, x in cells]
                pts = [(y, x) for y, x in pts if 0 <= y < h and 0 <= x < w]
                if not pts: break
                if stop == "block" and any(g[y][x] != bg and (y, x) not in s for y, x in pts): break
                for y, x in pts:
                    if g[y][x] == bg: out[y][x] = col
                k += 1
    return out


def induce_on(pairs, bgs, sel, srcsets, keymode, paint):
    """Vote directions on/off from the first ray cell. None if contradictory or nothing fires."""
    votes = {}
    for (g, o), bg, ss in zip(pairs, bgs, srcsets):
        for e in emissions(g, bg, sel, ss):
            oy, ox, dy, dx, own = e[:5]
            y, x = oy + dy, ox + dx
            if not (0 <= y < H(g) and 0 <= x < W(g)) or g[y][x] != bg: continue
            col = paint_of(e, paint)
            if col is None: return None
            key = (dy, dx) if keymode == "global" else (own, dy, dx)
            votes.setdefault(key, set()).add(o[y][x] == col)
    if any(len(v) > 1 for v in votes.values()): return None
    on = {k: True in v for k, v in votes.items()}
    if not any(on.values()): return None
    return on


def fam_corner_rays(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    bgs_in = [bg_of(p["input"]) for p in train]
    fixed_bg = bgs_in[0] if len(set(bgs_in)) == 1 else None
    def bgf(g):
        return fixed_bg if fixed_bg is not None and any(fixed_bg in r for r in g) else bg_of(g)
    ci_all = [{v for r in p["input"] for v in r} for p in train]
    co_all = [{v for r in p["output"] for v in r} for p in train]
    vanish = set.intersection(*ci_all) - set.union(*co_all)
    consts = set.intersection(*co_all) - {fixed_bg}
    found = 0
    for pname, pf in pre_candidates(train, fixed_bg):
        pairs = []
        for p in train:
            g = pf(p["input"], bgf(p["input"]))
            if g is None or (H(g), W(g)) != (H(p["output"]), W(p["output"])): pairs = None; break
            pairs.append((g, p["output"]))
        if not pairs: continue
        bgs = [bgf(p["input"]) for p in train]
        # cells that change must change from background (rays add, never erase) except sweep recolour
        for src in ("all", "big", "vanish"):
            if src == "vanish" and not vanish: continue
            def srcset_of(g, bg, src=src):
                if src == "all": return None
                if src == "vanish": return vanish
                cols = {v for r in g for v in r} - {bg}
                if not cols: return set()
                def area(c):
                    cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                    r0, c0, r1, c1 = bbox(cells); return (r1 - r0 + 1) * (c1 - c0 + 1)
                ar = sorted(((area(c), c) for c in cols), reverse=True)
                if len(ar) > 1 and ar[0][0] == ar[1][0]: return set()
                return {ar[0][1]}
            srcsets = [srcset_of(g, bg) for (g, _), bg in zip(pairs, bgs)]
            for sel in ("outer", "cell", "missing", "radial"):
                for paint in ["own", "other"] + sorted(consts):
                    for keymode in ("global", "colour"):
                        on = induce_on(pairs, bgs, sel, srcsets, keymode, paint)
                        if on is None: continue
                        for width in (1, "obj"):
                            for stop in ("edge", "block"):
                                def fn(g, pf=pf, sel=sel, srcset_of=srcset_of, on=on, keymode=keymode,
                                       paint=paint, width=width, stop=stop):
                                    bg = bgf(g); gg = pf(g, bg)
                                    if gg is None: return None
                                    ems = emissions(gg, bg, sel, srcset_of(gg, bg))
                                    if not ems: return None
                                    return cast(gg, bg, ems, on, keymode, paint, width, stop)
                                ok = True
                                for (g, o), p in zip(pairs, train):
                                    r = fn(p["input"])
                                    if r != o: ok = False; break
                                if not ok: continue
                                cost = 3 + (pname != "none") + (src != "all") + 2 * (keymode != "global") + (width != 1)
                                yield (f"diag-rays[{pname};{sel};{src};{keymode};{paint};w{width};{stop}]", cost, fn)
                                found += 1
                                if found >= 6: return
                                break  # one stop mode suffices when both fit


# ---------------------------------------------------------------- side segments of objects on a patterned bg
def bg_model(g):
    """background as a function (y,x)->colour: a 2-colour checkerboard covering most of the grid,
    else the plain majority colour."""
    h, w = H(g), W(g)
    cnt = [Counter(), Counter()]
    for y in range(h):
        for x in range(w): cnt[(y + x) % 2][g[y][x]] += 1
    a, na = cnt[0].most_common(1)[0]; b, nb = cnt[1].most_common(1)[0]
    if a != b and na + nb >= 0.8 * h * w:
        return lambda y, x: a if (y + x) % 2 == 0 else b
    c = bg_of(g)
    return lambda y, x: c

def side_segments(g, bgf):
    """per object (8-conn cells deviating from the background), each non-body colour must form straight
    diagonal runs on the object's boundary; yields (cells sorted along the run, outward direction)."""
    h, w = H(g), W(g)
    dev = {(y, x) for y in range(h) for x in range(w) if g[y][x] != bgf(y, x)}
    seen = set(); segs = []
    for st in sorted(dev):
        if st in seen: continue
        comp = []; stack = [st]; seen.add(st)
        while stack:
            y, x = stack.pop(); comp.append((y, x))
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    p = (y + dy, x + dx)
                    if p in dev and p not in seen: seen.add(p); stack.append(p)
        if len(comp) < 3: return None
        cols = Counter(g[y][x] for y, x in comp)
        body = cols.most_common(1)[0][0]
        cy = sum(y for y, _ in comp) / len(comp); cx = sum(x for _, x in comp) / len(comp)
        for c in cols:
            if c == body: continue
            cells = [p for p in comp if g[p[0]][p[1]] == c]
            # split into diagonal runs
            left = set(cells)
            while left:
                p0 = min(left); run = None
                for d in ((1, 1), (1, -1)):
                    r = [p0]
                    while (r[-1][0] + d[0], r[-1][1] + d[1]) in left: r.append((r[-1][0] + d[0], r[-1][1] + d[1]))
                    if len(r) >= 2 and (run is None or len(r) > len(run[0])): run = (r, d)
                if run is None: return None
                r, d = run
                left -= set(r)
                # outward: the perpendicular diagonal pointing away from the object's centre
                my = sum(y for y, _ in r) / len(r); mx = sum(x for _, x in r) / len(r)
                best = None
                for o in ((d[0], -d[1]), (-d[0], d[1])):
                    dot = o[0] * (my - cy) + o[1] * (mx - cx)
                    if best is None or dot > best[0]: best = (dot, o)
                if best[0] <= 0: return None
                segs.append((c, r, best[1]))
    return segs

def seg_rays(g, emit, stop):
    bgf = bg_model(g)
    segs = side_segments(g, bgf)
    if not segs: return None
    out = [r[:] for r in g]
    for c, r, (dy, dx) in segs:
        src = r if emit == "all" else [r[0], r[-1]]
        for y, x in src:
            yy, xx = y + dy, x + dx
            while 0 <= yy < H(g) and 0 <= xx < W(g):
                if g[yy][xx] != bgf(yy, xx):
                    if stop: break
                else: out[yy][xx] = c
                yy += dy; xx += dx
    return out

def fam_side_segment_rays(train):
    """objects on a (possibly checkerboard) background carry coloured straight diagonal side segments;
    each segment is swept outward (perpendicular, away from the object) to the edge.  Induced: which
    segment cells emit (the two ends, i.e. the band's outline, or all of them) and edge/block stop."""
    i0, o0 = train[0]["input"], train[0]["output"]
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if not side_segments(i0, bg_model(i0)): return
    for emit in ("ends", "all"):
        for stop in (False, True):
            yield ("diag-rays:side-segments[%s,%s]" % (emit, "block" if stop else "edge"), 5,
                   lambda g, emit=emit, stop=stop: seg_rays(g, emit, stop))

FAMILIES = (fam_corner_rays, fam_side_segment_rays)


In [ ]:
%%writefile /kaggle/working/probe/fam_ray_orthogonal_directed.py
"""ray.orthogonal_directed: seed cells emit axis-aligned half-rays whose direction is given by an
induced rule, painting the seed colour over background.

Seeds:   singleton cells (4-conn, by colour)  |  the odd (minority) cell of a multi-colour object.
Targets: a role, resolved per grid: 'big' = any cell of a non-singleton object,
         'bigsame' = non-singleton object of the seed's own colour, or one induced colour.
Direction rules (induced per seed colour from the training diffs, or uniform):
  toward  - every axis direction whose first non-bg cell is a target; paint up to contact
  away    - the opposite of each 'toward' direction; paint to the edge / first obstacle
  edges   - toward the nearest edge on each axis (ties -> no ray on that axis)
  through - odd cell: along the axis pointing from the odd cell into its object; the ray starts
            after the object and runs to the edge
Optional pre-step (only when outputs are smaller): crop to the rectangle bounded by the two
full rows and two full columns (lines with no background cell)."""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from gdsl import H, W, bg_of, objects, bbox

DIRS = ((-1, 0), (1, 0), (0, -1), (0, 1))


def _walk(g, bg, y, x, dy, dx, clear=None):
    """Cells from (y,x) exclusive in direction until the first non-bg cell; returns (path, hit).
    clear(cell) -> True marks a non-bg cell the ray passes over."""
    path = []; yy, xx = y + dy, x + dx
    while 0 <= yy < H(g) and 0 <= xx < W(g):
        if g[yy][xx] != bg and not (clear and clear((yy, xx))): return path, (yy, xx)
        path.append((yy, xx)); yy += dy; xx += dx
    return path, None


def _analyse(g):
    bg = bg_of(g)
    obs = objects(g, bg, False, True)
    size = {}
    for ob in obs:
        for c in ob: size[c] = len(ob)
    seeds = [ob[0] for ob in obs if len(ob) == 1]
    big = {}
    for ob in obs:
        c = g[ob[0][0]][ob[0][1]]
        if len(ob) > 1 and len(ob) > len(big.get(c, ())): big[c] = ob
    main = {cell for ob in big.values() for cell in ob}
    size['main'] = main
    size['selfseeds'] = [(y, x) for y in range(H(g)) for x in range(W(g))
                         if g[y][x] != bg and g[y][x] in big and (y, x) not in main]
    return bg, size, seeds


def _target(role, g, size, cell, seedcol):
    y, x = cell
    if role == 'big': return size.get(cell, 0) > 1
    if role == 'bigsame': return size.get(cell, 0) > 1 and g[y][x] == seedcol
    if role == 'mainself': return cell in size['main'] and g[y][x] == seedcol
    return g[y][x] == role


def _rays_cell(g, bg, size, y, x, rule, role):
    """Painted cells for one singleton seed under rule."""
    c = g[y][x]; out = []
    clear = (lambda cl: g[cl[0]][cl[1]] == c and cl not in size['main']) if role == 'mainself' else None
    if rule == 'edges':
        h, w = H(g), W(g)
        for a, b, d1, d2 in ((y, h - 1 - y, (-1, 0), (1, 0)), (x, w - 1 - x, (0, -1), (0, 1))):
            if a == b: continue
            d = d1 if a < b else d2
            out += _walk(g, bg, y, x, *d)[0]
        return out
    for dy, dx in DIRS:
        path, hit = _walk(g, bg, y, x, dy, dx, clear)
        if hit is None or not _target(role, g, size, hit, c): continue
        if rule == 'toward': out += path
        else: out += _walk(g, bg, y, x, -dy, -dx, clear)[0]
    return out


def _odd_rays(g, away=False):
    """Odd cell of each multi-colour object emits a ray through its object to the edge."""
    bg = bg_of(g); res = []
    for ob in objects(g, bg, True, False):
        cols = {}
        for y, x in ob: cols.setdefault(g[y][x], []).append((y, x))
        if len(cols) != 2: continue
        odd = min(cols.values(), key=len)
        if len(odd) != 1 or len(ob) < 3: continue
        (y, x), = odd
        cy = sum(a for a, _ in ob) / len(ob); cx = sum(b for _, b in ob) / len(ob)
        ddy, ddx = cy - y, cx - x
        if abs(ddy) == abs(ddx): continue
        d = ((1 if ddy > 0 else -1), 0) if abs(ddy) > abs(ddx) else (0, (1 if ddx > 0 else -1))
        if away: d = (-d[0], -d[1])
        obset = set(ob); yy, xx = y, x
        while (yy, xx) in obset: yy += d[0]; xx += d[1]
        while 0 <= yy < H(g) and 0 <= xx < W(g) and g[yy][xx] == bg:
            res.append((yy, xx, g[y][x])); yy += d[0]; xx += d[1]
    return res


def _paint_seeds(g, rules, role):
    """rules: dict colour->rule, or a str applied uniformly."""
    bg, size, seeds = _analyse(g)
    if role == 'mainself': seeds = size['selfseeds']
    out = [r[:] for r in g]; n = 0
    for y, x in seeds:
        c = g[y][x]
        if role not in ('big', 'bigsame', 'mainself') and c == role: continue
        r = rules if isinstance(rules, str) else rules.get(c, None if c in rules.get('_seen', ()) else '?')
        if r == '?': return None                    # unseen seed colour under per-colour rules
        if r is None: continue
        for yy, xx in _rays_cell(g, bg, size, y, x, r, role):
            out[yy][xx] = c; n += 1
    return out if n else None


def _frame_crop(g):
    bg = bg_of(g)
    rows = [y for y in range(H(g)) if all(v != bg for v in g[y])]
    cols = [x for x in range(W(g)) if all(g[y][x] != bg for y in range(H(g)))]
    if len(rows) != 2 or len(cols) != 2: return None
    return [r[cols[0]:cols[1] + 1] for r in g[rows[0]:rows[1] + 1]]


def _induce(pairs, role, rules=('toward', 'away', 'edges')):
    """Per seed colour, the rule that paints only correct changed cells and explains most of them."""
    stats = {}
    for i, o in pairs:
        bg, size, seeds = _analyse(i)
        if role == 'mainself': seeds = size['selfseeds']
        for y, x in seeds:
            c = i[y][x]
            if role not in ('big', 'bigsame', 'mainself') and c == role: continue
            s = stats.setdefault(c, {r: [True, 0] for r in rules})
            for r in rules:
                for yy, xx in _rays_cell(i, bg, size, y, x, r, role):
                    if o[yy][xx] == c: s[r][1] += 1
                    else: s[r][0] = False
    res = {}
    for c, s in stats.items():
        ok = [(v[1], r) for r, v in s.items() if v[0] and v[1] > 0]
        res[c] = max(ok)[1] if ok else None
    return res


def fam_directed_rays(train):
    i0, o0 = train[0]['input'], train[0]['output']
    same = (H(i0), W(i0)) == (H(o0), W(o0))
    pre = None
    if not same:
        c0 = _frame_crop(i0)
        if c0 is None or (H(c0), W(c0)) != (H(o0), W(o0)): return
        pre = _frame_crop
    pairs = []
    for p in train:
        i = pre(p['input']) if pre else p['input']
        if i is None or (H(i), W(i)) != (H(p['output']), W(p['output'])): return
        # rays only add colour onto background
        bg = bg_of(i)
        if pre is None and any(i[y][x] != bg and i[y][x] != p['output'][y][x] for y in range(H(i)) for x in range(W(i))): return
        pairs.append((i, p['output']))
    wrap = (lambda f: (lambda g: (lambda c: f(c) if c is not None else None)(pre(g)))) if pre else (lambda f: f)
    tag = 'frame-crop>' if pre else ''
    # colours present in every input (candidate explicit target colours)
    common = None
    for i, _ in pairs:
        cs = {v for r in i for v in r} - {bg_of(i)}
        common = cs if common is None else common & cs
    roles = ['big', 'bigsame', 'mainself'] + sorted(common or ())
    seen = set()
    for role in roles:
        rules = _induce(pairs, role)
        used = {r for r in rules.values() if r}
        if not used: continue
        if len(used) == 1:
            r = used.pop(); key = (r, role if r != 'edges' else '')
            if key in seen: continue
            seen.add(key)
            yield (f"{tag}ray:{r}[target={role}]" if r != 'edges' else f"{tag}ray:edges", 4,
                   wrap(lambda g, r=r, role=role: _paint_seeds(g, r, role)))
        else:
            rr = dict(rules); rr['_seen'] = set(rules)
            yield (f"{tag}ray:per-colour{sorted((k, v) for k, v in rules.items() if v)}[target={role}]", 5,
                   wrap(lambda g, rr=rr, role=role: _paint_seeds(g, rr, role)))
    if pre is None:
        for away in (False, True):
            def odd(g, away=away):
                res = _odd_rays(g, away)
                if not res: return None
                out = [r[:] for r in g]
                for y, x, c in res: out[y][x] = c
                return out
            yield ("ray:odd-cell-" + ("away-from-object" if away else "through-object"), 4, odd)


# ---------------------------------------------------------------- marker-keyed rays from congruent bars
def _canon(g, ob):
    """ob: cells of a solid rectangle t x L (t < L).  Returns (pattern, coord) where coord[r][p] is the
    original cell at canonical row r, position p, for the lexicographically smallest of the 8 frames
    (first such frame on ties)."""
    from gdsl import T, fh, fv
    y0, x0, y1, x1 = bbox(ob)
    if len(ob) != (y1 - y0 + 1) * (x1 - x0 + 1): return None
    cg = [[(y, x) for x in range(x0, x1 + 1)] for y in range(y0, y1 + 1)]
    if len(cg) > len(cg[0]): cg = T(cg)
    t, L = len(cg), len(cg[0])
    if t < 2 or t >= L: return None
    best = None
    for f in (lambda a: a, fh, fv, lambda a: fh(fv(a))):
        c = f(cg); pat = tuple(tuple(g[y][x] for y, x in r) for r in c)
        if best is None or pat < best[0]: best = (pat, c)
    return best

def _bars_markers(g):
    bg = bg_of(g); bars = []; marks = []
    for ob in objects(g, bg, False, False):
        if len(ob) == 1:
            y, x = ob[0]
            if y in (0, H(g) - 1) or x in (0, W(g) - 1): marks.append((y, x))
            else: return None
            continue
        cb = _canon(g, ob)
        if cb is None: return None
        bars.append(cb)
    if not bars or len({b[0] for b in bars}) != 1: return None
    return bg, bars, marks

def _face_rays(g, bg, coord, face, p):
    t = len(coord); a = coord[0 if face == 0 else t - 1][p]; b = coord[1 if face == 0 else t - 2][p]
    dy, dx = a[0] - b[0], a[1] - b[1]
    y, x = a[0] + dy, a[1] + dx; path = []
    while 0 <= y < H(g) and 0 <= x < W(g):
        path.append((y, x)); y += dy; x += dx
    return path

def _marker_legend(g):
    r = _bars_markers(g)
    if r is None: return None
    bg, bars, marks = r
    face = None; leg = {}
    for my, mx in marks:
        hit = []
        for pat, coord in bars:
            for f in (0, 1):
                for p in range(len(coord[0])):
                    path = _face_rays(g, bg, coord, f, p)
                    if path and path[-1] == (my, mx) and all(g[y][x] == bg for y, x in path[:-1]):
                        hit.append((f, p))
        if len(hit) != 1: return None
        f, p = hit[0]
        if face not in (None, f) or leg.get(p, g[my][mx]) != g[my][mx]: return None
        face = f; leg[p] = g[my][mx]
    if face is None: return None
    return bg, bars, face, leg

def _marker_bar_rays(g, stop):
    r = _marker_legend(g)
    if r is None: return None
    bg, bars, face, leg = r
    out = [row[:] for row in g]
    for pat, coord in bars:
        for p, c in leg.items():
            for y, x in _face_rays(g, bg, coord, face, p):
                if g[y][x] != bg:
                    if stop and g[y][x] != c: break
                    continue
                out[y][x] = c
    return out

def fam_marker_bar_rays(train):
    """Congruent solid bars (same pattern up to the dihedral frame) and single border markers.
    Each marker sits at the end of a straight clear line leaving one face of one bar at some position
    along it; the markers jointly give a legend (face, position -> colour) in the bars' canonical frame,
    and every bar emits those rays from that face to the edge."""
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)) or _marker_legend(i0) is None: return
    for stop in (False, True):
        yield ("ray:marker-legend-bars[%s]" % ("stop" if stop else "through"), 5,
               lambda g, stop=stop: _marker_bar_rays(g, stop))

FAMILIES = (fam_directed_rays, fam_marker_bar_rays)


In [ ]:
%%writefile /kaggle/working/probe/fam_ray_walker_turn_split.py
"""ray.walker_turn_split: a walker (ray) leaves a seed and travels straight; when the next cell is
blocked it reacts by an induced POLICY, and every visited background cell is painted.

One engine, one parameter set, all induced from the training pairs:
  seed    - cells of an induced colour s: straight segments (orth or diagonal) emit from both ends
            along their axis; single cells on the grid edge emit away from that edge.
            (spiral: the unique object of colour s - a single cell or a plus; portal: the object whose
            body colour is unique.)
  paint   - the seed colour, or the one colour new in every output.
  policy  -
    split    blocked -> two walkers, perpendicular both ways (and so on recursively)
    flow     global direction d0 (gravity-like): whenever the cell ahead in d0 is free go d0;
             blocked in d0 -> spread sideways both ways; a sideways walker stops when blocked
    reflect  diagonal walker bounces off obstacles (flip the blocked component)
    turn     blocked by colour x -> turn cw / ccw per an induced colour->turn map
    spiral   no deflection by obstacles (they stop the walker); the walker hugs its own/its
             siblings' trail with a 1-cell gap, turning cw/ccw (outward square spiral / pinwheel);
             drawn on a virtual canvas so arms may leave and re-enter the grid
    portal   the walker (a beam as wide as the emitter's marker) enters the object it hits, the
             object takes the paint colour and the beam re-emerges from that object's marker cells
             in the marker direction; objects never reached are erased (or kept)
Obstacles = non-background cells that are neither seed nor paint colour."""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, objects

ORTH = ((-1, 0), (0, 1), (1, 0), (0, -1))          # N E S W  (cw order)


def cw(d): return (d[1], -d[0])
def ccw(d): return (-d[1], d[0])


def _same_shape(train):
    return all(H(p['input']) == H(p['output']) and W(p['input']) == W(p['output']) for p in train)


def _new_colour(train):
    s = None
    for p in train:
        new = {v for r in p['output'] for v in r} - {v for r in p['input'] for v in r}
        if len(new) != 1: return None
        c = next(iter(new))
        if s is not None and c != s: return None
        s = c
    return s


def _only_bg_changes(train):
    for p in train:
        i, o = p['input'], p['output']; bg = bg_of(i)
        for y in range(H(i)):
            for x in range(W(i)):
                if i[y][x] != o[y][x] and i[y][x] != bg: return False
    return True


# ---------------------------------------------------------------- seeds
def _line_dir(cells):
    """Return unit axis (dy,dx) if cells form one straight contiguous line of length>=2, else None."""
    cells = sorted(cells)
    if len(cells) < 2: return None
    dy, dx = cells[1][0] - cells[0][0], cells[1][1] - cells[0][1]
    if max(abs(dy), abs(dx)) != 1: return None
    for a, b in zip(cells, cells[1:]):
        if (b[0] - a[0], b[1] - a[1]) != (dy, dx): return None
    return (dy, dx)


def _emitters(g, s, bg):
    """List of (y,x,dy,dx): walker positions (the seed end) and headings."""
    h, w = H(g), W(g); out = []
    for ob in objects(g, bg, True, True):
        if g[ob[0][0]][ob[0][1]] != s: continue
        if len(ob) == 1:
            y, x = ob[0]; ds = []
            if y == 0: ds.append((1, 0))
            if y == h - 1: ds.append((-1, 0))
            if x == 0: ds.append((0, 1))
            if x == w - 1: ds.append((0, -1))
            if len(ds) != 1: continue
            out.append((y, x) + ds[0])
            continue
        d = _line_dir(ob)
        if d is None: return None
        cs = sorted(ob)
        out.append(cs[-1] + d); out.append(cs[0] + (-d[0], -d[1]))
    return out


# ---------------------------------------------------------------- deflecting walkers
def _walk(g, bg, s, paint, emit, policy, tmap=None, minrun=0):
    h, w = H(g), W(g)
    def free(y, x): return g[y][x] in (bg, s, paint)
    def inside(y, x): return 0 <= y < h and 0 <= x < w
    seen = set(); st = []; painted = set()
    for e in emit:
        y, x, dy, dx = e
        if policy == 'reflect' and (dy == 0 or dx == 0): return None
        if policy != 'reflect' and dy and dx: return None
        st.append((y, x, dy, dx, (dy, dx), min(1, minrun)))
    guard = 0
    while st:
        guard += 1
        if guard > 40000: return None
        stt = st.pop()
        if stt in seen: continue
        seen.add(stt); y, x, dy, dx, d0, mv = stt
        painted.add((y, x))
        if policy == 'flow':
            ay, ax = y + d0[0], x + d0[1]
            if (dy, dx) != d0:
                if not inside(ay, ax): continue
                if free(ay, ax): st.append((y, x) + d0 + (d0, 0)); continue
        ny, nx = y + dy, x + dx
        if not inside(ny, nx): continue
        if free(ny, nx): st.append((ny, nx, dy, dx, d0, min(mv + 1, minrun))); continue
        if mv < minrun: continue
        if policy == 'split':
            for nd in (cw((dy, dx)), ccw((dy, dx))): st.append((y, x) + nd + (d0, 0))
        elif policy == 'flow':
            if (dy, dx) == d0:
                for nd in (cw(d0), ccw(d0)): st.append((y, x) + nd + (d0, 0))
        elif policy == 'turn':
            t = tmap.get(g[ny][nx])
            if t is None: return None
            nd = cw((dy, dx)) if t == 'cw' else ccw((dy, dx))
            st.append((y, x) + nd + (d0, 0))
        elif policy == 'reflect':
            by = inside(y + dy, x) and not free(y + dy, x)
            bx = inside(y, x + dx) and not free(y, x + dx)
            if by and not bx: nd = (-dy, dx)
            elif bx and not by: nd = (dy, -dx)
            else: nd = (-dy, -dx)
            st.append((y, x) + nd + (d0, 0))
    out = [r[:] for r in g]
    for y, x in painted:
        if g[y][x] == bg: out[y][x] = paint
    return out


def fam_walker(train):
    if not _same_shape(train) or not _only_bg_changes(train): return
    i0 = train[0]['input']; bg0 = bg_of(i0)
    common = set.intersection(*[{v for r in p['input'] for v in r} for p in train]) - {bg0}
    newc = _new_colour(train)
    n = 0
    for s in sorted(common):
        ems = [_emitters(p['input'], s, bg_of(p['input'])) for p in train]
        if any(not e for e in ems): continue
        paints = [('seed', s)] + ([('new', newc)] if newc is not None else [])
        obst = sorted({v for p in train for r in p['input'] for v in r} - {bg0, s} - {c for _, c in paints})
        for pn, pc in paints:
            pols = [('split', None), ('split2', None), ('flow', None), ('reflect', None)]
            oc = [c for c in obst if c != pc]
            if 1 <= len(oc) <= 3:
                for combo in product(('cw', 'ccw'), repeat=len(oc)):
                    if len(set(combo)) == 1: continue
                    pols.append(('turn', dict(zip(oc, combo))))
            for pol, tm in pols:
                def fn(g, s=s, pc=pc, pol=pol, tm=tm):
                    bg = bg_of(g); e = _emitters(g, s, bg)
                    if not e: return None
                    if pol == 'split2': return _walk(g, bg, s, pc, e, 'split', tm, 2)
                    return _walk(g, bg, s, pc, e, pol, tm)
                # cheap prefilter on train[0]
                p0 = train[0]
                if fn(p0['input']) != p0['output']: continue
                tag = pol if tm is None else 'turn[' + ','.join(f'{k}:{v}' for k, v in sorted(tm.items())) + ']'
                n += 1
                yield (f'walker[seed={s},paint={pn},policy={tag}]', 4 if tm is None else 5, fn)
                if n > 20: return


# ---------------------------------------------------------------- spiral / pinwheel
def _spiral(g, bg, s, arms, rot, runin, recent=3):
    """Lockstep walkers that keep a 1-cell gap from everything painted except their own last
    `recent` cells, preferring to turn toward `rot` (hug), else straight, else away."""
    h, w = H(g), W(g); P = 2 * max(h, w) + 4
    turn, unturn = (cw, ccw) if rot == 'cw' else (ccw, cw)
    painted = set()
    for y in range(h):
        for x in range(w):
            if g[y][x] == s: painted.add((y, x))
    walkers = []
    for (y, x, d) in arms:
        trail = [(y, x)]; yy, xx = y - d[0], x - d[1]
        while (yy, xx) in painted and len(trail) < recent: trail.insert(0, (yy, xx)); yy -= d[0]; xx -= d[1]
        walkers.append([y, x, d, 0, True, trail])
    NB = [(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if a or b]
    def ok(c, own):
        if c in painted: return False
        for a, b in NB:
            n = (c[0] + a, c[1] + b)
            if n in painted and n not in own: return False
        return True
    steps = 0
    while any(wk[4] for wk in walkers):
        for wk in walkers:
            if not wk[4]: continue
            steps += 1
            if steps > 200000: return None
            y, x, d, k, _, trail = wk
            own = set(trail[-recent:])
            opts = [turn(d), d, unturn(d)] if k >= runin else [d]
            nd = None
            for o in opts:
                if ok((y + o[0], x + o[1]), own): nd = o; break
            if nd is None: wk[4] = False; continue
            ny, nx = y + nd[0], x + nd[1]
            if not (-P <= ny < h + P and -P <= nx < w + P): wk[4] = False; continue
            if 0 <= ny < h and 0 <= nx < w and g[ny][nx] not in (bg, s): wk[4] = False; continue
            painted.add((ny, nx)); trail.append((ny, nx))
            if len(trail) > recent: del trail[0]
            wk[:4] = [ny, nx, nd, k + 1]
    out = [r[:] for r in g]
    for y, x in painted:
        if 0 <= y < h and 0 <= x < w and g[y][x] == bg: out[y][x] = s
    return out


def _spiral_seed(g, s, bg):
    obs = [ob for ob in objects(g, bg, True, True) if g[ob[0][0]][ob[0][1]] == s]
    if len(obs) != 1: return None
    ob = set(obs[0])
    if len(ob) == 1: return ('single', next(iter(ob)))
    ys = [y for y, _ in ob]; xs = [x for _, x in ob]
    if (min(ys) + max(ys)) % 2 or (min(xs) + max(xs)) % 2: return None
    cy, cx = (min(ys) + max(ys)) // 2, (min(xs) + max(xs)) // 2
    arms = []
    for d in ORTH:
        y, x = cy + d[0], cx + d[1]; last = None
        while (y, x) in ob: last = (y, x); y += d[0]; x += d[1]
        if last is None: return None
        arms.append(last + (d,))
    return ('plus', arms)


def fam_spiral(train):
    if not _same_shape(train) or not _only_bg_changes(train): return
    i0 = train[0]['input']; bg0 = bg_of(i0)
    for s in sorted({v for r in i0 for v in r} - {bg0}):
        kinds = [_spiral_seed(p['input'], s, bg_of(p['input'])) for p in train]
        if any(k is None for k in kinds) or len({k[0] for k in kinds}) != 1: continue
        kind = kinds[0][0]
        dirs = ORTH if kind == 'single' else (None,)
        for d0, rot, runin in product(dirs, ('cw', 'ccw'), (0, 1, 2)):
            def fn(g, s=s, d0=d0, rot=rot, runin=runin):
                bg = bg_of(g); sd = _spiral_seed(g, s, bg)
                if sd is None or sd[0] != kind: return None
                arms = [sd[1] + (d0,)] if kind == 'single' else sd[1]
                return _spiral(g, bg, s, arms, rot, runin)
            p0 = train[0]
            if fn(p0['input']) != p0['output']: continue
            yield (f'walker[seed={s},policy=spiral[{kind},{d0},{rot},runin={runin}]]', 5, fn)


# ---------------------------------------------------------------- portal (enter object, exit via marker)
def _marker_dir(ob, g):
    cnt = Counter(g[y][x] for y, x in ob)
    if len(cnt) != 2: return None
    body, mk = [c for c, _ in cnt.most_common()]
    bc = [(y, x) for y, x in ob if g[y][x] == body]; mc = [(y, x) for y, x in ob if g[y][x] == mk]
    obs, bs = set(ob), set(bc)
    ds = [d for d in ORTH if all((y + d[0], x + d[1]) not in obs and (y - d[0], x - d[1]) in bs for y, x in mc)]
    if len(ds) != 1: return None
    return body, mk, mc, ds[0]


def _portal(g, keep, sb=None):
    bg = bg_of(g); h, w = H(g), W(g)
    obs = objects(g, bg, False, False)
    info = []
    for ob in obs:
        m = _marker_dir(ob, g)
        if m is None: return None
        info.append(m)
    bodies = Counter(m[0] for m in info)
    seeds = [k for k, m in enumerate(info) if bodies[m[0]] == 1 and (sb is None or m[0] == sb)]
    if len(seeds) != 1 or len(info) < 2: return None
    sk = seeds[0]; paint = info[sk][0]
    owner = {}
    for k, ob in enumerate(obs):
        for c in ob: owner[c] = k
    out = [r[:] for r in g]
    if not keep:
        for ob in obs:
            for y, x in ob: out[y][x] = bg
    visited = {sk}; queue = [sk]
    while queue:
        k = queue.pop()
        for y, x in obs[k]: out[y][x] = paint
        _, _, mc, d = info[k]
        beam = list(mc)
        for _ in range(h + w):
            beam = [(y + d[0], x + d[1]) for y, x in beam]
            ins = [(y, x) for y, x in beam if 0 <= y < h and 0 <= x < w]
            if not ins: break
            hit = {owner[c] for c in ins if c in owner} - {k}
            if hit:
                for j in hit:
                    if j not in visited: visited.add(j); queue.append(j)
                break
            for y, x in ins: out[y][x] = paint
    return out


def _unique_bodies(g):
    bg = bg_of(g); info = [_marker_dir(ob, g) for ob in objects(g, bg, False, False)]
    if not info or any(m is None for m in info): return set()
    cnt = Counter(m[0] for m in info)
    return {c for c, n in cnt.items() if n == 1}


def fam_portal(train):
    if not _same_shape(train): return
    sbs = set.intersection(*[_unique_bodies(p['input']) for p in train])
    if len(sbs) != 1: return
    sb = sbs.pop()
    for keep in (False, True):
        fn = lambda g, keep=keep: _portal(g, keep, sb)
        if fn(train[0]['input']) != train[0]['output']: continue
        yield (f'walker[policy=portal,seed_body={sb},unreached={"keep" if keep else "erase"}]', 5, fn)


# ---------------------------------------------------------------- key-palette staircase walker
def _squares(g):
    """Solid k x k single-colour squares (k>=2) that are whole 4-connected components."""
    out = []
    for ob in objects(g, -1, False, True):
        ys = [y for y, _ in ob]; xs = [x for _, x in ob]
        hh, ww = max(ys) - min(ys) + 1, max(xs) - min(xs) + 1
        if hh == ww >= 2 and hh * ww == len(ob): out.append((min(ys), min(xs), hh, g[ob[0][0]][ob[0][1]], ob))
    return out


def _key(g):
    """Key = the largest row (>=2) of equal squares on the same rows, read left to right; terminal = the
    one other square of that size. Returns (palette, terminal colour, cells to erase) or None."""
    sq = _squares(g)
    rows = {}
    for q in sq: rows.setdefault((q[0], q[2]), []).append(q)
    best = max(rows.values(), key=len, default=[])
    if len(best) < 2: return None
    best.sort(key=lambda q: q[1]); k = best[0][2]
    gaps = {b[1] - a[1] for a, b in zip(best, best[1:])}
    if len(gaps) != 1: return None
    others = [q for q in sq if q[2] == k and q not in best]
    if len(others) > 1: return None
    pal = [q[3] for q in best]
    term = others[0][3] if others else None
    r0, c0 = best[0][0], best[0][1]; r1, c1 = r0 + k - 1, best[-1][1] + k - 1
    erase = [[(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)]] + ([others[0][4]] if others else [])
    return pal, term, erase


def _stair(g, first):
    h, w = H(g), W(g)
    kk = _key(g)
    if kk is None: return None
    pal, term, erase = kk
    out = [r[:] for r in g]
    for cells in erase:
        ring = Counter()
        cs = set(cells)
        for y, x in cells:
            for dy, dx in ORTH:
                n = (y + dy, x + dx)
                if n not in cs and 0 <= n[0] < h and 0 <= n[1] < w: ring[g[n[0]][n[1]]] += 1
        if not ring: return None
        fill = ring.most_common(1)[0][0]
        for y, x in cells: out[y][x] = fill
    base = [r[:] for r in out]
    seeds = []
    for y in range(h):
        for x in range(w):
            if base[y][x] != pal[0] or not (y in (0, h - 1) or x in (0, w - 1)): continue
            ds = [d for d, c in (((1, 0), y == 0), ((-1, 0), y == h - 1), ((0, 1), x == 0), ((0, -1), x == w - 1)) if c]
            if len(ds) != 1: continue
            seeds.append((y, x, ds[0]))
    if not seeds: return None
    for y, x, d in seeds:
        ny, nx = y + d[0], x + d[1]
        if not (0 <= ny < h and 0 <= nx < w): return None
        free = base[ny][nx]
        i = 0; t = first; steps = 0
        while True:
            steps += 1
            if steps > 4 * h * w: return None
            ny, nx = y + d[0], x + d[1]
            if not (0 <= ny < h and 0 <= nx < w): break
            if base[ny][nx] == free:
                y, x = ny, nx; i += 1; out[y][x] = pal[i % len(pal)]; continue
            if term is not None: out[ny][nx] = term
            d = cw(d) if t == 'cw' else ccw(d); t = 'ccw' if t == 'cw' else 'cw'
            ny, nx = y + d[0], x + d[1]
            if not (0 <= ny < h and 0 <= nx < w) or base[ny][nx] != free: break
    return out


def fam_stair(train):
    if not _same_shape(train): return
    if _key(train[0]['input']) is None: return
    for first in ('ccw', 'cw'):
        fn = lambda g, first=first: _stair(g, first)
        if fn(train[0]['input']) != train[0]['output']: continue
        yield (f'walker[policy=alternate({first}),paint=key_cycle,mark=terminal]', 6, fn)


FAMILIES = (fam_walker, fam_spiral, fam_portal, fam_stair)


In [ ]:
%%writefile /kaggle/working/probe/fam_recolour_by_colour_map.py
"""recolour.by_colour_map + recolour.by_key_position: recolour target cells from an in-grid key.

One primitive  out = paint(targets, K[corr(target)])  where the key K is read from the input grid and
indexed either by COLOUR (a colour map phi) or by POSITION (the key cell that corresponds to the target).

fam_key_position   (index by position; all parameters induced from the training pairs)
  T      : target colour role  rank0 | rank1 (by count among non-bg colours) | const c (same in every pair)
  frame  : drop the dominant non-target colour from the key (key sits inside a frame / behind lines) or not
  corr   : row | col           target cell takes the key colour found in its own row / column
           obj4 | obj8          each target component <-> one key cell, matched by normalised position
           rank-x | rank-y      target split into column (row) bands of identical profile, ordered,
                                matched to the key colours ordered the same way
           scaled               target bbox scaled onto the (rank-compressed) key matrix, per cell
           onoff                2-line key: line 0 colours target cells, line 1 colours the rest (per column/row)
  post   : keep | erase (key cells -> bg) | crop (to the target bbox; scaled/onoff)
fam_legend_map     (index by colour: phi read from legend pairs)
  source : dominoes (isolated 2-cell pairs) | rows / cols of an isolated 2-wide multicolour block
  dir    : fwd a->b | rev b->a | swap | chain (block rows applied one after another)
  scope  : whole grid outside the key | crop to the largest remaining object
fam_key_erase      phi = {k -> bg}, k = the single colour shown in a corner box cut off by a line.
fam_complement     binary complement (fg <-> bg) on scope whole | lattice blocks | object bbox,
                   the global colour-map post-step then names the new colour.
fam_variant_swap   repeated composite objects come in exactly two variants (up to integer scale);
                   every object is replaced by the other variant, aligned on their common part.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, crop

N4 = ((1, 0), (-1, 0), (0, 1), (0, -1))


def _same_shape(train):
    return all((H(p["input"]), W(p["input"])) == (H(p["output"]), W(p["output"])) for p in train)


def _comps(cells, diag):
    """connected components of a set of cells"""
    cells = set(cells); out = []
    nb = N4 + (((1, 1), (1, -1), (-1, 1), (-1, -1)) if diag else ())
    while cells:
        s = cells.pop(); st = [s]; comp = [s]
        while st:
            y, x = st.pop()
            for dy, dx in nb:
                p = (y + dy, x + dx)
                if p in cells:
                    cells.remove(p); st.append(p); comp.append(p)
        out.append(comp)
    return out


# ================================================================== key by position
def _ranked(g, bg):
    cnt = Counter(v for r in g for v in r if v != bg)
    return [c for c, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))]


def _target_truth(p):
    """input colour(s) of the target: colours of changed cells (same shape) or input colours absent in output"""
    i, o = p["input"], p["output"]
    if (H(i), W(i)) == (H(o), W(o)):
        return {i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]}
    ci = {v for r in i for v in r}; co = {v for r in o for v in r}
    return ci - co - {bg_of(i)}


def _t_roles(train):
    roles = []
    truths = [_target_truth(p) for p in train]
    for rk in (0, 1):
        ok = True
        for p, t in zip(train, truths):
            rs = _ranked(p["input"], bg_of(p["input"]))
            if len(rs) <= rk or rs[rk] not in t: ok = False; break
        if ok: roles.append(("rank%d" % rk, rk))
    common = set.intersection(*[set(t) for t in truths]) if truths else set()
    for c in sorted(common):
        roles.append(("const%d" % c, ("c", c)))
    return roles


def _target_colour(g, bg, role):
    if isinstance(role, tuple): return role[1]
    rs = _ranked(g, bg)
    return rs[role] if len(rs) > role else None


def _key(g, bg, T, frame):
    cells = {(y, x): g[y][x] for y in range(H(g)) for x in range(W(g)) if g[y][x] not in (bg, T)}
    if frame:
        cnt = Counter(cells.values())
        if len(cnt) < 2: return None
        F = cnt.most_common(1)[0][0]
        cells = {p: v for p, v in cells.items() if v != F}
    return cells or None


def _norm(pts):
    ys = [p[0] for p in pts]; xs = [p[1] for p in pts]
    y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)
    return [((y - y0) / (y1 - y0) if y1 > y0 else 0.0, (x - x0) / (x1 - x0) if x1 > x0 else 0.0) for y, x in pts]


def _match(a, b):
    """greedy min-distance bijection between two equal-size point lists -> list idx_a -> idx_b"""
    na, nb = _norm(a), _norm(b)
    pairs = sorted(((ya - yb) ** 2 + (xa - xb) ** 2, i, j) for i, (ya, xa) in enumerate(na) for j, (yb, xb) in enumerate(nb))
    ma, used = {}, set()
    for d, i, j in pairs:
        if i in ma or j in used: continue
        ma[i] = j; used.add(j)
    return ma


def _compress(cells):
    """rank-compress key cells into a matrix (None = hole)"""
    rows = sorted({y for y, _ in cells}); cols = sorted({x for _, x in cells})
    ri = {r: k for k, r in enumerate(rows)}; ci = {c: k for k, c in enumerate(cols)}
    M = [[None] * len(cols) for _ in rows]
    for (y, x), v in cells.items(): M[ri[y]][ci[x]] = v
    return M


def _bands(tc, axis):
    """group contiguous columns (axis=1) / rows (axis=0) holding target cells with identical profile"""
    prof = {}
    for y, x in tc:
        k, o = (x, y) if axis == 1 else (y, x)
        prof.setdefault(k, set()).add(o)
    bands = []
    for k in sorted(prof):
        if bands and bands[-1][-1] == k - 1 and prof[bands[-1][-1]] == prof[k]: bands[-1].append(k)
        else: bands.append([k])
    return bands, prof


def _apply_pos(g, role, frame, corr, post, diag=False, fill=None):
    bg = bg_of(g); T = _target_colour(g, bg, role)
    if T is None: return None
    if T == bg:
        cnt = Counter(v for r in g for v in r if v != T)
        if not cnt: return None
        bg = cnt.most_common(1)[0][0]
    tc = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == T]
    if not tc: return None
    K = _key(g, bg, T, frame)
    if not K: return None
    out = [r[:] for r in g]
    if corr in ("row", "col"):
        ax = 0 if corr == "row" else 1
        line = {}
        for p, v in K.items(): line.setdefault(p[ax], set()).add(v)
        for p in tc:
            s = line.get(p[ax])
            if not s or len(s) != 1: return None
            out[p[0]][p[1]] = next(iter(s))
    elif corr == "obj":
        comps = _comps(tc, diag)
        kp = sorted(K)
        if len(comps) != len(kp) or len(kp) < 2: return None
        cen = [(sum(y for y, _ in c) / len(c), sum(x for _, x in c) / len(c)) for c in comps]
        m = _match(cen, kp)
        for i, c in enumerate(comps):
            for y, x in c: out[y][x] = K[kp[m[i]]]
    elif corr in ("rank-x", "rank-y"):
        axis = 1 if corr == "rank-x" else 0
        bands, _ = _bands(tc, axis)
        first = {}
        for (y, x), v in K.items():
            k = x if axis == 1 else y
            first[v] = min(first.get(v, k), k)
        order = sorted(first, key=lambda v: first[v])
        if len(order) != len(bands) or len(order) < 2: return None
        for b, v in zip(bands, order):
            bs = set(b)
            for y, x in tc:
                if (x if axis == 1 else y) in bs: out[y][x] = v
    elif corr in ("scaled", "onoff"):
        r0, c0, r1, c1 = bbox(tc); h, w = r1 - r0 + 1, c1 - c0 + 1
        M = _compress(K)
        if any(v is None for r in M for v in r): return None
        if corr == "scaled":
            kh, kw = len(M), len(M[0])
            if (kh, kw) == (1, 1) or h % kh or w % kw: return None
            for y, x in tc: out[y][x] = M[(y - r0) * kh // h][(x - c0) * kw // w]
        else:
            rows = [r for k, r in enumerate(M) if k == 0 or r != M[k - 1]]
            if len(rows) == 2 and len(set(map(tuple, rows))) == 2:
                lines, vert = rows, False
            else:
                Mt = [list(c) for c in zip(*M)]
                cols = [r for k, r in enumerate(Mt) if k == 0 or r != Mt[k - 1]]
                if len(cols) != 2: return None
                lines, vert = cols, True
            n = len(lines[0]); span = h if vert else w
            if span % n: return None
            for y in range(r0, r1 + 1):
                for x in range(c0, c1 + 1):
                    k = ((y - r0) if vert else (x - c0)) * n // span
                    out[y][x] = lines[0 if g[y][x] == T else 1][k]
    else:
        return None
    if post == "erase":
        for y, x in K: out[y][x] = bg
    elif post == "crop":
        r0, c0, r1, c1 = bbox(tc)
        res = crop(out, (r0, c0, r1, c1))
        if corr == "scaled":
            f = bg if fill is None else fill
            res = [[v if g[r0 + y][c0 + x] == T else f for x, v in enumerate(r)] for y, r in enumerate(res)]
        return res
    return out


def fam_key_position(train):
    same = _same_shape(train)
    i0, o0 = train[0]["input"], train[0]["output"]
    if not same and (H(o0) > H(i0) or W(o0) > W(i0)): return
    roles = _t_roles(train)
    if not roles: return
    if same:
        combos = [(c, p) for c in ("row", "col", "obj4", "obj8", "rank-x", "rank-y") for p in ("keep", "erase")]
        combos += [("scaled", "keep"), ("onoff", "keep")]
    else:
        combos = [("scaled", "crop"), ("onoff", "crop")]
    fills = [None] + sorted(set.intersection(*[{v for r in p["output"] for v in r} & {v for r in p["input"] for v in r} for p in train]))[:2]
    if same: fills = [None]
    for rn, role in roles:
      for fill in fills:
        for frame in (False, True):
            for corr, post in combos:
                c = corr[:3] if corr.startswith("obj") else corr
                diag = corr == "obj8"
                if fill is not None and corr != "scaled": continue
                def fn(g, role=role, frame=frame, c=c, post=post, diag=diag, fill=fill):
                    return _apply_pos(g, role, frame, c, post, diag, fill)
                fs = "" if fill is None else f",fill={fill}"
                yield (f"key-position:{corr}[T={rn},frame={int(frame)},{post}{fs}]", 4, fn)


# ================================================================== key by colour (legend)
def _legend_sources(g, bg, how):
    """list of (pairs [(a,b)...], keycells set) for the chosen legend source"""
    res = []
    for ob in objects(g, bg, False, False):
        r0, c0, r1, c1 = bbox(ob); h, w = r1 - r0 + 1, c1 - c0 + 1
        if len(ob) != h * w: continue
        if how == "dominoes":
            if len(ob) != 2: continue
            (y1, x1), (y2, x2) = sorted(ob)
            prs = [(g[y1][x1], g[y2][x2])]
        elif how == "rows":
            if w != 2 or h > 10: continue
            prs = [(g[y][c0], g[y][c0 + 1]) for y in range(r0, r1 + 1)]
        else:
            if h != 2 or w > 10: continue
            prs = [(g[r0][x], g[r0 + 1][x]) for x in range(c0, c1 + 1)]
        if any(a == b for a, b in prs): continue
        res.append((prs, set(ob)))
    return res


def _phi(prs, d):
    m = {}
    for a, b in prs:
        if d == "rev": a, b = b, a
        if d == "swap":
            if m.get(b, a) != a: return None
            m[b] = a
        if m.get(a, b) != b: return None
        m[a] = b
    return m


def _apply_legend(g, how, d, scope):
    bg = bg_of(g)
    src = _legend_sources(g, bg, how)
    if not src: return None
    if how != "dominoes" and len(src) != 1: return None
    prs = [p for s, _ in src for p in s]; kc = set().union(*[k for _, k in src])
    if d == "chain":
        f = lambda v: _chain(v, prs)
    else:
        m = _phi(prs, d)
        if not m: return None
        f = lambda v: m.get(v, v)
    if scope == "all":
        return [[f(v) if (y, x) not in kc else v for x, v in enumerate(r)] for y, r in enumerate(g)]
    obs = [ob for ob in objects(g, bg, True, False) if not set(ob) & kc]
    if not obs: return None
    big = max(obs, key=len)
    if sum(1 for o in obs if len(o) == len(big)) > 1: return None
    return [[f(v) for v in r] for r in crop(g, bbox(big))]


def _chain(v, prs):
    for a, b in prs:
        if v == a: v = b
    return v


def fam_legend_map(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    same = _same_shape(train)
    scopes = ("all",) if same else ("crop",)
    if not same and (H(o0) >= H(i0) and W(o0) >= W(i0)): return
    for how in ("dominoes", "rows", "cols"):
        if not any(_legend_sources(p["input"], bg_of(p["input"]), how) for p in train[:1]): continue
        for d in ("fwd", "rev", "swap") + (("chain",) if how != "dominoes" else ()):
            for sc in scopes:
                def fn(g, how=how, d=d, sc=sc): return _apply_legend(g, how, d, sc)
                yield (f"legend-map:{how}[{d},{sc}]", 4, fn)


# ================================================================== erase the colour shown in a corner box
_D4 = [(lambda g: g, lambda g: g),
       (lambda g: [r[::-1] for r in g], lambda g: [r[::-1] for r in g]),
       (lambda g: g[::-1], lambda g: g[::-1]),
       (lambda g: [r[::-1] for r in g[::-1]], lambda g: [r[::-1] for r in g[::-1]])]


def _corner_box(g, bg):
    """top-left box: row r0 (cols 0..c0) and column c0 (rows 0..r0) all colour L; returns (r0,c0,L)"""
    for r0 in range(1, H(g) - 1):
        for c0 in range(1, W(g) - 1):
            L = g[r0][c0]
            if L == bg: continue
            if all(g[r0][x] == L for x in range(c0 + 1)) and all(g[y][c0] == L for y in range(r0 + 1)):
                inner = {g[y][x] for y in range(r0) for x in range(c0)} - {bg}
                if L not in inner: return r0, c0, L
    return None


def _apply_erase(g, how):
    bg = bg_of(g)
    for k, (f, fi) in enumerate(_D4):
        gg = f(g); cb = _corner_box(gg, bg)
        if not cb: continue
        r0, c0, L = cb
        inner = {gg[y][x] for y in range(r0) for x in range(c0)} - {bg}
        if len(inner) != 1: return None
        kcol = next(iter(inner))
        out = [[(bg if (v == kcol and not (y < r0 and x < c0)) else v) for x, v in enumerate(r)] for y, r in enumerate(gg)]
        return fi(out)
    return None


def fam_key_erase(train):
    if not _same_shape(train): return
    yield ("key-erase:corner-box", 4, lambda g: _apply_erase(g, "erase"))


# ================================================================== binary complement
def _lattice(g, bg):
    """smallest period p>=3 such that rows (cols) at one offset mod p are all background"""
    def per(lines):
        n = len(lines)
        for p in range(3, n):
            for off in range(p):
                if all(all(v == bg for v in lines[i]) for i in range(off, n, p)):
                    return p, off
        return None
    pr = per(g); pc = per([list(c) for c in zip(*g)])
    if not pr or not pc: return None
    def spans(n, p, off):
        cuts = list(range(off, n, p))
        edges = [-1] + cuts + [n]
        return [(a + 1, b - 1) for a, b in zip(edges, edges[1:]) if b - 1 >= a + 1]
    return [(a, b, c, d) for a, b in spans(H(g), *pr) for c, d in spans(W(g), *pc)]


def _complement(g, scope):
    bg = bg_of(g); out = [r[:] for r in g]
    if scope == "whole":
        regions = [(0, H(g) - 1, 0, W(g) - 1)]
    elif scope == "lattice":
        regions = _lattice(g, bg)
        if not regions: return None
    else:
        regions = [(b[0], b[2], b[1], b[3]) for b in (bbox(o) for o in objects(g, bg, True, False))]
    done = False
    for r0, r1, c0, c1 in regions:
        cs = {g[y][x] for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)} - {bg}
        if not cs: continue
        if len(cs) != 1: return None
        c = next(iter(cs)); done = True
        for y in range(r0, r1 + 1):
            for x in range(c0, c1 + 1):
                out[y][x] = bg if g[y][x] == c else c
    return out if done else None


def fam_complement(train):
    if not _same_shape(train): return
    for sc in ("whole", "lattice", "objbox"):
        yield (f"complement:{sc}", 3, lambda g, sc=sc: _complement(g, sc))


# ================================================================== two-variant swap
def _unit(g, ob, bg):
    """object -> (unit pattern dict (y,x)->colour, scale, top-left)"""
    r0, c0, r1, c1 = bbox(ob); h, w = r1 - r0 + 1, c1 - c0 + 1
    cells = {(y - r0, x - c0): g[y][x] for y, x in ob}
    for s in range(min(h, w), 0, -1):
        if h % s or w % s: continue
        ok = True
        for y in range(h):
            for x in range(w):
                if cells.get((y, x)) != cells.get((y - y % s, x - x % s)): ok = False; break
            if not ok: break
        if ok:
            return {(y // s, x // s): v for (y, x), v in cells.items() if y % s == 0 and x % s == 0}, s, (r0, c0)
    return None


def _align(A, B):
    """offset (dy,dx) to place B relative to A maximising shared equal-colour cells"""
    best = (0, None)
    for (ay, ax) in A:
        for (by, bx) in B:
            dy, dx = ay - by, ax - bx
            n = sum(1 for (y, x), v in B.items() if A.get((y + dy, x + dx)) == v)
            if n > best[0]: best = (n, (dy, dx))
    return best


def _variant_swap(g, diag):
    bg = bg_of(g)
    obs = objects(g, bg, diag, False)
    if len(obs) < 2 or len(obs) > 30: return None
    units = []
    for ob in obs:
        u = _unit(g, ob, bg)
        if u is None: return None
        units.append(u)
    keys = []
    for u, _, _ in units:
        k = frozenset(u.items())
        if k not in keys: keys.append(k)
    if len(keys) != 2: return None
    A, B = dict(keys[0]), dict(keys[1])
    n, off = _align(A, B)
    if not off or n < 2 or n >= min(len(A), len(B)): return None
    out = [r[:] for r in g]
    for ob in obs:
        for y, x in ob: out[y][x] = bg
    for u, s, (r0, c0) in units:
        if frozenset(u.items()) == keys[0]: new, (dy, dx) = B, off
        else: new, (dy, dx) = A, (-off[0], -off[1])
        for (y, x), v in new.items():
            for a in range(s):
                for b in range(s):
                    yy, xx = r0 + (y + dy) * s + a, c0 + (x + dx) * s + b
                    if not (0 <= yy < H(g) and 0 <= xx < W(g)): return None
                    out[yy][xx] = v
    return out


def fam_variant_swap(train):
    if not _same_shape(train): return
    for diag in (True, False):
        yield (f"variant-swap[{8 if diag else 4}]", 5, lambda g, diag=diag: _variant_swap(g, diag))


FAMILIES = (fam_key_position, fam_legend_map, fam_key_erase, fam_complement, fam_variant_swap)


In [ ]:
%%writefile /kaggle/working/probe/fam_recolour_by_distance_layers.py
"""recolour.by_distance_layers: cell colour = palette(distance of the cell from a source set, inside a mask).

One parametrised primitive, every parameter induced from the training pairs:
  mask/source  : objects<-exterior | enclosed<-exterior | enclosed<-seeds | open<-seeds | seedregion<-seeds | seedregion<-exterior
  metric       : 4-conn (manhattan/geodesic) | 8-conn (chebyshev/geodesic) BFS inside the mask
  palette      : cyc[k]  out = P[(d-1) mod k]      (d>=1; sources keep)
                 clamp[k] out = P[min(d-1,k-1)]
                 succ     out = S(out of BFS parent) (successor map learned from training)
                 P entries are roles fitted per residue: keep | seed (nearest source colour) | other (the other seed colour) | const c
                 tie cells (equidistant from different-coloured sources) get their own fitted role.
  plus a sibling mode: reverse the nested colour layers of each object (palette = reversed input layers).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter, deque
from gdsl import H, W, bg_of

N4 = ((1, 0), (-1, 0), (0, 1), (0, -1))
N8 = N4 + ((1, 1), (1, -1), (-1, 1), (-1, -1))


def _comp_from_border(g, ok):
    """cells with ok(y,x) reachable 4-conn from the grid border through ok cells"""
    h, w = H(g), W(g); seen = set(); dq = deque()
    for y in range(h):
        for x in range(w):
            if (y in (0, h - 1) or x in (0, w - 1)) and ok(y, x):
                seen.add((y, x)); dq.append((y, x))
    while dq:
        y, x = dq.popleft()
        for dy, dx in N4:
            p = (y + dy, x + dx)
            if 0 <= p[0] < h and 0 <= p[1] < w and p not in seen and ok(*p):
                seen.add(p); dq.append(p)
    return seen


def _setup(g, kind, pc):
    """return (mask set, sources dict cell->colour or 'EXT', open_colour) or None"""
    h, w = H(g), W(g)
    cnt = Counter(v for r in g for v in r)
    allc = {(y, x) for y in range(h) for x in range(w)}
    if kind in ('objects', 'enclosed_ext', 'enclosed_seed', 'open_seed'):
        bg = pc[1] if isinstance(pc, tuple) else bg_of(g)
        if kind in ('objects', 'objects_ie'):
            mask = {p for p in allc if g[p[0]][p[1]] != bg}
            return mask, ('EXT' if kind == 'objects' else 'EXTIN'), bg
        if kind == 'open_seed':
            src = {p: g[p[0]][p[1]] for p in allc if g[p[0]][p[1]] != bg}
            if not src or len(src) > 12: return None
            return allc, src, bg
        out = _comp_from_border(g, lambda y, x: g[y][x] == bg)
        frame = {p for p in allc - out if any((p[0] + dy, p[1] + dx) in out for dy, dx in N8)}
        enc = allc - out - frame
        if not enc: return None
        if kind == 'enclosed_ext':
            return {p for p in enc if g[p[0]][p[1]] == bg}, 'EXT', bg
        src = {p: g[p[0]][p[1]] for p in enc if g[p[0]][p[1]] != bg}
        if not src: return None
        return enc, src, bg
    # seed regions: open colour & wall colour are the two most frequent colours (order = pc)
    mc = [c for c, _ in cnt.most_common()]
    if len(mc) < 3: return None
    if isinstance(pc, tuple):
        op = pc[1]
        if op not in cnt: return None
        wall = [c for c in mc if c != op][0]
    else:
        op, wall = (mc[0], mc[1]) if pc == 0 else (mc[1], mc[0])
    seeds = {p: g[p[0]][p[1]] for p in allc if g[p[0]][p[1]] not in (op, wall)}
    if not seeds: return None
    reg = set(seeds); dq = deque(seeds)
    while dq:
        y, x = dq.popleft()
        for dy, dx in N4:
            p = (y + dy, x + dx)
            if p in allc and p not in reg and g[p[0]][p[1]] == op:
                reg.add(p); dq.append(p)
    if kind == 'region_seed':
        return reg, seeds, op
    # region_ext: fill holes of the region, sources = its exterior
    outside = _comp_from_border(g, lambda y, x: (y, x) not in reg)
    filled = allc - outside
    return filled, 'EXT', op


def _dist(g, mask, src, conn):
    """multi-source BFS inside mask. returns {cell: (d, colourset)}"""
    h, w = H(g), W(g); nb = N8 if conn == 8 else N4
    D = {}; dq = deque()
    if src in ('EXT', 'EXTIN'):
        for (y, x) in mask:
            if any(not ((y + dy, x + dx) in mask) and (src == 'EXT' or (0 <= y + dy < h and 0 <= x + dx < w)) for dy, dx in nb):
                D[(y, x)] = [1, set()]; dq.append((y, x))
    else:
        for p, c in src.items():
            if p in mask:
                D[p] = [0, {c}]; dq.append(p)
    while dq:
        p = dq.popleft(); d, cs = D[p]
        for dy, dx in nb:
            q = (p[0] + dy, p[1] + dx)
            if q not in mask or (isinstance(src, dict) and q in src): continue
            if q not in D:
                D[q] = [d + 1, set(cs)]; dq.append(q)
            elif D[q][0] == d + 1:
                D[q][1] |= cs
    return D


def _feats(g, kind, pc, conn, vor=None):
    s = _setup(g, kind, pc)
    if s is None: return None
    mask, src, op = s
    if not mask or len(mask) == 0: return None
    D = _dist(g, mask, src, conn)
    if kind == 'region_ext':  # seed cells are anchors, never repainted
        for q, v in D.items():
            if g[q[0]][q[1]] != op and g[q[0]][q[1]] != _wall(g, op): v[0] = 0
    if vor and isinstance(src, dict):
        S = list(src.items())
        for q, v in D.items():
            if v[0] == 0: continue
            ds = [abs(q[0] - p[0]) + abs(q[1] - p[1]) for p, c in S]
            m = min(ds)
            v[1] = {c for (p, c), d in zip(S, ds) if d == m}
    scols = sorted(set(src.values())) if isinstance(src, dict) else []
    rows = []
    for p, (d, cs) in D.items():
        if d == 0: continue
        seed = next(iter(cs)) if len(cs) == 1 else None
        other = None
        if len(scols) == 2 and seed is not None:
            other = scols[1] if seed == scols[0] else scols[0]
        rows.append((p, d, g[p[0]][p[1]], seed, other, len(cs) > 1))
    return rows, D, mask, op


def _role_val(role, inc, seed, other):
    if role == 'keep': return inc
    if role == 'seed': return seed
    if role == 'other': return other
    return role[1]


def _fit_role(samples):
    """samples: list of (inc, seed, other, out). pick first consistent role"""
    if not samples: return None
    for role in ('keep', 'seed', 'other'):
        if all(_role_val(role, a, b, c) == o and o is not None for a, b, c, o in samples):
            return role
    outs = {s[3] for s in samples}
    if len(outs) == 1: return ('c', outs.pop())
    return None


def _wall(g, op):
    cnt = Counter(v for r in g for v in r); cnt.pop(op, None)
    return cnt.most_common(1)[0][0] if cnt else None


def _clear_outside(g, mask, op):
    """objects (8-conn, non-open colour) outside the mask that do not touch it are erased"""
    out = [r[:] for r in g]; h, w = H(g), W(g); seen = set()
    for y in range(h):
        for x in range(w):
            if (y, x) in mask or (y, x) in seen or g[y][x] == op: continue
            comp = [(y, x)]; seen.add((y, x)); st = [(y, x)]; touch = False
            while st:
                a, b = st.pop()
                for dy, dx in N8:
                    q = (a + dy, b + dx)
                    if q in mask: touch = True
                    elif 0 <= q[0] < h and 0 <= q[1] < w and q not in seen and g[q[0]][q[1]] != op:
                        seen.add(q); comp.append(q); st.append(q)
            if not touch:
                for a, b in comp: out[a][b] = op
    return out


BASE_KINDS = (('objects', 0, None), ('enclosed_ext', 0, None), ('enclosed_seed', 0, None), ('enclosed_seed', 0, 'L1'),
              ('open_seed', 0, None), ('open_seed', 0, 'L1'))


def _kinds(train):
    bgs = {bg_of(p['input']) for p in train}
    bgp = ('c', next(iter(bgs))) if len(bgs) == 1 else 0  # training-constant background colour, else per-grid mode
    for kind, pc, vor in BASE_KINDS:
        yield (kind, bgp, vor)
    common = None
    for p in train:
        cs = {v for r in p['input'] for v in r}
        common = cs if common is None else common & cs
    ops = [0, 1] + [('c', c) for c in sorted(common or ())]
    for kind in ('region_seed', 'region_ext'):
        for pc in ops:
            yield (kind, pc, None)


def fam_distance_layers(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    nyield = 0
    seen = set()
    for kind, pc, vor in _kinds(train):
        for conn in (4, 8):
            F = []
            for p in train:
                f = _feats(p['input'], kind, pc, conn, vor)
                if f is None: F = None; break
                F.append(f)
            if F is None: continue
            # cells outside mask must be unchanged (or cleared for region_ext)
            clear_opts = (False, True) if kind == 'region_ext' else (False,)
            for clear in clear_opts:
                good = True
                for p, (rows, D, mask, op) in zip(train, F):
                    g, o = p['input'], p['output']
                    base = _clear_outside(g, mask, op) if clear else g
                    for y in range(H(g)):
                        for x in range(W(g)):
                            if (y, x) not in D or D[(y, x)][0] == 0:
                                if (y, x) in D or (y, x) not in mask:
                                    if base[y][x] != o[y][x]: good = False; break
                        if not good: break
                    if not good: break
                if not good: continue
                samples = []  # (d, tie, inc, seed, other, out)
                for p, (rows, D, mask, op) in zip(train, F):
                    o = p['output']
                    for (q, d, inc, seed, other, tie) in rows:
                        samples.append((d, tie, inc, seed, other, o[q[0]][q[1]]))
                if not samples: continue
                if all(s[2] == s[5] for s in samples): continue  # nothing recoloured
                maxd = max(s[0] for s in samples)
                ties = [s[2:] for s in samples if s[1]]
                tie_role = _fit_role(ties) if ties else None
                if ties and tie_role is None: tie_role = False
                base_s = [s for s in samples if not (s[1] and tie_role)]
                # palette modes
                for mode in ('cyc', 'clamp'):
                    for k in range(1, 5):
                        if mode == 'clamp' and k < 2: continue
                        if k > maxd: break
                        idx = (lambda d, k=k: (d - 1) % k) if mode == 'cyc' else (lambda d, k=k: min(d - 1, k - 1))
                        P = []
                        for j in range(k):
                            r = _fit_role([s[2:] for s in base_s if idx(s[0]) == j])
                            if r is None: P = None; break
                            P.append(r)
                        if P is None: continue
                        if tie_role is False: continue
                        name = f"distlayers[{kind}{pc}{vor or ''},c{conn},{mode}{k}:{P},tie={tie_role},clear={clear}]"
                        yield (name, 3 + min(k, 3), _make_fn(kind, pc, conn, vor, idx, P, tie_role, clear))
                        nyield += 1
                        break  # smallest k per mode
                # successor mode
                S = _fit_succ(train, F, conn) if vor is None else None
                if S is not None:
                    yield (f"distlayers[{kind}{pc},c{conn},succ,clear={clear}]", 5,
                           _make_succ(kind, pc, conn, S, clear))
                    nyield += 1
                if nyield > 40: return


def _make_fn(kind, pc, conn, vor, idx, P, tie_role, clear):
    def fn(g):
        f = _feats(g, kind, pc, conn, vor)
        if f is None: return None
        rows, D, mask, op = f
        out = _clear_outside(g, mask, op) if clear else [r[:] for r in g]
        for (q, d, inc, seed, other, tie) in rows:
            role = tie_role if (tie and tie_role) else P[idx(d)]
            v = _role_val(role, inc, seed, other)
            if v is None: return None
            out[q[0]][q[1]] = v
        return out
    return fn


def _parents(g, D, conn):
    nb = N8 if conn == 8 else N4
    for q, (d, cs) in D.items():
        if d == 0: continue
        par = [(q[0] + dy, q[1] + dx) for dy, dx in nb if (q[0] + dy, q[1] + dx) in D and D[(q[0] + dy, q[1] + dx)][0] == d - 1]
        yield q, d, par


def _fit_succ(train, F, conn):
    """out(cell) = S[out(parent)] with parents at d-1; sources keep their colour."""
    S = {}
    for p, (rows, D, mask, op) in zip(train, F):
        if not any(v[0] == 0 for v in D.values()): return None
        o = p['output']
        for q, d, par in _parents(p['input'], D, conn):
            v = o[q[0]][q[1]]
            for r in par:
                a = o[r[0]][r[1]]
                if S.get(a, v) != v: return None
                S[a] = v
    if not S or all(a == b for a, b in S.items()): return None
    return S


def _make_succ(kind, pc, conn, S, clear):
    def fn(g):
        f = _feats(g, kind, pc, conn)
        if f is None: return None
        rows, D, mask, op = f
        out = _clear_outside(g, mask, op) if clear else [r[:] for r in g]
        order = sorted((d, q) for q, (d, cs) in D.items())
        nb = N8 if conn == 8 else N4
        for d, q in order:
            if d == 0: continue
            vals = {S.get(out[q[0] + dy][q[1] + dx]) for dy, dx in nb
                    if (q[0] + dy, q[1] + dx) in D and D[(q[0] + dy, q[1] + dx)][0] == d - 1}
            if len(vals) != 1 or None in vals: return None
            out[q[0]][q[1]] = vals.pop()
        return out
    return fn


def _layers(g, bg):
    """per object (4-conn non-bg component; whole grid if bg None): nested colour layers from outside in"""
    h, w = H(g), W(g)
    cells = {(y, x) for y in range(h) for x in range(w) if bg is None or g[y][x] != bg}
    out = [r[:] for r in g]; seen = set(); changed = False
    for s in sorted(cells):
        if s in seen: continue
        comp = {s}; st = [s]
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                q = (y + dy, x + dx)
                if q in cells and q not in comp: comp.add(q); st.append(q)
        seen |= comp
        L = {}; dq = deque()
        for (y, x) in comp:
            if any((y + dy, x + dx) not in comp for dy, dx in N4):
                L[(y, x)] = 0; dq.append((y, x))
        cols0 = {g[y][x] for (y, x) in L}
        if len(cols0) != 1: return None
        while dq:
            p = dq.popleft()
            for dy, dx in N4:
                q = (p[0] + dy, p[1] + dx)
                if q not in comp: continue
                nd = L[p] + (g[q[0]][q[1]] != g[p[0]][p[1]])
                if q not in L or nd < L[q]:
                    L[q] = nd
                    if nd == L[p]: dq.appendleft(q)
                    else: dq.append(q)
        lc = {}
        for q, l in L.items():
            if lc.setdefault(l, g[q[0]][q[1]]) != g[q[0]][q[1]]: return None
        n = max(lc) + 1
        if n < 2: continue
        for q, l in L.items():
            out[q[0]][q[1]] = lc[n - 1 - l]
        changed = True
    return out if changed else None


def fam_reverse_layers(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    for mode in ('bg', 'none'):
        fn = (lambda g: _layers(g, bg_of(g))) if mode == 'bg' else (lambda g: _layers(g, None))
        r = fn(i0)
        if r is None: continue
        yield (f"distlayers:reverse[{mode}]", 4, fn)


FAMILIES = (fam_distance_layers, fam_reverse_layers)


In [ ]:
%%writefile /kaggle/working/probe/fam_recolour_objects.py
"""Recolour whole objects by an induced object feature / by shape match with exemplars.

Groups: recolour.object_by_property, recolour.by_shape_match.

fam_recolour_object_feature: objects (components 4/8, rows, cols, separator panels, uniform tiles) are
  mapped to an output behaviour (keep / literal colour / own colour / background, painted on the object
  cells or filled over the object's region) by a table over one computed feature, induced from all pairs.
fam_recolour_shape_match: candidate objects of the source colour take the colour of the exemplar object(s)
  with an equivalent shape (translation / dihedral / dihedral after collapsing repeated rows+cols);
  exemplar pool = all other objects / target-coloured objects / frame-enclosed objects; unique or majority.
  Variant: candidates are the holes of the largest object and the output is that object's crop.
fam_recolour_solid_blocks: cells covered by a monochrome TxT square (T induced) take one induced colour.
"""
import sys
from collections import Counter
from itertools import product
sys.path.insert(0, '/home/claude/work/widen')
from gdsl import H, W, bg_of, bbox

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def comps(g, pred, diag, same_colour=True):
    """Connected components of cells satisfying pred (optionally split by colour)."""
    h, w = H(g), W(g); seen = set(); out = []
    nb = N8 if diag else N4
    for r in range(h):
        for c in range(w):
            if (r, c) in seen or not pred(g[r][c]): continue
            col = g[r][c]; st = [(r, c)]; seen.add((r, c)); cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and pred(g[yy][xx]) \
                            and (not same_colour or g[yy][xx] == col):
                        seen.add((yy, xx)); st.append((yy, xx))
            out.append(sorted(cells))
    return out


def norm(cells):
    y0 = min(y for y, _ in cells); x0 = min(x for _, x in cells)
    return tuple(sorted((y - y0, x - x0) for y, x in cells))


def d8(cells):
    res = []
    for t in range(8):
        cs = []
        for y, x in cells:
            if t & 1: y, x = x, y
            if t & 2: y = -y
            if t & 4: x = -x
            cs.append((y, x))
        res.append(norm(cs))
    return min(res)


def collapse(cells):
    """Remove repeated adjacent rows and columns of the object's mask."""
    s = set(norm(cells)); h = max(y for y, _ in s) + 1; w = max(x for _, x in s) + 1
    m = [[(y, x) in s for x in range(w)] for y in range(h)]
    m = [r for i, r in enumerate(m) if i == 0 or r != m[i - 1]]
    t = [list(c) for c in zip(*m)]
    t = [c for i, c in enumerate(t) if i == 0 or c != t[i - 1]]
    return [(y, x) for x, c in enumerate(t) for y, v in enumerate(c) if v]


CANON = {"T": norm, "D8": d8, "C8": lambda c: d8(collapse(c))}


# ---------------------------------------------------------------- object features
def full_sides(cells):
    r0, c0, r1, c1 = bbox(cells); s = set(cells)
    return (all((r0, x) in s for x in range(c0, c1 + 1)) + all((r1, x) in s for x in range(c0, c1 + 1))
            + all((y, c0) in s for y in range(r0, r1 + 1)) + all((y, c1) in s for y in range(r0, r1 + 1)))


def n_parts4(cells):
    s = set(cells); seen = set(); n = 0
    for c in cells:
        if c in seen: continue
        n += 1; st = [c]; seen.add(c)
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                q = (y + dy, x + dx)
                if q in s and q not in seen: seen.add(q); st.append(q)
    return min(n, 3)


def path_turns(cells):
    """Topology of a 4-connected thin object: 'branch'/'cycle', or (n_turns, all turns same sense)."""
    s = set(cells)
    deg = {c: sum((c[0] + dy, c[1] + dx) in s for dy, dx in N4) for c in cells}
    if len(cells) == 1: return (0, True)
    ends = [c for c in cells if deg[c] == 1]
    if any(d > 2 for d in deg.values()): return "branch"
    if len(ends) != 2: return "cycle"
    path = [ends[0]]; prev = None
    while True:
        cur = path[-1]
        nx = [(cur[0] + dy, cur[1] + dx) for dy, dx in N4 if (cur[0] + dy, cur[1] + dx) in s and (cur[0] + dy, cur[1] + dx) != prev]
        if not nx: break
        prev = cur; path.append(nx[0])
    if len(path) != len(cells): return "branch"
    turns = []
    for a, b, c in zip(path, path[1:], path[2:]):
        d1 = (b[0] - a[0], b[1] - a[1]); d2 = (c[0] - b[0], c[1] - b[1])
        cr = d1[0] * d2[1] - d1[1] * d2[0]
        if cr: turns.append(cr > 0)
    return (len(turns), len(set(turns)) <= 1)


def rank_vals(vals, key):
    """Per-object label max/min/mid of a numeric value (ties allowed only at non-extremes)."""
    mx, mn = max(vals), min(vals)
    if mx == mn: return ["all"] * len(vals)
    return ["max" if v == mx else "min" if v == mn else "mid" for v in vals]


def feat_values(fname, objs, g):
    """objs: list of (fg_cells, region_cells, origin). Returns one hashable per object or None."""
    fg = [o[0] for o in objs]
    if fname == "size": return [len(c) for c in fg]
    if fname == "size_ext": return rank_vals([len(c) for c in fg], None)
    if fname == "size_rank":
        s = sorted({len(c) for c in fg}, reverse=True); return [s.index(len(c)) for c in fg]
    if fname == "shape": return [norm(c) if c else () for c in fg]
    if fname == "shape_d8": return [d8(c) if c else () for c in fg]
    if fname == "pattern":
        return [tuple(sorted(((y - o[2][0], x - o[2][1]), g[y][x]) for y, x in c)) for c, o in zip(fg, objs)]
    if fname == "full_sides": return [full_sides(c) if c else -1 for c in fg]
    if fname == "turns": return [path_turns(c) if c else None for c in fg]
    if fname == "parts4": return [n_parts4(c) if c else 0 for c in fg]
    if fname == "marker_pos":
        out = []
        for c, o in zip(fg, objs):
            if len(c) != 1: out.append(None); continue
            y, x = c[0]; out.append((y - o[2][0], x - o[2][1]))
        return out
    if fname == "colour_count_ext":
        return rank_vals([len(c) for c in fg], None)
    raise KeyError(fname)


NUMERIC = {"size", "full_sides", "parts4"}


def segment(g, mode, bg):
    """Returns (objs, sep) where objs = [(fg_cells, region_cells, origin)], or None."""
    h, w = H(g), W(g)
    if mode in ("c4", "c8", "m4", "m8"):
        cs = comps(g, lambda v: v != bg, mode[1] == "8", mode[0] == "c")
        return [(c, c, bbox(c)[:2]) for c in cs]
    if mode == "rows":
        return [([(y, x) for x in range(w) if g[y][x] != bg], [(y, x) for x in range(w)], (y, 0)) for y in range(h)]
    if mode == "cols":
        return [([(y, x) for y in range(h) if g[y][x] != bg], [(y, x) for y in range(h)], (0, x)) for x in range(w)]
    if mode == "panels":
        pr = panel_rects(g)
        if not pr: return None
        rects, sc = pr
        return [([(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if g[y][x] not in (bg, sc)],
                 [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)], (r0, c0)) for r0, c0, r1, c1 in rects]
    if mode.startswith("tile"):
        th, tw = map(int, mode[4:].split("x"))
        if h % th or w % tw or (h // th) * (w // tw) < 2: return None
        return [([(y, x) for y in range(r, r + th) for x in range(c, c + tw) if g[y][x] != bg],
                 [(y, x) for y in range(r, r + th) for x in range(c, c + tw)], (r, c))
                for r in range(0, h, th) for c in range(0, w, tw)]
    return None


def panel_rects(g):
    h, w = H(g), W(g)
    for sc in sorted({v for r in g for v in r}):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols or len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]; rects = []
        for a in range(len(rb) - 1):
            for b in range(len(cb) - 1):
                if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1:
                    rects.append((rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1))
        if len(rects) >= 2: return rects, sc
    return None


def outcome_roles(g, o, ob, paint, bg):
    """Candidate roles explaining the output over one object; [] if not explainable."""
    fg, reg, _ = ob
    cells = fg if paint else reg
    if not cells: return [("keep",)]
    vs = {o[y][x] for y, x in cells}
    roles = [("keep",)] if all(o[y][x] == g[y][x] for y, x in cells) else []
    if len(vs) != 1: return roles
    X = vs.pop(); roles.append(("lit", X))
    own = Counter(g[y][x] for y, x in fg).most_common(1)[0][0] if fg else None
    if X == own: roles.append(("own",))
    if X == bg: roles.append(("bg",))
    return roles


def induce_table(train, mode, fname, paint):
    obs_by_val = {}
    for p in train:
        g, o = p["input"], p["output"]; bg = bg_of(g)
        objs = segment(g, mode, bg)
        if not objs: return None
        vals = feat_values(fname, objs, g)
        for ob, v in zip(objs, vals):
            rs = outcome_roles(g, o, ob, paint, bg)
            if not rs: return None
            obs_by_val.setdefault(v, []).append(rs)
    table = {}
    for v, lst in obs_by_val.items():
        common = set(lst[0])
        for rs in lst[1:]: common &= set(rs)
        if not common: return None
        for pref in (("keep",), None, ("own",), ("bg",)):
            if pref is None:
                lits = [r for r in common if r[0] == "lit"]
                if lits: table[v] = lits[0]; break
            elif pref in common: table[v] = pref; break
    if all(r == ("keep",) for r in table.values()): return None
    return table


def apply_table(g, mode, fname, paint, table, nearest):
    bg = bg_of(g); objs = segment(g, mode, bg)
    if not objs: return None
    vals = feat_values(fname, objs, g); out = [r[:] for r in g]
    keys = [k for k in table if isinstance(k, int)]
    for ob, v in zip(objs, vals):
        if v not in table:
            if not nearest or not isinstance(v, int) or not keys: return None
            d = sorted(keys, key=lambda k: abs(k - v))
            if len(d) > 1 and abs(d[0] - v) == abs(d[1] - v) and table[d[0]] != table[d[1]]: return None
            v = d[0]
        r = table[v]
        if r == ("keep",): continue
        fg, reg, _ = ob
        if r[0] == "lit": c = r[1]
        elif r == ("bg",): c = bg
        else:
            if not fg: return None
            c = Counter(g[y][x] for y, x in fg).most_common(1)[0][0]
        for y, x in (fg if paint else reg): out[y][x] = c
    return out


def uniform_tile(train):
    """Smallest tile (th,tw), >=2 tiles, such that every training output is constant on each tile."""
    o0 = train[0]["output"]; h, w = H(o0), W(o0); res = []
    for th in range(1, h + 1):
        for tw in range(1, w + 1):
            if h % th or w % tw or th * tw < 2 or (h // th) * (w // tw) < 2: continue
            if all(H(p["output"]) % th == 0 and W(p["output"]) % tw == 0 and
                   all(len({p["output"][y][x] for y in range(r, r + th) for x in range(c, c + tw)}) == 1
                       for r in range(0, H(p["output"]), th) for c in range(0, W(p["output"]), tw)) for p in train):
                res.append((th * tw, th, tw))
    return sorted(res, reverse=True)[:3]


def fam_recolour_object_feature(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(i), W(i)) != (H(o), W(o)) or i == o: return
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    modes = [("c4", True), ("c8", True), ("m4", True), ("m8", True), ("rows", False), ("cols", False), ("panels", False)]
    for _, th, tw in uniform_tile(train): modes.append((f"tile{th}x{tw}", False))
    feats = ("size", "size_ext", "size_rank", "shape", "shape_d8", "pattern", "full_sides", "parts4", "turns", "marker_pos")
    n = 0
    for (mode, paint), fname in product(modes, feats):
        if mode in ("c4", "c8") and fname in ("size", "size_rank", "shape"): continue   # covered by gdsl
        t = induce_table(train, mode, fname, paint)
        if not t: continue
        for nearest in ((False, True) if fname in NUMERIC else (False,)):
            def fn(g, mode=mode, fname=fname, paint=paint, t=t, nearest=nearest):
                return apply_table(g, mode, fname, paint, t, nearest)
            yield (f"recolour-objects:{mode}[{fname}{',nearest' if nearest else ''}]", 4 + nearest, fn)
            n += 1
        if n >= 40: return


# ---------------------------------------------------------------- shape match
def change_colours(train):
    src, tgt = set(), set()
    for p in train:
        g, o = p["input"], p["output"]
        for y in range(H(g)):
            for x in range(W(g)):
                if g[y][x] != o[y][x]: src.add(g[y][x]); tgt.add(o[y][x])
    return src, tgt


def enclosed(g, cells, bg):
    """True if some colour F (not the object's, not bg) walls the object off from the grid border."""
    oc = g[cells[0][0]][cells[0][1]]
    for F in {v for r in g for v in r} - {oc, bg}:
        seen = set(cells); st = list(cells); ok = True
        while st and ok:
            y, x = st.pop()
            for dy, dx in N4:
                yy, xx = y + dy, x + dx
                if not (0 <= yy < H(g) and 0 <= xx < W(g)): ok = False; break
                if (yy, xx) not in seen and g[yy][xx] != F: seen.add((yy, xx)); st.append((yy, xx))
        if ok: return True
    return False


def vote(key, exemplars, resolve):
    cs = [c for k, c in exemplars if k == key]
    if not cs: return None
    cnt = Counter(cs).most_common()
    if resolve == "unique": return cnt[0][0] if len(cnt) == 1 else "amb"
    if len(cnt) > 1 and cnt[0][1] == cnt[1][1]: return "amb"
    return cnt[0][0]


def shape_match(g, s, diag, pool, equiv, resolve, tgt):
    cnt = Counter(v for r in g for v in r)
    bg = max((c for c in cnt if c != s), key=lambda c: cnt[c], default=None)
    can = CANON[equiv]
    cands = comps(g, lambda v: v == s, diag)
    if not cands: return None
    ex = []
    for e in comps(g, lambda v: v not in (s, bg), diag):
        c = g[e[0][0]][e[0][1]]
        if pool == "targets" and c not in tgt: continue
        if pool == "enclosed" and not enclosed(g, e, bg): continue
        ex.append((can(e), c))
    if not ex: return None
    out = [r[:] for r in g]
    for cd in cands:
        c = vote(can(cd), ex, resolve)
        if c == "amb": return None
        if c is not None:
            for y, x in cd: out[y][x] = c
    return out


def holes_crop(g, diag, equiv):
    """Largest single-colour object; its holes (other-coloured components inside its bbox) take the
    majority colour of equal-shape objects outside the bbox; return the crop."""
    cnt = Counter(v for r in g for v in r); bg = cnt.most_common(1)[0][0]
    objs = comps(g, lambda v: True, False)
    big = max(objs, key=len); cc = g[big[0][0]][big[0][1]]
    if cc == bg and len(objs) > 1:
        big = max((o for o in objs if g[o[0][0]][o[0][1]] != bg), key=len); cc = g[big[0][0]][big[0][1]]
    r0, c0, r1, c1 = bbox(big)
    if r1 - r0 < 2 or c1 - c0 < 2: return None
    sub = [row[c0:c1 + 1] for row in g[r0:r1 + 1]]
    can = CANON[equiv]
    ex = [(can(e), g[e[0][0]][e[0][1]]) for e in comps(g, lambda v: v not in (bg, cc), diag)
          if all(not (r0 <= y <= r1 and c0 <= x <= c1) for y, x in e)]
    if not ex: return None
    out = [r[:] for r in sub]; hit = False
    for hcells in comps(sub, lambda v: v != cc, False):
        c = vote(can(hcells), ex, "majority")
        if c == "amb" or c is None: return None
        hit = True
        for y, x in hcells: out[y][x] = c
    return out if hit else None


def fam_recolour_shape_match(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(i), W(i)) != (H(o), W(o)):
        if H(o) < H(i) and W(o) < W(i):
            for diag, equiv in product((False, True), ("D8", "T")):
                yield (f"shape-vote-holes-crop[{'8' if diag else '4'},{equiv}]", 5,
                       lambda g, diag=diag, equiv=equiv: holes_crop(g, diag, equiv))
        return
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    src, tgt = change_colours(train)
    if len(src) != 1 or not tgt: return
    s = src.pop()
    # changed cells must be whole source components (checked by verification); enumerate the rest
    for diag, pool, equiv, resolve in product((False, True), ("all", "targets", "enclosed"), ("T", "D8", "C8"),
                                              ("unique", "majority")):
        def fn(g, diag=diag, pool=pool, equiv=equiv, resolve=resolve):
            return shape_match(g, s, diag, pool, equiv, resolve, tgt)
        yield (f"recolour-shape-match[{'8' if diag else '4'},{pool},{equiv},{resolve}]",
               4 + (equiv != "T") + (resolve != "unique"), fn)



# ---------------------------------------------------------------- solid blocks
def solid_cover(g, T):
    h, w = H(g), W(g); cov = set()
    for r in range(h - T + 1):
        for c in range(w - T + 1):
            v = g[r][c]
            if all(g[y][x] == v for y in range(r, r + T) for x in range(c, c + T)):
                cov.update((y, x) for y in range(r, r + T) for x in range(c, c + T))
    return cov


def fam_recolour_solid_blocks(train):
    """Cells covered by a monochrome TxT square are recoloured to one induced colour (noise grids)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(i), W(i)) != (H(o), W(o)) or i == o: return
    src, tgt = change_colours(train)
    if len(tgt) != 1: return
    c = next(iter(tgt))
    for T in range(2, 7):
        ok = True
        for p in train:
            g, out = p["input"], p["output"]
            if (H(g), W(g)) != (H(out), W(out)): return
            cov = solid_cover(g, T)
            exp = [[c if (y, x) in cov else g[y][x] for x in range(W(g))] for y in range(H(g))]
            if exp != out: ok = False; break
        if ok:
            def fn(g, T=T):
                cov = solid_cover(g, T)
                return [[c if (y, x) in cov else g[y][x] for x in range(W(g))] for y in range(H(g))]
            yield (f"recolour-solid-blocks[{T}]", 4, fn)
            return


# ---------------------------------------------------------------- capped bars selected by key count
def capped_bars(g, bg, vert):
    """Maximal straight runs of non-bg cells (>=3, along columns if vert else rows) whose two end cells share
    colour a that does not occur in the interior."""
    bars = []
    lines = ([[(y, x) for y in range(H(g))] for x in range(W(g))] if vert else
             [[(y, x) for x in range(W(g))] for y in range(H(g))])
    for ln in lines:
        run = []
        for q in ln + [None]:
            if q is not None and g[q[0]][q[1]] != bg: run.append(q); continue
            if len(run) >= 3:
                a = g[run[0][0]][run[0][1]]
                if g[run[-1][0]][run[-1][1]] == a and a not in {g[y][x] for y, x in run[1:-1]}:
                    bars.append((a, run))
            run = []
    return bars


def key_rank_fill(g, desc, vert):
    bg = bg_of(g); bars = capped_bars(g, bg, vert)
    if not bars: return None
    inbar = {c for _, ob in bars for c in ob}
    out = [r[:] for r in g]; hit = False
    for a in {a for a, _ in bars}:
        k = sum(1 for y in range(H(g)) for x in range(W(g)) if g[y][x] == a and (y, x) not in inbar)
        grp = sorted((ob for c, ob in bars if c == a), key=len, reverse=desc)
        if k < 1 or k > len(grp): continue
        if k < len(grp) and len(grp[k - 1]) == len(grp[k]) or k > 1 and len(grp[k - 1]) == len(grp[k - 2]): return None
        for y, x in grp[k - 1]: out[y][x] = a
        hit = True
    return out if hit else None


def fam_recolour_bar_by_key_rank(train):
    """Per cap colour a: the k-th longest (or shortest) capped bar is filled with a, k = number of loose
    a-cells (the key)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(i), W(i)) != (H(o), W(o)) or i == o: return
    for vert, desc in product((True, False), (True, False)):
        if not capped_bars(i, bg_of(i), vert): continue
        yield (f"recolour-bar-by-key-rank[{'v' if vert else 'h'},{'desc' if desc else 'asc'}]", 5,
               lambda g, desc=desc, vert=vert: key_rank_fill(g, desc, vert))


FAMILIES = (fam_recolour_object_feature, fam_recolour_shape_match, fam_recolour_solid_blocks, fam_recolour_bar_by_key_rank)


In [ ]:
%%writefile /kaggle/working/probe/fam_segment_pack_pieces.py
"""segment.pack_pieces: the input's pieces (objects) are moved -- translated, optionally rotated /
reflected -- so that together they tile a target shape exactly; the output is that shape.

One parametrised primitive  pack[seg, target, moves, dims, canon, recolour]:
  seg      : c8 (single-colour 8-connected objects) | m8 (multi-colour 8-connected objects)
             | m8h (m8, enclosed background holes count as part of the piece)
  target   : full (solid h x w rectangle, h*w = #fg cells)
           | frame (border of an h x w rectangle, 2(h+w)-4 = #fg cells; interior = input background)
  moves    : T (translate only) | R (+ 90-degree rotations) | D (+ reflections)
  dims     : wide | tall  -- among the factorisations of the area with that orientation, the most
             square one that admits a (canonical, unique) tiling is used
  canon    : none   -- the tiling must be unique
           | keycell -- the single off-colour cell inside a piece (key) ends at the top-left corner
           | keypiece -- the piece carrying the key keeps its input orientation
           | largest -- the (unique) largest piece keeps its input orientation
  recolour : none | key -- the key is a kh x kw off-colour block inside one piece (kh*kw = #pieces);
             the output is cut into a kh x kw zone grid and every piece takes the key colour of the
             zone it mostly covers (pieces <-> zones must be a bijection)
All parameters are induced: a program is yielded only when it reproduces every training output.
Tilings are found by exact-cover backtracking (first empty cell in row-major order).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects

MAX_PIECES, MAX_AREA, MAX_SOL, NODE_BUDGET = 9, 225, 48, 60000
class _Budget(Exception): pass

# ------------------------------------------------------------------ pieces
def _holes(cells, h, w):
    """bg cells enclosed by the piece (not 4-connected to the bbox border outside the piece)."""
    s = {(y, x) for y, x, _ in cells}
    y0 = min(y for y, _ in s); x0 = min(x for _, x in s); y1 = max(y for y, _ in s); x1 = max(x for _, x in s)
    free = {(y, x) for y in range(y0, y1 + 1) for x in range(x0, x1 + 1) if (y, x) not in s}
    st = [c for c in free if c[0] in (y0, y1) or c[1] in (x0, x1)]; out = set(st)
    while st:
        y, x = st.pop()
        for q in ((y + 1, x), (y - 1, x), (y, x + 1), (y, x - 1)):
            if q in free and q not in out: out.add(q); st.append(q)
    return free - out

def _pieces(g, bg, seg):
    objs = objects(g, bg, diag=True, by_colour=(seg == 'c8'))
    ps = [[(y, x, g[y][x]) for y, x in o] for o in objs]
    if seg == 'm8h':  # enclosed background holes belong to the piece (take its main colour)
        for p in ps:
            main = Counter(v for _, _, v in p).most_common(1)[0][0]
            p.extend((y, x, main) for y, x in _holes(p, H(g), W(g)))
    return ps

def _norm(cells):
    y0 = min(c[0] for c in cells); x0 = min(c[1] for c in cells)
    s = sorted((y - y0, x - x0, v) for y, x, v in cells)
    # anchor = first cell in row-major order
    ay, ax = s[0][0], s[0][1]
    return tuple((y - ay, x - ax, v) for y, x, v in s)

def _variants(cells, moves):
    out, seen = [], set()
    cur = [(y, x, v) for y, x, v in cells]
    ops = []
    for refl in ((False, True) if moves == 'D' else (False,)):
        for k in (range(4) if moves in ('R', 'D') else range(1)):
            ops.append((refl, k))
    for refl, k in ops:
        c = [(y, -x, v) for y, x, v in cur] if refl else list(cur)
        for _ in range(k):
            c = [(x, -y, v) for y, x, v in c]  # rotate 90 cw
        n = _norm(c)
        if n not in seen:
            seen.add(n); out.append(n)
    return out  # out[0] is the identity orientation

def _key(pieces, kind, g=None):
    """Locate the key: off-colour cells inside the one multi-colour piece.
    kind 'cell' -> a single cell; 'block' -> the bbox of the off-colour cells, read from the grid
    (a kh x kw layout of colours, possibly with bg holes), with kh*kw == number of pieces."""
    found = None
    for i, p in enumerate(pieces):
        cnt = Counter(v for _, _, v in p)
        if len(cnt) < 2: continue
        main = cnt.most_common(1)[0][0]
        minor = [(y, x, v) for y, x, v in p if v != main]
        if found is not None: return None
        found = (i, main, minor)
    if found is None: return None
    i, main, minor = found
    if kind == 'cell':
        return (i, minor[0]) if len(minor) == 1 else None
    if len(minor) < 4: return None
    y0 = min(c[0] for c in minor); x0 = min(c[1] for c in minor)
    y1 = max(c[0] for c in minor); x1 = max(c[1] for c in minor)
    kh, kw = y1 - y0 + 1, x1 - x0 + 1
    if kh < 2 or kw < 2 or kh * kw != len(pieces): return None
    if len(minor) < kh * kw - 1: return None
    kb = [row[x0:x1 + 1] for row in g[y0:y1 + 1]]
    cells = {(y, x): v for y, x, v in pieces[i]}
    kb = [[cells.get((y, x), main) for x in range(x0, x1 + 1)] for y in range(y0, y1 + 1)]
    return (i, kb, main)

# ------------------------------------------------------------------ exact cover
def _tilings(h, w, target, pvars, cap):
    order = [(y, x) for y in range(h) for x in range(w) if target[y][x]]
    grid = [[-1] * w for _ in range(h)]
    n = len(pvars); used = [False] * n; place = [None] * n; sols = []; budget = [NODE_BUDGET]
    def rec(k):
        if len(sols) >= cap: return
        budget[0] -= 1
        if budget[0] < 0: raise _Budget()
        while k < len(order) and grid[order[k][0]][order[k][1]] != -1: k += 1
        if k == len(order):
            if all(used): sols.append(list(place))
            return
        y, x = order[k]
        for i in range(n):
            if used[i]: continue
            for vi, var in enumerate(pvars[i]):
                ok = True
                for dy, dx, _ in var:
                    yy, xx = y + dy, x + dx
                    if not (0 <= yy < h and 0 <= xx < w) or not target[yy][xx] or grid[yy][xx] != -1:
                        ok = False; break
                if not ok: continue
                for dy, dx, _ in var: grid[y + dy][x + dx] = i
                used[i] = True; place[i] = (vi, y, x)
                rec(k + 1)
                used[i] = False; place[i] = None
                for dy, dx, _ in var: grid[y + dy][x + dx] = -1
                if len(sols) >= cap: return
    try:
        rec(0)
    except _Budget:
        return None
    return sols

def _dims(n, target, orient):
    c = []
    if target == 'full':
        c = [(a, n // a) for a in range(1, n + 1) if n % a == 0]
    else:
        for a in range(3, n):
            b2 = n + 4 - 2 * a
            if b2 % 2 == 0 and b2 // 2 >= 3: c.append((a, b2 // 2))
    c = [(a, b) for a, b in c if (a <= b if orient == 'wide' else a >= b)]
    c.sort(key=lambda d: (abs(d[0] - d[1]), d))
    return c

def pack(g, seg, target, moves, orient, canon, recolour, tie='unique'):
    bg = bg_of(g)
    pieces = _pieces(g, bg, seg)
    if not (2 <= len(pieces) <= MAX_PIECES): return None
    n = sum(len(p) for p in pieces)
    if n > MAX_AREA: return None
    key = None
    if recolour == 'key':
        key = _key(pieces, 'block', g)
        if key is None: return None
    elif canon == 'keycell':
        key = _key(pieces, 'cell')
        if key is None: return None
    elif canon == 'keypiece':
        key = _key(pieces, 'cell') or _key(pieces, 'block', g)
        if key is None: return None
    pvars = [_variants(p, moves) for p in pieces]
    if canon == 'keypiece':
        pvars[key[0]] = pvars[key[0]][:1]
    elif canon == 'largest':
        sz = sorted((len(p) for p in pieces), reverse=True)
        if sz[0] == sz[1]: return None
        big = max(range(len(pieces)), key=lambda i: len(pieces[i]))
        pvars[big] = pvars[big][:1]
    for h, w in _dims(n, target, orient):
        if h > 30 or w > 30: continue
        tgt = [[1] * w for _ in range(h)] if target == 'full' else \
              [[1 if y in (0, h - 1) or x in (0, w - 1) else 0 for x in range(w)] for y in range(h)]
        sols = _tilings(h, w, tgt, pvars, MAX_SOL)
        if sols is None: return None
        outs = {}
        cen = [(sum(y for y, _, _ in p) / len(p), sum(x for _, x, _ in p) / len(p)) for p in pieces]
        for s in sols:
            o = [[bg] * w for _ in range(h)]
            owner = [[-1] * w for _ in range(h)]
            for i, (vi, y, x) in enumerate(s):
                for dy, dx, v in pvars[i][vi]:
                    o[y + dy][x + dx] = v; owner[y + dy][x + dx] = i
            if canon == 'keycell' and o[0][0] != key[1][2]: continue
            if canon == 'keycell' and owner[0][0] != key[0]: continue
            if recolour == 'key':
                kb = key[1]; kh, kw = len(kb), len(kb[0])
                ov = Counter((owner[y][x], y * kh // h, x * kw // w) for y in range(h) for x in range(w) if owner[y][x] >= 0)
                best = {}
                for (i, zy, zx), c in ov.items():
                    if i not in best or c > best[i][0]: best[i] = (c, zy, zx)
                zones = {(zy, zx) for _, zy, zx in best.values()}
                if len(zones) != len(pieces): continue
                cm = {i: kb[zy][zx] for i, (_, zy, zx) in best.items()}
                o = [[cm[owner[y][x]] if owner[y][x] >= 0 else o[y][x] for x in range(w)] for y in range(h)]
            sc = 0
            if tie == 'order':  # pairwise above/left relations kept from the input layout
                oc = []
                for i, (vi, y, x) in enumerate(s):
                    var = pvars[i][vi]
                    oc.append((y + sum(c[0] for c in var) / len(var), x + sum(c[1] for c in var) / len(var)))
                for i in range(len(pieces)):
                    for j in range(i + 1, len(pieces)):
                        for a in (0, 1):
                            d0 = cen[i][a] - cen[j][a]; d1 = oc[i][a] - oc[j][a]
                            if abs(d0) > 0.5 and d0 * d1 > 0: sc += 1
            k = tuple(map(tuple, o))
            outs[k] = max(outs.get(k, -1), sc)
            if tie == 'unique' and len(outs) > 1: break
        if outs:
            top = max(outs.values()); best = [k for k, v in outs.items() if v == top]
            if len(best) == 1 and (tie == 'order' or len(outs) == 1) and len(sols) < MAX_SOL:
                return [list(r) for r in best[0]]
            return None  # ambiguous at the preferred size
        if len(sols) >= MAX_SOL:
            return None
    return None

# ------------------------------------------------------------------ family
def _fg_count(g):
    bg = bg_of(g); return sum(v != bg for r in g for v in r)

def fam_pack_pieces(train):
    p0 = train[0]; i0, o0 = p0['input'], p0['output']
    if H(o0) * W(o0) >= H(i0) * W(i0): return
    n0 = _fg_count(i0)
    targets = []
    if H(o0) * W(o0) == n0: targets.append('full')
    if H(o0) >= 3 and W(o0) >= 3 and 2 * (H(o0) + W(o0)) - 4 == n0: targets.append('frame')
    if not targets: return
    orient = 'wide' if H(o0) <= W(o0) else 'tall'
    orients = [orient] + (['tall'] if H(o0) == W(o0) else [])
    combos = []
    for target in targets:
        for seg in ('c8', 'm8', 'm8h'):
            for moves in ('T', 'R', 'D'):
                canons = ['none'] + (['largest'] if moves != 'T' else []) + \
                         (['keycell', 'keypiece'] if seg != 'c8' and moves != 'T' else [])
                for canon in canons:
                    recs = ['none'] + (['key'] if seg != 'c8' and canon in ('none', 'keypiece') else [])
                    for rec in recs:
                        for ori in orients:
                            for tie in ('unique', 'order'):
                                combos.append((seg, target, moves, ori, canon, rec, tie))
    rank = {'unique': 0, 'order': 1, 'T': 0, 'R': 1, 'D': 2, 'c8': 0, 'm8': 1, 'm8h': 2}
    combos.sort(key=lambda c: (rank[c[6]], rank[c[2]], rank[c[0]], c[3] != orient))
    fits = []
    for c in combos:
        ok = True
        for p in train:
            try:
                r = pack(p['input'], *c)
            except RecursionError:
                r = None
            if r != p['output']: ok = False; break
        if ok:
            fits.append(c)
            if len(fits) >= 8: break
    if not fits: return
    # one program: the training-consistent parametrisations tried in order of simplicity;
    # the first that yields a (unique) packing of the test input answers.
    def fn(g, fits=fits):
        for c in fits:
            try:
                r = pack(g, *c)
            except RecursionError:
                r = None
            if r is not None: return r
        return None
    c = fits[0]
    name = 'pack[seg=%s,target=%s,moves=%s,dims=%s,canon=%s,recolour=%s,tie=%s]' % c
    yield (name + ('+%dalt' % (len(fits) - 1) if len(fits) > 1 else ''), 5, fn)

FAMILIES = (fam_pack_pieces,)


In [ ]:
%%writefile /kaggle/working/probe/fam_stamp_complete_partial_matches.py
"""STAMP.complete_partial_matches: complete every partial copy of a reference template.

One parametrised primitive.  A complete reference template (a multi-colour cluster, or the largest
shape) is present in the input; every other configuration that equals a part of the template, under a
transform group G and a block scale s, is completed in place by painting the template's missing cells.

Parameters, all induced from the training pairs (the family fits them internally and yields only
programs that reproduce every training output exactly):
  dom     : grid | lattice (grid-lines compress the picture to a cell grid; completion runs on cells)
  tsel    : which clusters are templates
            P      clusters containing a colour that training outputs add (optionally cropped to the
                   bbox of those colours, trimming touching noise)
            maxcol clusters with the maximal number of distinct colours (>= 2)
            big    the single cluster with the most (colours, scaled units)
            (clusters = non-bg cells linked at Chebyshev distance <= d, d in {1,2})
  krole   : which template colours form the key K that a fragment must contain completely
            fixed (template minus added colours) | least (least frequent colours) |
            nonmajor (all but the most frequent) | free (any present subset of cells; optionally
            colour-closed: each present colour complete) | none (with sigma=free, pure shape match)
  sigma   : id | free  (non-key colours may be recoloured by the fragment's own colour)
  G       : id | D4 ;  scale: 1 | 1..5 (each fragment block-scaled);  clip: template may leave the grid
  other   : bg  (non-key template cells must be bg or already the template colour; paint bg only)
            notK (non-key cells must merely not hold a key colour; paint over them)
  closure : none | comp (matched cells are whole 8-components) | class (matched colour classes whole)
Matches whose matched-cell set is strictly contained in another match are discarded; conflicting paints
reject the program.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
import time
from collections import Counter
from gdsl import H, W, bg_of

MAXC, MAXT = 64, 8   # a reference template is a small object; many/huge candidates -> no template
TRANS = [(sw, fy, fx) for sw in (0, 1) for fy in (0, 1) for fx in (0, 1)]


# ------------------------------------------------------------------ lattice domain
def lattice(g):
    """Return (line colour, row runs, col runs, compressed grid) or None."""
    h, w = H(g), W(g)
    for L in {v for r in g for v in r}:
        rows = [y for y in range(h) if all(v == L for v in g[y])]
        cols = [x for x in range(w) if all(g[y][x] == L for y in range(h))]
        if not rows or not cols or len(rows) == h or len(cols) == w: continue
        rr = runs([y for y in range(h) if y not in set(rows)])
        cc = runs([x for x in range(w) if x not in set(cols)])
        if len(rr) < 2 or len(cc) < 2: continue
        comp = []
        for a, b in rr:
            row = []
            for c, d in cc:
                vs = {g[y][x] for y in range(a, b + 1) for x in range(c, d + 1)}
                if len(vs) != 1: return None
                row.append(vs.pop())
            comp.append(row)
        return L, rr, cc, comp
    return None


def runs(idx):
    out = []
    for i in idx:
        if out and out[-1][1] == i - 1: out[-1][1] = i
        else: out.append([i, i])
    return [tuple(r) for r in out]


def expand(g, lat, comp):
    L, rr, cc, _ = lat
    out = [r[:] for r in g]
    for i, (a, b) in enumerate(rr):
        for j, (c, d) in enumerate(cc):
            for y in range(a, b + 1):
                for x in range(c, d + 1): out[y][x] = comp[i][j]
    return out


# ------------------------------------------------------------------ templates
def clusters(g, bg, d):
    h, w = H(g), W(g); seen = set(); out = []
    for r in range(h):
        for c in range(w):
            if g[r][c] == bg or (r, c) in seen: continue
            st = [(r, c)]; seen.add((r, c)); cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy in range(-d, d + 1):
                    for dx in range(-d, d + 1):
                        yy, xx = y + dy, x + dx
                        if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and g[yy][xx] != bg:
                            seen.add((yy, xx)); st.append((yy, xx))
            out.append(cells)
    return out


def block_scale(g, cells):
    """Largest s such that the cells are a union of aligned s x s uniform blocks."""
    cs = set(cells); y0 = min(y for y, _ in cells); x0 = min(x for _, x in cells)
    for s in (5, 4, 3, 2):
        ok = True
        for y, x in cells:
            by, bx = y0 + (y - y0) // s * s, x0 + (x - x0) // s * s
            if any((by + i, bx + j) not in cs or g[by + i][bx + j] != g[y][x] for i in range(s) for j in range(s)):
                ok = False; break
        if ok: return s
    return 1


def templates(g, bg, tsel, d, crop, pfix):
    cl = clusters(g, bg, d)
    if tsel == 'P':
        T = []
        for c in cl:
            pc = [(y, x) for y, x in c if g[y][x] in pfix]
            if not pc: continue
            if crop:
                y0 = min(y for y, _ in pc); y1 = max(y for y, _ in pc)
                x0 = min(x for _, x in pc); x1 = max(x for _, x in pc)
                c = [(y, x) for y, x in c if y0 <= y <= y1 and x0 <= x <= x1]
            T.append(c)
    elif tsel == 'maxcol':
        nc = [len({g[y][x] for y, x in c}) for c in cl]
        m = max(nc, default=0)
        if m < 2: return []
        T = [c for c, n in zip(cl, nc) if n == m]
    else:  # big
        key = lambda c: (len({g[y][x] for y, x in c}), len(c) / block_scale(g, c) ** 2)
        if not cl: return []
        ks = [key(c) for c in cl]; m = max(ks)
        if ks.count(m) != 1: return []
        T = [cl[ks.index(m)]]
    out = []; seen = set()
    if len(T) > MAXT: return []
    for c in T:
        if len(c) > MAXC: return []
        y0 = min(y for y, _ in c); x0 = min(x for _, x in c)
        t = tuple(sorted((y - y0, x - x0, g[y][x]) for y, x in c))
        if t not in seen: seen.add(t); out.append(t)
    return out


def key_colours(t, krole, pfix):
    cnt = Counter(c for _, _, c in t)
    if krole == 'fixed': K = set(cnt) - pfix
    elif krole == 'least':
        m = min(cnt.values()); K = {c for c in cnt if cnt[c] == m}
        if len(K) == len(cnt): return None
    elif krole == 'nonmajor':
        m = max(cnt.values()); top = [c for c in cnt if cnt[c] == m]
        if len(top) != 1: return None
        K = set(cnt) - set(top)
    else: K = set()
    if krole in ('fixed', 'least', 'nonmajor') and (not K or K == set(cnt)): return None
    return K


def variants(t, G):
    out = []; seen = set()
    for tr in (TRANS if G == 'D4' else TRANS[:1]):
        sw, fy, fx = tr
        h = max(a for a, _, _ in t) + 1; w = max(b for _, b, _ in t) + 1
        v = []
        for a, b, c in t:
            a2 = h - 1 - a if fy else a; b2 = w - 1 - b if fx else b
            v.append((b2, a2, c) if sw else (a2, b2, c))
        v = tuple(sorted(v))
        if v not in seen: seen.add(v); out.append(v)
    return out


# ------------------------------------------------------------------ matching
def block_tls(g, bg, s):
    h, w = H(g), W(g)
    if s == 1: return [(y, x) for y in range(h) for x in range(w) if g[y][x] != bg]
    return [(y, x) for y in range(h - s + 1) for x in range(w - s + 1) if g[y][x] != bg
            and all(g[y + i][x + j] == g[y][x] for i in range(s) for j in range(s))]


def raw_matches(g, bg, T, B):
    """All placements of all templates (every scale, clipping allowed); filters are applied later.
    Record = (M, foot, paints, s, offgrid, missing colours, present colours)."""
    h, w = H(g), W(g); res = []
    tls = {s: block_tls(g, bg, s) for s in SCALES}
    for t in T:
        K = key_colours(t, B['krole'], B['pfix'])
        if K is None: continue
        for v in variants(t, B['G']):
            kc = [x for x in v if x[2] in K]
            for s in SCALES:
                if not tls[s]: continue
                if kc:
                    a0, b0, c0 = kc[0]
                    origins = {(y - a0 * s, x - b0 * s) for y, x in tls[s] if g[y][x] == c0}
                elif B['sigma'] == 'id':
                    origins = {(y - a * s, x - b * s) for y, x in tls[s] for a, b, c in v if g[y][x] == c}
                else:
                    origins = {(y - a * s, x - b * s) for y, x in tls[s] for a, b, _ in v}
                for oy, ox in origins:
                    r = place(g, bg, v, K, s, oy, ox, B, h, w)
                    if r: res.append(r)
    return res


def place(g, bg, v, K, s, oy, ox, B, h, w):
    other, sigma = B['other'], B['sigma']
    M = set(); miss = []; present = set(); sig = {}; foot = set(); off = False
    for a, b, c in v:
        vals = set()
        for i in range(s):
            for j in range(s):
                y, x = oy + a * s + i, ox + b * s + j
                if 0 <= y < h and 0 <= x < w: vals.add(g[y][x]); foot.add((y, x))
                elif c in K: return None
                else: off = True
        if c in K:
            if vals != {c}: return None
            M.update((oy + a * s + i, ox + b * s + j) for i in range(s) for j in range(s)); continue
        if not vals: continue
        if other == 'notK':
            if vals & K: return None
            miss.append((a, b, c)); continue
        if len(vals) != 1: return None
        u = next(iter(vals))
        if u == bg: miss.append((a, b, c)); continue
        if sigma == 'id':
            if u != c: return None
        elif u in K or sig.setdefault(c, u) != u: return None
        present.add(c)
        M.update((y, x) for y in range(oy + a * s, oy + a * s + s) for x in range(ox + b * s, ox + b * s + s)
                 if 0 <= y < h and 0 <= x < w)
    if not miss or not M: return None
    if len(set(sig.values())) != len(sig): return None
    paints = {}
    for a, b, c in miss:
        col = sig.get(c, c)
        for i in range(s):
            for j in range(s):
                y, x = oy + a * s + i, ox + b * s + j
                if 0 <= y < h and 0 <= x < w and g[y][x] != col: paints[(y, x)] = col
    if not paints: return None
    return (frozenset(M), frozenset(foot), tuple(sorted(paints.items())), s, off,
            frozenset(c for _, _, c in miss), frozenset(present))


def closed(g, bg, M, foot, closure):
    h, w = H(g), W(g)
    if closure == 'comp':
        for y, x in M:
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and g[yy][xx] != bg and (yy, xx) not in foot: return False
    elif closure == 'class':
        cols = {g[y][x] for y, x in M}
        if any(g[y][x] in cols and (y, x) not in M for y in range(h) for x in range(w)): return False
    return True


def finalize(g, bg, recs, F):
    byM = {}
    for M, foot, pt, s, off, missc, presc in recs:
        if s > F['maxs'] or (off and not F['clip']): continue
        if F['colclosed'] and missc & presc: continue
        if not closed(g, bg, M, foot, F['closure']): continue
        byM.setdefault(M, set()).add(pt)
    keys = [M for M in byM if len(byM[M]) == 1]          # ambiguous fragments are left alone
    keys = [M for M in keys if not any(M < N for N in keys)]
    if not keys: return None
    out = [r[:] for r in g]; done = {}
    for M in keys:
        for (y, x), c in next(iter(byM[M])):
            if done.setdefault((y, x), c) != c: return None
            out[y][x] = c
    return out


def prepare(g, B):
    """-> (work grid, lattice info, bg, raw records) or None."""
    lat = None; work = g
    if B['dom'] == 'lattice':
        lat = lattice(g)
        if lat is None: return None
        work = lat[3]
    bg = bg_of(work)
    T = templates(work, bg, B['tsel'], B['d'], B['crop'], B['pfix'])
    if not T: return None
    return work, lat, bg, raw_matches(work, bg, T, B)


def apply(prep, F):
    if prep is None: return None
    work, lat, bg, recs = prep[:4]
    out = finalize(work, bg, recs, F)
    if out is None or out == work: return None
    return expand(prep[4], lat, out) if lat else out


def run_cfg(g, B, F):
    prep = prepare(g, B)
    if prep is None: return None
    prep = prep + (g,)
    return apply(prep, F)


# ------------------------------------------------------------------ family
SCALES = (1, 2, 3, 4, 5)
SEL = (('P', 1, True), ('P', 1, False), ('maxcol', 1, False), ('big', 1, False),
       ('P', 2, False), ('maxcol', 2, False), ('big', 2, False))


def bases(pfix, allbg, has_lat):
    others = ['bg'] if allbg else ['notK']
    for dom in ['grid'] + (['lattice'] if has_lat else []):
        for tsel, d, crop in SEL:
            if tsel == 'P' and not pfix: continue
            for krole, sigma in (('fixed', 'id'), ('least', 'id'), ('nonmajor', 'id'), ('least', 'free'),
                                 ('nonmajor', 'free'), ('free', 'id'), ('none', 'free')):
                if (krole == 'fixed') != (tsel == 'P'): continue
                for other in others:
                    if other == 'notK' and krole in ('free', 'none'): continue
                    for G in ('id', 'D4'):
                        yield dict(dom=dom, tsel=tsel, d=d, crop=crop, krole=krole, sigma=sigma, other=other,
                                   G=G, pfix=pfix)


def filters(B):
    anchorless = B['krole'] in ('free', 'none')
    for closure in ('none', 'comp', 'class'):
        if anchorless and closure == 'none': continue
        if B['other'] == 'notK' and closure != 'none': continue
        for colclosed in ((True, False) if B['krole'] == 'free' else (False,)):
            for maxs, clip in ((1, False), (1, True), (5, False), (5, True)):
                yield dict(closure=closure, colclosed=colclosed, maxs=maxs, clip=clip)


def fam_complete_partial(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    pfix = set(); allbg = True; has_lat = True
    for p in train:
        i, o = p['input'], p['output']
        if (H(i), W(i)) != (H(o), W(o)): return
        bg = bg_of(i); ch = 0
        for y in range(H(i)):
            for x in range(W(i)):
                if i[y][x] != o[y][x]:
                    ch += 1; pfix.add(o[y][x])
                    if i[y][x] != bg: allbg = False
        if ch == 0: return
        if lattice(i) is None: has_lat = False
    pfix = frozenset(pfix)
    t0 = time.time(); found = 0
    for B in bases(pfix, allbg, has_lat):
        if time.time() - t0 > BUDGET: return
        preps = []
        for p in train:
            try: pr = prepare(p['input'], B)
            except Exception: pr = None
            if pr is None or not pr[3]: preps = None; break
            preps.append(pr + (p['input'],))
            if time.time() - t0 > BUDGET: return
        if not preps: continue
        for F in filters(B):
            if all(apply(pr, F) == p['output'] for pr, p in zip(preps, train)):
                name = ("complete-partial[{dom},{tsel}{d}{c},K={krole},sig={sigma},{other},G={G},"
                        "{closure}{cc},s<={maxs},clip={clip}]").format(c='crop' if B['crop'] else '',
                        cc='+colclosed' if F['colclosed'] else '', **B, **F)
                yield (name, 4, lambda g, B=B, F=F: run_cfg(g, B, F))
                found += 1
                break
        if found >= 3: return


BUDGET = 1e9  # deterministic: wall-clock budget removed (Kaggle parity)


# ------------------------------------------------------------------ variant A: divided canvas, mirrored + scaled
def dividers(g):
    """a single full row and a single full column of one colour L -> (L, row, col)."""
    h, w = H(g), W(g)
    for L in {v for r in g for v in r}:
        rows = [y for y in range(h) if all(v == L for v in g[y])]
        cols = [x for x in range(w) if all(g[y][x] == L for y in range(h))]
        if len(rows) == 1 and len(cols) == 1: return L, rows[0], cols[0]
    return None


def _solid_rect(cells):
    y0 = min(y for y, _ in cells); y1 = max(y for y, _ in cells)
    x0 = min(x for _, x in cells); x1 = max(x for _, x in cells)
    return (y0, x0, y1, x1) if len(set(cells)) == (y1 - y0 + 1) * (x1 - x0 + 1) else None


def mirror_stamp(g, mirror, clip=False):
    """Quadrants split by a divider cross.  One quadrant holds the template: a solid single-colour core
    with decorations; every other object is a bare core (same colour, any size).  The decorations are
    re-drawn around each bare core, scaled by (core size / template core size) and, with mirror=True,
    reflected across each divider that separates the core from the template."""
    dv = dividers(g)
    if dv is None: return None
    L, ry, cx = dv
    bg = bg_of(g)
    # the divider cross would join everything: cluster with the divider row/column blanked out
    gg = [[bg if (y == ry or x == cx) else g[y][x] for x in range(W(g))] for y in range(H(g))]
    obs = clusters(gg, bg, 1)
    multi = [c for c in obs if len({g[y][x] for y, x in c}) > 1]
    if len(multi) != 1: return None
    tpl = multi[0]; bare = [c for c in obs if c is not tpl]
    kc = {g[y][x] for c in bare for y, x in c}
    if len(kc) != 1 or not bare: return None
    K = kc.pop()
    core = [(y, x) for y, x in tpl if g[y][x] == K]
    cr = _solid_rect(core)
    if cr is None: return None
    cy0, cx0, cy1, cx1 = cr; ch, cw = cy1 - cy0 + 1, cx1 - cx0 + 1
    sb = block_scale(g, tpl)
    while sb > 1 and (ch % sb or cw % sb or (cy0 - min(y for y, _ in tpl)) % sb
                      or (cx0 - min(x for _, x in tpl)) % sb): sb -= 1
    bh, bw = ch // sb, cw // sb
    deco = {}
    for y, x in tpl:
        if g[y][x] == K: continue
        a, b = (y - cy0), (x - cx0)
        deco[(a // sb if a >= 0 else -((-a + sb - 1) // sb), b // sb if b >= 0 else -((-b + sb - 1) // sb))] = g[y][x]
    tside = (cy0 < ry, cx0 < cx)
    out = [r[:] for r in g]; n = 0
    for c in bare:
        r = _solid_rect(c)
        if r is None: return None
        y0, x0, y1, x1 = r; hh, ww = y1 - y0 + 1, x1 - x0 + 1
        if hh % bh or ww % bw or hh // bh != ww // bw: return None
        u = hh // bh
        fy = mirror and (y0 < ry) != tside[0]; fx = mirror and (x0 < cx) != tside[1]
        for (a, b), col in deco.items():
            a2 = bh - 1 - a if fy else a; b2 = bw - 1 - b if fx else b
            for i in range(u):
                for j in range(u):
                    yy, xx = y0 + a2 * u + i, x0 + b2 * u + j
                    if not (0 <= yy < H(g) and 0 <= xx < W(g)):
                        if clip: continue
                        return None
                    if g[yy][xx] != bg: return None
                    out[yy][xx] = col; n += 1
    return out if n else None


def fam_mirror_scaled_stamp(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)) or dividers(i0) is None: return
    for mirror in (True, False):
        for clip in (False, True):
            yield ("complete-partial:divided[G=%s,scale=core,clip=%s]" % ("mirror" if mirror else "id", clip), 5,
                   lambda g, mirror=mirror, clip=clip: mirror_stamp(g, mirror, clip))


# ------------------------------------------------------------------ variant B: tile lattice, self template
def tile_lattice(g):
    """grid of equal k x k tiles separated by 1-wide lines of colour L -> (L, k, rows, cols, tiles)."""
    h, w = H(g), W(g)
    L = g[0][0]
    rows = [y for y in range(h) if all(v == L for v in g[y])]
    cols = [x for x in range(w) if all(g[y][x] == L for y in range(h))]
    if len(rows) < 3 or len(cols) < 3: return None
    k = rows[1] - rows[0] - 1
    if k < 2 or any(b - a - 1 != k for a, b in zip(rows, rows[1:])) or any(b - a - 1 != k for a, b in zip(cols, cols[1:])):
        return None
    tiles = [[tuple(tuple(g[y][x] for x in range(c + 1, c + 1 + k)) for y in range(r + 1, r + 1 + k))
              for c in cols[:-1]] for r in rows[:-1]]
    return L, k, rows, cols, tiles


def lattice_self_complete(g, mrole):
    """Tiles are all one common pattern except a few 'alt' tiles of one other pattern.  The alt tile's
    own mask (cells of its majority colour, or of the colour absent from the common tile) is the meta
    template: alt tiles are added so that the set of alt positions becomes that mask, placed at the
    unique offset that contains every existing alt tile."""
    lt = tile_lattice(g)
    if lt is None: return None
    L, k, rows, cols, tiles = lt
    cnt = Counter(t for r in tiles for t in r)
    if len(cnt) != 2: return None
    (common, _), (alt, na) = cnt.most_common()
    if na == 0: return None
    ac = Counter(v for r in alt for v in r)
    if mrole == 'major':
        m = ac.most_common()
        if len(m) < 2 or m[0][1] == m[1][1]: return None
        mc = m[0][0]
    else:
        new = set(ac) - {v for r in common for v in r}
        if len(new) != 1: return None
        mc = new.pop()
    mask = [(a, b) for a in range(k) for b in range(k) if alt[a][b] == mc]
    pos = {(i, j) for i, r in enumerate(tiles) for j, t in enumerate(r) if t == alt}
    R, C = len(tiles), len(tiles[0])
    fits = []
    for oy in range(-k + 1, R):
        for ox in range(-k + 1, C):
            ms = {(oy + a, ox + b) for a, b in mask}
            if pos <= ms and all(0 <= y < R and 0 <= x < C for y, x in ms): fits.append(ms)
    if len(fits) != 1 or fits[0] == pos: return None
    out = [r[:] for r in g]
    for i, j in fits[0]:
        for a in range(k):
            for b in range(k): out[rows[i] + 1 + a][cols[j] + 1 + b] = alt[a][b]
    return out


def fam_lattice_self_template(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)) or tile_lattice(i0) is None: return
    for mrole in ('major', 'new'):
        yield ("complete-partial:lattice-self[mask=%s]" % mrole, 5,
               lambda g, mrole=mrole: lattice_self_complete(g, mrole))


FAMILIES = (fam_complete_partial, fam_mirror_scaled_stamp, fam_lattice_self_template)


In [ ]:
%%writefile /kaggle/working/probe/fam_stamp_copy_to_congruent_target.py
"""STAMP.copy_to_congruent_target: a patterned source is copied onto target regions congruent with it.

One parametrised primitive, three ways of keying the target (all parameters induced on the training pairs;
a program is yielded only when it reproduces every training output):

  shape   objects = connected components of non-bg cells (conn 4|8, multicolour).
          src   : multi   (object with >= 2 colours) | nonplain (monochrome object that is neither a solid
                  rectangle nor a hollow rectangle)
          key   : mask   target cell mask == T(source mask)
                  bbox   target bbox dims == T(source bbox dims)
                  rect   target is a solid rectangle of the source's corner colour, any size; the source
                         pattern is resized by stretching the run that holds the innermost colour
          T     : dihedral transform of the source (identity first)
          recol : keep | recolour a monochrome source to the target colour
          erase : none | source | unmatched monochrome objects | both
  hole    source = the only colour forming exactly one monochrome component; key = the cells of the source
          bbox not in the source (all one colour z); every other placement where those cells are all z
          receives the source cells.
  glyph   blocks (solid monochrome squares, optionally inside a frame that is cropped) are replaced by the
          free-standing glyph of the same colour and size; colour of the copy = the colour paired with the
          block colour in a two-colour legend object (or kept).
"""
import sys; sys.path.insert(0, '/home/claude/work/widen'); sys.path.insert(0, '/home/claude/work/latent')
from collections import Counter
from gdsl import H, W, bg_of, D8
from fam_stamp_template_at_markers import components, safe, fits

N4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N8 = N4 + [(-1, -1), (-1, 1), (1, -1), (1, 1)]
TNAMES = ['id', 'fh', 'fv', 'r180', 'T', 'r90', 'r270', 'aT']


def comps(cells, nb):
    seen = set(); out = []
    for p in sorted(cells):
        if p in seen: continue
        seen.add(p); st = [p]; comp = []
        while st:
            y, x = st.pop(); comp.append((y, x))
            for dy, dx in nb:
                q = (y + dy, x + dx)
                if q in cells and q not in seen: seen.add(q); st.append(q)
        out.append({q: cells[q] for q in comp})
    return out


class Obj:
    def __init__(self, cells, bg):
        self.cells = cells
        ys = [y for y, _ in cells]; xs = [x for _, x in cells]
        self.y0, self.x0, self.y1, self.x1 = min(ys), min(xs), max(ys), max(xs)
        self.h, self.w = self.y1 - self.y0 + 1, self.x1 - self.x0 + 1
        self.cols = set(cells.values())
        self.pat = [[cells.get((self.y0 + i, self.x0 + j), bg) for j in range(self.w)] for i in range(self.h)]
        self.mask = [[(self.y0 + i, self.x0 + j) in cells for j in range(self.w)] for i in range(self.h)]
        n = len(cells)
        self.solid = n == self.h * self.w
        self.hollow = (self.h >= 3 and self.w >= 3 and n == 2 * (self.h + self.w) - 4 and
                       all(i in (0, self.h - 1) or j in (0, self.w - 1)
                           for i in range(self.h) for j in range(self.w) if self.mask[i][j]))


def objs_of(g, bg, conn):
    cells = {(y, x): g[y][x] for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg}
    return [Obj(c, bg) for c in comps(cells, N8 if conn == 8 else N4)]


# ------------------------------------------------------------------ resizing (stretch the innermost run)
def innermost(pat, bg):
    boxes = {}
    for i, r in enumerate(pat):
        for j, v in enumerate(r):
            if v == bg: continue
            b = boxes.get(v)
            boxes[v] = (i, j, i, j) if b is None else (min(b[0], i), min(b[1], j), max(b[2], i), max(b[3], j))
    if len(boxes) < 2: return None
    c = min(boxes, key=lambda k: (boxes[k][2] - boxes[k][0] + 1) * (boxes[k][3] - boxes[k][1] + 1))
    b = boxes[c]
    if not all(o[0] <= b[0] and o[1] <= b[1] and o[2] >= b[2] and o[3] >= b[3] for o in boxes.values()): return None
    return c


def axis_map(lines, c, n):
    runs = []
    for i, l in enumerate(lines):
        if runs and lines[runs[-1][0]] == l: runs[-1][1] += 1
        else: runs.append([i, 1])
    hit = [k for k, (s, ln) in enumerate(runs) if c in lines[s]]
    if len(hit) != 1: return None
    k = hit[0]; extra = n - len(lines)
    if runs[k][1] + extra < 1: return None
    idx = []
    for m, (s, ln) in enumerate(runs):
        idx += [s] * (ln + extra if m == k else ln)
    return idx


def resize(pat, h, w, bg):
    if (len(pat), len(pat[0])) == (h, w): return pat
    c = innermost(pat, bg)
    if c is None: return None
    ri = axis_map([tuple(r) for r in pat], c, h)
    ci = axis_map([tuple(r) for r in zip(*pat)], c, w)
    if ri is None or ci is None: return None
    return [[pat[i][j] for j in ci] for i in ri]


# ------------------------------------------------------------------ mode: shape
def run_shape(g, cfg):
    conn, src, key, tn, recol, erase = cfg
    bg = bg_of(g); tf = D8[tn]
    obs = objs_of(g, bg, conn)
    if len(obs) > 40: return None
    if src == 'multi': S = [o for o in obs if len(o.cols) >= 2]
    else: S = [o for o in obs if len(o.cols) == 1 and not o.solid and not o.hollow]
    if not S or len(S) > 4: return None
    if recol and any(len(s.cols) != 1 for s in S): return None
    Tg = [o for o in obs if o not in S and len(o.cols) == 1]
    out = [r[:] for r in g]; done = 0; unmatched = []; jobs = []
    for t in Tg:
        tc = next(iter(t.cols)); P = None
        for s in S:
            if key == 'rect':
                if not t.solid or s.pat[0][0] != tc or recol: continue
                q = resize(s.pat, t.h, t.w, bg)
                if q is None: continue
            else:
                q = tf(s.pat)
                if key == 'mask':
                    if tf(s.mask) != t.mask: continue
                elif (len(q), len(q[0])) != (t.h, t.w): continue
                if recol:
                    sc = next(iter(s.cols)); q = [[tc if v == sc else v for v in r] for r in q]
            if P is not None and P != q: return None
            P = q
        if P is None: unmatched.append(t); continue
        jobs.append((t, P)); done += 1
    if not done: return None
    if erase in ('src', 'both'):
        for s in S:
            for (y, x) in s.cells: out[y][x] = bg
    if erase in ('unm', 'both'):
        for t in unmatched:
            for (y, x) in t.cells: out[y][x] = bg
    for t, P in jobs:
        for i in range(t.h):
            for j in range(t.w):
                y, x = t.y0 + i, t.x0 + j; v = P[i][j]
                if v != bg: out[y][x] = v
                elif (y, x) in t.cells: out[y][x] = bg
    return out


def fam_shape(train):
    g0, o0 = train[0]['input'], train[0]['output']
    if (H(g0), W(g0)) != (H(o0), W(o0)) or g0 == o0: return
    n = 0
    for conn in (8, 4):
        for src in ('multi', 'nonplain'):
            for key in ('mask', 'bbox', 'rect'):
                for tn in (TNAMES if key != 'rect' else ['id']):
                    for recol in ((0, 1) if key != 'rect' else (0,)):
                        for erase in ('none', 'src', 'unm', 'both'):
                            cfg = (conn, src, key, tn, recol, erase)
                            fn = safe(lambda g, cfg=cfg: run_shape(g, cfg))
                            if fits(fn, train):
                                yield ('copy-congruent:shape[%s]' % ','.join(map(str, cfg)), 4, fn); n += 1
                                if n >= 4: return
        if n: return


# ------------------------------------------------------------------ mode: hole
def run_hole(g):
    h, w = H(g), W(g)
    bycol = {}
    for y in range(h):
        for x in range(w): bycol.setdefault(g[y][x], {})[(y, x)] = g[y][x]
    src = []
    for c, cells in bycol.items():
        if len(cells) < 4: continue
        if len(comps(cells, N8)) == 1: src.append(Obj(cells, -1))
    if len(src) != 1: return None
    s = src[0]
    K = [(i, j) for i in range(s.h) for j in range(s.w) if not s.mask[i][j]]
    if not K: return None
    zs = {g[s.y0 + i][s.x0 + j] for i, j in K}
    if len(zs) != 1: return None
    z = zs.pop(); sc = next(iter(s.cols))
    M = [(i, j) for i in range(s.h) for j in range(s.w) if s.mask[i][j]]
    out = [r[:] for r in g]; n = 0
    for y in range(h - s.h + 1):
        for x in range(w - s.w + 1):
            if (y, x) == (s.y0, s.x0): continue
            if all(g[y + i][x + j] == z for i, j in K) and not all(g[y + i][x + j] == z for i, j in M):
                for i, j in M: out[y + i][x + j] = sc
                n += 1
    return out if n else None


def fam_hole(train):
    g0, o0 = train[0]['input'], train[0]['output']
    if (H(g0), W(g0)) != (H(o0), W(o0)) or g0 == o0: return
    fn = safe(run_hole)
    if fits(fn, train): yield ('copy-congruent:hole', 4, fn)


# ------------------------------------------------------------------ mode: glyph (legend-keyed block substitution)
def big_frame(g):
    """Largest monochrome hollow rectangle (by-colour 8-component equal to its bbox border)."""
    bycol = {}
    for y in range(H(g)):
        for x in range(W(g)): bycol.setdefault(g[y][x], {})[(y, x)] = g[y][x]
    best = None
    for c, cells in bycol.items():
        for comp in comps(cells, N4):
            if len(comp) < 12: continue
            o = Obj(comp, -1)
            if o.hollow and (best is None or o.h * o.w > best.h * best.w): best = o
    return best


def run_glyph(g, crop, recol):
    bg = bg_of(g); h, w = H(g), W(g)
    if crop:
        f = big_frame(g)
        if f is None: return None
        inside = lambda y, x: f.y0 <= y <= f.y1 and f.x0 <= x <= f.x1
        fc = next(iter(f.cols))
    else:
        f = None; inside = lambda y, x: True; fc = None
    ib = Counter(g[y][x] for y in range(h) for x in range(w) if inside(y, x) and g[y][x] != fc).most_common(1)[0][0]
    cells_in = {(y, x): g[y][x] for y in range(h) for x in range(w)
                if inside(y, x) and g[y][x] not in (ib, fc)}
    blocks = [Obj(c, ib) for c in comps(cells_in, N4)]
    if not blocks or any(not b.solid or len(b.cols) != 1 for b in blocks): return None
    if len({(b.h, b.w) for b in blocks}) != 1: return None
    bh, bw = blocks[0].h, blocks[0].w
    if bh * bw < 4: return None
    outside = {(y, x): g[y][x] for y in range(h) for x in range(w) if not inside(y, x) and g[y][x] != bg}
    if not outside: return None
    oc = comps(outside, N8)
    glyph, legend = {}, {}
    mono = {}
    for comp in oc:
        cs = set(comp.values())
        if len(cs) == 1:
            mono.setdefault(next(iter(cs)), {}).update(comp)
        elif len(cs) == 2 and recol:
            for c in cs:
                cells = [p for p, v in comp.items() if v == c]
                o = Obj({p: c for p in cells}, bg)
                if o.x0 == min(x for _, x in comp) and o.y0 == min(y for y, _ in comp):
                    d = (cs - {c}).pop()
                    if legend.get(c, d) != d: return None
                    legend[c] = d
    for c, cells in mono.items():   # a glyph may consist of several pieces of its colour
        o = Obj(cells, bg)
        if (o.h, o.w) == (bh, bw) and not o.solid: glyph[c] = o
    out = [r[:] for r in g]
    for b in blocks:
        c = next(iter(b.cols))
        if c not in glyph: return None
        d = legend.get(c) if recol else c
        if d is None: return None
        P = glyph[c].pat
        for i in range(bh):
            for j in range(bw):
                out[b.y0 + i][b.x0 + j] = d if P[i][j] == c else ib
    if f is not None:
        out = [r[f.x0:f.x1 + 1] for r in out[f.y0:f.y1 + 1]]
    return out


def fam_glyph(train):
    for crop in (1, 0):
        for recol in (1, 0):
            fn = safe(lambda g, crop=crop, recol=recol: run_glyph(g, crop, recol))
            if fits(fn, train):
                yield ('copy-congruent:glyph[crop=%d,recol=%d]' % (crop, recol), 5, fn); return


FAMILIES = (fam_shape, fam_hole, fam_glyph)


In [ ]:
%%writefile /kaggle/working/probe/fam_stamp_local_stencil.py
"""STAMP.local_stencil: every seed (cell / object) gets a fixed relative stencil painted around it.

One parametrised primitive. Everything below is induced from the training pairs:
  seg    : how seeds are segmented   cell | obj4 | obj8 | multi4 | multi8 (non-background components)
  frame  : local coordinate frame    tl (bbox top-left) | near (offset from nearest bbox side, size-free)
           | d8 (shape canonicalised up to rotation/reflection; stencil turns with it)
           | wall (frame rotated so that the nearest uniform border line is "up")
  key    : seed signature selecting the stencil: () | colour | shape | colour+shape | square
           | unique-colour | colour+inside-other-object
  stencil: learned map canonical-offset -> colour spec, spec = constant colour | colour of the seed's
           own cell at a canonical position | colour of the nearest border line.  Offsets on the seed's
           own cells express erase / recolour of the seed.
  paint  : all | bg-only | protect (never overwrite a stamping seed)
The learner aggregates, per (key, offset), what the output holds in every training instance and keeps an
offset only when one spec explains >=75% of instances and at least two of them are real changes.
Programs are yielded only when they reproduce every training output exactly.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter, defaultdict
from gdsl import H, W, bg_of, objects, bbox

R = 2          # stencil radius around the seed bbox
MAXREF = 9     # seeds larger than this cannot be used as colour references
D8T = [(sw, fy, fx) for sw in (0, 1) for fy in (0, 1) for fx in (0, 1)]
ROT = [t for t in D8T if (t[1] ^ t[2]) == t[0]]   # the four proper rotations (chirality preserved)


def fwd(t, dy, dx, h, w):
    sw, fy, fx = t
    a = h - 1 - dy if fy else dy
    b = w - 1 - dx if fx else dx
    return (b, a) if sw else (a, b)


def dims(t, h, w):
    return (w, h) if t[0] else (h, w)


def anchor(a, n, near):
    if not near: return a
    return ('t', a) if 2 * a <= n - 1 else ('b', a - (n - 1))


def walls(g, bg):
    h, w = H(g), W(g); out = {}
    for side, line in (('top', g[0]), ('bottom', g[h - 1]), ('left', [r[0] for r in g]), ('right', [r[w - 1] for r in g])):
        if len(set(line)) == 1 and line[0] != bg: out[side] = line[0]
    return out


WALL_T = {'top': (0, 0, 0), 'bottom': (0, 1, 0), 'left': (1, 0, 0), 'right': (1, 0, 1)}


def segment(g, bg, seg):
    if seg == 'rare':   # marker cells on a two-tone canvas: every colour except the two most frequent ones
        cnt = Counter(c for r in g for c in r)
        if len(cnt) < 3: return []
        top2 = {c for c, _ in cnt.most_common(2)}
        return [[(y, x)] for y in range(H(g)) for x in range(W(g)) if g[y][x] not in top2]
    if seg == 'cell':
        return [[(y, x)] for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg]
    diag = seg.endswith('8'); byc = seg.startswith('obj')
    return objects(g, bg, diag, byc)


def seed_infos(g, seg, frame):
    """Per seed: cells, bbox, transform, canonical dims, features, wall colour.  None if the frame is undefined."""
    bg = bg_of(g); h, w = H(g), W(g)
    seeds = segment(g, bg, seg)
    if not seeds or len(seeds) > 400: return None, bg
    wl = walls(g, bg) if frame == 'wall' else None
    if frame == 'wall' and not wl: return None, bg
    colsets = [tuple(sorted({g[y][x] for y, x in s})) for s in seeds]
    ccount = Counter(colsets)
    infos = []
    for s, cs in zip(seeds, colsets):
        r0, c0, r1, c1 = bbox(s); sh, sw_ = r1 - r0 + 1, c1 - c0 + 1
        rel = [(y - r0, x - c0) for y, x in s]
        wcol = None
        if frame in ('d8', 'rot'):
            t = min(D8T if frame == 'd8' else ROT, key=lambda t: sorted(fwd(t, a, b, sh, sw_) for a, b in rel))
        elif frame == 'wall':
            d = {'top': r0, 'bottom': h - 1 - r1, 'left': c0, 'right': w - 1 - c1}
            ds = sorted((d[k], k) for k in wl)
            if len(ds) > 1 and ds[0][0] == ds[1][0]: t = None
            else: t = WALL_T[ds[0][1]]; wcol = wl[ds[0][1]]
        else:
            t = (0, 0, 0)
        if t is None:
            infos.append(None); continue
        ch, cw = dims(t, sh, sw_); near = frame == 'near'
        canon = {}
        for (a, b), (y, x) in zip(rel, s):
            p = fwd(t, a, b, sh, sw_)
            canon[(anchor(p[0], ch, near), anchor(p[1], cw, near))] = g[y][x]
        shape = tuple(sorted(fwd(t, a, b, sh, sw_) for a, b in rel))
        feat = {'colour': cs, 'shape': shape, 'square': sh == sw_ and sh >= 2,
                'unique': ccount[cs] == 1}
        infos.append(dict(cells=s, box=(r0, c0, r1, c1), t=t, cd=(ch, cw), canon=canon, feat=feat, wcol=wcol,
                          cs=cs))
    # surround: the colour (background included) forming a strict majority of the in-grid 4-border of the
    # seed's 4-connected cluster of same-coloured cells (a cell seed is judged together with its cluster)
    for inf in infos:
        if inf is None: continue
        own = set(inf['cells']); stack = list(own)
        while stack:
            y, x = stack.pop()
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in own and g[yy][xx] in inf['cs']:
                    own.add((yy, xx)); stack.append((yy, xx))
        nb = Counter()
        for y, x in own:
            for yy, xx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):
                if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in own: nb[g[yy][xx]] += 1
        tot = sum(nb.values())
        inf['feat']['nbr'] = next((c for c, n in nb.items() if 2 * n > tot and c not in inf['cs']), None)
    return infos, bg


def window(inf, h, w, near):
    r0, c0, r1, c1 = inf['box']; sh, sw_ = r1 - r0 + 1, c1 - c0 + 1
    t = inf['t']; ch, cw = inf['cd']
    for y in range(max(0, r0 - R), min(h, r1 + R + 1)):
        for x in range(max(0, c0 - R), min(w, c1 + R + 1)):
            a, b = fwd(t, y - r0, x - c0, sh, sw_)
            yield y, x, (anchor(a, ch, near), anchor(b, cw, near))


def keyof(inf, key):
    return tuple(inf['feat'][k] for k in key)


KEYS = [(), ('colour',), ('shape',), ('colour', 'shape'), ('square',), ('unique',), ('colour', 'nbr')]


def spec_colour(inf, sp):
    if sp[0] == 'c': return sp[1]
    if sp[0] == 's': return inf['canon'].get(sp[1])
    return inf['wcol']


def finalize(stats):
    sten = {}
    for v, offs in stats.items():
        m = {}
        for off, (n, mc, cc) in offs.items():
            if not cc: continue
            sp, c = max(mc.items(), key=lambda it: (it[1], cc[it[0]], -len(it[0])))
            if c >= 0.75 * n and cc[sp] >= 2 and 2 * cc[sp] >= c:
                m[off] = sp
        sten[v] = m
    return sten


def learn(train, seg, frame, keys):
    """Two phases.  Phase 1 learns what happens to each seed's OWN cells.  Phase 2 learns the full stencil,
    ignoring cells of another seed whose output that seed's own rule already explains (so neighbouring
    seeds do not leak spurious offsets into each other's stencils)."""
    near = frame == 'near'
    seen = {k: defaultdict(set) for k in keys}   # offsets observed per key value (no extrapolation at test)
    pairs = []
    for p in train:
        g, o = p['input'], p['output']
        infos, bg = seed_infos(g, seg, frame)
        if infos is None: return None
        h, w = H(g), W(g)
        owner, rows = {}, []
        for inf in infos:
            if inf is None: continue
            own = set(inf['cells'])
            win = list(window(inf, h, w, near))
            inf['ownoff'] = {(y, x): off for y, x, off in win if (y, x) in own}
            for c in own: owner[c] = inf
            kv = [(k, keyof(inf, k)) for k in keys]
            for k, v in kv: seen[k][v].update(off for _, _, off in win)
            rows.append((inf, own, win, kv))
        pairs.append((g, o, owner, rows))

    def gather(phase, ownmaps):
        stats = {k: defaultdict(lambda: defaultdict(lambda: [0, Counter(), Counter()])) for k in keys}
        for g, o, owner, rows in pairs:
            for inf, own, win, kv in rows:
                refs = inf['canon'] if len(inf['cells']) <= MAXREF else {}
                for y, x, off in win:
                    mine = (y, x) in own
                    if phase == 1 and not mine: continue
                    ov = o[y][x]
                    other = None if mine else owner.get((y, x))
                    for k, v in kv:
                        if other is not None:
                            sp = ownmaps[k].get(keyof(other, k), {}).get(other['ownoff'][(y, x)])
                            pred = g[y][x] if sp is None else spec_colour(other, sp)
                            if pred == ov: continue
                        chg = ov != g[y][x]
                        specs = [('c', ov)] + [('s', q) for q, c in refs.items() if c == ov]
                        if inf['wcol'] is not None and inf['wcol'] == ov: specs.append(('w',))
                        st = stats[k][v][off]; st[0] += 1
                        for sp in specs:
                            st[1][sp] += 1
                            if chg: st[2][sp] += 1
        return stats

    st1 = gather(1, None)
    ownmaps = {k: finalize(st1[k]) for k in keys}
    st2 = gather(2, ownmaps)
    res = {}
    for k in keys:
        sten = finalize(st2[k])
        if any(sten.values()):
            res[k] = (sten, seen[k])
    return res


def apply(g, seg, frame, key, learned, mode):
    sten, seen = learned
    infos, bg = seed_infos(g, seg, frame)
    if infos is None or any(i is None for i in infos): return None
    h, w = H(g), W(g); near = frame == 'near'
    out = [r[:] for r in g]
    plan = []
    for inf in infos:
        kv = keyof(inf, key)
        if kv not in sten: return None
        m = sten[kv]
        if m:
            if any(off not in seen[kv] for _, _, off in window(inf, h, w, near)): return None
            plan.append((inf, m))
    protect = {c for inf, _ in plan for c in inf['cells']}
    ext = []
    for inf, m in plan:
        own = set(inf['cells'])
        for y, x, off in window(inf, h, w, near):
            sp = m.get(off)
            if sp is None: continue
            if sp[0] == 'c': col = sp[1]
            elif sp[0] == 's':
                col = inf['canon'].get(sp[1])
                if col is None: return None
            else: col = inf['wcol']
            if (y, x) in own: out[y][x] = col
            else: ext.append((y, x, col))
    for y, x, col in ext:
        if mode == 'bg' and g[y][x] != bg: continue
        if mode == 'protect' and (y, x) in protect: continue
        out[y][x] = col
    return out


def fam_local_stencil(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    # every changed cell must lie within Chebyshev R of a non-background input cell
    for p in train:
        g, o = p['input'], p['output']; bg = bg_of(g); h, w = H(g), W(g)
        for y in range(h):
            for x in range(w):
                if g[y][x] != o[y][x] and not any(g[yy][xx] != bg for yy in range(max(0, y - R), min(h, y + R + 1))
                                                  for xx in range(max(0, x - R), min(w, x + R + 1))):
                    return
    outs = [p['output'] for p in train]
    found = 0
    for seg in ('cell', 'rare', 'obj4', 'obj8', 'multi4', 'multi8'):
        for frame in ('tl', 'wall', 'near', 'd8', 'rot'):
            if seg in ('cell', 'rare') and frame in ('near', 'd8', 'rot'): continue
            if frame in ('d8', 'rot'): keys = [('shape',), ('colour', 'shape')]
            elif seg in ('cell', 'rare'): keys = [k for k in KEYS if 'shape' not in k and 'square' not in k]
            else: keys = KEYS
            if frame == 'near': keys = [k for k in keys if 'shape' not in k]
            try:
                learned = learn(train, seg, frame, keys)
            except Exception:
                learned = None
            if not learned: continue
            for key, sten in learned.items():
                for mode in ('all', 'bg', 'protect'):
                    ok = all(apply(p['input'], seg, frame, key, sten, mode) == p['output'] for p in train)
                    if not ok: continue
                    nk = '+'.join(key) or 'any'
                    cost = 4 + (len(key) > 1) + (frame in ('d8', 'rot', 'wall', 'near'))
                    yield (f"stencil[{seg},{frame},key={nk},{mode}]", cost,
                           lambda g, seg=seg, frame=frame, key=key, sten=sten, mode=mode: apply(g, seg, frame, key, sten, mode))
                    found += 1
                    break
                if found >= 40: return


FAMILIES = (fam_local_stencil,)


In [ ]:
%%writefile /kaggle/working/probe/fam_stamp_template_at_markers.py
"""STAMP.template_at_markers: a multi-cell template is copied onto every marker.

One parametrised primitive; every parameter is induced from the training pairs (programs are only
yielded when they reproduce all training outputs exactly):
  seg     : how templates and markers are separated
            iso      markers = isolated non-bg cells (no non-bg 8-neighbour); templates = 8-connected
                     multicolour components with >= 2 cells.  Optionally full-length divider lines are
                     stripped first (they stay on the canvas but belong to no template).
            role:m   markers = every cell of colour m (m enumerated over colours present in every input);
                     templates = 8-connected components of the remaining non-bg cells.
  anchor  : where a template attaches to a marker
            same     the template's unique cell of the marker's colour sits on the marker
            centre   the (single, odd-sized) template's bbox centre sits on the marker
            col:a    the template's unique cell of colour a sits on the marker (role mode)
  paste   : all | noanchor (the anchor cell itself is not painted)
  recol   : keep template colours | recolour copy to the marker colour
  tf      : identity | dihedral about the anchor, learned per anchor colour
  canvas  : keep | erase markers | erase templates | erase both
  layout  : grid (in place) | spread (copies laid along the marker axis with the markers' gaps, anchored at
            the template's own marker) | concat (output = copies concatenated along the marker axis)
            | frame (grid split in bordered frames; templates pooled from all frames, output = the frame
            that holds only markers, stamped)
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of

N8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
TFS = [lambda a, b: (a, b), lambda a, b: (a, -b), lambda a, b: (-a, b), lambda a, b: (-a, -b),
       lambda a, b: (b, a), lambda a, b: (b, -a), lambda a, b: (-b, a), lambda a, b: (-b, -a)]
TFN = ['id', 'fh', 'fv', 'r180', 'T', 'r90', 'r270', 'aT']


# ------------------------------------------------------------------ segmentation
def line_cells(g, bg):
    h, w = H(g), W(g); s = set()
    for y in range(h):
        if g[y][0] != bg and all(v == g[y][0] for v in g[y]): s.update((y, x) for x in range(w))
    for x in range(w):
        c = g[0][x]
        if c != bg and all(g[y][x] == c for y in range(h)): s.update((y, x) for y in range(h))
    return s


def components(cells):
    """8-connected components of a dict {(y,x):colour}."""
    seen = set(); out = []
    for p in sorted(cells):
        if p in seen: continue
        seen.add(p); st = [p]; comp = []
        while st:
            y, x = st.pop(); comp.append((y, x))
            for dy, dx in N8:
                q = (y + dy, x + dx)
                if q in cells and q not in seen: seen.add(q); st.append(q)
        out.append({q: cells[q] for q in comp})
    return out


def segment(g, bg, seg, region=None):
    """-> (templates [dict cell->colour], markers [(y,x,c)]) or None."""
    h, w = H(g), W(g)
    ys, xs = (range(h), range(w)) if region is None else (range(region[0], region[2] + 1), range(region[1], region[3] + 1))
    cells = {(y, x): g[y][x] for y in ys for x in xs if g[y][x] != bg}
    if seg == 'iso' or seg == 'iso-strip':
        if seg == 'iso-strip':
            ls = line_cells(g, bg)
            if not ls: return None
            for p in ls: cells.pop(p, None)
        mk = [(y, x, c) for (y, x), c in cells.items() if not any((y + dy, x + dx) in cells for dy, dx in N8)]
        for y, x, _ in mk: del cells[(y, x)]
    else:
        m = seg[1]
        mk = [(y, x, c) for (y, x), c in cells.items() if c == m]
        for y, x, _ in mk: del cells[(y, x)]
    tpls = [t for t in components(cells) if len(t) >= 2]
    if len(tpls) > 12 or len(mk) > 40: return None
    return tpls, sorted(mk)


def unique_cell(t, c):
    ps = [p for p, v in t.items() if v == c]
    return ps[0] if len(ps) == 1 else None


def centre(t):
    ys = [y for y, _ in t]; xs = [x for _, x in t]
    if (max(ys) - min(ys)) % 2 or (max(xs) - min(xs)) % 2: return None
    return ((max(ys) + min(ys)) // 2, (max(xs) + min(xs)) // 2)


def match(tpls, marker, anchor):
    """-> (template, anchor cell, key) or None (marker unmatched); raises ValueError if ambiguous/undefined."""
    y, x, c = marker
    if anchor == 'centre':
        if len(tpls) != 1: raise ValueError
        a = centre(tpls[0])
        if a is None: raise ValueError
        return tpls[0], a, '*'
    ac = c if anchor == 'same' else anchor[1]
    hits = [(t, unique_cell(t, ac)) for t in tpls]
    hits = [(t, a) for t, a in hits if a is not None]
    if not hits: return None
    if len({frozenset(((y - a[0], x - a[1]), v) for (y, x), v in t.items()) for t, a in hits}) > 1:
        raise ValueError   # different templates claim the marker: ambiguous (identical copies are fine)
    return hits[0][0], hits[0][1], ac


def stamp_cells(t, a, marker, paste, recol, tf):
    y, x, c = marker; f = TFS[tf]
    for (ty, tx), v in t.items():
        if (ty, tx) == a and paste == 'noanchor': continue
        dy, dx = f(ty - a[0], tx - a[1])
        yield y + dy, x + dx, (c if recol and (ty, tx) != a else v)


# ------------------------------------------------------------------ in-place engine
def plan(g, bg, cfg, region=None, pool=None):
    seg = segment(g, bg, cfg['seg'], region)
    if seg is None: return None
    tpls, mk = seg
    use = pool if pool is not None else tpls
    if not use: return None
    jobs = []
    for m in mk:
        r = match(use, m, cfg['anchor'])
        if r is not None: jobs.append((m, r))
    return tpls, mk, jobs


def paint(g, bg, cfg, tpls, mk, jobs, tfmap, clip):
    o = [list(r) for r in g]
    er = cfg['canvas']
    if er in ('markers', 'both'):
        for y, x, _ in mk: o[y][x] = bg
    if er in ('tpl', 'both'):
        for t in tpls:
            for (y, x) in t: o[y][x] = bg
    r0, c0, r1, c1 = clip
    for m, (t, a, key) in jobs:
        tf = tfmap.get(key) if tfmap is not None else 0
        if tf is None: return None
        for y, x, v in stamp_cells(t, a, m, cfg['paste'], cfg['recol'], tf):
            if r0 <= y <= r1 and c0 <= x <= c1: o[y][x] = v
    return o


def learn_tf(train, cfg):
    ok = {}
    for p in train:
        g, o = p['input'], p['output']; bg = bg_of(g)
        pl = plan(g, bg, cfg)
        if pl is None or not pl[2]: return None
        for m, (t, a, key) in pl[2]:
            s = set()
            for k in range(8):
                if all(0 <= y < H(o) and 0 <= x < W(o) and o[y][x] == v
                       for y, x, v in stamp_cells(t, a, m, cfg['paste'], cfg['recol'], k)):
                    s.add(k)
            ok[key] = ok.get(key, s) & s
    if any(not s for s in ok.values()): return None
    return {k: min(s) for k, s in ok.items()}


def run_grid(g, cfg, tfmap):
    bg = bg_of(g)
    pl = plan(g, bg, cfg)
    if pl is None or not pl[2]: return None
    return paint(g, bg, cfg, *pl, tfmap, (0, 0, H(g) - 1, W(g) - 1))


# ------------------------------------------------------------------ layouts along the marker axis
def axis_of(mk):
    if len(mk) >= 2 and len({y for y, _, _ in mk}) == 1: return 'h'
    if len(mk) >= 2 and len({x for _, x, _ in mk}) == 1: return 'v'
    if len(mk) == 1: return 'any'
    return None


def tpl_box(t):
    ys = [y for y, _ in t]; xs = [x for _, x in t]
    return min(ys), min(xs), max(ys), max(xs)


def run_concat(g, cfg, axis_pref):
    bg = bg_of(g)
    seg = segment(g, bg, cfg['seg'])
    if seg is None: return None
    tpls, mk = seg
    if len(tpls) != 1 or not mk: return None
    ax = axis_of(mk)
    if ax is None: return None
    if ax == 'any': ax = axis_pref
    t = tpls[0]; r0, c0, r1, c1 = tpl_box(t); th, tw = r1 - r0 + 1, c1 - c0 + 1
    mk = sorted(mk, key=lambda m: (m[1], m[0]) if ax == 'h' else (m[0], m[1]))
    n = len(mk)
    o = [[bg] * (tw * n if ax == 'h' else tw) for _ in range(th if ax == 'h' else th * n)]
    for i, (_, _, c) in enumerate(mk):
        oy, ox = (0, i * tw) if ax == 'h' else (i * th, 0)
        for (y, x), v in t.items():
            o[oy + y - r0][ox + x - c0] = c if cfg['recol'] else v
    return o


def run_spread(g, cfg):
    bg = bg_of(g)
    seg = segment(g, bg, cfg['seg'])
    if seg is None: return None
    tpls, mk = seg
    if len(tpls) != 1 or len(mk) < 2: return None
    t = tpls[0]; cs = set(t.values())
    if len(cs) != 1: return None
    ax = axis_of(mk)
    if ax not in ('h', 'v'): return None
    mk = sorted(mk, key=lambda m: m[1] if ax == 'h' else m[0])
    own = [i for i, m in enumerate(mk) if m[2] in cs]
    if len(own) != 1: return None
    j = own[0]
    r0, c0, r1, c1 = tpl_box(t)
    pos = [m[1] if ax == 'h' else m[0] for m in mk]
    size = (c1 - c0 + 1) if ax == 'h' else (r1 - r0 + 1)
    start = {j: c0 if ax == 'h' else r0}
    for i in range(j + 1, len(mk)): start[i] = start[i - 1] + size - 1 + (pos[i] - pos[i - 1])
    for i in range(j - 1, -1, -1): start[i] = start[i + 1] - size + 1 - (pos[i + 1] - pos[i])
    o = [list(r) for r in g]
    for i, (_, _, c) in enumerate(mk):
        if i == j: continue
        for (y, x), v in t.items():
            yy, xx = (y, x - c0 + start[i]) if ax == 'h' else (y - r0 + start[i], x)
            if not (0 <= yy < H(g) and 0 <= xx < W(g)): return None
            o[yy][xx] = c
    return o


# ------------------------------------------------------------------ frames
def frames(g):
    """Maximal single-colour rectangular borders (>= 3x3) whose interior is not that colour."""
    h, w = H(g), W(g)
    R = [[0] * (w + 1) for _ in range(h + 1)]; D = [[0] * (w + 1) for _ in range(h + 1)]
    for y in range(h - 1, -1, -1):
        for x in range(w - 1, -1, -1):
            R[y][x] = R[y][x + 1] + 1 if x + 1 < w and g[y][x + 1] == g[y][x] else 1
            D[y][x] = D[y + 1][x] + 1 if y + 1 < h and g[y + 1][x] == g[y][x] else 1
    out = []
    for y in range(h):
        for x in range(w):
            if R[y][x] < 4 or D[y][x] < 4: continue
            if x > 0 and g[y][x - 1] == g[y][x] and R[y][x - 1] > R[y][x]: pass
            best = None
            for x1 in range(x + R[y][x] - 1, x + 2, -1):
                if D[y][x1] < 4: continue
                for y1 in range(y + min(D[y][x], D[y][x1]) - 1, y + 2, -1):
                    if R[y1][x] >= x1 - x + 1:
                        best = (y, x, y1, x1); break
                if best: break
            if best:
                y0, x0, y1, x1 = best
                inner = Counter(g[yy][xx] for yy in range(y0 + 1, y1) for xx in range(x0 + 1, x1))
                if inner.most_common(1)[0][0] != g[y][x]: out.append(best)
    out = [f for f in out if not any(o != f and o[0] <= f[0] and o[1] <= f[1] and o[2] >= f[2] and o[3] >= f[3] for o in out)]
    return out if 2 <= len(out) <= 6 else None


def run_frame(g, cfg, tfmap):
    fs = frames(g)
    if fs is None: return None
    info = []
    for f in fs:
        inner = (f[0] + 1, f[1] + 1, f[2] - 1, f[3] - 1)
        bg = Counter(g[y][x] for y in range(inner[0], inner[2] + 1) for x in range(inner[1], inner[3] + 1)).most_common(1)[0][0]
        seg = segment(g, bg, cfg['seg'], inner)
        if seg is None: return None
        info.append((f, inner, bg, seg))
    pool = [t for _, _, _, (ts, _) in info for t in ts]
    tg = [i for i in info if not i[3][0] and i[3][1]]
    if len(tg) != 1 or not pool: return None
    f, inner, bg, (ts, mk) = tg[0]
    jobs = []
    for m in mk:
        r = match(pool, m, cfg['anchor'])
        if r is not None: jobs.append((m, r))
    if not jobs: return None
    o = paint(g, bg, cfg, [], mk, jobs, tfmap, inner)
    if o is None: return None
    return [row[f[1]:f[3] + 1] for row in o[f[0]:f[2] + 1]]


# ------------------------------------------------------------------ induction
def safe(fn):
    def w(g):
        try: return fn(g)
        except (ValueError, KeyError, IndexError): return None
    return w


def fits(fn, train):
    for p in train:
        r = fn(p['input'])
        if r is None or r != p['output']: return False
    return True


def seg_options(train):
    opts = ['iso']
    if all(line_cells(p['input'], bg_of(p['input'])) for p in train): opts.append('iso-strip')
    common = None
    for p in train:
        g = p['input']; bg = bg_of(g)
        cs = {v for r in g for v in r} - {bg}
        common = cs if common is None else common & cs
    for m in sorted(common or ()):
        ok = True
        for p in train:
            g = p['input']
            n = sum(v == m for r in g for v in r)
            if n > 12: ok = False; break
        if ok: opts.append(('role', m))
    return opts


def anchors_for(seg, train):
    if seg in ('iso', 'iso-strip'): return ['same', 'centre']
    cs = set()
    for p in train:
        g = p['input']; bg = bg_of(g)
        cs |= {v for r in g for v in r} - {bg, seg[1]}
    return ['centre'] + [('col', a) for a in sorted(cs)]


def segname(s): return s if isinstance(s, str) else 'role=m%d' % s[1]


def fam_template_at_markers(train):
    g0, o0 = train[0]['input'], train[0]['output']
    same = all((H(p['input']), W(p['input'])) == (H(p['output']), W(p['output'])) for p in train)
    if same and all(p['input'] == p['output'] for p in train): return
    segs = seg_options(train)
    out = []
    if same:
        for seg in segs:
            for anc in anchors_for(seg, train):
                # cheap precondition: every pair has at least one matched marker
                ok = True
                for p in train:
                    try:
                        pl = plan(p['input'], bg_of(p['input']), {'seg': seg, 'anchor': anc})
                    except ValueError:
                        pl = None
                    if pl is None or not pl[2]: ok = False; break
                if not ok: continue
                for paste in ('all', 'noanchor'):
                    for recol in (False, True):
                        base = {'seg': seg, 'anchor': anc, 'paste': paste, 'recol': recol}
                        try: tfl = learn_tf(train, base)
                        except ValueError: tfl = None
                        tfopts = [None] + ([tfl] if tfl and any(v for v in tfl.values()) else [])
                        for tfm in tfopts:
                            for canvas in ('keep', 'markers', 'tpl', 'both'):
                                cfg = dict(base, canvas=canvas)
                                fn = safe(lambda g, cfg=cfg, tfm=tfm: run_grid(g, cfg, tfm))
                                if fits(fn, train):
                                    tn = 'id' if tfm is None else ','.join('%s:%s' % (k, TFN[v]) for k, v in sorted(tfm.items(), key=str))
                                    out.append(('stamp:template_at_markers[seg=%s,anchor=%s,paste=%s,recol=%d,tf=%s,erase=%s]'
                                                % (segname(seg), anc if isinstance(anc, str) else 'c%d' % anc[1], paste, recol, tn, canvas),
                                                4 + (tfm is not None) + (canvas != 'keep') // 2, fn))
                                    break
                            if len(out) >= 6: break
        for seg in segs:
            if not isinstance(seg, str): continue
            cfg = {'seg': seg}
            fn = safe(lambda g, cfg=cfg: run_spread(g, cfg))
            if fits(fn, train):
                out.append(('stamp:template_at_markers[layout=spread,seg=%s,recol=1]' % seg, 5, fn))
    else:
        for seg in segs:
            if not isinstance(seg, str): continue
            for recol in (True, False):
                for pref in ('h', 'v'):
                    cfg = {'seg': seg, 'recol': recol}
                    fn = safe(lambda g, cfg=cfg, pref=pref: run_concat(g, cfg, pref))
                    if fits(fn, train):
                        out.append(('stamp:template_at_markers[layout=concat,seg=%s,recol=%d,single=%s]' % (seg, recol, pref), 5, fn))
                        break
        if H(o0) < H(g0) and W(o0) < W(g0) and frames(g0) is not None:
            for anc in ('same',):
                for paste in ('all', 'noanchor'):
                    for recol in (False, True):
                        cfg = {'seg': 'iso', 'anchor': anc, 'paste': paste, 'recol': recol, 'canvas': 'keep'}
                        fn = safe(lambda g, cfg=cfg: run_frame(g, cfg, None))
                        if fits(fn, train):
                            out.append(('stamp:template_at_markers[layout=frame,anchor=%s,paste=%s,recol=%d]' % (anc, paste, recol), 5, fn))
    seen = set()
    for name, cost, fn in out[:8]:
        yield name, cost, fn


FAMILIES = (fam_template_at_markers,)


In [ ]:
%%writefile /kaggle/working/probe/fam_summarise_concentric_rings.py
"""summarise.concentric_rings: an ordered colour list is rendered as nested concentric square rings.

One primitive, ring(depth d) = L[min(d, n-1)] (or a periodic table of d), with three induced
parameter axes:
  items/key : colour classes of the input ranked by count | bbox size | occlusion layer |
              distance to grid centre;  or a legend read from a corner/edge ray of the input;
              or a periodic table of d learned from the training outputs
  canvas    : fresh square (2n-1 | 2n | max item bbox dim | input side)
              | interior of the unique rectangular frame (depth from the frame)
              | around a seed cell on the input (unique isolated cell | unique line crossing)
  paint     : overwrite | bg-only
All choices are enumerated from small generic domains and verified on every training pair.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox


# ------------------------------------------------------------------ helpers
def _ring_grid(S, L):
    n = len(L)
    return [[L[min(min(y, x, S - 1 - y, S - 1 - x), n - 1)] for x in range(S)] for y in range(S)]


def _decode_rings(o):
    """Colour per depth if o is a square grid of monochrome concentric rings, else None."""
    S = H(o)
    if S != W(o): return None
    D = (S + 1) // 2; L = [None] * D
    for y in range(S):
        for x in range(S):
            d = min(y, x, S - 1 - y, S - 1 - x)
            if L[d] is None: L[d] = o[y][x]
            elif L[d] != o[y][x]: return None
    while len(L) > 1 and L[-1] == L[-2]: L.pop()
    return L


def _cells_by_colour(g, bg):
    cs = {}
    for y, r in enumerate(g):
        for x, v in enumerate(r):
            if v != bg: cs.setdefault(v, []).append((y, x))
    return cs


def _layers(g, bg, cells):
    """Bottom-to-top order of colour classes from occlusion evidence on their bbox perimeters
    (colour d seen on c's perimeter = d lies above c); the order violating the least evidence,
    None if that order is not unique or there is no evidence."""
    from itertools import permutations
    if len(cells) > 7: return None
    wt = Counter()
    for c, cs in cells.items():
        y0, x0, y1, x1 = bbox(cs)
        per = {(y0, x) for x in range(x0, x1 + 1)} | {(y1, x) for x in range(x0, x1 + 1)} | \
              {(y, x0) for y in range(y0, y1 + 1)} | {(y, x1) for y in range(y0, y1 + 1)}
        for y, x in per:
            v = g[y][x]
            if v != c and v != bg and v in cells: wt[(c, v)] += 1
    if not wt: return None
    best, arg = None, []
    for perm in permutations(cells):
        pos = {c: i for i, c in enumerate(perm)}
        cost = sum(n for (c, v), n in wt.items() if pos[v] < pos[c])
        if best is None or cost < best: best, arg = cost, [perm]
        elif cost == best: arg.append(perm)
    return list(arg[0]) if len(arg) == 1 else None


def _feature(g, bg, cells, key):
    h, w = H(g), W(g)
    if key == "count": return {c: len(v) for c, v in cells.items()}
    if key == "size":
        out = {}
        for c, v in cells.items():
            y0, x0, y1, x1 = bbox(v); out[c] = max(y1 - y0 + 1, x1 - x0 + 1)
        return out
    if key == "area":
        out = {}
        for c, v in cells.items():
            y0, x0, y1, x1 = bbox(v); out[c] = (y1 - y0 + 1) * (x1 - x0 + 1)
        return out
    if key == "centre":  # doubled Chebyshev distance of the class's nearest cell to the grid centre
        return {c: min(max(abs(2 * y - (h - 1)), abs(2 * x - (w - 1))) for y, x in v) for c, v in cells.items()}
    return None


def _rank(g, key, desc):
    bg = bg_of(g); cells = _cells_by_colour(g, bg)
    if not cells or len(cells) > 10: return None
    if key == "layer":
        o = _layers(g, bg, cells)
        if o is None: return None
        return o if not desc else o[::-1]
    f = _feature(g, bg, cells, key)
    vals = list(f.values())
    if len(set(vals)) != len(vals): return None  # ties -> ambiguous
    return sorted(cells, key=lambda c: -f[c] if desc else f[c])


def _canvas_size(g, L, how):
    n = len(L)
    if how == "2n-1": return 2 * n - 1
    if how == "2n": return 2 * n
    if how == "side": return H(g) if H(g) == W(g) else None
    if how == "maxdim":
        bg = bg_of(g); cells = _cells_by_colour(g, bg)
        m = 0
        for v in cells.values():
            y0, x0, y1, x1 = bbox(v); m = max(m, y1 - y0 + 1, x1 - x0 + 1)
        return m
    return None


# ------------------------------------------------------------------ A: fresh square canvas
def fam_rings_summary(train):
    outs = [p["output"] for p in train]
    Ls = [_decode_rings(o) for o in outs]
    if any(L is None or len(L) < 2 for L in Ls): return
    for key in ("count", "size", "area", "layer", "centre"):
        for desc in (True, False):
            ranks = [_rank(p["input"], key, desc) for p in train]
            if any(r is None or r != L for r, L in zip(ranks, Ls)): continue
            for how in ("2n-1", "2n", "maxdim", "side"):
                if all(_canvas_size(p["input"], L, how) == H(o) for p, L, o in zip(train, Ls, outs)):
                    def fn(g, key=key, desc=desc, how=how):
                        L = _rank(g, key, desc)
                        if not L: return None
                        S = _canvas_size(g, L, how)
                        if not S or S < 2 * len(L) - 1 or S > 30: return None
                        return _ring_grid(S, L)
                    yield ("rings:summary[key=%s%s,canvas=%s]" % (key, "-desc" if desc else "-asc", how), 4, fn)


# ------------------------------------------------------------------ legends (rays from corners)
RAYS = {}
for _cn, (_cy, _cx) in {"tl": (0, 0), "tr": (0, -1), "bl": (-1, 0), "br": (-1, -1)}.items():
    for _dn in ("row", "col", "diag"):
        RAYS[_cn + "-" + _dn] = (_cy, _cx, _dn)


def _legend(g, bg, ray, stop):
    cy, cx, dn = RAYS[ray]; h, w = H(g), W(g)
    y = cy % h; x = cx % w
    dy = 1 if y == 0 else -1; dx = 1 if x == 0 else -1
    if dn == "row": dy = 0
    if dn == "col": dx = 0
    L = []
    while 0 <= y < h and 0 <= x < w and (y, x) not in stop:
        if dn == "diag" and g[y][x] == bg: break
        L.append(g[y][x]); y += dy; x += dx
    while L and L[-1] == bg: L.pop()
    return L or None


# ------------------------------------------------------------------ B: inside a rectangular frame
def _frame(g, bg):
    """Unique single-colour rectangular outline (>=3x3) with an all-bg interior."""
    found = []
    for ob in objects(g, bg):
        y0, x0, y1, x1 = bbox(ob)
        if y1 - y0 < 2 or x1 - x0 < 2: continue
        per = 2 * (y1 - y0 + x1 - x0)
        if len(ob) != per: continue
        s = set(ob)
        if not all((y0, x) in s and (y1, x) in s for x in range(x0, x1 + 1)): continue
        if not all((y, x0) in s and (y, x1) in s for y in range(y0, y1 + 1)): continue
        if all(g[y][x] == bg for y in range(y0 + 1, y1) for x in range(x0 + 1, x1)):
            found.append((y0, x0, y1, x1, s))
    return found[0] if len(found) == 1 else None


def _fill_frame(g, ray):
    bg = bg_of(g); f = _frame(g, bg)
    if not f: return None
    y0, x0, y1, x1, s = f
    L = _legend(g, bg, ray, s)
    if not L: return None
    o = [r[:] for r in g]
    for y in range(y0 + 1, y1):
        for x in range(x0 + 1, x1):
            d = min(y - y0 - 1, x - x0 - 1, y1 - 1 - y, x1 - 1 - x)
            o[y][x] = L[min(d, len(L) - 1)]
    return o


def fam_rings_frame(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if _frame(i0, bg_of(i0)) is None: return
    for ray in RAYS:
        if all(_fill_frame(p["input"], ray) == p["output"] for p in train):
            yield ("rings:frame[legend=%s]" % ray, 4, lambda g, ray=ray: _fill_frame(g, ray))


# ------------------------------------------------------------------ C: around a seed on the input
def _seed(g, bg, how):
    h, w = H(g), W(g)
    if how == "isolated":
        c = [(y, x) for y in range(h) for x in range(w) if g[y][x] != bg and
             all(g[yy][xx] == bg for yy in range(max(0, y - 1), min(h, y + 2))
                 for xx in range(max(0, x - 1), min(w, x + 2)) if (yy, xx) != (y, x))]
        return c[0] if len(c) == 1 else None
    if how == "crossing":
        rows = [y for y in range(h) if g[y][0] != bg and len(set(g[y])) == 1]
        cols = [x for x in range(w) if g[0][x] != bg and len({g[y][x] for y in range(h)}) == 1]
        return (rows[0], cols[0]) if len(rows) == 1 and len(cols) == 1 else None
    return None


def _paint_seed(g, how, L=None, ray=None, table=None):
    bg = bg_of(g); s = _seed(g, bg, how)
    if s is None: return None
    sy, sx = s; h, w = H(g), W(g); o = [r[:] for r in g]
    if ray is not None:
        L = _legend(g, bg, ray, {s})
        if not L or len(L) < 2 or L[0] != g[sy][sx]: return None
        for y in range(h):
            for x in range(w):
                d = max(abs(y - sy), abs(x - sx))
                if d < len(L): o[y][x] = L[d]
        return o
    p, T = table
    for y in range(h):
        for x in range(w):
            d = max(abs(y - sy), abs(x - sx))
            if d >= 1 and g[y][x] == bg: o[y][x] = T[d % p]
    return o


def _learn_table(train, how):
    for p in (1, 2, 3, 4):
        T = {}; ok = True
        for pr in train:
            g, o = pr["input"], pr["output"]; bg = bg_of(g); s = _seed(g, bg, how)
            if s is None: return None
            for y in range(H(g)):
                for x in range(W(g)):
                    d = max(abs(y - s[0]), abs(x - s[1]))
                    if d == 0 or g[y][x] != bg:
                        if o[y][x] != g[y][x]: return None
                        continue
                    k = d % p
                    if T.setdefault(k, o[y][x]) != o[y][x]: ok = False; break
                if not ok: break
            if not ok: break
        if ok and len(T) == p and len(set(T.values())) > 1: return (p, T)
    return None


def fam_rings_seed(train):
    i0, o0 = train[0]["input"], train[0]["output"]
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    for how in ("isolated", "crossing"):
        if any(_seed(p["input"], bg_of(p["input"]), how) is None for p in train): continue
        for ray in RAYS:
            if all(_paint_seed(p["input"], how, ray=ray) == p["output"] for p in train):
                yield ("rings:seed[%s,legend=%s]" % (how, ray), 4, lambda g, how=how, ray=ray: _paint_seed(g, how, ray=ray))
        tb = _learn_table(train, how)
        if tb and all(_paint_seed(p["input"], how, table=tb) == p["output"] for p in train):
            yield ("rings:seed[%s,periodic=%d]" % (how, tb[0]), 5, lambda g, how=how, tb=tb: _paint_seed(g, how, table=tb))


# ------------------------------------------------------------------ D: nested frames rebuilt from fragments
def _d4(y, x, s):
    m = s - 1
    return {(y, x), (x, y), (m - y, x), (y, m - x), (m - y, m - x), (m - x, y), (x, m - y), (m - x, m - y)}


def _complete(g, cs, s):
    """Unique D4-symmetric s x s pattern whose visible part (window inside the grid) equals the fragment cs."""
    h, w = H(g), W(g); y0, x0, y1, x1 = bbox(cs); fh, fw = y1 - y0 + 1, x1 - x0 + 1
    if fh > s or fw > s: return None
    rel = [(y - y0, x - x0) for y, x in cs]; found = set()
    for oy in range(s - fh + 1):
        for ox in range(s - fw + 1):
            P = set()
            for y, x in rel: P |= _d4(y + oy, x + ox, s)
            by, bx = y0 - oy, x0 - ox; ok = True
            for py, px in P:
                gy, gx = by + py, bx + px
                if 0 <= gy < h and 0 <= gx < w and (gy, gx) not in cs: ok = False; break
            if ok:
                # every visible fragment cell of the window must be in P (true by construction)
                found.add(frozenset(P))
                if len(found) > 1: return None
    return next(iter(found)) if found else None


def _nest_fragments(g):
    bg = bg_of(g); cells = _cells_by_colour(g, bg); n = len(cells)
    if n < 2 or n > 8: return None
    cells = {c: set(v) for c, v in cells.items()}
    vis = {}
    for c, v in cells.items():
        y0, x0, y1, x1 = bbox(v); vis[c] = max(y1 - y0 + 1, x1 - x0 + 1)
    lo = max(max(vis.values()), 2 * n - 1)
    for S in range(lo, min(30, lo + 6) + 1):
        sizes = [S - 2 * k for k in range(n)]
        if sizes[-1] < 1: continue
        feas = {c: {s: _complete(g, v, s) for s in sizes if s >= vis[c]} for c, v in cells.items()}
        feas = {c: {s: P for s, P in d.items() if P is not None} for c, d in feas.items()}
        # unique perfect matching colours -> sizes
        sols = []
        def rec(i, used, cur, cs=list(cells)):
            if len(sols) > 1: return
            if i == len(cs): sols.append(dict(cur)); return
            for s_, P in feas[cs[i]].items():
                if s_ not in used:
                    cur[cs[i]] = (s_, P); rec(i + 1, used | {s_}, cur); del cur[cs[i]]
        rec(0, frozenset(), {})
        if len(sols) == 1:
            o = [[bg] * S for _ in range(S)]
            for c, (s_, P) in sols[0].items():
                d = (S - s_) // 2
                for y, x in P: o[y + d][x + d] = c
            return o
        if len(sols) > 1: return None
    return None


def fam_rings_fragments(train):
    for p in train:
        o = p["output"]
        if H(o) != W(o) or o != [list(r) for r in zip(*o)] or o != o[::-1]: return
    if all(_nest_fragments(p["input"]) == p["output"] for p in train):
        yield ("rings:nest-fragments[D4]", 5, _nest_fragments)


FAMILIES = (fam_rings_summary, fam_rings_frame, fam_rings_seed, fam_rings_fragments)


In [ ]:
%%writefile /kaggle/working/probe/fam_summarise_count_to_glyph.py
"""summarise.count_to_glyph: count N items of an induced class, render N marks into a canvas.

One parametrised primitive, induced by decoding the training outputs first:
  canvas : fixed shape (all outputs equal) | 1xN / Nx1 bar | NxN diagonal | input-shaped canvas with
           a bar centred on the middle row/column
  slots  : for a fixed canvas, the fill order is one of the generic orders (reading / snake /
           diagonal / checkerboard-reading, under the 8 dihedral frames) or, failing that, the
           nested painted sets learned from the outputs (usable only for counts seen in training)
  cbg    : canvas background (0, input bg, or an output colour)
  target : the per-pair counts (and run colours) read off the outputs
Only then the item selector is searched (all of it must reproduce every target):
  role   : colour c | all fg (one group) | most/least frequent fg | each fg colour (sequential pour,
           ordered by colour value asc/desc)
  unit   : cells | 4/8-components | solid kxk squares (k>=2) | components of size>=2 | singletons |
           rows / columns of the item bbox that contain background
  scope  : whole grid | interior of the bbox of the largest object of another colour
  paint  : item colour | induced constant;  offset (size canvases only): N or N+1
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox

# ------------------------------------------------------------------ slot orders
def _walks(hh, ww):
    read = [(y, x) for y in range(hh) for x in range(ww)]
    snake = [(y, x if y % 2 == 0 else ww - 1 - x) for y in range(hh) for x in range(ww)]
    diag = [(i, i) for i in range(min(hh, ww))]
    cheq = [(y, x) for y in range(hh) for x in range(ww) if (y + x) % 2 == 0]
    return {"read": read, "snake": snake, "diag": diag, "cheq": cheq}

def slot_orders(h, w):
    out = {}; seen = set()
    for tname, tr, hh, ww in (
            ("id", lambda y, x: (y, x), h, w), ("fh", lambda y, x: (y, w - 1 - x), h, w),
            ("fv", lambda y, x: (h - 1 - y, x), h, w), ("r180", lambda y, x: (h - 1 - y, w - 1 - x), h, w),
            ("T", lambda y, x: (x, y), w, h), ("Tfh", lambda y, x: (x, h - 1 - y), w, h),
            ("Tfv", lambda y, x: (w - 1 - x, y), w, h), ("Tr180", lambda y, x: (w - 1 - x, h - 1 - y), w, h)):
        for wname, walk in _walks(hh, ww).items():
            o = tuple(tr(y, x) for y, x in walk)
            if o in seen: continue
            seen.add(o); out[wname + "/" + tname] = list(o)
    return out

# ------------------------------------------------------------------ item selectors
def _comps(mask, diag):
    h, w = len(mask), len(mask[0]); seen = set(); out = []
    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
    for y in range(h):
        for x in range(w):
            if not mask[y][x] or (y, x) in seen: continue
            st = [(y, x)]; seen.add((y, x)); cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in nb:
                    p = (a + dy, b + dx)
                    if 0 <= p[0] < h and 0 <= p[1] < w and mask[p[0]][p[1]] and p not in seen:
                        seen.add(p); st.append(p)
            out.append(cs)
    return out

def _is_square(cs):
    y0, x0, y1, x1 = bbox(cs)
    return y1 - y0 == x1 - x0 and y1 > y0 and len(cs) == (y1 - y0 + 1) ** 2

UNITS = ("cells", "comp4", "comp8", "square", "big4", "single8", "bgrows", "bgcols")

def count_unit(g, c, unit, bg, region):
    h, w = H(g), W(g)
    inr = (lambda y, x: True) if region is None else (lambda y, x: region[0] < y < region[2] and region[1] < x < region[3])
    mask = [[g[y][x] == c and inr(y, x) for x in range(w)] for y in range(h)]
    if unit == "cells": return sum(map(sum, mask))
    if unit in ("bgrows", "bgcols"):
        cs = [(y, x) for y in range(h) for x in range(w) if mask[y][x]]
        if not cs: return 0
        y0, x0, y1, x1 = bbox(cs)
        if unit == "bgrows": return sum(1 for y in range(y0, y1 + 1) if any(g[y][x] == bg for x in range(x0, x1 + 1)))
        return sum(1 for x in range(x0, x1 + 1) if any(g[y][x] == bg for y in range(y0, y1 + 1)))
    comps = _comps(mask, unit in ("comp8", "single8"))
    if unit in ("comp4", "comp8"): return len(comps)
    if unit == "square": return sum(1 for cs in comps if _is_square(cs))
    if unit == "big4": return sum(1 for cs in comps if len(cs) > 1)
    if unit == "single8": return sum(1 for cs in comps if len(cs) == 1)

def _region(g, bg, scope):
    """None (whole grid) or (y0,x0,y1,x1, frame colour) of the largest single-colour 8-object."""
    if scope == "all": return None, None
    obs = objects(g, bg, True, True)
    if not obs: return False, None
    big = max(obs, key=len)
    if sum(1 for o in obs if len(o) == len(big)) > 1: return False, None
    return bbox(big), g[big[0][0]][big[0][1]]

def grid_bg(g, fbg):
    """induced background: the colour that is the background of every training input (if present), else per grid."""
    return fbg if fbg is not None and any(fbg in r for r in g) else bg_of(g)

def groups(g, role, unit, scope, fbg=None):
    """list of (colour, count) for the selected item colours; None if the selector is undefined."""
    bg = grid_bg(g, fbg)
    region, frame = _region(g, bg, scope)
    if region is False: return None
    cnt = Counter(v for r in g for v in r if v != bg and v != frame)
    if role[0] == "c":
        cols = [role[1]]
    elif role[0] in ("fg", "each+", "each-"):
        cols = sorted(cnt, reverse=role[0] == "each-")
    elif role[0] in ("max", "min"):
        if not cnt: return None
        mc = cnt.most_common()
        v = mc[0][1] if role[0] == "max" else mc[-1][1]
        cs = [c for c, n in mc if n == v]
        if len(cs) != 1: return None
        cols = cs
    if role[0] == "fg":
        n = sum(count_unit(g, c, unit, bg, region) for c in cols) if unit in ("cells", "square", "big4", "single8") \
            else None
        if n is None:  # multi-colour components / rows: count on the union mask
            gg = [[1 if (v != bg and v != frame) else 0 for v in r] for r in g]
            n = count_unit(gg, 1, unit, 0, region)
        return [(cols[0] if len(cols) == 1 else None, n)]
    return [(c, count_unit(g, c, unit, bg, region)) for c in cols]

# ------------------------------------------------------------------ canvas decoding
def runs_along(o, order, cbg):
    """painted cells of o must be exactly a prefix of `order`; returns run-length colour list."""
    painted = {(y, x) for y in range(H(o)) for x in range(W(o)) if o[y][x] != cbg}
    n = len(painted)
    if n > len(order) or set(order[:n]) != painted: return None
    runs = []
    for y, x in order[:n]:
        v = o[y][x]
        if runs and runs[-1][0] == v: runs[-1][1] += 1
        else: runs.append([v, 1])
    return [tuple(r) for r in runs]

def learned_order(outs, cbg):
    """nested painted sets -> (dict n -> set). None if not nested."""
    sets = {}
    for o in outs:
        s = frozenset((y, x) for y in range(H(o)) for x in range(W(o)) if o[y][x] != cbg)
        if len(s) in sets and sets[len(s)] != s: return None
        sets[len(s)] = s
    ks = sorted(sets)
    for a, b in zip(ks, ks[1:]):
        if not sets[a] <= sets[b]: return None
    return sets

def decode(train):
    """yield (canvas_name, targets, render) where targets[i] = list of (colour, n) runs and
    render(runs, g) -> grid.  A run colour of None means 'unspecified'."""
    ins = [p["input"] for p in train]; outs = [p["output"] for p in train]
    shapes = {(H(o), W(o)) for o in outs}
    # (a) fixed canvas
    if len(shapes) == 1:
        h, w = shapes.pop()
        if h * w <= 36:
            cands = []
            for c in [0] + sorted({v for o in outs for v in (x for r in o for x in r)}):
                if c not in cands: cands.append(c)
            orders = slot_orders(h, w)
            for cbg in cands:
                got = 0
                for oname, order in orders.items():
                    tg = [runs_along(o, order, cbg) for o in outs]
                    if any(t is None for t in tg): continue
                    if all(len(t) == 0 for t in tg): continue
                    def render(runs, g, order=order, cbg=cbg, h=h, w=w):
                        out = [[cbg] * w for _ in range(h)]; k = 0
                        for col, n in runs:
                            if k + n > len(order): return None
                            for y, x in order[k:k + n]: out[y][x] = col
                            k += n
                        return out
                    yield ("fixed%dx%d:%s:bg%d" % (h, w, oname, cbg), tg, render)
                    got += 1
                    if got >= 2: break
                if got: continue
                sets = learned_order(outs, cbg)
                if sets and len(sets) >= 2:
                    tg = []
                    for o in outs:
                        cs = {o[y][x] for y in range(H(o)) for x in range(W(o)) if o[y][x] != cbg}
                        n = sum(1 for r in o for v in r if v != cbg)
                        if len(cs) > 1: tg = None; break
                        tg.append([(cs.pop(), n)] if n else [])
                    if tg is None: continue
                    def render(runs, g, sets=sets, cbg=cbg, h=h, w=w):
                        if len(runs) > 1: return None
                        n = runs[0][1] if runs else 0
                        if n not in sets: return None
                        out = [[cbg] * w for _ in range(h)]
                        for y, x in sets[n]: out[y][x] = runs[0][0]
                        return out
                    yield ("learned%dx%d:bg%d" % (h, w, cbg), tg, render)
        return
    # (b) uniform bar 1xN / Nx1
    for ax in ("row", "col"):
        if all((H(o) == 1 if ax == "row" else W(o) == 1) and len({v for r in o for v in r}) == 1 for o in outs):
            tg = [[(o[0][0], W(o) if ax == "row" else H(o))] for o in outs]
            def render(runs, g, ax=ax):
                if len(runs) != 1 or runs[0][1] < 1: return None
                c, n = runs[0]
                return [[c] * n] if ax == "row" else [[c] for _ in range(n)]
            yield ("bar-" + ax, tg, render)
            return
    # (c) NxN with diagonal painted
    if all(H(o) == W(o) for o in outs):
        for anti in (False, True):
            tg = []
            for o in outs:
                n = H(o); d = {(i, n - 1 - i if anti else i) for i in range(n)}
                dc = {o[y][x] for y, x in d}
                rest = {o[y][x] for y in range(n) for x in range(n) if (y, x) not in d}
                if len(dc) != 1 or len(rest) > 1 or dc == rest: tg = None; break
                tg.append((dc.pop(), n, rest.pop() if rest else None))
            if tg is None: continue
            cb = {t[2] for t in tg if t[2] is not None}
            if len(cb) != 1: continue
            cbg = cb.pop()
            def render(runs, g, anti=anti, cbg=cbg):
                if len(runs) != 1 or runs[0][1] < 1: return None
                c, n = runs[0]; out = [[cbg] * n for _ in range(n)]
                for i in range(n): out[i][n - 1 - i if anti else i] = c
                return out
            yield ("diag" + ("-anti" if anti else ""), [[(c, n)] for c, n, _ in tg], render)
            return
    # (d) input-shaped canvas, bar centred on the middle row / column
    if all((H(i), W(i)) == (H(o), W(o)) for i, o in zip(ins, outs)):
        for ax in ("row", "col"):
            tg = []
            for i, o in zip(ins, outs):
                bg = bg_of(i); h, w = H(o), W(o)
                cells = [(y, x) for y in range(h) for x in range(w) if o[y][x] != bg]
                cs = {o[y][x] for y, x in cells}
                if len(cs) != 1: tg = None; break
                n = len(cells); L = w if ax == "row" else h
                if ax == "row": want = {(h // 2, x) for x in range((L - n) // 2, (L - n) // 2 + n)}
                else: want = {(y, w // 2) for y in range((L - n) // 2, (L - n) // 2 + n)}
                if set(cells) != want: tg = None; break
                tg.append([(cs.pop(), n)])
            if tg is None: continue
            def render(runs, g, ax=ax):
                if len(runs) != 1: return None
                c, n = runs[0]; bg = bg_of(g); h, w = H(g), W(g)
                L = w if ax == "row" else h
                if n > L: return None
                out = [[bg] * w for _ in range(h)]
                for k in range((L - n) // 2, (L - n) // 2 + n):
                    if ax == "row": out[h // 2][k] = c
                    else: out[k][w // 2] = c
                return out
            yield ("centred-" + ax, tg, render)

# ------------------------------------------------------------------ the family
def fam_count_to_glyph(train):
    ins = [p["input"] for p in train]
    if len(train) < 2: return
    decs = list(decode(train))
    if not decs: return
    bgs = {bg_of(i) for i in ins}
    fbg = bgs.pop() if len(bgs) == 1 else None
    fgcols = sorted({v for i in ins for r in i for v in r} - {bg_of(i) for i in ins})
    roles = [("fg",), ("max",), ("min",)] + [("c", c) for c in fgcols]
    multi_roles = [("each-",), ("each+",)]
    cache = {}
    def G(k, role, unit, scope):
        key = (k, role, unit, scope)
        if key not in cache: cache[key] = groups(ins[k], role, unit, scope, fbg)
        return cache[key]
    n_yield = 0
    for cname, tg, render in decs:
        single = all(len(t) <= 1 for t in tg)
        offsets = (0, 1) if cname.startswith("bar") else (0,)
        if single:
            tcol = {t[0][0] for t in tg if t}
            for scope in ("all", "inbox"):
                for unit in UNITS:
                    for role in roles:
                        for off in offsets:
                            ok_item = True; ok = True
                            for k, t in enumerate(tg):
                                gs = G(k, role, unit, scope)
                                if gs is None: ok = False; break
                                col, n = gs[0]
                                tn = t[0][1] if t else 0
                                if n + off != tn: ok = False; break
                                if t and col != t[0][0]: ok_item = False
                            if not ok: continue
                            paints = []
                            if ok_item and any(tg): paints.append(("item", None))
                            if len(tcol) == 1: paints.append(("const", next(iter(tcol))))
                            for pmode, pc in paints:
                                def fn(g, role=role, unit=unit, scope=scope, off=off, pmode=pmode, pc=pc, render=render, fbg=fbg):
                                    gs = groups(g, role, unit, scope, fbg)
                                    if gs is None: return None
                                    col, n = gs[0]; n += off
                                    if pmode == "item" and col is None and n: return None
                                    c = col if pmode == "item" else pc
                                    return render([(c, n)] if n else [], g)
                                rn = role[0] + (str(role[1]) if len(role) > 1 else "")
                                yield ("count_to_glyph:%s[%s,%s,%s,+%d,%s]" % (cname, rn, unit, scope, off, pmode), 4, fn)
                                n_yield += 1
                                if n_yield >= 12: return
        else:
            for scope in ("all", "inbox"):
                for unit in UNITS:
                    for role in multi_roles:
                        ok = True
                        for k, t in enumerate(tg):
                            gs = G(k, role, unit, scope)
                            if gs is None or [(c, n) for c, n in gs if n] != list(t): ok = False; break
                        if not ok: continue
                        def fn(g, role=role, unit=unit, scope=scope, render=render, fbg=fbg):
                            gs = groups(g, role, unit, scope, fbg)
                            if gs is None: return None
                            return render([(c, n) for c, n in gs if n], g)
                        yield ("count_to_glyph:%s[%s,%s,%s,pour]" % (cname, role[0], unit, scope), 5, fn)
                        n_yield += 1
                        if n_yield >= 12: return

# ------------------------------------------------------------------ legend-box canvas
def find_legend(g):
    """a solid single-colour rectangle (colour != grid bg) whose interior holds isolated single-cell
    glyphs, one per row (or per column).  Returns (y0,x0,y1,x1, boxcol, [(y,x,colour)...]) or None."""
    bg = bg_of(g); h, w = H(g), W(g); found = []
    for c in sorted({v for r in g for v in r} - {bg}):
        cs = [(y, x) for y in range(h) for x in range(w) if g[y][x] == c]
        y0, x0, y1, x1 = bbox(cs)
        if (y1 - y0 + 1) * (x1 - x0 + 1) < 9: continue
        glyphs = []; ok = True
        for y in range(y0, y1 + 1):
            for x in range(x0, x1 + 1):
                v = g[y][x]
                if v == c: continue
                if not (y0 < y < y1 and x0 < x < x1): ok = False; break
                if any(g[y + dy][x + dx] != c for dy in (-1, 0, 1) for dx in (-1, 0, 1) if dy or dx): ok = False; break
                glyphs.append((y, x, v))
            if not ok: break
        if ok and len(glyphs) >= 1 and len({v for _, _, v in glyphs}) == len(glyphs):
            found.append((y0, x0, y1, x1, c, glyphs))
    return found[0] if len(found) == 1 else None

def _legend_render(g, counts, stride, axis):
    L = find_legend(g)
    if L is None: return None
    y0, x0, y1, x1, c, glyphs = L
    out = [r[x0:x1 + 1] for r in g[y0:y1 + 1]]
    for y, x, v in glyphs:
        n = counts.get(v)
        if n is None: return None
        for k in range(max(n, 0)):
            yy, xx = (y - y0, x - x0 + k * stride) if axis == "row" else (y - y0 + k * stride, x - x0)
            if not (0 <= yy < len(out) and 0 <= xx < len(out[0])): return None
            if k and out[yy][xx] != c: return None
            out[yy][xx] = v if n else c
    return out

def legend_counts(g, unit, scope):
    """count, for each legend colour, its items (unit) outside (or everywhere) the legend box."""
    L = find_legend(g)
    if L is None: return None
    y0, x0, y1, x1, c, glyphs = L
    gg = [list(r) for r in g]
    if scope == "outside":
        for y in range(y0, y1 + 1):
            for x in range(x0, x1 + 1): gg[y][x] = c
    return {v: count_unit(gg, v, unit, bg_of(g), None) for _, _, v in glyphs}

def fam_legend_count(train):
    """canvas = the legend box itself; every glyph in it is repeated N times along its row/column
    (stride induced), N = number of items of the glyph's colour elsewhere in the grid."""
    if len(train) < 2: return
    ins = [p["input"] for p in train]; outs = [p["output"] for p in train]
    Ls = [find_legend(g) for g in ins]
    if any(L is None for L in Ls): return
    if any((L[2] - L[0] + 1, L[3] - L[1] + 1) != (H(o), W(o)) for L, o in zip(Ls, outs)): return
    n = 0
    for unit in ("comp8", "comp4", "cells", "big4", "single8"):
        for scope in ("outside", "all"):
            for axis in ("row", "col"):
                for stride in (2, 1):
                    def fn(g, unit=unit, scope=scope, axis=axis, stride=stride):
                        cnt = legend_counts(g, unit, scope)
                        return None if cnt is None else _legend_render(g, cnt, stride, axis)
                    if all(fn(i) == o for i, o in zip(ins, outs)):
                        yield ("count_to_glyph:legend[%s,%s,%s,s%d]" % (unit, scope, axis, stride), 5, fn)
                        n += 1
                        if n >= 2: return

FAMILIES = (fam_count_to_glyph, fam_legend_count)


In [ ]:
%%writefile /kaggle/working/probe/fam_summarise_panel_to_pixel.py
"""summarise.panel_to_pixel: the grid is partitioned into panels; the output has one pixel per panel.

One parametrised primitive  out[i][j] = reduce(P[i][j])  followed by an optional layout post-step.
All parameters are induced from the training pairs (programs are pre-checked on every pair):
  partition : sep      - full uniform lines of one colour s split the grid into bands
                         (options: strip = peel uniform border layers first,
                                   seps  = separator runs are themselves output bands)
              lattice  - like sep, but the lines may carry marks (a line is a separator when it holds
                         no panel-background cell and is mostly s); panels know their 4 lattice corners
              blocks   - R x C equal blocks, (R,C) = constant output shape or input/k;
                         optional pre-crop to the bbox of the largest object
              layout   - solid rectangles (large, dense components) clustered into rows/columns
  reduce    : major | major_ex (majority ignoring bg and s) | second[fill] (most frequent colour other
              than the panel majority; fill = panel majority or global bg when absent) |
              uniform (active only when the panel is one colour) |
              corners (active when all 4 lattice corners are marked; colour when they agree, else bg) |
              table:<feature> (count feature -> colour, learnt from the pairs; threshold for unseen)
  post      : plain | crop (bbox of active panels) | frame (1-cell ring of s / bg around the result)
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, bbox, crop, fit_cmap, apply_cmap


# ------------------------------------------------------------------ partitions
# A partition is (matrix of panels, s) where a panel is dict(cells=[values], corners=[values|None]).

def _runs(idx_is_sep, n, with_seps):
    bands, cur, kind = [], [], None
    for i in range(n):
        k = idx_is_sep[i]
        if cur and k != kind:
            bands.append((kind, cur)); cur = []
        cur.append(i); kind = k
    if cur: bands.append((kind, cur))
    return [b for k, b in bands if with_seps or not k]


def _strip(g):
    r0, r1, c0, c1 = 0, H(g) - 1, 0, W(g) - 1
    changed = True
    while changed and r1 - r0 >= 2 and c1 - c0 >= 2:
        changed = False
        for side in range(4):
            if side == 0: vals = g[r0][c0:c1 + 1]
            elif side == 1: vals = g[r1][c0:c1 + 1]
            elif side == 2: vals = [g[r][c0] for r in range(r0, r1 + 1)]
            else: vals = [g[r][c1] for r in range(r0, r1 + 1)]
            if len(set(vals)) == 1:
                if side == 0: r0 += 1
                elif side == 1: r1 -= 1
                elif side == 2: c0 += 1
                else: c1 -= 1
                changed = True
                break
    if (r0, c0) == (0, 0) and (r1, c1) == (H(g) - 1, W(g) - 1): return None
    return crop(g, (r0, c0, r1, c1))


def _matrix(g, rb, cb, rsep, csep):
    h, w = H(g), W(g)
    M = []
    for rows in rb:
        line = []
        for cols in cb:
            cells = [g[r][c] for r in rows for c in cols]
            ys = (rows[0] - 1, rows[-1] + 1); xs = (cols[0] - 1, cols[-1] + 1)
            corners = [g[y][x] if 0 <= y < h and 0 <= x < w and rsep[y] and csep[x] else None
                       for y in ys for x in xs]
            line.append({"cells": cells, "corners": corners})
        M.append(line)
    return M


def part_sep(g, strip, with_seps, loose):
    gb0 = bg_of(g)
    if strip:
        g = _strip(g)
        if g is None: return None
    h, w = H(g), W(g)
    if h < 3 or w < 3: return None
    uni = Counter()
    for r in range(h):
        if len(set(g[r])) == 1: uni[g[r][0]] += 1
    for c in range(w):
        col = {g[r][c] for r in range(h)}
        if len(col) == 1: uni[g[0][c]] += 1
    if len(uni) != 1: return None
    s = next(iter(uni))
    if loose:
        rest = Counter(v for row in g for v in row if v != s)
        if not rest: return None
        pb = rest.most_common(1)[0][0]
        rsep = [pb not in g[r] and 2 * g[r].count(s) >= w for r in range(h)]
        colv = [[g[r][c] for r in range(h)] for c in range(w)]
        csep = [pb not in colv[c] and 2 * colv[c].count(s) >= h for c in range(w)]
    else:
        rsep = [len(set(g[r])) == 1 and g[r][0] == s for r in range(h)]
        csep = [all(g[r][c] == s for r in range(h)) for c in range(w)]
    if all(rsep) or all(csep): return None
    rb = _runs(rsep, h, with_seps); cb = _runs(csep, w, with_seps)
    if len(rb) * len(cb) < 2: return None
    return _matrix(g, rb, cb, rsep, csep), s, gb0


def part_blocks(g, R, C, precrop):
    gb = bg_of(g)
    if precrop:
        gb = g[0][0]
        if any(g[0][0] != v for v in (g[0][-1], g[-1][0], g[-1][-1])): return None
        cells = [(r, c) for r in range(H(g)) for c in range(W(g)) if g[r][c] != gb]
        if not cells: return None
        g = crop(g, bbox(cells))
    h, w = H(g), W(g)
    if R * C < 2 or h < 2 * R or w < 2 * C: return None   # >=2 blocks, each at least 2x2
    rb = [list(range(i * h // R, (i + 1) * h // R)) for i in range(R)]
    cb = [list(range(j * w // C, (j + 1) * w // C)) for j in range(C)]
    return _matrix(g, rb, cb, [False] * h, [False] * w), None, gb


def part_layout(g):
    bg = bg_of(g)
    objs = objects(g, bg, diag=False, by_colour=True)
    rects = []
    for o in objs:
        y0, x0, y1, x1 = bbox(o)
        if y1 - y0 < 1 or x1 - x0 < 1: continue
        if len(o) < 0.7 * (y1 - y0 + 1) * (x1 - x0 + 1): continue
        rects.append((len(o), y0, x0, y1, x1, g[o[0][0]][o[0][1]]))
    if len(rects) < 2: return None
    big = max(r[0] for r in rects)
    rects = [r for r in rects if r[0] >= 0.25 * big and r[0] >= 4]
    if len(rects) < 2: return None
    rects.sort(key=lambda r: r[1])
    rows = []
    for r in rects:
        if rows and r[1] <= max(q[3] for q in rows[-1]):
            rows[-1].append(r)
        else:
            rows.append([r])
    n = len(rows[0])
    if any(len(rw) != n for rw in rows): return None
    M = [[{"cells": [r[5]], "corners": [None] * 4} for r in sorted(rw, key=lambda q: q[2])] for rw in rows]
    return M, None, bg


# ------------------------------------------------------------------ reduces: panel -> (colour, active)

def _major(cells):
    return Counter(cells).most_common(1)[0][0]


def red_major(p, gb, s):
    return _major(p["cells"]), True


def red_major_ex(p, gb, s):
    cnt = Counter(v for v in p["cells"] if v != gb and v != s)
    return (cnt.most_common(1)[0][0] if cnt else gb), True


def _second(p, fill_gb, gb):
    cnt = Counter(p["cells"]).most_common()
    if len(cnt) >= 2:
        if len(cnt) >= 3 and cnt[1][1] == cnt[2][1]: return None, True
        return cnt[1][0], True
    return (gb if fill_gb else cnt[0][0]), True


def red_second_pb(p, gb, s): return _second(p, False, gb)
def red_second_gb(p, gb, s): return _second(p, True, gb)


def red_uniform(p, gb, s):
    st = set(p["cells"])
    return (next(iter(st)), True) if len(st) == 1 else (gb, False)


def red_corners(p, gb, s):
    cs = p["corners"]
    pb = _major(p["cells"])
    if any(c is None or c == s for c in cs): return pb, False
    return (cs[0] if len(set(cs)) == 1 else pb), True


REDUCES = {"major": red_major, "major_ex": red_major_ex, "second[fill=panel]": red_second_pb,
           "second[fill=bg]": red_second_gb, "uniform": red_uniform, "corners": red_corners}

FEATS = {"n_fg": lambda p, gb: sum(v != gb for v in p["cells"]),
         "n_minor": lambda p, gb: len(p["cells"]) - Counter(p["cells"]).most_common(1)[0][1],
         "n_colours": lambda p, gb: len(set(p["cells"]))}


# ------------------------------------------------------------------ post

def assemble(M, post, s, gb):
    """M: matrix of (colour, active). Returns output grid or None."""
    if post == "crop":
        act = [(i, j) for i, row in enumerate(M) for j, (c, a) in enumerate(row) if a]
        if not act: return None
        i0 = min(i for i, _ in act); i1 = max(i for i, _ in act)
        j0 = min(j for _, j in act); j1 = max(j for _, j in act)
        M = [row[j0:j1 + 1] for row in M[i0:i1 + 1]]
    out = [[c for c, a in row] for row in M]
    if any(c is None for row in out for c in row): return None
    if post == "frame":
        f = s if s is not None else gb
        w = len(out[0]) + 2
        out = [[f] * w] + [[f] + row + [f] for row in out] + [[f] * w]
    return out


# ------------------------------------------------------------------ family

def _partitioners(train):
    P = []
    for strip in (False, True):
        for ws in (False, True):
            P.append((f"sep[strip={int(strip)},seps={int(ws)}]", 0 + strip + ws,
                      lambda g, a=strip, b=ws: part_sep(g, a, b, False)))
    P.append(("lattice", 1, lambda g: part_sep(g, False, False, True)))
    shapes = {(H(p["output"]), W(p["output"])) for p in train}
    ks = set()
    for pc in (False, True):
        if len(shapes) == 1:
            R, C = next(iter(shapes))
            P.append((f"blocks[{R}x{C},crop={int(pc)}]", 1 + pc, lambda g, R=R, C=C, pc=pc: part_blocks(g, R, C, pc)))
    kk = {(H(p["input"]) / H(p["output"]), W(p["input"]) / W(p["output"])) for p in train}
    if len(kk) == 1:
        a, b = next(iter(kk))
        if a == int(a) and b == int(b) and a * b > 1:
            a, b = int(a), int(b)
            P.append((f"blocks[/{a}x{b}]", 1, lambda g, a=a, b=b: part_blocks(g, H(g) // a, W(g) // b, False)
                      if H(g) % a == 0 and W(g) % b == 0 else None))
    P.append(("layout", 2, part_layout))
    return P


def _fits(preds, outs):
    if any(p is None or H(p) != H(o) or W(p) != W(o) for p, o in zip(preds, outs)): return False
    if preds == outs: return True
    m = fit_cmap(preds, outs)
    return bool(m) and all(apply_cmap(a, m) == b for a, b in zip(preds, outs))


def _table_fn(table):
    keys = sorted(table)
    vals = [table[k] for k in keys]
    thr = None
    if len(set(vals)) == 2:
        cut = [i for i in range(1, len(vals)) if vals[i] != vals[i - 1]]
        if len(cut) == 1: thr = (keys[cut[0] - 1], keys[cut[0]], vals[0], vals[-1])

    def f(v):
        if v in table: return table[v]
        if thr: return thr[2] if v <= thr[0] else thr[3] if v >= thr[1] else None
        return None
    return f


def fam_panel_to_pixel(train):
    """Exact fits shadow fits that need the harness colour map (those extrapolate worse)."""
    outs = [p["output"] for p in train]
    progs = list(_programs(train))
    exact = [x for x in progs if [x[2](p["input"]) for p in train] == outs]
    for x in (exact or progs)[:40]:
        yield x


def _programs(train):
    outs = [p["output"] for p in train]
    ins = [p["input"] for p in train]
    if any(H(o) * W(o) >= H(i) * W(i) for i, o in zip(ins, outs)): return
    if max(H(o) * W(o) for o in outs) > 100: return
    n = 0
    for pname, pcost, part in _partitioners(train):
        parts = []
        for g in ins:
            r = part(g)
            if r is None: break
            parts.append(r)
        if len(parts) != len(ins): continue
        for post in ("plain", "crop", "frame"):
            # quick shape pre-check for plain/frame
            if post != "crop":
                d = 2 if post == "frame" else 0
                if any((len(M) + d, len(M[0]) + d) != (H(o), W(o)) for (M, s, gb), o in zip(parts, outs)): continue
            for rname, red in REDUCES.items():
                if post == "crop" and rname not in ("uniform", "corners"): continue
                if rname == "corners" and not pname.startswith("lattice"): continue
                preds = [assemble([[red(p, gb, s) for p in row] for row in M], post, s, gb)
                         for M, s, gb in parts]
                if not _fits(preds, outs): continue

                def fn(g, part=part, red=red, post=post):
                    r = part(g)
                    if r is None: return None
                    M, s, gb = r
                    return assemble([[red(p, gb, s) for p in row] for row in M], post, s, gb)
                n += 1
                yield (f"panel2px:{pname}:{rname}:{post}", min(6, 3 + pcost + (post != 'plain')), fn)
            if post == "crop": continue
            d = 1 if post == "frame" else 0
            for fname, feat in (FEATS.items() if pname.startswith(("sep", "lattice")) else ()):
                table = {}; ok = True
                for (M, s, gb), o in zip(parts, outs):
                    for i, row in enumerate(M):
                        for j, p in enumerate(row):
                            v = feat(p, gb); c = o[i + d][j + d]
                            if table.setdefault(v, c) != c: ok = False; break
                        if not ok: break
                    if not ok: break
                if not ok or len(set(table.values())) < 2: continue
                tf = _table_fn(table)
                if post == "frame":
                    f0 = {o[0][0] for o in outs}
                    if len(f0) != 1: continue
                    fc = f0.pop()
                    if any(set(o[0]) != {fc} or set(o[-1]) != {fc} or {r[0] for r in o} != {fc} or {r[-1] for r in o} != {fc}
                           for o in outs): continue
                else:
                    fc = None

                def fn(g, part=part, feat=feat, tf=tf, post=post, fc=fc):
                    r = part(g)
                    if r is None: return None
                    M, s, gb = r
                    out = [[tf(feat(p, gb)) for p in row] for row in M]
                    if any(c is None for row in out for c in row): return None
                    if post == "frame":
                        w = len(out[0]) + 2
                        out = [[fc] * w] + [[fc] + row + [fc] for row in out] + [[fc] * w]
                    return out
                if not _fits([fn(g) for g in ins], outs): continue
                n += 1
                yield (f"panel2px:{pname}:table[{fname}]:{post}", min(6, 4 + pcost + (post != 'plain')), fn)


FAMILIES = (fam_panel_to_pixel,)


In [ ]:
%%writefile /kaggle/working/probe/fam_summarise_ranked_bars.py
"""summarise.ranked_bars: colour classes are ranked by an induced key and rendered as bars.

One parametrised primitive.  Parameters, all induced from the training pairs by decoding the
outputs (never literal task constants):
  filter  : which colour classes are items   (all | no-grid: drop colours owning a full row/col |
                                               straight: only colours lying in one row or column)
  select  : all | drop-first (after sorting) | keep-max(f) (only items with the largest feature f)
  key/dir : ordering (numeric feature asc/desc, position incl. auto axis, occlusion layering)
  length  : bar length (a numeric feature | 1 | full canvas width)
  mode    : bars (one row per item) | wrap (item starts a row, wraps at width) | pour (continuous)
  canvas  : rows / cols from {#items, max length, fixed size, input H, input W}
  frame   : one of the 8 dihedral maps (bar axis, alignment side, order direction)
Zero-length bars are not drawn (their items vanish), which is how marker colours drop out.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from gdsl import H, W, bg_of, objects, D8

INV = {"id": "id", "r90": "r270", "r270": "r90", "r180": "r180", "fh": "fh", "fv": "fv", "T": "T", "aT": "aT"}

def _compose_T():
    """name of t∘T for each t (used when the ordering axis is x: canonical grid is transposed first)."""
    probe = [[1, 2, 3], [4, 5, 6]]
    imgs = {k: f(probe) for k, f in D8.items()}
    out = {}
    for k, f in D8.items():
        r = f(D8["T"](probe))
        out[k] = next(n for n, im in imgs.items() if im == r)
    return out
TCOMP = _compose_T()

NUM = ("ncells", "nobj4", "nobj8", "height", "width", "maxobj", "holes", "enclosed")
POS = ("miny", "minx", "cy", "cx", "first", "auto")
KEYS = NUM + ("value",) + POS + ("layer",)
FILTERS = ("all", "no-grid", "straight")

# ------------------------------------------------------------------ item features
def _enclosed(g, c, bg):
    """cells not of colour c that cannot reach the grid border without crossing colour c."""
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; st = []
    for y in range(h):
        for x in range(w):
            if (y in (0, h - 1) or x in (0, w - 1)) and g[y][x] != c:
                seen[y][x] = True; st.append((y, x))
    while st:
        y, x = st.pop()
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != c:
                seen[yy][xx] = True; st.append((yy, xx))
    hb = sum(1 for y in range(h) for x in range(w) if not seen[y][x] and g[y][x] == bg)
    ho = sum(1 for y in range(h) for x in range(w) if not seen[y][x] and g[y][x] not in (bg, c))
    return hb, ho

def _layers(g, bg, cells):
    """Bottom-to-top occlusion order of colour classes, or None if not uniquely determined."""
    above = {c: set() for c in cells}
    for c, cs in cells.items():
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        r0, r1, c0, c1 = min(ys), max(ys), min(xs), max(xs)
        if all(y in (r0, r1) or x in (c0, c1) for y, x in cs):
            exp = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if y in (r0, r1) or x in (c0, c1)]
        elif r1 - r0 == c1 - c0 and all(y - x == r0 - c0 for y, x in cs):
            exp = [(r0 + k, c0 + k) for k in range(r1 - r0 + 1)]
        elif r1 - r0 == c1 - c0 and all(y + x == r0 + c1 for y, x in cs):
            exp = [(r0 + k, c1 - k) for k in range(r1 - r0 + 1)]
        else:
            exp = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)]
        for y, x in exp:
            d = g[y][x]
            if d != c and d != bg and d in above: above[c].add(d)
    # Kahn: bottom = colour with nothing below it
    below = {c: {d for d in cells if c in above[d]} for c in cells}
    order, left = [], set(cells)
    while left:
        free = [c for c in left if not (below[c] & left)]
        if len(free) != 1: return None
        order.append(free[0]); left.discard(free[0])
    return order

def features(g, bgspec="mode"):
    bg = bg_of(g) if bgspec == "mode" else -1 if bgspec == "none" else bgspec
    h, w = H(g), W(g)
    cells = {}
    for y in range(h):
        for x in range(w):
            if g[y][x] != bg: cells.setdefault(g[y][x], []).append((y, x))
    o4 = Counter(g[ob[0][0]][ob[0][1]] for ob in objects(g, bg, False, True))
    obs8 = objects(g, bg, True, True)
    o8 = Counter(g[ob[0][0]][ob[0][1]] for ob in obs8)
    mo = {}
    for ob in obs8:
        c = g[ob[0][0]][ob[0][1]]; mo[c] = max(mo.get(c, 0), len(ob))
    F = {}
    for c, cs in cells.items():
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        hb, ho = _enclosed(g, c, bg)
        full = any(all(g[y][x] == c for x in range(w)) for y in set(ys)) or any(all(g[y][x] == c for y in range(h)) for x in set(xs))
        cy = sum(ys) / len(ys); cx = sum(xs) / len(xs)
        F[c] = {"ncells": len(cs), "nobj4": o4[c], "nobj8": o8[c], "height": max(ys) - min(ys) + 1,
                "width": max(xs) - min(xs) + 1, "maxobj": mo[c], "holes": hb, "enclosed": ho,
                "value": c, "miny": min(ys), "minx": min(xs), "cy": cy, "cx": cx,
                "first": min(cs), "grid": full, "straight": len(set(ys)) == 1 or len(set(xs)) == 1}
    return {"bg": bg, "F": F, "cells": cells, "h": h, "w": w, "g": g}

def order_items(info, flt, key, desc, sel):
    """Returns (ordered colours, transposed?) or None."""
    F = info["F"]
    items = [c for c in F if flt == "all" or (flt == "no-grid" and not F[c]["grid"]) or (flt == "straight" and F[c]["straight"])]
    if not items: return None
    tr = False
    if key == "layer":
        lay = _layers_cached(info, tuple(sorted(items)))
        if lay is None: return None
        seq = lay[::-1] if desc else lay
    else:
        k = key
        if key == "auto":
            sy = max(F[c]["cy"] for c in items) - min(F[c]["cy"] for c in items)
            sx = max(F[c]["cx"] for c in items) - min(F[c]["cx"] for c in items)
            if sx == sy: return None
            tr = sx > sy; k = "cx" if tr else "cy"
        vals = [F[c][k] for c in items]
        if len(set(vals)) != len(vals): return None      # ambiguous ranking
        seq = sorted(items, key=lambda c: F[c][k], reverse=desc)
    if sel == "drop-first":
        seq = seq[1:]
    elif sel.startswith("max:"):
        f = sel[4:]; m = max(F[c][f] for c in seq); seq = [c for c in seq if F[c][f] == m]
    return seq, tr

def _layers_cached(info, items):
    ck = ("layer", items)
    if ck not in info:
        info[ck] = _layers(info["g"], info["bg"], {c: info["cells"][c] for c in items})
    return info[ck]

# ------------------------------------------------------------------ decode / render
def decode(C, bg, mode):
    """Canonical grid -> list of (colour, length) or None."""
    h, w = H(C), W(C)
    if mode == "pour":
        flat = [v for r in C for v in r]
        while flat and flat[-1] == bg: flat.pop()
        seq = []
        for v in flat:
            if v == bg: return None
            if seq and seq[-1][0] == v: seq[-1][1] += 1
            else: seq.append([v, 1])
        return [tuple(s) for s in seq]
    rows = []
    for r in C:
        L = 0
        while L < w and r[L] != bg: L += 1
        if L and any(v != r[0] for v in r[:L]): return None
        if any(v != bg for v in r[L:]): return None
        rows.append((r[0] if L else None, L))
    while rows and rows[-1][1] == 0: rows.pop()
    if any(L == 0 for _, L in rows): return None
    if mode == "bars": return rows
    seq = []
    for i, (c, L) in enumerate(rows):
        if seq and seq[-1][0] == c and rows[i - 1][1] == w: seq[-1][1] += L
        else: seq.append([c, L])
    return [tuple(s) for s in seq]

def render(seq, R, Cn, bg, mode):
    if R <= 0 or Cn <= 0 or R > 30 or Cn > 30: return None
    out = [[bg] * Cn for _ in range(R)]
    if mode == "bars":
        if len(seq) > R: return None
        for i, (c, L) in enumerate(seq):
            if L > Cn: return None
            for x in range(L): out[i][x] = c
    elif mode == "wrap":
        y = 0
        for c, L in seq:
            while L > 0:
                if y >= R: return None
                k = min(L, Cn)
                for x in range(k): out[y][x] = c
                L -= k; y += 1
    else:
        pos = 0
        for c, L in seq:
            for _ in range(L):
                if pos >= R * Cn: return None
                out[pos // Cn][pos % Cn] = c; pos += 1
    return out

def dim(rule, g, n, lens):
    if rule == "n": return n
    if rule == "maxlen": return max(lens) if lens else 0
    if rule == "inH": return H(g)
    if rule == "inW": return W(g)
    return rule  # fixed int

def program(bgs, flt, key, desc, sel, length, mode, rrule, crule, t):
    def fn(g):
        info = features(g, bgs)
        r = order_items(info, flt, key, desc, sel)
        if r is None: return None
        seq, tr = r
        F = info["F"]
        n = len(seq)
        if length == "full":
            Cn = dim(crule, g, n, [])
            lens = [Cn] * n
        else:
            lens = [1 if length == "one" else F[c][length] for c in seq]
            Cn = dim(crule, g, n, lens)
        items = [(c, L) for c, L in zip(seq, lens) if L > 0]
        if not items: return None
        R = dim(rrule, g, len(items), lens)
        out = render(items, R, Cn, info["bg"], mode)
        if out is None: return None
        tt = TCOMP[t] if tr else t
        return D8[tt](out)
    return fn

# ------------------------------------------------------------------ induction
def _score(sig):
    bgs, mode, t, flt, key, desc, sel, length, rr, cr = sig
    s = (sel != "all") + (flt != "all") + (bgs != "mode") + isinstance(rr, int) + isinstance(cr, int)
    if key == length: pass
    elif key in ("layer", "auto", "value", "first"): s += 0.5
    elif key in POS: s += 1
    else: s += 2
    return s

def fam_ranked_bars(train):
    o0 = train[0]["output"]
    if H(o0) * W(o0) > 400: return
    common = set.intersection(*[{v for r in p["input"] for v in r} for p in train])
    bgspecs = ["mode", "none"] + sorted(c for c in common if any(bg_of(p["input"]) != c for p in train))
    found = []
    for bgs in bgspecs:
        infos = []
        for p in train:
            info = features(p["input"], bgs)
            if len(info["F"]) < 1 or len(info["F"]) > 12: infos = None; break
            infos.append(info)
        if not infos: continue
        dec = {}
        for mode in ("bars", "wrap", "pour"):
            for t in D8:
                ds = []
                for p, info in zip(train, infos):
                    C = D8[INV[t]](p["output"])
                    d = decode(C, info["bg"], mode)
                    if not d: ds = None; break
                    ds.append((d, H(C), W(C)))
                dec[(mode, t)] = ds
        if not any(dec.values()): continue
        sels = ["all", "drop-first"] + ["max:" + f for f in NUM]
        for mode in ("bars", "wrap", "pour"):
            for t in D8:
                if dec[(mode, t)] is None and dec[(mode, TCOMP[t])] is None: continue
                for flt in FILTERS:
                    for key in KEYS:
                        for desc in (False, True):
                            for sel in sels:
                                res = _match(train, infos, dec, mode, t, flt, key, desc, sel)
                                if not res: continue
                                for length, rr, cr in res:
                                    sig = (bgs, mode, t, flt, key, desc, sel, length, rr, cr)
                                    fn = program(bgs, flt, key, desc, sel, length, mode, rr, cr, t)
                                    if all(fn(p["input"]) == p["output"] for p in train):
                                        found.append((_score(sig), len(found), sig, fn))
                                if len(found) > 300: break
    found.sort(key=lambda x: (x[0], x[1]))
    for sc, _, sig, fn in found[:4]:
        bgs, mode, t, flt, key, desc, sel, length, rr, cr = sig
        name = f"ranked-bars:{mode}[bg={bgs},{flt},{sel},key={key}{'-' if desc else '+'},len={length},rows={rr},cols={cr},{t}]"
        yield (name, 5, fn)

def _match(train, infos, dec, mode, t, flt, key, desc, sel):
    per = []
    multi = False
    for p, info in zip(train, infos):
        r = order_items(info, flt, key, desc, sel)
        if r is None: return None
        seq, tr = r
        tt = TCOMP[t] if tr else t
        ds = dec.get((mode, tt))
        if ds is None: return None
        d, R, Cn = ds[len(per)]
        cols = [c for c, _ in d]
        if not set(cols) <= set(seq): return None
        if len(cols) >= 2: multi = True
        per.append((info, seq, d, R, Cn))
    if not multi: return None
    # lengths: items missing from the decoded sequence must have length 0 under the feature
    lens_ok = []
    for length in NUM + ("one", "full"):
        ok = True
        for info, seq, d, R, Cn in per:
            dd = dict(d)
            kept = [c for c in seq if c in dd]
            if [c for c, _ in d] != kept: ok = False; break
            for c in seq:
                if length == "one": L = 1
                elif length == "full": L = Cn
                else: L = info["F"][c][length]
                if dd.get(c, 0) != L: ok = False; break
            if not ok: break
        if ok: lens_ok.append(length)
    if not lens_ok: return None
    out = []
    for length in lens_ok:
        rrs = []; crs = []
        for rule in ("n", "maxlen", "inH", "inW"):
            if all(R == _dimv(rule, info, len(d), [L for _, L in d]) for info, seq, d, R, Cn in per):
                rrs.append(rule)
            if length != "full" and all(Cn == _dimv(rule, info, len(d), [L for _, L in d]) for info, seq, d, R, Cn in per):
                crs.append(rule)
        if len({R for _, _, _, R, _ in per}) == 1: rrs.append(per[0][3])
        if len({Cn for _, _, _, _, Cn in per}) == 1: crs.append(per[0][4])
        if length == "full":
            for rule in ("n", "inH", "inW"):
                if all(Cn == _dimv(rule, info, len(d), []) for info, seq, d, R, Cn in per): crs.append(rule)
        if rrs and crs: out.append((length, rrs[0], crs[0]))
    return out

def _dimv(rule, info, n, lens):
    if rule == "n": return n
    if rule == "maxlen": return max(lens) if lens else -1
    if rule == "inH": return info["h"]
    if rule == "inW": return info["w"]

FAMILIES = (fam_ranked_bars,)


In [ ]:
%%writefile /kaggle/working/probe/fam_transform_kronecker.py
"""transform.kronecker: output = blocks laid out by a mask grid; block(i,j) = tile_on(mask[i,j]) if pred(mask[i,j]) else tile_off.

One core (`kron`) + role inducers that decide where mask / tile / colours come from:
  self   : mask = tile source = the input itself (pred over cell colours, tile = input or self-colour mask)
  panels : input split into equal panels; mask = one shape panel, tile = another, fg/bg recolour from solid panels
  legend : mask = multicolour object, tile_on(c) = the motif paired (by legend order) with swatch colour c
  inplace: mask = sparse dot lattice, tile = mask(largest shape) painted by dot colour, drawn in place at the shape
All parameters (predicate colour, roles, order, colour swap) are enumerated over small domains and induced by fit.
"""
import sys
from collections import Counter
from itertools import permutations
from math import gcd
sys.path.insert(0, '/home/claude/work/widen')
from gdsl import H, W, bg_of, colours, objects, bbox, crop


def kron(mask, on_fn, off_tile, th, tw):
    """Assemble an (H(mask)*th) x (W(mask)*tw) grid; on_fn(c) returns a th x tw tile or None (use off_tile)."""
    out = []
    for y in range(H(mask)):
        blks = []
        for x in range(W(mask)):
            t = on_fn(mask[y][x])
            t = off_tile if t is None else t
            if t is None or H(t) != th or W(t) != tw: return None
            blks.append(t)
        for r in range(th): out.append(sum((b[r] for b in blks), []))
    return out


def solid(h, w, c): return [[c] * w for _ in range(h)]


def ranked(g, which):
    cnt = Counter(v for r in g for v in r).most_common()
    if which == "most":
        return cnt[0][0] if len(cnt) == 1 or cnt[0][1] > cnt[1][1] else None
    return cnt[-1][0] if len(cnt) == 1 or cnt[-1][1] < cnt[-2][1] else None


def _fits0(train, fn):
    try: return fn(train[0]["input"]) == train[0]["output"]
    except Exception: return False


# ---------------------------------------------------------------- self-similar (mask = tile = input)
def fam_kron_self(train):
    if not all((H(p["output"]), W(p["output"])) == (H(p["input"]) ** 2, W(p["input"]) ** 2) for p in train): return
    common = set.intersection(*(colours(p["input"]) for p in train))
    preds = [("nonbg", None), ("nonzero", None), ("most", None), ("least", None), ("all", None)] + [("eq", c) for c in sorted(common)]
    for (pk, pc) in preds:
        for inv in ((False,) if pk == "all" else (False, True)):
            for tile in ("input", "selfmask"):
                for z in ("zero", "bg"):
                    if tile == "input" and pk in ("nonbg", "nonzero") and not (inv and pk == "nonbg"):
                        continue  # plain fractal already in gdsl.fam_fractal
                    def fn(g, pk=pk, pc=pc, inv=inv, tile=tile, z=z):
                        bg = bg_of(g); zc = 0 if z == "zero" else bg
                        tgt = ranked(g, pk) if pk in ("most", "least") else pc
                        if pk in ("most", "least") and tgt is None: return None
                        def pred(v):
                            r = {"nonbg": v != bg, "nonzero": v != 0, "all": True}.get(pk, v == tgt)
                            return r != inv
                        h, w = H(g), W(g)
                        def on(v):
                            if not pred(v): return None
                            if tile == "input": return g
                            return [[v if q == v else zc for q in row] for row in g]
                        return kron(g, on, solid(h, w, zc), h, w)
                    name = f"kron:self[pred={pk}{'' if pc is None else pc}{',inv' if inv else ''},tile={tile},off={z}]"
                    if _fits0(train, fn): yield (name, 4, fn)


# ---------------------------------------------------------------- panels (mask panel x tile panel, recolour by solid panels)
def _split_any(g):
    """Split by full-length single-colour separator rows/cols (panels may differ in size)."""
    h, w = H(g), W(g)
    for sc in sorted(colours(g) - {bg_of(g)}):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols or len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]
        ps = [crop(g, (rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1))
              for a in range(len(rb) - 1) for b in range(len(cb) - 1) if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1]
        if len(ps) >= 3: return ps
    return None


def _panel_roles(g):
    panels = _split_any(g)
    if not panels: return None
    bg = Counter(v for p in panels for r in p for v in r).most_common(1)[0][0]
    shapes, cols, sc = [], [], []
    for p in panels:
        cells = [(y, x) for y in range(H(p)) for x in range(W(p)) if p[y][x] != bg]
        if not cells or len({p[y][x] for y, x in cells}) != 1: return None
        r0, c0, r1, c1 = bbox(cells)
        if len(cells) == (r1 - r0 + 1) * (c1 - c0 + 1) and len(cells) > 1: cols.append(p[cells[0][0]][cells[0][1]])
        else: shapes.append(p); sc.append(cells)
    if len(shapes) < 2: return None
    b = bbox([c for cs in sc for c in cs])
    if any(b[2] >= H(p) or b[3] >= W(p) for p in shapes): return None
    return [crop(p, b) for p in shapes], cols, bg


def fam_kron_panels(train):
    i, o = train[0]["input"], train[0]["output"]
    r = _panel_roles(i)
    if not r: return
    shapes, cols, _ = r
    if len(shapes) < 2 or len(shapes) > 4 or len(cols) > 4: return
    colsel = list(permutations(range(len(cols)), 2)) if len(cols) >= 2 else [None]
    for m, t in permutations(range(len(shapes)), 2):
        for cs in colsel:
            def fn(g, m=m, t=t, cs=cs):
                r = _panel_roles(g)
                if not r: return None
                shapes, cols, bg = r
                if max(m, t) >= len(shapes) or (cs and max(cs) >= len(cols)): return None
                mk, tl = shapes[m], shapes[t]
                if cs: fg, ob = cols[cs[0]], cols[cs[1]]
                else: fg, ob = None, bg
                on_t = [[(v if fg is None else fg) if v != bg else ob for v in row] for row in tl]
                return kron(mk, lambda v: on_t if v != bg else None, solid(H(tl), W(tl), ob), H(tl), W(tl))
            yield (f"kron:panels[mask={m},tile={t}{'' if cs is None else f',fg=p{cs[0]},bg=p{cs[1]}'}]", 5, fn)


# ---------------------------------------------------------------- legend (swatch colour -> motif)
def _merge_nested(os):
    """Merge same-colour components whose bounding boxes overlap (a motif with a detached inner stroke is one motif)."""
    os = [list(o) for o in os]; changed = True
    while changed:
        changed = False
        for a in range(len(os)):
            for b in range(a + 1, len(os)):
                A, B = bbox(os[a]), bbox(os[b])
                if A[0] <= B[2] and B[0] <= A[2] and A[1] <= B[3] and B[1] <= A[3]:
                    os[a] += os.pop(b); changed = True; break
            if changed: break
    return os


def _legend(g, order):
    bg = bg_of(g)
    obs = objects(g, bg, True, False)
    mono = [o for o in obs if len({g[y][x] for y, x in o}) == 1]
    multi = [o for o in obs if len({g[y][x] for y, x in o}) > 1]
    if len(multi) != 1: return None
    mask = crop(g, bbox(multi[0]))
    bycol = {}
    for o in mono: bycol.setdefault(g[o[0][0]][o[0][1]], []).append(o)
    bycol = {c: _merge_nested(os) for c, os in bycol.items()}
    mot = [c for c, os in bycol.items() if len(os) >= 2 and len({(bbox(o)[2] - bbox(o)[0], bbox(o)[3] - bbox(o)[1]) for o in os}) == 1]
    if len(mot) != 1: return None
    motifs = bycol.pop(mot[0])
    sw = [o for os in bycol.values() for o in os]
    if len(sw) != len(motifs) or len({g[o[0][0]][o[0][1]] for o in sw}) != len(sw): return None
    if any(len(o) != (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1) for o in sw): return None
    key = (lambda o: (bbox(o)[0], bbox(o)[1])) if order == "row" else (lambda o: (bbox(o)[1], bbox(o)[0]))
    sw.sort(key=key); motifs.sort(key=key)
    return mask, {g[s[0][0]][s[0][1]]: crop(g, bbox(m)) for s, m in zip(sw, motifs)}, mot[0], bg


def fam_kron_legend(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) <= H(i) and W(o) <= W(i) and (H(o), W(o)) != (H(i), W(i)) and H(o) * W(o) < 4: return
    if not _legend(i, "row"): return
    for order in ("row", "col"):
        def fn(g, order=order):
            L = _legend(g, order)
            if not L: return None
            mask, lut, mc, bg = L
            t0 = next(iter(lut.values())); th, tw = H(t0), W(t0)
            def on(v):
                if v not in lut: return None
                return [[v if q == mc else q for q in row] for row in lut[v]]
            return kron(mask, on, solid(th, tw, bg), th, tw)
        yield (f"kron:legend[order={order}]", 5, fn)


# ---------------------------------------------------------------- in place (dot lattice x shape mask, anchored at shape)
def _inplace(g, recol):
    bg = bg_of(g)
    obs = objects(g, bg, True, True)
    if len(obs) < 3: return None
    big = max(obs, key=len)
    if sum(1 for o in obs if len(o) == len(big)) != 1: return None
    dots = [o for o in obs if o is not big]
    if any(len(o) != 1 for o in dots): return None
    sc = g[big[0][0]][big[0][1]]
    pts = [(o[0][0], o[0][1], g[o[0][0]][o[0][1]]) for o in dots]
    r0 = min(p[0] for p in pts); c0 = min(p[1] for p in pts)
    py = px = 0
    for y, x, _ in pts: py = gcd(py, y - r0); px = gcd(px, x - c0)
    py = py or 1; px = px or 1
    anchor = [p for p in pts if p[2] == sc]
    if len(anchor) != 1: return None
    others = {p[2] for p in pts} - {sc}
    if recol == "swap" and len(others) != 1: return None
    oc = next(iter(others)) if others else sc
    R0, C0, R1, C1 = bbox(big); sh, sw = R1 - R0 + 1, C1 - C0 + 1
    ay, ax = (anchor[0][0] - r0) // py, (anchor[0][1] - c0) // px
    out = [row[:] for row in g]
    for y, x in big: out[y][x] = bg
    for y, x, _ in pts: out[y][x] = bg
    rel = [(y - R0, x - C0) for y, x in big]
    for y, x, c in pts:
        i, j = (y - r0) // py - ay, (x - c0) // px - ax
        col = c if recol == "dot" else (oc if c == sc else sc)
        for dy, dx in rel:
            yy, xx = R0 + i * sh + dy, C0 + j * sw + dx
            if 0 <= yy < H(g) and 0 <= xx < W(g): out[yy][xx] = col
    return out


def fam_kron_inplace(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for recol in ("dot", "swap"):
        fn = lambda g, recol=recol: _inplace(g, recol)
        if _fits0(train, fn): yield (f"kron:inplace[colour={recol}]", 5, fn)


FAMILIES = (fam_kron_self, fam_kron_panels, fam_kron_legend, fam_kron_inplace)


In [ ]:
%%writefile /kaggle/working/probe/fam_transform_objects.py
"""transform.dihedral + transform.reflect_object: dihedral ops applied to a SOURCE that is not the whole grid.

transform.dihedral   out = D(source)
  source : framed picture crop | largest-object crop | lattice summary (cut lines / separator panels)
           | each object about its own centre (in place) | object positions only (motifs keep orientation)
  D      : constant (the harness composes crop:/downscale: programs with every D8 element)
           | chosen per grid so that marker colours inside the crop take the layout of the outside markers
           | analogy: D (and colour relabelling) mapping block A to block B, applied to block C, per lattice row

transform.reflect_object   copy = reflect(obj, axis); optionally erase obj; paint obj colour or marker colour
  axis   : line through (template cell, marker cell) pair  (marker = singleton of another colour)
           | spine of nearest anchor object (row / col / diagonal line with most anchor cells)
           | centre of the fully-symmetric anchor object (H, V and HV copies)
           | own bbox edge on side d, d = constant or read from an indicator object's shape by rotation /
             D8 equivariance with the training examples
All colours / roles are induced from the training pairs; every program is verified by the harness.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, objects, bbox, crop, D8, split_panels, static_colours

ROT4 = ("id", "r90", "r180", "r270")
DIR4 = {"N": (-1, 0), "S": (1, 0), "W": (0, -1), "E": (0, 1)}


def same_shape(train):
    return all((H(p["input"]), W(p["input"])) == (H(p["output"]), W(p["output"])) for p in train)


def comps(g, bg, conn):
    """conn: c4 / c8 single colour, m4 / m8 multicolour."""
    return objects(g, bg, diag=conn[1] == "8", by_colour=conn[0] == "c")


def mask_of(cells):
    r0, c0, r1, c1 = bbox(cells)
    m = [[0] * (c1 - c0 + 1) for _ in range(r1 - r0 + 1)]
    for y, x in cells: m[y - r0][x - c0] = 1
    return m


def dir_under(d, D):
    """Image of a direction vector under grid transform D (applied to a 3x3 probe)."""
    p = [[0] * 3 for _ in range(3)]; p[1 + d[0]][1 + d[1]] = 1
    q = D8[D](p)
    for y in range(3):
        for x in range(3):
            if q[y][x]: return (y - 1, x - 1)


# ============================================================ transform.dihedral

def framed_picture(g, sides=None):
    """Largest multicolour component's bbox, peeling edge lines that are one colour absent from the rest."""
    bg = bg_of(g); h, w = H(g), W(g)
    obs = [ob for ob in comps(g, bg, "m8") if not any(y in (0, h - 1) or x in (0, w - 1) for y, x in ob)]
    if not obs: return None
    r0, c0, r1, c1 = bbox(max(obs, key=len)); peeled = 0
    while r1 - r0 >= 1 and c1 - c0 >= 1:
        done = False
        for side in "TBLR":
            if side == "T": line = [g[r0][x] for x in range(c0, c1 + 1)]; rest = (r0 + 1, c0, r1, c1)
            elif side == "B": line = [g[r1][x] for x in range(c0, c1 + 1)]; rest = (r0, c0, r1 - 1, c1)
            elif side == "L": line = [g[y][c0] for y in range(r0, r1 + 1)]; rest = (r0, c0 + 1, r1, c1)
            else: line = [g[y][c1] for y in range(r0, r1 + 1)]; rest = (r0, c0, r1, c1 - 1)
            cs = {v for v in line if v != bg}
            if len(cs) != 1 or sum(v != bg for v in line) * 2 < len(line): continue
            c = cs.pop()
            if any(g[y][x] == c for y in range(rest[0], rest[2] + 1) for x in range(rest[1], rest[3] + 1)): continue
            r0, c0, r1, c1 = rest; peeled += 1; done = True
            if sides is not None: sides[side] = c
            break
        if not done: break
    return crop(g, (r0, c0, r1, c1)) if peeled else None


SIDE_DIR = {"T": (-1, 0), "B": (1, 0), "L": (0, -1), "R": (0, 1)}


def framed_oriented(g):
    """Framed picture turned so that each frame line's colour lands on the grid side bordered by that colour."""
    sides = {}; pic = framed_picture(g, sides)
    if pic is None or not sides: return None
    h, w = H(g), W(g); bg = bg_of(g)
    edge = {"T": g[0], "B": g[h - 1], "L": [r[0] for r in g], "R": [r[w - 1] for r in g]}
    ecol = {}
    for s, line in edge.items():
        cnt = Counter(v for v in line if v != bg)
        if cnt: ecol[s] = cnt.most_common(1)[0][0]
    inv = {v: k for k, v in SIDE_DIR.items()}; hits = []
    for D in D8:
        ok = True
        for s, c in sides.items():
            t = inv[dir_under(SIDE_DIR[s], D)]
            if ecol.get(t) != c or list(ecol.values()).count(c) != 1: ok = False; break
        if ok: hits.append(D)
    return D8[hits[0]](pic) if len(hits) == 1 else None


def fam_crop_framed(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) >= H(i) * W(i): return
    yield ("crop:framed-picture", 3, framed_picture)
    yield ("crop-oriented:framed-by-border-colour", 4, framed_oriented)


def layout_sig(pos):
    ks = sorted(pos)
    sg = lambda v: (v > 0) - (v < 0)
    return tuple((a, b, sg(pos[b][0] - pos[a][0]), sg(pos[b][1] - pos[a][1])) for a in ks for b in ks if a < b)


def crop_marker_oriented(g):
    """Crop the largest multicolour object; pick the D8 element under which the singleton colours inside
    the crop are laid out like the same colours' singletons outside it."""
    bg = bg_of(g); obs = comps(g, bg, "m8")
    if len(obs) < 3: return None
    big = max(obs, key=len); b = bbox(big)
    outside = Counter(); opos = {}
    for ob in obs:
        if ob is big: continue
        if len(ob) != 1: return None
        y, x = ob[0]; outside[g[y][x]] += 1; opos[g[y][x]] = (y, x)
    if len(opos) < 2 or any(v > 1 for v in outside.values()): return None
    pic = crop(g, b); want = layout_sig(opos); hits = []
    for k, f in D8.items():
        q = f(pic); pos = {}
        for c in opos:
            cells = [(y, x) for y in range(H(q)) for x in range(W(q)) if q[y][x] == c]
            if len(cells) != 1: return None
            pos[c] = cells[0]
        if layout_sig(pos) == want: hits.append(q)
    if not hits or any(h != hits[0] for h in hits): return None
    return hits[0]


def fam_crop_marker_oriented(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) >= H(i) * W(i): return
    yield ("crop-oriented:by-marker-layout", 4, crop_marker_oriented)


def cut_positions(g, bg, axis):
    """Lattice cuts: positions where >= 2 lines show a change between two non-bg colours."""
    h, w = H(g), W(g); cuts = []
    n = w if axis == "col" else h
    for k in range(1, n):
        cnt = 0
        for t in range(h if axis == "col" else w):
            a, b = (g[t][k - 1], g[t][k]) if axis == "col" else (g[k - 1][t], g[k][t])
            if a != bg and b != bg and a != b: cnt += 1
        if cnt >= 2: cuts.append(k)
    return cuts


def lattice_summary(g, mode):
    bg = bg_of(g)
    if mode == "panels":
        sp = split_panels(g)
        if not sp: return None
        panels, sc = sp
        h, w = H(g), W(g)
        nr = 1 + sum(1 for r in range(h) if all(v == sc for v in g[r]) and 0 < r < h - 1 and not all(v == sc for v in g[r - 1]))
        nc = len(panels) // nr if nr and len(panels) % nr == 0 else 0
        if not nc: return None
        cells = []
        for p in panels:
            cnt = Counter(v for r in p for v in r)
            top = cnt.most_common(2)
            if len(top) > 1 and top[0][1] == top[1][1]: return None
            cells.append(top[0][0])
        return [cells[r * nc:(r + 1) * nc] for r in range(nr)]
    fg = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg]
    if not fg: return None
    s = crop(g, bbox(fg))
    rc = [0] + cut_positions(s, bg, "row") + [H(s)]; cc = [0] + cut_positions(s, bg, "col") + [W(s)]
    if len(rc) * len(cc) > 100 or (len(rc) == 2 and len(cc) == 2): return None
    out = []
    for a in range(len(rc) - 1):
        row = []
        for b in range(len(cc) - 1):
            cnt = Counter(s[y][x] for y in range(rc[a], rc[a + 1]) for x in range(cc[b], cc[b + 1]) if s[y][x] != bg)
            if not cnt: return None
            row.append(cnt.most_common(1)[0][0])
        out.append(row)
    return out


def fam_lattice_summary(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) >= H(i) or W(o) >= W(i) or H(o) * W(o) > 64: return
    for mode in ("panels", "cuts"):
        yield (f"downscale:lattice-summary[{mode}]", 3, lambda g, mode=mode: lattice_summary(g, mode))


def place(out, cells_vals, h, w):
    for (y, x), v in cells_vals:
        if 0 <= y < h and 0 <= x < w: out[y][x] = v
        else: return False
    return True


def objects_dihedral_inplace(g, D, conn):
    """Every object is replaced by D(object) about its own bbox centre."""
    bg = bg_of(g); h, w = H(g), W(g); out = [[bg] * w for _ in range(h)]
    obs = comps(g, bg, conn)
    if len(obs) < 2: return None
    for ob in obs:
        r0, c0, r1, c1 = bbox(ob); hh, ww = r1 - r0 + 1, c1 - c0 + 1
        if 2 * hh * ww > h * w: return None
        sub = [[g[y][x] if (y, x) in set(ob) else None for x in range(c0, c1 + 1)] for y in range(r0, r1 + 1)]
        q = D8[D](sub); nh, nw = H(q), W(q)
        if (hh - nh) % 2 or (ww - nw) % 2: return None
        y0, x0 = r0 + (hh - nh) // 2, c0 + (ww - nw) // 2
        if not place(out, [((y0 + y, x0 + x), q[y][x]) for y in range(nh) for x in range(nw) if q[y][x] is not None], h, w):
            return None
    return out


def fam_objects_dihedral_inplace(train):
    if not same_shape(train): return
    for D, conn in product(("T", "aT", "r90", "r270", "fh", "fv", "r180"), ("c4", "c8", "m8")):
        yield (f"object-dihedral-inplace:{D}[{conn}]", 4, lambda g, D=D, conn=conn: objects_dihedral_inplace(g, D, conn))


def mirror_positions(g, D, conn):
    """Objects keep their content; only their placement is mirrored (fh / fv / r180 of the grid)."""
    bg = bg_of(g); h, w = H(g), W(g); out = [[bg] * w for _ in range(h)]
    obs = comps(g, bg, conn)
    if len(obs) < 2: return None
    for ob in obs:
        r0, c0, r1, c1 = bbox(ob)
        nr0 = h - 1 - r1 if D in ("fv", "r180") else r0
        nc0 = w - 1 - c1 if D in ("fh", "r180") else c0
        for y, x in ob: out[nr0 + y - r0][nc0 + x - c0] = g[y][x]
    return out


def fam_mirror_positions(train):
    if not same_shape(train): return
    for D, conn in product(("fh", "fv", "r180"), ("m4", "m8")):
        yield (f"mirror-positions:{D}[{conn}]", 4, lambda g, D=D, conn=conn: mirror_positions(g, D, conn))


# ============================================================ transform.reflect_object

def reflect_to_markers(g, diag, paint, keep_marker=True):
    """Templates = single-colour components (8-conn) of size >= 2; markers = singleton cells of another colour.
    Each marker is the mirror image of its nearest aligned template cell; the template is reflected with that
    mapping (row-aligned: horizontal flip, column-aligned: vertical flip, diagonal: point reflection)."""
    bg = bg_of(g); h, w = H(g), W(g)
    obs = comps(g, bg, "c8")
    temps = [ob for ob in obs if len(ob) >= 2]; marks = [ob[0] for ob in obs if len(ob) == 1]
    if not temps or not marks: return None
    out = [r[:] for r in g]; used = 0
    for my, mx in marks:
        mc = g[my][mx]; best = []
        for ti, ob in enumerate(temps):
            tc = g[ob[0][0]][ob[0][1]]
            if tc == mc: continue
            for ty, tx in ob:
                dy, dx = my - ty, mx - tx
                if dy == 0 and dx: kind = "h"
                elif dx == 0 and dy: kind = "v"
                elif diag and abs(dy) == abs(dx) and dy: kind = "p"
                else: continue
                best.append((max(abs(dy), abs(dx)), kind == "p", ti, kind, ty, tx))
        if not best: continue
        best.sort(); top = [b for b in best if b[:2] == best[0][:2]]
        maps = {(b[2], b[3], b[4] + my if b[3] != "h" else None, b[5] + mx if b[3] != "v" else None) for b in top}
        if len(maps) != 1: return None
        ti, kind, sy, sx = maps.pop(); ob = temps[ti]
        col = mc if paint == "marker" else g[ob[0][0]][ob[0][1]]
        for y, x in ob:
            ny = sy - y if kind in "vp" else y
            nx = sx - x if kind in "hp" else x
            if 0 <= ny < h and 0 <= nx < w and out[ny][nx] == bg: out[ny][nx] = col
        used += 1
    return out if used else None


def fam_reflect_to_markers(train):
    if not same_shape(train): return
    for diag, paint in product((False, True), ("marker", "template")):
        yield (f"reflect-object:to-marker[{'diag' if diag else 'orth'},{paint}]", 4,
               lambda g, diag=diag, paint=paint: reflect_to_markers(g, diag, paint))


def spine(ob):
    """Straight line (kind, k) containing the most cells of the object: row y=k, col x=k, diag y-x=k, anti y+x=k."""
    cnt = Counter()
    for y, x in ob:
        cnt[("row", y)] += 1; cnt[("col", x)] += 1; cnt[("diag", y - x)] += 1; cnt[("anti", y + x)] += 1
    (best, n), = cnt.most_common(1)
    if n < 3 or sum(1 for v in cnt.values() if v == n) > 1: return None
    return best


def refl(y, x, ax):
    kind, k = ax
    if kind == "row": return 2 * k - y, x
    if kind == "col": return y, 2 * k - x
    if kind == "diag": return x + k, y - k
    return k - x, k - y


def line_dist(y, x, ax):
    kind, k = ax
    return abs({"row": y, "col": x, "diag": y - x, "anti": y + x}[kind] - k)


def colour_select(g, bg, axis_c, how, st):
    cnt = Counter(v for r in g for v in r if v not in (bg, axis_c))
    if not cnt: return set()
    if how == "all": return set(cnt)
    if how == "nonstatic": return {c for c in cnt if c not in st}
    vals = sorted(cnt.values())
    if how == "least":
        return {c for c, n in cnt.items() if n == vals[0]} if vals.count(vals[0]) == 1 else set()
    return {c for c, n in cnt.items() if n == vals[-1]} if vals.count(vals[-1]) == 1 else set()


def reflect_across_anchor(g, axis_c, how, keep, conn, st):
    bg = bg_of(g); h, w = H(g), W(g)
    anchors = [ob for ob in comps(g, bg, "c8") if g[ob[0][0]][ob[0][1]] == axis_c]
    axes = [spine(ob) for ob in anchors]
    if not axes or any(a is None for a in axes): return None
    movc = colour_select(g, bg, axis_c, how, st)
    if not movc: return None
    movers = [ob for ob in comps(g, bg, conn) if all(g[y][x] in movc for y, x in ob)]
    if not movers: return None
    out = [r[:] for r in g]
    if not keep:
        for ob in movers:
            for y, x in ob: out[y][x] = bg
    for ob in movers:
        ds = sorted((min(line_dist(y, x, a) for y, x in ob), i) for i, a in enumerate(axes))
        if len(ds) > 1 and ds[0][0] == ds[1][0] and axes[ds[0][1]] != axes[ds[1][1]]: return None
        ax = axes[ds[0][1]]
        for y, x in ob:
            ny, nx = refl(y, x, ax)
            if not (0 <= ny < h and 0 <= nx < w): return None
            if out[ny][nx] in (bg, g[y][x]): out[ny][nx] = g[y][x]
            else: return None
    return out


def reflect_about_centre(g, conn):
    """The one object symmetric under both flips is the centre; every other object gets its H, V and HV copies."""
    bg = bg_of(g); h, w = H(g), W(g); obs = comps(g, bg, conn)
    def sym(ob):
        s = set(ob); r0, c0, r1, c1 = bbox(ob)
        return all((r0 + r1 - y, x) in s and (y, c0 + c1 - x) in s and g[r0 + r1 - y][x] == g[y][x] and g[y][c0 + c1 - x] == g[y][x] for y, x in ob)
    cen = [ob for ob in obs if sym(ob)]
    if len(cen) != 1 or len(obs) < 2: return None
    r0, c0, r1, c1 = bbox(cen[0]); sy, sx = r0 + r1, c0 + c1
    out = [r[:] for r in g]
    for ob in obs:
        if ob is cen[0]: continue
        for y, x in ob:
            for ny, nx in ((sy - y, x), (y, sx - x), (sy - y, sx - x)):
                if not (0 <= ny < h and 0 <= nx < w): return None
                out[ny][nx] = g[y][x]
    return out


def fam_reflect_across_anchor(train):
    if not same_shape(train): return
    st = static_colours(train)
    common = None
    for p in train:
        cs = {v for r in p["input"] for v in r} - {bg_of(p["input"])}
        common = cs if common is None else common & cs
    for conn in ("c8", "m8"):
        yield (f"reflect-object:about-centre-object[{conn}]", 4, lambda g, conn=conn: reflect_about_centre(g, conn))
    for axis_c in sorted((common or set()) & st):
        for how, keep, conn in product(("nonstatic", "least", "most", "all"), (True, False), ("c8", "m8")):
            if how == "nonstatic" and keep: continue
            yield (f"reflect-object:across-anchor-spine[c{axis_c},{how},{'keep' if keep else 'move'},{conn}]", 4,
                   lambda g, a=axis_c, how=how, keep=keep, conn=conn: reflect_across_anchor(g, a, how, keep, conn, st))


def reflect_edge(g, ob, d, bg, out):
    r0, c0, r1, c1 = bbox(ob); h, w = H(g), W(g)
    for y, x in ob:
        if d == "N": ny, nx = 2 * r0 - 1 - y, x
        elif d == "S": ny, nx = 2 * r1 + 1 - y, x
        elif d == "W": ny, nx = y, 2 * c0 - 1 - x
        else: ny, nx = y, 2 * c1 + 1 - x
        if not (0 <= ny < h and 0 <= nx < w): return False
        out[ny][nx] = g[y][x]
    return True


def split_roles(g, bg, mc):
    mov = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == mc]
    ind = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] not in (bg, mc)]
    return mov, ind


def fam_reflect_own_edge(train):
    """One object of an induced colour is mirrored across its own bbox edge on side d; d is constant or is read
    from the shape of the other (indicator) object, equivariantly (rotations or all of D8) with the training pairs."""
    if not same_shape(train): return
    gain = None
    for p in train:
        i, o = p["input"], p["output"]; bg = bg_of(i)
        g_ = {o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] == bg and o[y][x] != bg}
        gain = g_ if gain is None else gain & g_
    if not gain or len(gain) != 1: return
    mc = gain.pop()
    lib = []; erase = None
    for p in train:
        i, o = p["input"], p["output"]; bg = bg_of(i)
        mov, ind = split_roles(i, bg, mc)
        if not mov or not ind: return
        er = all(o[y][x] != i[y][x] for y, x in ind)
        erase = er if erase is None else (erase if erase == er else None)
        if erase is None: return
        hit = []
        for d in DIR4:
            out = [r[:] for r in i]
            if erase:
                for y, x in ind: out[y][x] = bg
            if reflect_edge(i, mov, d, bg, out) and out == o: hit.append(d)
        if len(hit) != 1: return
        lib.append((mask_of(ind), DIR4[hit[0]]))
    inv = {v: k for k, v in DIR4.items()}

    def fn(g, mode):
        bg = bg_of(g); mov, ind = split_roles(g, bg, mc)
        if not mov or not ind: return None
        if mode == "const":
            ds = {d for _, d in lib}
        else:
            S = mask_of(ind); ds = set()
            for m, d in lib:
                for D in (ROT4 if mode == "rot" else D8):
                    if D8[D](m) == S: ds.add(dir_under(d, D))
        if len(ds) != 1: return None
        out = [r[:] for r in g]
        if erase:
            for y, x in ind: out[y][x] = bg
        return out if reflect_edge(g, mov, inv[ds.pop()], bg, out) else None
    for mode in ("const", "rot", "d8"):
        yield (f"reflect-object:own-edge[c{mc},{mode}{',erase' if erase else ''}]", 4, lambda g, mode=mode: fn(g, mode))


def bg_runs(n, is_sep):
    runs, cur = [], None
    for k in range(n):
        if is_sep(k):
            if cur is not None: runs.append(cur); cur = None
        else:
            cur = [k, k] if cur is None else [cur[0], k]
    if cur is not None: runs.append(cur)
    return runs


def analogy_map(A, B, C):
    """A : B :: C : ?  with ? = cmap(D(C)); (D, cmap) is the simplest pair (fewest recolourings, then D8 order)
    mapping A exactly onto B; None when C holds a colour the map does not know."""
    best = None
    for rank, k in enumerate(("id", "fh", "fv", "T", "aT", "r180", "r90", "r270")):
        f = D8[k]; q = f(A)
        if (H(q), W(q)) != (H(B), W(B)): continue
        m = {}; ok = True
        for ra, rb in zip(q, B):
            for a, b in zip(ra, rb):
                if m.setdefault(a, b) != b: ok = False
        if not ok: continue
        key = (sum(a != b for a, b in m.items()), rank)
        if best is None or key < best[0]: best = (key, f, m)
    if best is None: return None
    _, f, m = best; r = f(C)
    if best[0][0] and any(v not in m for row in r for v in row): return None
    return [[m.get(v, v) for v in row] for row in r]


def lattice_analogy(g, orient):
    """Blocks separated by background lines; each lattice row (or column) holds A, B, C; the output keeps the
    separator layout and replaces each row (column) by the analogy answer for C."""
    if orient == "col": g = [list(r) for r in zip(*g)]
    bg = bg_of(g); h, w = H(g), W(g)
    rr = bg_runs(h, lambda y: all(v == bg for v in g[y])); cr = bg_runs(w, lambda x: all(g[y][x] == bg for y in range(h)))
    if len(cr) != 3 or len(rr) < 1: return None
    out = []; prev = -1
    for a, b in rr:
        out += [None] * (a - prev - 1)
        A, B, C = (crop(g, (a, c0, b, c1)) for c0, c1 in cr)
        r = analogy_map(A, B, C)
        if r is None or H(r) != b - a + 1: return None
        out += r; prev = b
    out += [None] * (h - 1 - prev)
    wd = max(len(r) for r in out if r is not None)
    if any(r is not None and len(r) != wd for r in out): return None
    out = [r if r is not None else [bg] * wd for r in out]
    return [list(r) for r in zip(*out)] if orient == "col" else out


def fam_lattice_analogy(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) >= H(i) * W(i): return
    for orient in ("row", "col"):
        yield (f"lattice-analogy:{orient}", 4, lambda g, orient=orient: lattice_analogy(g, orient))


def slide_mirror_into_anchor(g, mc, ac):
    """The mover (colour mc) slides straight toward the anchor (colour ac) until they touch; the anchor is
    replaced by the mover's mirror image across the contact edge, painted in the anchor colour."""
    bg = bg_of(g); h, w = H(g), W(g)
    mov = [(y, x) for y in range(h) for x in range(w) if g[y][x] == mc]
    anc = [(y, x) for y in range(h) for x in range(w) if g[y][x] == ac]
    if not mov or not anc: return None
    m0, n0, m1, n1 = bbox(mov); a0, b0, a1, b1 = bbox(anc)
    if n0 <= b1 and b0 <= n1 and not (m0 <= a1 and a0 <= m1):
        d = "S" if a0 > m1 else "N"; k = (a0 - m1 - 1) if d == "S" else (m0 - a1 - 1); dy, dx = (k, 0) if d == "S" else (-k, 0)
    elif m0 <= a1 and a0 <= m1 and not (n0 <= b1 and b0 <= n1):
        d = "E" if b0 > n1 else "W"; k = (b0 - n1 - 1) if d == "E" else (n0 - b1 - 1); dy, dx = (0, k) if d == "E" else (0, -k)
    else:
        return None
    out = [r[:] for r in g]
    for y, x in mov + anc: out[y][x] = bg
    moved = [(y + dy, x + dx) for y, x in mov]
    for y, x in moved: out[y][x] = mc
    ok = reflect_edge([[mc if (y, x) in set(moved) else bg for x in range(w)] for y in range(h)], moved, d, bg, out)
    if not ok: return None
    for y in range(h):
        for x in range(w):
            if out[y][x] == mc and (y, x) not in set(moved): out[y][x] = ac
    return out


def fam_slide_mirror_into_anchor(train):
    if not same_shape(train): return
    cs = None
    for p in train:
        c = {v for r in p["input"] for v in r} - {bg_of(p["input"])}
        cs = c if cs is None else cs & c
    if not cs or len(cs) != 2: return
    for mc in cs:
        ac = (cs - {mc}).pop()
        yield (f"reflect-object:slide-mirror-into-anchor[c{mc}->c{ac}]", 5,
               lambda g, mc=mc, ac=ac: slide_mirror_into_anchor(g, mc, ac))


def orient_to_marker(g, head, mk):
    """Every straight run (row or column segment of >= 2 non-background, non-marker cells containing colour
    `head`) that shares its line with a marker (colour mk) is reversed in place if that brings its head cells
    closer to the marker."""
    bg = bg_of(g); h, w = H(g), W(g)
    out = [r[:] for r in g]; touched = set(); acted = 0
    for axis in ("row", "col"):
        for k in range(h if axis == "row" else w):
            line = [(k, x) for x in range(w)] if axis == "row" else [(y, k) for y in range(h)]
            vals = [g[y][x] for y, x in line]
            marks = [t for t, v in enumerate(vals) if v == mk]
            if not marks: continue
            t = 0
            while t < len(vals):
                if vals[t] in (bg, mk): t += 1; continue
                u = t
                while u + 1 < len(vals) and vals[u + 1] not in (bg, mk): u += 1
                seg = vals[t:u + 1]
                if len(seg) >= 2 and head in seg:
                    rev = seg[::-1]
                    d0 = min(abs(t + j - m) for j, v in enumerate(seg) if v == head for m in marks)
                    d1 = min(abs(t + j - m) for j, v in enumerate(rev) if v == head for m in marks)
                    if d1 < d0:
                        for j, v in enumerate(rev):
                            y, x = line[t + j]
                            if (y, x) in touched: return None
                            out[y][x] = v
                        acted += 1
                    for j in range(len(seg)): touched.add(line[t + j])
                u += 1; t = u
    return out if acted else None


def fam_orient_to_marker(train):
    if not same_shape(train): return
    i, o = train[0]["input"], train[0]["output"]; bg = bg_of(i)
    diff = {i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]}
    if not diff or bg in diff or len(diff) > 3: return
    cs = {v for r in i for v in r} - {bg}
    for head, mk in product(sorted(diff), sorted(cs - diff)):
        yield (f"reflect-object:orient-to-marker[head=c{head},marker=c{mk}]", 5,
               lambda g, head=head, mk=mk: orient_to_marker(g, head, mk))


FAMILIES = (fam_crop_framed, fam_crop_marker_oriented, fam_lattice_summary, fam_objects_dihedral_inplace,
            fam_mirror_positions, fam_reflect_to_markers, fam_reflect_across_anchor, fam_reflect_own_edge,
            fam_lattice_analogy, fam_slide_mirror_into_anchor, fam_orient_to_marker)


In [ ]:
%%writefile /kaggle/working/probe/gdsl.py
"""G-DSL: whole-grid primitives with parameters inferred from training pairs only.

Every program is verified exactly on all training pairs before it is applied to a test input.
Programs are enumerated in description-length order (cheapest first). No task identifiers,
no task-specific constants: every parameter (colours, factors, selectors, masks) is induced
from the task's own training pairs.
"""
from __future__ import annotations
from collections import Counter
from itertools import product

# ------------------------------------------------------------------ grid helpers
def H(g): return len(g)
def W(g): return len(g[0])
def T(g): return [list(r) for r in zip(*g)]
def fh(g): return [r[::-1] for r in g]
def fv(g): return g[::-1]
def r90(g): return fh(T(g))
def r180(g): return fv(fh(g))
def r270(g): return fv(T(g))
def at(g): return r180(T(g))
D8 = {"id": lambda g: [list(r) for r in g], "r90": r90, "r180": r180, "r270": r270,
      "fh": fh, "fv": fv, "T": T, "aT": at}
def bg_of(g):
    return Counter(v for r in g for v in r).most_common(1)[0][0]
def colours(g): return {v for r in g for v in r}
def eq(a, b): return a == b

def objects(g, bg, diag=False, by_colour=True):
    h, w = H(g), W(g); seen = [[False] * w for _ in range(h)]; out = []
    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
    for r in range(h):
        for c in range(w):
            if seen[r][c] or g[r][c] == bg: continue
            col = g[r][c]; st = [(r, c)]; seen[r][c] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != bg and (not by_colour or g[yy][xx] == col):
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out

def bbox(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    return min(ys), min(xs), max(ys), max(xs)

def crop(g, b):
    r0, c0, r1, c1 = b
    return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]

# ------------------------------------------------------------------ colour map (post-step)
def fit_cmap(preds, outs):
    """Global colour substitution consistent across all pairs; None if impossible."""
    m = {}
    for p, o in zip(preds, outs):
        if p is None or H(p) != H(o) or W(p) != W(o): return None
        for rp, ro in zip(p, o):
            for a, b in zip(rp, ro):
                if m.setdefault(a, b) != b: return None
    return m

def apply_cmap(g, m):
    """Apply an induced colour substitution; refuse colours never seen in training (no extrapolation)."""
    if g is None: return None
    if any(v not in m for r in g for v in r): return None
    return [[m[v] for v in r] for r in g]

# ------------------------------------------------------------------ primitive families
# Each family yields (name, cost, fn) where fn(grid) -> grid or None. Families may inspect
# training pairs to induce parameters (never test outputs).

def fam_geometric(train):
    for k, f in D8.items():
        if k != "id": yield ("dihedral:" + k, 1, f)

def fam_tile(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) % H(i) or W(o) % W(i): return
    n, m = H(o) // H(i), W(o) // W(i)
    if n * m == 1 or n * m > 16: return
    # per-tile dihedral transform inferred from the first pair, verified later on all pairs
    pat = []
    for a in range(n):
        row = []
        for b in range(m):
            blk = [r[b * W(i):(b + 1) * W(i)] for r in o[a * H(i):(a + 1) * H(i)]]
            ks = [k for k, f in D8.items() if (H(i) == W(i) or k in ("id", "r180", "fh", "fv")) and f(i) == blk]
            if not ks: return
            row.append(ks[0])
        pat.append(row)
    def fn(g, pat=pat, n=n, m=m):
        out = []
        for a in range(n):
            blks = [D8[pat[a][b]](g) for b in range(m)]
            if any(H(x) != H(blks[0]) for x in blks): return None
            for rr in range(H(blks[0])):
                out.append(sum((x[rr] for x in blks), []))
        return out
    yield (f"tile:{n}x{m}", 2, fn)

def fam_scale(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) % H(i) == 0 and W(o) % W(i) == 0 and (H(o) > H(i) or W(o) > W(i)):
        kh, kw = H(o) // H(i), W(o) // W(i)
        yield (f"upscale:{kh}x{kw}", 2, lambda g, kh=kh, kw=kw: [[v for v in r for _ in range(kw)] for r in g for _ in range(kh)])
    if H(i) % H(o) == 0 and W(i) % W(o) == 0 and (H(i) > H(o) or W(i) > W(o)):
        kh, kw = H(i) // H(o), W(i) // W(o)
        def down(g, kh=kh, kw=kw, mode="mode"):
            if H(g) % kh or W(g) % kw: return None
            bg = bg_of(g); out = []
            for a in range(0, H(g), kh):
                row = []
                for b in range(0, W(g), kw):
                    blk = [v for r in g[a:a + kh] for v in r[b:b + kw]]
                    nz = [v for v in blk if v != bg]
                    row.append(Counter(nz).most_common(1)[0][0] if (mode == "any" and nz) else (Counter(blk).most_common(1)[0][0] if mode == "mode" else bg))
                out.append(row)
            return out
        yield (f"downscale-mode:{kh}x{kw}", 2, down)
        yield (f"downscale-any:{kh}x{kw}", 3, lambda g, d=down: d(g, mode="any"))

def fam_fractal(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i) * H(i), W(i) * W(i)): return
    for inv, bgm in product((False, True), ("mode", "zero")):
        def fn(g, inv=inv, bgm=bgm):
            bg = bg_of(g) if bgm == "mode" else 0
            h, w = H(g), W(g); out = [[bg] * (w * w) for _ in range(h * h)]
            for a in range(h):
                for b in range(w):
                    if (g[a][b] != bg) != inv:
                        for y in range(h):
                            for x in range(w):
                                out[a * h + y][b * w + x] = g[y][x]
            return out
        yield ("fractal" + (":inverse" if inv else "") + ("" if bgm == "mode" else ":bg0"), 3, fn)

SELECTORS = ("largest", "smallest", "most_colours", "unique_colour", "unique_shape", "topmost", "bottommost",
             "leftmost", "rightmost", "densest_bbox", "sparsest_bbox", "most_frequent_shape")

def select(objs, g, how):
    if not objs: return None
    key = {
        "largest": lambda o: len(o), "smallest": lambda o: -len(o),
        "topmost": lambda o: -bbox(o)[0], "bottommost": lambda o: bbox(o)[2],
        "leftmost": lambda o: -bbox(o)[1], "rightmost": lambda o: bbox(o)[3],
        "most_colours": lambda o: len({g[y][x] for y, x in o}),
        "densest_bbox": lambda o: len(o) / ((bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)),
        "sparsest_bbox": lambda o: -len(o) / ((bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)),
    }
    if how in key:
        vals = [key[how](o) for o in objs]; best = max(vals)
        hits = [o for o, v in zip(objs, vals) if v == best]
        return hits[0] if len(hits) == 1 else None
    def shape(o):
        r0, c0, _, _ = bbox(o); return tuple(sorted((y - r0, x - c0) for y, x in o))
    if how == "unique_colour":
        cs = Counter(g[o[0][0]][o[0][1]] for o in objs)
        hits = [o for o in objs if cs[g[o[0][0]][o[0][1]]] == 1]
        return hits[0] if len(hits) == 1 else None
    if how in ("unique_shape", "most_frequent_shape"):
        cs = Counter(shape(o) for o in objs)
        if how == "unique_shape":
            hits = [o for o in objs if cs[shape(o)] == 1]
        else:
            top = cs.most_common(1)[0]
            if len(cs) > 1 and cs.most_common(2)[1][1] == top[1]: return None
            hits = [o for o in objs if shape(o) == top[0]][:1]
        return hits[0] if len(hits) == 1 else None
    return None

def fam_crop(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) > H(i) or W(o) > W(i) or (H(o), W(o)) == (H(i), W(i)): return
    def all_fg(g):
        bg = bg_of(g); cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg]
        return crop(g, bbox(cells)) if cells else None
    yield ("crop:all-foreground", 2, all_fg)
    for diag, byc, how in product((False, True), (True, False), SELECTORS):
        def fn(g, diag=diag, byc=byc, how=how):
            bg = bg_of(g); ob = select(objects(g, bg, diag, byc), g, how)
            return crop(g, bbox(ob)) if ob else None
        yield (f"crop:object[{'8' if diag else '4'}{'' if byc else ',multi'}]:{how}", 3, fn)

def split_panels(g):
    """Split by full-length single-colour separator rows/cols; returns (panels, sep_colour) or None."""
    h, w = H(g), W(g)
    for sc in colours(g):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols: continue
        if len(rows) == h or len(cols) == w: continue
        rb = [-1] + rows + [h]; cb = [-1] + cols + [w]
        panels = []
        for a in range(len(rb) - 1):
            for b in range(len(cb) - 1):
                if rb[a + 1] - rb[a] > 1 and cb[b + 1] - cb[b] > 1:
                    panels.append(crop(g, (rb[a] + 1, cb[b] + 1, rb[a + 1] - 1, cb[b + 1] - 1)))
        if len(panels) >= 2 and all((H(p), W(p)) == (H(panels[0]), W(panels[0])) for p in panels):
            return panels, sc
    # no separator: equal halves
    if h % 2 == 0 and h >= 2: pass
    return None

def halves(g):
    h, w = H(g), W(g); out = []
    if w % 2 == 0: out.append([[r[:w // 2] for r in g], [r[w // 2:] for r in g]])
    if h % 2 == 0: out.append([g[:h // 2], g[h // 2:]])
    return out

BOOLS = {"and": lambda a, b: a and b, "or": lambda a, b: a or b, "xor": lambda a, b: a != b,
         "nor": lambda a, b: not a and not b, "a_not_b": lambda a, b: a and not b, "b_not_a": lambda a, b: b and not a}

def fam_panels(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) == (H(i), W(i)): return
    def getp(g, mode):
        if mode == "sep":
            sp = split_panels(g); return sp[0] if sp else None
        hs = halves(g); k = int(mode[-1])
        return hs[k] if len(hs) > k else None
    oc = [c for c in colours(o)]
    for mode in ("sep", "half0", "half1"):
        ps = getp(i, mode)
        if not ps or (H(ps[0]), W(ps[0])) != (H(o), W(o)): continue
        # boolean combination of two panels -> constant colour on true, bg on false
        if len(ps) == 2:
            for bn, f in BOOLS.items():
                for on_c in oc:
                    def fn(g, mode=mode, f=f, on_c=on_c):
                        p = getp(g, mode)
                        if not p or len(p) != 2: return None
                        a, b = p; bg = bg_of(g)
                        return [[on_c if f(a[y][x] != bg, b[y][x] != bg) else bg for x in range(W(a))] for y in range(H(a))]
                    yield (f"panels[{mode}]:{bn}->colour", 3, fn)
        # overlay: later panels painted over earlier (non-zero wins), both orders
        for order in ("fwd", "rev"):
            def ov(g, mode=mode, order=order):
                p = getp(g, mode)
                if not p: return None
                p = p if order == "fwd" else p[::-1]
                bgg = bg_of(g); out = [r[:] for r in p[0]]
                for q in p[1:]:
                    for y in range(H(q)):
                        for x in range(W(q)):
                            if q[y][x] != bgg: out[y][x] = q[y][x]
                return out
            yield (f"panels[{mode}]:overlay-{order}", 3, ov)
        for how in ("most_fg", "least_fg", "unique", "most_colours", "least_colours"):
            def sel(g, mode=mode, how=how):
                p = getp(g, mode)
                if not p: return None
                if how == "unique":
                    cs = Counter(str(q) for q in p); hits = [q for q in p if cs[str(q)] == 1]
                    return hits[0] if len(hits) == 1 else None
                bgg = bg_of(g)
                key = {"most_fg": lambda q: sum(v != bgg for r in q for v in r), "least_fg": lambda q: -sum(v != bgg for r in q for v in r),
                       "most_colours": lambda q: len(colours(q)), "least_colours": lambda q: -len(colours(q))}[how]
                vals = [key(q) for q in p]; hits = [q for q, v in zip(p, vals) if v == max(vals)]
                return hits[0] if len(hits) == 1 else None
            yield (f"panels[{mode}]:select-{how}", 3, sel)

SYMS = {"fh": fh, "fv": fv, "r180": r180, "T": T, "aT": at, "r90": r90}

def fam_symmetry(train):
    """Occluder colour (present in input, absent in output) is replaced by symmetric counterparts.
    Output either the repaired grid (same size) or the repaired patch under the occluder."""
    i, o = train[0]["input"], train[0]["output"]
    occ = [c for c in colours(i) if all(c in colours(p["input"]) and c not in colours(p["output"]) for p in train)]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in occ:
        for combo in (("fh",), ("fv",), ("fh", "fv"), ("fh", "fv", "r180"), ("T",), ("fh", "fv", "T", "aT", "r90", "r180")):
            def rep(g, c=c, combo=combo):
                out = [r[:] for r in g]
                for _ in range(3):
                    changed = False
                    for k in combo:
                        if k in ("T", "aT", "r90") and H(g) != W(g): return None
                        m = SYMS[k](out)
                        for y in range(H(g)):
                            for x in range(W(g)):
                                if out[y][x] == c and m[y][x] != c:
                                    out[y][x] = m[y][x]; changed = True
                    if not changed: break
                return None if any(v == c for r in out for v in r) else out
            if same:
                yield (f"symmetry-repair:{'+'.join(combo)}", 3, rep)
            else:
                def patch(g, c=c, rep=rep):
                    cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                    if not cells: return None
                    r = rep(g)
                    return crop(r, bbox(cells)) if r else None
                yield (f"symmetry-patch:{'+'.join(combo)}", 4, patch)

def fam_gravity(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    # (to frame, from frame): transform so that the motion direction becomes 'down'
    frames = {"down": (lambda g: g, lambda g: g), "up": (fv, fv), "right": (T, T), "left": (lambda g: T(fh(g)), lambda g: fh(T(g)))}
    for d, (to, back) in frames.items():
        def fn(g, to=to, back=back):
            bg = bg_of(g); out = [r[:] for r in to(g)]
            for x in range(W(out)):
                col = [out[y][x] for y in range(H(out)) if out[y][x] != bg]
                for y in range(H(out)): out[y][x] = bg
                for k, v in enumerate(reversed(col)): out[H(out) - 1 - k][x] = v
            return back(out)
        yield (f"gravity-cells:{d}", 2, fn)

def fam_colour_count(train):
    """Output is a 1-row/col strip or square filled from colour counts (sorted by frequency)."""
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) > 30: return
    def strip(g, orient):
        bg = bg_of(g); cs = Counter(v for r in g for v in r if v != bg)
        seq = [c for c, _ in cs.most_common()]
        if not seq: return None
        return [seq] if orient == "row" else [[c] for c in seq]
    yield ("colours-by-frequency:row", 3, lambda g: strip(g, "row"))
    yield ("colours-by-frequency:col", 3, lambda g: strip(g, "col"))

FAMILIES = (fam_geometric, fam_tile, fam_scale, fam_fractal, fam_crop, fam_panels, fam_symmetry, fam_gravity, fam_colour_count)


# ------------------------------------------------------------------ cycle 2 families
def fam_crop_colour(train):
    """Crop to the bbox of one colour's cells, or to the interior of that bbox (frame contents).
    The colour is induced: it must reproduce every training output."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) == (H(i), W(i)) or H(o) > H(i) or W(o) > W(i): return
    for c in sorted(colours(i)):
        for inner in (False, True):
            def fn(g, c=c, inner=inner):
                cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                if not cells: return None
                r0, c0, r1, c1 = bbox(cells)
                if inner: r0, c0, r1, c1 = r0 + 1, c0 + 1, r1 - 1, c1 - 1
                if r1 < r0 or c1 < c0: return None
                return crop(g, (r0, c0, r1, c1))
            yield (f"crop:colour-bbox{'-interior' if inner else ''}[c{c}]", 3, fn)

def fam_fill_enclosed(train):
    """Background regions not connected to the border are filled with an induced colour."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x])
    for c, _ in new.most_common(2):
        for diag in (False, True):
            def fn(g, c=c, diag=diag):
                bg = bg_of(g); h, w = H(g), W(g); out = [r[:] for r in g]
                seen = [[False] * w for _ in range(h)]
                st = [(y, x) for y in range(h) for x in range(w) if (y in (0, h - 1) or x in (0, w - 1)) and g[y][x] == bg]
                for y, x in st: seen[y][x] = True
                nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
                while st:
                    y, x = st.pop()
                    for dy, dx in nb:
                        yy, xx = y + dy, x + dx
                        if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] == bg:
                            seen[yy][xx] = True; st.append((yy, xx))
                for y in range(h):
                    for x in range(w):
                        if g[y][x] == bg and not seen[y][x]: out[y][x] = c
                return out
            yield (f"fill-enclosed[c{c}{',8' if diag else ''}]", 2, fn)

def fam_symmetrize(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for combo in (("fh",), ("fv",), ("fh", "fv", "r180"), ("T",), ("fh", "fv", "T", "aT", "r90", "r180", "r270")):
        def fn(g, combo=combo):
            bg = bg_of(g); out = [r[:] for r in g]
            for k in combo:
                if k in ("T", "aT", "r90", "r270") and H(g) != W(g): return None
                m = (SYMS.get(k) or D8[k])(g)
                for y in range(H(g)):
                    for x in range(W(g)):
                        if out[y][x] == bg and m[y][x] != bg: out[y][x] = m[y][x]
            return out
        yield (f"symmetrize:{'+'.join(combo)}", 2, fn)

def fam_object_filter(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for diag, byc in product((False, True), (True, False)):
        for k in (1, 2, 3):
            def noise(g, k=k, diag=diag, byc=byc):
                bg = bg_of(g); out = [r[:] for r in g]
                for ob in objects(g, bg, diag, byc):
                    if len(ob) <= k:
                        for y, x in ob: out[y][x] = bg
                return out
            yield (f"remove-objects-size<={k}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 2, noise)
        for how in ("largest", "smallest", "unique_colour", "unique_shape", "most_colours"):
            def keep(g, how=how, diag=diag, byc=byc):
                bg = bg_of(g); obs = objects(g, bg, diag, byc); ob = select(obs, g, how)
                if not ob: return None
                out = [[bg] * W(g) for _ in range(H(g))]
                for y, x in ob: out[y][x] = g[y][x]
                return out
            yield (f"keep-object[{'8' if diag else '4'}{'' if byc else ',multi'}]:{how}", 3, keep)

RAYS = {"orth": [(-1, 0), (1, 0), (0, -1), (0, 1)], "diag": [(-1, -1), (-1, 1), (1, -1), (1, 1)],
        "up": [(-1, 0)], "down": [(1, 0)], "left": [(0, -1)], "right": [(0, 1)], "vert": [(-1, 0), (1, 0)], "horiz": [(0, -1), (0, 1)]}

def fam_rays(train):
    """Cells of an induced colour emit rays (own colour) in induced directions, stopping at non-background
    (or passing over it)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    src = sorted({i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != bg_of(i)})
    for c in src:
        for dn, dirs in RAYS.items():
            for stop in (True, False):
                def fn(g, c=c, dirs=dirs, stop=stop):
                    bg = bg_of(g); out = [r[:] for r in g]
                    for y in range(H(g)):
                        for x in range(W(g)):
                            if g[y][x] != c: continue
                            for dy, dx in dirs:
                                yy, xx = y + dy, x + dx
                                while 0 <= yy < H(g) and 0 <= xx < W(g):
                                    if g[yy][xx] != bg:
                                        if stop: break
                                    else: out[yy][xx] = c
                                    yy += dy; xx += dx
                    return out
                yield (f"rays[c{c}]:{dn}{':stop' if stop else ':through'}", 3, fn)

def fam_connect(train):
    """Same-colour cells sharing a row/column are joined by a line of their colour (or an induced colour)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = [None] + [c for c, _ in Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]).most_common(1)]
    for axes in ("both", "row", "col"):
        for lc in new:
            def fn(g, axes=axes, lc=lc):
                bg = bg_of(g); out = [r[:] for r in g]
                pts = {}
                for y in range(H(g)):
                    for x in range(W(g)):
                        if g[y][x] != bg: pts.setdefault(g[y][x], []).append((y, x))
                for c, ps in pts.items():
                    for (y1, x1), (y2, x2) in product(ps, ps):
                        if (y1, x1) >= (y2, x2): continue
                        if y1 == y2 and axes in ("both", "row"):
                            for x in range(min(x1, x2) + 1, max(x1, x2)):
                                if out[y1][x] == bg: out[y1][x] = lc if lc is not None else c
                        if x1 == x2 and axes in ("both", "col"):
                            for y in range(min(y1, y2) + 1, max(y1, y2)):
                                if out[y][x1] == bg: out[y][x1] = lc if lc is not None else c
                return out
            yield (f"connect-same-colour:{axes}{'' if lc is None else f'[c{lc}]'}", 3, fn)
    for lc in new:
        def dfn(g, lc=lc):
            bg = bg_of(g); out = [r[:] for r in g]; pts = {}
            for y in range(H(g)):
                for x in range(W(g)):
                    if g[y][x] != bg: pts.setdefault(g[y][x], []).append((y, x))
            for c, ps in pts.items():
                for (y1, x1), (y2, x2) in product(ps, ps):
                    if (y1, x1) >= (y2, x2) or abs(y1 - y2) != abs(x1 - x2): continue
                    sy = 1 if y2 > y1 else -1; sx = 1 if x2 > x1 else -1
                    for k in range(1, abs(y2 - y1)):
                        if out[y1 + k * sy][x1 + k * sx] == bg: out[y1 + k * sy][x1 + k * sx] = lc if lc is not None else c
            return out
        yield (f"connect-same-colour:diagonal{'' if lc is None else f'[c{lc}]'}", 3, dfn)

PROPS = {"size": lambda o, g: len(o), "height": lambda o, g: bbox(o)[2] - bbox(o)[0] + 1, "width": lambda o, g: bbox(o)[3] - bbox(o)[1] + 1,
         "size_rank": None, "is_rect": lambda o, g: len(o) == (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1),
         "shape": lambda o, g: tuple(sorted((y - bbox(o)[0], x - bbox(o)[1]) for y, x in o)),
         "touches_border": lambda o, g: any(y in (0, H(g) - 1) or x in (0, W(g) - 1) for y, x in o),
         "n_holes": None}

def fam_recolour_by_property(train):
    """Each object is recoloured by a mapping property-value -> colour induced from training pairs.
    Refuses property values never seen in training."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for diag, byc in product((False, True), (True, False)):
        for pn in ("size", "height", "width", "size_rank", "is_rect", "shape", "touches_border", "n_holes", "adj_colours",
                   "colour_set", "shape_count", "extreme_size", "area_rank"):
            def pval(obs, g, pn=pn):
                if pn == "size_rank":
                    sizes = sorted({len(x) for x in obs}, reverse=True)
                    return [sizes.index(len(x)) for x in obs]
                if pn == "area_rank":
                    ar = lambda o: (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)
                    vals = sorted({ar(x) for x in obs}, reverse=True)
                    return [vals.index(ar(x)) for x in obs]
                if pn == "extreme_size":
                    mx = max(len(x) for x in obs); mn = min(len(x) for x in obs)
                    return ["max" if len(x) == mx else "min" if len(x) == mn else "mid" for x in obs]
                if pn == "shape_count":
                    sh = [PROPS["shape"](x, g) for x in obs]; cs = Counter(sh)
                    return [min(cs[s], 3) for s in sh]
                if pn == "colour_set":
                    return [frozenset(g[y][x] for y, x in o) for o in obs]
                if pn == "adj_colours":
                    bg = bg_of(g); out = []
                    for o in obs:
                        cs = set(o); col = set()
                        for y, x in o:
                            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                                yy, xx = y + dy, x + dx
                                if 0 <= yy < H(g) and 0 <= xx < W(g) and (yy, xx) not in cs and g[yy][xx] != bg: col.add(g[yy][xx])
                        out.append(frozenset(col))
                    return out
                if pn == "n_holes":
                    bg = bg_of(g); out = []
                    for o in obs:
                        r0, c0, r1, c1 = bbox(o); cs = set(o)
                        sub = [[1 if (y, x) in cs else 0 for x in range(c0 - 1, c1 + 2)] for y in range(r0 - 1, r1 + 2)]
                        holes = objects(sub, 1, False, True)   # regions of 0 (non-object) cells
                        out.append(sum(1 for hcells in holes if not any(y in (0, len(sub) - 1) or x in (0, len(sub[0]) - 1) for y, x in hcells)))
                    return out
                return [PROPS[pn](x, g) for x in obs]
            m = {}; ok = True
            for p in train:
                gi, go = p["input"], p["output"]
                if (H(gi), W(gi)) != (H(go), W(go)): ok = False; break
                bg = bg_of(gi); obs = objects(gi, bg, diag, byc)
                for ob, v in zip(obs, pval(obs, gi)):
                    cs = {go[y][x] for y, x in ob}
                    if len(cs) != 1 or m.setdefault(v, cs.pop()) != go[ob[0][0]][ob[0][1]]: ok = False; break
                if not ok: break
            if not ok or not m: continue
            def fn(g, m=m, diag=diag, byc=byc, pval=pval):
                bg = bg_of(g); out = [r[:] for r in g]; obs = objects(g, bg, diag, byc)
                for ob, v in zip(obs, pval(obs, g)):
                    if v not in m: return None
                    for y, x in ob: out[y][x] = m[v]
                return out
            yield (f"recolour-by-{pn}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_panels_multi(train):
    i, o = train[0]["input"], train[0]["output"]
    sp = split_panels(i)
    if not sp or len(sp[0]) < 3 or (H(sp[0][0]), W(sp[0][0])) != (H(o), W(o)): return
    for rule in ("all", "any", "one", "none"):
        for on_c in sorted(colours(o)):
            def fn(g, rule=rule, on_c=on_c):
                s = split_panels(g)
                if not s: return None
                ps = s[0]; bg = bg_of(g)
                if any((H(p), W(p)) != (H(ps[0]), W(ps[0])) for p in ps): return None
                def f(y, x):
                    n = sum(p[y][x] != bg for p in ps)
                    return {"all": n == len(ps), "any": n > 0, "one": n == 1, "none": n == 0}[rule]
                return [[on_c if f(y, x) else bg for x in range(W(ps[0]))] for y in range(H(ps[0]))]
            yield (f"panels[sep]:{rule}-of-{len(sp[0])}->colour", 3, fn)

RESHAPERS = ("tile", "crop", "panels", "upscale", "downscale", "fractal", "symmetry-patch", "symmetry-offset-patch", "kronecker")

def compose_dihedral(base):
    """Any output-reshaping program followed by a dihedral transform (one extra step)."""
    for name, cost, fn in base:
        if not name.startswith(RESHAPERS): continue
        for k, f in D8.items():
            if k == "id": continue
            yield (name + ">" + k, cost + 1, lambda g, fn=fn, f=f: (lambda r: f(r) if r else None)(fn(g)))

def fam_codex(train):
    try:
        import codex_ops
    except Exception:
        return
    yield from codex_ops.programs(train)

FAMILIES = FAMILIES + (fam_crop_colour, fam_fill_enclosed, fam_symmetrize, fam_object_filter, fam_rays, fam_connect, fam_recolour_by_property, fam_panels_multi, fam_codex)

# ------------------------------------------------------------------ cycle 9 families (motivated by training + eval half A)
def _sym_transforms(h, w):
    """Candidate symmetries with arbitrary axis/centre positions: (name, f(y,x)->(y,x))."""
    out = []
    for s in range(w // 2, w + w // 2):           # horizontal mirror x -> s - x  (axis at s/2)
        out.append((f"mirror-x@{s}", lambda y, x, s=s: (y, s - x)))
    for s in range(h // 2, h + h // 2):
        out.append((f"mirror-y@{s}", lambda y, x, s=s: (s - y, x)))
    for d in range(-3, 4):                        # transpose about an offset main diagonal
        out.append((f"transpose@{d}", lambda y, x, d=d: (x - d, y + d)))
    return out

def infer_symmetries(g, occ, min_agree=0.98, min_pairs=0.3):
    h, w = H(g), W(g); acc = []
    for name, f in _sym_transforms(h, w):
        agree = tot = 0
        for y in range(h):
            for x in range(w):
                if g[y][x] == occ: continue
                yy, xx = f(y, x)
                if 0 <= yy < h and 0 <= xx < w and g[yy][xx] != occ:
                    tot += 1; agree += g[y][x] == g[yy][xx]
        if tot >= min_pairs * h * w and agree >= min_agree * tot:
            acc.append((name, f))
    return acc

def offset_repair(g, occ):
    syms = infer_symmetries(g, occ)
    if not syms: return None
    out = [r[:] for r in g]; h, w = H(g), W(g)
    for _ in range(4):
        changed = False
        for y in range(h):
            for x in range(w):
                if out[y][x] != occ: continue
                for _, f in syms:
                    yy, xx = f(y, x)
                    if 0 <= yy < h and 0 <= xx < w and out[yy][xx] != occ:
                        out[y][x] = out[yy][xx]; changed = True; break
        if not changed: break
    return None if any(v == occ for r in out for v in r) else out

def fam_symmetry_offset(train):
    i, o = train[0]["input"], train[0]["output"]
    occ = [c for c in colours(i) if all(c in colours(p["input"]) and c not in colours(p["output"]) for p in train)]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in occ:
        if same:
            yield (f"symmetry-offset-repair[c{c}]", 4, lambda g, c=c: offset_repair(g, c))
        else:
            def patch(g, c=c):
                cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == c]
                if not cells: return None
                r = offset_repair(g, c)
                return crop(r, bbox(cells)) if r else None
            yield (f"symmetry-offset-patch[c{c}]", 4, patch)

def fam_object_symmetrize(train):
    """Each object keeps only cells whose mirror partner (about the object's best axis) is present."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for axis in ("x", "y"):
        for diag, byc in product((True, False), (True, False)):
            def fn(g, axis=axis, diag=diag, byc=byc):
                bg = bg_of(g); out = [r[:] for r in g]
                for ob in objects(g, bg, diag, byc):
                    cells = set(ob); r0, c0, r1, c1 = bbox(ob); best = None
                    lo, hi = (2 * c0, 2 * c1) if axis == "x" else (2 * r0, 2 * r1)
                    for s in range(lo, hi + 1):
                        keep = {(y, x) for y, x in cells if ((y, s - x) if axis == "x" else (s - y, x)) in cells and
                                g[y][x] == g[(y if axis == "x" else s - y)][(s - x if axis == "x" else x)]}
                        if best is None or len(keep) > len(best): best = keep
                    for y, x in cells - (best or set()): out[y][x] = bg
                return out
            yield (f"object-symmetrize:{axis}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_kronecker(train):
    """Two objects: a single-colour mask and a pattern tile; output places the tile at each (non-)mask cell."""
    i, o = train[0]["input"], train[0]["output"]
    for inv in (False, True):
        for pick in ("mono_mask", "small_mask"):
            def fn(g, inv=inv, pick=pick):
                bg = bg_of(g); obs = objects(g, bg, True, False)
                if len(obs) != 2: return None
                a, b = [crop(g, bbox(x)) for x in obs]
                if pick == "mono_mask":
                    ma = len(colours(a) - {bg}) == 1; mb = len(colours(b) - {bg}) == 1
                    if ma == mb: return None
                    mask, tile = (a, b) if ma else (b, a)
                else:
                    mask, tile = (a, b) if H(a) * W(a) <= H(b) * W(b) else (b, a)
                out = []
                for y in range(H(mask)):
                    blk = [tile if ((mask[y][x] != bg) != inv) else [[bg] * W(tile) for _ in range(H(tile))] for x in range(W(mask))]
                    for rr in range(H(tile)): out.append(sum((q[rr] for q in blk), []))
                return out
            yield (f"kronecker:{pick}{':inverse' if inv else ''}", 3, fn)

def frame_interior(g):
    """Interior of the largest rectangle whose border cells all share one non-background colour."""
    bg = bg_of(g); best = None
    for ob in objects(g, bg, False, True):
        r0, c0, r1, c1 = bbox(ob)
        if r1 - r0 < 2 or c1 - c0 < 2: continue
        col = g[ob[0][0]][ob[0][1]]
        border = [(r0, x) for x in range(c0, c1 + 1)] + [(r1, x) for x in range(c0, c1 + 1)] + [(y, c0) for y in range(r0, r1 + 1)] + [(y, c1) for y in range(r0, r1 + 1)]
        if all(g[y][x] == col for y, x in border):
            area = (r1 - r0) * (c1 - c0)
            if best is None or area > best[0]: best = (area, (r0 + 1, c0 + 1, r1 - 1, c1 - 1))
    return crop(g, best[1]) if best else None

def fam_frame_crop(train):
    i, o = train[0]["input"], train[0]["output"]
    if H(o) >= H(i) and W(o) >= W(i): return
    yield ("crop:frame-interior", 2, frame_interior)

def fam_outline(train):
    """Background cells adjacent (4 or 8) to objects of an induced colour are painted an induced colour."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    new = Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x])
    srcs = sorted(colours(i) - {bg_of(i)})
    for c, _ in new.most_common(2):
        for s in srcs + [None]:
            for diag in (False, True):
                def fn(g, c=c, s=s, diag=diag):
                    bg = bg_of(g); out = [r[:] for r in g]
                    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
                    for y in range(H(g)):
                        for x in range(W(g)):
                            if g[y][x] != bg: continue
                            for dy, dx in nb:
                                yy, xx = y + dy, x + dx
                                if 0 <= yy < H(g) and 0 <= xx < W(g) and g[yy][xx] != bg and (s is None or g[yy][xx] == s):
                                    out[y][x] = c; break
                    return out
                yield (f"outline[c{c}{'' if s is None else f',src{s}'}{',8' if diag else ''}]", 3, fn)

def fam_period_extend(train):
    """Rows (or columns) containing a partial periodic sequence are continued periodically across the grid."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for axis in ("row", "col"):
        for mode in ("from_first", "full"):
            def fn(g, axis=axis, mode=mode):
                gg = g if axis == "row" else T(g); bg = bg_of(g); out = [r[:] for r in gg]
                for y, row in enumerate(gg):
                    idx = [x for x, v in enumerate(row) if v != bg]
                    if len(idx) < 2: continue
                    a, b = idx[0], idx[-1]; seg = row[a:b + 1]
                    for p in range(1, len(seg) + 1):   # smallest period consistent with the observed segment
                        if all(seg[k] == seg[k % p] for k in range(len(seg))) and p < len(seg): break
                    else:
                        continue
                    start = a if mode == "from_first" else a % p
                    for x in range(start, len(row)):
                        out[y][x] = seg[(x - a) % p]
                return out if axis == "row" else T(out)
            yield (f"period-extend:{axis}:{mode}", 3, fn)
        def prog(g, axis=axis):
            """First two marks set the step; marks repeat up to the last mark, in the last mark's colour."""
            gg = g if axis == "row" else T(g); bg = bg_of(g); out = [r[:] for r in gg]
            for y, row in enumerate(gg):
                idx = [x for x, v in enumerate(row) if v != bg]
                if len(idx) < 3: continue
                a, b, last = idx[0], idx[1], idx[-1]; p = b - a
                if p <= 0 or (last - a) % p: continue
                for x in range(a, last + 1, p): out[y][x] = row[last]
            return out if axis == "row" else T(out)
        yield (f"progression:{axis}", 3, prog)
    def prog_row(row, bg):
        """Two same-colour marks set a step; continue to the odd-coloured last mark (in its colour), else to the edge."""
        idx = [x for x, v in enumerate(row) if v != bg]
        if len(idx) < 2 or row[idx[0]] != row[idx[1]]: return None
        a, p = idx[0], idx[1] - idx[0]
        rest = [x for x in idx[2:] if row[x] != row[a]]
        if any(row[x] == row[a] for x in idx[2:]): return None
        if len(rest) > 1: return None
        keep_other = False
        if rest and (rest[0] - a) % p == 0:
            end, col = rest[0], row[rest[0]]
        else:                                   # odd mark off the lattice (or none): continue to the edge
            end, col, keep_other = len(row) - 1, row[a], bool(rest)
        new = list(row)
        for x in range(a, end + 1, p):
            if keep_other and row[x] != bg and row[x] != row[a]: continue
            new[x] = col
        return new
    def prog_row_bi(row, bg):
        """The anchoring same-colour pair sits at a grid edge; progress away from that edge."""
        if row and row[0] != bg:
            return prog_row(row, bg)
        if row and row[-1] != bg:
            r = prog_row(row[::-1], bg)
            return r[::-1] if r is not None else None
        return None
    def prog_auto(g):
        bg = bg_of(g)
        cand = {}
        for ax, gg in (("r", g), ("c", T(g))):
            rows = [prog_row_bi(r, bg) for r in gg]
            cand[ax] = (sum(1 for r in rows if r is not None), gg, rows)
        (nr, _, _), (nc, _, _) = cand["r"], cand["c"]
        if nr == nc: return None
        ax = "r" if nr > nc else "c"; _, gg, rows = cand[ax]
        out = [r if r is not None else list(o) for r, o in zip(rows, gg)]
        return out if ax == "r" else T(out)
    yield ("progression:auto-axis", 4, prog_auto)

FAMILIES = FAMILIES + (fam_symmetry_offset, fam_object_symmetrize, fam_kronecker, fam_frame_crop, fam_outline, fam_period_extend)

# ------------------------------------------------------------------ cycle 10b families
def key_pairs(g, bg):
    """In-grid colour key: isolated 2-cell objects made of two different colours define a->b."""
    m = {}; keycells = set()
    for ob in objects(g, bg, False, False):
        if len(ob) != 2: continue
        (y1, x1), (y2, x2) = sorted(ob)
        a, b = g[y1][x1], g[y2][x2]
        if a == b: continue
        if m.get(a, b) != b: return None, None
        m[a] = b; keycells |= set(ob)
    return (m, keycells) if m else (None, None)

def fam_key_recolour(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    for rev in (False, True):
        def fn(g, rev=rev):
            bg = bg_of(g); m, kc = key_pairs(g, bg)
            if not m: return None
            if rev: m = {v: k for k, v in m.items()}
            return [[m.get(v, v) if (y, x) not in kc else v for x, v in enumerate(r)] for y, r in enumerate(g)]
        yield (f"key-recolour{':reverse' if rev else ''}", 3, fn)

def fam_marker_object(train):
    """Crop / keep the (multi-colour) object that contains a cell of an induced marker colour."""
    i, o = train[0]["input"], train[0]["output"]
    same = (H(o), W(o)) == (H(i), W(i))
    for c in sorted(colours(i) - {bg_of(i)}):
        for diag in (True, False):
            def pick(g, c=c, diag=diag):
                bg = bg_of(g); hits = [ob for ob in objects(g, bg, diag, False) if any(g[y][x] == c for y, x in ob)]
                return hits[0] if len(hits) == 1 else None
            if same:
                def keep(g, pick=pick):
                    ob = pick(g)
                    if not ob: return None
                    bg = bg_of(g); out = [[bg] * W(g) for _ in range(H(g))]
                    for y, x in ob: out[y][x] = g[y][x]
                    return out
                yield (f"keep-object-with[c{c}{',8' if diag else ''}]", 3, keep)
            else:
                yield (f"crop:object-with[c{c}{',8' if diag else ''}]", 3, lambda g, pick=pick: (lambda ob: crop(g, bbox(ob)) if ob else None)(pick(g)))

def fam_count_output(train):
    """Output size equals a count of objects; filled with an induced colour or the counted colour."""
    shapes = {(H(p["output"]), W(p["output"])) for p in train}
    outs_c = [colours(p["output"]) for p in train]
    if not all(len(c) == 1 for c in outs_c): return
    for diag, byc in product((False, True), (True, False)):
        for orient in ("row", "col"):
            for colour_rule in ("most_common_object_colour", "constant"):
                const = list(outs_c[0])[0]
                def fn(g, diag=diag, byc=byc, orient=orient, colour_rule=colour_rule, const=const):
                    bg = bg_of(g); obs = objects(g, bg, diag, byc)
                    if not obs: return None
                    col = const if colour_rule == "constant" else Counter(g[o[0][0]][o[0][1]] for o in obs).most_common(1)[0][0]
                    n = len(obs) if colour_rule == "constant" else sum(1 for o in obs if g[o[0][0]][o[0][1]] == col)
                    return [[col] * n] if orient == "row" else [[col] for _ in range(n)]
                yield (f"count-objects[{'8' if diag else '4'}{'' if byc else ',multi'}]:{orient}:{colour_rule}", 3, fn)

def fam_colour_select_fill(train):
    """Output has a fixed shape (same in all pairs) filled with one colour chosen by a rule."""
    shapes = {(H(p["output"]), W(p["output"])) for p in train}
    if len(shapes) != 1 or not all(len(colours(p["output"])) == 1 for p in train): return
    h, w = shapes.pop()
    rules = {"most_frequent_fg": lambda g, bg: Counter(v for r in g for v in r if v != bg).most_common(1)[0][0],
             "least_frequent_fg": lambda g, bg: Counter(v for r in g for v in r if v != bg).most_common()[-1][0],
             "largest_object": lambda g, bg: (lambda ob: g[ob[0][0]][ob[0][1]])(max(objects(g, bg, False, True), key=len)),
             "smallest_object": lambda g, bg: (lambda ob: g[ob[0][0]][ob[0][1]])(min(objects(g, bg, False, True), key=len)),
             "most_objects": lambda g, bg: Counter(g[o[0][0]][o[0][1]] for o in objects(g, bg, False, True)).most_common(1)[0][0]}
    for rn, f in rules.items():
        def fn(g, f=f):
            bg = bg_of(g); c = f(g, bg)
            return [[c] * w for _ in range(h)]
        yield (f"fill-{h}x{w}-with:{rn}", 3, fn)

FAMILIES = FAMILIES + (fam_key_recolour, fam_marker_object, fam_count_output, fam_colour_select_fill)

def candidates(train):
    out = []
    for fam in FAMILIES:
        try:
            out.extend(fam(train))
        except Exception:
            continue
    out = out + list(compose_dihedral(out))
    out.sort(key=lambda x: x[1])
    return out

def run(fn, g):
    try:
        r = fn(g)
        if r is not None and not isinstance(r, list): r = [list(map(int, row)) for row in r]
        elif r is not None: r = [list(map(int, row)) for row in r]
        if r is None or not r or not r[0] or H(r) > 30 or W(r) > 30: return None
        return r
    except Exception:
        return None

# ------------------------------------------------------------------ cycle 11 families: object motion
DIRV = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}

def static_colours(train):
    """Colours whose cells never change between input and output in any training pair (obstacles/anchors)."""
    st = None
    for p in train:
        i, o = p["input"], p["output"]
        if (H(i), W(i)) != (H(o), W(o)): return set()
        cs = {v for r in i for v in r}
        ok = {c for c in cs if all(o[y][x] == c for y in range(H(i)) for x in range(W(i)) if i[y][x] == c)}
        st = ok if st is None else st & ok
    return st or set()

def slide(g, movers, statics, dy, dx, bg):
    """Move each mover object rigidly by (dy,dx) steps until the next step would leave the grid or hit
    a non-background cell that is not part of itself. Movers are processed from the leading edge."""
    out = [r[:] for r in g]
    key = lambda ob: -max(y * dy + x * dx for y, x in ob)
    for ob in sorted(movers, key=key):
        col = {(y, x): out[y][x] for y, x in ob}
        for y, x in ob: out[y][x] = bg
        cur = list(ob); k = 0
        while True:
            nxt = [(y + dy, x + dx) for y, x in cur]
            if any(not (0 <= y < H(g) and 0 <= x < W(g)) or out[y][x] != bg for y, x in nxt): break
            cur = nxt; k += 1
        for (y0, x0), (y, x) in zip(ob, cur): out[y][x] = col[(y0, x0)]
    return out

def fam_object_gravity(train):
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    st = static_colours(train)
    for d, (dy, dx) in DIRV.items():
        for diag, byc in product((False, True), (True, False)):
            def fn(g, dy=dy, dx=dx, diag=diag, byc=byc):
                bg = bg_of(g); obs = objects(g, bg, diag, byc)
                movers = [ob for ob in obs if not all(g[y][x] in st for y, x in ob)]
                if not movers or len(movers) == len(obs) and st: return None
                return slide(g, movers, st, dy, dx, bg)
            yield (f"object-gravity:{d}[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

def fam_move_to_target(train):
    """Each mover object moves in a straight line toward the static (anchor) object it faces, stopping on contact."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) != (H(i), W(i)): return
    st = static_colours(train)
    if not st: return
    for diag, byc in product((False, True), (True, False)):
        def fn(g, diag=diag, byc=byc):
            bg = bg_of(g); obs = objects(g, bg, diag, byc)
            anchors = [ob for ob in obs if all(g[y][x] in st for y, x in ob)]
            movers = [ob for ob in obs if ob not in anchors]
            if not anchors or not movers: return None
            out = [r[:] for r in g]
            for ob in movers:
                r0, c0, r1, c1 = bbox(ob); best = None
                for an in anchors:
                    a0, b0, a1, b1 = bbox(an)
                    if b0 <= c1 and c0 <= b1:           # vertical overlap -> move up/down
                        d = (1, 0) if a0 > r1 else (-1, 0) if a1 < r0 else None
                        dist = a0 - r1 if a0 > r1 else r0 - a1
                    elif a0 <= r1 and r0 <= a1:         # horizontal overlap -> move left/right
                        d = (0, 1) if b0 > c1 else (0, -1) if b1 < c0 else None
                        dist = b0 - c1 if b0 > c1 else c0 - b1
                    else:
                        continue
                    if d and (best is None or dist < best[0]): best = (dist, d)
                if best: out = slide(out, [ob], st, best[1][0], best[1][1], bg)
            return out
        yield (f"move-to-anchor[{'8' if diag else '4'}{'' if byc else ',multi'}]", 3, fn)

FAMILIES = FAMILIES + (fam_object_gravity, fam_move_to_target)

def residual_steps(train, k=4):
    """Same-size first steps that move every pair strictly closer to its output and never break a correct cell."""
    outs = [p["output"] for p in train]
    if any((H(p["input"]), W(p["input"])) != (H(o), W(o)) for p, o in zip(train, outs)): return []
    scored = []
    for name, cost, fn in candidates(train):
        if name.startswith(RESHAPERS) or ">" in name: continue
        gain = 0; ok = True; preds = []
        for p, o in zip(train, outs):
            i = p["input"]; r = run(fn, i)
            if r is None or (H(r), W(r)) != (H(o), W(o)) or r == i: ok = False; break
            broke = any(i[y][x] == o[y][x] and r[y][x] != o[y][x] for y in range(H(o)) for x in range(W(o)))
            fixed = sum(i[y][x] != o[y][x] and r[y][x] == o[y][x] for y in range(H(o)) for x in range(W(o)))
            if broke or fixed == 0: ok = False; break
            gain += fixed; preds.append(r)
        if ok: scored.append((-gain, cost, name, fn, preds))
    scored.sort(key=lambda x: (x[0], x[1]))
    return scored[:k]

def search2(task, max_programs=4):
    """Two-step programs: a residual first step, then a full single-step search on its outputs."""
    res = []
    for _, cost, name, fn, preds in residual_steps(task["train"]):
        sub = {"train": [{"input": r, "output": p["output"]} for r, p in zip(preds, task["train"])],
               "test": [{"input": run(fn, t["input"])} for t in task["test"]]}
        if any(t["input"] is None for t in sub["test"]): continue
        for r in search(sub, max_programs=2, allow2=False):
            res.append({"program": name + " ; " + r["program"], "cost": cost + r["cost"], "preds": r["preds"]})
        if len(res) >= max_programs: break
    return res

def search(task, max_programs=6, allow2=True):
    """Return verified programs: list of dicts {program, cost, preds}. Cheapest first; with and without colour map."""
    train = task["train"]; outs = [p["output"] for p in train]
    found = []
    # identity + colour map alone (pure recolouring)
    ident = [p["input"] for p in train]
    m = fit_cmap(ident, outs)
    if m and any(k != v for k, v in m.items()) and all(apply_cmap(a, m) == b for a, b in zip(ident, outs)):
        found.append(("colour-map", 1, lambda g, m=m: apply_cmap(g, m)))
    for name, cost, fn in candidates(train):
        preds = []
        for p, o in zip(train, outs):
            r = run(fn, p["input"])
            if r is None or H(r) != H(o) or W(r) != W(o): preds = None; break
            if not preds and fit_cmap([r], [o]) is None: preds = None; break
            preds.append(r)
        if not preds: continue
        if preds == outs:
            found.append((name, cost, fn))
        else:
            m = fit_cmap(preds, outs)
            if m and all(apply_cmap(a, m) == b for a, b in zip(preds, outs)):
                found.append((name + "+colour-map", cost + 1, lambda g, fn=fn, m=m: apply_cmap(run(fn, g), m)))
        if len(found) >= max_programs: break
    res, seen = [], set()
    for name, cost, fn in sorted(found, key=lambda x: x[1]):
        preds = [run(fn, t["input"]) for t in task["test"]]
        if any(p is None for p in preds): continue
        k = str(preds)
        if k in seen: continue
        seen.add(k); res.append({"program": name, "cost": cost, "preds": preds})
    if not res and allow2:
        res = search2(task)
    if not res and allow2 and not _IN_COMPOSE[0]:
        res = compose_fallback(task)
    if not res and allow2 and not _IN_COMPOSE[0]:
        res = nearmiss_fallback(task)
    return res

# ------------------------------------------------------------------ cycle 16: composition engines as a last fallback
# Deterministic work budgets inside each engine (no clocks), so Kaggle parity is preserved.
COMPOSE_ENGINES = ("compose_objmap2", "compose_lift", "compose_ctx")
NEARMISS_MAX_CELLS = 2400          # deterministic size cap (sum of training-input cells) for the near-miss stage
def nearmiss_fallback(task, max_attempts=2):
    """Best-effort attempts for a task with NO verified program: programs verified on all training pairs but one
    (leave-one-pair-out library search, k >= 3 pairs).  Ranked by how many subsets propose the same test prediction,
    then by cell accuracy on the left-out pair, then cost.  Names are prefixed 'nearmiss:'; these are guesses,
    used only to fill otherwise-empty attempt slots.  Deterministic: no clocks; bounded by the size cap."""
    train = task["train"]; k = len(train)
    if k < 3 or sum(len(p["input"]) * len(p["input"][0]) for p in train) > NEARMISS_MAX_CELLS:
        return []
    _IN_COMPOSE[0] = True
    cand = {}
    try:
        for i in range(k):
            sub = {"train": train[:i] + train[i + 1:], "test": [{"input": train[i]["input"]}] + task["test"]}
            try:
                rs = search(sub, max_programs=3, allow2=False)
            except Exception:
                rs = []
            o = train[i]["output"]
            for r in rs:
                p0 = r["preds"][0]
                acc = (sum(x == y for ra, rb in zip(p0, o) for x, y in zip(ra, rb)) / (H(o) * W(o))
                       if p0 is not None and H(p0) == H(o) and W(p0) == W(o) else 0.0)
                key = str(r["preds"][1:])
                c = cand.setdefault(key, [0, 0.0, r["cost"], r["program"], r["preds"][1:]])
                c[0] += 1; c[1] = max(c[1], acc)
    finally:
        _IN_COMPOSE[0] = False
    ranked = sorted(cand.values(), key=lambda c: (-c[0], -c[1], c[2], c[3]))[:max_attempts]
    return [{"program": "nearmiss:" + c[3], "cost": c[2] + 10, "preds": c[4]} for c in ranked]
_IN_COMPOSE = [False]
def compose_fallback(task):
    _IN_COMPOSE[0] = True
    try:
        for name in COMPOSE_ENGINES:
            try:
                eng = __import__(name)
                res = eng.SEARCH(task)
            except Exception:
                res = None
            if res:
                return [dict(r, program=name.split('_', 1)[1] + ':' + r['program']) for r in res[:3]]
        return []
    finally:
        _IN_COMPOSE[0] = False

# ------------------------------------------------------------------ cycle 12: own primitives (latent/)
import importlib as _il
for _m in ("fam_lines", "fam_move_by_vector", "fam_crop_by_score", "fam_summarise_ranked_bars",
           "fam_move_slide_until_contact", "fam_fill_region_by_property", "fam_stamp_local_stencil",
           "fam_ray_diagonal_from_object", "fam_stamp_complete_partial_matches", "fam_extend_periodic_fill",
           "fam_ray_orthogonal_directed", "fam_summarise_panel_to_pixel", "fam_summarise_count_to_glyph",
           "fam_ray_walker_turn_split", "fam_move_to_position", "fam_recolour_by_distance_layers", "fam_move_toward_anchor",
           "fam_stamp_template_at_markers", "fam_transform_kronecker", "fam_crop_fit_by_anchor", "fam_segment_pack_pieces",
           "fam_summarise_concentric_rings", "fam_combine_boolean_panels",
           "fam_stamp_copy_to_congruent_target", "fam_recolour_objects", "fam_recolour_by_colour_map",
           "fam_connect_aligned_pair", "fam_extend_repeat_stamp", "fam_transform_objects", "fam_fill_bg_windows",
           "prior_topology", "prior_geometry", "prior_combinatorics",
           "prior_optics", "prior_mechanics", "prior_fluids", "prior_action", "prior_arithmetic"):
    try:
        FAMILIES = FAMILIES + tuple(_il.import_module(_m).FAMILIES)
    except Exception:
        pass


In [ ]:
%%writefile /kaggle/working/probe/prior_action.py
"""ACTION prior: principle of least action and conservation laws on the grid.

Laws (the prior, stated before any task)
----------------------------------------
L1  Least action.  A curve drawn between terminals is a stationary (minimal) point of the action
        S = length  >>  turns  >>  tie terms            (lexicographic)
    length is octile (orthogonal step 1, diagonal step sqrt 2 = Euclidean step length) on the 4- or 8-lattice,
    obstacles (every non-background cell that is not an end point) have infinite potential, and among the
    shortest curves the straightest one is realised (inertia: a direction change costs).  The remaining tie
    terms are generic, induced per task: first leg along an axis at the source; turns as late as possible
    (keep going straight until forced); lateral steps to a preferred side; leave / enter a terminal that stands
    on a wall perpendicular to that wall.  A curve that passes through a terminal keeps its heading.
    If the minimum is degenerate the prior refuses (or, as an induced option, draws the union of all minimal
    curves - e.g. both one-turn king geodesics form a parallelogram).
L2  Least total length of a network.  Several terminals are joined by the network of minimal total length:
    rectilinear Steiner comb (one trunk at the median coordinate, fewest branches), minimal spanning tree of
    geodesics, minimal-total-length perfect matching (least-action assignment), or a chain / star along an
    induced order of the terminal colours.  A free end (a lone source) is joined to the far boundary.
L3  Conservation.  The cell count of every colour is invariant (checked on all training pairs).  A transformation
    is then a rearrangement whose target is fixed by the conserved quantity: the N mobile cells of a colour are
    poured into the unique container whose capacity (fluid-retaining cells, full or checkerboard lattice)
    equals N.
L4  Equilibrium.  Conserved mass settles into its minimal-potential configuration: a granular heap at the
    floor under its column with a 45-degree angle of repose (rows 1, 3, 5, ... cells, order of the cells kept);
    grains of a mobile colour fall until supported, the other colours being static supports (sand); a poured
    fluid fills the retaining cells of its container (gravity direction induced).

Search space (all parameters induced from the training pairs by internal fitting; each yielded program is
re-verified on every pair by the harness; no task constants)
  act-geodesic   preconditions: same shape, only background cells change.
                 terminals: components of one colour c (c unchanged and touching the drawn cells) | all single
                 cells | role 'pair2' (the unique colour with two components) | role 'edge' (components standing
                 on the border) | every single-colour component (4- / 8-connected with the metric), per colour.
                 curve colour: the unique new colour, or the terminal's colour.   metric N4 | N8.
                 network: pair | comb | chain / star (colour order) | emit (to the far boundary; heading away
                 from the wall the source stands on, or a fixed direction) | each colour class separately
                 (2 members -> geodesic, >2 -> spanning tree or matching, lone -> emit or keep; classes whose
                 least-action network is degenerate may be left alone, induced option).
                 tie rule: unique | union | first-h/v/d | late | late+side | perp.
  act-conserve   preconditions: same shape, colour histogram identical in every pair.
                 capacity[full|checker] x gravity (4 directions): containers = objects retaining fluid cells;
                 mobile = every other non-background cell; each mobile colour -> the container of equal capacity.
                 heap x gravity: every occupied column collapses into a 45-degree heap (count must be k^2).
                 sand[c] x gravity: grains of the mobile colour c (the colour whose cells move in every pair)
                 fall until supported; everything else is static.
Dropped / not kept: minimal-displacement assignment of dots to anchor sites (1b8318e3: the minimal total
displacement matching is degenerate and does not reproduce the outputs); hydrostatic settling of a liquid by
spill-over (28a6681f) - left to the FLUIDS prior.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import permutations, product
import heapq
from gdsl import H, W, bg_of, objects, bbox

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))
DIRS = {"N4": N4, "N8": N8}


def _same(train):
    return all((H(p["input"]), W(p["input"])) == (H(p["output"]), W(p["output"])) for p in train)


def _axis(d):
    dy, dx = d
    return "v" if dx == 0 else "h" if dy == 0 else "d"


# ------------------------------------------------------------------ L1: geodesics of minimal action
ORTH = 1000000
DIAG = 1414214                      # sqrt(2) in the same units: octile (Euclidean-step) length
MAXT = 1000


def geodesics(g, free, src, dst, metric, rule=(), d0=None, final_turn=True, cap=2, d1=None):
    """Curves of minimal action from cell set src to cell set dst (dst may lie outside the grid) through
    free cells.  Action is lexicographic: (length, number of turns, tie terms) where length is octile
    (orthogonal step 1, diagonal step sqrt 2) and the tie terms come from `rule`: ('first', axis) = first step
    along axis; ('late',) = turns as late as possible (inertia: keep going straight); ('side', d) = lateral
    steps prefer direction d (N4 only).  d0 = initial heading (a first step in another direction is a turn).
    The lexicographic action is encoded in one integer.  Returns (n_optimal (capped), one optimal path,
    union of optimal paths) or None."""
    h, w = H(g), W(g); nb = DIRS[metric]; dset = set(dst)
    first_ax = next((r[1] for r in rule if r[0] == "first"), None)
    late = any(r[0] == "late" for r in rule)
    side = next((r[1] for r in rule if r[0] == "side"), None)
    F = [[free(y, x) for x in range(w)] for y in range(h)]
    if metric == "N8":
        stepc = [(ORTH if d[0] == 0 or d[1] == 0 else DIAG) << 12 for d in nb]; TC = 1 << 2; FC = 1
        late = False; side = None
    else:
        stepc = [1 << 40] * 4; TC = 1 << 30; FC = 1 << 29
    opp = None if side is None else (-side[0], -side[1])
    sidec = [1 if (opp is not None and d == opp) else 0 for d in nb]
    firstc = [0 if first_ax is None or _axis(d) == first_ax else FC for d in nb]
    K = len(nb)
    def ext(c, k, k2):
        c2 = c + stepc[k2] + sidec[k2]
        if k is not None and k2 != k:
            c2 += TC
            if late: c2 += (MAXT - (c >> 40)) << 11
        return c2
    dist = {}; pq = []; pred = {}
    k0 = None if d0 is None else nb.index(d0)
    for (y, x) in src:
        for k in range(K):
            dy, dx = nb[k]; yy, xx = y + dy, x + dx
            if (yy, xx) in dset: return None               # terminals touch: nothing to draw
            if not (0 <= yy < h and 0 <= xx < w) or not F[yy][xx]: continue
            c = ext(firstc[k], k0, k)
            st = (yy, xx, k)
            if st not in dist or c < dist[st]:
                dist[st] = c; pred[st] = [None]; heapq.heappush(pq, (c, st))
            elif c == dist[st] and None not in pred[st]:
                pred[st].append(None)
    def endcost(st):
        y, x, k = st; m = None
        for k2 in range(K):
            dy, dx = nb[k2]
            if (y + dy, x + dx) in dset:
                cc = ext(dist[st], k if final_turn else None, k2)
                if d1 is not None and nb[k2] != d1: cc += TC
                if m is None or cc < m: m = cc
        return m
    best = None; ends = []
    while pq:
        c, st = heapq.heappop(pq)
        if c != dist.get(st): continue
        if best is not None and c > best: break
        ec = endcost(st)
        if ec is not None:
            if best is None or ec < best: best = ec; ends = [st]
            elif ec == best: ends.append(st)
        y, x, k = st
        for k2 in range(K):
            dy, dx = nb[k2]; yy, xx = y + dy, x + dx
            if not (0 <= yy < h and 0 <= xx < w) or not F[yy][xx]: continue
            cc = ext(c, k, k2); s2 = (yy, xx, k2)
            o = dist.get(s2)
            if o is None or cc < o:
                dist[s2] = cc; pred[s2] = [st]; heapq.heappush(pq, (cc, s2))
            elif cc == o and st not in pred[s2]:
                pred[s2].append(st)
    if best is None: return None
    ends = [e for e in ends if endcost(e) == best]
    # number of optimal curves (capped): predecessors always have a strictly smaller action, so a pass over
    # the states in increasing action is a topological order of the predecessor DAG (no recursion)
    memo = {}
    for st in sorted(pred, key=dist.__getitem__):
        memo[st] = min(cap, sum(1 if p is None else memo.get(p, 0) for p in pred[st]))
    n = sum(memo[e] for e in ends)
    union = set(); stack = list(ends); seen = set()
    while stack:
        st = stack.pop()
        if st is None or st in seen: continue
        seen.add(st); union.add(st[:2]); stack.extend(pred[st])
    one = []; st = ends[0]
    while st is not None:
        one.append(st[:2]); st = pred[st][0]
    return min(n, cap), one, union


_GEO_CACHE = {}


def _geo(g, bg, src, dst, metric, rule=(), d0=None, final_turn=True, d1=None):
    """Cached minimal-action curves over the background cells of g (many configurations share edges)."""
    key = (tuple(map(tuple, g)), bg, tuple(src), tuple(dst), metric, rule, d0, final_turn, d1)
    if key not in _GEO_CACHE:
        if len(_GEO_CACHE) > 4000: _GEO_CACHE.clear()
        _GEO_CACHE[key] = geodesics(g, lambda y, x: g[y][x] == bg, src, dst, metric, rule, d0, final_turn, d1=d1)
    return _GEO_CACHE[key]


def _terminals(g, bg, tsel):
    """Terminal cell groups. tsel = ('col', c) | ('singles',) | ('comp',) (every single-colour component)."""
    if tsel[0] == "comp":                   # single-colour components, connectivity of the curve metric
        return objects(g, bg, tsel[1] == 8, True)
    if tsel[0] == "col":
        c = tsel[1]
        return [o for o in objects(g, bg, True, True) if g[o[0][0]][o[0][1]] == c]
    if tsel[0] == "pair2":                  # role: the unique colour class made of exactly two components
        comps = objects(g, bg, True, True); cnt = Counter(g[o[0][0]][o[0][1]] for o in comps)
        two = [c for c, n in cnt.items() if n == 2]
        return [o for o in comps if g[o[0][0]][o[0][1]] == two[0]] if len(two) == 1 else []
    if tsel[0] == "edge":                   # role: components standing on the grid border
        h, w = H(g), W(g)
        return [o for o in objects(g, bg, True, True) if any(y in (0, h - 1) or x in (0, w - 1) for y, x in o)]
    return [o for o in objects(g, bg, True, False) if len(o) == 1]


def _key(g, t):
    return g[t[0][0]][t[0][1]]


def _paint(out, cells, pcol, bg):
    for y, x in cells:
        if out[y][x] == bg: out[y][x] = pcol


def comb_steiner(pts):
    """Rectilinear Steiner comb of minimal action for points: minimal total length, then fewest branches
    (most terminals lying on the trunk).  None when the optimum is not unique."""
    cands = []
    for ax in ("v", "h"):
        a = sorted(p[1] if ax == "v" else p[0] for p in pts)      # coordinate orthogonal to trunk
        b = [p[0] if ax == "v" else p[1] for p in pts]
        n = len(a)
        for m in range(a[(n - 1) // 2], a[n // 2] + 1):
            cost = sum(abs(v - m) for v in a) + max(b) - min(b)
            cands.append((cost, -a.count(m), ax, m, min(b), max(b)))
    cands.sort()
    if len(cands) > 1 and cands[0][:2] == cands[1][:2]: return None
    cost, _, ax, m, b0, b1 = cands[0]
    cells = set()
    for t in range(b0, b1 + 1):
        cells.add((t, m) if ax == "v" else (m, t))
    for (y, x) in pts:
        if ax == "v":
            for xx in range(min(x, m), max(x, m) + 1): cells.add((y, xx))
        else:
            for yy in range(min(y, m), max(y, m) + 1): cells.add((yy, x))
    return cells


DIRV = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}
TIES = {"unique": (), "union": (), "perp": (), "first-h": (("first", "h"),), "first-v": (("first", "v"),),
        "first-d": (("first", "d"),), "late": (("late",),)}
for _n, _d in DIRV.items():
    TIES["late-" + _n] = (("late",), ("side", _d))


def _emit_dir(g, t, emit):
    """Heading of an emitting source: fixed, or away from the single grid edge the source touches."""
    if emit in DIRV: return DIRV[emit]
    h, w = H(g), W(g)
    edges = [d for d, hit in ((DIRV["down"], any(y == 0 for y, _ in t)), (DIRV["up"], any(y == h - 1 for y, _ in t)),
                              (DIRV["right"], any(x == 0 for _, x in t)), (DIRV["left"], any(x == w - 1 for _, x in t))) if hit]
    if len(edges) > 1 and len(t) > 1:        # a bar in a corner is attached to the edge it stands on
        r0, c0, r1, c1 = bbox(t)
        if c0 == c1 and r1 > r0: edges = [d for d in edges if d[1] == 0]
        elif r0 == r1 and c1 > c0: edges = [d for d in edges if d[0] == 0]
    return edges[0] if len(edges) == 1 else None


def mst_network(g, bg, grp, metric):
    """Minimal spanning network of geodesics joining the terminals of one class (least total length).
    The spanning tree over pairwise geodesic lengths must be unique; its edges are drawn one at a time, each
    edge only when its minimal-action curve is unique given the headings inherited from curves already drawn
    (a curve passing through a single-cell terminal keeps going straight: inertia)."""
    n = len(grp); nb = DIRS[metric]
    free = lambda y, x: g[y][x] == bg
    D = {}
    for i in range(n):
        for j in range(i + 1, n):
            d = _bfs_steps(g, free, grp[i], grp[j], nb)
            if d is None: return None
            D[(i, j)] = d
    parent = list(range(n))
    def find(a):
        while parent[a] != a: a = parent[a]
        return a
    tree = []
    for (i, j), d in sorted(D.items(), key=lambda kv: (kv[1], kv[0])):
        if find(i) != find(j):
            parent[find(i)] = find(j); tree.append((i, j))
    adj = {i: [] for i in range(n)}
    for i, j in tree: adj[i].append((j, D[(i, j)])); adj[j].append((i, D[(i, j)]))
    def maxw(u, v):                        # heaviest edge on the tree path u..v
        st = [(u, -1, 0)]; seen = {u}
        while st:
            a, pa, m = st.pop()
            if a == v: return m
            for b, wgt in adj[a]:
                if b not in seen: seen.add(b); st.append((b, a, max(m, wgt)))
        return None
    for (i, j), d in D.items():
        if (i, j) not in tree and d == maxw(i, j): return None          # spanning tree not unique
    heading = {}; cells_all = set(); todo = list(tree)
    while todo:
        drawn = False
        for e in list(todo):
            i, j = e; a, b = grp[i], grp[j]
            ha = heading.get(i) if len(a) == 1 else None
            hb = heading.get(j) if len(b) == 1 else None
            r = _geo(g, bg, a, b, metric, (), d0=ha, d1=None if hb is None else (-hb[0], -hb[1]))
            if r is None or r[0] != 1: continue
            path = r[1][::-1]                 # from the cell next to a to the cell next to b
            cells_all |= set(path)
            if len(a) == 1: heading[i] = (a[0][0] - path[0][0], a[0][1] - path[0][1])
            if len(b) == 1: heading[j] = (b[0][0] - path[-1][0], b[0][1] - path[-1][1])
            todo.remove(e); drawn = True; break
        if not drawn: return None
    return cells_all


def matching_network(g, bg, grp, metric, rule):
    """Least-action assignment: the terminals of a class are paired by the perfect matching of minimal total
    geodesic length (it must be unique); each pair is joined by its minimal-action curve."""
    n = len(grp)
    if n % 2 or n > 8: return None
    free = lambda y, x: g[y][x] == bg
    D = {}
    for i in range(n):
        for j in range(i + 1, n):
            D[(i, j)] = _bfs_steps(g, free, grp[i], grp[j], DIRS[metric])
    best = []
    def rec(left, acc, cost):
        if not left:
            best.append((cost, acc)); return
        i = left[0]
        for j in left[1:]:
            d = D[(i, j)]
            if d is None: continue
            rec([k for k in left if k not in (i, j)], acc + [(i, j)], cost + d)
    rec(list(range(n)), [], 0)
    if not best: return None
    best.sort(key=lambda b: b[0])
    if len(best) > 1 and best[0][0] == best[1][0]: return None
    cells = set()
    for i, j in best[0][1]:
        r = _geo(g, bg, grp[i], grp[j], metric, rule)
        if r is None or r[0] != 1: return None
        cells |= set(r[1])
    return cells


def draw_network(g, cfg):
    """cfg = (tsel, pcol, metric, net, tie, order, emit)."""
    tsel, pcol, metric, net, tie, order, emit = cfg
    bg = bg_of(g); ts = _terminals(g, bg, tsel)
    if not ts or (len(ts) < 2 and net != "emit"): return None
    out = [r[:] for r in g]
    tcells = {c for t in ts for c in t}
    free = lambda y, x: g[y][x] == bg
    rule = TIES[tie]
    def colour(t): return pcol if pcol is not None else _key(g, t)
    def pick(r):
        if r is None: return None
        n, one, union = r
        if tie == "union": return union
        return one if n == 1 else None
    if net == "comb":
        if any(len(t) != 1 for t in ts) or len(ts) < 3: return None
        if any(v != bg and (y, x) not in tcells for y, row in enumerate(g) for x, v in enumerate(row)): return None
        cells = comb_steiner([t[0] for t in ts])
        if cells is None: return None
        _paint(out, cells, colour(ts[0]), bg)
        return out
    if net == "emit":
        h, w = H(g), W(g)
        for t in ts:
            D = _emit_dir(g, t, emit)
            if D is None: return None
            proj = max(y * D[0] + x * D[1] for y, x in t)
            tip = [(y, x) for y, x in t if y * D[0] + x * D[1] == proj]
            far = ([(h, x) for x in range(w)] if D == (1, 0) else [(-1, x) for x in range(w)] if D == (-1, 0) else
                   [(y, w) for y in range(h)] if D == (0, 1) else [(y, -1) for y in range(h)])
            if any(y in (0, h - 1) and D[0] or x in (0, w - 1) and D[1] for y, x in tip):
                if any((y + D[0], x + D[1]) in set(far) for y, x in tip): continue      # already at the far edge
            cells = pick(_geo(g, bg, tip, far, metric, rule, d0=D, final_turn=False))
            if cells is None: return None
            _paint(out, cells, colour(t), bg)
        return out
    if net == "each":
        # every colour class of terminals is its own system: two members are joined by a geodesic, a lone
        # member emits a curve away from the grid edge it touches (option), larger classes are refused
        byc = {}
        for t in ts: byc.setdefault(_key(g, t), []).append(t)
        h, w = H(g), W(g)
        for c, grp in sorted(byc.items()):
            if len(grp) == 1:
                if emit != "lone": continue
                t = grp[0]; D = _emit_dir(g, t, "away")
                if D is None: return None
                proj = max(y * D[0] + x * D[1] for y, x in t)
                tip = [(y, x) for y, x in t if y * D[0] + x * D[1] == proj]
                far = ([(h, x) for x in range(w)] if D == (1, 0) else [(-1, x) for x in range(w)] if D == (-1, 0) else
                       [(y, w) for y in range(h)] if D == (0, 1) else [(y, -1) for y in range(h)])
                cells = pick(_geo(g, bg, tip, far, metric, (("late",),), d0=D, final_turn=False))
            elif len(grp) == 2:
                a, b = sorted(grp)
                if tie == "perp":            # leave / enter each wall-attached terminal perpendicular to its wall
                    da = _emit_dir(g, a, "away"); db = _emit_dir(g, b, "away")
                    cells = pick(_geo(g, bg, a, b, metric, (), d0=da, d1=None if db is None else (-db[0], -db[1])))
                else:
                    cells = pick(_geo(g, bg, a, b, metric, rule))
            elif emit == "match":
                cells = matching_network(g, bg, sorted(grp), metric, rule)
            else:
                cells = mst_network(g, bg, sorted(grp), metric)
            if cells is None:
                if order == "skip": continue      # no unique least-action network: the class stays as it is
                return None
            _paint(out, cells, pcol if pcol is not None else c, bg)
        return out
    if net == "pair":
        if len(ts) != 2: return None
        edges = [(0, 1)]
        ks = [_key(g, t) for t in ts]
        if order is not None:
            if any(k not in order for k in ks) or ks[0] == ks[1]: return None
            if order.index(ks[0]) > order.index(ks[1]): edges = [(1, 0)]
    else:   # chain / star over distinct terminal colours in the induced order
        ks = [_key(g, t) for t in ts]
        if len(set(ks)) != len(ks) or order is None or set(ks) != set(order): return None
        idx = [ks.index(c) for c in order]
        edges = list(zip(idx, idx[1:])) if net == "chain" else [(idx[0], j) for j in idx[1:]]
    for a, b in edges:
        cells = pick(_geo(g, bg, ts[a], ts[b], metric, rule))
        if cells is None: return None
        _paint(out, cells, colour(ts[a]), bg)
    return out


def _new_colour(train):
    cs = set()
    for p in train:
        i, o = p["input"], p["output"]
        cs |= {o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]}
    return cs.pop() if len(cs) == 1 else None


def _only_bg_changes(train):
    for p in train:
        i, o = p["input"], p["output"]; bg = bg_of(i)
        if any(i[y][x] != o[y][x] and i[y][x] != bg for y in range(H(i)) for x in range(W(i))): return False
    return True


def _bfs_steps(g, free, src, dst, nb):
    """Plain BFS step count from cell set src to cell set dst (dst cells need not be free)."""
    h, w = H(g), W(g); dset = set(dst); seen = set(src); fr = list(src); d = 0
    while fr:
        d += 1; nx = []
        for y, x in fr:
            for dy, dx in nb:
                q = (y + dy, x + dx)
                if q in dset: return d
                if q in seen or not (0 <= q[0] < h and 0 <= q[1] < w) or not free(*q): continue
                seen.add(q); nx.append(q)
        fr = nx
    return None


def _changed(p):
    i, o = p["input"], p["output"]
    return {(y, x) for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]}


def _precheck(train, tsel, metric, net, emit=None):
    """Cheap necessary conditions on every training pair before any minimal-action search."""
    for p in train:
        g = p["input"]; bg = bg_of(g); ts = _terminals(g, bg, tsel); C = _changed(p)
        free = lambda y, x, g=g, bg=bg: g[y][x] == bg
        if net in ("pair", "chain", "star", "comb"):
            # every terminal is an end of some drawn curve: it touches a drawn cell
            nbm = DIRS[metric]
            if any(not any((y + dy, x + dx) in C for y, x in t for dy, dx in nbm) for t in ts): return False
        if net == "pair":
            if len(ts) != 2: return False
            d = _bfs_steps(g, free, ts[0], ts[1], DIRS[metric])
            if d is None or len(C) < d - 1: return False
        elif net in ("chain", "star"):
            ks = [_key(g, t) for t in ts]
            if len(ks) < 3 or len(set(ks)) != len(ks): return False
        elif net == "each":
            cnt = Counter(_key(g, t) for t in ts)
            if not any(v >= 2 for v in cnt.values()) or len(ts) > 24: return False
        elif net == "comb":
            if len(ts) < 3 or any(len(t) != 1 for t in ts): return False
        elif net == "emit":
            if not ts or len(ts) > 12: return False
            h, w = H(g), W(g)
            for t in ts:
                D = _emit_dir(g, t, emit)
                if D is None: return False
                # the curve leaves the source at its tip: a drawn cell must touch the tip (unless at the edge)
                proj = max(y * D[0] + x * D[1] for y, x in t)
                tip = [(y, x) for y, x in t if y * D[0] + x * D[1] == proj]
                if all(0 <= y + D[0] < h and 0 <= x + D[1] < w for y, x in tip) and \
                   not any((y + dy, x + dx) in C for y, x in tip for dy, dx in N4): return False
            if emit in DIRV:
                D = DIRV[emit]
                edge = {(h - 1, x) for x in range(w)} if D == (1, 0) else {(0, x) for x in range(w)} if D == (-1, 0) else \
                       {(y, w - 1) for y in range(h)} if D == (0, 1) else {(y, 0) for y in range(h)}
                if not (C & edge): return False
    return True


def fam_geodesic(train):
    if not _same(train) or not _only_bg_changes(train): return
    pnew = _new_colour(train)
    i0 = train[0]["input"]; bg = bg_of(i0)
    # terminal colour candidates: colours present in every input, touching the changed (drawn) cells
    cols = None
    for p in train:
        i, o = p["input"], p["output"]
        near = set()
        for y, x in _changed(p):
            for dy, dx in N8:
                yy, xx = y + dy, x + dx
                if 0 <= yy < H(i) and 0 <= xx < W(i) and i[yy][xx] != bg_of(i): near.add(i[yy][xx])
        cols = near if cols is None else cols & near
    tsels = [("col", c) for c in sorted(cols or [])] + [("singles",), ("pair2",), ("edge",), ("comp", 4), ("comp", 8)]
    pcols = [pnew] if pnew is not None else [None]
    found = 0; kinds = set()
    for tsel, pcol, metric in product(tsels, pcols, ("N4", "N8")):
        if tsel[0] == "comp" and tsel[1] != (4 if metric == "N4" else 8): continue
        if pcol is None:        # curves drawn in their terminal's colour: drawn colours must be terminal colours
            okc = True
            for p in train:
                tcols = {_key(p["input"], t) for t in _terminals(p["input"], bg_of(p["input"]), tsel)}
                if any(p["output"][y][x] not in tcols for y, x in _changed(p)): okc = False; break
            if not okc: continue
        ts0 = _terminals(i0, bg, tsel)
        ks0 = [_key(i0, t) for t in ts0]
        orders = []
        if 2 <= len(set(ks0)) <= 4 and len(set(ks0)) == len(ks0):
            orders = [list(o) for o in permutations(sorted(set(ks0)))]
        cfgs = []
        if metric == "N4" and _precheck(train, tsel, metric, "comb"):
            cfgs.append((tsel, pcol, metric, "comb", "unique", None, None))
        if _precheck(train, tsel, metric, "pair"):
            for tie in ("unique", "union", "late"):
                cfgs.append((tsel, pcol, metric, "pair", tie, None, None))
            for tie in ("first-h", "first-v", "first-d"):
                if tie == "first-d" and metric == "N4": continue
                for o in (orders if len(set(ks0)) == 2 else [None] if len(set(ks0)) == 1 else []):
                    cfgs.append((tsel, pcol, metric, "pair", tie, o, None))
        if orders and len(orders[0]) >= 3 and _precheck(train, tsel, metric, "chain"):
            for o in orders:
                for net in ("chain", "star"):
                    for tie in ("unique", "first-h", "first-v"):
                        cfgs.append((tsel, pcol, metric, net, tie, o, None))
        if tsel[0] == "comp" and _precheck(train, tsel, metric, "each"):
            for tie in ("unique", "perp", "late"):
                for lone in (None, "lone", "match"):
                    for amb in (None, "skip"):
                        cfgs.append((tsel, pcol, metric, "each", tie, amb, lone))
        if metric == "N4" and tsel[0] in ("col", "edge"):
            for emit in ("away",) + (tuple(DIRV) if tsel[0] == "col" else ()):
                if not _precheck(train, tsel, metric, "emit", emit): continue
                for tie in ("unique", "late") + tuple("late-" + d for d in DIRV):
                    cfgs.append((tsel, pcol, metric, "emit", tie, None, emit))
        for cfg in cfgs:
            if found >= 3 and tsel[0] in kinds: break          # keep a few programs, one per terminal role
            ok = True
            for p in train:
                if draw_network(p["input"], cfg) != p["output"]: ok = False; break
            if not ok: continue
            kinds.add(tsel[0])
            tsel_, pcol_, metric_, net_, tie_, o_, e_ = cfg
            name = (f"act-geodesic[{tsel_[0]}{'' if len(tsel_) == 1 else tsel_[1]},{metric_},{net_},{tie_}"
                    f"{'' if o_ is None else ',order=' + ''.join(map(str, o_)) if not isinstance(o_, str) else ',' + o_}{'' if e_ is None else ',emit=' + e_}]")
            yield (name, 4, lambda g, cfg=cfg: draw_network(g, cfg))
            found += 1
            if found >= 5: return


# ------------------------------------------------------------------ L2 + L3: conservation and equilibrium
ROT = {"down": (lambda g: [r[:] for r in g], lambda g: [r[:] for r in g]),
       "up": (lambda g: [r[:] for r in g[::-1]], lambda g: [r[:] for r in g[::-1]]),
       "right": (lambda g: [list(r) for r in zip(*g)], lambda g: [list(r) for r in zip(*g)]),
       "left": (lambda g: [list(r) for r in zip(*g)][::-1], lambda g: [list(r) for r in zip(*g[::-1])])}


def held(cells, h, w):
    """Cells of the bbox of an object that retain a fluid under gravity (down): starting there, moving
    down / left / right through non-object cells never leaves the bbox.  Closed frames retain their whole
    interior, cups their bowl."""
    cs = set(cells); r0, c0, r1, c1 = bbox(cells)
    esc = set()                          # cells from which the fluid escapes: reverse flood from outside
    st = []
    for y in range(r0, r1 + 1):
        for x in range(c0, c1 + 1):
            if (y, x) in cs: continue
            # a cell escapes if one move (down/left/right) leaves the bbox
            if y + 1 > r1 or x - 1 < c0 or x + 1 > c1: esc.add((y, x)); st.append((y, x))
    while st:
        y, x = st.pop()
        for py, px in ((y - 1, x), (y, x + 1), (y, x - 1)):     # predecessors that can move into (y, x)
            if r0 <= py <= r1 and c0 <= px <= c1 and (py, px) not in cs and (py, px) not in esc:
                esc.add((py, px)); st.append((py, px))
    return [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in cs and (y, x) not in esc]


def _pattern(cells, pat):
    if pat == "full": return list(cells)
    y0, x0 = min(cells)
    return [(y, x) for y, x in cells if (y + x - y0 - x0) % 2 == 0]


def capacity_pour(g, pat):
    """Mass conservation selects the target: every mobile colour (cells outside containers) is poured,
    count preserved, into the unique container whose capacity under the fill pattern equals its count."""
    bg = bg_of(g); h, w = H(g), W(g)
    objs = objects(g, bg, True, True)
    conts = []
    for o in objs:
        if len(o) < 3: continue
        hd = [c for c in held(o, h, w) if g[c[0]][c[1]] == bg]
        if hd: conts.append((o, _pattern(hd, pat)))
    if not conts: return None
    ccells = {c for o, _ in conts for c in o}
    mobile = {}
    for y in range(h):
        for x in range(w):
            if g[y][x] != bg and (y, x) not in ccells: mobile.setdefault(g[y][x], []).append((y, x))
    if not mobile: return None
    out = [r[:] for r in g]; used = set()
    for c, cells in mobile.items():
        hits = [k for k, (o, tgt) in enumerate(conts) if len(tgt) == len(cells)]
        if len(hits) != 1 or hits[0] in used: return None
        used.add(hits[0])
        for y, x in cells: out[y][x] = bg
        for y, x in conts[hits[0]][1]: out[y][x] = c
    return out


def heap(g):
    """Granular equilibrium with mass conservation: the cells of every occupied column collapse into a
    heap at the floor with a 45-degree angle of repose (row widths 1, 3, 5, ... centred on the column);
    cells keep their top-to-bottom order (apex first)."""
    bg = bg_of(g); h, w = H(g), W(g)
    out = [[bg] * w for _ in range(h)]
    for x in range(w):
        col = [g[y][x] for y in range(h) if g[y][x] != bg]
        if not col: continue
        k = int(round(len(col) ** 0.5))
        if k * k != len(col) or k > h: return None
        slots = []
        for j in range(k):                        # j = 0 apex row
            y = h - k + j; half = j
            if x - half < 0 or x + half >= w: return None
            slots += [(y, xx) for xx in range(x - half, x + half + 1)]
        for (y, xx), v in zip(slots, col):
            if out[y][xx] != bg: return None
            out[y][xx] = v
    return out


def sand(g, m):
    """Granular equilibrium of one mobile colour m under gravity (down): every grain falls until it rests on a
    non-background cell or the floor; all other colours are static supports.  Count is conserved."""
    bg = bg_of(g); h, w = H(g), W(g); out = [r[:] for r in g]
    if m == bg or not any(v == m for r in g for v in r): return None
    for x in range(w):
        for y in range(h - 1, -1, -1):
            if out[y][x] != m: continue
            yy = y
            while yy + 1 < h and out[yy + 1][x] == bg: yy += 1
            if yy != y: out[yy][x] = m; out[y][x] = bg
    return out


def _conserved(train):
    for p in train:
        i, o = p["input"], p["output"]
        if Counter(v for r in i for v in r) != Counter(v for r in o for v in r): return False
    return True


def fam_conserve(train):
    if not _same(train) or not _conserved(train): return
    # mobile colours: the colours of cells that move in every training pair
    mobile = None
    for p in train:
        i, o = p["input"], p["output"]; bg = bg_of(i)
        mv = {i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x] and i[y][x] != bg}
        mobile = mv if mobile is None else mobile & mv
    mobile = mobile or set()
    for grav, (to, back) in ROT.items():
        for pat in ("full", "checker"):
            fn = lambda g, to=to, back=back, pat=pat: (lambda r: back(r) if r else None)(capacity_pour(to(g), pat))
            if all(fn(p["input"]) == p["output"] for p in train):
                yield (f"act-conserve:capacity[{pat},gravity={grav}]", 4, fn)
        fn = lambda g, to=to, back=back: (lambda r: back(r) if r else None)(heap(to(g)))
        if all(fn(p["input"]) == p["output"] for p in train):
            yield (f"act-conserve:heap[gravity={grav}]", 4, fn)
        for m in sorted(mobile):
            fn = lambda g, to=to, back=back, m=m: (lambda r: back(r) if r else None)(sand(to(g), m))
            if all(fn(p["input"]) == p["output"] for p in train):
                yield (f"act-conserve:sand[c{m},gravity={grav}]", 4, fn)


FAMILIES = (fam_geodesic, fam_conserve)


In [ ]:
%%writefile /kaggle/working/probe/prior_arithmetic.py
"""ARITHMETIC prior: foundational arithmetic and set theory on the cell lattice.

Laws (the prior, stated before looking for tasks)
-------------------------------------------------
Z    The grid is a finite piece of Z x Z.  Every cell has integer coordinates; every item (object, line,
     band, panel, cell of a class) has integer attributes (position, extent, size, count) and an ORDINAL
     index in any total order of its kind.  Offsets u = coordinate - anchor are integers.
===  CONGRUENCE.  A periodic pattern is a function of the residue class of an offset: 1-D u mod k, diagonal
     (uy + ux) mod k, 2-D (uy mod ky, ux mod kx), or a general translation lattice <(p, s), (0, q)>.
     Parity is the case k = 2.  A class is either coloured by a table (colouring) or carries one colour
     that fills its unknown members (completion / continuation).
<    ORDER.  Integers are compared: sign(u) (before / at / after an anchor), the midpoint comparison
     sign(2u - L) (halves), thresholds (size >= t), ranks (ordinal index), and a total order on colours
     that resolves conflicts (priority).
+ *  COUNTING ARITHMETIC.  Multiplicities (scale factor, tile count) are affine a*n + b of a count n of the
     input; parallel bar lengths form arithmetic progressions L + a*d in the offset d from a seed bar.
|    DIVISIBILITY.  A side divisible by k gives a quotient window (side / k, or the minimal period p | side);
     the parity of the side can select which window.
S    SET ALGEBRA.  Panels (separator lattice, N x M blocks, overlapping corner windows) are sets of coloured
     cells: union with an induced colour order, any function of the MEMBERSHIP COUNT (intersection = all,
     union = any, symmetric difference = odd / exactly one, complement = none, consensus = n / n-1), and
     the complement (figure <-> ground) of a 2-colour grid.
AFF  Affine maps of offsets: every line (row / column) is translated by s(u) = a*u (shear) or by a periodic
     sequence s(u) = S[u mod k], u the line offset from an anchor line.

Families (all parameters induced from the training pairs; unseen keys / counts at test time -> None)
------------------------------------------------------------------------------------------------------
arith-residue[feature,class]  out(p) = F(class(p), phi(p)); class = input colour | figure/ground; F a table
        (class, phi) -> keep | literal colour | anchor colour (seed / object / the grid's figure colour).
        phi: grid-anchored u mod k (either edge, k = 2..12), (uy+ux) mod k from a corner, 2-D residues, grid
        half; object-anchored residues, halves, ordinal index mod k or rank (left/right/top/bottom), size
        parity and size threshold (4|8-conn, mono|multi colour); seed-anchored (the unique singleton-colour
        cell) lattices, diagonals, quadrants, half-plane x residue; ranks / residues of occupied lines,
        bands of identical lines, scan order of figure cells.  Guards: phi must matter (a class takes >= 2
        actions), >= 2 cells per key, every change rule recurs, >= 1.5 periods observed, periodic ordinals
        wrap, rank tables are palettes (injective).
arith-shift            each row / column translated by a*u or S[u mod k] from an anchor line (grid edge, figure
                       top/bottom); cyclic, padded, or on a widened canvas (fill colour induced); or a constant
                       cyclic shift whose amount is the index (a count) of a marker in removed border line(s).
arith-count-scale      output = upscale / tile (k x k, 1 x k, k x 1) of the grid or its figure bbox with
                       k = a*n + b, n a count (colours, cells of a colour, objects, figure cells, rarest colour).
arith-union            panels combined by colour-order union or by a membership-count table.
arith-progression      bars parallel to a seed bar with lengths L + a_side*d, colour by (side, d mod m).
arith-congruence       unknown cells (ground / occluder colour) filled from the smallest consistent lattice,
                       output the grid or the occluded window; or the input continued onto a larger canvas.
arith-complement       figure <-> ground swap, then identity / tiling / Kronecker placement at figure|ground.
arith-window           quotient window: constant size, side / k, or minimal period, at a fixed corner or the
                       corner selected by the parity of the grid size.

Measured (eval_fam, 1000 training + 50 half-A): 83 exact, WRONG 0; per family union 33, residue 21, shift 7,
congruence 7, window 5, count-scale 4, complement 3, progression 3.  Dropped / guarded after measurement:
height/width thresholds and parities (coincidental fits, no new solves), 1-cell windows and 1x1 corner
panels (coincidences), non-injective rank tables (over-fired), literal-vs-anchor ambiguity (resolved per class).
Seed-distance and bbox-ring residues solve nothing yet but never fire falsely; kept as part of the law.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
import numpy as np
from scipy import ndimage
from gdsl import H, W, bg_of

S4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])
S8 = np.ones((3, 3), dtype=int)
SEGS = ((4, True), (8, True), (4, False), (8, False))


def A(g):
    return np.asarray(g, dtype=np.int64)


def same_shape(train):
    return all(np.shape(p['input']) == np.shape(p['output']) for p in train)


# ------------------------------------------------------------------ segmentation
def label(a, bg, conn, mono):
    """Connected components of non-bg cells: label array (-1 = bg) and count."""
    st = S4 if conn == 4 else S8
    lab = -np.ones(a.shape, dtype=np.int64)
    if mono:
        n = 0
        for c in np.unique(a):
            if c == bg: continue
            l, m = ndimage.label(a == c, structure=st)
            lab[l > 0] = l[l > 0] - 1 + n
            n += m
        return lab, n
    l, m = ndimage.label(a != bg, structure=st)
    return l - 1, m


def boxes(lab, n):
    """(top, left, bottom, right) arrays per label."""
    t = np.zeros(n, dtype=np.int64); l = np.zeros(n, dtype=np.int64)
    b = np.zeros(n, dtype=np.int64); r = np.zeros(n, dtype=np.int64)
    for k, sl in enumerate(ndimage.find_objects(lab + 1)):
        if sl is None: continue
        t[k], b[k] = sl[0].start, sl[0].stop - 1
        l[k], r[k] = sl[1].start, sl[1].stop - 1
    return t, l, b, r


class Ctx:
    """Per-grid cache of arrays used by the features."""
    def __init__(self, g, bg=None):
        self.a = A(g); self.bg = bg_of(g) if bg is None or not np.any(self.a == bg) else bg
        self.h, self.w = self.a.shape
        self.Y, self.X = np.indices(self.a.shape)
        self.fg = self.a != self.bg
        self._seg = {}

    def seg(self, key):
        if key not in self._seg:
            lab, n = label(self.a, self.bg, *key)
            self._seg[key] = (lab, n, boxes(lab, n))
        return self._seg[key]

    def seed(self):
        """The unique cell whose colour occurs exactly once (the origin of a seed-anchored pattern)."""
        if not hasattr(self, '_seed'):
            vals, cnt = np.unique(self.a, return_counts=True)
            one = [int(v) for v, n in zip(vals, cnt) if n == 1 and v != self.bg]
            if len(one) != 1: self._seed = None
            else:
                ys, xs = np.nonzero(self.a == one[0])
                self._seed = (int(ys[0]), int(xs[0]), one[0])
        return self._seed


# ------------------------------------------------------------------ residue / order features
def _u(c, axis, anchor, lo=None, hi=None):
    """Offset of every cell along axis from an anchor edge (grid, or per-cell bounds lo/hi)."""
    P = c.Y if axis == 'y' else c.X
    if lo is None:
        n = c.h if axis == 'y' else c.w
        return P if anchor == 0 else n - 1 - P
    return P - lo if anchor == 0 else hi - P


def _rank(vals):
    """Dense rank of distinct values; None on ties."""
    if len(set(vals)) != len(vals): return None
    order = sorted(range(len(vals)), key=lambda i: vals[i])
    r = [0] * len(vals)
    for k, i in enumerate(order): r[i] = k
    return r


def feature(c, spec):
    """Integer feature per cell (-1 = undefined) or None when not computable (memoised per grid)."""
    key = ('feat',) + spec
    if key not in c._seg: c._seg[key] = _feature(c, spec)
    return c._seg[key]


def _feature(c, spec):
    kind = spec[0]
    if kind == 'g':                                    # ('g', axis, anchor, k)
        _, axis, an, k = spec
        return _u(c, axis, an) % k
    if kind == 'gd':                                   # ('gd', ay, ax, k)
        _, ay, ax, k = spec
        return (_u(c, 'y', ay) + _u(c, 'x', ax)) % k
    if kind == 'g2':                                   # ('g2', ay, ax, ky, kx)
        _, ay, ax, ky, kx = spec
        return (_u(c, 'y', ay) % ky) * kx + _u(c, 'x', ax) % kx
    if kind == 'gh':                                   # ('gh', axis)
        P, n = (c.Y, c.h) if spec[1] == 'y' else (c.X, c.w)
        return np.sign(2 * P - (n - 1)) + 1
    if kind in ('op', 'ot'):                           # ('op', seg, prop, k) | ('ot', seg, prop, t)
        _, seg, prop, k = spec
        lab, n, _ = c.seg(seg)
        if n == 0: return None
        vals = obj_prop(c, seg, prop)
        per = (vals % k) if kind == 'op' else (vals >= k).astype(np.int64)
        return np.where(lab >= 0, per[np.where(lab >= 0, lab, 0)], -1)
    if kind in ('o', 'od', 'oh', 'on'):
        seg = spec[1]
        lab, n, (t, l, b, r) = c.seg(seg)
        if n == 0: return None
        m = lab >= 0; L = np.where(m, lab, 0)
        T_, L_, B_, R_ = t[L], l[L], b[L], r[L]
        if kind == 'o':                                # ('o', seg, axis, anchor, k)
            _, _, axis, an, k = spec
            lo, hi = (T_, B_) if axis == 'y' else (L_, R_)
            f = _u(c, axis, an, lo, hi) % k
        elif kind == 'od':                             # ('od', seg, ay, ax, k)
            _, _, ay, ax, k = spec
            f = (_u(c, 'y', ay, T_, B_) + _u(c, 'x', ax, L_, R_)) % k
        elif kind == 'oh':                             # ('oh', seg, axis)
            f = np.sign(2 * c.Y - (T_ + B_)) + 1 if spec[2] == 'y' else np.sign(2 * c.X - (L_ + R_)) + 1
        else:                                          # ('on', seg, order, k)  k = 0: rank
            _, _, order, k = spec
            key = {'left': list(l), 'right': list(-r), 'top': list(t), 'bottom': list(-b)}[order]
            rk = _rank(key)
            if rk is None: return None
            rk = np.array(rk, dtype=np.int64)
            f = rk[L] if k == 0 else rk[L] % k
        return np.where(m, f, -1)
    if kind == 'rb':                                   # ('rb', seg, k): ring index inside a bbox mod k
        _, seg, k = spec
        lab, n, (t, l, b, r) = c.seg(seg)
        if n == 0: return None
        f = -np.ones(c.a.shape, dtype=np.int64)
        for j in sorted(range(n), key=lambda j: -(b[j] - t[j] + 1) * (r[j] - l[j] + 1)):
            if b[j] - t[j] < 2 or r[j] - l[j] < 2: continue
            Ys, Xs = c.Y[t[j]:b[j] + 1, l[j]:r[j] + 1], c.X[t[j]:b[j] + 1, l[j]:r[j] + 1]
            d = np.minimum(np.minimum(Ys - t[j], b[j] - Ys), np.minimum(Xs - l[j], r[j] - Xs))
            f[t[j]:b[j] + 1, l[j]:r[j] + 1] = d % k
        return f
    if kind == 'sr':                                   # ('sr', metric, k): distance from the seed mod k
        sd = c.seed()
        if sd is None: return None
        dy, dx = np.abs(c.Y - sd[0]), np.abs(c.X - sd[1])
        return (np.maximum(dy, dx) if spec[1] == 'cheb' else dy + dx) % spec[2]
    if kind in ('s2', 'sd', 'sq', 'sqm'):              # seed-anchored offsets dy, dx
        sd = c.seed()
        if sd is None: return None
        dy, dx = c.Y - sd[0], c.X - sd[1]
        if kind == 's2': return (dy % spec[1]) * spec[2] + dx % spec[2]
        if kind == 'sd': return (dy + spec[1] * dx) % spec[2]
        if kind == 'sq': return (np.sign(dy) + 1) * 3 + np.sign(dx) + 1
        _, axis, k = spec                              # half-plane along axis x residue along the other
        d1, d2 = (dy, dx) if axis == 'y' else (dx, dy)
        return (np.sign(d1) + 1) * k + d2 % k
    if kind == 'ln':                                   # ('ln', axis, anchor, k): line rank among fg lines
        _, axis, an, k = spec
        P = c.X if axis == 'x' else c.Y
        occ = sorted(set(P[c.fg].tolist()))
        if not occ: return None
        if an == 1: occ = occ[::-1]
        pos = -np.ones(max(c.h, c.w), dtype=np.int64); pos[occ] = np.arange(len(occ))
        idx = pos[P]
        f = idx if k == 0 else idx % k
        return np.where(c.fg, f, -1)
    if kind == 'bd':                                   # ('bd', axis, anchor, k): band index mod k
        _, axis, an, k = spec
        a = c.a if axis == 'y' else c.a.T
        n = a.shape[0]; bid = np.zeros(n, dtype=np.int64); cur = 0
        for i in range(1, n):
            if not np.array_equal(a[i], a[i - 1]): cur += 1
            bid[i] = cur
        if cur == 0: return None
        if an == 1: bid = cur - bid
        f = (bid % k)[:, None] * np.ones((1, a.shape[1]), dtype=np.int64)
        return f if axis == 'y' else f.T
    if kind == 'sc':                                   # ('sc', order, k): scan index of fg cells mod k
        _, order, k = spec
        ys, xs = np.nonzero(c.fg)
        if len(ys) == 0: return None
        if order == 'strict':
            if len(set(ys.tolist())) == len(ys): order = 'row'
            elif len(set(xs.tolist())) == len(xs): order = 'col'
            else: return None
        idx = np.lexsort((xs, ys)) if order == 'row' else np.lexsort((ys, xs))
        f = -np.ones(c.a.shape, dtype=np.int64)
        f[ys[idx], xs[idx]] = np.arange(len(ys)) % k
        return f
    return None


def obj_prop(c, seg, prop):
    """Integer property per object: size (cells), height, width."""
    key = ('prop', seg, prop)
    if key not in c._seg:
        lab, n, (t, l, b, r) = c.seg(seg)
        if prop == 'size': v = np.bincount(lab[lab >= 0], minlength=n)
        elif prop == 'h': v = b - t + 1
        else: v = r - l + 1
        c._seg[key] = np.asarray(v, dtype=np.int64)
    return c._seg[key]


def extent_ok(spec, ctxs):
    """The period k must be exercised: some training extent n along the axis holds >= 1.5 periods
    (3k <= 2n); on an axis of constant length the raw coordinate (k = n) is allowed."""
    kind = spec[0]
    def per(n, k): return 3 * k <= 2 * n
    def ax_ok(ns, k):
        return per(max(ns), k) or (len(set(ns)) == 1 and k == ns[0])
    if kind == 'g':
        _, axis, _, k = spec
        return per(max((c.h if axis == 'y' else c.w) for c in ctxs), k)
    if kind == 'gd':
        return per(max(c.h + c.w - 1 for c in ctxs), spec[3])
    if kind == 'g2':
        _, _, _, ky, kx = spec
        return ax_ok([c.h for c in ctxs], ky) and ax_ok([c.w for c in ctxs], kx)
    if kind == 'rb':
        ext = 0
        for c in ctxs:
            _, n, (t, l, b, r) = c.seg(spec[1])
            if n: ext = max(ext, int(((np.minimum(b - t, r - l)) // 2 + 1).max()))
        return per(ext, spec[2])
    if kind == 'sr':
        return per(max(max(c.h, c.w) for c in ctxs) // 2, spec[2])
    if kind in ('o', 'od'):
        ext = 0
        for c in ctxs:
            _, n, (t, l, b, r) = c.seg(spec[1])
            if n == 0: continue
            if kind == 'o':
                e = (b - t + 1) if spec[2] == 'y' else (r - l + 1)
            else:
                e = b - t + r - l + 1
            ext = max(ext, int(e.max()))
        return per(ext, spec[-1])
    return True


def feature_specs(train, ctxs):
    """Candidate features, cheapest description first: (level, spec)."""
    hs = [c.h for c in ctxs]; ws = [c.w for c in ctxs]
    out = []
    for axis, n in (('y', max(hs)), ('x', max(ws))):
        for k in range(2, min(12, n - 1) + 1):
            for an in (0, 1):
                out.append((0, ('g', axis, an, k)))
        out.append((0, ('gh', axis)))
    for ay, ax in product((0, 1), (0, 1)):
        for k in range(2, 7):
            out.append((1, ('gd', ay, ax, k)))
    for ay, ax in product((0, 1), (0, 1)):
        for ky in range(2, 5):
            for kx in range(2, 13):
                out.append((2, ('g2', ay, ax, ky, kx)))
                if kx <= 4 and kx != ky:
                    out.append((2, ('g2', ay, ax, kx, ky)))
    for seg in SEGS:
        for axis in ('y', 'x'):
            for an in (0, 1):
                for k in range(2, 5):
                    out.append((1, ('o', seg, axis, an, k)))
            out.append((1, ('oh', seg, axis)))
        for ay, ax in product((0, 1), (0, 1)):
            for k in (2, 3):
                out.append((2, ('od', seg, ay, ax, k)))
        for order in ('left', 'right', 'top', 'bottom'):
            for k in (2, 3, 0):
                out.append((2, ('on', seg, order, k)))
    for axis in ('x', 'y'):
        for an in (0, 1):
            for k in (0, 2, 3):
                out.append((2, ('ln', axis, an, k)))
            for k in (2, 3, 4):
                out.append((2, ('bd', axis, an, k)))
    for order in ('row', 'col', 'strict'):
        for k in (2, 3, 4):
            out.append((2, ('sc', order, k)))
    for seg in SEGS:
        for k in (2, 3, 4):
            out.append((2, ('rb', seg, k)))
    for seg in SEGS:
        for prop in ('size',):
            vals = sorted(set(np.concatenate([obj_prop(c, seg, prop) for c in ctxs]).tolist()))
            if len(vals) < 3: continue
            out.append((1, ('op', seg, prop, 2)))
            for v in vals[:-1]:                        # threshold just above an observed value
                out.append((1, ('ot', seg, prop, v + 1)))
    if all(c.seed() is not None for c in ctxs):
        out.append((1, ('sq',)))
        for ky, kx in product((1, 2, 3), (1, 2, 3)):
            if ky * kx > 1: out.append((1, ('s2', ky, kx)))
        for sg, k in product((1, -1), (2, 3)):
            out.append((1, ('sd', sg, k)))
        for axis, k in product(('y', 'x'), (2, 3)):
            out.append((2, ('sqm', axis, k)))
        for metric, k in product(('cheb', 'manh'), (2, 3, 4)):
            out.append((1, ('sr', metric, k)))
    return out


def anchor_colour(c, spec):
    """Colour of the anchor each cell is measured from (-1 = none): the seed, or a multicolour object's
    majority colour."""
    kind = spec[0]
    if kind in ('s2', 'sd', 'sq', 'sqm', 'sr'):
        return np.full(c.a.shape, c.seed()[2], dtype=np.int64)
    if kind in ('g', 'gd', 'g2', 'gh', 'ln', 'bd', 'sc'):        # the grid's unique figure colour
        if ('fig',) not in c._seg:
            fc = set(np.unique(c.a).tolist()) - {c.bg}
            c._seg[('fig',)] = np.full(c.a.shape, fc.pop() if len(fc) == 1 else -1, dtype=np.int64)
        return c._seg[('fig',)]
    if kind in ('o', 'od', 'oh', 'on', 'op', 'ot') and not spec[1][1]:
        key = ('maj',) + spec[1]
        if key not in c._seg:
            lab, n, _ = c.seg(spec[1])
            m = lab >= 0
            hist = np.zeros((n + 1, 16), dtype=np.int64)
            np.add.at(hist, (lab[m], c.a[m]), 1)
            maj = hist.argmax(axis=1); maj[n] = -1
            c._seg[key] = maj[np.where(m, lab, n)]
        return c._seg[key]
    return None


def fit_table(keys, ins, outs, anc=None):
    """Table key -> literal colour (>= 0), keep (-1) or anchor colour (-2); None if inconsistent.
    A group consistent with both a literal and the anchor colour is resolved per class: anchor when some
    group of the same class needs the anchor, else literal; keep is always preferred."""
    order = np.argsort(keys, kind='stable')
    k = keys[order]; i = ins[order]; o = outs[order]
    starts = np.flatnonzero(np.r_[True, k[1:] != k[:-1]])
    omin = np.minimum.reduceat(o, starts); omax = np.maximum.reduceat(o, starts)
    keep = np.minimum.reduceat((o == i).astype(np.int64), starts) == 1
    lit = omin == omax
    if anc is not None:
        an = anc[order]
        ancok = np.minimum.reduceat(((o == an) & (an >= 0)).astype(np.int64), starts) == 1
    else:
        ancok = np.zeros(len(starts), dtype=bool)
    if not np.all(lit | keep | ancok): return None
    kk = k[starts]
    need_anc = {int(x) // KM for x, kp, lt, ao in zip(kk, keep, lit, ancok) if ao and not kp and not lt}
    tab = {}
    for x, kp, lt, ao, v in zip(kk, keep, lit, ancok, omin):
        x = int(x)
        if kp: tab[x] = -1
        elif ao and (not lt or x // KM in need_anc): tab[x] = -2
        else: tab[x] = int(v)
    return tab


KM = 64   # key = class * KM + feature  (features < KM)


def residue_fit(ctxs, outs, spec, cmode):
    keys, ins, os_, ancs = [], [], [], []
    for c, o in zip(ctxs, outs):
        f = feature(c, spec)
        if f is None or f.max() >= KM: return None
        v = f >= 0
        if not np.array_equal(o[~v], c.a[~v]): return None
        cl = c.a if cmode == 'lit' else c.fg.astype(np.int64)
        keys.append((cl * KM + f)[v]); ins.append(c.a[v]); os_.append(o[v])
        an = anchor_colour(c, spec)
        ancs.append(an[v] if an is not None else np.full(int(v.sum()), -1, dtype=np.int64))
    keys = np.concatenate(keys); ins = np.concatenate(ins); os_ = np.concatenate(os_)
    if len(keys) == 0: return None
    tab = fit_table(keys, ins, os_, np.concatenate(ancs))
    if tab is None: return None
    # relevance: some class takes >= 2 different actions over the feature values
    acts = {}
    for kk, v in tab.items(): acts.setdefault(kk // KM, set()).add(v)
    if not any(len(s) > 1 for s in acts.values()): return None
    # it must change something, compress (>= 2 cells per key) and every change rule must recur
    nchg = int(np.sum(ins != os_))
    if nchg == 0: return None
    if 2 * len(tab) > len(keys): return None
    if 2 * sum(1 for v in tab.values() if v != -1) > nchg: return None
    if spec[0] in ('on', 'ln', 'bd', 'sc'):
        k = spec[-1]
        if k > 0:          # a periodic ordinal must wrap around in some pair
            if max(n_items(c, spec) for c in ctxs) <= k: return None
        else:              # a rank table is a palette: distinct ranks of a class get distinct actions
            for cl, s in acts.items():
                vals = [v for kk, v in tab.items() if kk // KM == cl]
                if len(vals) != len(set(vals)): return None
    return tab


def n_items(c, spec):
    kind = spec[0]
    if kind == 'on': return c.seg(spec[1])[1]
    if kind == 'ln':
        P = c.X if spec[1] == 'x' else c.Y
        return len(set(P[c.fg].tolist()))
    if kind == 'bd':
        a = c.a if spec[1] == 'y' else c.a.T
        return 1 + sum(not np.array_equal(a[i], a[i - 1]) for i in range(1, a.shape[0]))
    return int(c.fg.sum())


def residue_apply(g, spec, cmode, tab, bg=None):
    c = Ctx(g, bg)
    f = feature(c, spec)
    if f is None: return None
    out = c.a.copy()
    cl = c.a if cmode == 'lit' else c.fg.astype(np.int64)
    v = f >= 0
    keys = cl * KM + f
    anc = None
    for kk in np.unique(keys[v]).tolist():
        if kk not in tab: return None
        act = tab[kk]
        if act >= 0: out[v & (keys == kk)] = act
        elif act == -2:
            if anc is None: anc = anchor_colour(c, spec)
            m = v & (keys == kk)
            if np.any(anc[m] < 0): return None
            out[m] = anc[m]
    return out.tolist()


def task_bg(train):
    """Background induced over the task: the most frequent colour of all training inputs together, used when
    some input's own majority colour differs (a figure larger than its ground)."""
    tot = Counter(v for p in train for r in p['input'] for v in r)
    b = tot.most_common(1)[0][0]
    if all(bg_of(p['input']) == b for p in train): return None
    if not all(any(b in r for r in p['input']) for p in train): return None
    return b


def fam_arith_residue(train):
    if not same_shape(train): return
    if all(p['input'] == p['output'] for p in train): return
    outs = [A(p['output']) for p in train]
    fits = []
    tb = task_bg(train)
    for bg in (None, tb) if tb is not None else (None,):
        ctxs = [Ctx(p['input'], bg) for p in train]
        for level, spec in feature_specs(train, ctxs):
            if not extent_ok(spec, ctxs): continue
            for cmode in ('lit', 'fg'):
                tab = residue_fit(ctxs, outs, spec, cmode)
                if tab is None: continue
                fits.append((len(tab) + 2 * level, level, spec, cmode, tab, bg))
    fits.sort(key=lambda t: (t[0], t[1]))
    specs = []                       # the 3 best features, each with its literal and fg/bg class variants
    for score, level, spec, cmode, tab, bg in fits:
        if (spec, bg) not in specs:
            if len(specs) == 3: continue
            specs.append((spec, bg))
        yield (f"arith-residue[{spec_name(spec)},{cmode}{'' if bg is None else f',bg{bg}'}]", 4 + min(level, 1),
               lambda g, spec=spec, cmode=cmode, tab=tab, bg=bg: residue_apply(g, spec, cmode, tab, bg))


def spec_name(spec):
    def s(x):
        if isinstance(x, tuple): return f"{x[0]}{'m' if x[1] else 'M'}"
        return str(x)
    return ':'.join(s(x) for x in spec)


# ------------------------------------------------------------------ AFF: line shifts s(u) = a*u | S[u mod k]
def shift_line(r, s, mode, bg, width=None):
    n = len(r)
    if mode == 'roll':
        return np.roll(r, s)
    if mode == 'pad':
        out = np.full(n, bg, dtype=r.dtype)
        if abs(s) >= n: return out
        if s >= 0: out[s:] = r[:n - s]
        else: out[:n + s] = r[-s:]
        return out
    out = np.full(width, bg, dtype=r.dtype)       # expand: place the line at offset s
    out[s:s + n] = r
    return out


def allowed_shifts(r_in, r_out, mode, bg):
    """Set of shifts mapping r_in onto r_out, or None when every shift does (uniform line)."""
    n = len(r_in)
    if mode == 'roll':
        if np.all(r_in == r_in[0]): return None
        return {s for s in range(n) if np.array_equal(np.roll(r_in, s), r_out)}
    if np.all(r_in == bg):
        return None if np.all(r_out == bg) else set()
    if mode == 'pad':
        return {s for s in range(-n + 1, n) if np.array_equal(shift_line(r_in, s, 'pad', bg), r_out)}
    E = len(r_out) - n
    return {s for s in range(E + 1) if np.array_equal(shift_line(r_in, s, 'expand', bg, len(r_out)), r_out)}


def anchor_line(a, bg, anchor):
    if anchor == 'top': return 0
    if anchor == 'bottom': return a.shape[0] - 1
    rows = np.flatnonzero((a != bg).any(axis=1))
    if len(rows) == 0: return None
    return int(rows[0]) if anchor == 'fgtop' else int(rows[-1])


def shift_profile(n, y0, model, par):
    """Shift of every line 0..n-1 (None = undetermined)."""
    if model == 'lin':
        return [par * (y - y0) for y in range(n)]
    k, S = par
    return [S[(y - y0) % k] for y in range(n)]


def apply_shift(g, orient, mode, anchor, model, par, fill=None):
    a = A(g)
    if orient == 'col': a = a.T
    bg = bg_of(g); n, w = a.shape
    y0 = anchor_line(a, bg, anchor)
    if y0 is None: return None
    prof = shift_profile(n, y0, model, par)
    if mode == 'expand':
        lo, hi = min(prof), max(prof)
        width = w + hi - lo
        rows = [shift_line(a[y], prof[y] - lo, 'expand', bg if fill is None else fill, width) for y in range(n)]
    else:
        rows = []
        for y in range(n):
            s = prof[y]
            if s is None:
                if mode == 'roll' and np.all(a[y] == a[y][0]) or mode == 'pad' and np.all(a[y] == bg):
                    s = 0
                else:
                    return None
            rows.append(shift_line(a[y], s % w if mode == 'roll' else s, mode, bg))
    out = np.array(rows)
    return (out if orient == 'row' else out.T).tolist()


EDGE_SETS = (('left',), ('right',), ('top',), ('bottom',), ('top', 'left'), ('top', 'right'),
             ('bottom', 'left'), ('bottom', 'right'))


def border_line(a, e):
    return a[0] if e == 'top' else a[-1] if e == 'bottom' else a[:, 0] if e == 'left' else a[:, -1]


def marker_roll(g, edges, mcol, signs):
    """Remove the marker border line(s); roll the rest by the marker's index (a count of cells) along each."""
    a = A(g); h, w = a.shape
    r0, r1 = (1 if 'top' in edges else 0), h - (1 if 'bottom' in edges else 0)
    c0, c1 = (1 if 'left' in edges else 0), w - (1 if 'right' in edges else 0)
    R = a[r0:r1, c0:c1]
    if R.size == 0: return None
    sy = sx = 0
    for e in edges:
        pos = np.flatnonzero(border_line(a, e) == mcol)
        if len(pos) != 1: return None
        if e in ('top', 'bottom'): sx = int(pos[0]) - c0
        else: sy = int(pos[0]) - r0
    return np.roll(np.roll(R, signs[0] * sy, axis=0), signs[1] * sx, axis=1)


def fam_arith_shift(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    if all(np.array_equal(i, o) for i, o in zip(ins, outs)): return
    # constant cyclic shift read from a marker's index in removed border line(s)
    for edges in EDGE_SETS:
        dh, dw = sum(e in ('top', 'bottom') for e in edges), sum(e in ('left', 'right') for e in edges)
        if any(o.shape != (i.shape[0] - dh, i.shape[1] - dw) for i, o in zip(ins, outs)): continue
        mcols = set.intersection(*[{c for c in np.unique(border_line(i, e)).tolist()
                                    if int((border_line(i, e) == c).sum()) == 1} for i in ins for e in edges])
        for mcol, signs in product(sorted(mcols), product((1, -1), (1, -1))):
            if dh == 0 and signs[1] == -1 or dw == 0 and signs[0] == -1: continue
            preds = [marker_roll(p['input'], edges, mcol, signs) for p in train]
            if any(r is None or not np.array_equal(r, o) for r, o in zip(preds, outs)): continue
            if all(np.array_equal(r, marker_roll(p['input'], edges, mcol, (0, 0))) for r, p in zip(preds, train)):
                continue
            yield (f"arith-shift[marker{list(edges)},c{mcol}]:roll{signs}", 4,
                   lambda g, e=edges, m=mcol, sg=signs: (lambda r: None if r is None else r.tolist())(
                       marker_roll(g, e, m, sg)))
            return
    found = 0
    for orient in ('row', 'col'):
        I = [i if orient == 'row' else i.T for i in ins]
        O = [o if orient == 'row' else o.T for o in outs]
        if any(i.shape[0] != o.shape[0] for i, o in zip(I, O)): continue
        if all(i.shape == o.shape for i, o in zip(I, O)): modes = ('roll', 'pad')
        elif all(o.shape[1] > i.shape[1] for i, o in zip(I, O)): modes = ('expand',)
        else: continue
        bgs0 = [bg_of(p['input']) for p in train]
        fills = [None]
        if modes == ('expand',):
            newc = set.intersection(*[set(np.unique(o).tolist()) - set(np.unique(i).tolist()) for i, o in zip(I, O)])
            fills += sorted(newc)
        for mode, fill in product(modes, fills):
            bgs = bgs0 if fill is None else [fill] * len(train)
            AL = [[allowed_shifts(i[y], o[y], mode, bg) for y in range(i.shape[0])] for i, o, bg in zip(I, O, bgs)]
            if any(s is not None and not s for al in AL for s in al): continue
            if all(s is None or s == {0} for al in AL for s in al): continue
            for anchor in ('top', 'bottom', 'fgtop', 'fgbot'):
                y0s = [anchor_line(i, bg, anchor) for i, bg in zip(I, bgs0)]
                if any(y0 is None for y0 in y0s): continue
                # linear shear s = a * (y - y0)
                for aa in (1, -1, 2, -2):
                    ok = True
                    for i, al, y0 in zip(I, AL, y0s):
                        prof = shift_profile(i.shape[0], y0, 'lin', aa)
                        if mode == 'expand':
                            lo = min(prof); prof = [s - lo for s in prof]
                        for y, s in enumerate(prof):
                            if al[y] is None: continue
                            if (s % i.shape[1] if mode == 'roll' else s) not in al[y]: ok = False; break
                        if not ok: break
                    if ok:
                        found += 1
                        yield (f"arith-shift[{orient},{mode}{'' if fill is None else f',fill{fill}'},{anchor}]:lin{aa:+d}", 4,
                               lambda g, o_=orient, m=mode, an=anchor, aa=aa, fl=fill: apply_shift(g, o_, m, an, 'lin', aa, fl))
                        break
                if mode == 'expand': continue
                # periodic shift sequence s = S[(y - y0) mod k]
                for k in (2, 3, 4):
                    S = [None] * k; ok = True
                    for i, al, y0 in zip(I, AL, y0s):
                        w = i.shape[1]
                        for y in range(i.shape[0]):
                            if al[y] is None: continue
                            r = (y - y0) % k
                            cand = {s if mode != 'roll' else (s if s <= w // 2 else s - w) for s in al[y]}
                            S[r] = cand if S[r] is None else S[r] & cand
                            if not S[r]: ok = False; break
                        if not ok: break
                    if not ok or all(s is None for s in S): continue
                    SS = [None if s is None else min(s, key=lambda v: (abs(v), v)) for s in S]
                    if len({s for s in SS if s is not None}) < 2: continue
                    if sum(s is not None for s in SS) < k: continue
                    found += 1
                    yield (f"arith-shift[{orient},{mode},{anchor}]:per{k}{SS}", 5,
                           lambda g, o_=orient, m=mode, an=anchor, k=k, SS=tuple(SS): apply_shift(g, o_, m, an, 'per', (k, SS)))
                    break
                if found >= 4: return


# ------------------------------------------------------------------ + * : multiplicity = a * count + b
def count_features(train):
    """Count functions n(g) (name, fn), colours restricted to those present in every training input."""
    common = set.intersection(*[set(np.unique(A(p['input'])).tolist()) for p in train])
    feats = [('ncol', lambda g: len(set(np.unique(g).tolist()) - {bg_of(g.tolist())})),
             ('ncolall', lambda g: len(np.unique(g))),
             ('nfg', lambda g: int((g != bg_of(g.tolist())).sum())),
             ('nbg', lambda g: int((g == bg_of(g.tolist())).sum()))]
    for conn, mono in SEGS:
        feats.append((f'nobj{conn}{"m" if mono else "M"}', lambda g, s=(conn, mono): label(g, bg_of(g.tolist()), *s)[1]))
    for c in sorted(common):
        feats.append((f'ncell{c}', lambda g, c=c: int((g == c).sum())))
        for conn in (4, 8):
            feats.append((f'nobj{conn}c{c}', lambda g, c=c, conn=conn:
                          ndimage.label(g == c, structure=S4 if conn == 4 else S8)[1]))
    feats.append(('nminor', lambda g: min(Counter(g.ravel().tolist()).values())))
    return feats


def source(g, how):
    if how == 'grid': return g
    bg = bg_of(g.tolist()); ys, xs = np.nonzero(g != bg)
    if len(ys) == 0: return None
    return g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def multiply(S, op, k):
    if k < 1 or k * max(S.shape) > 30: return None
    if op == 'up': return np.kron(S, np.ones((k, k), dtype=S.dtype))
    if op == 'upy': return np.kron(S, np.ones((k, 1), dtype=S.dtype))
    if op == 'upx': return np.kron(S, np.ones((1, k), dtype=S.dtype))
    if op == 'tile': return np.tile(S, (k, k))
    if op == 'tiley': return np.tile(S, (k, 1))
    return np.tile(S, (1, k))


def fam_arith_count_scale(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    if len(train) < 2: return
    feats = None; n_yield = 0
    for how in ('grid', 'fg'):
        srcs = [source(i, how) for i in ins]
        if any(s is None for s in srcs): continue
        if how == 'fg' and all(s.shape == i.shape for s, i in zip(srcs, ins)): continue
        for op in ('up', 'tile', 'upx', 'upy', 'tilex', 'tiley'):
            ks = []
            for s, o in zip(srcs, outs):
                ky, kx = o.shape[0] / s.shape[0], o.shape[1] / s.shape[1]
                k = {'up': ky if ky == kx else 0, 'tile': ky if ky == kx else 0, 'upx': kx if ky == 1 else 0,
                     'upy': ky if kx == 1 else 0, 'tilex': kx if ky == 1 else 0, 'tiley': ky if kx == 1 else 0}[op]
                if k != int(k) or k < 1: break
                m = multiply(s, op, int(k))
                if m is None or not np.array_equal(m, o): break
                ks.append(int(k))
            if len(ks) != len(train) or len(set(ks)) < 2: continue
            if feats is None: feats = count_features(train)
            for fname, fn in feats:
                ns = [fn(i) for i in ins]
                for a, b in ((1, 0), (1, -1), (1, 1), (2, 0)):
                    if all(a * n + b == k for n, k in zip(ns, ks)):
                        n_yield += 1
                        yield (f"arith-count-scale[{how},{op}]:k={a}*{fname}{b:+d}", 4,
                               lambda g, how=how, op=op, fn=fn, a=a, b=b:
                               (lambda S: None if S is None else (lambda m: None if m is None else m.tolist())(
                                   multiply(S, op, a * fn(A(g)) + b)))(source(A(g), how)))
                        break
                if n_yield >= 3: return


# ------------------------------------------------------------------ S: union of panels, conflicts by colour order
def uniform_seps(a):
    """Candidate separators: (colour, full rows, full cols) of one uniform colour, most lines first."""
    out = []
    for c in np.unique(a).tolist():
        rs = [y for y in range(a.shape[0]) if np.all(a[y] == c)]
        cs = [x for x in range(a.shape[1]) if np.all(a[:, x] == c)]
        if (rs or cs) and len(rs) < a.shape[0] and len(cs) < a.shape[1]:
            out.append((c, rs, cs))
    out.sort(key=lambda t: -(len(t[1]) + len(t[2])))
    return out


def cut(lines, n):
    segs, s = [], 0
    for L in sorted(lines) + [n]:
        if L > s: segs.append((s, L))
        s = L + 1
    return segs


def panels(a, how, shape=None):
    """Equal-size panels of a grid: 'sep' (uniform separator lines), 'bNxM' (N x M blocks, no separators),
    'corners' (4 corner windows of a given shape, may overlap)."""
    if how == 'sep':
        for _, rs, cs in uniform_seps(a):
            P = [a[y0:y1, x0:x1] for y0, y1 in cut(rs, a.shape[0]) for x0, x1 in cut(cs, a.shape[1])]
            if len(P) >= 2 and all(p.shape == P[0].shape for p in P): return P
        return None
    elif how == 'corners':
        if shape is None: return None
        h, w = shape
        if h > a.shape[0] or w > a.shape[1] or (h, w) == a.shape: return None
        if min(h, w) < 2 or 2 * h < a.shape[0] - 1 or 2 * w < a.shape[1] - 1: return None
        P = [a[:h, :w], a[:h, -w:], a[-h:, :w], a[-h:, -w:]]
    else:
        n, m = map(int, how[1:].split('x'))
        if a.shape[0] % n or a.shape[1] % m: return None
        h, w = a.shape[0] // n, a.shape[1] // m
        P = [a[i * h:(i + 1) * h, j * w:(j + 1) * w] for i in range(n) for j in range(m)]
    if len(P) < 2 or any(p.shape != P[0].shape for p in P) or min(P[0].shape) < 2: return None
    return P


def panel_bg(P, pref=None):
    allv = np.concatenate([p.ravel() for p in P])
    if pref is not None and np.any(allv == pref): return pref
    return Counter(allv.tolist()).most_common(1)[0][0]


def closure(edges, cols):
    """Transitive closure of 'x beats y' edges; None when cyclic."""
    beats = {c: set() for c in cols}
    for x, y in edges: beats[x].add(y)
    changed = True
    while changed:
        changed = False
        for x in cols:
            new = set().union(*[beats[y] for y in beats[x]]) if beats[x] else set()
            if not new <= beats[x]: beats[x] |= new; changed = True
    if any(x in beats[x] for x in cols): return None
    return beats


def union_apply(g, how, shape, beats, tbg=None):
    P = panels(A(g), how, shape)
    if P is None: return None
    bg = panel_bg(P, tbg); h, w = P[0].shape
    out = np.full((h, w), bg, dtype=np.int64)
    for y in range(h):
        for x in range(w):
            vals = {int(p[y, x]) for p in P} - {bg}
            if not vals: continue
            if len(vals) == 1: out[y, x] = vals.pop(); continue
            top = [v for v in vals if v in beats and all(u in beats[v] for u in vals if u != v)]
            if len(top) != 1: return None
            out[y, x] = top[0]
    return out.tolist()


def fam_arith_union(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    if any(o.shape == i.shape for i, o in zip(ins, outs)): return
    oshapes = {o.shape for o in outs}
    hows = ['sep'] + [f'b{n}x{m}' for n in (1, 2, 3, 4) for m in (1, 2, 3, 4) if n * m >= 2]
    if len(oshapes) == 1: hows.append('corners')
    shape = next(iter(oshapes)) if len(oshapes) == 1 else None
    n_yield = 0
    for how in hows:
        Ps = [panels(i, how, shape) for i in ins]
        if any(P is None or P[0].shape != o.shape for P, o in zip(Ps, outs)): continue
        tbg = Counter(np.concatenate([p.ravel() for P in Ps for p in P]).tolist()).most_common(1)[0][0]
        edges, cols, ok, multi = set(), set(), True, 0
        for P, o in zip(Ps, outs):
            bg = panel_bg(P, tbg); h, w = o.shape
            for y in range(h):
                for x in range(w):
                    vals = {int(p[y, x]) for p in P} - {bg}
                    v = int(o[y, x])
                    if not vals:
                        if v != bg: ok = False
                        continue
                    if v not in vals: ok = False; break
                    if len(vals) > 1: multi += 1
                    cols |= vals
                    edges |= {(v, u) for u in vals if u != v}
                if not ok: break
            if not ok: break
        beats = closure(edges, cols) if ok else None
        if beats is not None:
            n_yield += 1
            yield (f"arith-union[{how}]:{'colour-order' if multi else 'or'}", 4,
                   lambda g, how=how, shape=shape, beats=beats, tbg=tbg: union_apply(g, how, shape, beats, tbg))
            if n_yield >= 2: return
            continue
        # membership count: out = T[number of panels holding a figure cell here]
        T = count_table(Ps, outs, tbg)
        if T is not None:
            n_yield += 1
            yield (f"arith-union[{how}]:count{sorted(T.items())}", 4,
                   lambda g, how=how, shape=shape, T=T, tbg=tbg: count_apply(g, how, shape, T, tbg))
            if n_yield >= 2: return


def count_table(Ps, outs, tbg=None):
    """count -> literal colour | 'val' (the common figure colour) | 'bg'; must differ from a plain union."""
    T = {}
    for P, o in zip(Ps, outs):
        bg = panel_bg(P, tbg); st = np.stack(P); fgm = st != bg
        cnt = fgm.sum(axis=0)
        for c in np.unique(cnt).tolist():
            m = cnt == c
            ov = o[m]
            opts = set()
            if np.all(ov == ov[0]): opts.add(('lit', int(ov[0])) if ov[0] != bg else ('bg',))
            if c > 0:
                # the figure colour of each cell (all figure panels agree)
                fv = np.where(fgm[:, m], st[:, m], -1).max(axis=0)
                agree = np.all((st[:, m] == fv) | ~fgm[:, m])
                if agree and np.array_equal(ov, fv): opts.add(('val',))
            T[c] = opts if c not in T else T[c] & opts
            if not T[c]: return None
    res = {}
    for c, opts in T.items():
        res[c] = sorted(opts)[0] if ('val',) not in opts or len(opts) == 1 else ('val',)
        if ('bg',) in opts and c == 0: res[c] = ('bg',)
    plain = all(v == ('val',) or (c == 0 and v == ('bg',)) for c, v in res.items())
    if plain or len(res) < 2: return None
    return res


def count_apply(g, how, shape, T, tbg=None):
    P = panels(A(g), how, shape)
    if P is None: return None
    bg = panel_bg(P, tbg); st = np.stack(P); fgm = st != bg
    cnt = fgm.sum(axis=0)
    out = np.full(cnt.shape, bg, dtype=np.int64)
    for c in np.unique(cnt).tolist():
        if c not in T: return None
        m = cnt == c; act = T[c]
        if act[0] == 'lit': out[m] = act[1]
        elif act[0] == 'val':
            fv = np.where(fgm[:, m], st[:, m], -1).max(axis=0)
            if not np.all((st[:, m] == fv) | ~fgm[:, m]): return None
            out[m] = fv
    return out.tolist()


# ------------------------------------------------------------------ + : arithmetic progression of parallel bars
def seed_bar(a, bg):
    """The single straight 1-thick bar of one colour formed by all fg cells, in a frame where the bar is a row
    segment anchored at the left edge: returns (transform name, r0, L, colour) or None."""
    ys, xs = np.nonzero(a != bg)
    if len(ys) < 2 or len(set(a[ys, xs].tolist())) != 1: return None
    col = int(a[ys[0], xs[0]])
    if len(set(ys.tolist())) == 1 and xs.max() - xs.min() + 1 == len(xs):
        if xs.min() == 0: return ('id', int(ys[0]), len(xs), col)
        if xs.max() == a.shape[1] - 1: return ('fh', int(ys[0]), len(xs), col)
    if len(set(xs.tolist())) == 1 and ys.max() - ys.min() + 1 == len(ys):
        if ys.min() == 0: return ('T', int(xs[0]), len(ys), col)
        if ys.max() == a.shape[0] - 1: return ('Tfh', int(xs[0]), len(ys), col)
    return None


def to_frame(a, t):
    if t == 'id': return a
    if t == 'fh': return a[:, ::-1]
    if t == 'T': return a.T
    return a.T[:, ::-1]


def from_frame(a, t):
    if t == 'id': return a
    if t == 'fh': return a[:, ::-1]
    if t == 'T': return a.T
    return a[:, ::-1].T


def progression_apply(g, steps, ctab):
    a = A(g); bg = bg_of(g); sb = seed_bar(a, bg)
    if sb is None: return None
    t, r0, L, col = sb
    f = to_frame(a, t).copy(); n, w = f.shape
    for side, step in zip((-1, 1), steps):
        if step is None: continue
        d = 1
        while 0 <= r0 + side * d < n:
            ln = min(w, L + step * d)
            if ln <= 0: break
            key = (side, d % ctab['m'])
            if key not in ctab['tab']: return None
            c = ctab['tab'][key]
            f[r0 + side * d, :ln] = col if c == 'seed' else c
            d += 1
    return from_frame(f, t).tolist()


def fam_arith_progression(train):
    """Bars parallel to a seed bar at offset d on each side have length L + a_side * d (a_side in -3..3, or no
    bars on that side), anchored at the seed's grid-edge end; colour = table (side, d mod m) of literal colours
    or the seed colour."""
    if not same_shape(train): return
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    seeds = [seed_bar(i, bg_of(p['input'])) for i, p in zip(ins, train)]
    if any(s is None for s in seeds): return
    lens = [[], []]                            # per side: (d, L, w, observed length)
    obs = []                                   # (side, d, colour, is_seed_colour)
    for (t, r0, L, col), i, o, p in zip(seeds, ins, outs, train):
        fi, fo = to_frame(i, t), to_frame(o, t); bg = bg_of(p['input']); n, w = fo.shape
        if not np.array_equal(fo[r0], fi[r0]): return
        if np.any(fi[np.arange(n) != r0] != bg): return
        for si, side in enumerate((-1, 1)):
            d = 1
            while 0 <= r0 + side * d < n:
                row = fo[r0 + side * d]
                nz = np.flatnonzero(row != bg)
                ln = len(nz)
                if ln and (nz[-1] != ln - 1 or len(set(row[:ln].tolist())) != 1): return
                lens[si].append((d, L, w, ln))
                if ln: obs.append((side, d, int(row[0]), int(row[0]) == col))
                d += 1
    if not obs: return
    st = []
    for si in (0, 1):
        if all(ln == 0 for _, _, _, ln in lens[si]):
            st.append(None); continue
        ok = [s for s in range(-3, 4) if all(min(w, max(0, L + s * d)) == ln for d, L, w, ln in lens[si])]
        if not ok: return
        st.append(min(ok, key=lambda v: (abs(v), v)))
    st = tuple(st)
    if all(s in (0, None) for s in st): return
    for m in (1, 2):
        tab = {}; ok = True
        for side, d, c, isseed in obs:
            key = (side, d % m)
            v = 'seed' if all(o[3] for o in obs if (o[0], o[1] % m) == key) else c
            if tab.setdefault(key, v) != v: ok = False; break
        if ok:
            yield (f"arith-progression:steps{st}:m{m}", 4,
                   lambda g, st=st, ct={'m': m, 'tab': tab}: progression_apply(g, st, ct))
            return


# ------------------------------------------------------------------ ===: completion of congruence classes
def lattice_classes(h, w, p, q, sft):
    """Class of every cell under the translation lattice generated by (p, sft) and (0, q)."""
    Y, X = np.indices((h, w))
    return (Y % p) * q + (X - sft * (Y // p)) % q


def vec_ok(a, known, dy, dx):
    """Known cells p and p + (dy, dx) (dy >= 0) always agree."""
    h, w = a.shape
    if dy >= h or abs(dx) >= w: return True
    x0, x1 = max(0, -dx), w - max(0, dx)
    A1, A2 = a[0:h - dy, x0:x1], a[dy:h, x0 + dx:x1 + dx]
    K = known[0:h - dy, x0:x1] & known[dy:h, x0 + dx:x1 + dx]
    return bool(np.all(A1[K] == A2[K]))


def congruence_fill(a, u, rep=2):
    """Fill cells of colour u from the smallest translation lattice <(p, sft), (0, q)> whose classes are
    single-coloured on known cells and all observed (>= 2 known cells per class on average)."""
    h, w = a.shape; known = a != u
    nk = int(known.sum())
    if nk == 0 or nk == a.size: return None
    Q = [q for q in range(1, w + 1) if q == w or vec_ok(a, known, 0, q)]
    cands = []
    for q in Q:
        for p in range(1, h + 1):
            if p * q < 2 or rep * p * q > nk or p * q >= nk or (p == h and q == w): continue
            for sft in (range(q) if p < h else (0,)):
                cands.append((p * q, sft != 0, p, q, sft))
    cands.sort()
    av = a[known]
    for _, _, p, q, sft in cands:
        if p < h and not vec_ok(a, known, p, sft if sft <= q // 2 else sft - q): continue
        full = lattice_classes(h, w, p, q, sft)
        cl = full[known]
        keys = np.unique(cl * 16 + av)
        ucl = np.unique(keys // 16)
        if len(ucl) != p * q or len(keys) != len(ucl): continue
        colour = np.zeros(p * q, dtype=np.int64); colour[keys // 16] = keys % 16
        out = a.copy()
        out[~known] = colour[full[~known]]
        return out
    return None


def unknown_colours(train):
    """Candidate unknown colours: the input background, and colours present in every input but in no output."""
    cands = []
    bgs = {bg_of(p['input']) for p in train}
    if len(bgs) == 1: cands.append(('bg', None))
    occ = set.intersection(*[set(np.unique(A(p['input'])).tolist()) - set(np.unique(A(p['output'])).tolist())
                             for p in train])
    for c in sorted(occ): cands.append(('c', c))
    return cands


def fam_arith_congruence(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    for kind, c in unknown_colours(train):
        us = [bg_of(p['input']) if kind == 'bg' else c for p in train]
        for mode in ('grid', 'patch'):
            ok = True
            for i, o, u in zip(ins, outs, us):          # cheap preconditions before any lattice search
                if mode == 'grid' and (i.shape != o.shape or not np.array_equal(i[i != u], o[i != u])
                                       or np.any(o[i == u] == u) or not np.any(i == u)):
                    ok = False; break
            if not ok: continue
            for i, o, u in zip(ins, outs, us):
                if mode == 'grid' and i.shape != o.shape: ok = False; break
                if mode == 'patch':
                    ys, xs = np.nonzero(i == u)
                    if len(ys) == 0 or (ys.max() - ys.min() + 1, xs.max() - xs.min() + 1) != o.shape: ok = False; break
                f = congruence_fill(i, u)
                if f is None: ok = False; break
                if mode == 'patch': f = f[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
                if not np.array_equal(f, o): ok = False; break
            if not ok: continue
            def fn(g, kind=kind, c=c, mode=mode):
                a = A(g); u = bg_of(g) if kind == 'bg' else c
                f = congruence_fill(a, u)
                if f is None: return None
                if mode == 'patch':
                    ys, xs = np.nonzero(a == u)
                    f = f[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
                return f.tolist()
            yield (f"arith-congruence[{kind}{'' if c is None else c},{mode}]", 4, fn)
            return
    # extension: the output canvas is larger; the input (all known) is continued by its smallest lattice
    if all(o.shape[0] >= i.shape[0] and o.shape[1] >= i.shape[1] and o.shape != i.shape for i, o in zip(ins, outs)):
        from fractions import Fraction as Fr
        rat = {(Fr(o.shape[0], i.shape[0]), Fr(o.shape[1], i.shape[1])) for i, o in zip(ins, outs)}
        shp = {o.shape for o in outs}
        rule = ('ratio', rat.pop()) if len(rat) == 1 else ('const', shp.pop()) if len(shp) == 1 else None
        if rule is None: return
        def ext(g, rule=rule):
            a = A(g)
            if rule[0] == 'ratio':
                ry, rx = rule[1]
                if (a.shape[0] * ry).denominator != 1 or (a.shape[1] * rx).denominator != 1: return None
                hh, ww = int(a.shape[0] * ry), int(a.shape[1] * rx)
            else:
                hh, ww = rule[1]
            if hh < a.shape[0] or ww < a.shape[1] or hh > 30 or ww > 30: return None
            cv = np.full((hh, ww), 15, dtype=np.int64); cv[:a.shape[0], :a.shape[1]] = a
            f = congruence_fill(cv, 15, rep=1)
            return None if f is None else f.tolist()
        from gdsl import fit_cmap, apply_cmap
        preds = [ext(p['input']) for p in train]
        if any(r is None for r in preds): return
        m = fit_cmap(preds, [p['output'] for p in train])
        if m is None or any(apply_cmap(r, m) != p['output'] for r, p in zip(preds, train)): return
        yield (f"arith-congruence[extend,{rule[0]}]", 4, ext)


# ------------------------------------------------------------------ S: complement (figure <-> ground) then repeat
def complement(a, ground=None):
    """Swap figure and ground of a 2-colour grid (ground = induced colour, else the majority colour)."""
    cs = np.unique(a).tolist()
    if len(cs) != 2: return None
    bg = ground if ground in cs else bg_of(a.tolist())
    fgc = cs[0] if cs[1] == bg else cs[1]
    return np.where(a == bg, fgc, bg), bg


def comp_apply(g, how, shape, ground):
    a = A(g); r = complement(a, ground)
    if r is None: return None
    c, bg = r
    if how == 'id': return c.tolist()
    if how == 'tile':
        n, m = shape
        return np.tile(c, (n, m)).tolist()
    h, w = a.shape
    out = np.full((h * h, w * w), bg, dtype=np.int64)
    for y in range(h):
        for x in range(w):
            if (a[y, x] != bg) == (how == 'kron-fg'):
                out[y * h:(y + 1) * h, x * w:(x + 1) * w] = c
    return out.tolist()


def fam_arith_complement(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    if any(len(np.unique(i)) != 2 for i in ins): return
    common = set.intersection(*[set(np.unique(i).tolist()) for i in ins])
    ground = common.pop() if len(common) == 1 else None
    i0, o0 = ins[0], outs[0]
    hows = []
    if all(i.shape == o.shape for i, o in zip(ins, outs)): hows.append(('id', None))
    if o0.shape[0] % i0.shape[0] == 0 and o0.shape[1] % i0.shape[1] == 0:
        sh = (o0.shape[0] // i0.shape[0], o0.shape[1] // i0.shape[1])
        if sh != (1, 1): hows.append(('tile', sh))
    if all(o.shape == (i.shape[0] ** 2, i.shape[1] ** 2) for i, o in zip(ins, outs)):
        hows += [('kron-fg', None), ('kron-bg', None)]
    for how, sh in hows:
        if all(comp_apply(p['input'], how, sh, ground) == p['output'] for p in train):
            yield (f"arith-complement[{how}{'' if sh is None else sh}]", 4,
                   lambda g, how=how, sh=sh, gr=ground: comp_apply(g, how, sh, gr))


# ------------------------------------------------------------------ |: quotient windows (divisibility, parity)
def min_period(a):
    """Smallest (p, q) with p | H, q | W such that the grid is p-periodic in y and q-periodic in x."""
    h, w = a.shape
    p = next(d for d in range(1, h + 1) if h % d == 0 and np.array_equal(a, np.tile(a[:d], (h // d, 1))))
    q = next(d for d in range(1, w + 1) if w % d == 0 and np.array_equal(a, np.tile(a[:, :d], (1, w // d))))
    return p, q


def win_size(a, rule):
    h, w = a.shape
    if rule[0] == 'const': return rule[1], rule[2]
    if rule[0] == 'div':
        ky, kx = rule[1], rule[2]
        if h % ky or w % kx: return None
        return h // ky, w // kx
    p, q = min_period(a)
    return (p, q) if (p, q) != (h, w) else None


def window(a, size, corner):
    h, w = a.shape; wh, ww = size
    if wh > h or ww > w or wh < 2 or ww < 2: return None
    if corner == 'C':
        if (h - wh) % 2 or (w - ww) % 2: return None
        y0, x0 = (h - wh) // 2, (w - ww) // 2
    else:
        y0 = 0 if corner[0] == 'T' else h - wh
        x0 = 0 if corner[1] == 'L' else w - ww
    return a[y0:y0 + wh, x0:x0 + ww]


def window_apply(g, rule, ctab):
    a = A(g); size = win_size(a, rule)
    if size is None: return None
    corner = ctab.get((a.shape[0] % 2, a.shape[1] % 2)) if isinstance(ctab, dict) else ctab
    if corner is None: return None
    r = window(a, size, corner)
    return None if r is None else r.tolist()


def fam_arith_window(train):
    ins = [A(p['input']) for p in train]; outs = [A(p['output']) for p in train]
    if any(o.shape[0] > i.shape[0] or o.shape[1] > i.shape[1] or o.shape == i.shape for i, o in zip(ins, outs)): return
    if any(min(o.shape) < 2 for o in outs): return
    rules = []
    if len({o.shape for o in outs}) == 1: rules.append(('const',) + outs[0].shape)
    rat = {(i.shape[0] / o.shape[0], i.shape[1] / o.shape[1]) for i, o in zip(ins, outs)}
    if len(rat) == 1:
        ky, kx = rat.pop()
        if ky == int(ky) and kx == int(kx): rules.append(('div', int(ky), int(kx)))
    rules.append(('period',))
    n = 0
    for rule in rules:
        sizes = [win_size(i, rule) for i in ins]
        if any(sz is None or sz != o.shape for sz, o in zip(sizes, outs)): continue
        hits = []
        for i, o, sz in zip(ins, outs, sizes):
            hits.append({c for c in ('TL', 'TR', 'BL', 'BR', 'C')
                         if (lambda r: r is not None and np.array_equal(r, o))(window(i, sz, c))})
        common = set.intersection(*hits)
        if common:
            c = sorted(common, key=lambda c: ('TL', 'TR', 'BL', 'BR', 'C').index(c))[0]
            n += 1
            yield (f"arith-window[{rule[0]}]:{c}", 3 if rule[0] != 'period' else 4,
                   lambda g, rule=rule, c=c: window_apply(g, rule, c))
        else:                          # corner chosen by the parity of the grid size
            tab = {}; ok = True
            for i, hs in zip(ins, hits):
                key = (i.shape[0] % 2, i.shape[1] % 2)
                tab[key] = tab.get(key, hs) & hs
                if not tab[key]: ok = False; break
            if not ok or len(tab) < 2: continue
            ct = {k: sorted(v)[0] for k, v in tab.items()}
            if len(set(ct.values())) < 2: continue
            n += 1
            yield (f"arith-window[{rule[0]}]:parity{sorted(ct.items())}", 4,
                   lambda g, rule=rule, ct=ct: window_apply(g, rule, ct))
        if n >= 2: return


FAMILIES = (fam_arith_residue, fam_arith_shift, fam_arith_count_scale, fam_arith_union, fam_arith_progression,
            fam_arith_congruence, fam_arith_complement, fam_arith_window)


In [ ]:
%%writefile /kaggle/working/probe/prior_combinatorics.py
"""COMBINATORICS prior: the output is a PERMUTATION of parts of the input.

Concept
-------
A grid is decomposed into an ordered sequence of parts (slots).  The output keeps the slot
structure but reassigns content to slots by a permutation pi.  pi is never memorised per task;
it is explained by one of a small set of generic permutation generators:

    rev            pi(i) = n-1-i                         (reversal)
    shift+k        pi(i) = (i+k) mod n, k in {+-1,+-2}   (cyclic rotation)
    sort:key:dir   items stably sorted by an induced key (asc/desc)
    fixed          a constant index permutation (only when n is identical in every pair and test)
    slot order     items (in canonical order) written into slots in another traversal
                   (row-major / col-major / snake / reversed) -- dihedral action on a panel lattice

Search space (all parameters induced from the training pairs, verified on every pair)
-------------------------------------------------------------------------------------
F1 axis parts   : rows|cols decomposed as unit lines, k-line groups, separator-delimited
                  strips (separator colour induced) or bands (maximal runs with equal majority colour);
                  the part sequence is permuted (rev / shift / sort by key / fixed).
F2 panel lattice: separator lattice of equal panels; panels sorted by key (or kept) and written
                  into a slot traversal (8 traversals).
F3 item attrs   : items = objects (or cells of a simple path); ordered along x, y, angle about
                  their common centroid, nesting (bbox area) or the path; the COLOUR sequence and
                  the SHAPE sequence (shape re-anchored at an alignment point) are permuted
                  independently (id / rev / shift / sort by size).  Rules are inferred by matching
                  the input and output item sequences, then re-verified by the harness.
F4 legend perm  : a small solid legend block of adjacent colour pairs defines a colour permutation
                  (swap = transpositions, or directed map) applied to the rest of the grid.
F5 sort & pack  : objects / panels cropped, ordered by key or position, concatenated along an axis
                  (optionally with an induced separator line between them).
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
import math
from gdsl import H, W, T, bg_of, objects, bbox, crop, colours, static_colours

# ------------------------------------------------------------------ permutation generators
SHIFTS = (1, -1, 2, -2)

def perm_rules(n, keys=None):
    """Yield (rule_name, perm) where new position i receives item perm[i]."""
    if n < 2: return
    yield 'rev', list(range(n - 1, -1, -1))
    for k in SHIFTS:
        if abs(k) < n:
            yield f'shift{k:+d}', [(i + k) % n for i in range(n)]
    if keys:
        for kn, kv in keys.items():
            if len(set(kv)) < 2: continue
            yield f'sort:{kn}:asc', sorted(range(n), key=lambda i: (kv[i], i))
            yield f'sort:{kn}:desc', sorted(range(n), key=lambda i: (-kv[i], i))

def apply_rule(rule, n, keys=None, fixed=None):
    if rule == 'id': return list(range(n))
    if rule == 'rev': return list(range(n - 1, -1, -1))
    if rule.startswith('shift'):
        k = int(rule[5:])
        if abs(k) >= n: return None
        return [(i + k) % n for i in range(n)]
    if rule.startswith('sort:'):
        _, kn, d = rule.split(':')
        kv = keys.get(kn) if keys else None
        if kv is None: return None
        s = 1 if d == 'asc' else -1
        return sorted(range(n), key=lambda i: (s * kv[i], i))
    if rule == 'fixed':
        return list(fixed) if fixed is not None and len(fixed) == n else None
    return None

def infer_fixed(src, dst):
    """Index permutation p with dst[i] == src[p[i]] (greedy on duplicates)."""
    if len(src) != len(dst): return None
    used = set(); p = []
    for d in dst:
        j = next((j for j, s in enumerate(src) if j not in used and s == d), None)
        if j is None: return None
        used.add(j); p.append(j)
    return p

def part_keys(parts, bg, cols):
    """Generic sort keys of sub-grids."""
    keys = {'fg': [sum(v != bg for r in p for v in r) for p in parts],
            'ncol': [len({v for r in p for v in r} - {bg}) for p in parts],
            'nobj': [len(objects(p, bg, False, True)) for p in parts]}
    for c in sorted(cols):
        keys[f'c{c}'] = [sum(v == c for r in p for v in r) for p in parts]
    return keys

# ------------------------------------------------------------------ F1: axis parts
def axis_parts(lines, mode):
    """lines: list of tuples. Returns (segments, movable flags) covering all lines in order."""
    n = len(lines)
    if mode == 'unit':
        return [(i, i + 1) for i in range(n)], [True] * n
    if mode.startswith('k'):
        k = int(mode[1:])
        if n % k or n // k < 2: return None
        return [(i, i + k) for i in range(0, n, k)], [True] * (n // k)
    if mode == 'sep':
        # separator colour: a colour whose uniform lines cut the axis into >= 2 equal parts
        ucols = Counter(ln[0] for ln in lines if len(set(ln)) == 1)
        for sc, _ in sorted(ucols.items(), key=lambda t: (t[1], t[0])):
            issep = [len(set(ln)) == 1 and ln[0] == sc for ln in lines]
            if all(issep): continue
            segs, mov = [], []; i = 0
            while i < n:
                j = i
                while j < n and issep[j] == issep[i]: j += 1
                segs.append((i, j)); mov.append(not issep[i]); i = j
            if sum(mov) < 2: continue
            if len({b - a for (a, b), m in zip(segs, mov) if m}) != 1: continue
            return segs, mov
        return None
    if mode == 'band':
        sig = [Counter(ln).most_common(1)[0][0] for ln in lines]
        segs = []; i = 0
        while i < n:
            j = i
            while j < n and sig[j] == sig[i]: j += 1
            segs.append((i, j)); i = j
        if len(segs) < 2 or len(segs) == n: return None
        return segs, [True] * len(segs)
    return None

def permute_axis(g, axis, mode, rule, keyname, fixed):
    lines = [tuple(r) for r in (g if axis == 'rows' else T(g))]
    ap = axis_parts(lines, mode)
    if not ap: return None
    segs, mov = ap
    items = [lines[a:b] for (a, b), m in zip(segs, mov) if m]
    n = len(items)
    keys = None
    if rule.startswith('sort:'):
        bg = bg_of(g)
        parts = [[list(l) for l in it] for it in items]
        keys = part_keys(parts, bg, colours(g))
    p = apply_rule(rule, n, keys, fixed)
    if p is None: return None
    new = [items[j] for j in p]
    if mode != 'sep' and any(len(a) != len(b) for a, b in zip(new, items)) and mode != 'band':
        return None
    out = []; it = iter(new)
    for (a, b), m in zip(segs, mov):
        out.extend(next(it) if m else lines[a:b])
    out = [list(r) for r in out]
    return out if axis == 'rows' else T(out)

def fam_axis_permute(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if all(p['input'] == p['output'] for p in train): return
    for axis in ('rows', 'cols'):
        # the multiset of lines across the axis must be preserved in every pair
        def L(g): return [tuple(r) for r in (g if axis == 'rows' else T(g))]
        if not all(Counter(L(p['input'])) == Counter(L(p['output'])) for p in train): continue
        n0 = len(L(i0))
        modes = ['unit', 'sep', 'band'] + [f'k{k}' for k in (2, 3, 4, 5) if n0 % k == 0]
        for mode in modes:
            aps = [axis_parts(L(p['input']), mode) for p in train]
            if not all(aps): continue
            # candidate rules consistent with pair 0
            ok_rules = []
            fixed = None
            segs, mov = aps[0]; lines = L(i0); lo = L(o0)
            items = [tuple(lines[a:b]) for (a, b), m in zip(segs, mov) if m]
            n = len(items)
            if n < 2: continue
            parts = [[list(l) for l in it] for it in items]
            keys = part_keys(parts, bg_of(i0), colours(i0))
            cand = ['rev'] + [f'shift{k:+d}' for k in SHIFTS] + \
                   [f'sort:{kn}:{d}' for kn in keys for d in ('asc', 'desc')]
            ns = {len([m for m in axis_parts(L(p['input']), mode)[1] if m]) for p in train}
            if len(ns) == 1:
                ns_test = True
                # fixed permutation inferred from pair 0 (output segmented like input)
                if mode != 'band':
                    outs_items = [tuple(lo[a:b]) for (a, b), m in zip(segs, mov) if m]
                    fixed = infer_fixed(items, outs_items)
                    if fixed and fixed != list(range(n)): cand.append('fixed')
            for rule in cand:
                if all(permute_axis(p['input'], axis, mode, rule, None, fixed) == p['output'] for p in train):
                    ok_rules.append(rule)
            for rule in ok_rules[:6]:
                yield (f'perm-axis[{axis},{mode}]:{rule}', 4 if rule != 'fixed' else 5,
                       lambda g, axis=axis, mode=mode, rule=rule, fixed=fixed: permute_axis(g, axis, mode, rule, None, fixed))

# ------------------------------------------------------------------ F2: panel lattice
def lattice(g):
    """Separator lattice: returns (row_ranges, col_ranges) of equal-size panels, or None."""
    h, w = H(g), W(g)
    best = None
    for sc in colours(g):
        rows = [r for r in range(h) if all(v == sc for v in g[r])]
        cols = [c for c in range(w) if all(g[r][c] == sc for r in range(h))]
        if not rows and not cols: continue
        if len(rows) == h or len(cols) == w: continue
        def runs(sep, n):
            out = []; s = 0
            for x in list(sep) + [n]:
                if x > s: out.append((s, x))
                s = x + 1
            return out
        rr, cc = runs(rows, h), runs(cols, w)
        if len(rr) * len(cc) < 2: continue
        if len({b - a for a, b in rr}) != 1 or len({b - a for a, b in cc}) != 1: continue
        if best is None or len(rr) * len(cc) > len(best[0]) * len(best[1]): best = (rr, cc)
    return best

def slot_order(R, C, how):
    """Traversal of an R x C lattice: primary axis (row/col), start corner, plain or boustrophedon."""
    prim, fr, fc, snake = how.split('.')
    rs = list(range(R))[::-1] if fr == '1' else list(range(R))
    cs = list(range(C))[::-1] if fc == '1' else list(range(C))
    o = []
    if prim == 'row':
        for k, r in enumerate(rs): o += [(r, c) for c in (cs[::-1] if snake == 's' and k % 2 else cs)]
    else:
        for k, c in enumerate(cs): o += [(r, c) for r in (rs[::-1] if snake == 's' and k % 2 else rs)]
    return o

SLOTS = tuple(f'{p}.{a}.{b}.{s}' for p in ('row', 'col') for a in '01' for b in '01' for s in ('p', 's'))

def permute_lattice_fixed(g, fixed):
    lt = lattice(g)
    if not lt: return None
    rr, cc = lt
    boxes = [(a, b, e, f) for (a, e) in rr for (b, f) in cc]
    if len(boxes) != len(fixed): return None
    panels = [crop(g, (a, b, e - 1, f - 1)) for a, b, e, f in boxes]
    out = [r[:] for r in g]
    for (a, b, e, f), j in zip(boxes, fixed):
        for y in range(e - a):
            for x in range(f - b): out[a + y][b + x] = panels[j][y][x]
    return out

def permute_lattice(g, key, d, slots):
    lt = lattice(g)
    if not lt: return None
    rr, cc = lt
    panels = [(r, c, crop(g, (a, b, e - 1, f - 1))) for r, (a, e) in enumerate(rr) for c, (b, f) in enumerate(cc)]
    bg = bg_of(g)
    if key == 'none':
        order = list(range(len(panels)))
    else:
        keys = part_keys([p[2] for p in panels], bg, colours(g))
        if key not in keys: return None
        s = 1 if d == 'asc' else -1
        order = sorted(range(len(panels)), key=lambda i: (s * keys[key][i], i))
    so = slot_order(len(rr), len(cc), slots)
    out = [r[:] for r in g]
    for (r, c), j in zip(so, order):
        (a, e), (b, f) = rr[r], cc[c]
        pg = panels[j][2]
        for y in range(e - a):
            for x in range(f - b):
                out[a + y][b + x] = pg[y][x]
    return out

def fam_lattice_permute(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if not lattice(i0): return
    keys = list(part_keys([i0], bg_of(i0), colours(i0)).keys())
    cnt = 0
    lt = lattice(i0); lo = lattice(o0)
    if lt and lt == lo and len({len(lattice(p['input'])[0]) * len(lattice(p['input'])[1]) if lattice(p['input']) else 0 for p in train}) == 1:
        rr, cc = lt
        pi = [str(crop(i0, (a, b, e - 1, f - 1))) for (a, e) in rr for (b, f) in cc]
        po = [str(crop(o0, (a, b, e - 1, f - 1))) for (a, e) in rr for (b, f) in cc]
        fx = infer_fixed(pi, po)
        if fx and fx != list(range(len(fx))) and len(set(pi)) == len(pi):
            fn = lambda g, fx=fx: permute_lattice_fixed(g, fx)
            if all(fn(p['input']) == p['output'] for p in train):
                cnt += 1
                yield ('perm-lattice:fixed', 5, fn)
    for key in ['none'] + keys:
        for d in (('asc',) if key == 'none' else ('asc', 'desc')):
            for sl in SLOTS:
                if key == 'none' and sl == 'row.0.0.p': continue
                if all(permute_lattice(p['input'], key, d, sl) == p['output'] for p in train):
                    cnt += 1
                    yield (f'perm-lattice:{key}:{d}->{sl}', 4 if key == 'none' else 5,
                           lambda g, key=key, d=d, sl=sl: permute_lattice(g, key, d, sl))
                    if cnt >= 4: return

# ------------------------------------------------------------------ F3: item attribute permutation
def get_items(g, bg, mode, st):
    """Items = list of cell lists; mode: obj4 / obj8 (single colour objects, static colours optionally excluded) / path."""
    if mode == 'path':
        obs = objects(g, bg, False, False)
        obs = [o for o in obs if not all(g[y][x] in st for y, x in o)]
        if len(obs) != 1: return None
        ob = set(obs[0])
        deg = {c: sum((c[0] + dy, c[1] + dx) in ob for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1))) for c in ob}
        ends = [c for c, k in deg.items() if k == 1]
        if len(ends) != 2 or any(k > 2 for k in deg.values()): return None
        cur = min(ends); seq = [cur]; seen = {cur}
        while True:
            nx = [(cur[0] + dy, cur[1] + dx) for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1))]
            nx = [c for c in nx if c in ob and c not in seen]
            if not nx: break
            cur = nx[0]; seq.append(cur); seen.add(cur)
        if len(seq) != len(ob): return None
        return [[c] for c in seq]
    diag = mode.endswith('8')
    obs = objects(g, -1 if 'all' in mode else bg, diag, True)
    if mode.startswith('ns'):
        obs = [o for o in obs if g[o[0][0]][o[0][1]] not in st]
    return obs

def order_items(items, how):
    if how == 'path': return items
    if how == 'x': return sorted(items, key=lambda o: (bbox(o)[1], bbox(o)[0]))
    if how == 'y': return sorted(items, key=lambda o: (bbox(o)[0], bbox(o)[1]))
    if how == 'nest':
        return sorted(items, key=lambda o: (-(bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1), bbox(o)))
    if how == 'angle':
        cs = [(sum(y for y, _ in o) / len(o), sum(x for _, x in o) / len(o)) for o in items]
        cy = sum(c[0] for c in cs) / len(cs); cx = sum(c[1] for c in cs) / len(cs)
        ang = [math.atan2(c[1] - cx, -(c[0] - cy)) % (2 * math.pi) for c in cs]
        if any(abs(c[0] - cy) < 1e-9 and abs(c[1] - cx) < 1e-9 for c in cs): return None
        return [o for _, o in sorted(zip(ang, items), key=lambda t: (round(t[0], 9), bbox(t[1])))]
    return None

def norm_shape(o):
    r0, c0, _, _ = bbox(o)
    return tuple(sorted((y - r0, x - c0) for y, x in o))

def anchor(o, align):
    r0, c0, r1, c1 = bbox(o)
    if align == 'tl': return (r0, c0)
    if align == 'tr': return (r0, c1)
    if align == 'bl': return (r1, c0)
    if align == 'br': return (r1, c1)
    return (r0 + r1, c0 + c1)          # centre (doubled coordinates)

def place(shape, a, align):
    hs = max(y for y, _ in shape) + 1; ws = max(x for _, x in shape) + 1
    if align == 'tl': r0, c0 = a
    elif align == 'tr': r0, c0 = a[0], a[1] - ws + 1
    elif align == 'bl': r0, c0 = a[0] - hs + 1, a[1]
    elif align == 'br': r0, c0 = a[0] - hs + 1, a[1] - ws + 1
    else:
        if (a[0] - hs + 1) % 2 or (a[1] - ws + 1) % 2: return None     # centre undefined at half cells
        r0, c0 = (a[0] - hs + 1) // 2, (a[1] - ws + 1) // 2
    return [(r0 + y, c0 + x) for y, x in shape]

SHAPE_KEYS = {'size': len, 'h': lambda s: max(y for y, _ in s) + 1, 'w': lambda s: max(x for _, x in s) + 1}

def is_rect(s):
    return len(s) == (max(y for y, _ in s) + 1) * (max(x for _, x in s) + 1)

def seq_rules(src, dst, allow_sort, keys=None):
    """Rules r with [src[p[i]]] == dst."""
    n = len(src); res = []
    if src == dst: res.append('id')
    if allow_sort and keys is None: keys = {k: [f(s) for s in src] for k, f in SHAPE_KEYS.items()}
    if not allow_sort: keys = None
    for name, p in perm_rules(n, keys):
        if [src[j] for j in p] == dst: res.append(name)
    return res

def attr_permute(g, st, imode, how, crule, srule, align):
    bg = bg_of(g)
    items = get_items(g, bg, imode, st)
    if not items or len(items) < 2: return None
    items = order_items(items, how)
    if items is None: return None
    n = len(items)
    if len({len({g[y][x] for y, x in o}) for o in items}) != 1 or any(len({g[y][x] for y, x in o}) != 1 for o in items): return None
    cols = [g[o[0][0]][o[0][1]] for o in items]
    shapes = [norm_shape(o) for o in items]
    keys = {k: [f(s) for s in shapes] for k, f in SHAPE_KEYS.items()}
    pc = apply_rule(crule, n)
    if srule.startswith('L'):                 # permute only the bar length along one axis
        if not all(is_rect(sh) for sh in shapes): return None
        ax, r = srule[1], srule[3:]
        dims = [(SHAPE_KEYS['h'](sh), SHAPE_KEYS['w'](sh)) for sh in shapes]
        lens = [d[0] if ax == 'h' else d[1] for d in dims]
        pl = apply_rule(r, n, {'len': lens})
        if pl is None: return None
        newdims = [(lens[pl[i]], dims[i][1]) if ax == 'h' else (dims[i][0], lens[pl[i]]) for i in range(n)]
        shapes = shapes + [tuple((y, x) for y in range(a) for x in range(b)) for a, b in newdims]
        ps = [n + i for i in range(n)]
    else:
        ps = apply_rule(srule, n, keys)
    if pc is None or ps is None: return None
    out = [r[:] for r in g]
    if srule == 'id':
        for o, j in zip(items, pc):
            for y, x in o: out[y][x] = cols[j]
        return out
    for o in items:
        for y, x in o: out[y][x] = bg
    for i, o in enumerate(items):
        cells = place(shapes[ps[i]], anchor(o, align), align)
        if cells is None: return None
        for y, x in cells:
            if not (0 <= y < H(g) and 0 <= x < W(g)): return None
            out[y][x] = cols[pc[i]]
    return out

IMODES = ('obj4', 'ns-obj4', 'obj8', 'ns-obj8', 'all4', 'ns-all4', 'path')
ORDERS = ('angle', 'nest', 'x', 'y')

def fam_attr_permute(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    if all(p['input'] == p['output'] for p in train): return
    st = static_colours(train)
    found = 0
    for imode in IMODES:
        if imode.startswith('ns') and not st: continue
        for how in (('path',) if imode == 'path' else ORDERS):
            cands = None
            for p in train:
                gi, go = p['input'], p['output']; bg = bg_of(gi)
                a = get_items(gi, bg, imode, st); b = get_items(go, bg, imode, st)
                if not a or not b or len(a) != len(b) or len(a) < 2: cands = set(); break
                a = order_items(a, how); b = order_items(b, how)
                if a is None or b is None: cands = set(); break
                if any(len({gi[y][x] for y, x in o}) != 1 for o in a) or any(len({go[y][x] for y, x in o}) != 1 for o in b):
                    cands = set(); break
                ca = [gi[o[0][0]][o[0][1]] for o in a]; cb = [go[o[0][0]][o[0][1]] for o in b]
                sa = [norm_shape(o) for o in a]; sb = [norm_shape(o) for o in b]
                cr = seq_rules(ca, cb, False)
                sr = seq_rules(sa, sb, True)
                if all(is_rect(x) for x in sa + sb) and 'id' not in sr:
                    for ax, k0, k1 in (('h', 0, 1), ('w', 1, 0)):
                        da = [(SHAPE_KEYS['h'](x), SHAPE_KEYS['w'](x)) for x in sa]
                        db = [(SHAPE_KEYS['h'](x), SHAPE_KEYS['w'](x)) for x in sb]
                        if [d[k1] for d in da] != [d[k1] for d in db]: continue
                        la = [d[k0] for d in da]; lb = [d[k0] for d in db]
                        sr += [f'L{ax}:{r}' for r in seq_rules(la, lb, True, {'len': la}) if r != 'id']
                al = [x for x in ('tl', 'tr', 'bl', 'br', 'c') if all(anchor(u, x) == anchor(v, x) for u, v in zip(a, b))]
                here = {(c, s, x) for c in cr for s in sr for x in (al if s != 'id' else ['-'])}
                if 'id' in sr and len(set(sa)) == 1:     # shapes all equal: shape rule irrelevant
                    here |= {(c, 'id', '-') for c in cr}
                cands = here if cands is None else cands & here
                if not cands: break
            if not cands: continue
            cands = [c for c in cands if (c[0], c[1]) != ('id', 'id')]
            if any(c[1] == 'id' for c in cands):          # colour-only explanation: simplest, keep it alone
                cands = [c for c in cands if c[1] == 'id']
            for crule, srule, align in sorted(cands)[:4]:
                fn = lambda g, imode=imode, how=how, crule=crule, srule=srule, align=align: \
                    attr_permute(g, st, imode, how, crule, srule, align)
                if all(fn(p['input']) == p['output'] for p in train):
                    found += 1
                    yield (f'perm-attr[{imode},{how}]:colour={crule},shape={srule}@{align}', 5, fn)
                    if found >= 6: return

# ------------------------------------------------------------------ F4: legend colour permutation
def legend_blocks(g, bg):
    """Solid (bg-free) rectangular multi-colour objects whose cells are all distinct colours."""
    out = []
    for o in objects(g, bg, True, False):
        r0, c0, r1, c1 = bbox(o)
        h, w = r1 - r0 + 1, c1 - c0 + 1
        if len(o) != h * w or len(o) < 2: continue
        vals = [g[y][x] for y, x in o]
        if len(set(vals)) != len(vals): continue
        out.append((r0, c0, r1, c1))
    return out

def legend_perm(g, orient, kind):
    bg = bg_of(g)
    bl = legend_blocks(g, bg)
    if len(bl) != 1: return None
    r0, c0, r1, c1 = bl[0]
    h, w = r1 - r0 + 1, c1 - c0 + 1
    m = {}
    if orient == 'h':
        if w != 2: return None
        pairs = [(g[y][c0], g[y][c0 + 1]) for y in range(r0, r1 + 1)]
    else:
        if h != 2: return None
        pairs = [(g[r0][x], g[r0 + 1][x]) for x in range(c0, c1 + 1)]
    for a, b in pairs:
        if kind in ('swap', 'fwd'): m[a] = b
        if kind in ('swap', 'bwd'): m[b] = a
    if kind == 'swap' and len(m) != 2 * len(pairs): return None
    out = [r[:] for r in g]
    for y in range(H(g)):
        for x in range(W(g)):
            if r0 <= y <= r1 and c0 <= x <= c1: continue
            out[y][x] = m.get(g[y][x], g[y][x])
    return out

def fam_legend_perm(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) != (H(o0), W(o0)): return
    if len(legend_blocks(i0, bg_of(i0))) != 1: return
    for orient, kind in product(('h', 'v'), ('swap', 'fwd', 'bwd')):
        fn = lambda g, orient=orient, kind=kind: legend_perm(g, orient, kind)
        if all(fn(p['input']) == p['output'] for p in train):
            yield (f'perm-legend[{orient}]:{kind}', 4, fn)

# ------------------------------------------------------------------ F5: sort & pack
PACK_ORDERS = {
    'x': lambda o: (bbox(o)[1], bbox(o)[0]), 'y': lambda o: (bbox(o)[0], bbox(o)[1]),
    'size-asc': lambda o: (len(o), bbox(o)[0], bbox(o)[1]), 'size-desc': lambda o: (-len(o), bbox(o)[0], bbox(o)[1]),
    'xr': lambda o: (-bbox(o)[3], bbox(o)[0]), 'yr': lambda o: (-bbox(o)[2], bbox(o)[1])}

def pack_items(g, conn, filt):
    bg = bg_of(g)
    obs = objects(g, bg, conn == 8, conn == 4)
    if filt == 'hollow':
        obs = [o for o in obs if len(o) < (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)]
    elif filt == 'solid':
        obs = [o for o in obs if len(o) == (bbox(o)[2] - bbox(o)[0] + 1) * (bbox(o)[3] - bbox(o)[1] + 1)]
    elif filt == 'common-shape':
        c = Counter(norm_shape(o) for o in obs)
        if not c: return None
        top = c.most_common(2)
        if len(top) > 1 and top[0][1] == top[1][1]: return None
        obs = [o for o in obs if norm_shape(o) == top[0][0]]
    return obs, bg

def pack(g, conn, filt, order, axis, canvas=None, slots=None):
    r = pack_items(g, conn, filt)
    if not r: return None
    obs, bg = r
    if not obs: return None
    obs = sorted(obs, key=PACK_ORDERS[order])
    crops = []
    for o in obs:
        r0, c0, r1, c1 = bbox(o); cs = set(o)
        crops.append([[g[y][x] if (y, x) in cs else bg for x in range(c0, c1 + 1)] for y in range(r0, r1 + 1)])
    if canvas:                                  # equal-size items written into a fixed lattice
        R, C = canvas
        h, w = H(crops[0]), W(crops[0])
        if any((H(c), W(c)) != (h, w) for c in crops) or R % h or C % w: return None
        so = slot_order(R // h, C // w, slots)
        if len(crops) > len(so): return None
        out = [[bg] * C for _ in range(R)]
        for (r, c), cr in zip(so, crops):
            for y in range(h):
                for x in range(w): out[r * h + y][c * w + x] = cr[y][x]
        return out
    if axis == 'auto':
        ys = [bbox(o)[0] for o in obs]; xs = [bbox(o)[1] for o in obs]
        sy, sx = max(ys) - min(ys), max(xs) - min(xs)
        if sy == sx: return None
        axis = 'h' if sx > sy else 'v'
        if order in ('x', 'y'):
            crops = [crops[i] for i in sorted(range(len(obs)), key=lambda i: (bbox(obs[i])[1], bbox(obs[i])[0]) if axis == 'h' else (bbox(obs[i])[0], bbox(obs[i])[1]))]
    if axis == 'h':
        if len({H(c) for c in crops}) != 1: return None
        return [sum((c[y] for c in crops), []) for y in range(H(crops[0]))]
    if len({W(c) for c in crops}) != 1: return None
    return [row for c in crops for row in c]

def fam_sort_pack(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if H(o0) * W(o0) >= H(i0) * W(i0): return
    shapes = {(H(p['output']), W(p['output'])) for p in train}
    canvas = shapes.pop() if len(shapes) == 1 else None
    n = 0
    for conn, filt in product((4, 8), ('all', 'hollow', 'solid', 'common-shape')):
        pre = pack_items(i0, conn, filt)
        if not pre or len(pre[0]) < 2: continue
        tried = []
        for order in PACK_ORDERS:
            for axis in ('auto', 'h', 'v'):
                if axis == 'auto' and order not in ('x', 'y'): continue
                tried.append((order, axis, None, None))
            if canvas:
                for sl in SLOTS: tried.append((order, None, canvas, sl))
        for order, axis, cv, sl in tried:
            fn = lambda g, conn=conn, filt=filt, order=order, axis=axis, cv=cv, sl=sl: pack(g, conn, filt, order, axis, cv, sl)
            if fn(i0) != o0: continue
            if all(fn(p['input']) == p['output'] for p in train[1:]):
                n += 1
                yield (f'sort-pack[{conn},{filt}]:{order}->{axis or sl}', 5, fn)
                if n >= 4: return

# ------------------------------------------------------------------ F6: panel chain (Hamiltonian path of edge-compatible panels)
def lattice_panels(g):
    lt = lattice(g)
    if not lt: return None
    rr, cc = lt
    sep = None
    if len(rr) > 1: sep = g[rr[0][1]][0]
    elif len(cc) > 1: sep = g[0][cc[0][1]]
    return [crop(g, (a, b, e - 1, f - 1)) for (a, e) in rr for (b, f) in cc], sep

def edge_set(p, side, bg):
    if side == 'top': ln = p[0]
    elif side == 'bottom': ln = p[-1]
    elif side == 'left': ln = [r[0] for r in p]
    else: ln = [r[-1] for r in p]
    return frozenset(i for i, v in enumerate(ln) if v != bg)

def chain_order(panels, axis, bg):
    far, near = ('bottom', 'top') if axis == 'v' else ('right', 'left')
    n = len(panels)
    F = [edge_set(p, far, bg) for p in panels]; N = [edge_set(p, near, bg) for p in panels]
    sols = []
    def rec(path, used):
        if len(sols) > 1: return
        if len(path) == n: sols.append(path[:]); return
        last = path[-1]
        for j in range(n):
            if j not in used and F[last] and F[last] == N[j]:
                path.append(j); used.add(j); rec(path, used); path.pop(); used.discard(j)
    for s0 in range(n):
        if N[s0]: continue                      # the chain starts at a panel with an empty near edge
        rec([s0], {s0})
    return sols[0] if len(sols) == 1 else None

def panel_chain(g, axis_rule):
    lp = lattice_panels(g)
    if not lp: return None
    panels, sep = lp
    bg = bg_of(g)
    if len({(H(p), W(p)) for p in panels}) != 1 or len(panels) < 2: return None
    found = []
    for axis in ('v', 'h'):
        o = chain_order(panels, axis, bg)
        if o: found.append((axis, o))
    if len(found) != 1: return None
    axis, o = found[0]
    ps = [panels[j] for j in o]
    if axis == 'v':
        out = []
        for k, p in enumerate(ps):
            if k: out.append([sep] * W(p))
            out += [r[:] for r in p]
        return out
    out = [[] for _ in range(H(ps[0]))]
    for k, p in enumerate(ps):
        for y in range(H(p)):
            out[y] += ([sep] if k else []) + p[y]
    return out

def fam_panel_chain(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if (H(i0), W(i0)) == (H(o0), W(o0)) and not lattice(i0): return
    if not lattice(i0): return
    fn = lambda g: panel_chain(g, 'auto')
    if all(fn(p['input']) == p['output'] for p in train):
        yield ('panel-chain:edge-match', 5, fn)

# ------------------------------------------------------------------ F7: class transposition (two shape classes swap)
def class_swap(g, conn, carry, align):
    bg = bg_of(g)
    obs = objects(g, bg, conn == 8, True)
    cl = {}
    for o in obs: cl.setdefault(g[o[0][0]][o[0][1]], []).append(o)
    if len(cl) != 2: return None
    (ca, A), (cb, B) = sorted(cl.items())
    sa = {norm_shape(o) for o in A}; sb = {norm_shape(o) for o in B}
    if len(sa) != 1 or len(sb) != 1 or sa == sb: return None
    sa, sb = sa.pop(), sb.pop()
    out = [r[:] for r in g]
    for o in obs:
        for y, x in o: out[y][x] = bg
    for group, own, other_shape, other_col in ((A, ca, sb, cb), (B, cb, sa, ca)):
        for o in group:
            cells = place(other_shape, anchor(o, align), align)
            if cells is None: return None
            for y, x in cells:
                if not (0 <= y < H(g) and 0 <= x < W(g)): return None
                out[y][x] = other_col if carry else own
    return out

def fam_class_swap(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    n = 0
    for conn, carry, align in product((4, 8), (True, False), ('tl', 'tr', 'bl', 'br', 'c')):
        fn = lambda g, conn=conn, carry=carry, align=align: class_swap(g, conn, carry, align)
        if fn(i0) != o0: continue
        if all(fn(p['input']) == p['output'] for p in train[1:]):
            n += 1
            yield (f'class-swap[{conn}]:{"carry" if carry else "keep"}-colour@{align}', 4, fn)
            if n >= 3: return

# ------------------------------------------------------------------ F8: sorted stack against a wall
def sorted_stack(g, st, edge, order, cross):
    """Remove the movable items, sort them, and stack them one per band starting at the first free line
    next to `edge` (the wall), each aligned to the start/end of the cross axis."""
    gg = g if edge in ('bottom', 'top') else T(g)
    if edge in ('top', 'left'): gg = gg[::-1]
    bg = bg_of(g)
    obs = [o for o in objects(gg, bg, False, True) if gg[o[0][0]][o[0][1]] not in st]
    if len(obs) < 2: return None
    out = [r[:] for r in gg]
    for o in obs:
        for y, x in o: out[y][x] = bg
    y = H(out) - 1
    while y >= 0 and any(v != bg for v in out[y]): y -= 1
    if y < 0: return None
    obs = sorted(obs, key=PACK_ORDERS[order])
    for o in obs:
        sh = norm_shape(o); col = gg[o[0][0]][o[0][1]]
        hs = max(a for a, _ in sh) + 1; ws = max(b for _, b in sh) + 1
        r0 = y - hs + 1; c0 = 0 if cross == 'start' else W(out) - ws
        if r0 < 0: return None
        for a, b in sh:
            if out[r0 + a][c0 + b] != bg: return None
            out[r0 + a][c0 + b] = col
        y = r0 - 1
    if edge in ('top', 'left'): out = out[::-1]
    return out if edge in ('bottom', 'top') else T(out)

def fam_sorted_stack(train):
    i0, o0 = train[0]['input'], train[0]['output']
    if any((H(p['input']), W(p['input'])) != (H(p['output']), W(p['output'])) for p in train): return
    st = static_colours(train)
    n = 0
    for edge, order, cross in product(('bottom', 'top', 'left', 'right'), ('size-desc', 'size-asc'), ('start', 'end')):
        fn = lambda g, edge=edge, order=order, cross=cross: sorted_stack(g, st, edge, order, cross)
        if fn(i0) != o0: continue
        if all(fn(p['input']) == p['output'] for p in train[1:]):
            n += 1
            yield (f'sorted-stack[{edge}]:{order},{cross}', 5, fn)
            if n >= 2: return

FAMILIES = (fam_axis_permute, fam_lattice_permute, fam_attr_permute, fam_legend_perm, fam_sort_pack, fam_panel_chain, fam_class_swap, fam_sorted_stack)


In [ ]:
%%writefile /kaggle/working/probe/prior_fluids.py
"""FLUIDS prior: common-sense hydraulics on the grid, used as a general search space.

Laws (the prior)
----------------
The medium is the background colour; every other colour is solid unless it is the fluid itself.
Gravity g is one of the four lattice directions; every engine runs in the frame where g points 'down'.
  L1 gravity      a fluid parcel moves along g whenever the next cell is free.
  L2 support      a parcel blocked along g (by a wall, standing fluid or a closed floor) spreads perpendicular
                  to g, both ways, along the supporting surface; where the support ends it falls again
                  (waterfalls that wrap obstacles; films that run along walls and pass through their gaps).
  L3 hydrostatics a horizontal layer of fluid that is supported everywhere and bounded by walls at both ends is
                  at rest (level free surface); a basin therefore fills up to its SPILL level and then
                  overflows by L2.
  L4 conservation a finite body of fluid keeps its volume: parcels (lowest first, repeated to equilibrium)
                  descend to the lowest cell they can reach by falling or sliding along a support
                  (communicating vessels level out; fluid above a rim spills into the next basin).
  L5 thin stream  the zero-volume limit of L1-L2: a trickle does not split; blocked along g it slides to the
                  NEAREST point from which it can fall again (least detour; tie -> induced side), or always to
                  one side when the surface is tilted; it stops when both ways are blocked.
  L6 pressure     an incompressible fluid injected into a container fills its interior; at every gap in the
                  container wall it escapes as a jet running straight outward (normal to that wall) until an
                  obstacle or the grid edge, optionally with a spray: diagonal rays fanning out from the
                  gap's rims.  A container is a wall set whose openings are all GAPS (flanked by wall cells).

Search space (every parameter induced from the training pairs; unseen situations refuse -> None)
  fluid colour F   a non-background colour present in every training input (sources / liquid / marker), the
                   role 'minor' (least frequent non-bg colour of the grid), or 'multi': all colours that grow in
                   every pair, each pouring its own colour (colliding colours refuse)
  paint            F itself, or the constant colour of all changed cells (induced, may be new)
  gravity          down | up | left | right | edge (each source flows away from the grid edge it touches);
                   frames are rotations, so 'lo'/'hi' sides keep one rotational sense for every gravity
  source term      infinite (F cells are springs) | finite (F cells ARE the liquid, volume conserved) |
                   rain (every cell of the top edge is a spring)
  boundaries       floor open (fluid leaves the grid) / closed; sides open / closed (walls)
  render           wet (every wetted cell) | standing (fluid at rest: basins) | two-tone (flowing and standing
                   fluid in two induced colours); sources kept / drained
  trickle policy   nearest (tie lo/hi side) | one-sided lo/hi (tilted surface) | tilt table: obstacle colour ->
                   side (each obstacle colour is a surface tilted one way)
  jets             container = each single-colour 8-connected object (any colour / induced colour) or all cells
                   of one colour (induced colour / the most frequent non-bg colour); fluid = the unique
                   marker colour inside the container, or the induced constant colour; spray on/off

Families kept (full eval: training 1000 + dev-eval half A; each solves >= 3 tasks, WRONG = 0)
  fluid:gravity  L1-L4   36a08778 (half A: streams wrap bars), 28a6681f (half A: conserved liquid settles into
                         basins, spill-over into the next basin), f9a67cb5 (films run along walls, through gaps)
  fluid:trickle  L5      c87289bb (nearest end, tie), d9f24cd1 (rising streams), 712bf12e (one-sided / tilted),
                         96a8c0cd (tilt table: obstacle colour -> side)
  fluid:jet      L6      d4f3cd78, 292dd178, 551d5bf1 (fill + jet through the gap), aba27056 (with spray),
                         78e78cff (bracket container, marker colour, jets through every gap)
Dropped after measurement (0 tasks fit on the training pairs): contagion along the object adjacency graph
(diffusion/infection between touching objects), competing diffusion from several sources (geodesic Voronoi
flood), look-ahead trickles (shortest remaining path), a 'standing + film' render (fluid at rest on every
surface).  Plain flood fill from seeds is left to prior_topology (seed-region) / distance layers.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter, deque
from itertools import product
from gdsl import H, W, r90, r180, r270, bg_of, objects, bbox

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
# frames: (to, back) ROTATIONS so that the gravity direction becomes 'down' (orientation preserved, so the
# frame's +x side is the same rotational sense - counter-clockwise of the flow - for every gravity)
FRAMES = {"down": (lambda g: [r[:] for r in g], lambda g: [r[:] for r in g]), "up": (r180, r180),
          "right": (r90, r270), "left": (r270, r90)}
GRAVITIES = ('down', 'up', 'left', 'right', 'edge')
MAXC = 200


# ------------------------------------------------------------------ induction helpers
def same_shape(train):
    return all(H(p['input']) == H(p['output']) and W(p['input']) == W(p['output']) for p in train)


def const_colour(train):
    """The single colour taken by every changed cell of every training pair (None if not unique)."""
    cs = {p['output'][y][x] for p in train for y in range(H(p['input'])) for x in range(W(p['input']))
          if p['input'][y][x] != p['output'][y][x]}
    return next(iter(cs)) if len(cs) == 1 else None


def common_colours(train):
    return sorted(set.intersection(*[{v for r in p['input'] for v in r} for p in train]) - {bg_of(train[0]['input'])})


def growing_colours(train):
    """Colours whose cell count grows in every training pair."""
    res = None
    for p in train:
        a = Counter(v for r in p['input'] for v in r); b = Counter(v for r in p['output'] for v in r)
        s = {c for c in b if b[c] > a[c] and a[c] > 0}
        res = s if res is None else res & s
    return sorted(res or ())


def edge_dir(cells, h, w):
    ds = set()
    for y, x in cells:
        if y == 0: ds.add('down')
        if y == h - 1: ds.add('up')
        if x == 0: ds.add('right')
        if x == w - 1: ds.add('left')
    return ds.pop() if len(ds) == 1 else None


def resolve(g, bg, Fs):
    """Source colours: literal tuple, or a role resolved on the grid ('minor' = least frequent non-bg colour)."""
    if Fs != 'minor': return Fs
    cs = Counter(v for r in g for v in r if v != bg)
    if len(cs) < 2: return None
    (a, na), (b, nb) = cs.most_common()[-1], cs.most_common()[-2]
    return (a,) if na < nb else None


def source_groups(g, bg, Fs, gravity):
    """[(gravity dir, colour, cells)]: one group per source colour, or per source object for 'edge' gravity."""
    h, w = H(g), W(g); groups = []
    Fs = resolve(g, bg, Fs)
    if not Fs: return None
    for F in Fs:
        cells = [(y, x) for y in range(h) for x in range(w) if g[y][x] == F]
        if not cells: continue
        if gravity != 'edge':
            groups.append((gravity, F, cells)); continue
        for ob in objects(g, bg, True, True):
            if g[ob[0][0]][ob[0][1]] != F: continue
            d = edge_dir(ob, h, w)
            if d is None: return None
            groups.append((d, F, ob))
    return groups


def in_frame(g, gdir, cells):
    """Grid and a cell mask transformed into the gravity frame."""
    to, _ = FRAMES[gdir]
    mark = [[0] * W(g) for _ in range(H(g))]
    for y, x in cells: mark[y][x] = 1
    gg = to(g); mm = to(mark)
    return gg, [(y, x) for y in range(H(gg)) for x in range(W(gg)) if mm[y][x]]


def from_frame(cells, gdir, hf, wf):
    _, back = FRAMES[gdir]
    lay = [[0] * wf for _ in range(hf)]
    for y, x in cells: lay[y][x] = 1
    lay = back(lay)
    return [(y, x) for y in range(H(lay)) for x in range(W(lay)) if lay[y][x]]


def paint_layers(g, layers, bg, erase=()):
    """layers: [(colour, cells)]; cells painted only over background; conflicting colours refuse."""
    out = [r[:] for r in g]
    for y, x in erase: out[y][x] = bg
    got = {}
    for c, cells in layers:
        for y, x in cells:
            if got.setdefault((y, x), c) != c: return None
            out[y][x] = c
    return out


# ------------------------------------------------------------------ engines (gravity frame: down)
def reservoir(free, h, w, sources, floor, sides):
    """L1-L3 with infinite springs.  Returns (wet, standing) cell sets or None."""
    src = set(sources); standing = set(); wet = set(src)
    def openc(y, x): return 0 <= y < h and 0 <= x < w and (free[y][x] or (y, x) in src) and (y, x) not in standing
    def solid(y, x):
        if y >= h: return floor
        if x < 0 or x >= w: return sides
        return not openc(y, x)
    stack = list(src); dirty = {y for y, _ in src}
    for _ in range(3000):
        while stack:
            y, x = stack.pop()
            if (y, x) in standing: continue
            if y + 1 < h and openc(y + 1, x):
                if (y + 1, x) not in wet: wet.add((y + 1, x)); stack.append((y + 1, x)); dirty.add(y + 1)
                continue
            if y + 1 >= h and not floor: continue
            for nx in (x - 1, x + 1):
                if openc(y, nx) and (y, nx) not in wet:
                    wet.add((y, nx)); stack.append((y, nx)); dirty.add(y)
        newly = []
        for y in sorted(dirty):
            xs = sorted(x for (yy, x) in wet if yy == y and (yy, x) not in standing)
            i = 0
            while i < len(xs):
                j = i
                while j + 1 < len(xs) and xs[j + 1] == xs[j] + 1: j += 1
                a, b = xs[i], xs[j]
                if solid(y, a - 1) and solid(y, b + 1) and all(solid(y + 1, x) for x in range(a, b + 1)):
                    newly.extend((y, x) for x in range(a, b + 1))
                i = j + 1
        dirty = set()
        if not newly: return wet, standing
        standing.update(newly)
        for y, x in newly:
            if (y - 1, x) in wet and (y - 1, x) not in standing: stack.append((y - 1, x)); dirty.add(y - 1)
    return None


def settle(free, h, w, liquid, floor, sides):
    """L4: finite volume.  Parcels processed lowest first, repeated until nothing moves; each goes to the
    lowest cell reachable by falling (unsupported) or sliding sideways (supported); ties -> nearest, stay.
    A parcel that can reach an open floor / open side leaves the grid."""
    occ = set(liquid)
    def openc(y, x): return 0 <= y < h and 0 <= x < w and free[y][x] and (y, x) not in occ
    for _ in range(60):
        moved = False
        for c in sorted(occ, key=lambda c: (-c[0], c[1])):
            occ.discard(c)
            dist = {c: 0}; q = deque([c]); best = None; drain = False
            while q and not drain:
                y, x = q.popleft()
                if y + 1 >= h:
                    if not floor: drain = True; break
                    sup = True
                else:
                    sup = not openc(y + 1, x)
                if sup:
                    key = (y, -dist[(y, x)], (y, x) == c)
                    if best is None or key > best[0]: best = (key, (y, x))
                    nb = ((y, x - 1), (y, x + 1))
                else:
                    nb = ((y + 1, x),)
                for yy, xx in nb:
                    if not 0 <= xx < w:
                        if not sides: drain = True
                        continue
                    if (yy, xx) not in dist and openc(yy, xx):
                        dist[(yy, xx)] = dist[(y, x)] + 1; q.append((yy, xx))
            if drain or best is None:
                moved = True; continue
            d = c if best[0][0] == c[0] else best[1]
            occ.add(d)
            if d != c: moved = True
        if not moved: return occ
    return None


SIDES = {'lo': (-1, 1), 'hi': (1, -1), 'lo!': (-1,), 'hi!': (1,)}


def trickle_paths(free, h, w, starts, policy, gg=None):
    """L5: thin streams from the start cells; returns the set of visited cells.  policy: a side rule, or a
    dict obstacle colour -> side rule (surfaces of each colour tilted one way; gg = the grid in the frame)."""
    painted = set()
    def ok(y, x): return 0 <= y < h and 0 <= x < w and (free[y][x] or (y, x) in painted)
    for sy, sx in starts:
        y, x = sy, sx
        for _ in range(4 * (h + w)):
            painted.add((y, x))
            if y + 1 >= h: break
            if ok(y + 1, x): y += 1; continue
            if isinstance(policy, dict):
                rule = policy.get(gg[y + 1][x])
                if rule is None: return None
                order = SIDES[rule]
            else:
                order = SIDES[policy]
            best = None
            for side in order:
                k = 1
                while ok(y, x + side * k):
                    if ok(y + 1, x + side * k):
                        if best is None or k < best[0]: best = (k, side)
                        break
                    k += 1
            if best is None: break
            k, side = best
            for j in range(1, k + 1): painted.add((y, x + side * j))
            x += side * k
    return painted


# ------------------------------------------------------------------ family 1: gravity-driven liquid (L1-L4)
def simulate(g, bg, Fs, gravity, source, floor, sides):
    """Physics only (no colours): [(F, erase cells, wet cells, standing cells)] in grid coordinates."""
    if source == 'rain':
        if gravity == 'edge': return None
        gg, _ = in_frame(g, gravity, [])
        h, w = H(gg), W(gg)
        free = [[gg[y][x] == bg for x in range(w)] for y in range(h)]
        res = reservoir(free, h, w, [(0, x) for x in range(w) if free[0][x]], floor, sides)
        if res is None: return None
        return [(None, [], from_frame([c for c in res[0] if free[c[0]][c[1]]], gravity, h, w),
                 from_frame([c for c in res[1] if free[c[0]][c[1]]], gravity, h, w))]
    groups = source_groups(g, bg, Fs, gravity)
    if not groups or sum(len(cs) for _, _, cs in groups) > MAXC: return None
    sim = []
    for gdir, F, cells in groups:
        gg, s = in_frame(g, gdir, cells)
        h, w = H(gg), W(gg)
        if source == 'finite':
            free = [[gg[y][x] == bg or gg[y][x] == F for x in range(w)] for y in range(h)]
            occ = settle(free, h, w, s, floor, sides)
            if occ is None: return None
            res = from_frame(occ, gdir, h, w)
            sim.append((F, cells, res, res))
        else:
            free = [[gg[y][x] == bg for x in range(w)] for y in range(h)]
            res = reservoir(free, h, w, s, floor, sides)
            if res is None: return None
            wet, standing = res
            sim.append((F, cells, from_frame([c for c in wet | standing if free[c[0]][c[1]]], gdir, h, w),
                        from_frame([c for c in standing if free[c[0]][c[1]]], gdir, h, w)))
    return sim


def render_liquid(g, bg, sim, source, paint, render, keep):
    """render: 'wet' | 'standing' | ('two', flowing colour, standing colour)."""
    if sim is None: return None
    layers = []; erase = []
    for F, cells, wet, standing in sim:
        if isinstance(render, tuple):
            st = set(standing)
            layers.append((render[1], [c for c in wet if c not in st])); layers.append((render[2], standing))
        else:
            pc = F if paint is None else paint
            if pc is None: return None
            layers.append((pc, wet if render == 'wet' else standing))
        if source == 'finite' or not keep: erase.extend(cells)
    return paint_layers(g, layers, bg, erase)


def gravity_liquid(g, bg, Fs, gravity, source, floor, sides, paint, render, keep):
    """Fs: source/liquid colours (empty for rain).  paint: None -> each source's own colour."""
    return render_liquid(g, bg, simulate(g, bg, Fs, gravity, source, floor, sides), source, paint, render, keep)


def fam_fluid_gravity(train):
    if not same_shape(train): return
    i0, o0 = train[0]['input'], train[0]['output']
    if i0 == o0: return
    bg0 = bg_of(i0)
    changed = {o0[y][x] for y in range(H(i0)) for x in range(W(i0)) if i0[y][x] != o0[y][x]}
    cc = const_colour(train)
    srcsets = [(str(F), (F,)) for F in common_colours(train)]
    grow = growing_colours(train)
    if len(grow) >= 2: srcsets.append(('multi', tuple(grow)))
    mi = resolve(i0, bg0, 'minor')
    if mi and all(resolve(p['input'], bg_of(p['input']), 'minor') for p in train): srcsets.append(('minor', 'minor'))
    n = 0
    two = sorted(changed - {bg0})                  # flowing and standing fluid drawn in two induced colours
    for (fname, Fs), source in product(srcsets, ('infinite', 'finite')):
        F0 = resolve(i0, bg0, Fs)
        paints = [('own', None)] + ([('c%d' % cc, cc)] if cc is not None and (cc,) != F0 else [])
        # the only colours a program can write are its paint colours (and bg where sources drain)
        views = []
        for pn, pc in paints:
            if not changed <= (set(F0) if pc is None else {pc}) | {bg0}: continue
            if source == 'finite': views.append((pn, pc, 'wet', False))
            else: views += [(pn, pc, r, k) for r, k in product(('wet', 'standing'), (True, False))]
        if source == 'infinite' and len(two) == 2:
            views += [('own', None, ('two', a, b), k) for a, b in (two, two[::-1]) for k in (True, False)]
        if not views: continue
        if source == 'finite':
            geo = [(gr, fl, sd) for gr in GRAVITIES[:4] for fl, sd in product((True, False), (True, False))]
        else:
            geo = [(gr, fl, sd) for gr in GRAVITIES for fl, sd in product((False, True), (False, True))]
        for gravity, floor, sides in geo:
            sim = simulate(i0, bg0, Fs, gravity, source, floor, sides)
            if sim is None: continue
            for pn, pc, render, keep in views:
                if render_liquid(i0, bg0, sim, source, pc, render, keep) != o0: continue
                def fn(g, Fs=Fs, gravity=gravity, source=source, floor=floor, sides=sides, pc=pc, render=render, keep=keep):
                    return gravity_liquid(g, bg_of(g), Fs, gravity, source, floor, sides, pc, render, keep)
                n += 1
                rn = render if isinstance(render, str) else f'flow=c{render[1]}/still=c{render[2]}'
                yield (f"fluid:gravity[F{fname},{pn},{gravity},{source},{'floor' if floor else 'nofloor'},"
                       f"{'walls' if sides else 'open'},{rn}{'' if keep else ',drain'}]", 4, fn)
                if n >= 12: return
    if cc is not None and changed <= {cc}:
        for gravity, floor, sides in product(GRAVITIES[:4], (False, True), (False, True)):
            sim = simulate(i0, bg0, (), gravity, 'rain', floor, sides)
            for render in ('standing', 'wet'):
                if render_liquid(i0, bg0, sim, 'rain', cc, render, True) != o0: continue
                def fn(g, gravity=gravity, floor=floor, sides=sides, render=render):
                    return gravity_liquid(g, bg_of(g), (), gravity, 'rain', floor, sides, cc, render, True)
                n += 1
                yield (f"fluid:gravity[rain,c{cc},{gravity},{'floor' if floor else 'nofloor'},{'walls' if sides else 'open'},{render}]", 4, fn)
                if n >= 16: return


# ------------------------------------------------------------------ family 2: thin streams (L5)
def trickle(g, bg, Fs, gravity, policy, paint):
    groups = source_groups(g, bg, Fs, gravity)
    if not groups: return None
    if sum(len(cs) for _, _, cs in groups) > MAXC: return None
    layers = []
    for gdir, F, cells in groups:
        gg, s = in_frame(g, gdir, cells)
        h, w = H(gg), W(gg); ss = set(s)
        free = [[gg[y][x] == bg or gg[y][x] == F for x in range(w)] for y in range(h)]
        starts = [(y, x) for y, x in s if (y + 1, x) not in ss]         # leading cells of each source
        tp = trickle_paths(free, h, w, starts, policy, gg)
        if tp is None: return None
        cells = [(y, x) for y, x in tp if gg[y][x] == bg]
        layers.append((F if paint is None else paint, from_frame(cells, gdir, h, w)))
    return paint_layers(g, layers, bg)


def fam_fluid_trickle(train):
    if not same_shape(train): return
    i0, o0 = train[0]['input'], train[0]['output']
    if i0 == o0: return
    cc = const_colour(train)
    srcsets = [(str(F), (F,)) for F in common_colours(train)]
    grow = growing_colours(train)
    if len(grow) >= 2: srcsets.append(('multi', tuple(grow)))
    n = 0
    bg0 = bg_of(i0)
    changed = {o0[y][x] for y in range(H(i0)) for x in range(W(i0)) if i0[y][x] != o0[y][x]}
    if bg0 in changed: return
    mi = resolve(i0, bg0, 'minor')
    if mi and all(resolve(p['input'], bg_of(p['input']), 'minor') for p in train): srcsets.append(('minor', 'minor'))
    allc = sorted({v for p in train for r in p['input'] for v in r} - {bg0})
    for (fname, Fs), gravity in product(srcsets, GRAVITIES):
        F0 = resolve(i0, bg0, Fs)
        # side rules: uniform (nearest with a tie side, or always one side), or one side per obstacle colour
        obst = [c for c in allc if c not in F0]
        policies = ['lo', 'hi', 'lo!', 'hi!']
        if 2 <= len(obst) <= 3:
            policies += [dict(zip(obst, t)) for t in product(('lo!', 'hi!'), repeat=len(obst)) if len(set(t)) > 1]
        paints = [('own', None)] + ([('c%d' % cc, cc)] if cc is not None and (cc,) != F0 else [])
        for policy, (pn, pc) in product(policies, paints):
            if not changed <= (set(F0) if pc is None else {pc}): continue
            def fn(g, Fs=Fs, gravity=gravity, policy=policy, pc=pc):
                return trickle(g, bg_of(g), Fs, gravity, policy, pc)
            if fn(i0) != o0: continue
            n += 1
            pname = policy if isinstance(policy, str) else 'tilt:' + ','.join(f'c{k}{v}' for k, v in sorted(policy.items()))
            yield (f"fluid:trickle[F{fname},{pn},{gravity},{pname}]", 4 if isinstance(policy, str) else 5, fn)
            if n >= 10: return


# ------------------------------------------------------------------ family 3: pressure jets through gaps (L6)
def _runs(vals):
    vals = sorted(vals); runs = []
    for v in vals:
        if runs and v == runs[-1][1] + 1: runs[-1][1] = v
        else: runs.append([v, v])
    return runs


def jet(g, bg, wall, mode, paint, spray):
    """wall: a colour, None (any colour, 'obj' mode) or 'major' ('colour' mode).  paint: 'marker' or a colour."""
    h, w = H(g), W(g); out = [r[:] for r in g]
    conts = []
    if mode == 'obj':
        for ob in objects(g, bg, True, True):
            if wall is None or g[ob[0][0]][ob[0][1]] == wall: conts.append(ob)
    else:
        wc = wall
        if wc == 'major':
            cs = Counter(v for r in g for v in r if v != bg)
            if not cs: return None
            wc = cs.most_common(1)[0][0]
        cells = [(y, x) for y in range(h) for x in range(w) if g[y][x] == wc]
        if cells: conts.append(cells)
    n = 0
    for ob in conts:
        r0, c0, r1, c1 = bbox(ob)
        if r1 - r0 < 2 or c1 - c0 < 2: continue
        obs = set(ob)
        inner = [(y, x) for y in range(r0 + 1, r1) for x in range(c0 + 1, c1) if (y, x) not in obs]
        if not inner: continue
        marks = [(y, x) for y, x in inner if g[y][x] != bg]
        if paint == 'marker':
            mc = {g[y][x] for y, x in marks}
            if len(mc) != 1: continue
            pc = mc.pop(); seeds = marks
        else:
            if marks: continue
            pc = paint; seeds = inner
        seen = set(seeds); st = list(seeds)
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                yy, xx = y + dy, x + dx
                if r0 <= yy <= r1 and c0 <= xx <= c1 and (yy, xx) not in seen and (yy, xx) not in obs \
                        and g[yy][xx] in (bg, pc):
                    seen.add((yy, xx)); st.append((yy, xx))
        # openings per side: (outward direction) -> positions along the side
        sides = {(-1, 0): [x for y, x in seen if y == r0], (1, 0): [x for y, x in seen if y == r1],
                 (0, -1): [y for y, x in seen if x == c0], (0, 1): [y for y, x in seen if x == c1]}
        runs = {}
        ok = True
        for d, vals in sides.items():
            rs = _runs(vals); runs[d] = rs
            for a, b in rs:           # every opening must be a GAP: flanked by container cells along the side
                if d[0]:
                    fy = r0 if d[0] < 0 else r1
                    if (fy, a - 1) not in obs or (fy, b + 1) not in obs: ok = False
                else:
                    fx = c0 if d[1] < 0 else c1
                    if (a - 1, fx) not in obs or (b + 1, fx) not in obs: ok = False
        if not ok: continue
        n += 1
        for y, x in seen:
            if g[y][x] == bg: out[y][x] = pc
        for (dy, dx), rs in runs.items():
            for a, b in rs:
                fixed = (r0 if dy < 0 else r1) if dy else (c0 if dx < 0 else c1)
                for v in range(a, b + 1):
                    y, x = (fixed, v) if dy else (v, fixed)
                    y += dy; x += dx
                    while 0 <= y < h and 0 <= x < w and g[y][x] == bg:
                        out[y][x] = pc; y += dy; x += dx
                if spray:
                    for end, s in ((a, -1), (b, 1)):
                        if dy: y, x, sy, sx = fixed + dy, end + s, dy, s
                        else: y, x, sy, sx = end + s, fixed + dx, s, dx
                        while 0 <= y < h and 0 <= x < w and g[y][x] == bg:
                            out[y][x] = pc; y += sy; x += sx
    return out if n else None


def fam_fluid_jet(train):
    if not same_shape(train): return
    i0, o0 = train[0]['input'], train[0]['output']
    if i0 == o0: return
    cc = const_colour(train)
    walls = [('obj', c) for c in common_colours(train)] + [('obj', None)] + \
            [('colour', c) for c in common_colours(train)] + [('colour', 'major')]
    paints = ['marker'] + ([cc] if cc is not None else [])
    if bg_of(i0) in {o0[y][x] for y in range(H(i0)) for x in range(W(i0)) if i0[y][x] != o0[y][x]}: return
    for (mode, wall), paint, spray in product(walls, paints, (False, True)):
        def fn(g, mode=mode, wall=wall, paint=paint, spray=spray):
            return jet(g, bg_of(g), wall, mode, paint, spray)
        if fn(i0) != o0: continue
        wn = 'any' if wall is None else (wall if wall == 'major' else 'c%d' % wall)
        yield (f"fluid:jet[{mode}:{wn},{'marker' if paint == 'marker' else 'c%d' % paint}{',spray' if spray else ''}]", 4, fn)


FAMILIES = (fam_fluid_gravity, fam_fluid_trickle, fam_fluid_jet)


In [ ]:
%%writefile /kaggle/working/probe/prior_geometry.py
"""GEOMETRY prior: symmetry groups, lattices and similarity transforms with arbitrary centres, fitted exhaustively.

Core concept
------------
An ARC picture P (a whole grid, one object, the contents of a frame, the panels around a separator crossing)
is assumed invariant under a subgroup G of the plane isometries that preserve the square lattice: D4 elements
(fh, fv, r180, T, aT, r90, r270) about an UNKNOWN centre c that may sit on a cell or between cells (doubled
coordinates c2 = 2c), translations (lattices) and glide reflections.  The input shows P with unknown cells
(background holes or an occluder colour); the output restores each unknown cell from its G-orbit.

Exhaustive fitting: for every element e and EVERY doubled centre, the counts
    agree = #{known fg p : e_c(p) known, same colour}, conflict = #{... different colour},
    fill  = #{known fg p : e_c(p) unknown},  oob = #{fg p : e_c(p) outside the grid}
are computed at once by 1-D/2-D correlation (scipy.signal.correlate); a group's table is the sum over its
elements.  The chosen centre has zero conflicts and maximal (agree - oob) (ties: fewer fills, nearest the
region centre).  'auto' takes every element (each with its own centre) plus the best translations and glides
that are exact symmetries of the known cells and closes the orbits by union-find (a wallpaper-like group);
'local' additionally uses PARTIAL symmetries: an element fills a cell when it is exact inside the smallest
window around that cell holding >= 12 known pairs.

Families (all parameters induced per task; every program is verified on all training pairs by the harness)
  geom-sym-complete        unknown = bg; group H|V|HV|R2|D|A|DA|R4|D4 (+ lattice|auto on the grid);
                           scope grid | object | frame (group+centre fitted on the frame = its stabiliser,
                           imposed on its contents) | anchor (unique G-invariant object is the centre) |
                           crossing (each separator-line crossing is a centre for its 4 panels);
                           paint orbit colour | the new output colour
  geom-sym-repair(-patch)  unknown = occluder colour (absent from outputs) or the unique solid rectangle;
                           group fixed | auto | local; output repaired grid or only the occluded window
  geom-symmetrize-colour   the largest blob's colour becomes G-invariant (overwriting), centre grid | fitted
  geom-fold-dominant       each object made mirror-symmetric about its in-object line / best axis, the fuller
                           half wins
  geom-twisted-quadrants   D2 symmetry up to a colour substitution per quadrant, induced from hint cells
  geom-similarity-complete template = most-colourful object; partial copies elsewhere under D8 x integer scale
                           (of the template's coarsest resolution) x translation are found by exact matching of
                           their visible colours and completed (ties broken by positional reflection)
  geom-product-sites       congruent motifs at anchors: complete the anchor set to rows x cols
  geom-frieze              output = the input continued as a 1-D frieze per axis (translation, mirror with
                           repeated or shared boundary), copies / orientation induced
  geom-fold                fold along the central separator line: base half overlaid by the mirrored half
  geom-select-by-symmetry  crop the unique object / panel / block that has / lacks a D4 element, the odd
                           symmetry signature, or the most / least symmetric one
  geom-fundamental-domain  output = top-left quadrant / half of the symmetric picture
  geom-orbit-recolour      recolour cells by whether their whole G-orbit is present ((colour, paired) table)
Beyond gdsl fam_symmetry / fam_symmetry_offset (mirrors + offset transposes about thresholded axes, occluder
only): rotations and anti/diagonal mirrors about any half-integer centre, translations and glides, partial
(local) symmetries, bg-as-unknown completion, per-object / frame / anchor / crossing scopes, symmetry used as a
feature (selection, recolouring, fundamental domain), colour-twisted symmetry and similarity transforms.
Dropped: symmetry-consistent denoising (majority over orbits, no occluder) -- solved nothing, over-fire risk.
"""
import sys; sys.path.insert(0, '/home/claude/work/widen')
import numpy as np
from scipy.signal import correlate
from itertools import product
from gdsl import H, W, bg_of, objects, bbox

# (dy, dx) -> (a*dy + b*dx, c*dy + d*dx)
EL = {"fh": (1, 0, 0, -1), "fv": (-1, 0, 0, 1), "r180": (-1, 0, 0, -1), "T": (0, 1, 1, 0),
      "aT": (0, -1, -1, 0), "r90": (0, 1, -1, 0), "r270": (0, -1, 1, 0)}
GROUPS = {"H": ("fh",), "V": ("fv",), "HV": ("fh", "fv", "r180"), "R2": ("r180",), "D": ("T",), "A": ("aT",),
          "DA": ("T", "aT", "r180"), "R4": ("r90", "r180", "r270"), "D4": tuple(EL)}


def image(e, cy2, cx2, y, x):
    a, b, c, d = EL[e]; dy, dx = 2 * y - cy2, 2 * x - cx2
    ny, nx = cy2 + a * dy + b * dx, cx2 + c * dy + d * dx
    if ny % 2 or nx % 2: return None
    return ny // 2, nx // 2


def pair_counts(e, A, B):
    """N[cy2, cx2] = #{p : A[p] and B[e_c(p)]} for every doubled centre, shape (2h-1, 2w-1)."""
    h, w = A.shape; N = np.zeros((2 * h - 1, 2 * w - 1), dtype=np.int64)
    if not A.any() or not B.any(): return N
    Af, Bf = A.astype(np.int64), B.astype(np.int64)
    if e == "r180":                                  # q = c2 - p  -> full convolution
        return np.rint(correlate(Bf, Af[::-1, ::-1], mode="full", method="auto")).astype(np.int64)
    if e == "fh":                                    # q = (y, cx2 - x)
        v = sum(np.convolve(Af[y], Bf[y]) for y in range(h)); N[:, :] = v[None, :]; return N
    if e == "fv":
        v = sum(np.convolve(Af[:, x], Bf[:, x]) for x in range(w)); N[:, :] = v[:, None]; return N
    # T: q = (x + k, y - k), k = (cy2 - cx2)/2 ;  aT: q = (s - x, s - y), s = (cy2 + cx2)/2
    # r90: q = (x + u, v - y), u = (cy2 - cx2)/2, v = (cy2 + cx2)/2 ; r270 is its inverse
    Y, X = np.mgrid[0:2 * h - 1, 0:2 * w - 1]; par = (Y - X) % 2 == 0
    P = Af.T                                          # P[x, y] = A[y, x]
    if e == "T":                                      # sum P[x,y] B[x+k, y-k] = corr(B,P)[k+w-1, -k+h-1]
        C = np.rint(correlate(Bf, P, mode="full", method="auto")).astype(np.int64)
        k = (Y - X) // 2; i, j = k + w - 1, -k + h - 1
    elif e == "aT":                                   # sum P[x,y] B[s-x, s-y] = conv(B,P)[s, s]
        C = np.rint(correlate(Bf, P[::-1, ::-1], mode="full", method="auto")).astype(np.int64)
        s_ = (Y + X) // 2; i, j = s_, s_
    elif e == "r270":                                 # #{p: A[p], B[r270 p]} = #{q: B[q], A[r90 q]}
        return pair_counts("r90", B, A)
    else:                                             # r90: sum P[x,y] B[x+u, v-y] = C[u + w - 1, v]
        C = np.rint(correlate(Bf, P[:, ::-1], mode="full", method="auto")).astype(np.int64)
        i, j = (Y - X) // 2 + w - 1, (Y + X) // 2
    ok = par & (i >= 0) & (i < C.shape[0]) & (j >= 0) & (j < C.shape[1])
    N[ok] = C[i[ok], j[ok]]
    return N


def glide_counts(A, B, axis):
    """axis 'h': C[t + h - 1, c] = #{p : A[p] and B[y + t, c - x]} (mirror in x, shift t along y);
    axis 'v': the transpose analogue (mirror in y, shift along x), indexed [t + w - 1, c]."""
    if axis == "v": return glide_counts(A.T, B.T, "h")
    return np.rint(correlate(B.astype(np.int64), A[:, ::-1].astype(np.int64), mode="full", method="auto")).astype(np.int64)


def shift_counts(A, B):
    """S[dy + h - 1, dx + w - 1] = #{p : A[p] and B[p + (dy, dx)]}."""
    return np.rint(correlate(B.astype(np.int64), A.astype(np.int64), mode="full", method="auto")).astype(np.int64)


class Fit:
    """Correlation tables for one picture: fg (known foreground), K (known), U (unknown)."""
    def __init__(self, g, fg, K):
        self.g = g; self.fg = fg; self.K = K; self.U = ~K; self.t = {}
        self.cols = [c for c in np.unique(g[fg])]

    def tables(self, e):
        if e not in self.t:
            agree = sum(pair_counts(e, self.fg & (self.g == c), self.K & (self.g == c)) for c in self.cols)
            pairs = pair_counts(e, self.fg, self.K)
            fill = pair_counts(e, self.fg, self.U)
            h, w = self.g.shape
            oob = int(self.fg.sum()) - pair_counts(e, self.fg, np.ones_like(self.fg))
            valid = np.ones((2 * h - 1, 2 * w - 1), bool)
            if e in ("T", "aT", "r90", "r270"):
                Y, X = np.mgrid[0:2 * h - 1, 0:2 * w - 1]; valid = (Y - X) % 2 == 0
            self.t[e] = (agree, pairs - agree, fill, valid)
            self.t["oob:" + e] = oob
        return self.t[e]

    def best(self, group, box=None, min_frac=0.0):
        h, w = self.g.shape
        A = np.zeros((2 * h - 1, 2 * w - 1), np.int64); Cf = A.copy(); F = A.copy(); O = A.copy(); V = np.ones_like(A, bool)
        for e in GROUPS[group]:
            a, c, f, v = self.tables(e); A = A + a; Cf = Cf + c; F = F + f; V &= v; O = O + self.t["oob:" + e]
        ok = V & (Cf == 0) & (A > 0)
        if box is not None:
            r0, c0, r1, c1 = box; m = np.zeros_like(ok); m[2 * r0:2 * r1 + 1, 2 * c0:2 * c1 + 1] = True; ok &= m
        nfg = int(self.fg.sum())
        if not ok.any(): return None
        if min_frac and A[ok].max() < min_frac * nfg * len(GROUPS[group]): return None
        ys, xs = np.nonzero(ok)
        if box is None: my, mx = h - 1, w - 1
        else: my, mx = box[0] + box[2], box[1] + box[3]
        key = [(-(A[y, x] - O[y, x]), -F[y, x], abs(y - my) + abs(x - mx), y, x) for y, x in zip(ys, xs)]
        key.sort()
        return key[0][3], key[0][4]


_CACHE = {}


def get_fit(G, fg, K):
    key = (G.shape, G.tobytes(), fg.tobytes(), K.tobytes())
    f = _CACHE.get(key)
    if f is None:
        if len(_CACHE) > 200: _CACHE.clear()
        f = _CACHE[key] = Fit(G, fg, K)
    return f


def orbit_fill(g, known, group, cy2, cx2, cells, paint, bgc, keep=False):
    """For each unknown cell in `cells` take the colour of known fg orbit members (must agree)."""
    h, w = g.shape; out = g.copy(); changed = False
    for y, x in cells:
        vals = set()
        for e in GROUPS[group]:
            q = image(e, cy2, cx2, y, x)
            if q is None: continue
            qy, qx = q
            if 0 <= qy < h and 0 <= qx < w and known[qy, qx]: vals.add(int(g[qy, qx]))
        if not vals: continue
        if len(vals) > 1: return None
        v = vals.pop()
        if v == bgc and paint is not None: continue
        out[y, x] = v if paint is None else paint; changed = True
    return out if changed or keep else None


def to_np(g): return np.array(g, dtype=np.int64)


# --------------------------------------------------------------------------- bg-as-unknown completion

def complete_bg(g, group, scope, paint, min_frac):
    G = to_np(g); h, w = G.shape; bgc = bg_of(g)
    fg = G != bgc
    if not fg.any(): return None
    out = G.copy(); acted = False
    if group in ("lattice", "auto"):             # every exact symmetry of the foreground, closed by union-find
        f = get_fit(G, fg, fg)
        el = [t for t in auto_elements(f, h, w, ratio=0.75) if group == "auto" or t[0] == "shift"]
        if not el: return None
        K = fg.copy(); r = union_repair(G, K, el, partial=True)
        if r is None or (r == G).all(): return None
        if paint is not None: r[(r != G)] = paint
        return r.tolist()
    if scope == "grid":
        units = [(fg, None, list(zip(*np.nonzero(~fg))))]
    elif scope == "object":
        units = []
        for ob in objects(g, bgc, True, False):
            if len(ob) < 3: continue
            m = np.zeros_like(fg); ys, xs = zip(*ob); m[list(ys), list(xs)] = True
            units.append((m, bbox(ob), list(zip(*np.nonzero(~fg)))))
        if len(units) < 1: return None
    elif scope == "crossing":  # every crossing of separator lines is a candidate centre for its 4 panels
        rows = [y for y in range(h) if len(set(g[y])) == 1 and g[y][0] != bgc]
        cols = [x for x in range(w) if len({g[y][x] for y in range(h)}) == 1 and g[0][x] != bgc]
        if not rows or not cols: return None
        sep = np.zeros_like(fg); sep[rows, :] = True; sep[:, cols] = True
        for r in rows:
            for c in cols:
                ra = max([y for y in rows if y < r], default=-1); rb = min([y for y in rows if y > r], default=h)
                ca = max([x for x in cols if x < c], default=-1); cb = min([x for x in cols if x > c], default=w)
                m = np.zeros_like(fg); m[ra + 1:rb, ca + 1:cb] = True; m &= ~sep
                pic = m & fg
                if not pic.any(): continue
                cnt_a = 0; ok = True
                for y, x in zip(*np.nonzero(pic)):
                    for e in GROUPS[group]:
                        q = image(e, 2 * r, 2 * c, y, x)
                        if q is None or not (0 <= q[0] < h and 0 <= q[1] < w) or not m[q]: continue
                        if pic[q]:
                            if G[q] == G[y, x]: cnt_a += 1
                            else: ok = False
                if not ok or cnt_a == 0: continue
                cells = list(zip(*np.nonzero(m & ~fg)))
                rr = orbit_fill(out, pic, group, 2 * r, 2 * c, cells, paint, bgc)
                if rr is not None: out = rr; acted = True
        return out.tolist() if acted else None
    elif scope == "anchor":   # the unique G-invariant object (size >= 2) is the centre for everything else
        anc = []
        for ob in objects(g, bgc, True, True):
            if len(ob) < 2: continue
            r0, c0, r1, c1 = bbox(ob); st = set(ob)
            if all(image(e, r0 + r1, c0 + c1, y, x) in st for e in GROUPS[group] for y, x in ob): anc.append((ob, r0 + r1, c0 + c1))
        if len(anc) != 1: return None
        ob, cy2, cx2 = anc[0]
        if int(fg.sum()) == len(ob): return None
        r = orbit_fill(out, fg, group, cy2, cx2, list(zip(*np.nonzero(~fg))), paint, bgc, keep=True)
        return None if r is None else r.tolist()
    else:   # frame: fit on a single-colour container, impose on the content inside its bbox
        units = []
        for ob in objects(g, bgc, True, True):
            if len(ob) < 6: continue
            r0, c0, r1, c1 = bbox(ob)
            if r1 - r0 < 2 or c1 - c0 < 2: continue
            s = set(ob)
            inner = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s and fg[y, x]]
            if not inner: continue
            m = np.zeros_like(fg); ys, xs = zip(*ob); m[list(ys), list(xs)] = True
            units.append((m, (r0, c0, r1, c1), inner))
        if not units: return None
    for m, box, _ in units:
        if scope == "frame":
            f = get_fit(G, m, m)
            c = f.best(group, box, min_frac=0.99)            # the frame must be fully symmetric under G
            if c is None: continue
            r0, c0, r1, c1 = box
            content = np.zeros_like(fg); s = m
            content[r0:r1 + 1, c0:c1 + 1] = fg[r0:r1 + 1, c0:c1 + 1] & ~s[r0:r1 + 1, c0:c1 + 1]
            cells = [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if not fg[y, x]]
            r = orbit_fill(out, content, group, c[0], c[1], cells, paint, bgc)
        else:
            f = get_fit(G, m, m)
            c = f.best(group, box, min_frac=min_frac)
            if c is None: continue
            cells = list(zip(*np.nonzero(~fg)))
            r = orbit_fill(out, m & (out != bgc), group, c[0], c[1], cells, paint, bgc)
        if r is not None: out = r; acted = True
    return out.tolist() if acted else None


# --------------------------------------------------------------------------- occluder completion

def occ_window(G, occ):
    ys, xs = np.nonzero(G == occ)
    if not len(ys): return None
    return ys.min(), xs.min(), ys.max(), xs.max()


def auto_elements(f, h, w, shifts=True, ratio=0.0):
    """Every D4 element that is an exact symmetry of the known picture (each with its own best centre),
    plus the (up to 4) best exact translation symmetries (a lattice)."""
    nfg = int(f.fg.sum()); acc = []
    for e in EL:
        a, c, fl, v = f.tables(e)
        ok = v & (c == 0) & (a >= 0.25 * nfg) & (a >= ratio * (nfg - f.t["oob:" + e]))
        if not ok.any(): continue
        ys, xs = np.nonzero(ok)
        best = max(zip(ys, xs), key=lambda t: (a[t], -abs(t[0] - (h - 1)) - abs(t[1] - (w - 1))))
        acc.append((e, int(best[0]), int(best[1])))
    if shifts:
        if "shift" not in f.t:
            ag = sum(shift_counts(f.fg & (f.g == c), f.K & (f.g == c)) for c in f.cols)
            f.t["shift"] = (ag, shift_counts(f.fg, f.K) - ag, shift_counts(f.fg, np.ones_like(f.fg)))
        ag, cf, inb = f.t["shift"]
        ok = (cf == 0) & (ag >= 0.25 * nfg) & (ag >= ratio * inb); ok[h - 1, w - 1] = False
        cand = sorted(((-ag[y, x], abs(y - h + 1) + abs(x - w + 1), y - h + 1, x - w + 1) for y, x in zip(*np.nonzero(ok))))
        got = []
        for _, _, dy, dx in cand:
            if (-dy, -dx) in got: continue
            got.append((dy, dx))
            if len(got) == 4: break
        acc += [("shift", dy, dx) for dy, dx in got]
        for ax in ("h", "v"):                            # glide reflections (mirror + shift along the axis)
            key = "glide" + ax
            if key not in f.t:
                ag = sum(glide_counts(f.fg & (f.g == c), f.K & (f.g == c), ax) for c in f.cols)
                f.t[key] = (ag, glide_counts(f.fg, f.K, ax) - ag, glide_counts(f.fg, np.ones_like(f.fg), ax))
            ag, cf, inb = f.t[key]
            n0 = (h if ax == "h" else w) - 1
            ok = (cf == 0) & (ag >= 0.25 * nfg) & (ag >= ratio * inb); ok[n0, :] = False    # t = 0 is a mirror
            if ok.any():
                t_, c_ = max(zip(*np.nonzero(ok)), key=lambda q: ag[q])
                acc.append(("glide" + ax, int(t_) - n0, int(c_)))
    return acc


def union_repair(G, K, elems, partial=False):
    h, w = G.shape; par = list(range(h * w))
    def find(i):
        while par[i] != i: par[i] = par[par[i]]; i = par[i]
        return i
    for e, cy2, cx2 in elems:
        for y in range(h):
            for x in range(w):
                q = ((y + cy2, x + cx2) if e == "shift" else (y + cy2, cx2 - x) if e == "glideh" else
                     (cx2 - y, x + cy2) if e == "glidev" else image(e, cy2, cx2, y, x))
                if q is None: continue
                qy, qx = q
                if 0 <= qy < h and 0 <= qx < w:
                    a, b = find(y * w + x), find(qy * w + qx)
                    if a != b: par[a] = b
    vals = {}
    for y in range(h):
        for x in range(w):
            if K[y, x]: vals.setdefault(find(y * w + x), set()).add(int(G[y, x]))
    out = G.copy()
    for y in range(h):
        for x in range(w):
            if not K[y, x]:
                s = vals.get(find(y * w + x))
                if not s or len(s) != 1:
                    if partial and not s: continue
                    return None
                out[y, x] = next(iter(s)); K[y, x] = True
    return out


def image_maps(e, cy2, cx2, h, w):
    Y, X = np.mgrid[0:h, 0:w]
    a, b, c, d = EL[e]; dy, dx = 2 * Y - cy2, 2 * X - cx2
    ny, nx = cy2 + a * dy + b * dx, cx2 + c * dy + d * dx
    if (ny % 2).any() or (nx % 2).any(): return None
    ny, nx = ny // 2, nx // 2
    return ny, nx, (ny >= 0) & (ny < h) & (nx >= 0) & (nx < w)


def local_candidates(f, per=2):
    """Partial symmetries: for each element the best centres whose agreement ratio on known cells >= 1/2."""
    nfg = int(f.fg.sum()); out = []
    for e in EL:
        a, cf, fl, v = f.tables(e)
        sc = np.where(v & (a >= 0.1 * nfg) & (2 * a >= a + cf), a, -1)
        for i in np.argsort(-sc.ravel())[:per]:
            y, x = np.unravel_index(i, sc.shape)
            if sc[y, x] > 0: out.append((e, int(y), int(x)))
    return out


def box_sum(M, r):
    h, w = M.shape; P = np.zeros((h + 1, w + 1), np.int64); P[1:, 1:] = M.cumsum(0).cumsum(1)
    Y, X = np.mgrid[0:h, 0:w]
    y0, y1 = np.clip(Y - r, 0, h), np.clip(Y + r + 1, 0, h); x0, x1 = np.clip(X - r, 0, w), np.clip(X + r + 1, 0, w)
    return P[y1, x1] - P[y0, x1] - P[y1, x0] + P[y0, x0]


def local_repair(G, K, cands, need=12, rmax=6, rounds=3):
    """Fill unknown p from e(p) when e is an exact symmetry of the known cells in the smallest window around p
    holding >= `need` known pairs; all locally valid candidates must agree."""
    h, w = G.shape; G = G.copy(); K = K.copy()
    maps = [m for m in (image_maps(e, cy2, cx2, h, w) for e, cy2, cx2 in cands) if m is not None]
    for _ in range(rounds):
        if K.all(): break
        votes = {}
        for ny, nx, inb in maps:
            qy, qx = np.where(inb, ny, 0), np.where(inb, nx, 0)
            kn = K & inb & K[qy, qx]
            agr = kn & (G == G[qy, qx]); con = kn & ~agr
            dec = np.zeros((h, w), bool); ok = np.zeros((h, w), bool)
            for r in range(1, rmax + 1):
                n = box_sum(agr.astype(np.int64) + con, r); c = box_sum(con.astype(np.int64), r)
                newly = ~dec & (n >= need)
                ok |= newly & (c == 0); dec |= newly
            src = ~K & ok & inb & K[qy, qx]
            for y, x in zip(*np.nonzero(src)): votes.setdefault((y, x), set()).add(int(G[qy[y, x], qx[y, x]]))
        got = False
        for (y, x), vs in votes.items():
            if len(vs) == 1: G[y, x] = vs.pop(); K[y, x] = True; got = True
        if not got: break
    return G, K


def rect_colour(G):
    """The unique non-background colour whose cells exactly fill their bounding box (area >= 4)."""
    bgc = np.bincount(G.ravel()).argmax(); hits = []
    for c in np.unique(G):
        if c == bgc: continue
        ys, xs = np.nonzero(G == c)
        a = (ys.max() - ys.min() + 1) * (xs.max() - xs.min() + 1)
        if a >= 4 and a == len(ys): hits.append(int(c))
    return hits[0] if len(hits) == 1 else None


def repair_occ(g, occ, group, patch):
    G = to_np(g); h, w = G.shape
    if occ == "rect":
        occ = rect_colour(G)
        if occ is None: return None
    win = occ_window(G, occ)
    if win is None: return None
    K = G != occ
    # every known cell is informative (bg included): fit on all known cells
    f = get_fit(G, K, K)
    if group in ("auto", "local"):
        el = auto_elements(f, h, w)
        if not el and group == "auto": return None
        K2 = K.copy()
        out = union_repair(G, K2, el, partial=group == "local")
        if out is None: return None
        if group == "local" and not K2.all():
            out, K2 = local_repair(out, K2, local_candidates(f))
            if not K2.all(): return None
    else:
        c = f.best(group, None, min_frac=0.3)
        if c is None: return None
        el = [(e, c[0], c[1]) for e in GROUPS[group]]
        out = union_repair(G, K, el)
    if out is None: return None
    if patch:
        r0, c0, r1, c1 = win
        return out[r0:r1 + 1, c0:c1 + 1].tolist()
    return out.tolist()


# --------------------------------------------------------------------------- families

def _same(train): return all((H(p["input"]), W(p["input"])) == (H(p["output"]), W(p["output"])) for p in train)


def _new_colour(train):
    new = None
    for p in train:
        ci = {v for r in p["input"] for v in r}; co = {v for r in p["output"] for v in r}
        n = co - ci; new = n if new is None else new & n
    return next(iter(new)) if new and len(new) == 1 else None


def fam_sym_complete_bg(train):
    """Background holes restored from the G-orbit (G, centre fitted per grid / object / frame)."""
    if not _same(train): return
    i0, o0 = train[0]["input"], train[0]["output"]; bg = bg_of(i0)
    # output must only add foreground over background
    if all(p["input"] == p["output"] for p in train): return
    for p in train:
        i, o = p["input"], p["output"]; b = bg_of(i)
        if any(i[y][x] != o[y][x] and i[y][x] != b for y in range(H(i)) for x in range(W(i))): return
    nc = _new_colour(train)
    paints = [None] + ([nc] if nc is not None else [])
    for scope in ("grid", "object", "frame", "anchor", "crossing"):
        for grp in tuple(GROUPS) + (("lattice", "auto") if scope == "grid" else ()):
            for paint in paints:
                mf = 0.0 if scope == "frame" else 0.15
                yield (f"geom-sym-complete:{scope}[{grp}{'' if paint is None else ',paint=c%d' % paint}]", 4,
                       lambda g, s=scope, G=grp, p=paint, mf=mf: complete_bg(g, G, s, p, mf))


def fam_sym_repair_occ(train):
    """Occluder colour cells restored from the fitted symmetry group (or all exact symmetries: auto)."""
    i0, o0 = train[0]["input"], train[0]["output"]
    same = _same(train)
    occs = []
    if same:
        occs = [c for c in {v for r in i0 for v in r}
                if all(c in {v for r in p["input"] for v in r} and c not in {v for r in p["output"] for v in r} for p in train)]
    else:
        for c in {v for r in i0 for v in r}:
            ok = True
            for p in train:
                win = occ_window(to_np(p["input"]), c)
                if win is None or (win[2] - win[0] + 1, win[3] - win[1] + 1) != (H(p["output"]), W(p["output"])): ok = False; break
            if ok: occs.append(c)
    if all(rect_colour(to_np(p["input"])) is not None for p in train):
        if same and all(rect_colour(to_np(p["input"])) not in {v for r in p["output"] for v in r} for p in train):
            occs.append("rect")
        if not same and all(occ_window(to_np(p["input"]), rect_colour(to_np(p["input"]))) is not None and
                            (lambda wn, p: (wn[2] - wn[0] + 1, wn[3] - wn[1] + 1) == (H(p["output"]), W(p["output"])))(
                                occ_window(to_np(p["input"]), rect_colour(to_np(p["input"]))), p) for p in train):
            occs.append("rect")
    for c in occs:
        for grp in ("auto", "local") + tuple(GROUPS):
            yield (f"geom-sym-repair{'-patch' if not same else ''}[{c if c == 'rect' else 'c%d' % c},{grp}]", 4,
                   lambda g, c=c, G=grp, pt=not same: repair_occ(g, c, G, pt))


def symmetrize_colour(g, group, centre):
    """The colour of the largest single-colour component becomes G-invariant (union with its orbit),
    overwriting whatever lies under the images; centre = grid centre or fitted on that colour alone."""
    G = to_np(g); h, w = G.shape; bgc = bg_of(g)
    obs = objects(g, -1, False, True)          # every colour competes (the blob may be the mode colour)
    if not obs: return None
    big = max(obs, key=len)
    if sum(len(o) == len(big) for o in obs) > 1: return None
    c = g[big[0][0]][big[0][1]]; m = G == c
    if centre == "grid": cy2, cx2 = h - 1, w - 1
    else:
        f = get_fit(G, m, m); r = f.best(group, None, min_frac=0.15)
        if r is None: return None
        cy2, cx2 = r
    out = G.copy()
    for y, x in zip(*np.nonzero(m)):
        for e in GROUPS[group]:
            q = image(e, cy2, cx2, y, x)
            if q is None: return None
            if 0 <= q[0] < h and 0 <= q[1] < w: out[q] = c
    return out.tolist()


def fam_symmetrize_colour(train):
    if not _same(train): return
    for grp, centre in product(GROUPS, ("grid", "fit")):
        yield (f"geom-symmetrize-colour:{centre}[{grp}]", 4, lambda g, grp=grp, centre=centre: symmetrize_colour(g, grp, centre))


def fold_dominant(g, e0, conn):
    """Each object is made mirror-symmetric under the single reflection e about its best-agreeing axis inside
    its bbox: the half holding more cells is kept and mirrored, the other half is replaced."""
    G = to_np(g); h, w = G.shape; bgc = bg_of(g); out = G.copy(); acted = False
    for ob in objects(g, bgc, True, conn == "c8"):
        if len(ob) < 4: continue
        m = np.zeros((h, w), bool); ys, xs = zip(*ob); m[list(ys), list(xs)] = True
        f = get_fit(G, m, m); r0, c0, r1, c1 = bbox(ob)
        best = []
        if e0 == "line":                                   # axis = straight line of one colour inside the object
            lines = []
            for col in {int(G[y, x]) for y, x in ob}:
                cs = [(y, x) for y, x in ob if G[y, x] == col]
                if len(cs) < 2 or len(cs) * 2 > len(ob): continue
                from collections import Counter
                cnt = Counter()
                for y, x in cs: cnt[("fv", 2 * y)] += 1; cnt[("fh", 2 * x)] += 1; cnt[("T", y - x)] += 1; cnt[("aT", y + x)] += 1
                (k, n), = cnt.most_common(1)
                if n >= 3 and 2 * n > len(cs) and sum(v == n for v in cnt.values()) == 1:
                    lines.append((k[0], k[1], k[1]))
            if len(lines) != 1: continue
            e, a_, b_ = lines[0]
            sc = np.full((2 * h - 1, 2 * w - 1), -1)
            if e == "fv": sc[a_, :] = 1
            elif e == "fh": sc[:, a_] = 1
            elif e == "T": sc[(np.subtract.outer(np.arange(2 * h - 1), np.arange(2 * w - 1))) == 2 * a_] = 1
            else: sc[(np.add.outer(np.arange(2 * h - 1), np.arange(2 * w - 1))) == 2 * a_] = 1
            best.append((1, e, sc))
        for e in (("fh", "fv", "T", "aT") if e0 == "best" else () if e0 == "line" else (e0,)):
            a, cf, fl, v = f.tables(e)
            box = np.zeros_like(v); box[2 * r0:2 * r1 + 1, 2 * c0:2 * c1 + 1] = True
            sc = np.where(v & box, a - cf, -10 ** 6); best.append((sc.max(), e, sc))
        best.sort(key=lambda t: -t[0])
        if len(best) > 1 and best[0][0] == best[1][0]: return None
        _, e, sc = best[0]
        if sc.max() <= 0: continue
        top = np.argwhere(sc == sc.max())
        keys = {(int(y - x) if e == "T" else int(y + x) if e == "aT" else int(y) if e == "fv" else int(x)) for y, x in top}
        if len(keys) != 1: return None
        cy2, cx2 = map(int, top[0])
        side = lambda y, x: np.sign((2 * y - cy2) if e == "fv" else (2 * x - cx2) if e == "fh" else
                                    ((2 * y - cy2) - (2 * x - cx2)) if e == "T" else ((2 * y - cy2) + (2 * x - cx2)))
        cnt = {1: 0, -1: 0}
        for y, x in ob:
            s_ = side(y, x)
            if s_: cnt[int(s_)] += 1
        if cnt[1] == cnt[-1]: continue
        keep = 1 if cnt[1] > cnt[-1] else -1
        for y, x in ob:
            if side(y, x) == -keep: out[y, x] = bgc
        for y, x in ob:
            if side(y, x) == keep:
                q = image(e, cy2, cx2, y, x)
                if q is None or not (0 <= q[0] < h and 0 <= q[1] < w): return None
                out[q] = G[y, x]
        acted = True
    return out.tolist() if acted else None


def fam_fold_dominant(train):
    if not _same(train): return
    for e, conn in product(("line", "best", "fh", "fv", "T", "aT"), ("m8", "c8")):
        yield (f"geom-fold-dominant[{e},{conn}]", 4, lambda g, e=e, conn=conn: fold_dominant(g, e, conn))


def twisted_quadrants(g, dihedral):
    """Colour-twisted D2 symmetry: the picture (fg bbox, even sides) is split into quadrants; the fullest
    quadrant Q is the motif; every other quadrant equals e(Q) (e = mirror / rotation taking Q there) up to a
    colour substitution sigma_e induced from that quadrant's visible hint cells."""
    G = to_np(g); bgc = bg_of(g); fg = G != bgc
    ys, xs = np.nonzero(fg)
    if not len(ys): return None
    h, w = G.shape; R0, C0, R1, C1 = ys.min(), xs.min(), ys.max(), xs.max(); cands = []
    for sa, sb in ((0, 0), (0, 1), (1, 0), (1, 1)):     # motif = largest solid square at a bbox corner
        k = 0
        while True:
            k1 = k + 1
            y0 = R0 if sa == 0 else R1 - k1 + 1; x0 = C0 if sb == 0 else C1 - k1 + 1
            if y0 < 0 or x0 < 0 or y0 + k1 > h or x0 + k1 > w or not fg[y0:y0 + k1, x0:x0 + k1].all(): break
            k = k1
        if k < 2: continue
        r0 = R0 if sa == 0 else R1 - 2 * k + 1; c0 = C0 if sb == 0 else C1 - 2 * k + 1
        r1, c1 = r0 + 2 * k - 1, c0 + 2 * k - 1
        if r0 < 0 or c0 < 0 or r1 >= h or c1 >= w or R1 > r1 or C1 > c1: continue
        cands.append((k, sa, sb, r0, c0, r1, c1))
    if len(cands) != 1: return None
    k, sa, sb, r0, c0, r1, c1 = cands[0]
    P = G[r0:r1 + 1, c0:c1 + 1].copy(); qh = qw = k
    quads = {(a, b): P[a * qh:(a + 1) * qh, b * qw:(b + 1) * qw] for a in (0, 1) for b in (0, 1)}
    full = [(sa, sb)]
    sa, sb = full[0]; Q = quads[full[0]]
    for (a, b), T_ in quads.items():
        if (a, b) == (sa, sb): continue
        if dihedral == "mirror":
            M = Q[::-1 if a != sa else 1, ::-1 if b != sb else 1]
        else:                                           # rotation about the picture centre
            if qh != qw: return None
            k = {(0, 1): 3, (1, 1): 2, (1, 0): 1}[((a - sa) % 2, (b - sb) % 2)] if (sa, sb) == (0, 0) else None
            if k is None: return None
            M = np.rot90(Q, k)
        sig = {}
        for u, v in zip(M.ravel(), T_.ravel()):
            if v == bgc: continue
            if sig.setdefault(int(u), int(v)) != v: return None
        if not sig: return None
        quads[(a, b)][:, :] = np.vectorize(lambda u: sig.get(int(u), bgc))(M)   # unseen colours vanish
    out = G.copy(); out[r0:r1 + 1, c0:c1 + 1] = P
    return out.tolist()


def fam_twisted_quadrants(train):
    if not _same(train): return
    for d in ("mirror", "rotation"):
        yield (f"geom-twisted-quadrants[{d}]", 5, lambda g, d=d: twisted_quadrants(g, d))


# --------------------------------------------------------------------------- product lattice of sites

def product_sites(g, paint, conn):
    """Congruent copies of one motif sit at (row, col) anchors; the anchor set is completed to the full
    Cartesian product rows x cols (a possibly irregular rectangular lattice), new copies painted `paint`."""
    bgc = bg_of(g); h, w = H(g), W(g)
    obs = objects(g, bgc, True, conn == "c8")
    if len(obs) < 3: return None
    shapes = {}
    for ob in obs:
        r0, c0, _, _ = bbox(ob)
        shapes.setdefault(tuple(sorted((y - r0, x - c0, g[y][x]) for y, x in ob)), []).append((r0, c0))
    if len(shapes) != 1: return None
    (shape, anchors), = shapes.items()
    rows = sorted({a for a, _ in anchors}); cols = sorted({b for _, b in anchors})
    if len(rows) * len(cols) == len(anchors): return None
    out = [r[:] for r in g]
    for a in rows:
        for b in cols:
            if (a, b) in anchors: continue
            for dy, dx, v in shape:
                y, x = a + dy, b + dx
                if not (0 <= y < h and 0 <= x < w) or g[y][x] != bgc: return None
                out[y][x] = v if paint is None else paint
    return out


def fam_product_sites(train):
    if not _same(train): return
    nc = _new_colour(train)
    for paint, conn in product([None] + ([nc] if nc is not None else []), ("c8", "m8")):
        yield (f"geom-product-sites[{conn}{'' if paint is None else ',paint=c%d' % paint}]", 4,
               lambda g, paint=paint, conn=conn: product_sites(g, paint, conn))


# --------------------------------------------------------------------------- similarity-transform completion

def block_factor(P, bgc):
    """Largest f such that the picture is an f-fold upscaling of a smaller one."""
    h, w = len(P), len(P[0])
    for f in range(min(h, w), 1, -1):
        if h % f or w % f: continue
        if all(P[y][x] == P[y - y % f][x - x % f] for y in range(h) for x in range(w)): return f
    return 1


def similarity_complete(g, tie):
    """A template (the multicolour object with the most colours) exists; elsewhere only its 'shown' colours
    appear (colours of the template also present outside it), as copies of the template under a similarity
    transform (D8 x integer scale of the template's coarsest resolution x translation). Each copy is found by
    exact matching of its shown cells and completed with the template's hidden colours."""
    bgc = bg_of(g); h, w = H(g), W(g)
    seprow = {y for y in range(h) if len(set(g[y])) == 1 and g[y][0] != bgc}
    sepcol = {x for x in range(w) if len({g[y][x] for y in range(h)}) == 1 and g[0][x] != bgc}
    m = [[bgc if (y in seprow or x in sepcol) else g[y][x] for x in range(w)] for y in range(h)]
    obs = objects(m, bgc, True, False)
    if len(obs) < 2: return None
    ncol = [len({m[y][x] for y, x in o}) for o in obs]
    top = max(ncol)
    if top < 2 or ncol.count(top) != 1: return None
    T = obs[ncol.index(top)]; Tset = set(T)
    r0, c0, r1, c1 = bbox(T)
    P = [[m[y][x] if (y, x) in Tset else bgc for x in range(c0, c1 + 1)] for y in range(r0, r1 + 1)]
    f = block_factor(P, bgc); P0 = [row[::f] for row in P[::f]]
    tcols = {v for r in P0 for v in r} - {bgc}
    outside = {m[y][x] for y in range(h) for x in range(w) if m[y][x] != bgc and (y, x) not in Tset}
    shown = tcols & outside; hidden = tcols - shown
    if not shown or not hidden or outside - tcols: return None
    seeds = [o for o in objects(m, bgc, True, True) if (o[0][0], o[0][1]) not in Tset]
    used = set(); out = [r[:] for r in g]; acted = False
    tcy, tcx = (r0 + r1) / 2 - (h - 1) / 2, (c0 + c1) / 2 - (w - 1) / 2
    for sd in seeds:
        if (sd[0][0], sd[0][1]) in used: continue
        sc = m[sd[0][0]][sd[0][1]]; sb = bbox(sd); sshape = {(y - sb[0], x - sb[1]) for y, x in sd}
        cands = {}
        for D, k in product(D8, range(1, 7)):
            Q = D8[D](P0); Q = [[v for v in row for _ in range(k)] for row in Q for _ in range(k)]
            qh, qw = len(Q), len(Q[0])
            qcells = [(y, x) for y in range(qh) for x in range(qw) if Q[y][x] == sc]
            for comp in objects([[1 if Q[y][x] == sc else 0 for x in range(qw)] for y in range(qh)], 0, True, True):
                cb = bbox(comp)
                if {(y - cb[0], x - cb[1]) for y, x in comp} != sshape: continue
                oy, ox = sb[0] - cb[0], sb[1] - cb[1]
                ok = True; paint = []; cover = []
                for y in range(qh):
                    for x in range(qw):
                        v = Q[y][x]
                        if v == bgc: continue
                        yy, xx = y + oy, x + ox
                        if not (0 <= yy < h and 0 <= xx < w):
                            if v in shown: ok = False; break
                            continue                                  # hidden parts are clipped at the edge
                        if v in shown:
                            if m[yy][xx] != v: ok = False; break
                            cover.append((yy, xx))
                        elif m[yy][xx] != bgc and m[yy][xx] != v: ok = False; break
                        else: paint.append((yy, xx, v))
                    if not ok: break
                if ok and paint:
                    key = tuple(sorted(paint))
                    cands.setdefault(key, (D, cover))
        if not cands: continue
        if len(cands) > 1:
            if tie != "positional": return None
            scy, scx = (sb[0] + sb[2]) / 2 - (h - 1) / 2, (sb[1] + sb[3]) / 2 - (w - 1) / 2
            fy, fx = (scy * tcy < 0), (scx * tcx < 0)
            want = {(False, False): "id", (False, True): "fh", (True, False): "fv", (True, True): "r180"}[(fy, fx)]
            pick = [k_ for k_, (D, _) in cands.items() if D == want]
            if len(pick) != 1: return None
            key = pick[0]
        else:
            key = next(iter(cands))
        for y, x, v in key: out[y][x] = v
        for p in cands[key][1]: used.add(p)
        acted = True
    return out if acted else None


def fam_similarity_complete(train):
    if not _same(train): return
    for tie in ("unique", "positional"):
        yield (f"geom-similarity-complete[{tie}]", 5, lambda g, tie=tie: similarity_complete(g, tie))


# --------------------------------------------------------------------------- folding along a mirror line

def fold(g, base):
    """The grid is folded along its central separator line (a full row / column of one colour): the output is
    the `base` half with the mirror image of the other half showing through its background cells."""
    gg = g if base in ("L", "R") else [list(r) for r in zip(*g)]
    h, w = H(gg), W(gg); bgc = bg_of(g)
    if w % 2 == 0: return None
    m = w // 2
    if len({gg[y][m] for y in range(h)}) != 1 or gg[0][m] == bgc: return None
    A = [r[:m] for r in gg]; B = [r[m + 1:][::-1] for r in gg]
    if base in ("R", "B"): A, B = [r[::-1] for r in B], [r[::-1] for r in A]
    out = [[a if a != bgc else b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]
    return out if base in ("L", "R") else [list(r) for r in zip(*out)]


def fam_fold(train):
    i, o = train[0]["input"], train[0]["output"]
    if not ((H(o) == H(i) and 2 * W(o) + 1 == W(i)) or (W(o) == W(i) and 2 * H(o) + 1 == H(i))): return
    for base in ("L", "R", "T", "B"):
        yield (f"geom-fold[base={base}]", 3, lambda g, base=base: fold(g, base))


# --------------------------------------------------------------------------- reflective (frieze) extension

def frieze_index(L, n, mode, rev):
    """Index map of a 1-D frieze: tile (translation), pp-dup (glide/mirror, boundary repeated),
    pp-shared (mirror sharing the boundary cell)."""
    if mode == "tile": p, f = n, (lambda t: t)
    elif mode == "dup": p, f = 2 * n, (lambda t: t if t < n else 2 * n - 1 - t)
    else:
        if n < 2: return None
        p, f = 2 * n - 2, (lambda t: t if t < n else 2 * n - 2 - t)
    idx = [f(k % p) for k in range(L)]
    return [n - 1 - v for v in idx] if rev else idx


def frieze_len(n, mode, m):
    return m * (n - 1) + 1 if mode == "shared" else m * n


def fam_frieze(train):
    """Output = the input continued along each axis as a frieze (translation, mirror with repeated or shared
    boundary), m copies; per-axis mode, copy count and starting orientation induced from the training pairs."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(o), W(o)) == (H(i), W(i)) or H(o) < H(i) or W(o) < W(i): return
    def ms(n, L, mode):
        if mode == "shared": return (L - 1) // (n - 1) if n > 1 and (L - 1) % (n - 1) == 0 else None
        return L // n if L % n == 0 else None
    for my, mx in product(("tile", "dup", "shared"), repeat=2):
        ky, kx = ms(H(i), H(o), my), ms(W(i), W(o), mx)
        if not ky or not kx or ky * kx == 1 or ky > 8 or kx > 8: continue
        if all(ms(H(p["input"]), H(p["output"]), my) == ky and ms(W(p["input"]), W(p["output"]), mx) == kx for p in train) is False: continue
        for ry, rx in product((False, True), repeat=2):
            if (ky == 1 and ry) or (kx == 1 and rx): continue
            def fn(g, my=my, mx=mx, ky=ky, kx=kx, ry=ry, rx=rx):
                n, m = H(g), W(g)
                iy = frieze_index(frieze_len(n, my, ky), n, my, ry); ix = frieze_index(frieze_len(m, mx, kx), m, mx, rx)
                if iy is None or ix is None: return None
                return [[g[a][b] for b in ix] for a in iy]
            yield (f"geom-frieze[{my}x{ky}{',rev' if ry else ''};{mx}x{kx}{',rev' if rx else ''}]", 3, fn)


# --------------------------------------------------------------------------- symmetry as a feature

from gdsl import D8, crop, split_panels

INV = {"fh": "fh", "fv": "fv", "r180": "r180", "T": "T", "aT": "aT", "r90": "r90", "r270": "r270"}


def signature(p):
    """Set of non-identity D4 elements leaving the picture (a list grid) invariant about its own centre."""
    sq = H(p) == W(p)
    return frozenset(e for e in ("fh", "fv", "r180", "T", "aT", "r90") if (sq or e in ("fh", "fv", "r180")) and D8[e](p) == p)


def parts(g, src):
    bg = bg_of(g)
    if src == "panels":
        sp = split_panels(g)
        return [p for p in sp[0]] if sp else []
    if src == "blocks":                               # k equal square blocks along the long side
        h, w = H(g), W(g)
        if h > w and h % w == 0: return [g[k:k + w] for k in range(0, h, w)]
        if w > h and w % h == 0: return [[r[k:k + h] for r in g] for k in range(0, w, h)]
        return []
    obs = objects(g, bg, True, src == "c8")
    return [crop(g, bbox(o)) for o in obs if len(o) >= 3]


def pick_by_symmetry(g, src, how):
    ps = parts(g, src)
    if len(ps) < 2: return None
    sig = [signature(p) for p in ps]
    if how.startswith("has:") or how.startswith("lacks:"):
        e = how.split(":")[1]; want = how.startswith("has:")
        hits = [p for p, s in zip(ps, sig) if (e in s) == want]
    elif how == "odd-signature":
        from collections import Counter
        cnt = Counter(sig); hits = [p for p, s in zip(ps, sig) if cnt[s] == 1]
        if len(cnt) != 2: return None
    else:
        k = [len(s) for s in sig]; t = max(k) if how == "most-symmetric" else min(k)
        hits = [p for p, v in zip(ps, k) if v == t]
    return hits[0] if len(hits) == 1 else None


def fam_select_by_symmetry(train):
    """Crop the unique object / panel whose symmetry group has (lacks) an element, or is the odd one out."""
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) >= H(i) * W(i): return
    hows = [f"{m}:{e}" for m in ("has", "lacks") for e in ("fh", "fv", "r180", "T", "aT", "r90")] + \
           ["odd-signature", "most-symmetric", "least-symmetric"]
    for src in ("c8", "m8", "panels", "blocks"):
        for how in hows:
            yield (f"geom-select-by-symmetry:{src}[{how}]", 4, lambda g, src=src, how=how: pick_by_symmetry(g, src, how))


def fundamental_domain(g, part, req):
    bg = bg_of(g); h, w = H(g), W(g)
    cells = [(y, x) for y in range(h) for x in range(w) if g[y][x] != bg]
    if not cells: return None
    p = crop(g, bbox(cells)); sg = signature(p)
    if not set(req) <= sg: return None
    ph, pw = H(p), W(p)
    if part == "quadrant": return [r[:(pw + 1) // 2] for r in p[:(ph + 1) // 2]]
    if part == "left": return [r[:(pw + 1) // 2] for r in p]
    return [r[:] for r in p[:(ph + 1) // 2]]


def fam_fundamental_domain(train):
    """Output = the fundamental domain (top-left quadrant / half) of the symmetric foreground picture."""
    i, o = train[0]["input"], train[0]["output"]
    if H(o) * W(o) >= H(i) * W(i): return
    for part, req in (("quadrant", ("r180",)), ("left", ("fh",)), ("top", ("fv",))):
        yield (f"geom-fundamental-domain:{part}", 3, lambda g, part=part, req=req: fundamental_domain(g, part, req))


def paired_mask(g, group, centre, bgc=None):
    G = to_np(g); h, w = G.shape; bgc = bg_of(g) if bgc is None else bgc; fg = G != bgc
    if centre == "grid": cy2, cx2 = h - 1, w - 1
    else:
        ys, xs = np.nonzero(fg)
        if not len(ys): return None
        cy2, cx2 = ys.min() + ys.max(), xs.min() + xs.max()
    M = np.zeros_like(fg)
    for y, x in zip(*np.nonzero(fg)):
        ok = True
        for e in GROUPS[group]:
            q = image(e, cy2, cx2, y, x)
            if q is None or not (0 <= q[0] < h and 0 <= q[1] < w) or G[q] != G[y, x]: ok = False; break
        M[y, x] = ok
    return G, M


def fam_orbit_recolour(train):
    """Foreground cells are recoloured according to whether their whole G-orbit (about the grid / picture
    centre) is present; the (colour, paired) -> colour table is induced from training."""
    if not _same(train): return
    st = None
    for p in train:
        i, o = to_np(p["input"]), to_np(p["output"])
        s_ = {int(c) for c in np.unique(i) if (o[i == c] == c).all()}
        st = s_ if st is None else st & s_
    bgs = [None] + ([next(iter(st))] if st and len(st) == 1 else [])
    for grp, centre, bgx in product(("H", "V", "HV", "R2", "D", "A", "R4", "D4"), ("grid", "fg"), bgs):
        if True:
            table = {}; ok = True
            for p in train:
                r = paired_mask(p["input"], grp, centre, bgx)
                if r is None: ok = False; break
                G, M = r; O = to_np(p["output"]); bgc = bg_of(p["input"]) if bgx is None else bgx
                for y, x in zip(*np.nonzero(G != bgc)):
                    k = (int(G[y, x]), bool(M[y, x]))
                    if table.setdefault(k, int(O[y, x])) != O[y, x]: ok = False; break
                if not ok or (O[G == bgc] != bgc).any(): ok = False; break
            if not ok or all(k[0] == v for k, v in table.items()): continue
            if len({v for k, v in table.items() if k[1]} ^ {v for k, v in table.items() if not k[1]}) == 0: continue

            def fn(g, grp=grp, centre=centre, table=table, bgx=bgx):
                r = paired_mask(g, grp, centre, bgx)
                if r is None: return None
                G, M = r; out = G.copy(); bgc = bg_of(g) if bgx is None else bgx
                for y, x in zip(*np.nonzero(G != bgc)):
                    k = (int(G[y, x]), bool(M[y, x]))
                    if k not in table: return None
                    out[y, x] = table[k]
                return out.tolist()
            yield (f"geom-orbit-recolour[{grp},{centre}{'' if bgx is None else ',bg=static'}]", 4, fn)


FAMILIES = (fam_sym_complete_bg, fam_sym_repair_occ, fam_symmetrize_colour, fam_fold_dominant, fam_twisted_quadrants, fam_frieze, fam_product_sites, fam_fold, fam_similarity_complete, fam_select_by_symmetry, fam_fundamental_domain, fam_orbit_recolour)


In [ ]:
%%writefile /kaggle/working/probe/prior_mechanics.py
"""MECHANICS prior: rigid bodies in mechanical equilibrium, and common-sense machines.

Laws (the prior, stated before looking for tasks)
------------------------------------------------
Bodies.  A picture is a set of rigid BODIES (connected components: 4/8-connected, single- or multi-colour;
         or finer units: single cells, 1-wide strips parallel to the motion) lying on a background.
L0 conservation     bodies are neither created nor destroyed by motion: the foreground mass of the output
                    equals the input's (gravity, levers) or is smaller only by swept-away debris / unused keys
                    (lock and key).  Used as a precondition.
L1 rigidity         a body moves as a whole: translation (and, only when fitting a key, a D4 rotation);
                    its shape and colours are preserved (optionally a class-dependent recolouring on landing).
L2 impenetrability  two bodies never overlap; STATIC bodies (colours that never change in any training pair,
                    full-length walls / floors, the grid frame) never move.
L3 field            every movable body is driven by a field direction f(body): a constant direction, a
                    table class -> direction (class = colour / topology (holed | open | solid) / thickness /
                    mass threshold), or an ATTRACTOR (the static body it is aligned with: nearest one, the one
                    of its own colour, or only if that is its nearest anchor overall).
L4 support          a body moves along f until the next step would violate L2 (contact) or leave the grid;
                    the equilibrium is reached by settling bodies leading edge first (piling / stacking).
                    Bodies driven in different directions live in different layers (they may pass each other);
                    bodies of one layer pile on each other.  A body may take the colour of what it lands on
                    (absorption).  In time order: pieces released one after the other build a tower (LIFO).
L5 complementarity  LOCK AND KEY: a lock is a body with cavities (enclosed holes, or notches of its bounding
                    box); a key fits a cavity when a rigid motion maps it onto the cavity so that the cavity is
                    filled exactly and nothing overlaps (the key may protrude only outside the lock's hull), or
                    so that lock + key close into a solid rectangle (all such rectangles congruent).  Keys go
                    to the locks they fit: the unique maximum matching (ties: least total displacement = least
                    action), or an exact cover of all holes by several keys.
L6 levers           an ARM attached to (or sighted from) a PIVOT swings 90 degrees about it: fixed sense (cw /
                    ccw), or it falls to the side with more free room; standing bars topple about their base
                    and knock over the bars they reach (dominoes, chain reaction).

Search space (all parameters induced from the training pairs; every program is verified on all pairs)
  mech:lockkey[seg,lock,fit,group,rest(,side)(,rc)]
      seg   c4|c8|m4|m8          body segmentation
      lock  holed (bodies with enclosed empty holes; cavities = holes) | largest (cavities = its bbox notches)
            | static (bodies of static colours) | pairs (two bodies of one colour: the smaller one / the one on an
              induced side is the key)
      fit   fill (one key per cavity) | tile (exact cover of all holes, several keys per hole) | rect / rect=
            (lock + key = solid rectangle / congruent rectangles)
      group T | R4 | D4          allowed key motions
      rest  keep | erase         unmatched keys; debris colours (in every input, absent from its output) erased
      rc    optional recolour of placed keys to the one new output colour
  mech:lockkey-panels[lock]      two panels: the key panel is inserted in place when it is the exact complement of
                                 the lock panel, else the lock panel is output
  mech:lockkey-crop[seg,cav,fit,group]  the unique body whose cavities can all be filled exactly is the lock;
                                 output = the completed lock's bounding box
  mech:gravity[anchors,seg,unit,field(,recol)]
      anchors none | colour (static colours) | lines (full rows / columns)
      unit    body | cell | vstrip | hstrip
      field   up|down|left|right | table:<key> (key in colour, topo, thick, mass>k; value = direction or stay)
              | attract | attract-colour | attract-near
      recol   none | absorb | table (landing colour as a function of colour / topo / thick)
  mech:gravity[stack-panels,d,order]    panel contents dropped one after the other into one frame
  mech:lever[pivot p,arm a,attached|aligned,cw|ccw|room(,new c)(,old c)]
  mech:lever[topple,floor,trigger t,d]  dominoes
Dropped after measurement (solved < 3 tasks): SLOT MACHINE (a reel = the grid inside a border line is rolled
cyclically until its origin aligns with the single marker on the border; 6d1d5c90, 79cce52d only).
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, objects, bbox, static_colours, split_panels

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


def same_shape(train):
    return all((H(p['input']), W(p['input'])) == (H(p['output']), W(p['output'])) for p in train)


def comps4(cells):
    """4-connected components of a set of cells."""
    cells = set(cells); out = []
    while cells:
        s = cells.pop(); st = [s]; comp = [s]
        while st:
            y, x = st.pop()
            for dy, dx in N4:
                q = (y + dy, x + dx)
                if q in cells:
                    cells.remove(q); st.append(q); comp.append(q)
        out.append(sorted(comp))
    return out


def segment(g, bg, seg):
    return objects(g, bg, diag=seg[1] == '8', by_colour=seg[0] == 'c')


# ------------------------------------------------------------------ D4 variants of a key
def _tf(k, y, x):
    return ((y, x), (y, -x), (-y, x), (-y, -x), (x, y), (x, -y), (-x, y), (-x, -y))[k]


GROUPS = {'T': (0,), 'R4': (0, 3, 6, 5), 'D4': tuple(range(8))}   # R4: id, r180, r90, r270 in _tf indices


def variants(cells, g, group):
    """cells -> list of distinct normalised coloured shapes [(dy,dx,c), ...] under the group."""
    out = []; seen = set()
    for k in GROUPS[group]:
        t = [(*_tf(k, y, x), g[y][x]) for y, x in cells]
        my = min(a for a, _, _ in t); mx = min(b for _, b, _ in t)
        v = tuple(sorted((a - my, b - mx, c) for a, b, c in t))
        if v not in seen: seen.add(v); out.append((k, v))
    return out


# ------------------------------------------------------------------ cavities
def enclosed_holes(L, h, w):
    """Components of non-lock cells inside bbox(L) that cannot reach the bbox margin (4-leak)."""
    r0, c0, r1, c1 = bbox(L); s = set(L)
    R0, C0, R1, C1 = r0 - 1, c0 - 1, r1 + 1, c1 + 1
    st = [(y, x) for y in range(R0, R1 + 1) for x in (C0, C1)] + [(y, x) for x in range(C0, C1 + 1) for y in (R0, R1)]
    seen = set(st)
    while st:
        y, x = st.pop()
        for dy, dx in N4:
            q = (y + dy, x + dx)
            if R0 <= q[0] <= R1 and C0 <= q[1] <= C1 and q not in seen and q not in s:
                seen.add(q); st.append(q)
    return comps4([(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s and (y, x) not in seen])


def notches(L):
    r0, c0, r1, c1 = bbox(L); s = set(L)
    return comps4([(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s])


# ------------------------------------------------------------------ fitting
class Scene:
    """Bodies split into locks, keys and debris; cavities of the locks."""
    def __init__(self, g, bg, seg, lock, cav, stat, pairside, debris=frozenset()):
        self.g = g; self.bg = bg; self.h, self.w = H(g), W(g); self.cav_kind = cav
        bodies = segment(g, bg, seg)
        self.debris = [b for b in bodies if {g[y][x] for y, x in b} <= debris]
        bodies = [b for b in bodies if not {g[y][x] for y, x in b} <= debris]
        self.ok = True; self.fixed = []
        locks, keys = [], []
        if lock == 'pairs':
            byc = {}
            for b in bodies:
                cs = {g[y][x] for y, x in b}
                if len(cs) == 1: byc.setdefault(cs.pop(), []).append(b)
            for c, bs in byc.items():
                if len(bs) != 2: continue
                a, b = bs
                if pairside == 'smaller':
                    if len(a) == len(b): self.ok = False; return
                    key, lk = (a, b) if len(a) < len(b) else (b, a)
                else:
                    ya = sum(y for y, _ in a) / len(a); xa = sum(x for _, x in a) / len(a)
                    yb = sum(y for y, _ in b) / len(b); xb = sum(x for _, x in b) / len(b)
                    va = {'left': xa, 'right': -xa, 'top': ya, 'bottom': -ya}[pairside]
                    vb = {'left': xb, 'right': -xb, 'top': yb, 'bottom': -yb}[pairside]
                    if va == vb: self.ok = False; return
                    key, lk = (a, b) if va < vb else (b, a)
                locks.append(lk); keys.append(key)
            if not locks: self.ok = False; return
            ids = {id(x) for x in locks + keys}
            self.fixed = [c for b in bodies if id(b) not in ids for c in b]
        elif lock == 'holed':
            for b in bodies:
                hs = [hc for hc in enclosed_holes(b, self.h, self.w) if all(g[y][x] == bg for y, x in hc)]
                (locks if hs else keys).append(b)
        elif lock == 'largest':
            m = max((len(b) for b in bodies), default=0)
            big = [b for b in bodies if len(b) == m]
            if len(big) != 1: self.ok = False; return
            locks = big; keys = [b for b in bodies if b is not big[0]]
        elif lock == 'static':
            for b in bodies:
                (locks if all(g[y][x] in stat for y, x in b) else keys).append(b)
        if not locks or not keys or len(keys) > 24: self.ok = False; return
        self.locks, self.keys = locks, keys
        self.occ = set(c for b in locks for c in b) | set(self.fixed)
        self._cav = None

    def cavities(self):
        """[(lock index, cells)] of all-background cavities."""
        if self._cav is None:
            g, bg = self.g, self.bg; out = []
            for i, L in enumerate(self.locks):
                cs = enclosed_holes(L, self.h, self.w) if self.cav_kind == 'hole' else notches(L)
                out += [(i, C) for C in cs if all(g[y][x] == bg for y, x in C)]
            self._cav = out
        return self._cav

    def units(self, fit):
        return [(i, None) for i in range(len(self.locks))] if fit.startswith('rect') else self.cavities()


def fill_options(S, key_id, unit, group, mode):
    """Placements of one key into one unit: 'fill' = the key covers the whole cavity and protrudes only outside
    the lock bbox (exactly the cavity for enclosed holes); 'rect' = lock + key form a solid rectangle."""
    g = S.g; h, w = S.h, S.w; li, C = unit; L = S.locks[li]
    K = S.keys[key_id]; opts = []
    if mode == 'rect':
        r0, c0, r1, c1 = bbox(L); Ls = set(L); n = len(L) + len(K)
        for _, v in variants(K, g, group):
            shape = sorted((a, b) for a, b, _ in v)
            for hh in range(r1 - r0 + 1, n + 1):
                if n % hh: continue
                ww = n // hh
                if ww < c1 - c0 + 1: continue
                for R0 in range(r1 - hh + 1, r0 + 1):
                    for C0 in range(c1 - ww + 1, c0 + 1):
                        if R0 < 0 or C0 < 0 or R0 + hh > h or C0 + ww > w: continue
                        P = [(y, x) for y in range(R0, R0 + hh) for x in range(C0, C0 + ww) if (y, x) not in Ls]
                        if len(P) != len(shape): continue
                        my = min(y for y, _ in P); mx = min(x for _, x in P)
                        if sorted((y - my, x - mx) for y, x in P) != shape: continue
                        if any(p in S.occ for p in P): continue
                        opts.append([(a + my, b + mx, c) for a, b, c in v])
        return opts
    Cs = set(C); lb = bbox(L); c0 = min(C)
    for _, v in variants(K, g, group):
        if len(v) < len(C): continue
        for a, b, _ in v:
            ty, tx = c0[0] - a, c0[1] - b
            P = [(a2 + ty, b2 + tx, c) for a2, b2, c in v]
            cells = {(y, x) for y, x, _ in P}
            if not Cs <= cells: continue
            if any(not (0 <= y < h and 0 <= x < w) or (y, x) in S.occ for y, x in cells): continue
            extra = cells - Cs
            if (S.cav_kind == 'hole' or getattr(S, 'exact', False)) and extra: continue
            if any(lb[0] <= y <= lb[2] and lb[1] <= x <= lb[3] for y, x in extra): continue
            opts.append(P)
    return opts


def _disp(K, P):
    """Manhattan displacement of the centroid (least-action tie-break)."""
    ky = sum(y for y, _ in K) / len(K); kx = sum(x for _, x in K) / len(K)
    py = sum(y for y, _, _ in P) / len(P); px = sum(x for _, x, _ in P) / len(P)
    return abs(ky - py) + abs(kx - px)


def solve_match(S, group, fit):
    """One key per unit: maximum assignment of keys to units (cavities or locks); ties broken by least total
    displacement.  Returns (used key ids, placements) or None when empty / still ambiguous."""
    mode = 'rect' if fit.startswith('rect') else 'fill'
    congruent = fit == 'rect='
    opts = []
    for u in S.units(fit):
        o = []
        for k in range(len(S.keys)):
            for P in fill_options(S, k, u, group, mode):
                dims = None
                if congruent:
                    b = bbox([(y, x) for y, x, _ in P] + S.locks[u[0]]); dims = (b[2] - b[0], b[3] - b[1])
                o.append((k, P, _disp(S.keys[k], P), dims))
        if o: opts.append(o)
    if not opts or len(opts) > 16 or sum(len(o) for o in opts) > 200: return None
    best = {}; nodes = [0]
    def rec(i, used, occ, chosen, score, cost, dims):
        nodes[0] += 1
        if nodes[0] > 4000: raise OverflowError
        if i == len(opts):
            best.setdefault(score, []).append((cost, list(chosen))); return
        top = max(best) if best else -1
        if score + (len(opts) - i) < top: return
        for k, P, c, d in opts[i]:
            if k in used or (dims is not None and d != dims): continue
            cells = {(y, x) for y, x, _ in P}
            if cells & occ: continue
            chosen.append((k, P)); rec(i + 1, used | {k}, occ | cells, chosen, score + 1, cost + c, d); chosen.pop()
        rec(i + 1, used, occ, chosen, score, cost, dims)
    try:
        rec(0, frozenset(), set(), [], 0, 0.0, None)
    except OverflowError:
        return None
    if not best or max(best) <= 0: return None
    sols = best[max(best)]
    mc = min(c for c, _ in sols)
    sols = [ch for c, ch in sols if c <= mc + 1e-9]
    outs = {tuple(sorted(t for _, P in ch for t in P)) for ch in sols}
    if len(outs) != 1: return None
    return [k for k, _ in sols[0]], [P for _, P in sols[0]]


def solve_tile(S, group, cap=300, budget=8000):
    """All cavities are covered EXACTLY by keys lying inside them (several keys per cavity allowed): a global
    exact cover; identical keys are interchangeable.  The painted result must be unique."""
    cavs = S.cavities()
    if not cavs: return None
    cav_of = {}
    for j, (_, C) in enumerate(cavs):
        for c in C: cav_of[c] = j
    U = set(cav_of)
    types = {}
    for k, K in enumerate(S.keys):
        r0, c0, _, _ = bbox(K)
        sig = tuple(sorted((y - r0, x - c0, S.g[y][x]) for y, x in K))
        types.setdefault(sig, []).append(k)
    tlist = list(types.items())
    cand = []                                  # (type index, frozenset cells, P)
    by_cell = {c: [] for c in U}
    for ti, (sig, ks) in enumerate(tlist):
        seen = set()
        for _, v in variants(S.keys[ks[0]], S.g, group):
            a, b, _ = v[0]
            for u in U:
                ty, tx = u[0] - a, u[1] - b
                P = tuple((a2 + ty, b2 + tx, c) for a2, b2, c in v)
                cells = frozenset((y, x) for y, x, _ in P)
                if cells in seen or not cells <= U or len({cav_of[c] for c in cells}) != 1: continue
                seen.add(cells); ci = len(cand); cand.append((ti, cells, P))
                for c in cells: by_cell[c].append(ci)
    left = {ti: len(ks) for ti, (_, ks) in enumerate(tlist)}
    sigs = set(); first = [None]; nodes = [0]; nsol = [0]
    class Stop(Exception): pass
    def rec(rem, chosen):
        nodes[0] += 1
        if nodes[0] > budget: raise Stop
        if not rem:
            s = tuple(sorted(t for ci in chosen for t in cand[ci][2]))
            sigs.add(s); nsol[0] += 1
            if first[0] is None: first[0] = list(chosen)
            if len(sigs) > 1 or nsol[0] >= cap: raise Stop
            return
        best = None
        for c in rem:
            opts = [ci for ci in by_cell[c] if left[cand[ci][0]] > 0 and cand[ci][1] <= rem]
            if best is None or len(opts) < len(best[1]): best = (c, opts)
            if not opts: return
        for ci in best[1]:
            ti, cells, _ = cand[ci]
            left[ti] -= 1; chosen.append(ci)
            rec(rem - cells, chosen)
            chosen.pop(); left[ti] += 1
    try:
        rec(frozenset(U), [])
    except Stop:
        if len(sigs) != 1 or nsol[0] < cap: return None
    if len(sigs) != 1: return None
    used = []; placed = []
    pools = {ti: list(ks) for ti, (_, ks) in enumerate(tlist)}
    for ci in first[0]:
        ti, _, P = cand[ci]
        used.append(pools[ti].pop()); placed.append(list(P))
    return used, placed


def solve(S, group, fit):
    return solve_tile(S, group) if fit == 'tile' else solve_match(S, group, fit)


def lk_scene(g, seg, lock, cav, stat, side, debris=frozenset()):
    S = Scene(g, bg_of(g), seg, lock, cav, stat, side, debris)
    return S if S.ok else None


def lk_render(S, sol, rest, rc, debris):
    g, bg = S.g, S.bg
    used, placed = sol
    out = [row[:] for row in g]
    for i, K in enumerate(S.keys):
        if i in used or rest == 'erase':
            for y, x in K: out[y][x] = bg
    for K in S.debris:
        for y, x in K: out[y][x] = bg
    for P in placed:
        for y, x, c in P:
            out[y][x] = rc if rc is not None else c
    return out


def lockkey(g, seg, lock, cav, fit, group, rest, rc, stat, debris, side):
    S = lk_scene(g, seg, lock, cav, stat, side, debris if rc is None else frozenset())
    if S is None: return None
    sol = solve(S, group, fit)
    return lk_render(S, sol, rest, rc, debris) if sol else None


LOCK_CAV = (('holed', 'hole'), ('largest', 'notch'), ('static', 'notch'), ('pairs', 'notch'))
FITS = (('fill', 'T'), ('rect=', 'T'), ('rect', 'T'), ('tile', 'T'), ('fill', 'R4'), ('rect=', 'R4'), ('rect', 'R4'),
        ('fill', 'D4'), ('rect=', 'D4'), ('rect', 'D4'))


def n_fg(g):
    bg = bg_of(g)
    return sum(v != bg for r in g for v in r)


def fam_mech_lockkey(train):
    if not same_shape(train):
        yield from fam_mech_lockkey_panels(train)
        yield from fam_mech_lockkey_crop(train)
        return
    i0, o0 = train[0]['input'], train[0]['output']
    if i0 == o0: return
    # conservation: keys only move (or vanish as debris / leftovers), so the mass cannot grow
    if any(n_fg(p['output']) > n_fg(p['input']) or bg_of(p['output']) != bg_of(p['input']) for p in train): return
    stat = static_colours(train)
    # debris: colours present in every input and absent from the paired output (dust that is swept away)
    debris = set.intersection(*[{v for r in p['input'] for v in r} - {v for r in p['output'] for v in r} for p in train])
    newc = set.intersection(*[{v for r in p['output'] for v in r} - {v for r in p['input'] for v in r} for p in train])
    rcs = [None] + ([next(iter(newc))] if len(newc) == 1 else [])
    n = 0
    for seg, (lock, cav) in product(('c4', 'c8', 'm4', 'm8'), LOCK_CAV):
        if lock == 'static' and not stat: continue
        for side in (('smaller', 'left', 'right', 'top', 'bottom') if lock == 'pairs' else (None,)):
            for rc in rcs:
              S0 = lk_scene(i0, seg, lock, cav, stat, side, debris if rc is None else frozenset())
              if S0 is None: continue
              for fit, group in FITS:
                if fit == 'tile' and cav != 'hole': continue
                sol0 = solve(S0, group, fit)
                if not sol0: continue
                for rest in ('keep', 'erase'):
                    if lk_render(S0, sol0, rest, rc, debris) != o0: continue
                    args = (seg, lock, cav, fit, group, rest, rc, stat, debris, side)
                    ok = True
                    for p in train[1:]:
                        try:
                            r = lockkey(p['input'], *args)
                        except Exception:
                            r = None
                        if r != p['output']: ok = False; break
                    if not ok: continue
                    name = f"mech:lockkey[{seg},{lock},{fit},{group},{rest}{',' + side if side else ''}{',rc' if rc is not None else ''}]"
                    yield (name, 4, lambda g, a=args: lockkey(g, *a))
                    n += 1
                    if n >= 3: return


def lockkey_panels(g, li):
    """Two panels: the key panel is inserted into the lock panel when it is the exact complement of the lock
    inside the panel frame (in place); otherwise the lock panel is returned unchanged."""
    sp = split_panels(g)
    if not sp: return None
    ps, sc = sp
    if len(ps) != 2: return None
    L, K = ps[li], ps[1 - li]
    bg = bg_of(g)
    if bg == sc: return None
    lf = [[v != bg for v in r] for r in L]; kf = [[v != bg for v in r] for r in K]
    if not any(any(r) for r in lf) or not any(any(r) for r in kf): return None
    fits = all(a != b for ra, rb in zip(lf, kf) for a, b in zip(ra, rb))
    return [[(K[y][x] if kf[y][x] else L[y][x]) if fits else L[y][x] for x in range(W(L))] for y in range(H(L))]


def lockkey_crop(g, seg, cav, fit, group, rc, debris):
    """Scope 'crop': among the bodies that have empty cavities, the unique one whose cavities can ALL be filled by
    the other bodies (keys) is the lock; the keys are inserted and the output is the lock's bounding box."""
    bg = bg_of(g); h, w = H(g), W(g)
    bodies = segment(g, bg, seg)
    bodies = [b for b in bodies if not {g[y][x] for y, x in b} <= debris]
    if len(bodies) < 2 or len(bodies) > 16: return None
    hits = []
    for L in bodies:
        cs = enclosed_holes(L, h, w) if cav == 'hole' else notches(L)
        cs = [C for C in cs if all(g[y][x] == bg for y, x in C)]
        if not cs: continue
        S = Scene.__new__(Scene)
        S.g, S.bg, S.h, S.w, S.cav_kind = g, bg, h, w, cav
        S.locks = [L]; S.keys = [b for b in bodies if b is not L]; S.fixed = []; S.debris = []
        S.occ = set(L); S._cav = [(0, C) for C in cs]; S.ok = True; S.exact = True
        sol = solve(S, group, fit)
        if not sol: continue
        filled = {c for P in sol[1] for y, x, _ in P for c in [(y, x)]}
        if not all(set(C) <= filled for _, C in S._cav): continue
        hits.append((S, sol))
        if len(hits) > 1: return None
    if len(hits) != 1: return None
    S, sol = hits[0]
    out = lk_render(S, sol, 'keep', rc, debris)
    return crop_cells(out, S.locks[0])


def crop_cells(g, cells):
    r0, c0, r1, c1 = bbox(cells)
    return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]


def fam_mech_lockkey_crop(train):
    """(scope of the lock-and-key law: the output is the completed lock)"""
    i0, o0 = train[0]['input'], train[0]['output']
    if H(o0) > H(i0) or W(o0) > W(i0): return
    # debris: colours present in every input and absent from the paired output (dust that is swept away)
    debris = set.intersection(*[{v for r in p['input'] for v in r} - {v for r in p['output'] for v in r} for p in train])
    for seg, cav, (fit, group) in product(('c4', 'c8', 'm4', 'm8'), ('hole', 'notch'), (('fill', 'T'), ('fill', 'D4'), ('tile', 'T'))):
        if fit == 'tile' and cav != 'hole': continue
        try:
            r0 = lockkey_crop(i0, seg, cav, fit, group, None, debris)
        except Exception:
            r0 = None
        if r0 != o0: continue
        ok = True
        for p in train[1:]:
            try:
                r = lockkey_crop(p['input'], seg, cav, fit, group, None, debris)
            except Exception:
                r = None
            if r != p['output']: ok = False; break
        if ok:
            yield (f'mech:lockkey-crop[{seg},{cav},{fit},{group}]', 4,
                   lambda g, a=(seg, cav, fit, group, None, debris): lockkey_crop(g, *a))
            return


def fam_mech_lockkey_panels(train):
    """(scope of the lock-and-key law: panels)"""
    for li in (0, 1):
        if all(lockkey_panels(p['input'], li) == p['output'] for p in train):
            yield (f'mech:lockkey-panels[lock={li}]', 4, lambda g, li=li: lockkey_panels(g, li))
            return


# ================================================================== L3/L4: fields, support and piling
DIRS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}


def full_lines(g, bg):
    """Cells of full-length single-colour rows / columns (walls, floors, border lines)."""
    h, w = H(g), W(g); cells = set()
    for y in range(h):
        if g[y][0] != bg and all(v == g[y][0] for v in g[y]): cells |= {(y, x) for x in range(w)}
    for x in range(w):
        if g[0][x] != bg and all(g[y][x] == g[0][x] for y in range(h)): cells |= {(y, x) for y in range(h)}
    return cells


def anchor_cells(g, bg, mode, stat):
    if mode == 'colour':
        return {(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] != bg and g[y][x] in stat}
    if mode == 'lines':
        return full_lines(g, bg)
    return set()


def movers_of(g, bg, anchors, seg, unit):
    """Movable units: bodies of non-anchor cells, split into single cells or strips when asked."""
    sub = [[v if (y, x) not in anchors else bg for x, v in enumerate(r)] for y, r in enumerate(g)]
    bodies = segment(sub, bg, seg)
    if unit == 'body': return bodies
    if unit == 'cell': return [[c] for b in bodies for c in b]
    out = []                                   # strips: split each body into columns ('v') or rows ('h')
    for b in bodies:
        groups = {}
        for y, x in b: groups.setdefault(x if unit == 'vstrip' else y, []).append((y, x))
        for cells in groups.values():          # a strip may itself be split by gaps
            out += comps4(cells) if len(cells) > 1 else [cells]
    return out


def body_key(cells, g, bg, key):
    if key == 'colour':
        cs = sorted({g[y][x] for y, x in cells})
        return cs[0] if len(cs) == 1 else tuple(cs)
    if key == 'topo':
        if enclosed_holes(cells, H(g), W(g)): return 'holed'
        r0, c0, r1, c1 = bbox(cells)
        return 'open' if len(cells) < (r1 - r0 + 1) * (c1 - c0 + 1) else 'solid'
    if key == 'thick':
        s = set(cells)
        return any((y + 1, x) in s and (y, x + 1) in s and (y + 1, x + 1) in s for y, x in cells)
    if key == 'size':
        return len(cells)
    if key.startswith('mass>'):
        return len(cells) > int(key[5:])
    return None


def attract_dir(cells, g, anchors_bodies, same_colour, near=False):
    """Direction toward the nearest anchor body whose row / column span overlaps the mover (None: none / tie).
    near: the attractor is the nearest anchor overall (Chebyshev gap between boxes); the mover moves only when
    that attractor is aligned with it (a unique nearest)."""
    r0, c0, r1, c1 = bbox(cells)
    col = {g[y][x] for y, x in cells}
    if near:
        gaps = []
        for A, ac in anchors_bodies:
            a0, b0, a1, b1 = bbox(A)
            dy = max(0, a0 - r1 - 1, r0 - a1 - 1); dx = max(0, b0 - c1 - 1, c0 - b1 - 1)
            gaps.append((max(dy, dx), A, ac))
        if not gaps: return None
        m = min(x[0] for x in gaps); near_ab = [(A, ac) for d, A, ac in gaps if d == m]
        if len(near_ab) != 1: return None
        d = attract_dir(cells, g, near_ab, same_colour)
        return None if d == 'tie' else d
    best = []
    for A, ac in anchors_bodies:
        if same_colour and not (len(col) == 1 and ac == next(iter(col))): continue
        a0, b0, a1, b1 = bbox(A)
        if b0 <= c1 and c0 <= b1:
            if a0 > r1: best.append((a0 - r1, 'down'))
            elif a1 < r0: best.append((r0 - a1, 'up'))
        if a0 <= r1 and r0 <= a1:
            if b0 > c1: best.append((b0 - c1, 'right'))
            elif b1 < c0: best.append((c0 - b1, 'left'))
    if not best: return None
    m = min(d for d, _ in best); ds = {d for dd, d in best if dd == m}
    return ds.pop() if len(ds) == 1 else 'tie'


def settle_positions(g, bg, movers, dirs):
    """L4: lift every driven mover, then settle each field layer (one direction) leading edge first; a mover stops
    when its next step would leave the grid or enter an occupied cell (anchors, resting bodies, bodies already
    settled in its layer).  Layers are independent.  Returns (landing cells per mover, settling order)."""
    h, w = H(g), W(g)
    if 'tie' in dirs: return None
    lifted = {c for m, d in zip(movers, dirs) if d is not None for c in m}
    base = {(y, x) for y in range(h) for x in range(w) if g[y][x] != bg and (y, x) not in lifted}
    lands = [list(m) for m in movers]; order = []
    layers = {}
    for i, d in enumerate(dirs):
        if d is not None: layers.setdefault(d, []).append(i)
    for d, ids in layers.items():
        dy, dx = DIRS[d]; occ = set(base)
        ids.sort(key=lambda i: -max(y * dy + x * dx for y, x in movers[i]))
        for i in ids:
            cells = movers[i]; k = 0
            while True:
                nxt = [(y + (k + 1) * dy, x + (k + 1) * dx) for y, x in cells]
                if any(not (0 <= y < h and 0 <= x < w) or (y, x) in occ for y, x in nxt): break
                k += 1
            lands[i] = [(y + k * dy, x + k * dx) for y, x in cells]
            occ |= set(lands[i]); order.append(i)
    return lands, order


def settle(g, bg, movers, dirs, recol=None, absorb=False):
    """Paint the equilibrium: moved bodies keep their colours, take a class colour (recol), or take the colour of
    what they come to rest against (absorb).  Overlaps between layers are inconsistent (None)."""
    r = settle_positions(g, bg, movers, dirs)
    if r is None: return None
    lands, order = r
    h, w = H(g), W(g)
    out = [row[:] for row in g]
    for m, d in zip(movers, dirs):
        if d is not None:
            for y, x in m: out[y][x] = bg
    final = {}; layer = {}                     # layer: colours of bodies already settled in the same field layer
    for n, i in enumerate(order):
        if n and dirs[order[n - 1]] != dirs[i]: layer = {}
        new = lands[i]
        if absorb:
            dy, dx = DIRS[dirs[i]]; s = set(new)
            hit = [(y + dy, x + dx) for y, x in new if (y + dy, x + dx) not in s]
            hc = Counter(layer.get(q, out[q[0]][q[1]]) for q in hit
                         if 0 <= q[0] < h and 0 <= q[1] < w and (q in layer or out[q[0]][q[1]] != bg))
            if not hc: return None
            c_abs = hc.most_common(1)[0][0]
        for (y0, x0), q in zip(movers[i], new):
            c = c_abs if absorb else (recol[i] if recol and recol[i] is not None else g[y0][x0])
            if q in final and final[q] != c: return None
            final[q] = c; layer[q] = c
    for (y, x), c in final.items():
        if out[y][x] != bg: return None
        out[y][x] = c
    return out


def consistent_dirs(cells, g, o, bg, strict):
    """Directions d (and 'stay') such that the mover's footprint shifted k>=0 steps along d lies on cells of the
    output that carry its colours (strict) or any non-background colour (recolouring allowed)."""
    h, w = H(g), W(g); res = set()
    if all((o[y][x] == g[y][x]) if strict else (o[y][x] != bg) for y, x in cells): res.add(None)
    for d, (dy, dx) in DIRS.items():
        for k in range(1, max(h, w)):
            ok = True; inside = True
            for y, x in cells:
                yy, xx = y + k * dy, x + k * dx
                if not (0 <= yy < h and 0 <= xx < w): inside = False; break
                if (o[yy][xx] != g[y][x]) if strict else (o[yy][xx] == bg): ok = False; break
            if not inside: break
            if ok: res.add(d); break
    return res


FIELD_KEYS = ('colour', 'topo', 'thick')


def grav_program(g, P):
    """Apply one gravity program P (dict) to grid g."""
    bg = bg_of(g)
    anchors = anchor_cells(g, bg, P['anchor'], P['stat'])
    unit = P['unit']
    movers = movers_of(g, bg, anchors, P['seg'], unit)
    if not movers or len(movers) > 120: return None
    dirs = grav_dirs(g, bg, P, movers)
    if dirs is None: return None
    recol = None
    if P['recol'] == 'table':
        recol = []
        for m in movers:
            v = body_key(m, g, bg, P['rkey'])
            if v not in P['rtable']: return None
            recol.append(P['rtable'][v])
    return settle(g, bg, movers, dirs, recol, P['recol'] == 'absorb')


def stack_panels(g, d, order):
    """Sequential drop (tower building, L4 in time order): the contents of the panels enter one panel frame one
    after the other from the side opposite to d, keep their lateral position and slide along d until they rest
    on the frame or on earlier pieces."""
    sp = split_panels(g)
    if not sp: return None
    ps, sc = sp
    if len(ps) < 2: return None
    bg = bg_of(g)
    if bg == sc: return None
    h, w = H(ps[0]), W(ps[0]); dy, dx = DIRS[d]
    canvas = [[bg] * w for _ in range(h)]
    seq = ps if order == 'fwd' else ps[::-1]
    for P in seq:
        cells = [(y, x, P[y][x]) for y in range(h) for x in range(w) if P[y][x] != bg]
        if not cells: continue
        # start fully outside the frame on the side opposite to d
        k0 = max(h, w) + 1
        cur = [(y - dy * k0, x - dx * k0, c) for y, x, c in cells]
        def free(cs):
            return all(not (0 <= y < h and 0 <= x < w) or canvas[y][x] == bg for y, x, _ in cs) and \
                   all((y < h if dy > 0 else y >= 0 if dy < 0 else True) and (x < w if dx > 0 else x >= 0 if dx < 0 else True)
                       for y, x, _ in cs)
        while True:
            nxt = [(y + dy, x + dx, c) for y, x, c in cur]
            if not free(nxt): break
            cur = nxt
        if any(not (0 <= y < h and 0 <= x < w) for y, x, _ in cur): return None
        for y, x, c in cur: canvas[y][x] = c
    return canvas


def fam_mech_stack_panels(train):
    """(scope of the support law: pieces from panels dropped in sequence into one frame)"""
    for d, order in product(('down', 'up', 'left', 'right'), ('fwd', 'rev')):
        ok = True
        for p in train:
            try:
                r = stack_panels(p['input'], d, order)
            except Exception:
                r = None
            if r != p['output']: ok = False; break
        if ok:
            yield (f'mech:gravity[stack-panels,{d},{order}]', 4, lambda g, a=(d, order): stack_panels(g, *a))
            return


FIELD_COST = {'const': 0, 'attract': 1, 'attract-colour': 1, 'attract-near': 2, 'table': 2}


def fam_mech_gravity(train):
    if not same_shape(train):
        yield from fam_mech_stack_panels(train)
        return
    if all(p['input'] == p['output'] for p in train): return
    # conservation of mass: bodies only move (recolouring allowed)
    if any(n_fg(p['output']) != n_fg(p['input']) or bg_of(p['output']) != bg_of(p['input']) for p in train): return
    import time
    t0 = time.time()
    stat = static_colours(train)
    anchors_modes = ['none'] + (['colour'] if stat else [])
    if all(full_lines(p['input'], bg_of(p['input'])) for p in train): anchors_modes.append('lines')
    found = []; seen = set()
    for amode, seg, unit in product(anchors_modes, ('c4', 'c8', 'm4', 'm8'), ('body', 'cell', 'vstrip', 'hstrip')):
        if unit in ('cell', 'vstrip', 'hstrip') and seg[1] == '8': continue
        if len(found) >= 12: break  # deterministic: wall-clock budget removed (Kaggle parity)
        info = []
        for p in train:
            g, o = p['input'], p['output']; bg = bg_of(g)
            A = anchor_cells(g, bg, amode, stat)
            if amode != 'none' and not A: info = None; break
            ms = movers_of(g, bg, A, seg, unit)
            if not ms or len(ms) > 120: info = None; break
            info.append((g, o, bg, ms))
        if info is None: continue
        sig = tuple(tuple(sorted(tuple(sorted(m)) for m in ms)) for _, _, _, ms in info) + (amode,)
        if sig in seen: continue
        seen.add(sig)
        for recol in ('none', 'absorb', 'table'):
            strict = recol == 'none'
            cons = [[consistent_dirs(m, g, o, bg, strict) for m in ms] for g, o, bg, ms in info]
            if any(not c for cs in cons for c in cs): continue
            fields = []
            for d in DIRS:
                if all(d in c or None in c for cs in cons for c in cs): fields.append((d, None, 'const'))
            if amode != 'none':
                fields += [('attract', None, 'attract'), ('attract-colour', None, 'attract-colour'),
                           ('attract-near', None, 'attract-near')]
            sizes = sorted({len(m) for _, _, _, ms in info for m in ms})
            keys = list(FIELD_KEYS) + [f'mass>{k}' for k in sizes[:-1]][:6]
            for key in keys:
                tab = {}; ok = True
                for (g, o, bg, ms), cs in zip(info, cons):
                    for m, c in zip(ms, cs):
                        v = body_key(m, g, bg, key)
                        tab[v] = tab.get(v, c) & c
                        if not tab[v]: ok = False; break
                    if not ok: break
                if not ok or len(tab) < 2 or len(tab) > 4: continue
                opts = [sorted(tab[v], key=lambda d: (d is not None, str(d))) for v in tab]
                for combo in product(*opts):
                    if len(set(combo)) < 2: continue
                    fields.append(('table:' + key, dict(zip(tab, combo)), 'table'))
            for fname, table, fkind in fields[:40]:
                P = {'anchor': amode, 'stat': stat, 'seg': seg, 'unit': unit, 'field': fname, 'table': table,
                     'recol': recol}
                if recol == 'table':
                    P = induce_recolour(P, train)
                    if P is None: continue
                ok = True
                for p in train:
                    try:
                        r = grav_program(p['input'], P)
                    except Exception:
                        r = None
                    if r != p['output']: ok = False; break
                if not ok: continue
                tname = '' if table is None else ':' + ','.join(f'{k}>{v}' for k, v in sorted(table.items(), key=str))
                rname = '' if recol == 'none' else (f',recol={P.get("rkey")}' if recol == 'table' else ',absorb')
                score = (FIELD_COST[fkind] + (1 if fname.startswith('table:mass') else 0)
                         + {'none': 0, 'absorb': 1, 'table': 2}[recol] + (unit.endswith('strip')) + (seg[0] == 'm')
                         + (amode == 'lines'))
                found.append((score, len(found), f'mech:gravity[{amode},{seg},{unit},{fname}{tname}{rname}]', dict(P)))
    found.sort(key=lambda x: (x[0], x[1]))
    for score, _, name, P in found[:3]:
        yield (name, 4, lambda g, P=P: grav_program(g, P))


def induce_recolour(P, train):
    """Find a key such that the landing colour of every mover is a function of it (moved units only)."""
    for rkey in ('colour', 'topo', 'thick'):
        Q = dict(P); Q['recol'] = 'none'
        rt = {}; ok = True; changed = False
        for p in train:
            g, o = p['input'], p['output']; bg = bg_of(g)
            anchors = anchor_cells(g, bg, Q['anchor'], Q['stat'])
            movers = movers_of(g, bg, anchors, Q['seg'], Q['unit'])
            # simulate without recolour to know where each mover lands
            r = grav_program_trace(g, Q, movers)
            if r is None: ok = False; break
            for m, land in zip(movers, r):
                cs = {o[y][x] for y, x in land}
                if len(cs) != 1: ok = False; break
                v = body_key(m, g, bg, rkey); c = cs.pop()
                if rt.setdefault(v, c) != c: ok = False; break
                if any(g[y][x] != c for y, x in m): changed = True
            if not ok: break
        if ok and rt and changed:
            Q['recol'] = 'table'; Q['rkey'] = rkey; Q['rtable'] = rt
            return Q
    return None


def grav_dirs(g, bg, P, movers):
    f = P['field']
    if f in DIRS:
        return [f] * len(movers)
    if f.startswith('table:'):
        dirs = []
        for m in movers:
            v = body_key(m, g, bg, f[6:])
            if v not in P['table']: return None
            dirs.append(P['table'][v])
        return dirs
    anchors = anchor_cells(g, bg, P['anchor'], P['stat'])      # attract / attract-colour / attract-near
    ab = [(A, g[A[0][0]][A[0][1]]) for A in segment([[v if (y, x) in anchors else bg for x, v in enumerate(r)]
                                                      for y, r in enumerate(g)], bg, 'c4')]
    if not ab: return None
    return [attract_dir(m, g, ab, f == 'attract-colour', f == 'attract-near') for m in movers]


def grav_program_trace(g, P, movers):
    """Landing cells of every mover under program P (no recolouring)."""
    bg = bg_of(g)
    dirs = grav_dirs(g, bg, P, movers)
    if dirs is None: return None
    r = settle_positions(g, bg, movers, dirs)
    return r[0] if r else None


# ================================================================== L6: levers (rotation about a pivot)
ROT = {'cw': lambda dy, dx: (dx, -dy), 'ccw': lambda dy, dx: (-dx, dy)}


def lever_arms(g, bg, p, a, mode):
    """[(pivot cell, [arm cells])].  attached: straight runs of arm colour leaving the pivot whose far end is free
    (next cell background or outside); aligned: arm-colour cells matched one-to-one to pivots on their row /
    column with a clear line of sight, minimum total distance (the matching must be unique)."""
    h, w = H(g), W(g)
    piv = [(y, x) for y in range(h) for x in range(w) if g[y][x] == p]
    out = []
    if mode == 'attached':
        for py, px in piv:
            for dy, dx in N4:
                run = []; y, x = py + dy, px + dx
                while 0 <= y < h and 0 <= x < w and g[y][x] == a:
                    run.append((y, x)); y += dy; x += dx
                if not run: continue
                if 0 <= y < h and 0 <= x < w and g[y][x] != bg: continue
                out.append(((py, px), run))
        used = [c for _, r in out for c in r]
        if len(used) != len(set(used)): return None
        return out
    arms = [(y, x) for y in range(h) for x in range(w) if g[y][x] == a]
    if not arms or len(arms) > len(piv) or len(arms) > 12: return None
    cand = []
    for q in arms:
        cs = []
        for pv in piv:
            if (pv[0] == q[0] or pv[1] == q[1]) and all(g[yy][xx] == bg for yy, xx in _between(pv, q)):
                cs.append((abs(pv[0] - q[0]) + abs(pv[1] - q[1]), pv))
        if not cs: return None
        cand.append(sorted(cs))
    best = [None, []]
    def rec(i, used, cost, ch):
        if best[0] is not None and cost > best[0]: return
        if i == len(arms):
            if best[0] is None or cost < best[0]: best[0] = cost; best[1] = [list(ch)]
            elif cost == best[0]: best[1].append(list(ch))
            return
        for d, pv in cand[i]:
            if pv in used: continue
            ch.append(pv); rec(i + 1, used | {pv}, cost + d, ch); ch.pop()
    rec(0, frozenset(), 0, [])
    if len(best[1]) != 1: return None
    return [(pv, [q]) for pv, q in zip(best[1][0], arms)]


def _between(a, b):
    (y0, x0), (y1, x1) = a, b
    if y0 == y1: return [(y0, x) for x in range(min(x0, x1) + 1, max(x0, x1))]
    return [(y, x0) for y in range(min(y0, y1) + 1, max(y0, y1))]


def _room(g, bg, py, px, dy, dx):
    h, w = H(g), W(g); k = 0; y, x = py + dy, px + dx
    while 0 <= y < h and 0 <= x < w and g[y][x] == bg:
        k += 1; y += dy; x += dx
    return k


def lever(g, p, a, mode, rot, new_c, old_c):
    """L7: every arm swings 90 degrees about its pivot.  rot = cw | ccw (fixed sense, clipped by the frame)
    | room (it falls to the perpendicular side with more free room; that side must hold the whole arm)."""
    bg = bg_of(g); h, w = H(g), W(g)
    arms = lever_arms(g, bg, p, a, mode)
    if not arms: return None
    out = [r[:] for r in g]
    taken = set()
    for (py, px), cells in arms:
        for y, x in cells: out[y][x] = bg if old_c is None else old_c
    for (py, px), cells in arms:
        if rot in ROT:
            f = ROT[rot]
        else:
            s = lambda v: (v > 0) - (v < 0)
            dy0, dx0 = s(cells[0][0] - py), s(cells[0][1] - px)
            r1 = _room(g, bg, py, px, *ROT['cw'](dy0, dx0)); r2 = _room(g, bg, py, px, *ROT['ccw'](dy0, dx0))
            if r1 == r2: return None
            f = ROT['cw'] if r1 > r2 else ROT['ccw']
            if max(r1, r2) < max(abs(y - py) + abs(x - px) for y, x in cells): return None
        new = [(py + f(y - py, x - px)[0], px + f(y - py, x - px)[1]) for y, x in cells]
        for (y, x), (y0, x0) in zip(new, cells):
            if 0 <= y < h and 0 <= x < w:
                if (y, x) in taken: return None
                out[y][x] = g[y0][x0] if new_c is None else new_c
                taken.add((y, x))
    return out


def topple(g, trig, d):
    """Dominoes: vertical bars standing on the floor (bottom row).  Bars of the trigger colour fall toward d,
    rotating 90 degrees about their base and lying on the floor; a falling bar of height h knocks over every
    standing bar whose base lies within distance h (chain reaction)."""
    bg = bg_of(g); h, w = H(g), W(g)
    bars = {}
    for x in range(w):
        if g[h - 1][x] == bg: continue
        c = g[h - 1][x]; y = h - 1
        while y >= 0 and g[y][x] == c: y -= 1
        if y >= 0 and g[y][x] != bg: return None
        if any(g[yy][x] != bg for yy in range(y + 1)): return None
        bars[x] = (c, h - 1 - y)
    if not bars: return None
    for y in range(h):
        for x in range(w):
            if g[y][x] != bg and x not in bars: return None
    s = 1 if d == 'right' else -1
    fallen = set(); todo = [x for x, (c, _) in bars.items() if c == trig]
    if not todo: return None
    while todo:
        x = todo.pop()
        if x in fallen: continue
        fallen.add(x); hh = bars[x][1]
        for k in range(1, hh + 1):
            if x + s * k in bars and x + s * k not in fallen: todo.append(x + s * k)
    out = [r[:] for r in g]
    for x in fallen:
        for y in range(h): out[y][x] = bg
    for x in sorted(fallen, key=lambda x: s * x):
        c, hh = bars[x]
        for k in range(hh):
            xx = x + s * k
            if 0 <= xx < w: out[h - 1][xx] = c
    return out


# frames: which grid side is the floor (the grid is turned so that this side is at the bottom)
FRAMES = {'bottom': (lambda g: g, lambda g: g),
          'top': (lambda g: [r[::-1] for r in g[::-1]], lambda g: [r[::-1] for r in g[::-1]]),
          'left': (lambda g: [list(r) for r in zip(*g)][::-1], lambda g: [list(r) for r in zip(*g[::-1])]),
          'right': (lambda g: [list(r) for r in zip(*g[::-1])], lambda g: [list(r) for r in zip(*g)][::-1])}


def topple_framed(g, fr, trig, d):
    to, back = FRAMES[fr]
    r = topple(to(g), trig, d)
    return back(r) if r is not None else None


def fam_mech_lever(train):
    if not same_shape(train): return
    i0, o0 = train[0]['input'], train[0]['output']
    if i0 == o0: return
    bg0 = bg_of(i0)
    stat = static_colours(train)
    cin = set.intersection(*[{v for r in p['input'] for v in r} for p in train])
    changed = set()
    for p in train:
        for ri, ro in zip(p['input'], p['output']):
            for u, v in zip(ri, ro):
                if u != v: changed.add(u)
    newc = set.intersection(*[{v for r in p['output'] for v in r} - {v for r in p['input'] for v in r} for p in train])
    n = 0
    for fr, trig, d in product(FRAMES, sorted(changed - {bg0}), ('right', 'left')):
        if all(topple_framed(p['input'], fr, trig, d) == p['output'] for p in train):
            yield (f'mech:lever[topple,floor={fr},trigger{trig},{d}]', 4,
                   lambda g, a=(fr, trig, d): topple_framed(g, *a))
            return
    for pc in sorted(c for c in stat & cin if c != bg0):
        for ac in sorted(changed & cin - {bg0, pc}):
            olds = [None] + sorted(c for c in {v for r in o0 for v in r} if c not in (bg0, ac, pc))[:2]
            news = [None] + sorted(newc)[:1]
            for mode, rot, nc, oc in product(('attached', 'aligned'), ('cw', 'ccw', 'room'), news, olds):
                ok = True
                for p in train:
                    try:
                        r = lever(p['input'], pc, ac, mode, rot, nc, oc)
                    except Exception:
                        r = None
                    if r != p['output']: ok = False; break
                if not ok: continue
                rn = rot
                yield (f'mech:lever[pivot{pc},arm{ac},{mode},{rn}{",new" + str(nc) if nc is not None else ""}'
                       f'{",old" + str(oc) if oc is not None else ""}]', 4,
                       lambda g, a=(pc, ac, mode, rot, nc, oc): lever(g, *a))
                n += 1
                if n >= 2: return


FAMILIES = (fam_mech_lockkey, fam_mech_gravity, fam_mech_lever)


In [ ]:
%%writefile /kaggle/working/probe/prior_optics.py
"""OPTICS prior: occlusion, layering, amodal completion, projection and reflection of light in a flat scene.

Laws (the prior; every family below is a search over one of them)
----
A picture is a stack of opaque LAYERS (shapes, each of one colour) over a background, seen from above.
  L1 Painter's law      render(stack)[p] = colour of the front-most layer whose shape covers p, else bg.
  L2 Depth order        "A in front of B" is a strict partial order, inferred from overlaps only: A is in
                        front of B iff A is visible at a cell that B's (amodal) shape covers (or, for explicit
                        layers, iff A's colour wins where both are opaque in a training output).  It must be
                        acyclic; a comparison the evidence does not decide is never guessed: the program
                        refuses (fn -> None), so an under-determined scene cannot over-fire.
  L3 Amodal completion  the hidden part of a layer is the smallest admissible completion of its visible part
                        (Occam): diagonal / anti-diagonal segment, rectangle outline, solid rectangle (bbox);
                        irregular shapes only bridge row/column gaps (run closure).  Visible fragments of one
                        colour are one layer when their union completes (good continuation; rectangles need
                        aligned fragments).  Admissible = every hidden cell is covered by another colour:
                        nothing hides behind the background, and a hidden cell is never of the layer's colour.
  L4 Projection         light travels straight; an opaque body blocks it.  A body's image on a screen is the
                        sweep of its cells toward the screen, and a body nearer the screen occludes the images
                        of bodies behind it.
  L5 Reflection         a diagonal light ray bounces off opaque bodies like a mirror (the blocked velocity
                        component flips: angle of incidence = angle of reflection), passes transparent colours,
                        is absorbed or reflected by the grid walls.  Beams never interact, so two beams that
                        would meet (same cell in two colours, crossing between cells, hitting another source)
                        are outside the evidence and the program refuses.

Search space (every parameter induced from the training pairs; the harness verifies every program)
------------
  fam_overlay   optics:overlay[stack,order]           L1+L2 on explicit layers.
                  stack: equal panels (separator lattice, a x b grid with 0/1-wide uniform separators, the
                         four corner windows of a constant output size), equal-size windows (framed / holed
                         objects), fragments registered on a common anchor colour (the unique colour seen once
                         in every fragment), or the scene's amodal layers / colour classes slid into a common
                         corner or centre of a canvas as large as the largest;  transparent = the stack ground.
                  order: colour priority or layer-index priority induced as a partial order from training
                         overlaps (undecided test comparisons refuse), or size (smaller / larger in front).
  fam_scene     one scene parse (L3 layers + L2 depth order), three read-outs:
                  optics:depth[view]      depth order of the colour layers as a row / column (back->front or
                                          front->back) or the front-most / back-most colour (1x1); only for
                                          clean scenes (one completed layer per colour).
                  optics:disocclude[X]    occluder X (fixed colour absent from every output, or the rarest /
                                          most fragmented colour) removed; each hidden cell shows the front-most
                                          remaining layer covering it, else bg; whole grid or the hidden window.
                  optics:relayer[key]     the layers re-rendered with a new depth order: reversed, smaller in
                                          front, larger in front.
  fam_project   optics:project[screen,mode,paint]     L4.  screen = a fixed colour or the largest object;
                  each body cell shines along the unique axis direction in which the screen lies; mode: sweep
                  (paint the beam up to the first opaque cell), sweep-reach (only beams reaching the screen),
                  image (recolour the screen cell hit), land (paint the last free cell); paint = own / constant.
  fam_light     optics:light[emitter,transp,adopt,edge,paint]   L5.  emitters: L-triominoes (shine away from
                  the missing corner), diagonal segments (continue from free ends), grid-corner cells;
                  transparent colours (<= 2, passed and painted over); adopt the mirror colour after a bounce or
                  keep; walls absorb / reflect (all, left-right, top-bottom); paint own colour or a new colour.
Tested and dropped (0 tasks each): shadows cast away from a light source, x-ray reveal of hidden parts,
colour-filter transparency (colour of a layer seen through a translucent rectangle), cropping the front-most /
back-most completed layer, point sources shining in four diagonals.

Measured (eval_fam.py, 1000 training + 50 half-A tasks): 38 exact, WRONG 0 (4 of the 38 are the harness's own
identity+colour-map).  overlay 21 (281123b4 3d31c5b3 75b8110e e99362f0 ea9794b1 bc1d5164 25e02866 7c9b52a0
137eaa0f 20818e16 c8cbb738 6a11f6da 2dc579da a68b268e cf98881b e98196ab 0c9aba6e 506d28a5 ce4f8723 d19f7514
e133d23d); scene 6 (depth: 68bc2e87 1a2e2828; disocclude: b7955b3c 7e0986d6; relayer: 52df9849 ba97ae07);
project 4 (13713586 1f642eb9 2c608aff d43fd935); light 3 (7e2bad24 e179c5f4 508bd3b6).  Half-A 142ca369 fits
fam_light on all training pairs, but its test beams cross / hit other sources, so the program refuses.
"""
import sys
sys.path.insert(0, '/home/claude/work/widen')
from collections import Counter
from itertools import product
from gdsl import H, W, bg_of, objects, bbox, crop, split_panels

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))


# ------------------------------------------------------------------ partial orders (L2)
class Order:
    """Strict partial order from pairwise 'a in front of b' evidence, with transitive closure."""
    def __init__(self):
        self.front = {}                          # a -> set of items a is in front of

    def add(self, a, b):
        if a == b: return True
        self.front.setdefault(a, set()).add(b)
        return True

    def close(self):
        """Transitive closure; False if a cycle exists."""
        items = set(self.front) | {b for s in self.front.values() for b in s}
        clo = {a: set(self.front.get(a, ())) for a in items}
        changed = True
        while changed:
            changed = False
            for a in items:
                add = set()
                for b in clo[a]: add |= clo.get(b, set())
                if not add <= clo[a]:
                    clo[a] |= add; changed = True
        self.clo = clo
        return all(a not in clo[a] for a in items)

    def above(self, a, b):
        return b in self.clo.get(a, ())

    def top(self, items):
        """The unique item in front of all others (decided by evidence), else None."""
        items = list(items)
        if len(items) == 1: return items[0]
        for a in items:
            if all(b == a or self.above(a, b) for b in items): return a
        return None

    def chain(self, items):
        """Back-to-front total order of items if the evidence decides it completely, else None."""
        items = list(items)
        srt = sorted(items, key=lambda a: sum(self.above(a, b) for b in items))
        for i in range(len(srt) - 1):
            if not self.above(srt[i + 1], srt[i]): return None
        return srt


# ------------------------------------------------------------------ explicit layer stacks (L1)
def split_grid(g, a, b, s):
    """a x b equal panels separated by s-wide uniform lines (s in {0,1}); reading order."""
    h, w = H(g), W(g)
    if (h - (a - 1) * s) % a or (w - (b - 1) * s) % b: return None
    ph, pw = (h - (a - 1) * s) // a, (w - (b - 1) * s) // b
    if ph < 1 or pw < 1: return None
    if s:
        seps = {g[i * (ph + s) + ph][x] for i in range(a - 1) for x in range(w)}
        seps |= {g[y][j * (pw + s) + pw] for j in range(b - 1) for y in range(h)}
        if len(seps) != 1: return None
    return [[row[j * (pw + s): j * (pw + s) + pw] for row in g[i * (ph + s): i * (ph + s) + ph]]
            for i in range(a) for j in range(b)]


def split_windows(g, diag):
    """Equal-size windows: multicolour components of non-background, all with the same bbox size."""
    bg = bg_of(g)
    obs = objects(g, bg, diag, False)
    if len(obs) < 2 or len(obs) > 12: return None
    boxes = sorted(bbox(o) for o in obs)
    if len({(r1 - r0, c1 - c0) for r0, c0, r1, c1 in boxes}) != 1: return None
    if boxes[0][2] == boxes[0][0] and boxes[0][3] == boxes[0][1]: return None
    return [crop(g, b) for b in boxes]


def splitters(train):
    """(name, fn(g) -> list of equal layers) whose layers have the output's shape on every training pair."""
    i0, o0 = train[0]["input"], train[0]["output"]
    if H(o0) > H(i0) or W(o0) > W(i0) or (H(o0), W(o0)) == (H(i0), W(i0)): return []
    cands = [("sep", lambda g: (lambda sp: sp[0] if sp else None)(split_panels(g)))]
    for a, b in product(range(1, 5), range(1, 5)):
        if a * b < 2: continue
        for s in (0, 1):
            cands.append((f"grid{a}x{b}s{s}", lambda g, a=a, b=b, s=s: split_grid(g, a, b, s)))
    cands.append(("win4", lambda g: split_windows(g, False)))
    cands.append(("win8", lambda g: split_windows(g, True)))
    if len({(H(p["input"]), W(p["input"])) for p in train}) == 1 and len({(H(p["output"]), W(p["output"])) for p in train}) == 1:
        h, w = H(o0), W(o0)
        def corners(g, h=h, w=w):
            if H(g) < h or W(g) < w or (H(g), W(g)) != (H(i0), W(i0)): return None
            return [crop(g, (r, c, r + h - 1, c + w - 1)) for r in (0, H(g) - h) for c in (0, W(g) - w)]
        cands.append(("corners", corners))
    out = []
    for name, fn in cands:
        ok = True
        for p in train:
            try:
                L = fn(p["input"])
            except Exception:
                L = None
            if not L or len(L) < 2 or any((H(x), W(x)) != (H(p["output"]), W(p["output"])) for x in L):
                ok = False; break
        if ok: out.append((name, fn))
    return out


def ground(panels):
    """Transparent colour of a panel stack: the most common colour over all panel cells."""
    return Counter(v for L in panels for r in L for v in r).most_common(1)[0][0]


# A STACK is (layers, h, w, t): each layer a dict {(y, x): colour} of its opaque cells in canvas coordinates,
# a canvas of h x w cells and the transparent ground colour t.
def panel_stack(panels):
    t = ground(panels)
    return [{(y, x): v for y, r in enumerate(P) for x, v in enumerate(r) if v != t} for P in panels], H(panels[0]), W(panels[0]), t


def anchor_stack(g, diag):
    """Fragments (multicolour components) that each hold exactly one cell of a common anchor colour,
    registered on that cell (the anchor colour is the unique colour occurring once in every fragment)."""
    bg = bg_of(g)
    obs = objects(g, bg, diag, False)
    if len(obs) < 2 or len(obs) > 12: return None
    cand = None
    for ob in obs:
        cnt = Counter(g[y][x] for y, x in ob)
        ones = {c for c, n in cnt.items() if n == 1}
        cand = ones if cand is None else cand & ones
    if not cand or len(cand) != 1: return None
    a = cand.pop(); rel = []
    for ob in obs:
        ay, ax = next((y, x) for y, x in ob if g[y][x] == a)
        rel.append({(y - ay, x - ax): g[y][x] for y, x in ob})
    ys = [y for L in rel for y, _ in L]; xs = [x for L in rel for _, x in L]
    y0, x0 = min(ys), min(xs)
    return [{(y - y0, x - x0): v for (y, x), v in L.items()} for L in rel], max(ys) - y0 + 1, max(xs) - x0 + 1, bg


def class_stack(g, source, align):
    """Layers = the scene's amodal layers (L3) or its colour classes, registered at a common corner or at
    their common centre on a canvas as large as the largest layer."""
    bg = bg_of(g)
    if source == "amodal":
        S = scene(g, bg)
        if not S or len(S) < 2 or any(L["model"] == "runs" for L in S): return None
        parts = [(L["colour"], L["M"]) for L in S]
    else:
        cells = {}
        for y, r in enumerate(g):
            for x, v in enumerate(r):
                if v != bg: cells.setdefault(v, set()).add((y, x))
        if len(cells) < 2: return None
        parts = sorted(cells.items())
    boxes = [bbox(M) for _, M in parts]
    h = max(r1 - r0 + 1 for r0, _, r1, _ in boxes); w = max(c1 - c0 + 1 for _, c0, _, c1 in boxes)
    layers = []
    for (c, M), (r0, c0, r1, c1) in zip(parts, boxes):
        if align == "centre":
            if (h - (r1 - r0 + 1)) % 2 or (w - (c1 - c0 + 1)) % 2: return None
            dy = (h - (r1 - r0 + 1)) // 2 - r0; dx = (w - (c1 - c0 + 1)) // 2 - c0
        else:
            dy = -r0 if align[0] == "t" else h - 1 - r1
            dx = -c0 if align[1] == "l" else w - 1 - c1
        layers.append({(y + dy, x + dx): c for y, x in M})
    return layers, h, w, bg


def stackers(train):
    """(name, fn(g) -> stack) whose canvas has the output's shape on every training pair."""
    i0, o0 = train[0]["input"], train[0]["output"]
    if H(o0) > H(i0) or W(o0) > W(i0) or (H(o0), W(o0)) == (H(i0), W(i0)): return []
    cands = [(n, lambda g, f=f: (lambda P: panel_stack(P) if P and len(P) >= 2 else None)(f(g))) for n, f in splitters(train)]
    cands += [(f"anchor{'8' if d else '4'}", lambda g, d=d: anchor_stack(g, d)) for d in (True, False)]
    cands += [(f"{s}-{a}", lambda g, s=s, a=a: class_stack(g, s, a))
              for s in ("amodal", "colour") for a in ("tl", "tr", "bl", "br", "centre")]
    out = []
    for name, fn in cands:
        ok = True
        for p in train:
            try:
                S = fn(p["input"])
            except Exception:
                S = None
            if not S or (S[1], S[2]) != (H(p["output"]), W(p["output"])): ok = False; break
        if ok: out.append((name, fn))
    return out


def induce_stack_order(train, stk, key):
    """Pairwise depth evidence from training outputs; key 'colour' (items are colours) or 'panel'
    (items are layer indices).  Returns a closed Order or None if the painter's law cannot explain a pair."""
    od = Order(); nlay = None
    for p in train:
        layers, h, w, t = stk(p["input"]); o = p["output"]
        if key == "panel":
            if nlay is None: nlay = len(layers)
            elif nlay != len(layers): return None
        for y in range(h):
            for x in range(w):
                vals = [(k, L[(y, x)]) for k, L in enumerate(layers) if (y, x) in L]
                v = o[y][x]
                if not vals:
                    if v != t: return None
                    continue
                if v not in {c for _, c in vals}: return None
                if key == "colour":
                    for _, c in vals:
                        if c != v: od.add(v, c)
                else:
                    win = [k for k, c in vals if c == v]
                    if len(win) == 1:
                        for k, c in vals:
                            if c != v: od.add(win[0], k)
    return od if od.close() else None


def compose(stack, key, od=None):
    """Painter's law (L1): each canvas cell shows its front-most opaque layer; undecided order -> None."""
    layers, h, w, t = stack
    out = [[t] * w for _ in range(h)]
    size = [len(L) for L in layers]
    for y in range(h):
        for x in range(w):
            vals = [(k, L[(y, x)]) for k, L in enumerate(layers) if (y, x) in L]
            if not vals: continue
            cols = {c for _, c in vals}
            if len(cols) == 1:
                out[y][x] = vals[0][1]; continue
            if key == "colour":
                win = od.top(cols)
            elif key == "panel":
                k = od.top([k for k, _ in vals])
                win = None if k is None else layers[k][(y, x)]
            else:
                best = (min if key == "small" else max)(size[k] for k, _ in vals)
                ws = {c for k, c in vals if size[k] == best}
                win = ws.pop() if len(ws) == 1 else None
            if win is None: return None
            out[y][x] = win
    return out


def fam_overlay(train):
    """L1+L2 on explicit layers (panels, windows, anchor-registered fragments, corner-stacked amodal layers)
    composed by the painter's law; depth order induced from training overlaps (by colour or layer index)
    or given by size (smaller / larger in front)."""
    for sname, stk in stackers(train):
        for key in ("colour", "panel", "small", "large"):
            if key in ("colour", "panel"):
                od = induce_stack_order(train, stk, key)
                if od is None: continue
            else:
                od = None
            def fn(g, stk=stk, od=od, key=key):
                S = stk(g)
                if not S or len(S[0]) < 2: return None
                if key == "panel" and od.clo and max(max(od.clo), max((b for s in od.clo.values() for b in s), default=0)) >= len(S[0]):
                    return None
                return compose(S, key, od)
            yield (f"optics:overlay[{sname},{key}]", 4, fn)


# ------------------------------------------------------------------ scene analysis: amodal completion (L3)
MODELS = ("diag", "anti", "outline", "rect")          # smallest completion first (Occam)


class Pic:
    """A grid with O(1) per-colour rectangle counts (2-D prefix sums)."""
    def __init__(self, g, bg):
        self.g, self.bg, self.h, self.w = g, bg, H(g), W(g)
        self.P = {}
        for c in {v for r in g for v in r}:
            P = [[0] * (self.w + 1) for _ in range(self.h + 1)]
            for y in range(self.h):
                acc = 0; row = g[y]; up = P[y]; cur = P[y + 1]
                for x in range(self.w):
                    acc += row[x] == c
                    cur[x + 1] = up[x + 1] + acc
            self.P[c] = P

    def count(self, c, r0, c0, r1, c1):
        P = self.P.get(c)
        if P is None or r1 < r0 or c1 < c0: return 0
        return P[r1 + 1][c1 + 1] - P[r0][c1 + 1] - P[r1 + 1][c0] + P[r0][c0]

    def ring(self, c, r0, c0, r1, c1):
        return (self.count(c, r0, c0, r0, c1) + self.count(c, r1, c0, r1, c1) +
                self.count(c, r0 + 1, c0, r1 - 1, c0) + self.count(c, r0 + 1, c1, r1 - 1, c1))


def fit_model(pic, V, col, box, model):
    """Is `model` an admissible completion of fragment V (colour col, bbox box)?  Hidden cells must all be
    covered by other colours: no background and no other cell of the layer's own colour inside the model."""
    r0, c0, r1, c1 = box
    if model == "rect":
        return pic.count(pic.bg, r0, c0, r1, c1) == 0 and pic.count(col, r0, c0, r1, c1) == len(V)
    if model == "outline":
        if r1 - r0 < 2 or c1 - c0 < 2: return False
        if any(r0 < y < r1 and c0 < x < c1 for y, x in V): return False
        return pic.ring(pic.bg, r0, c0, r1, c1) == 0 and pic.ring(col, r0, c0, r1, c1) == len(V)
    if model in ("diag", "anti"):
        if r1 == r0 or r1 - r0 != c1 - c0: return False
        if model == "diag":
            if len({y - x for y, x in V}) != 1: return False
            cells = [(r0 + k, c0 + k) for k in range(r1 - r0 + 1)]
        else:
            if len({y + x for y, x in V}) != 1: return False
            cells = [(r0 + k, c1 - k) for k in range(r1 - r0 + 1)]
        g = pic.g
        return all((y, x) in V or (g[y][x] != pic.bg and g[y][x] != col) for y, x in cells)
    return False


def model_cells(box, model):
    r0, c0, r1, c1 = box
    if model == "rect":
        return {(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1)}
    if model == "outline":
        return {(y, x) for y in range(r0, r1 + 1) for x in (c0, c1)} | {(y, x) for y in (r0, r1) for x in range(c0, c1 + 1)}
    if model == "diag":
        return {(r0 + k, c0 + k) for k in range(r1 - r0 + 1)}
    return {(r0 + k, c1 - k) for k in range(r1 - r0 + 1)}


def best_model(pic, V, col, box=None):
    """Smallest admissible completion (Occam); irregular shapes fall back to run closure."""
    box = box or bbox(V)
    for m in MODELS:
        if fit_model(pic, V, col, box, m): return m, model_cells(box, m)
    return "runs", run_closure(pic.g, pic.bg, V)


def run_closure(g, bg, V):
    """Fallback completion of an irregular shape: a gap between two visible cells of the shape on one row
    or column is hidden (part of the shape) when every gap cell is covered by another colour."""
    M = set(V); col = g[next(iter(V))[0]][next(iter(V))[1]]
    rows, cols = {}, {}
    for y, x in V:
        rows.setdefault(y, []).append(x); cols.setdefault(x, []).append(y)
    for y, xs in rows.items():
        xs.sort()
        for a, b in zip(xs, xs[1:]):
            if b > a + 1 and all(g[y][x] != bg and g[y][x] != col for x in range(a + 1, b)):
                M |= {(y, x) for x in range(a + 1, b)}
    for x, ys in cols.items():
        ys.sort()
        for a, b in zip(ys, ys[1:]):
            if b > a + 1 and all(g[y][x] != bg and g[y][x] != col for y in range(a + 1, b)):
                M |= {(y, x) for y in range(a + 1, b)}
    return M


def _union_box(a, b):
    return min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])


def _aligned(a, b):
    return (a[0] <= b[2] and b[0] <= a[2]) or (a[1] <= b[3] and b[1] <= a[3])


_SCENES = {}


def scene(g, bg, skip=(), max_frag=80):
    """Layers of a picture: list of dicts {colour, V (visible cells), M (amodal cells), model}.
    Fragments of one colour (8-connected) are merged by good continuation when a non-trivial model of
    their union is admissible (rectangle merges also need aligned fragments).  Colours in `skip` are pure
    occluders (not layers).  Cached per grid."""
    key = (tuple(map(tuple, g)), bg, tuple(sorted(skip)))
    if key in _SCENES: return _SCENES[key]
    if len(_SCENES) > 64: _SCENES.clear()
    res = _scene(g, bg, skip, max_frag)
    _SCENES[key] = res
    return res


def _scene(g, bg, skip, max_frag):
    frags = {}
    for ob in objects(g, bg, True, True):
        c = g[ob[0][0]][ob[0][1]]
        if c in skip: continue
        frags.setdefault(c, []).append((set(ob), bbox(ob)))
    if sum(len(v) for v in frags.values()) > max_frag: return None
    pic = Pic(g, bg); layers = []
    for c, F in frags.items():
        if len(F) > 1:
            U = set().union(*(V for V, _ in F)); box = bbox(U)
            if any(fit_model(pic, U, c, box, m) for m in MODELS): F = [(U, box)]
        changed = True
        while changed and len(F) > 1:
            changed = False
            a = 0
            while a < len(F):
                b = a + 1
                while b < len(F):
                    (A, ba), (B, bb) = F[a], F[b]
                    U = A | B; box = _union_box(ba, bb)
                    ok = any(fit_model(pic, U, c, box, m) for m in ("diag", "anti", "outline")) or \
                        (_aligned(ba, bb) and fit_model(pic, U, c, box, "rect"))
                    if ok:
                        F[a] = (U, box); del F[b]; changed = True
                    else:
                        b += 1
                a += 1
        for V, box in F:
            m, M = best_model(pic, V, c, box)
            layers.append({"colour": c, "V": V, "M": M, "model": m})
    return layers


def layer_order(g, layers):
    """Occlusion evidence between layers (L2): layer j in front of layer i if j is visible inside M_i."""
    where = {}
    for j, L in enumerate(layers):
        for p in L["V"]: where[p] = j
    od = Order()
    for i, L in enumerate(layers):
        for p in L["M"] - L["V"]:
            j = where.get(p)
            if j is not None and j != i: od.add(j, i)
    for i in range(len(layers)): od.front.setdefault(i, set())
    return od if od.close() else None


def colour_order(layers, od):
    co = Order()
    for a, s in od.front.items():
        for b in s:
            ca, cb = layers[a]["colour"], layers[b]["colour"]
            if ca != cb: co.add(ca, cb)
    for L in layers: co.front.setdefault(L["colour"], set())
    return co if co.close() else None


# ------------------------------------------------------------------ depth-order summaries
VIEWS = ("row-b2f", "row-f2b", "col-b2f", "col-f2b", "top", "bottom")


def depth_view(g, view):
    bg = bg_of(g)
    layers = scene(g, bg)
    if not layers: return None
    od = layer_order(g, layers)
    if od is None: return None
    co = colour_order(layers, od)
    if co is None: return None
    cols = sorted({L["colour"] for L in layers})
    if len(cols) < 2 or len(cols) != len(layers) or any(L["model"] == "runs" for L in layers): return None
    if view in ("top", "bottom"):
        if view == "top":
            w = co.top(cols)
        else:
            w = [a for a in cols if all(b == a or co.above(b, a) for b in cols)]
            w = w[0] if len(w) == 1 else None
        return [[w]] if w is not None else None
    ch = co.chain(cols)
    if ch is None: return None
    if view.endswith("f2b"): ch = ch[::-1]
    return [ch] if view.startswith("row") else [[c] for c in ch]


def depth_programs(train):
    """Read-out 1: the depth order of the scene's colour layers (row/column list or front/back colour)."""
    o = train[0]["output"]
    if not (H(o) == 1 or W(o) == 1) or H(o) * W(o) > 10: return
    for view in VIEWS:
        if view in ("top", "bottom") and (H(o), W(o)) != (1, 1): continue
        if view.startswith("row") and H(o) != 1: continue
        if view.startswith("col") and W(o) != 1: continue
        yield (f"optics:depth[{view}]", 4, lambda g, view=view: depth_view(g, view))


# ------------------------------------------------------------------ disocclusion and re-layering
def _bottom(od, items):
    items = list(items)
    if len(items) == 1: return items[0]
    hits = [a for a in items if all(b == a or od.above(b, a) for b in items)]
    return hits[0] if len(hits) == 1 else None


def occluders(train):
    """Colours present in every training input and absent from every training output."""
    X = None
    for p in train:
        ci = {v for r in p["input"] for v in r}; co = {v for r in p["output"] for v in r}
        X = (ci - co) if X is None else X & (ci - co)
    return sorted(X or ())


def disocclude(g, X, out):
    cells = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] == X]
    if not cells: return None
    rest = Counter(v for r in g for v in r if v != X)
    if not rest: return None
    bg = rest.most_common(1)[0][0]
    layers = scene(g, bg, skip={X})
    if layers is None: return None
    od = layer_order(g, layers)
    if od is None: return None
    res = [r[:] for r in g]
    for y, x in cells:
        cov = [j for j, L in enumerate(layers) if (y, x) in L["M"]]
        if not cov:
            res[y][x] = bg; continue
        cs = {layers[j]["colour"] for j in cov}
        if len(cs) == 1:
            res[y][x] = cs.pop(); continue
        w = od.top(cov)
        if w is None: return None
        res[y][x] = layers[w]["colour"]
    return res if out == "grid" else crop(res, bbox(cells))


def occluder_role(g, role):
    """Occluder colour of a grid by role: 'rare' = least frequent colour, 'frag' = the colour split into
    the most 8-connected pieces (scattered noise); unique or None."""
    cnt = Counter(v for r in g for v in r)
    if len(cnt) < 3: return None
    bg = cnt.most_common(1)[0][0]
    if role == "rare":
        m = min(n for c, n in cnt.items() if c != bg)
        hits = [c for c, n in cnt.items() if c != bg and n == m]
    else:
        pieces = Counter(g[o[0][0]][o[0][1]] for o in objects(g, bg, True, True))
        m = max(pieces.values())
        hits = [c for c, n in pieces.items() if n == m]
    return hits[0] if len(hits) == 1 else None


def disocclude_programs(train):
    """Read-out 2: remove an occluder; hidden cells show the front-most remaining amodal layer (L1-L3).
    The occluder is a fixed colour (absent from all outputs) or a per-grid role (rarest / most fragmented)."""
    i, o = train[0]["input"], train[0]["output"]
    same = (H(i), W(i)) == (H(o), W(o))
    out = "grid" if same else "patch"
    for X in occluders(train):
        yield (f"optics:disocclude[c{X},{out}]", 4, lambda g, X=X, out=out: disocclude(g, X, out))
    for role in ("rare", "frag"):
        if all(occluder_role(p["input"], role) is not None and occluder_role(p["input"], role) not in {v for r in p["output"] for v in r}
               for p in train):
            yield (f"optics:disocclude[{role},{out}]", 5,
                   lambda g, role=role, out=out: (lambda X: disocclude(g, X, out) if X is not None else None)(occluder_role(g, role)))


def relayer(g, key):
    bg = bg_of(g)
    layers = scene(g, bg)
    if not layers or len(layers) < 2: return None
    od = layer_order(g, layers)
    if od is None: return None
    res = [[bg] * W(g) for _ in range(H(g))]
    cover = {}
    for j, L in enumerate(layers):
        for p in L["M"]: cover.setdefault(p, []).append(j)
    for (y, x), cov in cover.items():
        cs = {layers[j]["colour"] for j in cov}
        if len(cs) == 1:
            res[y][x] = cs.pop(); continue
        if key == "reverse":
            w = _bottom(od, cov)
        else:
            sz = {j: len(layers[j]["M"]) for j in cov}
            best = (min if key == "small-front" else max)(sz.values())
            ws = {layers[j]["colour"] for j in cov if sz[j] == best}
            w = next(j for j in cov if sz[j] == best) if len(ws) == 1 else None
        if w is None: return None
        res[y][x] = layers[w]["colour"]
    return res


def relayer_programs(train):
    """Read-out 3: re-render the amodal layers with a new depth order (smaller/larger in front, reversed)."""
    i, o = train[0]["input"], train[0]["output"]
    if (H(i), W(i)) != (H(o), W(o)): return
    for key in ("small-front", "large-front", "reverse"):
        yield (f"optics:relayer[{key}]", 4, lambda g, key=key: relayer(g, key))


def fam_scene(train):
    """One layered-scene parse (amodal layers L3 + depth order L2) with three read-outs:
    depth summary, disocclusion (occluder removed), re-layering (new depth order)."""
    yield from depth_programs(train)
    yield from disocclude_programs(train)
    yield from relayer_programs(train)


# ------------------------------------------------------------------ projection onto a screen (L4)
def screen_colour(g, bg, role):
    """Screen colour by role: a fixed colour, or the colour of the unique largest object."""
    if role == "largest":
        obs = objects(g, bg, False, True)
        if not obs: return None
        m = max(len(o) for o in obs)
        cs = {g[o[0][0]][o[0][1]] for o in obs if len(o) == m}
        return cs.pop() if len(cs) == 1 else None
    return role if any(role in r for r in g) else None


_BEAMS = {}


def beams(g, bg, sc):
    """Beam of every body cell toward the screen colour sc (cached per grid): (colour, path, end, reach)."""
    key = (tuple(map(tuple, g)), bg, sc)
    if key in _BEAMS: return _BEAMS[key]
    if len(_BEAMS) > 64: _BEAMS.clear()
    h, w = H(g), W(g); res = []
    ahead = {}                                        # screen cell ahead of (y, x) in each direction
    for d in N4:
        dy, dx = d
        ys = range(h) if dy <= 0 else range(h - 1, -1, -1)
        xs = range(w) if dx <= 0 else range(w - 1, -1, -1)
        seen = {}
        for y in ys:
            for x in xs:
                py, px = y + dy, x + dx
                nxt = (0 <= py < h and 0 <= px < w) and (g[py][px] == sc or seen.get((py, px), False))
                seen[(y, x)] = nxt
        ahead[d] = seen
    for y in range(h):
        for x in range(w):
            c = g[y][x]
            if c == bg or c == sc: continue
            ds = [d for d in N4 if ahead[d][(y, x)]]
            if len(ds) != 1: continue
            dy, dx = ds[0]; yy, xx = y + dy, x + dx; path = []
            while 0 <= yy < h and 0 <= xx < w and g[yy][xx] == bg:
                path.append((yy, xx)); yy += dy; xx += dx
            reach = 0 <= yy < h and 0 <= xx < w and g[yy][xx] == sc
            res.append((c, path, (yy, xx), reach))
    _BEAMS[key] = res
    return res


def project(g, role, mode, paint):
    """Every body cell (not bg, not screen) casts a beam toward the screen: the unique axis direction in
    which a screen cell lies ahead.  The beam crosses background and stops at the first opaque cell, so a
    body nearer the screen occludes the images of bodies behind it.
    mode: sweep (paint the beam), sweep-reach (only beams that reach the screen), image (recolour the first
    screen cell hit), land (paint the last free cell before the screen); paint: body colour or a constant."""
    bg = bg_of(g)
    sc = screen_colour(g, bg, role)
    if sc is None or sc == bg: return None
    out = [r[:] for r in g]; n = 0
    for c, path, (yy, xx), reach in beams(g, bg, sc):
        col = c if paint is None else paint
        if mode == "sweep" or (mode == "sweep-reach" and reach):
            for y, x in path: out[y][x] = col
            n += len(path)
        elif mode == "image" and reach:
            out[yy][xx] = col; n += 1
        elif mode == "land" and reach and path:
            out[path[-1][0]][path[-1][1]] = col; n += 1
    return out if n else None


def fam_project(train):
    """L4: bodies project onto a screen (fixed colour or the largest object) along the axis facing it."""
    i, o = train[0]["input"], train[0]["output"]
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    bg = bg_of(i)
    new = Counter(o[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x])
    if not new: return
    cols = set.intersection(*[{v for r in p["input"] for v in r} for p in train]) - {bg}
    paints = [None] + [c for c, _ in new.most_common(2)]
    for role, mode, paint in product(sorted(cols) + ["largest"], ("sweep", "sweep-reach", "image", "land"), paints):
        fn = lambda g, role=role, mode=mode, paint=paint: project(g, role, mode, paint)
        try:
            if fn(i) != o: continue
        except Exception:
            continue
        yield (f"optics:project[{role},{mode},{'own' if paint is None else paint}]", 4, fn)


# ------------------------------------------------------------------ light rays and mirrors (L5)
def emitters(g, bg, kind):
    """Light sources (start cell, diagonal direction, colour): an L-triomino shines away from its missing
    corner; a diagonal segment continues from each free end; a single cell in a grid corner shines inward."""
    h, w = H(g), W(g); out = []
    for ob in objects(g, bg, True, True):
        c = g[ob[0][0]][ob[0][1]]; r0, c0, r1, c1 = bbox(ob); S = set(ob)
        if kind == "L" and len(ob) == 3 and r1 - r0 == 1 and c1 - c0 == 1:
            my, mx = next((y, x) for y in (r0, r1) for x in (c0, c1) if (y, x) not in S)
            cy, cx = r0 + r1 - my, c0 + c1 - mx; d = (cy - my, cx - mx)
            out.append(((cy + d[0], cx + d[1]), d, c, S))
        elif kind == "seg" and len(ob) >= 2 and r1 - r0 + 1 == len(ob) and c1 - c0 + 1 == len(ob):
            if len({y - x for y, x in ob}) == 1: ends = (((r0, c0), (-1, -1)), ((r1, c1), (1, 1)))
            elif len({y + x for y, x in ob}) == 1: ends = (((r0, c1), (-1, 1)), ((r1, c0), (1, -1)))
            else: continue
            for (y, x), d in ends:
                q = (y + d[0], x + d[1])
                if 0 <= q[0] < h and 0 <= q[1] < w: out.append((q, d, c, S))
        elif kind == "corner" and len(ob) == 1 and ob[0][0] in (0, h - 1) and ob[0][1] in (0, w - 1):
            y, x = ob[0]; d = (1 if y == 0 else -1, 1 if x == 0 else -1)
            out.append(((y + d[0], x + d[1]), d, c, S))
    return out


def trace(g, bg, out, start, d, col, transp, adopt, edge, lit, rid, sources):
    """Follow one diagonal ray (id rid), painting background (and transparent) cells.  Reflection law:
    when the vertical / horizontal neighbour in the direction of travel is opaque (a mirror) the
    corresponding velocity component flips; a head-on corner hit reverses the ray.  Grid edges absorb the
    ray, or reflect it (edge = all | x: left/right walls | y: top/bottom walls).  adopt: after a bounce the
    ray takes the mirror's colour.  lit maps painted cells to (colour, ray ids).  Returns the number of
    painted cells, or -1 when the picture is undecided: two beams lighting one cell with different colours,
    or a beam running into another light source -- interactions the training pairs never show."""
    h, w = H(g), W(g)
    def opaque(y, x, axis):
        if not (0 <= y < h and 0 <= x < w):
            return edge == "all" or (edge == "x" and axis == "h") or (edge == "y" and axis == "v")
        v = g[y][x]
        return v != bg and v not in transp
    def paint(y, x, col):
        c0, ids = lit.setdefault((y, x), (col, set()))
        if c0 != col: return False
        ids.add(rid); out[y][x] = col
        return True
    y, x = start; dy, dx = d
    if not (0 <= y < h and 0 <= x < w) or opaque(y, x, "d"): return 0
    seen = set(); n = 0
    while (y, x, dy, dx) not in seen:
        seen.add((y, x, dy, dx))
        if not paint(y, x, col): return -1
        n += 1
        for _ in range(2):
            vb, hb = opaque(y + dy, x, "v"), opaque(y, x + dx, "h")
            db = 0 <= y + dy < h and 0 <= x + dx < w and opaque(y + dy, x + dx, "d")
            if not (vb or hb or db): break
            src = (y + dy, x) if vb and not hb else (y, x + dx) if hb and not vb else (y + dy, x + dx)
            if src in sources or (vb and (y + dy, x) in sources) or (hb and (y, x + dx) in sources): return -1
            if vb and not hb: dy = -dy
            elif hb and not vb: dx = -dx
            else: dy, dx = -dy, -dx
            if adopt and 0 <= src[0] < h and 0 <= src[1] < w and g[src[0]][src[1]] != col:
                if lit[(y, x)][1] != {rid}: return -1
                col = g[src[0]][src[1]]; lit[(y, x)] = (col, {rid}); out[y][x] = col
        ny, nx = y + dy, x + dx
        if not (0 <= ny < h and 0 <= nx < w) or opaque(ny, nx, "d"): break
        y, x = ny, nx
    return n


def light(g, kind, transp, adopt, edge, paint=None):
    """All emitters shine; beams that cross each other between cells (an X of two different rays) are an
    interaction never shown in training -> undecided (None)."""
    bg = bg_of(g); out = [r[:] for r in g]; n = 0; lit = {}
    em = emitters(g, bg, kind)
    sources = set().union(*(S for *_, S in em)) if em else set()
    for rid, (start, d, c, S) in enumerate(em):
        k = trace(g, bg, out, start, d, c if paint is None else paint, transp, adopt, edge, lit, rid, sources - S)
        if k < 0: return None
        n += k
    if not n: return None
    ids = lambda y, x: lit.get((y, x), (None, set()))[1]
    for y in range(H(g) - 1):
        for x in range(W(g) - 1):
            a, d = ids(y, x) & ids(y + 1, x + 1), ids(y, x + 1) & ids(y + 1, x)
            if a and d and (a | d) != (a & d): return None
    return out


def _fits_up_to_bg(pred, o, bg):
    """pred equals o, except that the background may be rendered in one other colour."""
    if pred is None or (H(pred), W(pred)) != (H(o), W(o)): return False
    m = None
    for rp, ro in zip(pred, o):
        for a, b in zip(rp, ro):
            if a == b and a != bg: continue
            if a != bg: return False
            if m is None: m = b
            elif m != b: return False
    return True


def fam_light(train):
    """L5: diagonal light rays from emitters bounce off opaque bodies (mirrors), pass transparent colours."""
    i, o = train[0]["input"], train[0]["output"]
    if any((H(p["input"]), W(p["input"])) != (H(p["output"]), W(p["output"])) for p in train): return
    bg = bg_of(i)
    new = sorted(set.intersection(*[{v for r in p["output"] for v in r} - {v for r in p["input"] for v in r} for p in train]))
    for kind in ("L", "seg", "corner"):
        if not all(emitters(p["input"], bg_of(p["input"]), kind) for p in train): continue
        own = {c for _, _, c, _ in emitters(i, bg, kind)}
        cols = sorted({v for r in i for v in r} - {bg} - own)
        subsets = [()] + [(c,) for c in cols] + [(a, b) for k, a in enumerate(cols) for b in cols[k + 1:]]
        for transp, adopt, edge, paint in product(subsets, (False, True), ("none", "all", "x", "y"), [None] + new[:2]):
            if adopt and paint is not None: continue
            fn = lambda g, t=frozenset(transp), a=adopt, e=edge, k=kind, pc=paint: light(g, k, t, a, e, pc)
            if not _fits_up_to_bg(fn(i), o, bg): continue
            yield (f"optics:light[{kind},transp={''.join(map(str, transp)) or '-'},{'adopt' if adopt else 'keep'},"
                   f"edge={edge},paint={'own' if paint is None else paint}]", 5, fn)


FAMILIES = (fam_overlay, fam_scene, fam_project, fam_light)


In [ ]:
%%writefile /kaggle/working/probe/prior_topology.py
"""TOPOLOGY prior: invariants under continuous deformation, used as a general search space.

Concept
-------
A grid is read as a planar cell complex.  Foreground objects (connected components; 4- or 8-connected,
single-colour or multicolour) are the "curves/solids"; everything else is ambient space.  For every
object O we compute its INTERIOR I(O): the cells of its bbox (+1 margin) that cannot be reached from
outside without crossing O, where leakage through the complement uses the DUAL connectivity (8-connected
objects leak 4-wise, 4-connected objects leak 8-wise, i.e. a diagonal gap in a 4-object is a hole in its
boundary).  From the interiors we get the classical topological invariants:

  * holes / genus        n_holes(O) = #components of I(O)  (Euler number = 1 - n_holes)
  * containment tree     P inside O  iff  P ⊂ I(O);  depth(P) = #objects containing P; parent = innermost
  * nesting counts       n_children (objects directly inside), has_children
  * boundary contact     touches_border (the object meets the frame of the plane)
  * adjacency graph      degree = #distinct objects 8-adjacent (region adjacency graph)
  * cell depth           depth(cell) = #objects whose interior contains the cell (winding-number analogue)
  * zones                a zone is a component of I(O); its content = colours of objects inside it

Search space (all parameters induced from training pairs; unseen feature values refuse -> None)
  segmentation  seg = (kind, bg): kind ∈ {c4, c8, m4, m8} (conn × single/multi colour); bg = the input mode
                colour (fixed when shared by all training inputs), or the single input colour of all changed
                cells; plus the full colour partition p4/p8 (every colour is a region: region adjacency graph).
  1. topo:recolour[seg,feat(,col)]  object -> action table over a topological feature (n_holes, has_hole,
                                    depth, is_inside, n_children, parent colour, inside colours, touches
                                    border, RAG degree, euler class, empty-holes, linked colours), optionally
                                    keyed with the object colour.  action = keep | delete | literal colour |
                                    role colour (parent / root of containment tree / unique child / mirror
                                    position in the nesting chain).  Table must compress (#objects > #keys).
  2. topo:fill[seg,rule,paint(,clean)(,glob|comp)]
                                    zones (innermost object holes; or global enclosed background regions; or
                                    all background components) painted (bg cells | whole zone) by rule:
                                    encloser colour | unique / majority content colour | table over a zone
                                    feature (encloser colour, rim = outermost ring of rooms, open = reaches
                                    frame, encloser n_holes, size, width/height/size parity, depth, is_rect,
                                    content set, shape).  'clean' also erases loose single dots outside zones.
  3. topo:cellmap[seg,key(,role)]   cellwise map (colour or bg/fg role, containment key) -> keep | colour,
                                    key ∈ {inside, depth, innermost encloser colour, globally enclosed}.
  4. topo:select-colour/keep/crop   the object with the unique max / min / unique value of a topological
                                    measure (interior area, n_holes, #objects inside, n_children, depth,
                                    degree) -> its colour as a constant output, kept alone, or cropped
                                    (bbox | bbox shrunk by its wall | interior).
  5. topo:seed-region               path-connected background component of each seed cell (non-bg, non-wall
                                    colour) painted whole / on its boundary layer / on its OUTER boundary
                                    component only, with the seed colour or an induced colour; walls not
                                    touching any seeded region kept or deleted.
  6. topo:hole-vote                 host = object with most holes; each empty hole takes the majority colour
                                    of free objects congruent to it (translation or dihedral); output grid or
                                    host crop.
Dropped after measurement (0 solves): inner/outer halo of closed curves; recolour by exemplar with the same
hole count; per-object hole-area count rendering.
"""
import sys
from collections import Counter
from itertools import product
sys.path.insert(0, '/home/claude/work/widen')
from gdsl import H, W, bg_of, bbox

N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
N8 = N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


# ------------------------------------------------------------------ core topology
def _comps(h, w, ok, nb):
    """Components of cells (y,x) with ok(y,x) True."""
    seen = set(); out = []
    for r in range(h):
        for c in range(w):
            if (r, c) in seen or not ok(r, c): continue
            st = [(r, c)]; seen.add((r, c)); cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and ok(yy, xx):
                        seen.add((yy, xx)); st.append((yy, xx))
            out.append(cells)
    return out


def _objects(g, bg, seg):
    h, w = H(g), W(g); nb = N8 if seg[1] == '8' else N4; mono = seg[0] == 'c'
    seen = [[False] * w for _ in range(h)]; out = []
    for r in range(h):
        for c in range(w):
            if seen[r][c] or g[r][c] == bg: continue
            col = g[r][c]; st = [(r, c)]; seen[r][c] = True; cells = []
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and g[yy][xx] != bg \
                            and (not mono or g[yy][xx] == col):
                        seen[yy][xx] = True; st.append((yy, xx))
            out.append(cells)
    return out


def _interior(cells, h, w, leak):
    """Cells enclosed by `cells` (not reachable from outside the bbox margin through non-object cells)."""
    r0, c0, r1, c1 = bbox(cells)
    if r1 - r0 < 2 or c1 - c0 < 2: return []
    s = set(cells)
    R0, C0, R1, C1 = r0 - 1, c0 - 1, r1 + 1, c1 + 1
    seen = set(); st = []
    for y in range(R0, R1 + 1):
        for x in (C0, C1):
            if (y, x) not in seen: seen.add((y, x)); st.append((y, x))
    for x in range(C0, C1 + 1):
        for y in (R0, R1):
            if (y, x) not in seen: seen.add((y, x)); st.append((y, x))
    while st:
        y, x = st.pop()
        for dy, dx in leak:
            yy, xx = y + dy, x + dx
            if R0 <= yy <= R1 and C0 <= xx <= C1 and (yy, xx) not in seen and (yy, xx) not in s:
                seen.add((yy, xx)); st.append((yy, xx))
    return [(y, x) for y in range(r0, r1 + 1) for x in range(c0, c1 + 1) if (y, x) not in s and (y, x) not in seen]


_CACHE = {}


class Topo:
    def __init__(self, g, seg, bg):
        self.g = g; h, w = self.h, self.w = H(g), W(g)
        self.bg = bg
        self.objs = objs = _objects(g, bg, seg)
        leak = N4 if seg[1] == '8' else N8
        self.inner = [_interior(o, h, w, leak) for o in objs]
        self.innerset = [set(I) for I in self.inner]
        self.owner = {}
        for k, o in enumerate(objs):
            for c in o: self.owner[c] = k
        n = len(objs)
        # containment
        self.contains = [[] for _ in range(n)]
        self.parents = [[] for _ in range(n)]
        for a in range(n):
            if not self.inner[a]: continue
            Ia = self.innerset[a]
            for b in range(n):
                if b != a and objs[b][0] in Ia:
                    self.contains[a].append(b); self.parents[b].append(a)
        self.depth = [len(p) for p in self.parents]
        self.parent = [min(p, key=lambda a: len(self.inner[a])) if p else None for p in self.parents]
        self.children = [[b for b in self.contains[a] if self.parent[b] == a] for a in range(n)]
        # zones = components of each interior (dual connectivity)
        self.zones = []          # (owner index, cells)
        self.nholes = [0] * n
        for a in range(n):
            if not self.inner[a]: continue
            Ia = self.innerset[a]
            zs = _comps(h, w, lambda y, x, Ia=Ia: (y, x) in Ia, leak)
            self.nholes[a] = len(zs)
            for z in zs: self.zones.append((a, z))
        # cell depth
        self.cdepth = [[0] * w for _ in range(h)]
        self.cenc = [[None] * w for _ in range(h)]    # innermost encloser object index
        best = [[10 ** 9] * w for _ in range(h)]
        for a in range(n):
            L = len(self.inner[a])
            for y, x in self.inner[a]:
                self.cdepth[y][x] += 1
                if L < best[y][x]: best[y][x] = L; self.cenc[y][x] = a

    def root(self, k):
        while self.parent[k] is not None: k = self.parent[k]
        return k

    def outside(self):
        """Background cells connected (4-wise) to the grid border through background cells."""
        if getattr(self, '_out', None) is None:
            h, w, g, bg = self.h, self.w, self.g, self.bg
            st = [(y, x) for y in range(h) for x in range(w) if (y in (0, h - 1) or x in (0, w - 1)) and g[y][x] == bg]
            seen = set(st)
            while st:
                y, x = st.pop()
                for dy, dx in N4:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in seen and g[yy][xx] == bg:
                        seen.add((yy, xx)); st.append((yy, xx))
            self._out = seen
        return self._out

    def colour(self, k):
        cs = Counter(self.g[y][x] for y, x in self.objs[k]); return cs.most_common(1)[0][0]

    def degree(self, k):
        s = set(); g = self.g
        for y, x in self.objs[k]:
            for dy, dx in N8:
                yy, xx = y + dy, x + dx
                if 0 <= yy < self.h and 0 <= xx < self.w:
                    j = self.owner.get((yy, xx))
                    if j is not None and j != k: s.add(j)
        return len(s)


def seg_bg(g, seg):
    """seg = (kind, bgmode): bgmode 'mode' (most frequent colour), None (partition: every colour is a region)
    or an int colour induced from the training pairs."""
    kind, bm = seg
    if bm == 'mode': return bg_of(g)
    return bm


def topo(g, seg):
    key = (seg, tuple(map(tuple, g)))
    t = _CACHE.get(key)
    if t is None:
        if len(_CACHE) > 256: _CACHE.clear()
        t = _CACHE[key] = Topo(g, seg[0], seg_bg(g, seg))
    return t


def segs_for(train, partition=True):
    """Segmentations to try: 4 kinds x background modes. Besides the most frequent colour, a colour c is
    tried as background when every changed cell of every training pair had input colour c (the painted
    ambient space) and c is not the mode."""
    modes = {bg_of(p['input']) for p in train}
    bms = [next(iter(modes))] if len(modes) == 1 else ['mode']   # a background shared by all pairs is fixed
    ch = set()
    for p in train:
        i, o = p['input'], p['output']
        if (H(i), W(i)) != (H(o), W(o)): ch = None; break
        ch |= {i[y][x] for y in range(H(i)) for x in range(W(i)) if i[y][x] != o[y][x]}
    if ch and len(ch) == 1:
        c = next(iter(ch))
        if c not in bms: bms.append(c)
    out = [(k, b) for b in bms for k in ('c4', 'c8', 'm4', 'm8')]
    if partition: out += [('c4', None), ('c8', None)]
    return out


def sname(seg):
    k, b = seg
    return k if b == 'mode' else (k.replace('c', 'p') if b is None else f"{k}/bg{b}")


def fixed_bg(train):
    modes = {bg_of(p['input']) for p in train}
    return next(iter(modes)) if len(modes) == 1 else 'mode'


def same_shape(train):
    return all((H(p['input']), W(p['input'])) == (H(p['output']), W(p['output'])) for p in train)


# ------------------------------------------------------------------ object features
OFEATS = ('n_holes', 'has_hole', 'depth', 'is_inside', 'n_children', 'has_children', 'parent_colour',
          'inside_colours', 'touches_border', 'degree', 'euler_class', 'zone_empty', 'linked_colours')


def ofeat(t, k, f):
    if f == 'n_holes': return t.nholes[k]
    if f == 'has_hole': return t.nholes[k] > 0
    if f == 'depth': return t.depth[k]
    if f == 'is_inside': return t.depth[k] > 0
    if f == 'n_children': return len(t.children[k])
    if f == 'has_children': return len(t.children[k]) > 0
    if f == 'parent_colour': return None if t.parent[k] is None else t.colour(t.parent[k])
    if f == 'inside_colours': return frozenset(t.g[y][x] for y, x in t.inner[k] if t.g[y][x] != t.bg)
    if f == 'touches_border': return any(y in (0, t.h - 1) or x in (0, t.w - 1) for y, x in t.objs[k])
    if f == 'degree': return t.degree(k)
    if f == 'euler_class': return (t.nholes[k] > 0, len(t.children[k]) > 0)
    if f == 'linked_colours':   # other colours in the same 8-connected foreground component (path-connected to it)
        if t.bg is None: return None
        if not hasattr(t, '_m8'):
            t._m8 = {}
            for cells in _objects(t.g, t.bg, 'm8'):
                cs = frozenset(t.g[y][x] for y, x in cells)
                for c in cells: t._m8[c] = cs
        return t._m8[t.objs[k][0]] - {t.colour(k)}
    if f == 'zone_empty':   # has a hole and every hole is empty (bg only)
        if not t.inner[k]: return 'nohole'
        return all(t.g[y][x] == t.bg for y, x in t.inner[k])
    raise KeyError(f)


def role_colour(t, k, role):
    if role == 'parent': return None if t.parent[k] is None else t.colour(t.parent[k])
    if role == 'root': return t.colour(t.root(k))
    if role == 'child':
        cs = {t.colour(b) for b in t.children[k]}
        return cs.pop() if len(cs) == 1 else None
    if role == 'mirror':   # nesting chain root..leaf read backwards (each level has one child)
        ch = chain(t, k)
        return None if ch is None else t.colour(ch[len(ch) - 1 - ch.index(k)])
    raise KeyError(role)


def chain(t, k):
    r = t.root(k); ch = [r]
    while t.children[ch[-1]]:
        if len(t.children[ch[-1]]) != 1: return None
        ch.append(t.children[ch[-1]][0])
    return ch if k in ch and len(ch) > 1 else None


ROLES = ('parent', 'root', 'child', 'mirror')


def _choose(acts):
    if 'keep' in acts: return 'keep'
    for r in ROLES:
        if ('r', r) in acts: return ('r', r)
    if 'del' in acts: return 'del'
    return sorted(a for a in acts if isinstance(a, tuple))[0]


# ------------------------------------------------------------------ 1. recolour objects by topological class
def fam_topo_recolour(train):
    if not same_shape(train): return
    for seg in segs_for(train):
        tops = [topo(p['input'], seg) for p in train]
        if any(not t.objs or len(t.objs) > 250 for t in tops): continue
        obs = []; ok = True
        for t, p in zip(tops, train):
            o = p['output']; g = p['input']
            if any(g[y][x] != o[y][x] and (y, x) not in t.owner for y in range(t.h) for x in range(t.w)):
                ok = False; break
            beh = []
            for k, cells in enumerate(t.objs):
                outs = {o[y][x] for y, x in cells}
                acts = set()
                if all(o[y][x] == g[y][x] for y, x in cells): acts.add('keep')
                if len(outs) == 1:
                    c = outs.pop(); acts.add(('c', c))
                    if c == t.bg: acts.add('del')
                    for r in ROLES:
                        if role_colour(t, k, r) == c: acts.add(('r', r))
                if not acts: ok = False; break
                beh.append(acts)
            if not ok: break
            obs.append(beh)
        if not ok or all('keep' in a for beh in obs for a in beh): continue
        mono = seg[0][0] == 'c'
        for f, withcol in product(OFEATS, (False, True)):
            if withcol and not mono: continue
            table = {}; good = True
            for t, beh in zip(tops, obs):
                for k, acts in enumerate(beh):
                    v = ofeat(t, k, f)
                    key = (v, t.colour(k)) if withcol else v
                    table[key] = table[key] & acts if key in table else set(acts)
                    if not table[key]: good = False; break
                if not good: break
            if not good or len({k[0] if withcol else k for k in table}) < 2: continue
            if sum(len(b) for b in obs) <= len(table): continue   # table must compress
            fin = {key: _choose(a) for key, a in table.items()}
            if len(set(fin.values())) < 2: continue

            def fn(g, seg=seg, f=f, withcol=withcol, fin=fin):
                t = topo(g, seg); out = [r[:] for r in g]
                if not t.objs or len(t.objs) > 400: return None
                for k, cells in enumerate(t.objs):
                    v = ofeat(t, k, f)
                    key = (v, t.colour(k)) if withcol else v
                    if key not in fin: return None
                    a = fin[key]
                    if a == 'keep': continue
                    if a == 'del': c = t.bg
                    elif a[0] == 'r': c = role_colour(t, k, a[1])
                    else: c = a[1]
                    if c is None: return None
                    for y, x in cells: out[y][x] = c
                return out
            yield (f"topo:recolour[{sname(seg)},{f}{',col' if withcol else ''}]", 4 + withcol + (seg[1] is None), fn)


# ------------------------------------------------------------------ 2. fill zones (holes) by a topological rule
ZFEATS = ('enc_colour', 'rim', 'open', 'enc_nholes', 'size', 'w_parity', 'h_parity', 'size_parity', 'depth', 'is_rect',
          'content', 'enc_colour_size', 'shape')


def enc_colour(t, a, z):
    """Colour of the zone's encloser: the enclosing object, or (global zones, a is None) the majority colour
    of the non-background cells 4-adjacent to the zone."""
    if a is not None: return t.colour(a)
    cs = Counter(); zs = set(z)
    for y, x in z:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx
            if 0 <= yy < t.h and 0 <= xx < t.w and (yy, xx) not in zs and t.g[yy][xx] != t.bg: cs[t.g[yy][xx]] += 1
    if not cs: return None
    top = cs.most_common(2)
    return top[0][0] if len(top) == 1 or top[0][1] > top[1][1] else None


def global_zones(t):
    """Components (4-wise) of non-exterior cells that contain background: regions sealed off from the
    grid border by the union of all objects (the encloser may be several objects)."""
    out = t.outside(); g, bg = t.g, t.bg
    zs = _comps(t.h, t.w, lambda y, x: (y, x) not in out and g[y][x] == bg, N4)
    return [(None, z, z) for z in zs]


def _rim(t, a, z):
    key = 'obj' if a is not None else 'glob'
    if not hasattr(t, '_zcell'): t._zcell = {}
    if key not in t._zcell:
        Z = _innermost_zones(t) if a is not None else bg_components(t)   # enclosed bg zones are bg components
        m = {}
        for zi, (_, zz, _) in enumerate(Z):
            for c in zz: m[c] = zi
        t._zcell[key] = m
    m = t._zcell[key]; zs = set(z); me = m.get(z[0])
    for y, x in z:
        for dy, dx in N4:
            yy, xx = y + dy, x + dx; ok = True
            while 0 <= yy < t.h and 0 <= xx < t.w:
                j = m.get((yy, xx))
                if j is not None and j != me: ok = False; break
                yy += dy; xx += dx
            if ok: return True
    return False


def bg_components(t):
    """Every 4-connected component of background cells (open and closed regions of the complement)."""
    g, bg = t.g, t.bg
    return [(None, z, z) for z in _comps(t.h, t.w, lambda y, x: g[y][x] == bg, N4)]


def zfeat(t, a, z, f):
    if f == 'enc_colour': return enc_colour(t, a, z)
    if f == 'rim':   # outermost ring of rooms: some axis ray leaves the grid without entering another zone
        return _rim(t, a, z)
    if f == 'enc_nholes': return None if a is None else t.nholes[a]
    if f == 'open': return any(y in (0, t.h - 1) or x in (0, t.w - 1) for y, x in z)   # reaches the frame
    if f == 'size': return len(z)
    if f in ('w_parity', 'h_parity'):
        r0, c0, r1, c1 = bbox(z); return ((c1 - c0 + 1) if f[0] == 'w' else (r1 - r0 + 1)) % 2
    if f == 'size_parity': return len(z) % 2
    if f == 'depth': return None if a is None else t.depth[a] + 1
    if f == 'is_rect':
        r0, c0, r1, c1 = bbox(z); return len(z) == (r1 - r0 + 1) * (c1 - c0 + 1)
    if f == 'content': return frozenset(t.g[y][x] for y, x in z if t.g[y][x] != t.bg)
    if f == 'enc_colour_size': return (enc_colour(t, a, z), len(z))
    if f == 'shape':
        r0, c0, _, _ = bbox(z); return tuple(sorted((y - r0, x - c0) for y, x in z))
    raise KeyError(f)


def content_colour(t, z, how):
    cs = Counter(t.g[y][x] for y, x in z if t.g[y][x] != t.bg)
    if not cs: return None
    if how == 'content': return next(iter(cs)) if len(cs) == 1 else None
    top = cs.most_common(2)
    return top[0][0] if len(top) == 1 or top[0][1] > top[1][1] else None


def _innermost_zones(t):
    """Each enclosed cell belongs to its smallest enclosing zone -> list of (owner, zone, cells)."""
    best = {}
    for zi, (a, z) in enumerate(t.zones):
        L = len(z)
        for c in z:
            if c not in best or L < len(t.zones[best[c]][1]): best[c] = zi
    per = {}
    for c, zi in best.items(): per.setdefault(zi, []).append(c)
    return [(t.zones[zi][0], t.zones[zi][1], cells) for zi, cells in sorted(per.items())]


def _loose_dots(t, Z):
    """Single-cell objects lying in no zone (noise in the ambient space)."""
    inz = set()
    for _, z, _ in Z: inz.update(z)
    return [cells[0] for cells in t.objs if len(cells) == 1 and cells[0] not in inz]


def fam_topo_fill(train):
    if not same_shape(train): return
    for seg in segs_for(train, partition=False):
        tops = [topo(p['input'], seg) for p in train]
        if any(len(t.objs) > 250 for t in tops): continue
        for zk in ('obj', 'glob', 'comp'):
            if zk != 'obj' and seg[0] != 'c4': continue    # global zones do not depend on the segmentation kind
            zfun = {'obj': _innermost_zones, 'glob': global_zones, 'comp': bg_components}[zk]
            zl = [zfun(t) for t in tops]
            if not any(zl): continue
            nz_ok = not all(zl)   # a zone-less training pair was left unchanged
            for paint, clean in product(('bg', 'all'), (False, True)):
                if zk != 'obj' and paint == 'all': continue
                obsz = []; ok = True; ndots = 0
                for t, p, Z in zip(tops, train, zl):
                    g, o = p['input'], p['output']
                    cover = set(); row = []
                    for a, z, cells in Z:
                        tgt = [c for c in cells if paint == 'all' or g[c[0]][c[1]] == t.bg]
                        cover.update(tgt)
                        if not tgt: row.append(None); continue
                        outs = {o[y][x] for y, x in tgt}
                        if len(outs) != 1: ok = False; break
                        c = outs.pop()
                        row.append(('keep', c) if all(o[y][x] == g[y][x] for y, x in tgt) else ('set', c))
                    if not ok: break
                    if clean:
                        dots = _loose_dots(t, Z)
                        if any(o[y][x] != t.bg for y, x in dots): ok = False; break
                        cover.update(dots); ndots += len(dots)
                    if any(g[y][x] != o[y][x] and (y, x) not in cover for y in range(t.h) for x in range(t.w)):
                        ok = False; break
                    obsz.append(row)
                if not ok or not any(r and r[0] == 'set' for row in obsz for r in row): continue
                if clean and not ndots: continue

                def apply(g, seg, paint, clean, colour_of, nz_ok=nz_ok, zfun=zfun):
                    t = topo(g, seg); out = [r[:] for r in g]
                    if len(t.objs) > 400: return None
                    Z = zfun(t)
                    if not Z and not nz_ok: return None
                    if clean:
                        for y, x in _loose_dots(t, Z): out[y][x] = t.bg
                    for a, z, cells in Z:
                        tgt = [c for c in cells if paint == 'all' or g[c[0]][c[1]] == t.bg]
                        if not tgt: continue
                        c = colour_of(t, a, z)
                        if c == 'refuse': return None
                        if c is None: continue
                        for y, x in tgt: out[y][x] = c
                    return out

                tag = paint + (',clean' if clean else '') + ('' if zk == 'obj' else ',' + zk)
                for rule in ('encloser', 'content', 'majority'):
                    col = enc_colour if rule == 'encloser' else \
                          (lambda t, a, z, rule=rule: content_colour(t, z, rule))
                    good = True
                    for t, Z, row in zip(tops, zl, obsz):
                        for (a, z, cells), r in zip(Z, row):
                            if r is None: continue
                            c = col(t, a, z)
                            if (c is None and r[0] == 'set') or (c is not None and c != r[1]): good = False; break
                        if not good: break
                    if good:
                        yield (f"topo:fill[{sname(seg)},{rule},{tag}]", 4,
                               lambda g, seg=seg, paint=paint, clean=clean, col=col: apply(g, seg, paint, clean, col))
                for f in ZFEATS:
                    table = {}; good = True
                    for t, Z, row in zip(tops, zl, obsz):
                        for (a, z, cells), r in zip(Z, row):
                            if r is None: continue
                            v = zfeat(t, a, z, f)
                            acts = {('c', r[1])} | ({'keep'} if r[0] == 'keep' else set())
                            table[v] = table[v] & acts if v in table else acts
                            if not table[v]: good = False; break
                        if not good: break
                    nobs = sum(r is not None for row in obsz for r in row)
                    if not good or len(table) < 2 or nobs <= len(table): continue   # table must compress
                    fin = {v: ('keep' if 'keep' in a else next(iter(a))[1]) for v, a in table.items()}
                    if len(set(fin.values())) < 2: continue

                    def col(t, a, z, f=f, fin=fin):
                        v = zfeat(t, a, z, f)
                        if v not in fin: return 'refuse'
                        return None if fin[v] == 'keep' else fin[v]
                    yield (f"topo:fill[{sname(seg)},{f},{tag}]", 5,
                           lambda g, seg=seg, paint=paint, clean=clean, col=col: apply(g, seg, paint, clean, col))


# ------------------------------------------------------------------ 3. inside/outside-dependent cellwise colour map
CKEYS = ('inside', 'depth', 'enc_colour', 'enclosed')


def ckey(t, y, x, kname):
    if kname == 'inside': return t.cdepth[y][x] > 0
    if kname == 'depth': return t.cdepth[y][x]
    if kname == 'enc_colour':
        a = t.cenc[y][x]; return None if a is None else t.colour(a)
    if kname == 'enclosed':    # global: not reachable from the border through background
        out = t.outside()
        if t.g[y][x] == t.bg: return (y, x) not in out
        return not any((y + dy, x + dx) in out or not (0 <= y + dy < t.h and 0 <= x + dx < t.w) for dy, dx in N4)
    raise KeyError(kname)


def fam_topo_cellmap(train):
    if not same_shape(train): return
    for seg in segs_for(train, partition=False):
        tops = [topo(p['input'], seg) for p in train]
        if any(len(t.objs) > 250 for t in tops): continue
        for vrole, kname in product((True, False), CKEYS):
            if kname != 'enclosed' and not all(t.zones for t in tops): continue
            if kname == 'enclosed' and seg[0] != 'c4': continue   # global notion: independent of segmentation
            table = {}; good = True
            for t, p in zip(tops, train):
                g, o = p['input'], p['output']
                for y in range(t.h):
                    for x in range(t.w):
                        v = g[y][x]
                        k = (('bg' if v == t.bg else 'fg') if vrole else v, ckey(t, y, x, kname))
                        acts = {('c', o[y][x])} | ({'keep'} if o[y][x] == v else set())
                        table[k] = table[k] & acts if k in table else acts
                        if not table[k]: good = False; break
                    if not good: break
                if not good: break
            if not good: continue
            fin = {k: ('keep' if 'keep' in a else next(iter(a))[1]) for k, a in table.items()}
            if all(a == 'keep' for a in fin.values()): continue
            byv = {}
            for (v, _), a in fin.items(): byv.setdefault(v, set()).add(a)
            if not any(len(s) > 1 for s in byv.values()): continue   # containment must matter

            def fn(g, seg=seg, kname=kname, vrole=vrole, fin=fin, byv=byv):
                t = topo(g, seg); out = [r[:] for r in g]
                if len(t.objs) > 400: return None
                for y in range(t.h):
                    for x in range(t.w):
                        v = g[y][x]
                        k = (('bg' if v == t.bg else 'fg') if vrole else v, ckey(t, y, x, kname))
                        a = fin.get(k, None)
                        if a is None:
                            # unseen containment value: allowed only if the colour's action never depended on it
                            s = byv.get(k[0])
                            if s is None or len(s) != 1: return None
                            a = next(iter(s))
                        if a != 'keep': out[y][x] = a
                return out
            yield (f"topo:cellmap[{sname(seg)},{kname}{',role' if vrole else ''}]", 4 + (not vrole), fn)


# ------------------------------------------------------------------ 4. select the object by topological class
SFEATS = ('inner_size', 'n_holes', 'n_inside_objs', 'n_children', 'depth', 'degree')


def sfeat(t, k, f):
    if f == 'inner_size': return len(t.inner[k])
    if f == 'n_inside_objs': return len(t.contains[k])
    return ofeat(t, k, f)


def pick(t, f, ext):
    n = len(t.objs)
    if n < 2: return None
    vals = [sfeat(t, k, f) for k in range(n)]
    if ext == 'unique':
        cs = Counter(vals); hits = [k for k in range(n) if cs[vals[k]] == 1]
    else:
        b = max(vals) if ext == 'max' else min(vals)
        hits = [k for k in range(n) if vals[k] == b]
    return hits[0] if len(hits) == 1 else None


def fam_topo_select(train):
    same = same_shape(train)
    shp = {(H(p['output']), W(p['output'])) for p in train}
    const = len(shp) == 1 and all(len({v for r in p['output'] for v in r}) == 1 for p in train)
    for seg in segs_for(train):
        tops = [topo(p['input'], seg) for p in train]
        if any(len(t.objs) < 2 or len(t.objs) > 250 for t in tops): continue
        for f, ext in product(SFEATS, ('max', 'min', 'unique')):
            ks = [pick(t, f, ext) for t in tops]
            if any(k is None for k in ks): continue
            if const:
                if all(t.colour(k) == p['output'][0][0] for t, k, p in zip(tops, ks, train)):
                    oh, ow = next(iter(shp))

                    def fn(g, seg=seg, f=f, ext=ext, oh=oh, ow=ow):
                        t = topo(g, seg); k = pick(t, f, ext)
                        return None if k is None else [[t.colour(k)] * ow for _ in range(oh)]
                    yield (f"topo:select-colour[{sname(seg)},{f},{ext}]", 4 + (seg[1] is None), fn)
            elif same:
                if seg[1] is None: continue

                def fn(g, seg=seg, f=f, ext=ext):
                    t = topo(g, seg); k = pick(t, f, ext)
                    if k is None: return None
                    out = [[t.bg] * t.w for _ in range(t.h)]
                    for y, x in t.objs[k] + t.inner[k]: out[y][x] = g[y][x]
                    return out
                yield (f"topo:keep[{sname(seg)},{f},{ext}]", 4, fn)
            else:
                for part in ('bbox', 'shrink', 'inner'):
                    def fn(g, seg=seg, f=f, ext=ext, part=part):
                        t = topo(g, seg); k = pick(t, f, ext)
                        if k is None: return None
                        cells = t.inner[k] if part == 'inner' else t.objs[k]
                        if not cells: return None
                        r0, c0, r1, c1 = bbox(cells)
                        if part == 'shrink':
                            r0, c0, r1, c1 = r0 + 1, c0 + 1, r1 - 1, c1 - 1
                            if r1 < r0 or c1 < c0: return None
                        return [row[c0:c1 + 1] for row in g[r0:r1 + 1]]
                    yield (f"topo:crop[{sname(seg)},{f},{ext},{part}]", 4 + (seg[1] is None), fn)


# ------------------------------------------------------------------ 5. path-connected component of a seed point
def fam_topo_seed_region(train):
    """Seeds = cells that are neither background nor the dominant wall colour.  The region of a seed is the
    path-connected component of background cells touching it (walls = every other non-background cell).
    The region (whole, or only its boundary layer: cells adjacent to a wall or to the grid frame) is painted
    with the seed's colour or an induced colour; walls that do not touch any seeded region are kept or
    deleted (induced)."""
    if not same_shape(train): return

    fb = fixed_bg(train)

    def parse(g, fixed_wall=None):
        bg = bg_of(g) if fb == 'mode' else fb
        cs = Counter(v for r in g for v in r if v != bg)
        if len(cs) < 2: return None
        wall = cs.most_common(1)[0][0] if fixed_wall is None else fixed_wall
        seeds = [(y, x) for y in range(H(g)) for x in range(W(g)) if g[y][x] not in (bg, wall)]
        return bg, wall, seeds

    def regions(g, bg, seeds, conn):
        nb = N8 if conn == 8 else N4; h, w = H(g), W(g)
        lab = {}; res = []
        for sy, sx in seeds:
            st = [(sy + dy, sx + dx) for dy, dx in nb if 0 <= sy + dy < h and 0 <= sx + dx < w and g[sy + dy][sx + dx] == bg]
            st = [c for c in st if c not in lab]
            idx = len(res); cells = []
            for c in st: lab[c] = idx
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in nb:
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < h and 0 <= xx < w and (yy, xx) not in lab and g[yy][xx] == bg:
                        lab[(yy, xx)] = idx; st.append((yy, xx))
            res.append(((sy, sx), cells))
        return res

    def render(g, conn, paint, colour, walls, fw=None):
        pr = parse(g, fw)
        if pr is None: return None
        bg, wall, seeds = pr
        if not seeds or len(seeds) > 30: return None
        h, w = H(g), W(g); out = [r[:] for r in g]
        regs = regions(g, bg, seeds, conn)
        allr = set(seeds)
        island = set()      # wall cells of components not touching the grid frame (inner boundary components)
        if paint == 'outer8':
            for cells in _objects(g, bg, 'm8'):
                if not any(y in (0, h - 1) or x in (0, w - 1) for y, x in cells): island.update(cells)
        for (sy, sx), cells in regs:
            allr.update(cells)
            c = g[sy][sx] if colour == 'seed' else colour
            mine = set(cells) | {(sy, sx)}
            for y, x in cells:
                if paint != 'whole':
                    nb = N4 if paint == 'edge4' else N8
                    if all(0 <= y + dy < h and 0 <= x + dx < w and ((y + dy, x + dx) in mine or (y + dy, x + dx) in island)
                           for dy, dx in nb): continue
                out[y][x] = c
        if walls == 'del':
            for cells in _objects(g, bg, 'm8'):
                if any(g[y][x] != wall for y, x in cells): continue
                if not any((y + dy, x + dx) in allr for y, x in cells for dy, dx in N8):
                    for y, x in cells: out[y][x] = bg
        return out

    pr = [parse(p['input']) for p in train]
    if any(q is None or not q[2] for q in pr): return
    ws = {q[1] for q in pr}
    fw = next(iter(ws)) if len(ws) == 1 else None     # wall colour shared by all pairs is fixed
    # literal colour candidates: output colours on changed cells
    lit = Counter(p['output'][y][x] for p in train for y in range(H(p['input'])) for x in range(W(p['input']))
                  if p['input'][y][x] != p['output'][y][x])
    if not lit: return
    cols = ['seed'] + [c for c, _ in lit.most_common(2)]
    for conn, paint, colour, walls in product((4, 8), ('whole', 'edge8', 'edge4', 'outer8'), cols, ('keep', 'del')):
        fn = lambda g, conn=conn, paint=paint, colour=colour, walls=walls: render(g, conn, paint, colour, walls, fw)
        if fn(train[0]['input']) != train[0]['output']: continue
        yield (f"topo:seed-region[{conn},{paint},{colour},{walls}]", 4, fn)


# ------------------------------------------------------------------ 6. holes filled by homotopy-free shape vote
def _norm(cells, d8=False):
    res = []
    for t in (range(8) if d8 else (0,)):
        cs = []
        for y, x in cells:
            if t & 1: y, x = x, y
            if t & 2: y = -y
            if t & 4: x = -x
            cs.append((y, x))
        y0 = min(y for y, _ in cs); x0 = min(x for _, x in cs)
        res.append(tuple(sorted((y - y0, x - x0) for y, x in cs)))
    return min(res)


def fam_topo_hole_vote(train):
    """Host = the object with the most holes.  Each hole (a zone of background cells) takes the majority
    colour of the free objects (outside the host's interior) congruent to the hole (translation, or up to
    dihedral maps); holes with no congruent object stay.  Output: the whole grid, or the host's crop."""
    same = same_shape(train)
    fb = fixed_bg(train)
    for seg, d8 in product((('c8', fb), ('c4', fb)), (False, True)):
        def render(g, seg=seg, d8=d8, crop_out=not same):
            t = topo(g, seg)
            if len(t.objs) < 2 or len(t.objs) > 400: return None
            k = pick(t, 'n_holes', 'max')
            if k is None or not t.nholes[k]: return None
            votes = {}
            for j, cells in enumerate(t.objs):
                if j == k or cells[0] in t.innerset[k]: continue
                votes.setdefault(_norm(cells, d8), Counter())[t.colour(j)] += 1
            out = [r[:] for r in g]; n = 0
            for a, z in t.zones:
                if a != k: continue
                hole = [c for c in z if g[c[0]][c[1]] == t.bg]
                if len(hole) != len(z): continue
                v = votes.get(_norm(hole, d8))
                if not v: continue
                top = v.most_common(2)
                if len(top) > 1 and top[0][1] == top[1][1]: return None
                n += 1
                for y, x in hole: out[y][x] = top[0][0]
            if not n: return None
            if crop_out:
                r0, c0, r1, c1 = bbox(t.objs[k]); return [row[c0:c1 + 1] for row in out[r0:r1 + 1]]
            return out
        yield (f"topo:hole-vote[{sname(seg)}{',d8' if d8 else ''}{',crop' if not same else ''}]", 5, render)


FAMILIES = (fam_topo_recolour, fam_topo_fill, fam_topo_cellmap, fam_topo_select, fam_topo_seed_region, fam_topo_hole_vote)


In [ ]:
%%writefile /kaggle/working/predict_m1.py
"""Kaggle driver for the M1b lattice/RDR probe (occupancy2.py).

Every prediction comes from rules induced on the task's own training pairs over the frozen prior
vocabulary; unresolved slots get a 1x1 [[0]] placeholder. Deterministic (sorted iteration), so a
local run and the Kaggle run over the same file must produce the same submission digest.
"""
import argparse, hashlib, json, os, signal, sys, time

PLACEHOLDER = [[0]]


class TaskTimeout(BaseException):
    pass


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--candidate", required=True)      # dir containing image.py / ARCGraph.py
    ap.add_argument("--probe", required=True)          # dir containing occupancy2.py
    ap.add_argument("--vocab", required=True)
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--task-seconds", type=int, default=120)
    args = ap.parse_args()
    os.environ["M1B_VOCAB"] = args.vocab
    sys.path[:0] = [args.probe, args.candidate]
    import occupancy2 as probe
    probe.Timeout = TaskTimeout
    ch = json.load(open(args.challenges))
    signal.signal(signal.SIGALRM, lambda *a: (_ for _ in ()).throw(TaskTimeout()))
    sub, solved_slots = {}, 0
    for k in sorted(ch):
        task = ch[k]
        slots = [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in task["test"]]
        try:
            signal.alarm(args.task_seconds)
            attempts, _, _ = probe.solve(task)
            for rank, a in enumerate(attempts[:2]):
                for i, g in enumerate(a["preds"]):
                    if g and all(isinstance(r, list) and r for r in g):
                        slots[i]["attempt_1" if rank == 0 else "attempt_2"] = g
        except TaskTimeout:
            pass
        except Exception:
            pass
        finally:
            signal.alarm(0)
        sub[k] = slots
    blob = json.dumps(sub, sort_keys=True).encode()
    open(args.out, "w").write(json.dumps(sub))
    print(json.dumps({"tasks": len(sub), "digest": hashlib.sha256(blob).hexdigest()}))


if __name__ == "__main__":
    main()


In [ ]:
# Parity gate: the notebook must replicate the local result on the public 120-task evaluation set.
PROBE_SHA = '68f1332ce4f972a676f027ba9519bdf53cd1bb054d07f57a289c214887af87f2'
VOCAB = 'fill_interior,ray,connect,border,bbox_fill,outline8,L_inertia,D12,S_holes,D4,D1,S_size,P_pixel,P_frame,D10,S_atoms,G_dsl'
EVAL_SHA = 'e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8'
EXPECTED_DIGEST = 'c93d287bfd585cd4c60e39d00ca09a52e09e50c8f745d6a4f4862be4e8569f32'
EXPECTED_CORRECT = 49
assert hashlib.sha256(Path('/kaggle/working/probe/occupancy2.py').read_bytes()).hexdigest() == PROBE_SHA
evals = [p for p in Path('/kaggle/input').rglob('arc-agi_evaluation_challenges.json')
         if hashlib.sha256(p.read_bytes()).hexdigest() == EVAL_SHA]
assert evals, 'identical public evaluation file not found; parity cannot be verified'
sols = list(evals[0].parent.glob('arc-agi_evaluation_solutions.json'))
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
def run(ch, out):
    subprocess.run([sys.executable, '/kaggle/working/predict_m1.py', '--candidate', str(cand),
                    '--probe', '/kaggle/working/probe', '--vocab', VOCAB, '--challenges', str(ch),
                    '--out', out, '--task-seconds', '300'], check=True, env=env)
    return json.load(open(out))
ev = run(evals[0], '/kaggle/working/parity_eval.json')
digest = hashlib.sha256(json.dumps(ev, sort_keys=True).encode()).hexdigest()
correct = None
if sols:
    ans = json.load(open(sols[0]))
    correct = sum(any(ev[k][i][a] == t for a in ('attempt_1', 'attempt_2')) for k in ans for i, t in enumerate(ans[k]))
report = {'eval_file': str(evals[0]), 'prediction_digest': digest, 'digest_match': digest == EXPECTED_DIGEST,
          'correct_of_172': correct, 'expected_correct': EXPECTED_CORRECT}
json.dump(report, open('/kaggle/working/parity_report.json', 'w'), indent=1)
print(report)
assert digest == EXPECTED_DIGEST, 'PARITY FAILED: predictions differ from the frozen local run'
assert correct in (None, EXPECTED_CORRECT), f'PARITY FAILED: expected {EXPECTED_CORRECT} correct, got {correct}'


In [ ]:
sub = run(challenges[0], '/kaggle/working/submission.json')
src = json.load(open(challenges[0]))
assert set(sub) == set(src) and all(len(sub[k]) == len(src[k]['test']) for k in src)
print(len(sub), 'tasks written')
